In [ ]:
from networkx.convert import to_networkx_graph
from torch_geometric.utils import to_networkx


import os
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
os.chdir(PROJECT_DIR)

print("PROJECT_DIR:", PROJECT_DIR)
print("Current directory:", Path.cwd())

import numpy as np

from matplotlib import pyplot as plt

from experiments import (
    experiment_train,
    experiment_local_attack_direct,
    experiment_global_attack_direct
)
import helpers.selector_pipeline_helpers

from sparse_smoothing.utils import load_and_standardize
from sparse_smoothing.models import GCN



%load_ext autoreload
%autoreload 2

files = ["cache/demo.json", "cache/demo/demo_1.pt", "cache/evasion_global_adj.json", "cache/evasion_global_attr.json", "cache/evasion_global_adj/evasion_global_adj_1.pt", "cache/evasion_global_attr/evasion_global_attr_1.pt"]

for file_path in files:
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"{file_path} has been deleted.")
    else:
        print(f"{file_path} does not exist.")

%matplotlib inline


In [ ]:
# ── Dataset / split ─────────────────────────────────────────────────────────
DATASET = 'cora_ml'
SAMPLING = 'stratified'
SPLIT = 'ratio'

# All experiments in this notebook are repeated for these seeds.
# Add/remove values here; every raw table keeps `seed`, while displayed tables
# and plots aggregate over this list.
SEEDS = [0]
if not SEEDS:
    raise ValueError("SEEDS must contain at least one integer seed.")
SEEDS = [int(seed) for seed in SEEDS]
SEED = SEEDS[0]  # compatibility alias; never use this for an experiment loop
N_SEEDS = len(SEEDS)

TRAIN_RATIO = 0.15
VAL_RATIO = 0.10
TEST_RATIO = 0.30

# ── Victim model ─────────────────────────────────────────────────────────────
MODEL_NAME = 'GCN'
MODEL_LABEL = 'GCN'
TRAIN_EPOCHS = 200
HIDDEN = 64
INDUCTIVE = True
DROPOUT_VICTIM = 0.5
LR_VICTIM = 1e-2
WEIGHT_DECAY_VICTIM = 1e-3
PATIENCE_VICTIM = 300
MAX_EPOCHS_VICTIM = 3000

# ── V4 mining params ─────────────────────────────────────────────────────────
USE_PRBCD_CANDIDATES = True
N_PRBCD_RUNS = 1
MINER = 'prbcd'
MINER_LOCAL_FACTOR = 0.5
DEGREE_CAP_PCT = None
PRBCD_BUDGET_FRACTION = 0.5
CANDIDATE_SET_SIZES = [2500, 5000, 10000]
PRBCD_CANDIDATE_FRACTIONS = [0.0]
SCORING_MODES = ["endpoint","subset_accuracy_drop"]
ENDPOINT_MINING_HOPS = [2]
LABEL_MODES = ["continuous"]
EXTREME_FRACTIONS = [0.3]
PRBCD_EPOCHS = 5
PRBCD_SEARCH_SPACE = 200_000
PRBCD_N_EPOCHS_RESAMPLING = 1


# ── RQ3 subgraph-selector sweep ─────────────────────────────────────────────
# Every requested combination is constructed and trained. Only §8 consumes
# the complete Cartesian product. Legacy RQ3 diagnostics use the primary
# combination selected below.
# A value of 1.0 is an explicit whole-graph selector condition. It uses all
# graph nodes/edges for message passing, while candidate labels are still
# obtained from a finite sampled candidate set.
RQ3_SUBGRAPH_FRACTIONS = [1.0]
RQ3_SUBGRAPH_TRAINING_CANDIDATE_SIZES = [2500,5000,10000]
RQ3_SUBGRAPH_SEEDS = [20]
RQ3_SUBGRAPH_SCORING_MODES = ["endpoint","subset_accuracy_drop"]
RQ3_SUBGRAPH_METHODS = ["stratified_context", "forest_fire"]

# One combination exposed through the legacy names `mining_runs`,
# `lp_training_runs`, and `LP_MODEL_VARIANTS`. This prevents all experiments
# except §8 from silently expanding over the new sweep dimensions.
RQ3_PRIMARY_SUBGRAPH_FRACTION = 1.0
RQ3_PRIMARY_TRAINING_CANDIDATE_SIZE = 10000
RQ3_PRIMARY_SUBGRAPH_SEED = 20
RQ3_PRIMARY_SUBGRAPH_METHOD = "stratified_context"

# ── V4 subset-scoring params ─────────────────────────────────────────────────
N_SUBSETS = 1000
SUBSET_FRACTION = 0.05

# The helper owns the exact schema identifiers. The notebook imports them in
# the mining cell and writes them into every cache key and run dictionary.
# Canonical raw target:
#   mean harmful accuracy drop of subsets in which an edge was selected.

# ── Scorer training params ───────────────────────────────────────────────────
SCORER_HIDDEN = 64
SCORER_EPOCHS = 200
SCORER_LR = 0.01
VAL_FRACTION = 0.2
PATIENCE = 20
# Continuous subset scores are not class-balanced. Keep None unless deliberately
# running a separate projected-label ablation later.
BALANCE_RATIO = None

# ── Reproducibility / aggregation helpers ────────────────────────────────────
import os
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from datetime import datetime

# ── Thesis-ready run output ──────────────────────────────────────────────────
# Created once per kernel session so re-running the configuration cell does not
# scatter figures across multiple folders.
THESIS_RUN_ID = globals().get(
    "THESIS_RUN_ID",
    datetime.now().strftime("%Y%m%d_%H%M%S"),
)
THESIS_RUN_DIR = (
    Path("extendedPlotting")
    / "thesis_ready_runs"
    / f"{DATASET}__{THESIS_RUN_ID}"
)
RUN_PLOTS_DIR = THESIS_RUN_DIR / "plots"
RUN_PLOTS_DIR.mkdir(parents=True, exist_ok=True)


def thesis_plot_path(filename: str) -> Path:
    """Return a path inside the flat, run-specific plot directory."""
    path = RUN_PLOTS_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    return path

print("Thesis run directory:", THESIS_RUN_DIR.resolve())
print("All figures will be saved to:", RUN_PLOTS_DIR.resolve())


def set_global_seed(seed: int, deterministic: bool = True) -> None:
    """Seed Python, NumPy and PyTorch before every independent run."""
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def activate_seed_context(context: dict) -> None:
    """Expose one trained victim context through the legacy notebook globals."""
    keys = [
        "seed", "train_statistics", "clean_acc", "model", "graph",
        "idx_train", "idx_val", "idx_test", "device", "n_nodes",
        "n_edges_directed", "n_undirected", "attr_matrix", "adj_matrix",
        "labels_raw", "edge_index", "edge_weight", "attr", "labels",
    ]
    for key in keys:
        if key in context:
            globals()["SEED" if key == "seed" else key] = context[key]


def aggregate_over_seeds(
    frame: pd.DataFrame,
    group_cols: list[str],
    metric_cols: list[str] | None = None,
) -> pd.DataFrame:
    """Return mean/std/SEM and number of distinct seeds for numeric metrics."""
    if frame is None or frame.empty:
        return pd.DataFrame()
    missing = [column for column in group_cols if column not in frame.columns]
    if missing:
        raise KeyError(f"Missing grouping columns: {missing}")
    if "seed" not in frame.columns:
        raise KeyError("Every raw experiment table must contain a 'seed' column.")
    if metric_cols is None:
        metric_cols = [
            column for column in frame.select_dtypes(include=[np.number]).columns
            if column not in set(group_cols) | {"seed"}
        ]
    metric_cols = [column for column in metric_cols if column in frame.columns]

    if not group_cols:
        row = {"n_seeds": int(frame["seed"].nunique())}
        for metric in metric_cols:
            values = pd.to_numeric(frame[metric], errors="coerce")
            count = int(values.notna().sum())
            mean = float(values.mean()) if count else np.nan
            std = float(values.std(ddof=1)) if count > 1 else 0.0
            row[f"{metric}_mean"] = mean
            row[f"{metric}_std"] = std
            row[f"{metric}_count"] = count
            row[f"{metric}_sem"] = std / np.sqrt(max(1, count))
        return pd.DataFrame([row])

    grouped = frame.groupby(group_cols, dropna=False)
    pieces = []
    if metric_cols:
        stats = grouped[metric_cols].agg(["mean", "std", "count"])
        stats.columns = [f"{metric}_{stat}" for metric, stat in stats.columns]
        stats = stats.reset_index()
        for metric in metric_cols:
            std_col = f"{metric}_std"
            count_col = f"{metric}_count"
            if std_col in stats:
                stats[std_col] = stats[std_col].fillna(0.0)
                stats[f"{metric}_sem"] = stats[std_col] / np.sqrt(
                    stats[count_col].clip(lower=1)
                )
        pieces.append(stats)
    seed_counts = grouped["seed"].nunique().rename("n_seeds").reset_index()
    if not pieces:
        return seed_counts
    return pieces[0].merge(seed_counts, on=group_cols, how="left")


def add_mean_std_band(ax, x, mean, std, *, label=None, **plot_kwargs):
    """Plot a seed mean and a ±1 standard-deviation band."""
    x = np.asarray(x)
    mean = np.asarray(mean, dtype=float)
    std = np.nan_to_num(np.asarray(std, dtype=float), nan=0.0)
    line = ax.plot(x, mean, label=label, **plot_kwargs)[0]
    ax.fill_between(x, mean - std, mean + std, alpha=0.18, color=line.get_color())
    return line

set_global_seed(SEED)
print(f"Configured {N_SEEDS} seeds: {SEEDS}")


In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

seed_contexts = []
train_curve_rows = []
train_summary_rows = []

for seed in SEEDS:
    set_global_seed(seed)
    print(f"Training victim model for seed={seed}")

    stats = experiment_train.run(
        data_dir='./data',
        dataset=DATASET,
        model_params=dict(
            label=MODEL_LABEL,
            model=MODEL_NAME,
            do_cache_adj_prep=True,
            n_filters=64,
            dropout=DROPOUT_VICTIM,
            svd_params=None,
            jaccard_params=None,
            gdc_params={"alpha": 0.15, "k": 64},
        ),
        train_params=dict(
            lr=LR_VICTIM,
            weight_decay=WEIGHT_DECAY_VICTIM,
            patience=PATIENCE_VICTIM,
            max_epochs=MAX_EPOCHS_VICTIM,
        ),
        binary_attr=False,
        make_undirected=True,
        seed=seed,
        artifact_dir='cache',
        model_storage_type='demo_custom_split',
        ppr_cache_params=dict(),
        device="cpu",
        data_device="cpu",
        display_steps=100,
        debug_level="info",
        custom_split_ratios=None,
    )

    model_seed = stats["model"]
    graph_seed = stats["graph"]
    idx_train_seed = stats["idx_train"]
    idx_val_seed = stats["idx_val"]
    idx_test_seed = stats["idx_test"]
    model_seed.eval()

    attr_matrix_seed, adj_matrix_seed, labels_raw_seed = graph_seed[:3]
    model_device = next(model_seed.parameters()).device
    row, col, value = adj_matrix_seed.coo()
    edge_index_seed = torch.stack([row, col], dim=0).long().to(model_device)
    edge_weight_seed = (
        torch.ones(edge_index_seed.size(1), dtype=torch.float32, device=model_device)
        if value is None else value.float().to(model_device)
    )
    attr_seed = attr_matrix_seed.float().to(model_device)
    labels_seed = labels_raw_seed.long().to(model_device)
    n_nodes_seed = int(adj_matrix_seed.sizes()[0])
    n_edges_directed_seed = int(adj_matrix_seed.nnz())
    n_undirected_seed = n_edges_directed_seed // 2
    clean_acc_seed = float(stats["accuracy"])

    context = {
        "seed": int(seed),
        "train_statistics": stats,
        "clean_acc": clean_acc_seed,
        "model": model_seed,
        "graph": graph_seed,
        "idx_train": idx_train_seed,
        "idx_val": idx_val_seed,
        "idx_test": idx_test_seed,
        "device": model_device,
        "n_nodes": n_nodes_seed,
        "n_edges_directed": n_edges_directed_seed,
        "n_undirected": n_undirected_seed,
        "attr_matrix": attr_matrix_seed,
        "adj_matrix": adj_matrix_seed,
        "labels_raw": labels_raw_seed,
        "edge_index": edge_index_seed,
        "edge_weight": edge_weight_seed,
        "attr": attr_seed,
        "labels": labels_seed,
    }
    seed_contexts.append(context)

    for split, values in (
        ("train", stats.get("trace_train", [])),
        ("validation", stats.get("trace_val", [])),
    ):
        for epoch, loss in enumerate(values, start=1):
            train_curve_rows.append({
                "seed": int(seed),
                "split": split,
                "epoch": int(epoch),
                "loss": float(loss),
            })

    train_summary_rows.append({
        "seed": int(seed),
        "clean_accuracy": clean_acc_seed,
        "n_train_epochs": len(stats.get("trace_train", [])),
        "n_val_epochs": len(stats.get("trace_val", [])),
    })

SEED_CONTEXTS = seed_contexts
SEED_CONTEXT_BY_SEED = {context["seed"]: context for context in seed_contexts}
activate_seed_context(seed_contexts[0])

train_curve_df = pd.DataFrame(train_curve_rows)
train_summary_raw_df = pd.DataFrame(train_summary_rows)
train_summary_df = aggregate_over_seeds(
    train_summary_raw_df,
    group_cols=[],
    metric_cols=["clean_accuracy", "n_train_epochs", "n_val_epochs"],
)

curve_summary = (
    train_curve_df.groupby(["split", "epoch"], as_index=False)
    .agg(loss_mean=("loss", "mean"), loss_std=("loss", "std"), n_seeds=("seed", "nunique"))
)
curve_summary["loss_std"] = curve_summary["loss_std"].fillna(0.0)
curve_summary = curve_summary[curve_summary["n_seeds"] == N_SEEDS].copy()
if curve_summary.empty:
    raise RuntimeError("No victim-training epochs are shared by every configured seed.")

fig, ax = plt.subplots(figsize=(8, 4.5))
for split, group in curve_summary.groupby("split", sort=False):
    group = group.sort_values("epoch")
    add_mean_std_band(
        ax,
        group["epoch"],
        group["loss_mean"],
        group["loss_std"],
        label=f"{split} mean ± SD",
    )
ax.set_xlabel('Epoch $t$')
ax.set_ylabel("Loss")
ax.set_title("Victim-model training loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    RUN_PLOTS_DIR / "victim_training_loss.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

clean_mean = train_summary_raw_df["clean_accuracy"].mean()
clean_std = train_summary_raw_df["clean_accuracy"].std(ddof=1) if N_SEEDS > 1 else 0.0
print(f"Victim accuracy over {N_SEEDS} seeds: {100*clean_mean:.2f}% ± {100*clean_std:.2f}% SD")
display(train_summary_df)


In [ ]:
import torch

# Dataset structure is seed-invariant; inspect the first trained context.
activate_seed_context(SEED_CONTEXTS[0])
attr_matrix, adj_matrix, labels_raw = graph[:3]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Display ────────────────────────────────────────────────────────────────
n_nodes = adj_matrix.sizes()[0]
n_edges_directed = adj_matrix.nnz()
n_undirected = n_edges_directed // 2

print(f"nodes={n_nodes}")
print(f"Directed edges: {n_edges_directed}")
print(f"Undirected edges: {n_undirected}")
print(f"features={attr_matrix.shape}")
print(f"classes={len(torch.unique(labels_raw))}")

# ── Convert adjacency SparseTensor -> edge_index / edge_weight ─────────────
row, col, value = adj_matrix.coo()

edge_index = torch.stack([row, col], dim=0).long().to(device)

if value is None:
    edge_weight = torch.ones(edge_index.size(1), dtype=torch.float32, device=device)
else:
    edge_weight = value.float().to(device)

# ── Convert features / labels ──────────────────────────────────────────────
attr = attr_matrix.float().to(device)
labels = labels_raw.long().to(device)

# ── Safety checks ──────────────────────────────────────────────────────────
print("edge_index shape:", edge_index.shape)
print("edge_weight shape:", edge_weight.shape)
print("max node id in edge_index:", int(edge_index.max()))
print("n_nodes:", n_nodes)

assert int(edge_index.max()) < n_nodes
assert attr.shape[0] == n_nodes
assert labels.shape[0] == n_nodes
'''
train_idx_np, val_idx_np, test_idx_np = train_val_test_split(
    graph.labels,
    train_size=0.05,
    val_size=0.05,
    test_size=0.9,
    mode="stratified",
    seed=SEED,
)

train_idx = torch.as_tensor(train_idx_np, dtype=torch.long, device=device)
val_idx = torch.as_tensor(val_idx_np, dtype=torch.long, device=device)
test_idx = torch.as_tensor(test_idx_np, dtype=torch.long, device=device)'''

# RQ2 — Mining and Evaluating Harmful Edge Perturbations

RQ1 showed that the scalability of PR-BCD depends on restricting optimization to a sampled candidate block. RQ2 therefore studies whether victim-model evaluations can produce a useful harmfulness ranking over candidate edge flips.

For `subset_accuracy_drop`, one random subset of candidate flips is applied at a time. Let \(d_t=A(G)-A(G\oplus S_t)\) be the signed evaluation-accuracy drop of subset \(S_t\). For candidate edge \(e_i\), the canonical mined target is

\[
s_i
=
\frac{\sum_{t:e_i\in S_t}\max(0,d_t)}
     {\sum_t \mathbf{1}[e_i\in S_t]}.
\]

Thus, \(s_i\) is the **mean harmful subset drop observed when edge \(e_i\) was selected**. It corrects for unequal inclusion frequency.

This is an edge-associated subset score, not an isolated causal effect. Interactions, redundancy, and cancellation between jointly flipped edges remain part of the signal. The later oracle and PR-BCD experiments test whether the resulting noisy score nevertheless provides a useful ranking.

Every mining run stores:

- signed and harmful subset drops;
- inclusion counts;
- raw mean harmful scores;
- normalized scores in \([0,1]\);
- within-run percentile ranks;
- an explicit schema version and score definition.


In [ ]:
from helpers.selector_pipeline_helpers import EndpointPRBCDV4Scorer, _edge_set
from IPython.display import display
import math
import random
import pandas as pd


def _safe_config_value(value):
    text = f"{value:.12g}" if isinstance(value, float) else str(value)
    return (text.strip().replace(" ", "").replace("/", "-").replace("\\", "-")
            .replace(".", "p").replace("+", "").replace("-", "m"))


def _resolve_candidate_set_sizes():
    configured = globals().get("CANDIDATE_SET_SIZES", None)
    if configured is None:
        fallback_size = max(1, round(SEED_CONTEXTS[0]["n_undirected"] * PRBCD_BUDGET_FRACTION * N_PRBCD_RUNS))
        return [fallback_size]
    if isinstance(configured, int):
        configured = [configured]
    sizes = [int(value) for value in configured]
    if not sizes or any(size <= 0 for size in sizes):
        raise ValueError("CANDIDATE_SET_SIZES must contain positive values.")
    return sizes


def _resolve_prbcd_candidate_fractions():
    configured = globals().get("PRBCD_CANDIDATE_FRACTIONS", [globals().get("PRBCD_CANDIDATE_FRACTION", 0.5)])
    if isinstance(configured, (int, float)):
        configured = [configured]
    fractions = [float(value) for value in configured]
    if not fractions or any(not 0.0 <= fraction <= 1.0 for fraction in fractions):
        raise ValueError("PRBCD_CANDIDATE_FRACTIONS must be in [0, 1].")
    return fractions


def _candidate_config_id(candidate_set_size: int, prbcd_candidate_fraction: float) -> str:
    return (f"candN-{_safe_config_value(candidate_set_size)}"
            f"__prbcdFrac-{_safe_config_value(prbcd_candidate_fraction)}")


def sample_random_candidates(num_nodes: int, count: int, *, forbidden=None, seed: int):
    forbidden = set() if forbidden is None else set(forbidden)
    n_possible = num_nodes * (num_nodes - 1) // 2
    n_available = n_possible - len(forbidden)
    if count > n_available:
        raise ValueError(f"Cannot sample {count} candidates; only {n_available} remain.")
    rng = random.Random(int(seed))
    sampled = set()
    while len(sampled) < count:
        u, v = rng.randrange(num_nodes), rng.randrange(num_nodes)
        if u == v:
            continue
        edge = (min(u, v), max(u, v))
        if edge not in forbidden:
            sampled.add(edge)
    return sampled


candidate_set_sizes = _resolve_candidate_set_sizes()
prbcd_candidate_fractions = _resolve_prbcd_candidate_fractions()
candidate_runs = []
_candidate_mining_cache = {}

for context in SEED_CONTEXTS:
    activate_seed_context(context)
    seed = context["seed"]
    set_global_seed(seed)
    n_possible_candidates = n_nodes * (n_nodes - 1) // 2
    orig_edges = _edge_set(edge_index)

    for size_index, candidate_set_size in enumerate(candidate_set_sizes):
        if candidate_set_size > n_possible_candidates:
            raise ValueError(f"candidate_set_size={candidate_set_size} exceeds {n_possible_candidates} pairs.")
        budget = max(1, math.ceil(candidate_set_size / max(1, int(N_PRBCD_RUNS))))
        prbcd_budget_fraction = budget / max(1, int(n_undirected))

        for fraction_index, prbcd_candidate_fraction in enumerate(prbcd_candidate_fractions):
            config_id = _candidate_config_id(candidate_set_size, prbcd_candidate_fraction)
            target_prbcd_count = round(candidate_set_size * prbcd_candidate_fraction)
            prbcd_candidates = set()

            if USE_PRBCD_CANDIDATES and target_prbcd_count > 0:
                cache_key = (seed, candidate_set_size)
                if cache_key not in _candidate_mining_cache:
                    scorer = EndpointPRBCDV4Scorer(
                        n=n_nodes, edge_index=edge_index, edge_weight=edge_weight,
                        attr=attr, labels=labels, attacked_model=model,
                        idx_attack=idx_test, test_idx=idx_val, device=device,
                        block_size=PRBCD_SEARCH_SPACE,
                        n_epochs_resampling=PRBCD_N_EPOCHS_RESAMPLING,
                        n_subsets=N_SUBSETS, subset_fraction=SUBSET_FRACTION,
                        balance_ratio=BALANCE_RATIO, store_candidates=True,
                    )
                    mined = scorer.mine_prbcd_endpoint_candidates(
                        n_perturbations=budget,
                        tag=f"endpointPRBCD__{config_id}__seed-{seed}",
                        rng_seed=seed,
                        n_candidates=PRBCD_SEARCH_SPACE,
                        prbcd_epochs=PRBCD_EPOCHS,
                        n_prbcd_runs=N_PRBCD_RUNS,
                        n_epochs_resampling=PRBCD_N_EPOCHS_RESAMPLING,
                    )
                    _candidate_mining_cache[cache_key] = {
                        (min(int(u), int(v)), max(int(u), int(v)))
                        for u, v in mined if int(u) != int(v)
                    }
                prbcd_candidates = set(_candidate_mining_cache[cache_key])

            selection_rng = random.Random(seed + 10_000 * size_index + 101 * fraction_index)
            selected_prbcd = (
                set(selection_rng.sample(sorted(prbcd_candidates), target_prbcd_count))
                if len(prbcd_candidates) > target_prbcd_count else set(prbcd_candidates)
            )
            n_random_needed = candidate_set_size - len(selected_prbcd)
            random_candidates = sample_random_candidates(
                n_nodes, n_random_needed, forbidden=selected_prbcd,
                seed=seed + 20_000 * size_index + 211 * fraction_index + 17,
            )
            candidates_for_config = list(selected_prbcd | random_candidates)
            selection_rng.shuffle(candidates_for_config)
            if len(candidates_for_config) != candidate_set_size:
                raise RuntimeError(f"Expected {candidate_set_size} candidates, got {len(candidates_for_config)}")

            n_removals = sum(edge in orig_edges for edge in candidates_for_config)
            candidate_runs.append({
                "seed": int(seed),
                "context": context,
                "candidate_config_id": config_id,
                "candidate_set_size": int(candidate_set_size),
                "prbcd_candidate_fraction": float(prbcd_candidate_fraction),
                "budget": int(budget),
                "prbcd_budget_fraction": float(prbcd_budget_fraction),
                "n_prbcd_requested": int(target_prbcd_count),
                "n_prbcd_mined_unique": int(len(prbcd_candidates)),
                "n_prbcd_selected": int(len(selected_prbcd)),
                "n_random_candidates": int(len(random_candidates)),
                "actual_prbcd_fraction": float(len(selected_prbcd) / len(candidates_for_config)),
                "n_add_candidates": int(len(candidates_for_config) - n_removals),
                "n_delete_candidates": int(n_removals),
                "candidates": candidates_for_config,
                "selected_prbcd": selected_prbcd,
                "random_candidates": random_candidates,
            })

if not candidate_runs:
    raise RuntimeError("No candidate configurations were created.")

candidate_config_raw_df = pd.DataFrame([
    {key: value for key, value in run.items()
     if key not in {"context", "candidates", "selected_prbcd", "random_candidates"}}
    for run in candidate_runs
])
_candidate_group_cols = ["candidate_config_id", "candidate_set_size", "prbcd_candidate_fraction"]
_candidate_metric_cols = [
    "budget", "prbcd_budget_fraction", "n_prbcd_requested", "n_prbcd_mined_unique",
    "n_prbcd_selected", "n_random_candidates", "actual_prbcd_fraction",
    "n_add_candidates", "n_delete_candidates",
]
candidate_config_df = aggregate_over_seeds(candidate_config_raw_df, _candidate_group_cols, _candidate_metric_cols)
display(candidate_config_df)

candidate_runs_by_key = {(run["seed"], run["candidate_config_id"]): run for run in candidate_runs}
print(f"Created {len(candidate_runs)} raw candidate runs = {len(SEEDS)} seeds × "
      f"{len(candidate_set_sizes)} sizes × {len(prbcd_candidate_fractions)} fractions.")


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

CANDIDATE_DIAGNOSTIC_OUT_DIR = Path("extendedPlotting") / "candidate_diagnostics_multiseed"
CANDIDATE_DIAGNOSTIC_OUT_DIR.mkdir(parents=True, exist_ok=True)

candidate_degree_summary_rows = []
degree_profile_rows = []

for run in candidate_runs:
    context = run["context"]
    degrees_run = np.bincount(context["edge_index"][0].cpu().numpy(), minlength=context["n_nodes"])
    candidates_run = run["candidates"]
    cand_nodes = sorted(set(u for u, v in candidates_run) | set(v for u, v in candidates_run))
    cand_degs = degrees_run[cand_nodes] if cand_nodes else np.array([], dtype=int)
    endpoint_mask = np.zeros(context["n_nodes"], dtype=bool)
    endpoint_mask[cand_nodes] = True
    all_counts = np.bincount(degrees_run)
    endpoint_counts = np.bincount(degrees_run[endpoint_mask], minlength=len(all_counts))
    fractions = np.divide(endpoint_counts, all_counts, out=np.zeros_like(endpoint_counts, dtype=float), where=all_counts > 0)

    candidate_degree_summary_rows.append({
        "seed": run["seed"],
        "candidate_config_id": run["candidate_config_id"],
        "candidate_set_size": run["candidate_set_size"],
        "prbcd_candidate_fraction": run["prbcd_candidate_fraction"],
        "actual_prbcd_fraction": run["actual_prbcd_fraction"],
        "n_candidate_endpoint_nodes": len(cand_nodes),
        "mean_candidate_endpoint_degree": float(cand_degs.mean()) if cand_degs.size else np.nan,
        "median_candidate_endpoint_degree": float(np.median(cand_degs)) if cand_degs.size else np.nan,
        "max_candidate_endpoint_degree": float(cand_degs.max()) if cand_degs.size else np.nan,
        "n_add_candidates": run["n_add_candidates"],
        "n_delete_candidates": run["n_delete_candidates"],
    })
    for degree, (all_count, endpoint_count, fraction) in enumerate(zip(all_counts, endpoint_counts, fractions)):
        degree_profile_rows.append({
            "seed": run["seed"], "candidate_config_id": run["candidate_config_id"],
            "degree": degree, "all_node_count": all_count,
            "candidate_endpoint_count": endpoint_count, "candidate_endpoint_fraction": fraction,
        })

candidate_degree_raw_df = pd.DataFrame(candidate_degree_summary_rows)
candidate_degree_summary_df = aggregate_over_seeds(
    candidate_degree_raw_df,
    ["candidate_config_id", "candidate_set_size", "prbcd_candidate_fraction"],
    ["actual_prbcd_fraction", "n_candidate_endpoint_nodes", "mean_candidate_endpoint_degree",
     "median_candidate_endpoint_degree", "max_candidate_endpoint_degree",
     "n_add_candidates", "n_delete_candidates"],
)
candidate_degree_raw_df.to_csv(CANDIDATE_DIAGNOSTIC_OUT_DIR / "candidate_degree_summary_raw.csv", index=False)
candidate_degree_summary_df.to_csv(CANDIDATE_DIAGNOSTIC_OUT_DIR / "candidate_degree_summary_averaged.csv", index=False)
display(candidate_degree_summary_df)

degree_profile_raw_df = pd.DataFrame(degree_profile_rows)
degree_profile_df = aggregate_over_seeds(
    degree_profile_raw_df, ["candidate_config_id", "degree"],
    ["all_node_count", "candidate_endpoint_count", "candidate_endpoint_fraction"],
)

for config_id, summary_group in candidate_degree_summary_df.groupby("candidate_config_id", sort=False):
    summary = summary_group.iloc[0]
    profile = degree_profile_df[degree_profile_df["candidate_config_id"] == config_id].sort_values("degree")
    profile = profile[profile["degree"] <= min(100, profile["degree"].max())]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    labels_bar = ["additions", "removals"]
    means = [summary["n_add_candidates_mean"], summary["n_delete_candidates_mean"]]
    stds = [summary["n_add_candidates_std"], summary["n_delete_candidates_std"]]
    axes[0].bar(labels_bar, means, yerr=stds, capsize=4)
    axes[0].set_ylabel("mean count ± SD")
    axes[0].set_title("Candidate action breakdown")

    add_mean_std_band(
        axes[1], profile["degree"], profile["candidate_endpoint_fraction_mean"],
        profile["candidate_endpoint_fraction_std"], label="mean ± SD", marker="o", linewidth=1,
    )
    axes[1].set_xlabel("node degree")
    axes[1].set_ylabel("fraction used as candidate endpoint")
    axes[1].set_ylim(0, 1.05)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    axes[1].set_title("Endpoint coverage by degree")
    fig.suptitle("Candidate-set diagnostics")
    fig.tight_layout()
    fig.savefig(RUN_PLOTS_DIR / f"candidate_diagnostics__{_safe_config_value(config_id)}.png", dpi=200, bbox_inches="tight")
    plt.show()


In [ ]:
# Debug breakpoint removed in the multiseed notebook.

---
## §2  Subset Scoring

We run `N_SUBSETS` random subsets of the candidate pool and apply each subset jointly.

For each subset, the notebook stores both:

- the signed accuracy drop \(A(G)-A(G\oplus S)\);
- the harmful-only drop \(\max(0, A(G)-A(G\oplus S))\).

For every edge, the harmful drops are divided by the number of subsets in which that edge appeared. The canonical raw score is therefore the **mean harmful subset drop when selected**, rather than the accumulated sum.

The run also stores a max-normalized score in \([0,1]\), a percentile rank, the inclusion count, and an observed mask. Unobserved candidates are marked as unobserved rather than being treated as genuine zero-score edges.


---

## Experiment 1 — Direct Mining of Harmful Candidates

### Research objective

This experiment constructs candidate edge perturbations and evaluates them against the fixed victim model.

For `subset_accuracy_drop`, random candidate subsets are flipped jointly. Every selected edge receives the subset's harmful accuracy drop, and its final raw score is the average of those assigned drops over all subsets in which it appeared.

The resulting value is an observational, subset-associated harmfulness score. It should not be interpreted as the isolated accuracy effect of the edge.

### Execution block

The following cell:

- loads or computes versioned mining results;
- rejects caches created with the previous accumulated-sum target;
- filters candidates that were never observed;
- exposes raw, normalized, and percentile score representations;
- preserves signed-drop, harmful-drop, and inclusion-count diagnostics;
- creates the standardized `mining_runs` objects used downstream.


In [ ]:
# PATCH: h-hop endpoint mining monkey patch
# Keeps endpoint_mining_hop/h available inside the notebook even if the helper
# module was loaded before the h-hop implementation was edited.
from typing import Any
import torch
import helpers.selector_pipeline_helpers as _selector_helpers


@torch.no_grad()
def _mine_endpoint_flips_with_h(
    *,
    model: torch.nn.Module,
    attr: torch.Tensor,
    labels: torch.Tensor,
    adj_orig: torch.Tensor,
    cand_src: torch.Tensor,
    cand_dst: torch.Tensor,
    clean_preds: torch.Tensor,
    clean_correct: torch.Tensor,
    clean_accuracy: float,
    h: int = 0,
    verbose: bool = True,
) -> dict[str, Any]:
    """
    Evaluate every original candidate edge exactly once.

    h = 0: query the full graph.
    h > 0: query only the h-hop neighborhood around the candidate endpoints.

    If the local subgraph is too small for a victim GCN with GDC/PPR top-k
    preprocessing, the query falls back to the full graph for that candidate.
    """
    if h < 0:
        raise ValueError("h must be non-negative. Use h=0 for the full graph.")

    device = adj_orig.device

    attr = attr.to(device)
    labels = labels.to(device=device, dtype=torch.long)
    cand_src = cand_src.to(device=device, dtype=torch.long).view(-1)
    cand_dst = cand_dst.to(device=device, dtype=torch.long).view(-1)
    clean_preds = clean_preds.to(device=device, dtype=torch.long)
    clean_correct = clean_correct.to(device=device, dtype=torch.bool)

    if cand_src.numel() != cand_dst.numel():
        raise ValueError("cand_src and cand_dst must have equal length.")

    n_samples = cand_src.numel()
    if n_samples == 0:
        raise ValueError("The candidate list is empty.")

    def _h_hop_nodes_from_dense_adj(adj: torch.Tensor, seeds: torch.Tensor, num_hops: int) -> torch.Tensor:
        seeds = seeds.to(device=adj.device, dtype=torch.long).view(-1)
        n_nodes = adj.size(0)

        visited = torch.zeros(n_nodes, device=adj.device, dtype=torch.bool)
        frontier = torch.zeros_like(visited)
        visited[seeds] = True
        frontier[seeds] = True

        adj_bool = adj > 0.5

        for _ in range(num_hops):
            neigh_out = adj_bool[frontier].any(dim=0)
            neigh_in = adj_bool[:, frontier].any(dim=1)
            neighbors = neigh_out | neigh_in
            new_frontier = neighbors & ~visited
            visited |= neighbors
            frontier = new_frontier

            if not bool(frontier.any().item()):
                break

        return torch.nonzero(visited, as_tuple=True)[0]

    def _clear_adj_cache_and_call(x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # Some rgnn_at_scale GCN variants cache preprocessed adjacency.
        # Candidate flips change adjacency every iteration, so clear it for the call.
        has_cache = hasattr(model, "adj_preped")
        old_cache = getattr(model, "adj_preped", None) if has_cache else None
        try:
            if has_cache:
                model.adj_preped = None
            return model(x, adj)
        finally:
            if has_cache:
                model.adj_preped = old_cache

    def _local_query_is_unsafe_for_gdc(sub_adj: torch.Tensor) -> bool:
        gdc_params = getattr(model, "gdc_params", None)
        if gdc_params is None:
            return False
        if "k" not in gdc_params:
            return False
        try:
            return int(sub_adj.size(0)) < int(gdc_params["k"])
        except Exception:
            return False

    model.eval()
    adj_work = adj_orig.clone()

    endpoint_labels = torch.zeros(n_samples, device=device, dtype=torch.float32)
    src_hit_labels = torch.zeros_like(endpoint_labels)
    dst_hit_labels = torch.zeros_like(endpoint_labels)
    both_hit_labels = torch.zeros_like(endpoint_labels)
    exists_clean = torch.zeros(n_samples, device=device, dtype=torch.bool)

    src_pert_predictions = torch.full((n_samples,), -1, device=device, dtype=torch.long)
    dst_pert_predictions = torch.full_like(src_pert_predictions, -1)
    neighborhood_sizes = torch.zeros(n_samples, device=device, dtype=torch.long)
    used_full_graph_fallback = torch.zeros(n_samples, device=device, dtype=torch.bool)

    rows: list[dict[str, Any]] = []

    for i in range(n_samples):
        u = int(cand_src[i].item())
        v = int(cand_dst[i].item())

        original_uv = float(adj_orig[u, v].item())
        original_vu = float(adj_orig[v, u].item())

        edge_exists = original_uv > 0.5 or original_vu > 0.5
        exists_clean[i] = edge_exists

        flipped_value = 0.0 if edge_exists else 1.0

        # Flip exactly this original candidate edge.
        adj_work[u, v] = flipped_value
        adj_work[v, u] = flipped_value

        neighborhood_size = int(adj_work.size(0))
        full_graph_fallback = False

        if h == 0:
            pert_preds = _clear_adj_cache_and_call(attr, adj_work).argmax(dim=-1)
            u_pert_pred = int(pert_preds[u].item())
            v_pert_pred = int(pert_preds[v].item())
        else:
            sub_nodes = _h_hop_nodes_from_dense_adj(
                adj=adj_work,
                seeds=torch.tensor([u, v], device=device, dtype=torch.long),
                num_hops=h,
            )
            sub_attr = attr[sub_nodes]
            sub_adj = adj_work[sub_nodes][:, sub_nodes]
            neighborhood_size = int(sub_nodes.numel())

            if _local_query_is_unsafe_for_gdc(sub_adj):
                full_graph_fallback = True
            else:
                try:
                    sub_preds = _clear_adj_cache_and_call(sub_attr, sub_adj).argmax(dim=-1)

                    global_to_local = torch.full(
                        (adj_work.size(0),),
                        -1,
                        device=device,
                        dtype=torch.long,
                    )
                    global_to_local[sub_nodes] = torch.arange(
                        sub_nodes.numel(),
                        device=device,
                        dtype=torch.long,
                    )

                    u_local = int(global_to_local[u].item())
                    v_local = int(global_to_local[v].item())
                    u_pert_pred = int(sub_preds[u_local].item())
                    v_pert_pred = int(sub_preds[v_local].item())
                except RuntimeError:
                    # Conservative fallback: keep the run alive and preserve the
                    # victim model's original preprocessing instead of changing it.
                    full_graph_fallback = True

            if full_graph_fallback:
                pert_preds = _clear_adj_cache_and_call(attr, adj_work).argmax(dim=-1)
                u_pert_pred = int(pert_preds[u].item())
                v_pert_pred = int(pert_preds[v].item())

        neighborhood_sizes[i] = neighborhood_size
        used_full_graph_fallback[i] = full_graph_fallback

        u_clean_pred = int(clean_preds[u].item())
        v_clean_pred = int(clean_preds[v].item())
        u_true = int(labels[u].item())
        v_true = int(labels[v].item())

        u_hit = bool(clean_correct[u].item()) and u_pert_pred != u_true
        v_hit = bool(clean_correct[v].item()) and v_pert_pred != v_true
        endpoint_hit = u_hit or v_hit
        both_hit = u_hit and v_hit

        endpoint_labels[i] = float(endpoint_hit)
        src_hit_labels[i] = float(u_hit)
        dst_hit_labels[i] = float(v_hit)
        both_hit_labels[i] = float(both_hit)
        src_pert_predictions[i] = u_pert_pred
        dst_pert_predictions[i] = v_pert_pred

        rows.append({
            "sample_index": i,
            "u": u,
            "v": v,
            "exists_clean": edge_exists,
            "action": "del" if edge_exists else "add",
            "h": int(h),
            "endpoint_mining_hop": int(h),
            "neighborhood_size": neighborhood_size,
            "used_full_graph_fallback": bool(full_graph_fallback),
            "u_label": u_true,
            "v_label": v_true,
            "u_clean_pred": u_clean_pred,
            "v_clean_pred": v_clean_pred,
            "u_pert_pred": u_pert_pred,
            "v_pert_pred": v_pert_pred,
            "u_was_correct": bool(clean_correct[u].item()),
            "v_was_correct": bool(clean_correct[v].item()),
            "u_hit": u_hit,
            "v_hit": v_hit,
            "both_hit": both_hit,
            "endpoint_hit": endpoint_hit,
        })

        # Restore the graph exactly.
        adj_work[u, v] = original_uv
        adj_work[v, u] = original_vu

    endpoint_hits = int(endpoint_labels.sum().item())

    if verbose:
        print(f"Clean evaluation accuracy: {clean_accuracy:.4f}")
        print(f"Original candidate edges evaluated: {n_samples}")
        if h == 0:
            print("Victim queries used the full graph.")
        else:
            print(f"Victim queries used h-hop neighborhoods with h={h}.")
            print(f"Average neighborhood size: {float(neighborhood_sizes.float().mean().item()):.2f}")
            print(f"Full-graph fallbacks: {int(used_full_graph_fallback.sum().item())}/{n_samples}")
        print(f"Endpoint hits: {endpoint_hits}/{n_samples} ({endpoint_hits / n_samples:.2%})")

    edge_index_lab = torch.stack([cand_src, cand_dst], dim=0)

    return {
        "mode": "endpoint",
        "h": int(h),
        "endpoint_mining_hop": int(h),
        "labels_raw": endpoint_labels.detach().cpu().numpy(),
        "labels_norm": endpoint_labels.detach().cpu().numpy(),
        "endpoint_labels": endpoint_labels.detach().cpu().numpy(),
        "sampled_edge_index": edge_index_lab.detach().cpu(),
        "sampled_u": cand_src.detach().cpu().numpy(),
        "sampled_v": cand_dst.detach().cpu().numpy(),
        "u_hit_labels": src_hit_labels.detach().cpu().numpy(),
        "v_hit_labels": dst_hit_labels.detach().cpu().numpy(),
        "both_hit_labels": both_hit_labels.detach().cpu().numpy(),
        "u_pert_predictions": src_pert_predictions.detach().cpu().numpy(),
        "v_pert_predictions": dst_pert_predictions.detach().cpu().numpy(),
        "exists": exists_clean.detach().cpu().numpy(),
        "neighborhood_sizes": neighborhood_sizes.detach().cpu().numpy(),
        "used_full_graph_fallback": used_full_graph_fallback.detach().cpu().numpy(),
        "clean_accuracy": clean_accuracy,
        "endpoint_hits": endpoint_hits,
        "n_samples": n_samples,
        "rows": rows,
    }


_selector_helpers._mine_endpoint_flips = _mine_endpoint_flips_with_h
_selector_helpers.mine_candidate_edge_scores.__globals__["_mine_endpoint_flips"] = _mine_endpoint_flips_with_h
print("Patched helpers.selector_pipeline_helpers._mine_endpoint_flips with h-hop support.")


In [ ]:
from pathlib import Path
from IPython.display import display
import hashlib
import numpy as np
import pandas as pd
import torch

from helpers.selector_pipeline_helpers import (
    MINING_SCHEMA_VERSION,
    SUBSET_SCORE_SCHEMA_VERSION,
    SUBSET_SCORE_AGGREGATION,
    SUBSET_SCORE_NORMALIZATION,
    SUBSET_SCORE_CLIPPING,
    mine_candidate_edge_scores,
    save_mining_output,
    load_mining_output,
    _dense_adj,
)


def _resolve_scoring_modes():
    configured = globals().get(
        "SCORING_MODES",
        [globals().get("SCORING_MODE", "subset_accuracy_drop")],
    )
    if isinstance(configured, str):
        configured = [configured]
    modes = [str(mode) for mode in configured]
    if not modes:
        raise ValueError("SCORING_MODES must not be empty.")
    return modes


def _resolve_endpoint_mining_hops():
    configured = globals().get("ENDPOINT_MINING_HOPS", [0])
    if isinstance(configured, int):
        configured = [configured]
    hops = [int(value) for value in configured]
    if not hops or any(h < 0 for h in hops):
        raise ValueError(
            "ENDPOINT_MINING_HOPS must contain non-negative integers."
        )
    return hops


def _score_source_pairs(mining_result, fallback_candidates):
    """Return the candidate order corresponding to the returned score arrays."""
    sampled_u = mining_result.get("sampled_u")
    sampled_v = mining_result.get("sampled_v")

    if sampled_u is None or sampled_v is None:
        return list(fallback_candidates)

    sampled_u = np.asarray(sampled_u, dtype=np.int64).reshape(-1)
    sampled_v = np.asarray(sampled_v, dtype=np.int64).reshape(-1)

    if sampled_u.size != sampled_v.size:
        raise ValueError("sampled_u and sampled_v have unequal length.")

    return [
        (min(int(u), int(v)), max(int(u), int(v)))
        for u, v in zip(sampled_u, sampled_v)
    ]


scoring_modes = _resolve_scoring_modes()
endpoint_mining_hops = _resolve_endpoint_mining_hops()

# Expand h only for endpoint scoring. subset_accuracy_drop creates exactly one
# mining run per candidate configuration and seed.
scoring_jobs = []
for scoring_mode in scoring_modes:
    if scoring_mode == "endpoint":
        scoring_jobs.extend(
            {
                "scoring_mode": scoring_mode,
                "endpoint_h": int(endpoint_h),
            }
            for endpoint_h in endpoint_mining_hops
        )
    else:
        scoring_jobs.append(
            {
                "scoring_mode": scoring_mode,
                "endpoint_h": 0,
            }
        )

mining_runs = []
mining_summary_rows = []

for candidate_run in candidate_runs:
    context = candidate_run["context"]
    activate_seed_context(context)

    seed = int(candidate_run["seed"])
    set_global_seed(seed)

    adj_orig_run = _dense_adj(
        edge_index,
        n_nodes,
        device=device,
    )

    candidates_for_config = list(candidate_run["candidates"])

    for scoring_job in scoring_jobs:
        scoring_mode = scoring_job["scoring_mode"]
        endpoint_h = int(scoring_job["endpoint_h"])

        candidate_hash = hashlib.sha1(
            repr(candidates_for_config).encode("utf-8")
        ).hexdigest()[:10]

        score_schema_version = (
            SUBSET_SCORE_SCHEMA_VERSION
            if scoring_mode == "subset_accuracy_drop"
            else f"{scoring_mode}_v1"
        )

        cache_path = Path("cache") / (
            f"candidate_mining_schema-{MINING_SCHEMA_VERSION}_"
            f"scoreSchema-{score_schema_version}_"
            f"dataset-{DATASET}_model-{MODEL_LABEL}_"
            f"cfg-{candidate_run['candidate_config_id']}_"
            f"scoringMode-{scoring_mode}_h-{endpoint_h}_seed-{seed}_"
            f"candidates-{len(candidates_for_config)}_"
            f"subsetFrac-{_safe_config_value(SUBSET_FRACTION)}_"
            f"nSubsets-{N_SUBSETS}_hash-{candidate_hash}.pt"
        )

        metadata = {
            "mining_schema_version": MINING_SCHEMA_VERSION,
            "score_schema_version": score_schema_version,
            "scoring_mode": scoring_mode,
            "target_kind": (
                "binary" if scoring_mode == "endpoint" else "continuous"
            ),
            "score_aggregation": (
                SUBSET_SCORE_AGGREGATION
                if scoring_mode == "subset_accuracy_drop"
                else "individual_evaluation"
            ),
            "score_normalization": (
                SUBSET_SCORE_NORMALIZATION
                if scoring_mode == "subset_accuracy_drop"
                else "mode_specific"
            ),
            "score_clipping": (
                SUBSET_SCORE_CLIPPING
                if scoring_mode == "subset_accuracy_drop"
                else "mode_specific"
            ),
            "endpoint_mining_hop": endpoint_h,
            "seed": seed,
            "n_nodes": int(n_nodes),
            "n_candidates": len(candidates_for_config),
            "candidate_config_id": candidate_run["candidate_config_id"],
            "candidate_set_size": candidate_run["candidate_set_size"],
            "prbcd_candidate_fraction": (
                candidate_run["prbcd_candidate_fraction"]
            ),
            "actual_prbcd_fraction": (
                candidate_run["actual_prbcd_fraction"]
            ),
            "subset_fraction": float(SUBSET_FRACTION),
            "n_subsets": int(N_SUBSETS),
            "endpoint_require_correct_to_incorrect": True,
        }

        print(
            f"Scoring seed={seed} | "
            f"{candidate_run['candidate_config_id']} | "
            f"mode={scoring_mode} | h={endpoint_h} | "
            f"schema={score_schema_version}"
        )

        if cache_path.exists():
            mining_result, cached_candidates, _ = load_mining_output(
                cache_path,
                device=device,
                expected_metadata=metadata,
                expected_schema_version=MINING_SCHEMA_VERSION,
            )

            if cached_candidates != candidates_for_config:
                raise ValueError(
                    "Cached candidates differ from current candidates."
                )

            cache_status = "loaded"
        else:
            mining_result = mine_candidate_edge_scores(
                model=model,
                attr=attr,
                edge_index=edge_index,
                labels=labels,
                eval_idx=idx_test,
                candidates=candidates_for_config,
                n_nodes=n_nodes,
                mode=scoring_mode,
                subset_fraction=SUBSET_FRACTION,
                n_subsets=N_SUBSETS,
                endpoint_k_samples=None,
                endpoint_require_correct_to_incorrect=True,
                two_hop_k_samples=None,
                h=endpoint_h,
                seed=seed,
                device=device,
                verbose=True,
            )

            save_mining_output(
                cache_path,
                mining_result,
                candidates=candidates_for_config,
                metadata=metadata,
            )
            cache_status = "computed"

        if mining_result.get("schema_version") != MINING_SCHEMA_VERSION:
            raise RuntimeError(
                "Mining result has an incompatible schema_version."
            )

        source_pairs_all = _score_source_pairs(
            mining_result,
            candidates_for_config,
        )

        score_raw_all = np.asarray(
            mining_result["score_raw"],
            dtype=np.float64,
        ).reshape(-1)

        score_norm_all = np.asarray(
            mining_result["score_norm"],
            dtype=np.float32,
        ).reshape(-1)

        score_percentile_all = np.asarray(
            mining_result["score_percentile"],
            dtype=np.float32,
        ).reshape(-1)

        observed_mask_all = np.asarray(
            mining_result["observed_mask"],
            dtype=bool,
        ).reshape(-1)

        n_score_rows = len(source_pairs_all)

        for name, values in {
            "score_raw": score_raw_all,
            "score_norm": score_norm_all,
            "score_percentile": score_percentile_all,
            "observed_mask": observed_mask_all,
        }.items():
            if len(values) != n_score_rows:
                raise ValueError(
                    f"{name} has length {len(values)}, but the score source "
                    f"contains {n_score_rows} candidate pairs."
                )

        if not observed_mask_all.any():
            raise RuntimeError(
                "No candidate was observed during mining. Increase N_SUBSETS "
                "or SUBSET_FRACTION."
            )

        observed_indices = np.flatnonzero(observed_mask_all)

        scored_candidates = [
            source_pairs_all[index]
            for index in observed_indices
        ]

        score_raw_run = score_raw_all[observed_indices]
        score_norm_run = score_norm_all[observed_indices]
        score_percentile_run = score_percentile_all[observed_indices]

        if not (
            np.isfinite(score_raw_run).all()
            and np.isfinite(score_norm_run).all()
            and np.isfinite(score_percentile_run).all()
        ):
            raise ValueError(
                "Observed score arrays must contain only finite values."
            )

        src_run = np.asarray(
            [u for u, _ in scored_candidates],
            dtype=np.int64,
        )
        dst_run = np.asarray(
            [v for _, v in scored_candidates],
            dtype=np.int64,
        )

        exists_all = np.asarray(
            mining_result.get(
                "exists",
                [
                    float(adj_orig_run[u, v].item())
                    for u, v in source_pairs_all
                ],
            ),
            dtype=np.float32,
        ).reshape(-1)

        if exists_all.size != n_score_rows:
            raise ValueError(
                "exists must align with the score source pairs."
            )

        exists_run = exists_all[observed_indices]
        clean_accuracy = float(mining_result["clean_accuracy"])

        inclusion_count_all = np.asarray(
            mining_result.get(
                "inclusion_count",
                np.ones(n_score_rows, dtype=np.int64),
            ),
            dtype=np.int64,
        ).reshape(-1)

        if inclusion_count_all.size != n_score_rows:
            raise ValueError(
                "inclusion_count must align with the score source pairs."
            )

        inclusion_count_run = inclusion_count_all[observed_indices]

        n_positive = int(np.sum(score_norm_run > 0))
        n_observed = int(observed_indices.size)
        n_unobserved = int(n_score_rows - n_observed)

        if scoring_mode == "endpoint":
            diagnostic_name = "endpoint_hit_rate"
            diagnostic_value = float(
                np.asarray(
                    mining_result["endpoint_labels"],
                    dtype=float,
                ).mean()
            )
            mean_subset_signed_drop = np.nan
            mean_subset_harmful_drop = np.nan
        else:
            signed_subset_drops = np.asarray(
                mining_result.get(
                    "signed_drop_per_subset",
                    mining_result.get("drop_per_subset", []),
                ),
                dtype=float,
            )
            harmful_subset_drops = np.asarray(
                mining_result.get(
                    "harmful_drop_per_subset",
                    mining_result.get("drop_per_subset", []),
                ),
                dtype=float,
            )
            mean_subset_signed_drop = (
                float(signed_subset_drops.mean())
                if signed_subset_drops.size
                else np.nan
            )
            mean_subset_harmful_drop = (
                float(harmful_subset_drops.mean())
                if harmful_subset_drops.size
                else np.nan
            )
            diagnostic_name = "mean_subset_drop"
            diagnostic_value = mean_subset_harmful_drop

        run = {
            **{
                key: value
                for key, value in candidate_run.items()
                if key not in {
                    "selected_prbcd",
                    "random_candidates",
                    "candidates",
                }
            },

            "seed": seed,
            "context": context,
            "adj_orig": adj_orig_run,

            "schema_version": mining_result["schema_version"],
            "score_schema_version": mining_result[
                "score_schema_version"
            ],
            "target_kind": mining_result["target_kind"],
            "score_definition": mining_result["score_definition"],
            "score_aggregation": mining_result["score_aggregation"],
            "score_normalization": mining_result[
                "score_normalization"
            ],
            "score_clipping": mining_result.get(
                "score_clipping"
            ),

            "scoring_mode": scoring_mode,
            "endpoint_mining_hop": endpoint_h,
            "candidate_hash": candidate_hash,
            "cache_path": str(cache_path),
            "cache_status": cache_status,

            # The complete candidate pool remains available for diagnostics.
            "candidate_pool": candidates_for_config,
            "n_candidates_total": n_score_rows,
            "n_observed_candidates": n_observed,
            "n_unobserved_candidates": n_unobserved,
            "observed_fraction": n_observed / max(1, n_score_rows),

            # Downstream edge arrays contain observed candidates only.
            "candidates": scored_candidates,
            "cand_src": src_run.tolist(),
            "cand_dst": dst_run.tolist(),
            "exists_list": exists_run.tolist(),
            "n_cands": n_observed,

            "mining_result": mining_result,
            "observed_mask_all": observed_mask_all,
            "observed_indices": observed_indices,

            "score_raw": score_raw_run,
            "score_norm": score_norm_run,
            "score_percentile": score_percentile_run,
            "inclusion_count": inclusion_count_run,

            # Compatibility aliases for the existing downstream notebook.
            "labels_raw": score_raw_run,
            "labels_norm": score_norm_run,
            "labels_changed": torch.tensor(
                score_norm_run,
                dtype=torch.float32,
                device=device,
            ),

            "src": torch.tensor(
                src_run,
                dtype=torch.long,
                device=device,
            ),
            "dst": torch.tensor(
                dst_run,
                dtype=torch.long,
                device=device,
            ),
            "exists": torch.tensor(
                exists_run,
                dtype=torch.float32,
                device=device,
            ),

            "clean_accuracy": clean_accuracy,
            "n_positive_labels": n_positive,
        }

        mining_runs.append(run)

        mining_summary_rows.append({
            "seed": seed,
            "candidate_config_id": candidate_run[
                "candidate_config_id"
            ],
            "candidate_set_size": candidate_run[
                "candidate_set_size"
            ],
            "prbcd_candidate_fraction": candidate_run[
                "prbcd_candidate_fraction"
            ],
            "actual_prbcd_fraction": candidate_run[
                "actual_prbcd_fraction"
            ],
            "schema_version": mining_result["schema_version"],
            "score_schema_version": mining_result[
                "score_schema_version"
            ],
            "target_kind": mining_result["target_kind"],
            "score_definition": mining_result["score_definition"],
            "score_aggregation": mining_result[
                "score_aggregation"
            ],
            "score_normalization": mining_result[
                "score_normalization"
            ],
            "scoring_mode": scoring_mode,
            "endpoint_mining_hop": endpoint_h,

            "n_candidates": n_score_rows,
            "n_observed_candidates": n_observed,
            "n_unobserved_candidates": n_unobserved,
            "observed_fraction": n_observed / max(1, n_score_rows),

            # Retained as a compatibility diagnostic only.
            "n_positive_labels": n_positive,
            "positive_fraction": n_positive / max(1, n_observed),

            "score_raw_min": float(np.min(score_raw_run)),
            "score_raw_mean": float(np.mean(score_raw_run)),
            "score_raw_max": float(np.max(score_raw_run)),
            "score_norm_mean": float(np.mean(score_norm_run)),

            "inclusion_count_min": int(inclusion_count_run.min()),
            "inclusion_count_mean": float(
                inclusion_count_run.mean()
            ),
            "inclusion_count_max": int(inclusion_count_run.max()),

            "clean_accuracy": clean_accuracy,
            "mean_subset_signed_drop": mean_subset_signed_drop,
            "mean_subset_harmful_drop": mean_subset_harmful_drop,
            diagnostic_name: diagnostic_value,

            "cache_status": cache_status,
            "cache_path": str(cache_path),
        })


# Defensive check: subset scoring must not be multiplied by endpoint-hop settings.
_subset_counts = {}
for run in mining_runs:
    if run["scoring_mode"] == "subset_accuracy_drop":
        key = (
            run["seed"],
            run["candidate_config_id"],
        )
        _subset_counts[key] = _subset_counts.get(key, 0) + 1

_duplicate_subset_runs = {
    key: count
    for key, count in _subset_counts.items()
    if count != 1
}

if _duplicate_subset_runs:
    raise RuntimeError(
        "subset_accuracy_drop must create exactly one run per "
        "seed/configuration; found: "
        f"{_duplicate_subset_runs}"
    )

if not mining_runs:
    raise RuntimeError("No mining runs were created.")

mining_summary_raw_df = pd.DataFrame(mining_summary_rows)

mining_group_cols = [
    "candidate_config_id",
    "candidate_set_size",
    "prbcd_candidate_fraction",
    "schema_version",
    "score_schema_version",
    "target_kind",
    "score_definition",
    "score_aggregation",
    "score_normalization",
    "scoring_mode",
    "endpoint_mining_hop",
]

mining_summary_df = aggregate_over_seeds(
    mining_summary_raw_df,
    mining_group_cols,
    [
        "actual_prbcd_fraction",
        "n_candidates",
        "n_observed_candidates",
        "n_unobserved_candidates",
        "observed_fraction",
        "n_positive_labels",
        "positive_fraction",
        "score_raw_min",
        "score_raw_mean",
        "score_raw_max",
        "score_norm_mean",
        "inclusion_count_min",
        "inclusion_count_mean",
        "inclusion_count_max",
        "clean_accuracy",
        "endpoint_hit_rate",
        "mean_subset_drop",
        "mean_subset_signed_drop",
        "mean_subset_harmful_drop",
    ],
)

Path("extendedPlotting").mkdir(parents=True, exist_ok=True)

mining_summary_raw_df.to_csv(
    "extendedPlotting/mining_summary_raw_multiseed.csv",
    index=False,
)

mining_summary_df.to_csv(
    "extendedPlotting/mining_summary_averaged_multiseed.csv",
    index=False,
)

display(mining_summary_df)

mining_runs_by_key = {
    (
        run["seed"],
        run["candidate_config_id"],
        run["scoring_mode"],
        run["endpoint_mining_hop"],
    ): run
    for run in mining_runs
}


In [ ]:
subset_runs = [
    run
    for run in mining_runs
    if run["scoring_mode"] == "subset_accuracy_drop"
]

if subset_runs:
    from pathlib import Path

    out_dir = RUN_PLOTS_DIR
    out_dir.mkdir(parents=True, exist_ok=True)

    group_keys = [
        "candidate_config_id",
        "candidate_set_size",
        "prbcd_candidate_fraction",
        "score_schema_version",
    ]

    grouped_input = pd.DataFrame([
        {
            **{key: run[key] for key in group_keys},
            "run": run,
        }
        for run in subset_runs
    ])

    for key, runs in grouped_input.groupby(
        group_keys,
        dropna=False,
    ):
        run_list = runs["run"].tolist()

        subset_rows = []
        raw_score_arrays = []
        inclusion_arrays = []

        for run in run_list:
            result = run["mining_result"]

            signed_drops = np.asarray(
                result["signed_drop_per_subset"],
                dtype=float,
            )
            harmful_drops = np.asarray(
                result["harmful_drop_per_subset"],
                dtype=float,
            )

            subset_rows.extend(
                {
                    "seed": run["seed"],
                    "subset_index": subset_index + 1,
                    "signed_drop": signed_drop,
                    "harmful_drop": harmful_drop,
                }
                for subset_index, (
                    signed_drop,
                    harmful_drop,
                ) in enumerate(
                    zip(signed_drops, harmful_drops)
                )
            )

            raw_score_arrays.append(
                np.asarray(
                    run["score_raw"],
                    dtype=float,
                )
            )

            inclusion_arrays.append(
                np.asarray(
                    run["inclusion_count"],
                    dtype=float,
                )
            )

        subset_df = pd.DataFrame(subset_rows)

        curve = (
            subset_df
            .groupby("subset_index", as_index=False)
            .agg(
                harmful_drop_mean=(
                    "harmful_drop",
                    "mean",
                ),
                harmful_drop_std=(
                    "harmful_drop",
                    "std",
                ),
            )
        )

        curve["harmful_drop_std"] = (
            curve["harmful_drop_std"].fillna(0.0)
        )

        all_harmful_drops = subset_df[
            "harmful_drop"
        ].to_numpy()

        harmful_bins = np.linspace(
            all_harmful_drops.min(),
            all_harmful_drops.max() + 1e-12,
            21,
        )

        hist_harmful = np.vstack([
            np.histogram(
                subset_df.loc[
                    subset_df.seed == seed,
                    "harmful_drop",
                ],
                bins=harmful_bins,
            )[0]
            for seed in sorted(subset_df.seed.unique())
        ])

        pooled_scores = np.concatenate(raw_score_arrays)
        score_bins = np.linspace(
            pooled_scores.min(),
            pooled_scores.max() + 1e-12,
            31,
        )

        hist_scores = np.vstack([
            np.histogram(
                scores,
                bins=score_bins,
            )[0]
            for scores in raw_score_arrays
        ])

        pooled_inclusions = np.concatenate(
            inclusion_arrays
        )
        inclusion_bins = np.arange(
            pooled_inclusions.min(),
            pooled_inclusions.max() + 2,
        ) - 0.5

        hist_inclusions = np.vstack([
            np.histogram(
                counts,
                bins=inclusion_bins,
            )[0]
            for counts in inclusion_arrays
        ])

        fig, axes = plt.subplots(
            1,
            4,
            figsize=(19, 4),
        )

        add_mean_std_band(
            axes[0],
            curve["subset_index"],
            curve["harmful_drop_mean"],
            curve["harmful_drop_std"],
            label="mean ± SD",
            marker="o",
            markersize=3,
        )
        axes[0].set_xlabel("subset index")
        axes[0].set_ylabel("harmful accuracy drop")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        harmful_centers = (
            harmful_bins[:-1] + harmful_bins[1:]
        ) / 2

        axes[1].step(
            harmful_centers,
            hist_harmful.mean(0),
            where="mid",
            label="mean count",
        )
        axes[1].fill_between(
            harmful_centers,
            hist_harmful.mean(0)
            - hist_harmful.std(0),
            hist_harmful.mean(0)
            + hist_harmful.std(0),
            alpha=0.18,
            step="mid",
        )
        axes[1].set_xlabel(
            "harmful accuracy drop per subset"
        )
        axes[1].set_ylabel("mean bin count ± SD")

        score_centers = (
            score_bins[:-1] + score_bins[1:]
        ) / 2

        axes[2].step(
            score_centers,
            hist_scores.mean(0),
            where="mid",
            label="mean count",
        )
        axes[2].fill_between(
            score_centers,
            hist_scores.mean(0)
            - hist_scores.std(0),
            hist_scores.mean(0)
            + hist_scores.std(0),
            alpha=0.18,
            step="mid",
        )
        axes[2].set_xlabel(
            "mean harmful subset drop when selected"
        )
        axes[2].set_ylabel("mean bin count ± SD")

        inclusion_centers = (
            inclusion_bins[:-1]
            + inclusion_bins[1:]
        ) / 2

        axes[3].step(
            inclusion_centers,
            hist_inclusions.mean(0),
            where="mid",
            label="mean count",
        )
        axes[3].fill_between(
            inclusion_centers,
            hist_inclusions.mean(0)
            - hist_inclusions.std(0),
            hist_inclusions.mean(0)
            + hist_inclusions.std(0),
            alpha=0.18,
            step="mid",
        )
        axes[3].set_xlabel("candidate inclusion count")
        axes[3].set_ylabel("mean bin count ± SD")

        title = " | ".join(map(str, key))
        fig.suptitle(
            "Exposure-corrected subset scoring"
        )
        fig.tight_layout()

        fig.savefig(
            out_dir
            / (
                "subset_scoring__"
                f"{_safe_config_value(title)}.png"
            ),
            dpi=200,
            bbox_inches="tight",
        )

        plt.show()
else:
    print(
        "Skipping subset-drop plots because no "
        "subset_accuracy_drop mining run exists."
    )


---
## §3  Continuous Score Distribution

The canonical raw label is the mean harmful subset accuracy drop observed when an edge was selected. It remains in the original accuracy-drop scale.

For selector training, the raw score is divided by the largest observed raw score within the mining run, producing `score_norm` in \([0,1]\). This is a relative harmfulness score, not a probability.

`score_percentile` stores the tie-aware within-run rank. `observed_mask` distinguishes evaluated zero-score candidates from candidates that were never included in any sampled subset.


### Analysis and plotting block

The following cells characterize the candidates obtained through direct mining.

The analysis examines:

- the number and proportion of positive labels;
- the balance between label-1 and label-0 candidates;
- edge additions versus edge deletions;
- the number of distinct affected endpoints;
- structural differences between label groups;
- the distribution of accuracy-drop scores;
- the overlap between local endpoint harmfulness and global accuracy effects.

This analysis addresses the first step of the RQ2 argument:

> Are harmful candidates observable through direct local or global evaluations, and how are they distributed within the candidate space?

These descriptive results do not yet establish that the candidates remain harmful when several perturbations are applied jointly. That question is addressed by the oracle and injection experiments.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

LABEL_DIAGNOSTIC_OUT_DIR = Path("extendedPlotting") / "label_diagnostics_multiseed"
LABEL_DIAGNOSTIC_OUT_DIR.mkdir(parents=True, exist_ok=True)
label_summary_rows = []

for run in mining_runs:
    values = run["labels_changed"].detach().cpu().numpy().astype(float)
    label_summary_rows.append({
        "seed": run["seed"], "candidate_config_id": run["candidate_config_id"],
        "candidate_set_size": run["candidate_set_size"],
        "prbcd_candidate_fraction": run["prbcd_candidate_fraction"],
        "actual_prbcd_fraction": run["actual_prbcd_fraction"],
        "scoring_mode": run["scoring_mode"], "endpoint_mining_hop": run["endpoint_mining_hop"],
        "n_candidates": len(values), "n_positive_labels": int((values > 0).sum()),
        "label_min": values.min(), "label_mean": values.mean(), "label_median": np.median(values),
        "label_max": values.max(), "label_std": values.std(),
        "fraction_label_gt_0": (values > 0).mean(), "fraction_label_gt_0_5": (values > .5).mean(),
        "n_label_lt_0_2": int((values < 0.2).sum()),
        "n_label_gt_0_8": int((values > 0.8).sum()),
        "n_extreme_labels": int(((values < 0.2) | (values > 0.8)).sum()),
    })

label_summary_raw_df = pd.DataFrame(label_summary_rows)
label_group_cols = ["candidate_config_id", "candidate_set_size", "prbcd_candidate_fraction", "scoring_mode", "endpoint_mining_hop"]
label_summary_df = aggregate_over_seeds(label_summary_raw_df, label_group_cols, [
    "actual_prbcd_fraction", "n_candidates", "n_positive_labels", "label_min", "label_mean",
    "label_median", "label_max", "label_std", "fraction_label_gt_0", "fraction_label_gt_0_5",
])
label_summary_raw_df.to_csv(LABEL_DIAGNOSTIC_OUT_DIR / "label_summary_raw.csv", index=False)
label_summary_df.to_csv(LABEL_DIAGNOSTIC_OUT_DIR / "label_summary_averaged.csv", index=False)
display(label_summary_df)

extreme_label_rows = []

for run in mining_runs:
    if run["scoring_mode"] != "subset_accuracy_drop":
        continue

    values = run["labels_changed"].detach().cpu().numpy().astype(float)

    n_below_0_2 = int((values < 0.2).sum())
    n_above_0_8 = int((values > 0.8).sum())

    extreme_label_rows.append({
        "seed": run["seed"],
        "candidate_config_id": run["candidate_config_id"],
        "candidate_set_size": run["candidate_set_size"],
        "prbcd_candidate_fraction": run["prbcd_candidate_fraction"],
        "endpoint_mining_hop": run["endpoint_mining_hop"],
        "n_labels_below_0_2": n_below_0_2,
        "n_labels_above_0_8": n_above_0_8,
        "n_extreme_labels": n_below_0_2 + n_above_0_8,
    })

extreme_label_df = pd.DataFrame(extreme_label_rows)

print("Extreme normalized labels for subset_accuracy_drop:")
display(extreme_label_df)

print(
    f"Total below 0.2: {extreme_label_df['n_labels_below_0_2'].sum()}\n"
    f"Total above 0.8: {extreme_label_df['n_labels_above_0_8'].sum()}\n"
    f"Total extreme labels: {extreme_label_df['n_extreme_labels'].sum()}"
)

for key, run_group in pd.DataFrame([{**{k: r[k] for k in label_group_cols}, "run": r} for r in mining_runs]).groupby(label_group_cols, dropna=False):
    arrays = [row["labels_changed"].detach().cpu().numpy().astype(float) for row in run_group["run"]]
    pooled = np.concatenate(arrays)
    bins = np.linspace(pooled.min(), pooled.max() + 1e-12, 41)
    counts = np.vstack([np.histogram(values, bins=bins)[0] for values in arrays])
    positive_arrays = [values[values > 0] for values in arrays]
    pooled_pos = np.concatenate([v for v in positive_arrays if v.size]) if any(v.size for v in positive_arrays) else np.array([0.0])
    bins_pos = np.linspace(pooled_pos.min(), pooled_pos.max() + 1e-12, 31)
    counts_pos = np.vstack([np.histogram(values, bins=bins_pos)[0] for values in positive_arrays])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, bins_i, counts_i, title in [(axes[0], bins, counts, "All labels"), (axes[1], bins_pos, counts_pos, "Labels > 0")]:
        centers = (bins_i[:-1] + bins_i[1:]) / 2
        mean, std = counts_i.mean(0), counts_i.std(0)
        ax.step(centers, mean, where="mid", label="mean bin count")
        ax.fill_between(centers, mean-std, mean+std, step="mid", alpha=.18)
        ax.set_xlabel("normalised label"); ax.set_ylabel("mean count ± SD"); ax.set_title(title); ax.grid(True, alpha=.3)
    title_key = " | ".join(map(str, key))
    fig.suptitle("Mined-label distribution")
    fig.tight_layout(); fig.savefig(RUN_PLOTS_DIR / f"label_distribution__{_safe_config_value(title_key)}.png", dpi=200, bbox_inches="tight"); plt.show()


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


# ============================================================
# Output configuration
# ============================================================

MINING_MODE_OUT_DIR = (
    Path("extendedPlotting")
    / "mining_mode_rates_multiseed"
)

MINING_MODE_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Collect one row per mining run
# ============================================================

endpoint_rows = []
subset_rows = []


for run in mining_runs:
    scoring_mode = run.get("scoring_mode")
    result = run["mining_result"]

    common = {
        "seed": run["seed"],
        "candidate_config_id": run["candidate_config_id"],
        "candidate_set_size": run["candidate_set_size"],
        "prbcd_candidate_fraction": (
            run["prbcd_candidate_fraction"]
        ),
        "actual_prbcd_fraction": (
            run["actual_prbcd_fraction"]
        ),
    }

    # ========================================================
    # Binary endpoint mining
    # ========================================================

    if scoring_mode == "endpoint":
        endpoint_labels = np.asarray(
            result["endpoint_labels"],
            dtype=np.float64,
        ).reshape(-1)

        n_samples = int(
            result.get(
                "n_samples",
                endpoint_labels.size,
            )
        )

        n_positive = int(
            endpoint_labels.sum()
        )

        endpoint_rows.append({
            **common,
            "scoring_mode": "endpoint",
            "endpoint_mining_hop": run.get(
                "endpoint_mining_hop",
                0,
            ),
            "n_candidates": run.get(
                "n_cands",
                endpoint_labels.size,
            ),
            "n_positive": n_positive,
            "n_samples": n_samples,
            "positive_per_sample": (
                n_positive / n_samples
                if n_samples > 0
                else np.nan
            ),
        })

    # ========================================================
    # Continuous subset-accuracy-drop mining
    # ========================================================

    elif scoring_mode == "subset_accuracy_drop":
        if "score_raw" not in result:
            raise KeyError(
                "A subset_accuracy_drop result does not contain "
                "'score_raw'. Recompute this run using the new "
                "mean-drop schema."
            )

        score_raw = np.asarray(
            result["score_raw"],
            dtype=np.float64,
        ).reshape(-1)

        observed_mask = np.asarray(
            result.get(
                "observed_mask",
                np.isfinite(score_raw),
            ),
            dtype=bool,
        ).reshape(-1)

        if observed_mask.shape != score_raw.shape:
            raise ValueError(
                "observed_mask and score_raw have incompatible "
                f"shapes: {observed_mask.shape} and "
                f"{score_raw.shape}."
            )

        valid_mask = (
            observed_mask
            & np.isfinite(score_raw)
        )

        observed_scores = score_raw[valid_mask]

        n_candidates = int(score_raw.size)
        n_observed = int(valid_mask.sum())
        n_unobserved = n_candidates - n_observed

        if n_observed > 0:
            mean_score_raw = float(
                observed_scores.mean()
            )

            median_score_raw = float(
                np.median(observed_scores)
            )

            max_score_raw = float(
                observed_scores.max()
            )

            n_positive = int(
                (observed_scores > 0).sum()
            )

            positive_score_fraction = (
                n_positive / n_observed
            )

        else:
            mean_score_raw = np.nan
            median_score_raw = np.nan
            max_score_raw = np.nan
            n_positive = 0
            positive_score_fraction = np.nan

        inclusion_count = np.asarray(
            result.get(
                "inclusion_count",
                np.full(n_candidates, np.nan),
            ),
            dtype=np.float64,
        ).reshape(-1)

        if (
            inclusion_count.size == n_candidates
            and n_observed > 0
        ):
            mean_inclusion_count = float(
                inclusion_count[valid_mask].mean()
            )
        else:
            mean_inclusion_count = np.nan

        subset_rows.append({
            **common,
            "scoring_mode": "subset_accuracy_drop",
            "n_candidates": n_candidates,
            "n_observed": n_observed,
            "n_unobserved": n_unobserved,
            "observed_fraction": (
                n_observed / n_candidates
                if n_candidates > 0
                else np.nan
            ),
            "n_positive": n_positive,
            "positive_score_fraction": (
                positive_score_fraction
            ),
            "mean_score_raw": mean_score_raw,
            "median_score_raw": median_score_raw,
            "max_score_raw": max_score_raw,
            "mean_inclusion_count": (
                mean_inclusion_count
            ),
            "n_subsets": result.get(
                "n_subsets",
                np.nan,
            ),
            "subset_size": result.get(
                "subset_size",
                np.nan,
            ),
            "subset_fraction": result.get(
                "subset_fraction",
                np.nan,
            ),
        })


# ============================================================
# Validate available modes
# ============================================================

if not endpoint_rows and not subset_rows:
    raise RuntimeError(
        "No endpoint or subset_accuracy_drop mining runs found."
    )


# ============================================================
# Build and aggregate endpoint results
# ============================================================

if endpoint_rows:
    endpoint_rate_df = pd.DataFrame(
        endpoint_rows
    )

    endpoint_plot_df = aggregate_over_seeds(
        endpoint_rate_df,
        [
            "candidate_set_size",
            "endpoint_mining_hop",
            "prbcd_candidate_fraction",
        ],
        [
            "actual_prbcd_fraction",
            "n_positive",
            "n_samples",
            "positive_per_sample",
        ],
    )

    endpoint_rate_df.to_csv(
        MINING_MODE_OUT_DIR
        / "endpoint_hit_rate_raw.csv",
        index=False,
    )

    endpoint_plot_df.to_csv(
        MINING_MODE_OUT_DIR
        / "endpoint_hit_rate_averaged.csv",
        index=False,
    )

else:
    endpoint_rate_df = pd.DataFrame()
    endpoint_plot_df = pd.DataFrame()


# ============================================================
# Build and aggregate subset-drop results
# ============================================================

if subset_rows:
    subset_score_df = pd.DataFrame(
        subset_rows
    )

    subset_plot_df = aggregate_over_seeds(
        subset_score_df,
        [
            "candidate_set_size",
            "prbcd_candidate_fraction",
        ],
        [
            "actual_prbcd_fraction",
            "mean_score_raw",
            "median_score_raw",
            "max_score_raw",
            "positive_score_fraction",
            "observed_fraction",
            "n_positive",
            "n_observed",
            "mean_inclusion_count",
        ],
    )

    subset_score_df.to_csv(
        MINING_MODE_OUT_DIR
        / "subset_accuracy_drop_raw.csv",
        index=False,
    )

    subset_plot_df.to_csv(
        MINING_MODE_OUT_DIR
        / "subset_accuracy_drop_averaged.csv",
        index=False,
    )

else:
    subset_score_df = pd.DataFrame()
    subset_plot_df = pd.DataFrame()


# ============================================================
# Display result tables
# ============================================================

if not endpoint_plot_df.empty:
    print("Endpoint mining:")
    display(endpoint_plot_df)

if not subset_plot_df.empty:
    print("Subset-accuracy-drop mining:")
    display(subset_plot_df)


# ============================================================
# Plot both mining modes in one figure
#
# Separate axes are essential because:
# - endpoint uses a binary positive rate;
# - subset mining uses a continuous accuracy-drop score.
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 5),
)

endpoint_ax, subset_ax = axes


# ------------------------------------------------------------
# Panel A: endpoint hit rate
# ------------------------------------------------------------

if not endpoint_plot_df.empty:
    for (
        candidate_set_size,
        endpoint_hop,
    ), group in endpoint_plot_df.groupby(
        [
            "candidate_set_size",
            "endpoint_mining_hop",
        ]
    ):
        group = group.sort_values(
            "actual_prbcd_fraction_mean"
        )

        add_mean_std_band(
            endpoint_ax,
            group["actual_prbcd_fraction_mean"],
            group["positive_per_sample_mean"],
            group["positive_per_sample_std"],
            marker="o",
            label=(
                f"{candidate_set_size} candidates, "
                f"h={endpoint_hop}"
            ),
        )

    endpoint_ax.set_xlabel(
        "actual PRBCD candidate fraction "
        "(seed mean)"
    )

    endpoint_ax.set_ylabel(
        "endpoint positive hit rate"
    )

    endpoint_ax.set_title(
        "Endpoint mining"
    )

    endpoint_ax.grid(
        True,
        alpha=0.3,
    )

    endpoint_ax.legend(
        fontsize=8,
    )

else:
    endpoint_ax.text(
        0.5,
        0.5,
        "No endpoint runs",
        ha="center",
        va="center",
        transform=endpoint_ax.transAxes,
    )

    endpoint_ax.set_axis_off()


# ------------------------------------------------------------
# Panel B: subset mean raw score
# ------------------------------------------------------------

if not subset_plot_df.empty:
    for candidate_set_size, group in (
        subset_plot_df.groupby(
            "candidate_set_size"
        )
    ):
        group = group.sort_values(
            "actual_prbcd_fraction_mean"
        )

        add_mean_std_band(
            subset_ax,
            group["actual_prbcd_fraction_mean"],
            group["mean_score_raw_mean"],
            group["mean_score_raw_std"],
            marker="o",
            label=(
                f"{candidate_set_size} candidates"
            ),
        )

    subset_ax.set_xlabel(
        "actual PRBCD candidate fraction "
        "(seed mean)"
    )

    subset_ax.set_ylabel(
        "mean harmful subset accuracy drop"
    )

    subset_ax.set_title(
        "Subset-accuracy-drop mining"
    )

    subset_ax.grid(
        True,
        alpha=0.3,
    )

    subset_ax.legend(
        fontsize=8,
    )

else:
    subset_ax.text(
        0.5,
        0.5,
        "No subset-accuracy-drop runs",
        ha="center",
        va="center",
        transform=subset_ax.transAxes,
    )

    subset_ax.set_axis_off()


fig.suptitle(
    "Mining-label behaviour"
)

fig.tight_layout()

fig.savefig(
    RUN_PLOTS_DIR
    / "mining_mode_behaviour_by_prbcd_fraction.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ============================================================
# RQ2: Mode-aware comprehensive graph-statistics comparison
# endpoint: label 1 versus label 0 (unchanged)
# subset_accuracy_drop: top versus bottom score decile
# Run directly after candidate labeling / creation of `mining_runs`.
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import math
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import networkx as nx

from IPython.display import display
from scipy import sparse
from scipy.sparse.csgraph import shortest_path
from scipy.stats import (
    chi2_contingency,
    fisher_exact,
    ks_2samp,
    mannwhitneyu,
    pointbiserialr,
    ttest_rel,
    wilcoxon,
)

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

RQ2_GRAPH_STATS_SCORING_MODES = {"endpoint", "subset_accuracy_drop"}
RQ2_GRAPH_STATS_CANDIDATE_CONFIG_ID = None
# None -> use every endpoint hop available for the selected configuration.
# This filter applies only to endpoint runs; subset runs are never duplicated by hop.
RQ2_GRAPH_STATS_ENDPOINT_HOPS = None

# Endpoint mode keeps the old binary split exactly.
RQ2_GRAPH_STATS_LABEL_THRESHOLD = 0.5

# Subset mode compares equal-sized score extremes within every seed/run.
RQ2_GRAPH_STATS_SUBSET_EXTREME_FRACTION = 0.10

RQ2_GRAPH_STATS_TOP_PLOT_FEATURES = 30
RQ2_GRAPH_STATS_MAX_CATEGORICAL_LEVELS = 30
RQ2_GRAPH_STATS_COMPUTE_EXACT_BETWEENNESS = True
RQ2_GRAPH_STATS_COMPUTE_ALL_PAIRS_DISTANCE = True
RQ2_GRAPH_STATS_PRINT_FULL_TABLES = True
RQ2_GRAPH_STATS_OUT_BASE = Path("extendedPlotting") / "rq2_mode_aware_graph_stats"

# ------------------------------------------------------------
# General helpers
# ------------------------------------------------------------

def _rq2_stats_safe(value):
    text = str(value)
    return "".join(ch if ch.isalnum() or ch in "._=-" else "_" for ch in text).strip("_")


def _to_cpu_tensor(value, *, dtype=None):
    if torch.is_tensor(value):
        tensor = value.detach().cpu()
    elif hasattr(value, "to_dense"):
        tensor = value.to_dense().detach().cpu()
    else:
        tensor = torch.as_tensor(value)
    if dtype is not None:
        tensor = tensor.to(dtype=dtype)
    return tensor


def _dense_features(context):
    x = context["attr"]
    if hasattr(x, "to_dense") and not torch.is_tensor(x):
        x = x.to_dense()
    elif torch.is_tensor(x) and x.is_sparse:
        x = x.to_dense()
    x = _to_cpu_tensor(x, dtype=torch.float32)
    if x.ndim != 2:
        raise ValueError(f"Expected a 2-D feature matrix, got shape={tuple(x.shape)}")
    return x


def _canonical_pair(u, v):
    u, v = int(u), int(v)
    return (u, v) if u < v else (v, u)


def _canonical_pair_series(u_values, v_values):
    return [_canonical_pair(u, v) for u, v in zip(u_values, v_values)]


def _clear_model_cache_and_call(model, x, adj):
    has_cache = hasattr(model, "adj_preped")
    old_cache = getattr(model, "adj_preped", None) if has_cache else None
    try:
        if has_cache:
            model.adj_preped = None
        return model(x, adj)
    finally:
        if has_cache:
            model.adj_preped = old_cache


def _clean_prediction_statistics(run):
    context = run["context"]
    model = context["model"]
    model.eval()
    device = next(model.parameters()).device
    x = context["attr"].to(device)
    adj = run["adj_orig"].to(device)
    with torch.no_grad():
        logits = _clear_model_cache_and_call(model, x, adj)
        probabilities = torch.softmax(logits, dim=-1)
        predictions = logits.argmax(dim=-1)
        confidence = probabilities.max(dim=-1).values
        entropy = -(probabilities * probabilities.clamp_min(1e-12).log()).sum(dim=-1)
        top2 = torch.topk(probabilities, k=min(2, probabilities.size(1)), dim=-1).values
        if top2.size(1) == 1:
            margin = top2[:, 0]
        else:
            margin = top2[:, 0] - top2[:, 1]
        labels = context["labels"].to(device=device, dtype=torch.long)
        true_probability = probabilities.gather(1, labels.view(-1, 1)).squeeze(1)
        correct = predictions.eq(labels)
    return {
        "clean_prediction": predictions.detach().cpu().numpy(),
        "clean_confidence": confidence.detach().cpu().numpy(),
        "clean_entropy": entropy.detach().cpu().numpy(),
        "clean_margin": margin.detach().cpu().numpy(),
        "clean_true_probability": true_probability.detach().cpu().numpy(),
        "clean_correct": correct.detach().cpu().numpy().astype(bool),
    }


def _build_clean_graph_stats(context):
    """Compute graph-level and per-node quantities once for a clean graph."""
    n_nodes = int(context["n_nodes"])
    edge_index = _to_cpu_tensor(context["edge_index"], dtype=torch.long).numpy()

    # Canonical undirected edge set.
    edge_set = {
        _canonical_pair(u, v)
        for u, v in zip(edge_index[0].tolist(), edge_index[1].tolist())
        if int(u) != int(v)
    }

    graph = nx.Graph()
    graph.add_nodes_from(range(n_nodes))
    graph.add_edges_from(edge_set)

    neighbors = [set(graph.neighbors(node)) for node in range(n_nodes)]
    degree = np.asarray([graph.degree(node) for node in range(n_nodes)], dtype=float)
    clustering = np.asarray([nx.clustering(graph, node) for node in range(n_nodes)], dtype=float)
    triangles_map = nx.triangles(graph)
    triangles = np.asarray([triangles_map[node] for node in range(n_nodes)], dtype=float)

    try:
        core_map = nx.core_number(graph)
    except nx.NetworkXError:
        core_map = {node: 0 for node in graph.nodes}
    core_number = np.asarray([core_map.get(node, 0) for node in range(n_nodes)], dtype=float)

    pagerank_map = nx.pagerank(graph, alpha=0.85, max_iter=1000)
    pagerank = np.asarray([pagerank_map.get(node, 0.0) for node in range(n_nodes)], dtype=float)

    closeness_map = nx.closeness_centrality(graph)
    closeness = np.asarray([closeness_map.get(node, 0.0) for node in range(n_nodes)], dtype=float)

    if RQ2_GRAPH_STATS_COMPUTE_EXACT_BETWEENNESS:
        betweenness_map = nx.betweenness_centrality(graph, normalized=True)
    else:
        k_approx = min(256, n_nodes)
        betweenness_map = nx.betweenness_centrality(
            graph,
            k=k_approx,
            normalized=True,
            seed=0,
        )
    betweenness = np.asarray([betweenness_map.get(node, 0.0) for node in range(n_nodes)], dtype=float)

    try:
        eigenvector_map = nx.eigenvector_centrality(graph, max_iter=5000, tol=1e-09)
    except Exception:
        eigenvector_map = {node: 0.0 for node in graph.nodes}
    eigenvector = np.asarray([eigenvector_map.get(node, 0.0) for node in range(n_nodes)], dtype=float)

    components = list(nx.connected_components(graph))
    component_id = np.full(n_nodes, -1, dtype=int)
    component_size = np.zeros(n_nodes, dtype=float)
    for component_index, nodes in enumerate(components):
        size = len(nodes)
        idx = np.fromiter(nodes, dtype=int)
        component_id[idx] = component_index
        component_size[idx] = size

    bridges = {_canonical_pair(u, v) for u, v in nx.bridges(graph)}

    distance_matrix = None
    if RQ2_GRAPH_STATS_COMPUTE_ALL_PAIRS_DISTANCE:
        rows = np.fromiter((w for u, v in edge_set for w in (u, v)), dtype=int)
        cols = np.fromiter((w for u, v in edge_set for w in (v, u)), dtype=int)
        data = np.ones(rows.size, dtype=np.uint8)
        adjacency_csr = sparse.csr_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes))
        distance_matrix = shortest_path(
            adjacency_csr,
            directed=False,
            unweighted=True,
            return_predecessors=False,
        )

    x = _dense_features(context)
    x_np = x.numpy()
    feature_norm_l2 = np.linalg.norm(x_np, axis=1)
    feature_norm_l1 = np.abs(x_np).sum(axis=1)
    feature_nnz = np.count_nonzero(x_np, axis=1).astype(float)

    labels = _to_cpu_tensor(context["labels"], dtype=torch.long).numpy()
    class_counts = Counter(labels.tolist())
    class_frequency = np.asarray([class_counts[int(label)] for label in labels], dtype=float)

    split_name = np.full(n_nodes, "other", dtype=object)
    for name, key in (("train", "idx_train"), ("validation", "idx_val"), ("test", "idx_test")):
        if key in context:
            idx = _to_cpu_tensor(context[key], dtype=torch.long).numpy()
            split_name[idx] = name

    graph_summary = {
        "n_nodes": n_nodes,
        "n_edges": graph.number_of_edges(),
        "density": nx.density(graph),
        "n_components": nx.number_connected_components(graph),
        "largest_component_fraction": max((len(c) for c in components), default=0) / max(1, n_nodes),
        "average_degree": float(degree.mean()),
        "degree_std": float(degree.std(ddof=1)) if n_nodes > 1 else 0.0,
        "average_clustering": float(nx.average_clustering(graph)),
        "transitivity": float(nx.transitivity(graph)),
        "assortativity_degree": float(nx.degree_assortativity_coefficient(graph)),
        "n_bridges": len(bridges),
    }

    return {
        "graph": graph,
        "graph_summary": graph_summary,
        "edge_set": edge_set,
        "neighbors": neighbors,
        "bridges": bridges,
        "distance_matrix": distance_matrix,
        "features": x_np,
        "labels": labels,
        "split_name": split_name,
        "degree": degree,
        "clustering": clustering,
        "triangles": triangles,
        "core_number": core_number,
        "pagerank": pagerank,
        "closeness": closeness,
        "betweenness": betweenness,
        "eigenvector": eigenvector,
        "component_id": component_id,
        "component_size": component_size,
        "feature_norm_l2": feature_norm_l2,
        "feature_norm_l1": feature_norm_l1,
        "feature_nnz": feature_nnz,
        "class_frequency": class_frequency,
    }


def _pair_topology_metrics(u, v, clean):
    nu = clean["neighbors"][u]
    nv = clean["neighbors"][v]
    common = nu & nv
    union = nu | nv
    du = len(nu)
    dv = len(nv)
    n_common = len(common)

    common_degrees = np.asarray([len(clean["neighbors"][w]) for w in common], dtype=float)
    aa_degrees = common_degrees[common_degrees > 1]
    adamic_adar = float(np.sum(1.0 / np.log(aa_degrees))) if aa_degrees.size else 0.0
    resource_allocation = float(np.sum(1.0 / common_degrees[common_degrees > 0])) if np.any(common_degrees > 0) else 0.0

    min_degree = min(du, dv)
    max_degree = max(du, dv)
    product_degree = du * dv

    if clean["distance_matrix"] is not None:
        path_length = float(clean["distance_matrix"][u, v])
        reachable = np.isfinite(path_length)
    else:
        try:
            path_length = float(nx.shortest_path_length(clean["graph"], u, v))
            reachable = True
        except nx.NetworkXNoPath:
            path_length = np.inf
            reachable = False

    return {
        "common_neighbors": float(n_common),
        "neighbor_union_size": float(len(union)),
        "jaccard_coefficient": n_common / max(1, len(union)),
        "salton_cosine_index": n_common / math.sqrt(product_degree) if product_degree > 0 else 0.0,
        "sorensen_index": (2.0 * n_common) / max(1, du + dv),
        "hub_promoted_index": n_common / min_degree if min_degree > 0 else 0.0,
        "hub_depressed_index": n_common / max_degree if max_degree > 0 else 0.0,
        "leicht_holme_newman_index": n_common / product_degree if product_degree > 0 else 0.0,
        "adamic_adar_index": adamic_adar,
        "resource_allocation_index": resource_allocation,
        "preferential_attachment": float(product_degree),
        "same_connected_component": bool(clean["component_id"][u] == clean["component_id"][v]),
        "clean_shortest_path_length": path_length if reachable else np.nan,
        "inverse_shortest_path": 1.0 / path_length if reachable and path_length > 0 else 0.0,
        "has_two_hop_path": bool(n_common > 0),
    }


def _feature_pair_metrics(u, v, clean):
    xu = clean["features"][u]
    xv = clean["features"][v]
    dot = float(np.dot(xu, xv))
    norm_u = float(clean["feature_norm_l2"][u])
    norm_v = float(clean["feature_norm_l2"][v])
    cosine = dot / (norm_u * norm_v) if norm_u > 0 and norm_v > 0 else 0.0
    diff = xu - xv
    support_u = xu != 0
    support_v = xv != 0
    support_intersection = int(np.logical_and(support_u, support_v).sum())
    support_union = int(np.logical_or(support_u, support_v).sum())
    return {
        "feature_dot_product": dot,
        "feature_cosine_similarity": cosine,
        "feature_l1_distance": float(np.abs(diff).sum()),
        "feature_l2_distance": float(np.linalg.norm(diff)),
        "feature_support_intersection": float(support_intersection),
        "feature_support_union": float(support_union),
        "feature_support_jaccard": support_intersection / max(1, support_union),
    }


def _add_endpoint_pair_summaries(row, u, v, clean, prediction_stats):
    structural_metrics = [
        "degree",
        "clustering",
        "triangles",
        "core_number",
        "pagerank",
        "closeness",
        "betweenness",
        "eigenvector",
        "component_size",
        "feature_norm_l2",
        "feature_norm_l1",
        "feature_nnz",
        "class_frequency",
    ]
    prediction_metrics = [
        "clean_confidence",
        "clean_entropy",
        "clean_margin",
        "clean_true_probability",
    ]

    for metric in structural_metrics:
        u_value = float(clean[metric][u])
        v_value = float(clean[metric][v])
        row[f"u_{metric}"] = u_value
        row[f"v_{metric}"] = v_value
        row[f"pair_{metric}_mean"] = (u_value + v_value) / 2.0
        row[f"pair_{metric}_min"] = min(u_value, v_value)
        row[f"pair_{metric}_max"] = max(u_value, v_value)
        row[f"pair_{metric}_sum"] = u_value + v_value
        row[f"pair_{metric}_absdiff"] = abs(u_value - v_value)
        row[f"pair_{metric}_product"] = u_value * v_value

    for metric in prediction_metrics:
        u_value = float(prediction_stats[metric][u])
        v_value = float(prediction_stats[metric][v])
        row[f"u_{metric}"] = u_value
        row[f"v_{metric}"] = v_value
        row[f"pair_{metric}_mean"] = (u_value + v_value) / 2.0
        row[f"pair_{metric}_min"] = min(u_value, v_value)
        row[f"pair_{metric}_max"] = max(u_value, v_value)
        row[f"pair_{metric}_sum"] = u_value + v_value
        row[f"pair_{metric}_absdiff"] = abs(u_value - v_value)
        row[f"pair_{metric}_product"] = u_value * v_value


def _gini(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0 or np.allclose(values.sum(), 0):
        return 0.0
    values = np.sort(np.clip(values, 0, None))
    n = values.size
    return float((2.0 * np.sum((np.arange(1, n + 1)) * values) / (n * values.sum())) - (n + 1) / n)


def _endpoint_concentration(edge_df, n_nodes):
    counts = np.zeros(n_nodes, dtype=float)
    np.add.at(counts, edge_df["u"].to_numpy(dtype=int), 1)
    np.add.at(counts, edge_df["v"].to_numpy(dtype=int), 1)
    active = counts[counts > 0]
    total = active.sum()
    shares = np.sort(active / total)[::-1] if total > 0 else np.array([])

    def top_share(fraction):
        if shares.size == 0:
            return 0.0
        k = max(1, int(math.ceil(fraction * shares.size)))
        return float(shares[:k].sum())

    return {
        "n_unique_endpoint_nodes": int((counts > 0).sum()),
        "endpoint_node_coverage": float((counts > 0).mean()),
        "mean_candidate_incidence_active_node": float(active.mean()) if active.size else 0.0,
        "max_candidate_incidence": float(active.max()) if active.size else 0.0,
        "candidate_incidence_gini": _gini(active),
        "candidate_incidence_hhi": float(np.square(shares).sum()) if shares.size else 0.0,
        "top_1pct_endpoint_share": top_share(0.01),
        "top_5pct_endpoint_share": top_share(0.05),
        "top_10pct_endpoint_share": top_share(0.10),
    }


def _candidate_pair_network_summary(edge_df, n_nodes):
    graph = nx.Graph()
    graph.add_nodes_from(range(n_nodes))
    graph.add_edges_from(edge_df[["u", "v"]].itertuples(index=False, name=None))
    active_nodes = [node for node, degree in graph.degree() if degree > 0]
    active_graph = graph.subgraph(active_nodes).copy()
    if active_graph.number_of_nodes() == 0:
        return {
            "candidate_pair_network_density": 0.0,
            "candidate_pair_network_components": 0,
            "candidate_pair_network_largest_component_fraction": 0.0,
            "candidate_pair_network_average_clustering": 0.0,
            "candidate_pair_network_transitivity": 0.0,
        }
    components = list(nx.connected_components(active_graph))
    return {
        "candidate_pair_network_density": float(nx.density(active_graph)),
        "candidate_pair_network_components": int(nx.number_connected_components(active_graph)),
        "candidate_pair_network_largest_component_fraction": (
            max(len(component) for component in components) / active_graph.number_of_nodes()
        ),
        "candidate_pair_network_average_clustering": float(nx.average_clustering(active_graph)),
        "candidate_pair_network_transitivity": float(nx.transitivity(active_graph)),
    }


def _cohen_d(x1, x0):
    x1 = np.asarray(x1, dtype=float)
    x0 = np.asarray(x0, dtype=float)
    x1 = x1[np.isfinite(x1)]
    x0 = x0[np.isfinite(x0)]
    if x1.size < 2 or x0.size < 2:
        return np.nan
    variance = ((x1.size - 1) * x1.var(ddof=1) + (x0.size - 1) * x0.var(ddof=1)) / (x1.size + x0.size - 2)
    if variance <= 0:
        return 0.0 if np.isclose(x1.mean(), x0.mean()) else np.sign(x1.mean() - x0.mean()) * np.inf
    return float((x1.mean() - x0.mean()) / math.sqrt(variance))


def _hedges_g(x1, x0):
    d = _cohen_d(x1, x0)
    n1 = np.isfinite(np.asarray(x1, dtype=float)).sum()
    n0 = np.isfinite(np.asarray(x0, dtype=float)).sum()
    if not np.isfinite(d) or n1 + n0 <= 3:
        return d
    correction = 1.0 - 3.0 / (4.0 * (n1 + n0) - 9.0)
    return float(correction * d)


def _benjamini_hochberg(p_values):
    p = np.asarray(p_values, dtype=float)
    adjusted = np.full(p.shape, np.nan, dtype=float)
    finite_idx = np.flatnonzero(np.isfinite(p))
    if finite_idx.size == 0:
        return adjusted
    order = finite_idx[np.argsort(p[finite_idx])]
    ranked = p[order] * finite_idx.size / np.arange(1, finite_idx.size + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adjusted[order] = np.minimum(ranked, 1.0)
    return adjusted


def _numeric_comparison_table(frame, group_cols, excluded_columns):
    rows = []
    numeric_columns = [
        column for column in frame.select_dtypes(include=[np.number, "bool"]).columns
        if column not in excluded_columns
    ]

    for group_key, group in frame.groupby(group_cols, dropna=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        group_meta = dict(zip(group_cols, group_key))
        positive = group[group["label_group"] == 1]
        zero = group[group["label_group"] == 0]

        for column in numeric_columns:
            x1 = pd.to_numeric(positive[column], errors="coerce").to_numpy(dtype=float)
            x0 = pd.to_numeric(zero[column], errors="coerce").to_numpy(dtype=float)
            x1 = x1[np.isfinite(x1)]
            x0 = x0[np.isfinite(x0)]
            if x1.size == 0 or x0.size == 0:
                continue

            try:
                mw = mannwhitneyu(x1, x0, alternative="two-sided")
                mw_u = float(mw.statistic)
                mw_p = float(mw.pvalue)
                rank_biserial = 2.0 * mw_u / (x1.size * x0.size) - 1.0
            except Exception:
                mw_u, mw_p, rank_biserial = np.nan, np.nan, np.nan

            try:
                ks = ks_2samp(x1, x0, alternative="two-sided", method="auto")
                ks_stat, ks_p = float(ks.statistic), float(ks.pvalue)
            except Exception:
                ks_stat, ks_p = np.nan, np.nan

            pooled_values = np.concatenate([x1, x0])
            pooled_labels = np.concatenate([np.ones(x1.size), np.zeros(x0.size)])
            try:
                point_biserial, point_biserial_p = pointbiserialr(pooled_labels, pooled_values)
            except Exception:
                point_biserial, point_biserial_p = np.nan, np.nan

            q1_1, q3_1 = np.quantile(x1, [0.25, 0.75])
            q1_0, q3_0 = np.quantile(x0, [0.25, 0.75])
            mean1, mean0 = float(x1.mean()), float(x0.mean())

            rows.append({
                **group_meta,
                "feature": column,
                "n_label_1": int(x1.size),
                "n_label_0": int(x0.size),
                "label_1_mean": mean1,
                "label_1_std": float(x1.std(ddof=1)) if x1.size > 1 else 0.0,
                "label_1_median": float(np.median(x1)),
                "label_1_q1": float(q1_1),
                "label_1_q3": float(q3_1),
                "label_1_iqr": float(q3_1 - q1_1),
                "label_1_min": float(x1.min()),
                "label_1_max": float(x1.max()),
                "label_0_mean": mean0,
                "label_0_std": float(x0.std(ddof=1)) if x0.size > 1 else 0.0,
                "label_0_median": float(np.median(x0)),
                "label_0_q1": float(q1_0),
                "label_0_q3": float(q3_0),
                "label_0_iqr": float(q3_0 - q1_0),
                "label_0_min": float(x0.min()),
                "label_0_max": float(x0.max()),
                "mean_difference_1_minus_0": mean1 - mean0,
                "median_difference_1_minus_0": float(np.median(x1) - np.median(x0)),
                "mean_ratio_1_over_0": mean1 / mean0 if not np.isclose(mean0, 0.0) else np.nan,
                "cohen_d": _cohen_d(x1, x0),
                "hedges_g": _hedges_g(x1, x0),
                "mann_whitney_u": mw_u,
                "rank_biserial_correlation": rank_biserial,
                "mann_whitney_p": mw_p,
                "ks_statistic": ks_stat,
                "ks_p": ks_p,
                "point_biserial_correlation": float(point_biserial),
                "point_biserial_p": float(point_biserial_p),
            })

    result = pd.DataFrame(rows)
    if not result.empty:
        for p_column in ("mann_whitney_p", "ks_p", "point_biserial_p"):
            result[f"{p_column}_fdr_bh"] = np.nan
            for _, idx in result.groupby(group_cols, dropna=False).groups.items():
                result.loc[idx, f"{p_column}_fdr_bh"] = _benjamini_hochberg(result.loc[idx, p_column])
    return result


def _cramers_v(contingency):
    contingency = np.asarray(contingency, dtype=float)
    n = contingency.sum()
    if n <= 0 or min(contingency.shape) <= 1:
        return np.nan
    chi2 = chi2_contingency(contingency, correction=False)[0]
    phi2 = chi2 / n
    r, k = contingency.shape
    phi2_corr = max(0.0, phi2 - ((k - 1) * (r - 1)) / max(1, n - 1))
    r_corr = r - ((r - 1) ** 2) / max(1, n - 1)
    k_corr = k - ((k - 1) ** 2) / max(1, n - 1)
    denom = min(k_corr - 1, r_corr - 1)
    return float(math.sqrt(phi2_corr / denom)) if denom > 0 else np.nan


def _categorical_comparison_tables(frame, group_cols, excluded_columns):
    summary_rows = []
    level_rows = []

    candidate_columns = []
    for column in frame.columns:
        if column in excluded_columns or column == "label_group":
            continue
        nunique = frame[column].nunique(dropna=False)
        if (
            frame[column].dtype == object
            or str(frame[column].dtype).startswith("category")
            or frame[column].dtype == bool
        ) and 1 < nunique <= RQ2_GRAPH_STATS_MAX_CATEGORICAL_LEVELS:
            candidate_columns.append(column)

    for group_key, group in frame.groupby(group_cols, dropna=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        group_meta = dict(zip(group_cols, group_key))

        for column in candidate_columns:
            table = pd.crosstab(group["label_group"], group[column].fillna("<NA>"), dropna=False)
            table = table.reindex(index=[0, 1], fill_value=0)
            if table.shape[1] < 2:
                continue
            try:
                chi2, chi2_p, dof, _ = chi2_contingency(table.to_numpy(), correction=False)
                chi2 = float(chi2)
                chi2_p = float(chi2_p)
                dof = int(dof)
            except Exception:
                chi2, chi2_p, dof = np.nan, np.nan, 0

            fisher_odds_ratio = np.nan
            fisher_p = np.nan
            risk_difference = np.nan
            risk_ratio = np.nan
            odds_ratio_corrected = np.nan
            positive_level = None

            if table.shape == (2, 2):
                levels = list(table.columns)
                positive_level = levels[-1]
                a = float(table.loc[1, positive_level])
                b = float(table.loc[1].sum() - a)
                c = float(table.loc[0, positive_level])
                d = float(table.loc[0].sum() - c)
                try:
                    fisher_odds_ratio, fisher_p = fisher_exact([[a, b], [c, d]])
                    fisher_odds_ratio = float(fisher_odds_ratio)
                    fisher_p = float(fisher_p)
                except Exception:
                    pass
                p1 = a / max(1.0, a + b)
                p0 = c / max(1.0, c + d)
                risk_difference = p1 - p0
                risk_ratio = p1 / p0 if p0 > 0 else np.inf if p1 > 0 else np.nan
                odds_ratio_corrected = ((a + 0.5) * (d + 0.5)) / ((b + 0.5) * (c + 0.5))

            summary_rows.append({
                **group_meta,
                "feature": column,
                "n_levels": int(table.shape[1]),
                "chi_square": chi2,
                "chi_square_df": dof,
                "chi_square_p": chi2_p,
                "cramers_v": _cramers_v(table.to_numpy()),
                "binary_positive_level": positive_level,
                "risk_difference_label1_minus_label0": risk_difference,
                "risk_ratio_label1_over_label0": risk_ratio,
                "odds_ratio_haldane_anscombe": odds_ratio_corrected,
                "fisher_odds_ratio": fisher_odds_ratio,
                "fisher_p": fisher_p,
            })

            for level in table.columns:
                n1 = int(table.loc[1, level])
                n0 = int(table.loc[0, level])
                total1 = int(table.loc[1].sum())
                total0 = int(table.loc[0].sum())
                level_rows.append({
                    **group_meta,
                    "feature": column,
                    "level": level,
                    "label_1_count": n1,
                    "label_1_fraction": n1 / max(1, total1),
                    "label_0_count": n0,
                    "label_0_fraction": n0 / max(1, total0),
                    "fraction_difference_1_minus_0": n1 / max(1, total1) - n0 / max(1, total0),
                })

    summary = pd.DataFrame(summary_rows)
    levels = pd.DataFrame(level_rows)
    if not summary.empty:
        for p_column in ("chi_square_p", "fisher_p"):
            summary[f"{p_column}_fdr_bh"] = np.nan
            for _, idx in summary.groupby(group_cols, dropna=False).groups.items():
                summary.loc[idx, f"{p_column}_fdr_bh"] = _benjamini_hochberg(summary.loc[idx, p_column])
    return summary, levels


def _seed_level_paired_differences(edge_df, group_cols, numeric_features):
    rows = []
    seed_means = (
        edge_df.groupby(group_cols + ["seed", "label_group"], dropna=False)[numeric_features]
        .mean()
        .reset_index()
    )

    for group_key, group in seed_means.groupby(group_cols, dropna=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        meta = dict(zip(group_cols, group_key))
        for feature in numeric_features:
            pivot = group.pivot(index="seed", columns="label_group", values=feature)
            if 0 not in pivot.columns or 1 not in pivot.columns:
                continue
            paired = pivot[[0, 1]].dropna()
            differences = paired[1] - paired[0]
            n_seeds = int(differences.size)
            if n_seeds >= 2:
                try:
                    t_stat, t_p = ttest_rel(paired[1], paired[0])
                except Exception:
                    t_stat, t_p = np.nan, np.nan
                try:
                    w_stat, w_p = wilcoxon(differences)
                except Exception:
                    w_stat, w_p = np.nan, np.nan
            else:
                t_stat = t_p = w_stat = w_p = np.nan
            rows.append({
                **meta,
                "feature": feature,
                "n_seeds": n_seeds,
                "seed_mean_difference_1_minus_0": float(differences.mean()) if n_seeds else np.nan,
                "seed_sd_difference": float(differences.std(ddof=1)) if n_seeds > 1 else 0.0,
                "seed_sem_difference": float(differences.std(ddof=1) / math.sqrt(n_seeds)) if n_seeds > 1 else 0.0,
                "paired_t_statistic": float(t_stat),
                "paired_t_p": float(t_p),
                "wilcoxon_statistic": float(w_stat),
                "wilcoxon_p": float(w_p),
            })
    result = pd.DataFrame(rows)
    if not result.empty:
        for p_column in ("paired_t_p", "wilcoxon_p"):
            result[f"{p_column}_fdr_bh"] = np.nan
            for _, idx in result.groupby(group_cols, dropna=False).groups.items():
                result.loc[idx, f"{p_column}_fdr_bh"] = _benjamini_hochberg(result.loc[idx, p_column])
    return result


# ------------------------------------------------------------
# Mode-aware group construction
# ------------------------------------------------------------

def _resolve_rq2_graph_stats_modes():
    """Resolve configured modes against the modes actually present in mining_runs.

    This makes the cell work with endpoint-only, subset-only, or mixed runs while
    keeping RQ2_GRAPH_STATS_SCORING_MODES as the user-facing configuration.
    """
    configured = RQ2_GRAPH_STATS_SCORING_MODES
    if isinstance(configured, str):
        configured = [configured]

    requested_modes = [str(mode) for mode in configured]
    allowed_modes = {"endpoint", "subset_accuracy_drop"}

    unknown_modes = sorted(set(requested_modes) - allowed_modes)
    if unknown_modes:
        raise ValueError(
            f"Unsupported graph-statistics scoring modes: {unknown_modes}"
        )
    if not requested_modes:
        raise ValueError("RQ2_GRAPH_STATS_SCORING_MODES must not be empty.")

    available_modes = {
        str(run.get("scoring_mode"))
        for run in mining_runs
    }
    active_modes = [
        mode for mode in requested_modes
        if mode in available_modes
    ]

    missing_modes = sorted(set(requested_modes) - available_modes)
    if missing_modes:
        warnings.warn(
            "Configured scoring modes not present in mining_runs will be skipped: "
            f"{missing_modes}. Available modes: {sorted(available_modes)}"
        )

    if not active_modes:
        raise RuntimeError(
            "None of the configured scoring modes are available in mining_runs. "
            f"Requested={requested_modes}, available={sorted(available_modes)}"
        )

    return active_modes


def _rq2_endpoint_hop(run):
    """Return an integer endpoint hop without requiring endpoint metadata.

    endpoint runs use their actual hop. subset_accuracy_drop runs have no
    endpoint-local mining hop, so zero is used only as a neutral grouping value.
    """
    if str(run.get("scoring_mode", "")) != "endpoint":
        return 0

    value = run.get("endpoint_mining_hop", 0)
    if value is None or pd.isna(value):
        return 0
    return int(value)


def _fallback_mining_rows(run):
    """Create one aligned row per observed candidate when a miner has no row table."""
    n_cands = int(run["n_cands"])
    return pd.DataFrame({
        "sample_index": np.arange(n_cands, dtype=int),
        "u": np.asarray(run["cand_src"], dtype=int).reshape(-1),
        "v": np.asarray(run["cand_dst"], dtype=int).reshape(-1),
        "exists_clean": np.asarray(run["exists_list"], dtype=float).reshape(-1) > 0.5,
    })


def _rank_percentiles(values):
    values = np.asarray(values, dtype=float).reshape(-1)
    out = np.full(values.shape, np.nan, dtype=float)
    valid = np.isfinite(values)
    if valid.any():
        out[valid] = pd.Series(values[valid]).rank(method="average", pct=True).to_numpy()
    return out


def _prepare_comparison_rows(run):
    """Return the candidate rows used in the mode-specific comparison.

    Internal group coding is retained for compatibility with the original
    statistics helpers:
        label_group == 1:
            endpoint -> endpoint label 1
            subset   -> top score fraction
        label_group == 0:
            endpoint -> endpoint label 0
            subset   -> bottom score fraction
    """
    scoring_mode = str(run["scoring_mode"])
    mining_result = run["mining_result"]

    mining_rows = pd.DataFrame(mining_result.get("rows", []))
    if mining_rows.empty:
        mining_rows = _fallback_mining_rows(run)
    mining_rows = mining_rows.sort_values("sample_index").reset_index(drop=True)

    n_rows = len(mining_rows)
    if n_rows != int(run["n_cands"]):
        raise RuntimeError(
            f"Mining row count ({n_rows}) does not match run['n_cands'] "
            f"({run['n_cands']}) for mode={scoring_mode}."
        )

    if scoring_mode == "endpoint":
        endpoint_labels = np.asarray(
            mining_result.get("endpoint_labels", run["labels_norm"]),
            dtype=float,
        ).reshape(-1)
        if endpoint_labels.size != n_rows:
            raise RuntimeError("Endpoint rows and endpoint labels are not aligned.")

        score_raw = np.asarray(run.get("score_raw", endpoint_labels), dtype=float).reshape(-1)
        score_norm = np.asarray(run.get("score_norm", endpoint_labels), dtype=float).reshape(-1)
        if score_raw.size != n_rows or score_norm.size != n_rows:
            raise RuntimeError("Endpoint score arrays are not aligned with mining rows.")

        mining_rows["score_raw"] = score_raw
        mining_rows["score_norm"] = score_norm
        mining_rows["score_percentile"] = _rank_percentiles(score_raw)
        mining_rows["inclusion_count"] = 1
        mining_rows["label_value"] = endpoint_labels
        mining_rows["label_group"] = (
            endpoint_labels > float(RQ2_GRAPH_STATS_LABEL_THRESHOLD)
        ).astype(int)
        mining_rows["label_name"] = np.where(
            mining_rows["label_group"].to_numpy() == 1,
            "endpoint_label_1_harmful",
            "endpoint_label_0_non_harmful",
        )
        mining_rows["comparison_rule"] = (
            f"endpoint_binary_threshold_gt_{RQ2_GRAPH_STATS_LABEL_THRESHOLD:g}"
        )
        mining_rows["selection_rank"] = np.nan

        n_group_1 = int((mining_rows["label_group"] == 1).sum())
        n_group_0 = int((mining_rows["label_group"] == 0).sum())
        diagnostics = {
            "scoring_mode": scoring_mode,
            "comparison_rule": mining_rows["comparison_rule"].iloc[0],
            "n_candidates_available": n_rows,
            "n_candidates_valid": n_rows,
            "n_group_1": n_group_1,
            "n_group_0": n_group_0,
            "selected_fraction_per_group": np.nan,
            "bottom_cutoff_score": 0.0,
            "top_cutoff_score": 1.0,
            "score_min": float(np.nanmin(score_raw)) if n_rows else np.nan,
            "score_max": float(np.nanmax(score_raw)) if n_rows else np.nan,
            "constant_score_run": bool(n_rows and np.allclose(score_raw, score_raw[0])),
        }
        return mining_rows, diagnostics

    if scoring_mode == "subset_accuracy_drop":
        score_raw = np.asarray(
            run.get("score_raw", mining_result.get("score_raw")),
            dtype=float,
        ).reshape(-1)
        score_norm = np.asarray(
            run.get("score_norm", mining_result.get("score_norm")),
            dtype=float,
        ).reshape(-1)
        score_percentile = np.asarray(
            run.get(
                "score_percentile",
                mining_result.get("score_percentile", _rank_percentiles(score_raw)),
            ),
            dtype=float,
        ).reshape(-1)
        inclusion_count = np.asarray(
            run.get(
                "inclusion_count",
                mining_result.get("inclusion_count", np.ones(n_rows, dtype=int)),
            ),
            dtype=float,
        ).reshape(-1)

        for name, values in {
            "score_raw": score_raw,
            "score_norm": score_norm,
            "score_percentile": score_percentile,
            "inclusion_count": inclusion_count,
        }.items():
            if values.size != n_rows:
                raise RuntimeError(
                    f"{name} contains {values.size} values but the subset run has "
                    f"{n_rows} aligned candidate rows."
                )

        valid = np.isfinite(score_raw)
        valid_indices = np.flatnonzero(valid)
        n_valid = int(valid_indices.size)
        if n_valid < 2:
            raise RuntimeError(
                "subset_accuracy_drop needs at least two observed finite scores "
                "for a top-versus-bottom comparison."
            )

        fraction = float(RQ2_GRAPH_STATS_SUBSET_EXTREME_FRACTION)
        if not 0.0 < fraction <= 0.5:
            raise ValueError(
                "RQ2_GRAPH_STATS_SUBSET_EXTREME_FRACTION must lie in (0, 0.5]."
            )

        n_per_group = max(1, int(math.floor(fraction * n_valid)))
        n_per_group = min(n_per_group, n_valid // 2)
        if n_per_group < 1:
            raise RuntimeError("Not enough valid subset scores to form two groups.")

        # Deterministic exact-size groups. sample_index breaks score ties only at
        # the boundary; the diagnostics below expose whether boundary ties exist.
        sample_index_series = pd.to_numeric(
            mining_rows["sample_index"], errors="coerce"
        )
        sample_index_fallback = pd.Series(
            np.arange(n_rows, dtype=float),
            index=mining_rows.index,
        )
        sample_index_values = sample_index_series.where(
            sample_index_series.notna(),
            sample_index_fallback,
        ).to_numpy(dtype=float)
        order_within_valid = np.lexsort(
            (sample_index_values[valid_indices], score_raw[valid_indices])
        )
        ordered_indices = valid_indices[order_within_valid]
        bottom_indices = ordered_indices[:n_per_group]
        top_indices = ordered_indices[-n_per_group:]

        selected_indices = np.concatenate([bottom_indices, top_indices])
        selected = mining_rows.iloc[selected_indices].copy().reset_index(drop=True)

        selected["score_raw"] = score_raw[selected_indices]
        selected["score_norm"] = score_norm[selected_indices]
        selected["score_percentile"] = score_percentile[selected_indices]
        selected["inclusion_count"] = inclusion_count[selected_indices]
        selected["label_value"] = selected["score_raw"]
        selected["label_group"] = np.concatenate([
            np.zeros(n_per_group, dtype=int),
            np.ones(n_per_group, dtype=int),
        ])

        pct_label = f"{100.0 * fraction:g}%"
        selected["label_name"] = np.where(
            selected["label_group"].to_numpy() == 1,
            f"top_{pct_label}_subset_score",
            f"bottom_{pct_label}_subset_score",
        )
        selected["comparison_rule"] = f"subset_top_vs_bottom_{pct_label}"

        full_rank = np.full(n_rows, np.nan, dtype=float)
        full_rank[ordered_indices] = np.arange(1, n_valid + 1, dtype=float)
        selected["selection_rank"] = full_rank[selected_indices]

        bottom_cutoff = float(score_raw[bottom_indices[-1]])
        top_cutoff = float(score_raw[top_indices[0]])
        score_min = float(np.min(score_raw[valid_indices]))
        score_max = float(np.max(score_raw[valid_indices]))
        constant_score_run = bool(np.isclose(score_min, score_max))
        if constant_score_run:
            warnings.warn(
                "All observed subset scores are equal. The deterministic top/bottom "
                "split is formally computable but has no score separation and should "
                "not be interpreted as a harmfulness contrast."
            )

        diagnostics = {
            "scoring_mode": scoring_mode,
            "comparison_rule": selected["comparison_rule"].iloc[0],
            "n_candidates_available": n_rows,
            "n_candidates_valid": n_valid,
            "n_group_1": n_per_group,
            "n_group_0": n_per_group,
            "selected_fraction_per_group": n_per_group / n_valid,
            "bottom_cutoff_score": bottom_cutoff,
            "top_cutoff_score": top_cutoff,
            "score_min": score_min,
            "score_max": score_max,
            "constant_score_run": constant_score_run,
            "n_at_bottom_cutoff": int(np.isclose(score_raw[valid_indices], bottom_cutoff).sum()),
            "n_at_top_cutoff": int(np.isclose(score_raw[valid_indices], top_cutoff).sum()),
        }
        return selected, diagnostics

    raise ValueError(f"Unsupported scoring mode: {scoring_mode!r}")

# ------------------------------------------------------------
# Select the largest common candidate configuration
# ------------------------------------------------------------

requested_scoring_modes = _resolve_rq2_graph_stats_modes()
eligible_runs = [
    run for run in mining_runs
    if str(run["scoring_mode"]) in requested_scoring_modes
]
if not eligible_runs:
    raise RuntimeError(
        "No mining runs matched RQ2_GRAPH_STATS_SCORING_MODES="
        f"{requested_scoring_modes}."
    )

if RQ2_GRAPH_STATS_CANDIDATE_CONFIG_ID is None:
    largest_size = max(int(run["candidate_set_size"]) for run in eligible_runs)
    largest_runs = [
        run for run in eligible_runs
        if int(run["candidate_set_size"]) == largest_size
    ]
    # If several candidate mixtures have the same largest size, choose the one
    # with the largest requested PRBCD-derived fraction.
    selected_config_id = sorted(
        {
            (float(run["prbcd_candidate_fraction"]), str(run["candidate_config_id"]))
            for run in largest_runs
        },
        key=lambda item: (-item[0], item[1]),
    )[0][1]
else:
    selected_config_id = str(RQ2_GRAPH_STATS_CANDIDATE_CONFIG_ID)

requested_hops = (
    None
    if RQ2_GRAPH_STATS_ENDPOINT_HOPS is None
    else {int(value) for value in RQ2_GRAPH_STATS_ENDPOINT_HOPS}
)

selected_runs = []
for run in eligible_runs:
    if str(run["candidate_config_id"]) != selected_config_id:
        continue
    scoring_mode = str(run["scoring_mode"])
    if (
        scoring_mode == "endpoint"
        and requested_hops is not None
        and _rq2_endpoint_hop(run) not in requested_hops
    ):
        continue
    selected_runs.append(run)

if not selected_runs:
    raise RuntimeError(
        f"No runs matched candidate_config_id={selected_config_id!r}, "
        f"modes={requested_scoring_modes}, and the requested endpoint hops."
    )

missing_modes = sorted(
    set(requested_scoring_modes)
    - {str(run["scoring_mode"]) for run in selected_runs}
)
if missing_modes:
    warnings.warn(
        "The selected candidate configuration has no runs for these modes; "
        f"they will be skipped: {missing_modes}"
    )

selected_runs = sorted(
    selected_runs,
    key=lambda run: (
        str(run["scoring_mode"]),
        _rq2_endpoint_hop(run),
        int(run["seed"]),
    ),
)
selected_candidate_size = max(int(run["candidate_set_size"]) for run in selected_runs)
run_signature = hashlib.sha1(
    repr(sorted(
        (
            int(run["seed"]),
            str(run["scoring_mode"]),
            _rq2_endpoint_hop(run),
            str(run["candidate_hash"]),
        )
        for run in selected_runs
    )).encode()
).hexdigest()[:10]
mode_tag = "-".join(sorted({str(run["scoring_mode"]) for run in selected_runs}))
RQ2_GRAPH_STATS_OUT_DIR = (
    RQ2_GRAPH_STATS_OUT_BASE
    / f"{DATASET}__{_rq2_stats_safe(selected_config_id)}__{_rq2_stats_safe(mode_tag)}__hash-{run_signature}"
)
RQ2_GRAPH_STATS_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("RQ2 MODE-AWARE COMPREHENSIVE GRAPH-STATISTICS EXPERIMENT")
print("=" * 100)
print("Selected candidate configuration:", selected_config_id)
print("Candidate-set size:", selected_candidate_size)
print("Scoring modes:", sorted({str(run['scoring_mode']) for run in selected_runs}))
print("Seeds:", sorted({int(run['seed']) for run in selected_runs}))
print(
    "Endpoint hops:",
    sorted({
        _rq2_endpoint_hop(run)
        for run in selected_runs
        if str(run['scoring_mode']) == 'endpoint'
    }),
)
print(
    "Subset comparison:",
    f"top {100 * RQ2_GRAPH_STATS_SUBSET_EXTREME_FRACTION:g}% versus "
    f"bottom {100 * RQ2_GRAPH_STATS_SUBSET_EXTREME_FRACTION:g}% of observed score_raw",
)
print("Number of mining runs:", len(selected_runs))
print("Output directory:", RQ2_GRAPH_STATS_OUT_DIR.resolve())

# ------------------------------------------------------------
# Build the edge-level feature table
# ------------------------------------------------------------

edge_rows = []
graph_summary_rows = []
comparison_selection_rows = []
clean_graph_cache = {}
prediction_cache = {}

for run in selected_runs:
    seed = int(run["seed"])
    scoring_mode = str(run["scoring_mode"])
    hop = _rq2_endpoint_hop(run)
    context = run["context"]

    # The clean topology/features are generally dataset-invariant, but cache by
    # a simple topology signature to remain correct if that ever changes.
    edge_index_cpu = _to_cpu_tensor(context["edge_index"], dtype=torch.long)
    topology_signature = (
        int(context["n_nodes"]),
        int(edge_index_cpu.size(1)),
        int(edge_index_cpu[:, : min(100, edge_index_cpu.size(1))].sum().item()),
    )
    if topology_signature not in clean_graph_cache:
        print(f"Computing clean graph/node statistics for topology {topology_signature} ...")
        clean_graph_cache[topology_signature] = _build_clean_graph_stats(context)
    clean = clean_graph_cache[topology_signature]

    prediction_key = (seed, scoring_mode, hop, run["candidate_hash"])
    if prediction_key not in prediction_cache:
        prediction_cache[prediction_key] = _clean_prediction_statistics(run)
    prediction_stats = prediction_cache[prediction_key]

    candidate_source_run = candidate_runs_by_key[(seed, run["candidate_config_id"])]
    selected_prbcd = {_canonical_pair(*edge) for edge in candidate_source_run["selected_prbcd"]}
    random_candidates = {_canonical_pair(*edge) for edge in candidate_source_run["random_candidates"]}

    mining_rows, selection_diagnostics = _prepare_comparison_rows(run)
    selection_diagnostics.update({
        "dataset": DATASET,
        "seed": seed,
        "candidate_config_id": run["candidate_config_id"],
        "candidate_set_size": int(run["candidate_set_size"]),
        "prbcd_candidate_fraction": float(run["prbcd_candidate_fraction"]),
        "actual_prbcd_fraction": float(run["actual_prbcd_fraction"]),
        "endpoint_mining_hop": hop,
        "candidate_hash": run["candidate_hash"],
    })
    comparison_selection_rows.append(selection_diagnostics)

    for index, mining_row in mining_rows.iterrows():
        u = int(mining_row["u"])
        v = int(mining_row["v"])
        pair = _canonical_pair(u, v)
        label_value = float(mining_row["label_value"])
        label_group = int(mining_row["label_group"])
        label_name = str(mining_row["label_name"])
        comparison_rule = str(mining_row["comparison_rule"])
        exists_clean = bool(mining_row.get("exists_clean", pair in clean["edge_set"]))

        if pair in selected_prbcd:
            candidate_source = "prbcd_derived"
        elif pair in random_candidates:
            candidate_source = "random_candidate"
        else:
            candidate_source = "unknown"

        u_true = int(clean["labels"][u])
        v_true = int(clean["labels"][v])
        u_clean_pred = int(prediction_stats["clean_prediction"][u])
        v_clean_pred = int(prediction_stats["clean_prediction"][v])
        u_correct = bool(prediction_stats["clean_correct"][u])
        v_correct = bool(prediction_stats["clean_correct"][v])

        if scoring_mode == "endpoint":
            u_pert_pred = int(mining_row.get("u_pert_pred", -1))
            v_pert_pred = int(mining_row.get("v_pert_pred", -1))
            u_hit = bool(mining_row.get("u_hit", False))
            v_hit = bool(mining_row.get("v_hit", False))
            both_hit = bool(mining_row.get("both_hit", u_hit and v_hit))
            u_prediction_changed = bool(u_pert_pred >= 0 and u_pert_pred != u_clean_pred)
            v_prediction_changed = bool(v_pert_pred >= 0 and v_pert_pred != v_clean_pred)
            n_endpoint_predictions_changed = int(u_prediction_changed) + int(v_prediction_changed)
            exactly_one_hit = bool(u_hit ^ v_hit)
            neither_hit = bool(not u_hit and not v_hit)
            u_adopts_partner = bool(u_pert_pred >= 0 and u_pert_pred == v_true)
            v_adopts_partner = bool(v_pert_pred >= 0 and v_pert_pred == u_true)
            either_adopts_partner = bool(u_adopts_partner or v_adopts_partner)
            neighborhood_size = float(mining_row.get("neighborhood_size", np.nan))
            used_full_graph_fallback = bool(
                mining_row.get("used_full_graph_fallback", False)
            )
        else:
            # Subset mining does not produce per-edge perturbed endpoint states.
            u_pert_pred = np.nan
            v_pert_pred = np.nan
            u_hit = np.nan
            v_hit = np.nan
            both_hit = np.nan
            u_prediction_changed = np.nan
            v_prediction_changed = np.nan
            n_endpoint_predictions_changed = np.nan
            exactly_one_hit = np.nan
            neither_hit = np.nan
            u_adopts_partner = np.nan
            v_adopts_partner = np.nan
            either_adopts_partner = np.nan
            neighborhood_size = np.nan
            used_full_graph_fallback = np.nan

        row = {
            "dataset": DATASET,
            "seed": seed,
            "candidate_config_id": run["candidate_config_id"],
            "candidate_set_size": int(run["candidate_set_size"]),
            "prbcd_candidate_fraction": float(run["prbcd_candidate_fraction"]),
            "actual_prbcd_fraction": float(run["actual_prbcd_fraction"]),
            "scoring_mode": scoring_mode,
            "comparison_rule": comparison_rule,
            "endpoint_mining_hop": hop,
            "candidate_hash": run["candidate_hash"],
            "sample_index": int(mining_row.get("sample_index", index)),
            "u": u,
            "v": v,
            "canonical_u": pair[0],
            "canonical_v": pair[1],
            "candidate_source": candidate_source,
            "label_value": label_value,
            "label_group": label_group,
            "label_name": label_name,
            "score_raw": float(mining_row["score_raw"]),
            "score_norm": float(mining_row["score_norm"]),
            "score_percentile": float(mining_row["score_percentile"]),
            "inclusion_count": float(mining_row.get("inclusion_count", np.nan)),
            "selection_rank": float(mining_row.get("selection_rank", np.nan)),
            "exists_clean": exists_clean,
            "action": "delete" if exists_clean else "add",
            "is_clean_bridge": bool(pair in clean["bridges"]),
            "u_true_label": u_true,
            "v_true_label": v_true,
            "canonical_true_label_pair": f"{min(u_true, v_true)}-{max(u_true, v_true)}",
            "same_true_label": bool(u_true == v_true),
            "u_clean_prediction": u_clean_pred,
            "v_clean_prediction": v_clean_pred,
            "canonical_clean_prediction_pair": f"{min(u_clean_pred, v_clean_pred)}-{max(u_clean_pred, v_clean_pred)}",
            "same_clean_prediction": bool(u_clean_pred == v_clean_pred),
            "u_clean_correct": u_correct,
            "v_clean_correct": v_correct,
            "n_clean_correct_endpoints": int(u_correct) + int(v_correct),
            "both_endpoints_clean_correct": bool(u_correct and v_correct),
            "neither_endpoint_clean_correct": bool(not u_correct and not v_correct),
            "u_pert_prediction": u_pert_pred,
            "v_pert_prediction": v_pert_pred,
            "u_prediction_changed": u_prediction_changed,
            "v_prediction_changed": v_prediction_changed,
            "n_endpoint_predictions_changed": n_endpoint_predictions_changed,
            "u_hit": u_hit,
            "v_hit": v_hit,
            "both_hit": both_hit,
            "exactly_one_hit": exactly_one_hit,
            "neither_hit": neither_hit,
            "u_adopts_partner_true_label": u_adopts_partner,
            "v_adopts_partner_true_label": v_adopts_partner,
            "either_adopts_partner_true_label": either_adopts_partner,
            "u_split": str(clean["split_name"][u]),
            "v_split": str(clean["split_name"][v]),
            "same_split": bool(clean["split_name"][u] == clean["split_name"][v]),
            "canonical_split_pair": "-".join(
                sorted([str(clean["split_name"][u]), str(clean["split_name"][v])])
            ),
            "neighborhood_size": neighborhood_size,
            "used_full_graph_fallback": used_full_graph_fallback,
        }

        row.update(_pair_topology_metrics(u, v, clean))
        row.update(_feature_pair_metrics(u, v, clean))
        _add_endpoint_pair_summaries(row, u, v, clean, prediction_stats)
        edge_rows.append(row)

    graph_summary_rows.append({
        "seed": seed,
        "scoring_mode": scoring_mode,
        "comparison_rule": selection_diagnostics["comparison_rule"],
        "endpoint_mining_hop": hop,
        "candidate_config_id": run["candidate_config_id"],
        **clean["graph_summary"],
    })

edge_stats_df = pd.DataFrame(edge_rows)
graph_summary_df = pd.DataFrame(graph_summary_rows).drop_duplicates()
comparison_selection_df = pd.DataFrame(comparison_selection_rows)
if edge_stats_df.empty:
    raise RuntimeError("No edge-level graph-statistics rows were created.")

# Replace infinite path values, if any, with missing values for summaries.
edge_stats_df = edge_stats_df.replace([np.inf, -np.inf], np.nan)

# ------------------------------------------------------------
# Group-level summaries and endpoint concentration
# ------------------------------------------------------------

base_group_cols = [
    "candidate_config_id",
    "scoring_mode",
    "comparison_rule",
    "endpoint_mining_hop",
]
raw_group_cols = base_group_cols + ["seed"]


def _numeric_mean_or_nan(series):
    values = pd.to_numeric(series, errors="coerce")
    return float(values.mean()) if values.notna().any() else np.nan


run_group_sizes = edge_stats_df.groupby(raw_group_cols, dropna=False).size()
label_group_summary_rows = []
for group_key, group in edge_stats_df.groupby(raw_group_cols + ["label_group"], dropna=False):
    if not isinstance(group_key, tuple):
        group_key = (group_key,)
    group_meta = dict(zip(raw_group_cols + ["label_group"], group_key))
    run_key = tuple(group_meta[column] for column in raw_group_cols)
    run_total = int(run_group_sizes.loc[run_key])
    n_nodes = int(selected_runs[0]["context"]["n_nodes"])

    summary = {
        **group_meta,
        "label_name": str(group["label_name"].iloc[0]),
        "n_candidate_edges": int(len(group)),
        "candidate_fraction": float(len(group) / max(1, run_total)),
        "score_raw_mean": _numeric_mean_or_nan(group["score_raw"]),
        "score_raw_min": float(pd.to_numeric(group["score_raw"], errors="coerce").min()),
        "score_raw_max": float(pd.to_numeric(group["score_raw"], errors="coerce").max()),
        "score_percentile_mean": _numeric_mean_or_nan(group["score_percentile"]),
        "inclusion_count_mean": _numeric_mean_or_nan(group["inclusion_count"]),
        "addition_fraction": float((group["action"] == "add").mean()),
        "deletion_fraction": float((group["action"] == "delete").mean()),
        "prbcd_derived_fraction": float((group["candidate_source"] == "prbcd_derived").mean()),
        "same_true_label_fraction": float(group["same_true_label"].mean()),
        "same_clean_prediction_fraction": float(group["same_clean_prediction"].mean()),
        "both_clean_correct_fraction": float(group["both_endpoints_clean_correct"].mean()),
        "neither_clean_correct_fraction": float(group["neither_endpoint_clean_correct"].mean()),
        "both_hit_fraction": _numeric_mean_or_nan(group["both_hit"]),
        "exactly_one_hit_fraction": _numeric_mean_or_nan(group["exactly_one_hit"]),
        "partner_label_adoption_fraction": _numeric_mean_or_nan(
            group["either_adopts_partner_true_label"]
        ),
        "full_graph_fallback_fraction": _numeric_mean_or_nan(
            group["used_full_graph_fallback"]
        ),
    }
    summary.update(_endpoint_concentration(group, n_nodes))
    summary.update(_candidate_pair_network_summary(group, n_nodes))
    label_group_summary_rows.append(summary)

label_group_summary_raw_df = pd.DataFrame(label_group_summary_rows)

summary_metric_cols = [
    column for column in label_group_summary_raw_df.select_dtypes(include=[np.number]).columns
    if column not in {"seed", "label_group", "endpoint_mining_hop"}
]
label_group_summary_df = aggregate_over_seeds(
    label_group_summary_raw_df,
    base_group_cols + ["label_group", "label_name"],
    summary_metric_cols,
)

# ------------------------------------------------------------
# Numeric and categorical comparisons
# ------------------------------------------------------------

excluded_numeric = {
    "seed",
    "sample_index",
    "u",
    "v",
    "canonical_u",
    "canonical_v",
    "candidate_set_size",
    "endpoint_mining_hop",
    "label_group",
    "label_value",
    "score_raw",
    "score_norm",
    "score_percentile",
    "selection_rank",
    "u_true_label",
    "v_true_label",
    "u_clean_prediction",
    "v_clean_prediction",
    "u_pert_prediction",
    "v_pert_prediction",
}

numeric_comparison_raw_df = _numeric_comparison_table(
    edge_stats_df,
    raw_group_cols,
    excluded_numeric,
)

# Average the descriptive/effect-size outputs across victim seeds. P-values are
# retained in the raw table; seed-level inference appears separately below.
numeric_aggregate_metrics = [
    column for column in numeric_comparison_raw_df.select_dtypes(include=[np.number]).columns
    if column not in {"seed", "endpoint_mining_hop"}
]
numeric_comparison_df = aggregate_over_seeds(
    numeric_comparison_raw_df,
    base_group_cols + ["feature"],
    numeric_aggregate_metrics,
)

excluded_categorical = {
    "dataset",
    "candidate_config_id",
    "scoring_mode",
    "candidate_hash",
    "label_name",
    "canonical_u",
    "canonical_v",
}
categorical_comparison_raw_df, categorical_levels_raw_df = _categorical_comparison_tables(
    edge_stats_df,
    raw_group_cols,
    excluded_categorical,
)

categorical_aggregate_metrics = [
    column for column in categorical_comparison_raw_df.select_dtypes(include=[np.number]).columns
    if column not in {"seed", "endpoint_mining_hop"}
]
categorical_comparison_df = aggregate_over_seeds(
    categorical_comparison_raw_df,
    base_group_cols + ["feature"],
    categorical_aggregate_metrics,
)

categorical_level_metric_cols = [
    "label_1_count",
    "label_1_fraction",
    "label_0_count",
    "label_0_fraction",
    "fraction_difference_1_minus_0",
]
categorical_levels_df = aggregate_over_seeds(
    categorical_levels_raw_df,
    base_group_cols + ["feature", "level"],
    categorical_level_metric_cols,
)

numeric_feature_names = sorted(set(numeric_comparison_raw_df["feature"]))
seed_level_comparison_df = _seed_level_paired_differences(
    edge_stats_df,
    base_group_cols,
    numeric_feature_names,
)

# ------------------------------------------------------------
# Specialized tables that are useful for interpretation
# ------------------------------------------------------------

class_pair_table_raw_df = (
    edge_stats_df.groupby(raw_group_cols + ["canonical_true_label_pair", "label_group"], dropna=False)
    .size()
    .rename("count")
    .reset_index()
)
class_pair_totals = (
    class_pair_table_raw_df.groupby(raw_group_cols + ["canonical_true_label_pair"], dropna=False)["count"]
    .sum()
    .rename("pair_total")
    .reset_index()
)
class_pair_table_raw_df = class_pair_table_raw_df.merge(
    class_pair_totals,
    on=raw_group_cols + ["canonical_true_label_pair"],
    how="left",
)
class_pair_table_raw_df["within_pair_label_fraction"] = (
    class_pair_table_raw_df["count"] / class_pair_table_raw_df["pair_total"].clip(lower=1)
)

source_action_table_raw_df = (
    edge_stats_df.groupby(raw_group_cols + ["candidate_source", "action", "label_group"], dropna=False)
    .size()
    .rename("count")
    .reset_index()
)

# Endpoint-hit patterns exist only for endpoint mining. Subset scoring evaluates
# simultaneous subsets and therefore has no per-edge perturbed endpoint state.
_endpoint_only = edge_stats_df[edge_stats_df["scoring_mode"] == "endpoint"].copy()
if _endpoint_only.empty:
    endpoint_hit_pattern_raw_df = pd.DataFrame()
else:
    endpoint_hit_pattern_raw_df = (
        _endpoint_only.assign(
            hit_pattern=np.select(
                [
                    _endpoint_only["both_hit"].fillna(False).astype(bool),
                    _endpoint_only["u_hit"].fillna(False).astype(bool)
                    & ~_endpoint_only["v_hit"].fillna(False).astype(bool),
                    ~_endpoint_only["u_hit"].fillna(False).astype(bool)
                    & _endpoint_only["v_hit"].fillna(False).astype(bool),
                ],
                ["both", "u_only", "v_only"],
                default="neither",
            )
        )
        .groupby(raw_group_cols + ["label_group", "hit_pattern"], dropna=False)
        .size()
        .rename("count")
        .reset_index()
    )

# Node-level involvement: compare which nodes participate in group 1 versus
# group 0 candidate pairs and how frequently they occur.
node_rows = []
for group_key, group in edge_stats_df.groupby(
    raw_group_cols + ["label_group"], dropna=False
):
    if not isinstance(group_key, tuple):
        group_key = (group_key,)
    group_meta = dict(zip(raw_group_cols + ["label_group"], group_key))
    config_id = group_meta["candidate_config_id"]
    scoring_mode = group_meta["scoring_mode"]
    comparison_rule = group_meta["comparison_rule"]
    hop = int(group_meta["endpoint_mining_hop"])
    seed = int(group_meta["seed"])
    label_group = int(group_meta["label_group"])

    counts = Counter(group["u"].tolist() + group["v"].tolist())
    matching_run = next(
        run for run in selected_runs
        if int(run["seed"]) == seed
        and str(run["scoring_mode"]) == scoring_mode
        and _rq2_endpoint_hop(run) == hop
        and run["candidate_config_id"] == config_id
    )
    context = matching_run["context"]
    edge_index_cpu = _to_cpu_tensor(context["edge_index"], dtype=torch.long)
    topology_signature = (
        int(context["n_nodes"]),
        int(edge_index_cpu.size(1)),
        int(edge_index_cpu[:, : min(100, edge_index_cpu.size(1))].sum().item()),
    )
    clean = clean_graph_cache[topology_signature]
    prediction_stats = prediction_cache[
        (seed, scoring_mode, hop, matching_run["candidate_hash"])
    ]

    for node, incidence in counts.items():
        node_rows.append({
            "candidate_config_id": config_id,
            "scoring_mode": scoring_mode,
            "comparison_rule": comparison_rule,
            "endpoint_mining_hop": hop,
            "seed": seed,
            "label_group": label_group,
            "label_name": str(group["label_name"].iloc[0]),
            "node": int(node),
            "candidate_incidence": int(incidence),
            "degree": float(clean["degree"][node]),
            "clustering": float(clean["clustering"][node]),
            "triangles": float(clean["triangles"][node]),
            "core_number": float(clean["core_number"][node]),
            "pagerank": float(clean["pagerank"][node]),
            "closeness": float(clean["closeness"][node]),
            "betweenness": float(clean["betweenness"][node]),
            "eigenvector": float(clean["eigenvector"][node]),
            "feature_nnz": float(clean["feature_nnz"][node]),
            "feature_norm_l2": float(clean["feature_norm_l2"][node]),
            "clean_confidence": float(prediction_stats["clean_confidence"][node]),
            "clean_margin": float(prediction_stats["clean_margin"][node]),
            "clean_entropy": float(prediction_stats["clean_entropy"][node]),
            "clean_correct": bool(prediction_stats["clean_correct"][node]),
            "true_label": int(clean["labels"][node]),
            "split": str(clean["split_name"][node]),
        })
node_involvement_df = pd.DataFrame(node_rows)

# ------------------------------------------------------------
# Print everything requested for comparison
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)
if RQ2_GRAPH_STATS_PRINT_FULL_TABLES:
    pd.set_option("display.max_rows", None)

print("\n" + "=" * 100)
print("STATS INCLUDED IN THE MODE-AWARE GROUP-1 VS GROUP-0 COMPARISON")
print("=" * 100)
print("""
Group definitions
   - endpoint: group 1 = endpoint label 1; group 0 = endpoint label 0.
   - subset_accuracy_drop: group 1 = exact top score decile; group 0 = exact bottom score decile.
   - score columns are excluded from the feature tests to avoid a tautological comparison.

1. Candidate identity and provenance
   - PRBCD-derived versus random candidate; addition versus deletion; bridge status.
2. Endpoint ground truth and victim state
   - true classes, clean predictions, clean correctness, confidence, entropy,
     true-class probability, classification margin, endpoint hit pattern, and
     adoption of the partner endpoint's true class.
3. Endpoint node structure
   - degree, clustering coefficient, triangle count, core number, PageRank,
     closeness, betweenness, eigenvector centrality, component size, and class frequency.
4. Pairwise topology
   - common neighbors, neighborhood union, Jaccard, Salton, Sørensen,
     hub-promoted, hub-depressed, Leicht-Holme-Newman, Adamic-Adar,
     resource allocation, preferential attachment, clean shortest path,
     inverse path length, two-hop reachability, and component membership.
5. Node attributes and pair similarity
   - feature L1/L2 norms, feature nonzero count, dot product, cosine similarity,
     L1/L2 distance, support intersection/union, and feature-support Jaccard.
6. Mining mechanics
   - h-hop neighborhood size and full-graph fallback use.
7. Set-level organization
   - number and coverage of endpoint nodes, candidate-incidence concentration,
     Gini, HHI, top-node shares, and candidate-pair-network density/components.
8. Statistical comparisons
   - complete descriptive summaries, mean/median differences, ratios,
     Cohen's d, Hedges' g, rank-biserial correlation, point-biserial correlation,
     Mann-Whitney U, KS tests, chi-square/Fisher tests, Cramér's V, FDR-adjusted
     p-values, and paired seed-level differences where multiple seeds exist.
""")

print("\nCOMPARISON-SELECTION DIAGNOSTICS")
display(comparison_selection_df.sort_values(["scoring_mode", "endpoint_mining_hop", "seed"]))

print("\nCLEAN GRAPH SUMMARY")
display(graph_summary_df)

print("\nCOMPARISON-GROUP / SET-LEVEL SUMMARY — RAW PER SEED")
display(label_group_summary_raw_df.sort_values(raw_group_cols + ["label_group"]))

print("\nCOMPARISON-GROUP / SET-LEVEL SUMMARY — AGGREGATED ACROSS SEEDS")
display(label_group_summary_df.sort_values(base_group_cols + ["label_group"]))

print("\nNUMERIC COMPARISONS — SORTED BY ABSOLUTE HEDGES' G")
_numeric_display = numeric_comparison_df.copy()
if "hedges_g_mean" in _numeric_display.columns:
    _numeric_display["abs_hedges_g_mean"] = _numeric_display["hedges_g_mean"].abs()
    _numeric_display = _numeric_display.sort_values(
        base_group_cols + ["abs_hedges_g_mean"],
        ascending=[True] * len(base_group_cols) + [False],
    )
display(_numeric_display)

print("\nCATEGORICAL COMPARISONS — SORTED BY CRAMÉR'S V")
_categorical_display = categorical_comparison_df.copy()
if "cramers_v_mean" in _categorical_display.columns:
    _categorical_display = _categorical_display.sort_values(
        base_group_cols + ["cramers_v_mean"],
        ascending=[True] * len(base_group_cols) + [False],
    )
display(_categorical_display)

print("\nCATEGORICAL LEVEL PROPORTIONS")
display(
    categorical_levels_df.sort_values(
        base_group_cols + ["feature", "fraction_difference_1_minus_0_mean"],
        ascending=[True] * len(base_group_cols) + [True, False],
    )
)

print("\nPAIRED SEED-LEVEL GROUP-1 MINUS GROUP-0 DIFFERENCES")
display(
    seed_level_comparison_df.sort_values(
        base_group_cols + ["seed_mean_difference_1_minus_0"],
        ascending=[True] * len(base_group_cols) + [False],
    )
)

print("\nTRUE-CLASS PAIR COUNTS")
display(
    class_pair_table_raw_df.sort_values(
        raw_group_cols + ["label_group", "count"],
        ascending=[True] * len(raw_group_cols) + [True, False],
    )
)

print("\nCANDIDATE SOURCE × ACTION × COMPARISON GROUP")
display(source_action_table_raw_df.sort_values(raw_group_cols + ["candidate_source", "action", "label_group"]))

if not endpoint_hit_pattern_raw_df.empty:
    print("\nENDPOINT HIT PATTERNS — ENDPOINT MODE ONLY")
    display(endpoint_hit_pattern_raw_df.sort_values(raw_group_cols + ["label_group", "hit_pattern"]))

print("\nNODE INVOLVEMENT TABLE — TOP 50 BY CANDIDATE INCIDENCE")
display(
    node_involvement_df.sort_values(
        raw_group_cols + ["label_group", "candidate_incidence"],
        ascending=[True] * len(raw_group_cols) + [True, False],
    ).groupby(raw_group_cols + ["label_group"], dropna=False).head(50)
)

# ------------------------------------------------------------
# Plots: largest standardized numeric and categorical differences
# ------------------------------------------------------------

for config_key, frame in numeric_comparison_df.groupby(base_group_cols, dropna=False):
    if "hedges_g_mean" not in frame:
        continue
    plotted = (
        frame.assign(abs_effect=frame["hedges_g_mean"].abs())
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["hedges_g_mean"])
        .nlargest(RQ2_GRAPH_STATS_TOP_PLOT_FEATURES, "abs_effect")
        .sort_values("hedges_g_mean")
    )
    if plotted.empty:
        continue
    fig, ax = plt.subplots(figsize=(11, max(6, 0.32 * len(plotted))))
    ax.barh(plotted["feature"], plotted["hedges_g_mean"])
    ax.axvline(0.0, linewidth=0.9)
    ax.set_xlabel("Hedges' g: group 1 minus group 0")
    ax.set_title(
        "Largest standardized numeric differences"
    )
    ax.grid(True, axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig(
        RUN_PLOTS_DIR
        / f"numeric_effects__{_rq2_stats_safe('__'.join(map(str, config_key)))}.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()

for config_key, frame in categorical_comparison_df.groupby(base_group_cols, dropna=False):
    if "cramers_v_mean" not in frame:
        continue
    plotted = (
        frame.replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["cramers_v_mean"])
        .nlargest(RQ2_GRAPH_STATS_TOP_PLOT_FEATURES, "cramers_v_mean")
        .sort_values("cramers_v_mean")
    )
    if plotted.empty:
        continue
    fig, ax = plt.subplots(figsize=(11, max(5, 0.34 * len(plotted))))
    ax.barh(plotted["feature"], plotted["cramers_v_mean"])
    ax.set_xlabel("Cramér's V")
    ax.set_title(
        "Largest categorical associations"
    )
    ax.set_xlim(left=0)
    ax.grid(True, axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig(
        RUN_PLOTS_DIR
        / f"categorical_effects__{_rq2_stats_safe('__'.join(map(str, config_key)))}.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()

# A focused prevalence chart for the most interpretable binary/categorical variables.
focused_features = [
    "action",
    "candidate_source",
    "same_true_label",
    "same_clean_prediction",
    "n_clean_correct_endpoints",
    "used_full_graph_fallback",
    "has_two_hop_path",
    "same_connected_component",
    "canonical_split_pair",
]
for config_key, levels_frame in categorical_levels_df.groupby(base_group_cols, dropna=False):
    plotted = levels_frame[levels_frame["feature"].isin(focused_features)].copy()
    if plotted.empty:
        continue
    plotted["display_name"] = plotted["feature"].astype(str) + " = " + plotted["level"].astype(str)
    plotted = plotted.sort_values("fraction_difference_1_minus_0_mean")
    fig, ax = plt.subplots(figsize=(12, max(6, 0.28 * len(plotted))))
    ax.barh(plotted["display_name"], plotted["fraction_difference_1_minus_0_mean"])
    ax.axvline(0.0, linewidth=0.9)
    ax.set_xlabel("Proportion difference: group 1 minus group 0")
    ax.set_title(
        "Selected categorical prevalence differences"
    )
    ax.grid(True, axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig(
        RUN_PLOTS_DIR
        / f"categorical_prevalence__{_rq2_stats_safe('__'.join(map(str, config_key)))}.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()

# ------------------------------------------------------------
# Save every table
# ------------------------------------------------------------

edge_stats_df.to_csv(RQ2_GRAPH_STATS_OUT_DIR / "edge_level_stats_raw.csv", index=False)
comparison_selection_df.to_csv(RQ2_GRAPH_STATS_OUT_DIR / "comparison_selection_diagnostics.csv", index=False)
graph_summary_df.to_csv(RQ2_GRAPH_STATS_OUT_DIR / "clean_graph_summary.csv", index=False)
label_group_summary_raw_df.to_csv(RQ2_GRAPH_STATS_OUT_DIR / "label_group_summary_raw.csv", index=False)
label_group_summary_df.to_csv(RQ2_GRAPH_STATS_OUT_DIR / "label_group_summary_averaged.csv", index=False)
numeric_comparison_raw_df.to_csv(RQ2_GRAPH_STATS_OUT_DIR / "numeric_comparisons_raw_per_seed.csv", index=False)
numeric_comparison_df.to_csv(RQ2_GRAPH_STATS_OUT_DIR / "numeric_comparisons_averaged.csv", index=False)
categorical_comparison_raw_df.to_csv(RQ2_GRAPH_STATS_OUT_DIR / "categorical_comparisons_raw_per_seed.csv", index=False)
categorical_comparison_df.to_csv(RQ2_GRAPH_STATS_OUT_DIR / "categorical_comparisons_averaged.csv", index=False)
categorical_levels_raw_df.to_csv(RQ2_GRAPH_STATS_OUT_DIR / "categorical_level_proportions_raw_per_seed.csv", index=False)
categorical_levels_df.to_csv(RQ2_GRAPH_STATS_OUT_DIR / "categorical_level_proportions_averaged.csv", index=False)
seed_level_comparison_df.to_csv(RQ2_GRAPH_STATS_OUT_DIR / "paired_seed_level_differences.csv", index=False)
class_pair_table_raw_df.to_csv(RQ2_GRAPH_STATS_OUT_DIR / "true_class_pair_counts.csv", index=False)
source_action_table_raw_df.to_csv(RQ2_GRAPH_STATS_OUT_DIR / "candidate_source_action_counts.csv", index=False)
if not endpoint_hit_pattern_raw_df.empty:
    endpoint_hit_pattern_raw_df.to_csv(RQ2_GRAPH_STATS_OUT_DIR / "endpoint_hit_patterns.csv", index=False)
node_involvement_df.to_csv(RQ2_GRAPH_STATS_OUT_DIR / "node_involvement_raw.csv", index=False)

experiment_config = {
    "dataset": DATASET,
    "candidate_config_id": selected_config_id,
    "candidate_set_size": selected_candidate_size,
    "scoring_modes": sorted({str(run["scoring_mode"]) for run in selected_runs}),
    "endpoint_hops": sorted({
        _rq2_endpoint_hop(run)
        for run in selected_runs
        if str(run["scoring_mode"]) == "endpoint"
    }),
    "victim_seeds": sorted({int(run["seed"]) for run in selected_runs}),
    "endpoint_label_threshold": RQ2_GRAPH_STATS_LABEL_THRESHOLD,
    "subset_extreme_fraction": RQ2_GRAPH_STATS_SUBSET_EXTREME_FRACTION,
    "subset_group_definition": "exact equal-sized top and bottom score_raw fractions within each run",
    "score_columns_excluded_from_feature_tests": [
        "label_value", "score_raw", "score_norm", "score_percentile", "selection_rank"
    ],
    "compute_exact_betweenness": RQ2_GRAPH_STATS_COMPUTE_EXACT_BETWEENNESS,
    "compute_all_pairs_distance": RQ2_GRAPH_STATS_COMPUTE_ALL_PAIRS_DISTANCE,
    "n_edge_rows": int(len(edge_stats_df)),
    "numeric_features_compared": numeric_feature_names,
}
(RQ2_GRAPH_STATS_OUT_DIR / "experiment_config.json").write_text(
    json.dumps(experiment_config, indent=2),
    encoding="utf-8",
)

print("\n" + "=" * 100)
print("COMPLETED")
print("=" * 100)
print("Edge-level rows used in comparisons:", len(edge_stats_df))
print("Selection-diagnostic rows:", len(comparison_selection_df))
print("Numeric features compared:", len(numeric_feature_names))
print("Categorical features compared:", categorical_comparison_raw_df["feature"].nunique() if not categorical_comparison_raw_df.empty else 0)
print("All raw tables, aggregated tables, and plots saved to:")
print(RQ2_GRAPH_STATS_OUT_DIR.resolve())


In [ ]:
##%%
# ============================================================
# RQ2: Mode-aware contextual output and plotting
#
# endpoint:
#     group 1 = endpoint label 1
#     group 0 = endpoint label 0
#
# subset_accuracy_drop:
#     group 1 = top score decile
#     group 0 = bottom score decile
#
# Run after the mode-aware graph-statistics cell.
# ============================================================

from pathlib import Path
import json
import math
import re
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

RQ2_CONTEXT_TOP_NUMERIC = 30
RQ2_CONTEXT_TOP_CATEGORICAL = 25
RQ2_CONTEXT_DISTRIBUTION_FEATURES = 12
RQ2_CONTEXT_EFFECTS_PER_PAGE = 32
RQ2_CONTEXT_TOP_NODES = 25
RQ2_CONTEXT_ALPHA = 0.05
RQ2_CONTEXT_SAVE_DPI = 220
RQ2_CONTEXT_PRINT_FULL_TABLES = True
RQ2_CONTEXT_SHOW_ALL_FIGURES = True
RQ2_CONTEXT_BASE_DIR = Path("extendedPlotting") / "rq2_label_graph_stats"

RQ2_CONTEXT_BINNED_FEATURES = [
    "pair_degree_mean",
    "pair_degree_min",
    "common_neighbors",
    "jaccard_coefficient",
    "adamic_adar_index",
    "clean_shortest_path_length",
    "feature_cosine_similarity",
    "feature_support_jaccard",
    "pair_clean_confidence_min",
    "pair_clean_margin_min",
    "pair_clean_entropy_mean",
    "neighborhood_size",
]

RQ2_CONTEXT_CATEGORICAL_FEATURES = [
    "action",
    "candidate_source",
    "same_true_label",
    "same_clean_prediction",
    "n_clean_correct_endpoints",
    "both_endpoints_clean_correct",
    "neither_endpoint_clean_correct",
    "same_connected_component",
    "has_two_hop_path",
    "used_full_graph_fallback",
    "canonical_split_pair",
    "canonical_true_label_pair",
]

RQ2_CONTEXT_SET_METRIC_GROUPS = {
    "Candidate composition": [
        "candidate_fraction",
        "addition_fraction",
        "deletion_fraction",
        "prbcd_derived_fraction",
    ],
    "Clean endpoint state": [
        "same_true_label_fraction",
        "same_clean_prediction_fraction",
        "both_clean_correct_fraction",
        "neither_clean_correct_fraction",
    ],
    "Endpoint-hit mechanism (endpoint mode only)": [
        "both_hit_fraction",
        "exactly_one_hit_fraction",
        "partner_label_adoption_fraction",
        "full_graph_fallback_fraction",
    ],
    "Endpoint-node coverage and concentration": [
        "endpoint_node_coverage",
        "candidate_incidence_gini",
        "candidate_incidence_hhi",
        "top_1pct_endpoint_share",
        "top_5pct_endpoint_share",
        "top_10pct_endpoint_share",
    ],
    "Candidate-pair network": [
        "candidate_pair_network_density",
        "candidate_pair_network_largest_component_fraction",
        "candidate_pair_network_average_clustering",
        "candidate_pair_network_transitivity",
    ],
}


# ------------------------------------------------------------
# Load outputs when run after a restart
# ------------------------------------------------------------

def _rq2_context_safe(value):
    return re.sub(r"[^a-zA-Z0-9_.=-]+", "_", str(value)).strip("_")


def _find_latest_stats_directory(base_dir):
    base_dir = Path(base_dir)
    if not base_dir.exists():
        raise FileNotFoundError(
            f"No graph-statistics output directory exists at {base_dir.resolve()}. "
            "Run the mode-aware statistics cell first."
        )

    candidates = [
        path
        for path in base_dir.iterdir()
        if path.is_dir() and (path / "edge_level_stats_raw.csv").exists()
    ]

    if not candidates:
        raise FileNotFoundError(
            f"No completed graph-statistics run was found below {base_dir.resolve()}."
        )

    return max(
        candidates,
        key=lambda path: (path.stat().st_mtime, path.name),
    )


if "RQ2_GRAPH_STATS_OUT_DIR" not in globals():
    RQ2_GRAPH_STATS_OUT_DIR = _find_latest_stats_directory(
        RQ2_CONTEXT_BASE_DIR
    )
else:
    RQ2_GRAPH_STATS_OUT_DIR = Path(RQ2_GRAPH_STATS_OUT_DIR)


def _load_dataframe_if_missing(
    variable_name,
    filename,
    *,
    required=True,
):
    if (
        variable_name in globals()
        and isinstance(globals()[variable_name], pd.DataFrame)
    ):
        return globals()[variable_name]

    path = RQ2_GRAPH_STATS_OUT_DIR / filename

    if not path.exists():
        if required:
            raise FileNotFoundError(
                f"Required table is missing: {path.resolve()}"
            )
        frame = pd.DataFrame()
        globals()[variable_name] = frame
        return frame

    frame = pd.read_csv(path)
    globals()[variable_name] = frame
    return frame


edge_stats_df = _load_dataframe_if_missing(
    "edge_stats_df",
    "edge_level_stats_raw.csv",
)
comparison_selection_df = _load_dataframe_if_missing(
    "comparison_selection_df",
    "comparison_selection_diagnostics.csv",
    required=False,
)
graph_summary_df = _load_dataframe_if_missing(
    "graph_summary_df",
    "clean_graph_summary.csv",
)
label_group_summary_raw_df = _load_dataframe_if_missing(
    "label_group_summary_raw_df",
    "label_group_summary_raw.csv",
)
label_group_summary_df = _load_dataframe_if_missing(
    "label_group_summary_df",
    "label_group_summary_averaged.csv",
)
numeric_comparison_raw_df = _load_dataframe_if_missing(
    "numeric_comparison_raw_df",
    "numeric_comparisons_raw_per_seed.csv",
)
numeric_comparison_df = _load_dataframe_if_missing(
    "numeric_comparison_df",
    "numeric_comparisons_averaged.csv",
)
categorical_comparison_raw_df = _load_dataframe_if_missing(
    "categorical_comparison_raw_df",
    "categorical_comparisons_raw_per_seed.csv",
)
categorical_comparison_df = _load_dataframe_if_missing(
    "categorical_comparison_df",
    "categorical_comparisons_averaged.csv",
)
categorical_levels_raw_df = _load_dataframe_if_missing(
    "categorical_levels_raw_df",
    "categorical_level_proportions_raw_per_seed.csv",
)
categorical_levels_df = _load_dataframe_if_missing(
    "categorical_levels_df",
    "categorical_level_proportions_averaged.csv",
)
seed_level_comparison_df = _load_dataframe_if_missing(
    "seed_level_comparison_df",
    "paired_seed_level_differences.csv",
)
class_pair_table_raw_df = _load_dataframe_if_missing(
    "class_pair_table_raw_df",
    "true_class_pair_counts.csv",
)
source_action_table_raw_df = _load_dataframe_if_missing(
    "source_action_table_raw_df",
    "candidate_source_action_counts.csv",
)
endpoint_hit_pattern_raw_df = _load_dataframe_if_missing(
    "endpoint_hit_pattern_raw_df",
    "endpoint_hit_patterns.csv",
    required=False,
)
node_involvement_df = _load_dataframe_if_missing(
    "node_involvement_df",
    "node_involvement_raw.csv",
)

required_mode_columns = {
    "scoring_mode",
    "comparison_rule",
    "endpoint_mining_hop",
}
missing_mode_columns = required_mode_columns - set(edge_stats_df.columns)
if missing_mode_columns:
    raise RuntimeError(
        "The loaded edge-level table was produced by the old endpoint-only "
        "statistics cell. Missing columns: "
        f"{sorted(missing_mode_columns)}. Run the mode-aware statistics cell first."
    )


RQ2_CONTEXT_PLOT_DIR = RUN_PLOTS_DIR
RQ2_CONTEXT_TABLE_DIR = (
    RQ2_GRAPH_STATS_OUT_DIR / "contextual_tables"
)

RQ2_CONTEXT_PLOT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
RQ2_CONTEXT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)
if RQ2_CONTEXT_PRINT_FULL_TABLES:
    pd.set_option("display.max_rows", None)


# ------------------------------------------------------------
# Mode-aware naming and interpretation
# ------------------------------------------------------------

def _subset_fraction_text(comparison_rule):
    match = re.search(
        r"subset_top_vs_bottom_([0-9.]+)%",
        str(comparison_rule),
    )
    return f"{match.group(1)}%" if match else "selected extreme"


def _mode_info(scoring_mode, comparison_rule):
    scoring_mode = str(scoring_mode)

    if scoring_mode == "endpoint":
        return {
            "mode_title": "Endpoint mining",
            "group_1": "Endpoint label 1 (harmful)",
            "group_0": "Endpoint label 0 (no endpoint hit)",
            "short_1": "endpoint label 1",
            "short_0": "endpoint label 0",
            "contrast": "endpoint label 1 minus endpoint label 0",
            "group_1_rate": "endpoint-label-1 rate",
            "main_context": (
                "Each candidate edge was flipped individually. Group 1 means "
                "that at least one endpoint changed from a correct clean "
                "prediction to an incorrect perturbed prediction. Group 0 "
                "means that this endpoint criterion was not met."
            ),
        }

    if scoring_mode == "subset_accuracy_drop":
        fraction_text = _subset_fraction_text(comparison_rule)
        return {
            "mode_title": "Subset-accuracy-drop mining",
            "group_1": f"Top {fraction_text} subset score",
            "group_0": f"Bottom {fraction_text} subset score",
            "short_1": f"top {fraction_text}",
            "short_0": f"bottom {fraction_text}",
            "contrast": f"top {fraction_text} minus bottom {fraction_text}",
            "group_1_rate": f"top-{fraction_text} group fraction",
            "main_context": (
                f"Only the equal-sized top and bottom {fraction_text} of "
                "observed raw subset scores are compared within each mining "
                "run. The middle candidates are excluded. The score is the "
                "mean harmful accuracy drop among sampled subsets containing "
                "the edge; it is an edge-associated subset score, not an "
                "individual causal effect."
            ),
        }

    raise ValueError(f"Unsupported scoring mode: {scoring_mode!r}")


def _label_name(label_group, scoring_mode, comparison_rule):
    info = _mode_info(scoring_mode, comparison_rule)
    return info["group_1"] if int(label_group) == 1 else info["group_0"]


def _pretty_feature(name):
    replacements = {
        "prbcd": "PRBCD",
        "hhi": "HHI",
        "l1": "L1",
        "l2": "L2",
        "nnz": "nonzero features",
    }
    text = (
        str(name)
        .replace("pair_", "endpoint-pair ")
        .replace("_", " ")
    )
    words = [
        replacements.get(word, word)
        for word in text.split()
    ]
    return " ".join(words)


base_group_cols = [
    "candidate_config_id",
    "scoring_mode",
    "comparison_rule",
    "endpoint_mining_hop",
]
raw_group_cols = base_group_cols + ["seed"]


def _config_meta(config_key):
    if not isinstance(config_key, tuple):
        config_key = (config_key,)
    return dict(zip(base_group_cols, config_key))


def _config_title(config_key):
    meta = _config_meta(config_key)
    hop_text = (
        f"h={int(meta['endpoint_mining_hop'])}"
        if str(meta["scoring_mode"]) == "endpoint"
        else "subset scoring"
    )
    return (
        f"configuration={meta['candidate_config_id']} | "
        f"mode={meta['scoring_mode']} | {hop_text}"
    )


def _effect_label(value):
    value = abs(float(value))
    if value < 0.2:
        return "negligible"
    if value < 0.5:
        return "small"
    if value < 0.8:
        return "medium"
    return "large"


def _association_label(value):
    value = abs(float(value))
    if value < 0.1:
        return "weak"
    if value < 0.3:
        return "small-to-moderate"
    if value < 0.5:
        return "moderate-to-strong"
    return "strong"


def _mean_sd_sem(values):
    values = (
        pd.to_numeric(
            pd.Series(values),
            errors="coerce",
        )
        .dropna()
        .to_numpy(dtype=float)
    )
    n = int(values.size)
    if n == 0:
        return np.nan, np.nan, np.nan, 0

    mean = float(values.mean())
    sd = float(values.std(ddof=1)) if n > 1 else 0.0
    sem = sd / math.sqrt(n) if n > 1 else 0.0
    return mean, sd, sem, n


# Normalize Boolean columns after CSV reload.
for boolean_column in [
    "exists_clean",
    "same_true_label",
    "same_clean_prediction",
    "both_endpoints_clean_correct",
    "neither_endpoint_clean_correct",
    "same_connected_component",
    "has_two_hop_path",
    "used_full_graph_fallback",
    "u_hit",
    "v_hit",
    "both_hit",
    "exactly_one_hit",
]:
    if (
        boolean_column in edge_stats_df.columns
        and edge_stats_df[boolean_column].dtype == object
    ):
        edge_stats_df[boolean_column] = (
            edge_stats_df[boolean_column]
            .astype(str)
            .str.lower()
            .map({"true": True, "false": False})
        )


# ------------------------------------------------------------
# General output helpers
# ------------------------------------------------------------

plot_manifest_rows = []
context_report_lines = []


def _section(title, context=None):
    print("\n" + "=" * 112)
    print(title)
    print("=" * 112)
    context_report_lines.extend(
        ["", title, "-" * len(title)]
    )
    if context:
        wrapped = textwrap.fill(
            str(context),
            width=112,
        )
        print(wrapped)
        context_report_lines.append(str(context))


def _subsection(title, context=None):
    print("\n" + "-" * 112)
    print(title)
    print("-" * 112)
    context_report_lines.extend(["", title])
    if context:
        wrapped = textwrap.fill(
            str(context),
            width=112,
        )
        print(wrapped)
        context_report_lines.append(str(context))


def _display_and_save(
    frame,
    filename,
    *,
    max_rows=None,
    sort_by=None,
    ascending=True,
):
    output = frame.copy()

    if (
        sort_by is not None
        and all(
            column in output.columns
            for column in np.atleast_1d(sort_by)
        )
    ):
        output = output.sort_values(
            sort_by,
            ascending=ascending,
        )

    output.to_csv(
        RQ2_CONTEXT_TABLE_DIR / filename,
        index=False,
    )

    if max_rows is not None:
        display(output.head(max_rows))
        if len(output) > max_rows:
            print(
                f"Displayed {max_rows} of {len(output)} rows. "
                f"Full table saved as {filename}."
            )
    else:
        display(output)

    return output


def _save_figure(
    fig,
    filename,
    title,
    context,
    config_key=None,
):
    path = RQ2_CONTEXT_PLOT_DIR / filename
    fig.tight_layout()
    fig.savefig(
        path,
        dpi=RQ2_CONTEXT_SAVE_DPI,
        bbox_inches="tight",
    )

    manifest = {
        "title": title,
        "context": context,
        "candidate_config_id": None,
        "scoring_mode": None,
        "comparison_rule": None,
        "endpoint_mining_hop": None,
        "path": str(path.resolve()),
    }

    if config_key is not None:
        manifest.update(_config_meta(config_key))

    plot_manifest_rows.append(manifest)
    print("Saved plot:", path.resolve())

    if RQ2_CONTEXT_SHOW_ALL_FIGURES:
        plt.show()
    else:
        plt.close(fig)


# ------------------------------------------------------------
# Derived summaries
# ------------------------------------------------------------

numeric_effect_summary = (
    numeric_comparison_raw_df
    .groupby(
        base_group_cols + ["feature"],
        dropna=False,
    )
    .agg(
        n_seeds=("seed", "nunique"),
        group_1_mean=("label_1_mean", "mean"),
        group_0_mean=("label_0_mean", "mean"),
        mean_difference=(
            "mean_difference_1_minus_0",
            "mean",
        ),
        mean_difference_sd=(
            "mean_difference_1_minus_0",
            "std",
        ),
        hedges_g_mean=("hedges_g", "mean"),
        hedges_g_sd=("hedges_g", "std"),
        rank_biserial_mean=(
            "rank_biserial_correlation",
            "mean",
        ),
        point_biserial_mean=(
            "point_biserial_correlation",
            "mean",
        ),
        mann_whitney_fdr_significant_fraction=(
            "mann_whitney_p_fdr_bh",
            lambda values: float(
                (
                    pd.to_numeric(
                        values,
                        errors="coerce",
                    )
                    <= RQ2_CONTEXT_ALPHA
                ).mean()
            ),
        ),
    )
    .reset_index()
)

numeric_effect_summary["hedges_g_sd"] = (
    numeric_effect_summary["hedges_g_sd"]
    .fillna(0.0)
)
numeric_effect_summary["hedges_g_sem"] = (
    numeric_effect_summary["hedges_g_sd"]
    / np.sqrt(
        numeric_effect_summary["n_seeds"].clip(lower=1)
    )
)
numeric_effect_summary["abs_hedges_g"] = (
    numeric_effect_summary["hedges_g_mean"].abs()
)

direction_consistency = (
    numeric_comparison_raw_df
    .assign(
        effect_positive=lambda frame: (
            frame["hedges_g"] > 0
        )
    )
    .groupby(
        base_group_cols + ["feature"],
        dropna=False,
    )
    .agg(
        fraction_positive_direction=(
            "effect_positive",
            "mean",
        ),
        n_finite_effects=(
            "hedges_g",
            lambda values: int(
                pd.to_numeric(
                    values,
                    errors="coerce",
                ).notna().sum()
            ),
        ),
    )
    .reset_index()
)

direction_consistency["direction_consistency"] = np.maximum(
    direction_consistency["fraction_positive_direction"],
    1.0
    - direction_consistency["fraction_positive_direction"],
)

numeric_effect_summary = numeric_effect_summary.merge(
    direction_consistency,
    on=base_group_cols + ["feature"],
    how="left",
)

categorical_effect_summary = (
    categorical_comparison_raw_df
    .groupby(
        base_group_cols + ["feature"],
        dropna=False,
    )
    .agg(
        n_seeds=("seed", "nunique"),
        n_levels_mean=("n_levels", "mean"),
        cramers_v_mean=("cramers_v", "mean"),
        cramers_v_sd=("cramers_v", "std"),
        chi_square_fdr_significant_fraction=(
            "chi_square_p_fdr_bh",
            lambda values: float(
                (
                    pd.to_numeric(
                        values,
                        errors="coerce",
                    )
                    <= RQ2_CONTEXT_ALPHA
                ).mean()
            ),
        ),
        fisher_fdr_significant_fraction=(
            "fisher_p_fdr_bh",
            lambda values: float(
                (
                    pd.to_numeric(
                        values,
                        errors="coerce",
                    )
                    <= RQ2_CONTEXT_ALPHA
                ).mean()
            ),
        ),
    )
    .reset_index()
)

categorical_effect_summary["cramers_v_sd"] = (
    categorical_effect_summary["cramers_v_sd"]
    .fillna(0.0)
)
categorical_effect_summary["cramers_v_sem"] = (
    categorical_effect_summary["cramers_v_sd"]
    / np.sqrt(
        categorical_effect_summary["n_seeds"].clip(lower=1)
    )
)

numeric_effect_summary.to_csv(
    RQ2_CONTEXT_TABLE_DIR
    / "numeric_effect_summary_contextual.csv",
    index=False,
)
categorical_effect_summary.to_csv(
    RQ2_CONTEXT_TABLE_DIR
    / "categorical_effect_summary_contextual.csv",
    index=False,
)


# ------------------------------------------------------------
# Experiment context
# ------------------------------------------------------------

_section(
    "RQ2 mode-aware graph-statistics output",
    (
        "Endpoint runs retain the original binary label-1 versus label-0 "
        "comparison. Subset-accuracy-drop runs compare equal-sized top and "
        "bottom score groups within every seed/run. Results are always "
        "separated by scoring mode and comparison rule."
    ),
)

print(
    "Loaded statistics directory:",
    RQ2_GRAPH_STATS_OUT_DIR.resolve(),
)
print(
    "Contextual figures directory:",
    RQ2_CONTEXT_PLOT_DIR.resolve(),
)
print(
    "Contextual tables directory:",
    RQ2_CONTEXT_TABLE_DIR.resolve(),
)
print(
    "Candidate configurations:",
    sorted(
        edge_stats_df[
            "candidate_config_id"
        ].astype(str).unique().tolist()
    ),
)
print(
    "Scoring modes:",
    sorted(
        edge_stats_df[
            "scoring_mode"
        ].astype(str).unique().tolist()
    ),
)
print(
    "Comparison rules:",
    sorted(
        edge_stats_df[
            "comparison_rule"
        ].astype(str).unique().tolist()
    ),
)
print(
    "Victim seeds:",
    sorted(
        pd.to_numeric(
            edge_stats_df["seed"]
        ).unique().tolist()
    ),
)
print(
    "Edge-level observations used in comparisons:",
    len(edge_stats_df),
)
print(
    "Important inferential note: candidate edges share endpoints and "
    "are not independent. Seed-level consistency and paired seed "
    "summaries are the primary robustness evidence."
)

_subsection(
    "Clean graph context",
    (
        "These quantities describe the clean graph. They provide scale "
        "for later endpoint and pair statistics but are not themselves "
        "group comparisons."
    ),
)
_display_and_save(
    graph_summary_df,
    "clean_graph_context.csv",
)


# ------------------------------------------------------------
# 1. Comparison-group construction and stability
# ------------------------------------------------------------

_section(
    "1. Comparison-group construction and seed stability",
    (
        "For endpoint runs, the group sizes reflect the naturally occurring "
        "binary endpoint-label prevalence. For subset runs, the top and "
        "bottom groups are deliberately equal-sized, so a 50/50 split among "
        "selected edges is expected by construction and is not a prevalence "
        "estimate."
    ),
)

if not comparison_selection_df.empty:
    _display_and_save(
        comparison_selection_df.sort_values(
            raw_group_cols
        ),
        "comparison_selection_diagnostics_contextual.csv",
    )

group_counts_per_seed = (
    edge_stats_df
    .groupby(
        raw_group_cols + ["label_group"],
        dropna=False,
    )
    .size()
    .rename("count")
    .reset_index()
)

group_totals_per_seed = (
    group_counts_per_seed
    .groupby(
        raw_group_cols,
        dropna=False,
    )["count"]
    .sum()
    .rename("total_selected")
    .reset_index()
)

group_counts_per_seed = group_counts_per_seed.merge(
    group_totals_per_seed,
    on=raw_group_cols,
    how="left",
)

group_counts_per_seed["selected_fraction"] = (
    group_counts_per_seed["count"]
    / group_counts_per_seed["total_selected"].clip(lower=1)
)

group_counts_per_seed["group_name"] = [
    _label_name(
        row.label_group,
        row.scoring_mode,
        row.comparison_rule,
    )
    for row in group_counts_per_seed.itertuples()
]

_display_and_save(
    group_counts_per_seed.sort_values(
        raw_group_cols + ["label_group"]
    ),
    "comparison_group_counts_per_seed.csv",
)

for config_key, frame in group_counts_per_seed.groupby(
    base_group_cols,
    dropna=False,
):
    meta = _config_meta(config_key)
    info = _mode_info(
        meta["scoring_mode"],
        meta["comparison_rule"],
    )

    if meta["scoring_mode"] == "endpoint":
        group_1 = frame[
            frame["label_group"] == 1
        ].sort_values("seed")

        mean_fraction, sd_fraction, _, n_seeds = _mean_sd_sem(
            group_1["selected_fraction"]
        )

        context = (
            f"{_config_title(config_key)}. The mean endpoint-label-1 "
            f"fraction is {mean_fraction:.3f} (SD={sd_fraction:.3f}) "
            f"across {n_seeds} victim seed(s)."
        )

        _subsection(
            "Endpoint-label-1 prevalence by victim seed",
            context,
        )
        display(
            group_1[
                [
                    "seed",
                    "count",
                    "total_selected",
                    "selected_fraction",
                ]
            ]
        )

        fig, ax = plt.subplots(
            figsize=(9, 4.8)
        )
        x = np.arange(len(group_1))
        ax.bar(
            x,
            group_1["selected_fraction"],
        )
        ax.axhline(
            mean_fraction,
            linestyle="--",
            linewidth=1.0,
            label="mean across seeds",
        )
        ax.set_xticks(x)
        ax.set_xticklabels(
            group_1["seed"].astype(str)
        )
        ax.set_xlabel("Victim-model seed")
        ax.set_ylabel(
            "Fraction of candidates with endpoint label 1"
        )
        ax.set_ylim(
            0,
            min(
                1.0,
                max(
                    0.05,
                    group_1["selected_fraction"].max()
                    * 1.2,
                ),
            ),
        )
        ax.set_title(
            "Endpoint-harmful candidate prevalence"
        )
        ax.grid(
            True,
            axis="y",
            alpha=0.3,
        )
        ax.legend()

        _save_figure(
            fig,
            "01_endpoint_prevalence__"
            + _rq2_context_safe(
                "__".join(map(str, config_key))
            )
            + ".png",
            "Endpoint-label-1 prevalence",
            context,
            config_key,
        )

    else:
        subset_edges = edge_stats_df[
            (edge_stats_df["candidate_config_id"] == meta["candidate_config_id"])
            & (edge_stats_df["scoring_mode"] == meta["scoring_mode"])
            & (edge_stats_df["comparison_rule"] == meta["comparison_rule"])
            & (
                edge_stats_df["endpoint_mining_hop"]
                == meta["endpoint_mining_hop"]
            )
        ]

        score_summary = (
            subset_edges
            .groupby(
                ["seed", "label_group"],
                dropna=False,
            )["score_raw"]
            .agg(["count", "mean", "std", "min", "max"])
            .reset_index()
        )
        score_summary["group_name"] = score_summary[
            "label_group"
        ].map(
            lambda value: _label_name(
                value,
                meta["scoring_mode"],
                meta["comparison_rule"],
            )
        )

        _subsection(
            "Subset top/bottom score separation",
            (
                f"{_config_title(config_key)}. Equal group counts are "
                "expected by construction. The relevant diagnostic is "
                "whether the raw score distributions are clearly separated "
                "and whether the cutoffs are stable across seeds."
            ),
        )
        display(score_summary)

        seeds = sorted(
            score_summary["seed"].unique()
        )
        x = np.arange(len(seeds))
        width = 0.38

        fig, ax = plt.subplots(
            figsize=(10, 5)
        )

        for offset_index, label_group in enumerate([0, 1]):
            group_frame = (
                score_summary[
                    score_summary["label_group"]
                    == label_group
                ]
                .set_index("seed")
                .reindex(seeds)
                .reset_index()
            )
            offset = (
                offset_index - 0.5
            ) * width
            ax.bar(
                x + offset,
                group_frame["mean"],
                width=width,
                yerr=group_frame["std"].fillna(0.0),
                capsize=3,
                label=_label_name(
                    label_group,
                    meta["scoring_mode"],
                    meta["comparison_rule"],
                ),
            )

        ax.set_xticks(x)
        ax.set_xticklabels(
            [str(seed) for seed in seeds]
        )
        ax.set_xlabel("Victim-model seed")
        ax.set_ylabel("Mean raw subset score")
        ax.set_title(
            "Top/bottom subset-score separation"
        )
        ax.grid(
            True,
            axis="y",
            alpha=0.3,
        )
        ax.legend()

        _save_figure(
            fig,
            "01_subset_score_separation__"
            + _rq2_context_safe(
                "__".join(map(str, config_key))
            )
            + ".png",
            "Subset top/bottom score separation",
            info["main_context"],
            config_key,
        )


# ------------------------------------------------------------
# 2. Set-level organization and concentration
# ------------------------------------------------------------

_section(
    "2. Set-level organization of the two comparison groups",
    (
        "These summaries compare the two complete edge sets. Endpoint-hit "
        "mechanism metrics are shown only when they contain data; they are "
        "undefined for subset scoring because subset mining does not produce "
        "one individually perturbed prediction per candidate edge."
    ),
)

_display_and_save(
    label_group_summary_raw_df.sort_values(
        raw_group_cols + ["label_group"]
    ),
    "set_level_summary_per_seed.csv",
)

for config_key, frame in label_group_summary_raw_df.groupby(
    base_group_cols,
    dropna=False,
):
    meta = _config_meta(config_key)
    info = _mode_info(
        meta["scoring_mode"],
        meta["comparison_rule"],
    )

    for metric_group_name, metrics in (
        RQ2_CONTEXT_SET_METRIC_GROUPS.items()
    ):
        available = []

        for metric in metrics:
            if metric not in frame.columns:
                continue

            numeric = pd.to_numeric(
                frame[metric],
                errors="coerce",
            )

            if numeric.notna().any():
                available.append(metric)

        if not available:
            continue

        rows = []

        for metric in available:
            for label_group, label_frame in frame.groupby(
                "label_group"
            ):
                mean, sd, sem, n = _mean_sd_sem(
                    label_frame[metric]
                )
                rows.append({
                    "metric": metric,
                    "metric_display": _pretty_feature(metric),
                    "label_group": int(label_group),
                    "group_name": _label_name(
                        label_group,
                        meta["scoring_mode"],
                        meta["comparison_rule"],
                    ),
                    "mean": mean,
                    "sd": sd,
                    "sem": sem,
                    "n_seeds": n,
                })

        summary = pd.DataFrame(rows)

        _subsection(
            metric_group_name,
            (
                f"{_config_title(config_key)}. Bars are means across "
                "victim seeds and error bars are ±1 SD. "
                f"{info['main_context']}"
            ),
        )
        display(summary)

        metrics_order = (
            summary["metric"]
            .drop_duplicates()
            .tolist()
        )
        x = np.arange(len(metrics_order))
        width = 0.38

        fig, ax = plt.subplots(
            figsize=(
                max(10, 1.25 * len(metrics_order)),
                5.2,
            )
        )

        for offset_index, label_group in enumerate([0, 1]):
            label_frame = (
                summary[
                    summary["label_group"]
                    == label_group
                ]
                .set_index("metric")
                .reindex(metrics_order)
                .reset_index()
            )

            offset = (
                offset_index - 0.5
            ) * width

            ax.bar(
                x + offset,
                label_frame["mean"],
                width=width,
                yerr=label_frame["sd"],
                capsize=3,
                label=_label_name(
                    label_group,
                    meta["scoring_mode"],
                    meta["comparison_rule"],
                ),
            )

        ax.set_xticks(x)
        ax.set_xticklabels(
            [
                _pretty_feature(metric)
                for metric in metrics_order
            ],
            rotation=25,
            ha="right",
        )
        ax.set_ylabel("Mean across victim seeds")
        ax.set_title(
            metric_group_name
        )
        ax.grid(
            True,
            axis="y",
            alpha=0.3,
        )
        ax.legend()

        _save_figure(
            fig,
            "02_set_"
            + _rq2_context_safe(metric_group_name)
            + "__"
            + _rq2_context_safe(
                "__".join(map(str, config_key))
            )
            + ".png",
            metric_group_name,
            (
                "Direct set-level comparison for "
                + _config_title(config_key)
            ),
            config_key,
        )


# ------------------------------------------------------------
# 3. Candidate source and flip action
# ------------------------------------------------------------

_section(
    "3. Candidate source and flip action",
    (
        "For endpoint mode, the conditional group-1 rate is the endpoint-"
        "harmful yield. For subset mode, it is the fraction of selected "
        "extreme candidates belonging to the top-score group rather than "
        "the bottom-score group. Because subset groups are balanced overall, "
        "values above 0.5 indicate enrichment in the top-score extreme."
    ),
)


def _category_group_1_rate_table(
    frame,
    category_columns,
):
    rows = []
    group_columns = (
        base_group_cols
        + ["seed"]
        + category_columns
    )

    for group_key, group in frame.groupby(
        group_columns,
        dropna=False,
    ):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        meta = dict(
            zip(group_columns, group_key)
        )

        rows.append({
            **meta,
            "n_candidates": int(len(group)),
            "n_group_1": int(
                group["label_group"].sum()
            ),
            "group_1_rate": float(
                group["label_group"].mean()
            ),
        })

    raw = pd.DataFrame(rows)

    aggregate = (
        raw.groupby(
            base_group_cols + category_columns,
            dropna=False,
        )
        .agg(
            n_seeds=("seed", "nunique"),
            n_candidates_mean=(
                "n_candidates",
                "mean",
            ),
            group_1_rate_mean=(
                "group_1_rate",
                "mean",
            ),
            group_1_rate_sd=(
                "group_1_rate",
                "std",
            ),
        )
        .reset_index()
    )

    aggregate["group_1_rate_sd"] = (
        aggregate["group_1_rate_sd"]
        .fillna(0.0)
    )

    return raw, aggregate


source_action_raw, source_action_rates = (
    _category_group_1_rate_table(
        edge_stats_df,
        ["candidate_source", "action"],
    )
)

source_action_raw.to_csv(
    RQ2_CONTEXT_TABLE_DIR
    / "source_action_group1_rates_per_seed.csv",
    index=False,
)
source_action_rates.to_csv(
    RQ2_CONTEXT_TABLE_DIR
    / "source_action_group1_rates_averaged.csv",
    index=False,
)

for config_key, frame in source_action_rates.groupby(
    base_group_cols,
    dropna=False,
):
    meta = _config_meta(config_key)
    info = _mode_info(
        meta["scoring_mode"],
        meta["comparison_rule"],
    )

    frame = frame.copy()
    frame["category"] = (
        frame["candidate_source"].astype(str)
        + " | "
        + frame["action"].astype(str)
    )
    frame = frame.sort_values(
        "group_1_rate_mean"
    )

    _subsection(
        "Group-1 rate by candidate source and action",
        (
            f"{_config_title(config_key)}. A higher bar means that "
            f"the source/action stratum contains a larger proportion "
            f"of {info['group_1']} edges."
        ),
    )
    display(frame)

    fig, ax = plt.subplots(
        figsize=(
            10,
            max(4.5, 0.65 * len(frame)),
        )
    )
    y = np.arange(len(frame))

    ax.barh(
        y,
        frame["group_1_rate_mean"],
        xerr=frame["group_1_rate_sd"],
        capsize=3,
    )
    ax.axvline(
        0.5,
        linestyle=":",
        linewidth=0.9,
        label="0.5 reference",
    )
    ax.set_yticks(y)
    ax.set_yticklabels(frame["category"])
    ax.set_xlabel(
        f"Mean {info['group_1_rate']} across victim seeds"
    )
    ax.set_xlim(0, 1)
    ax.set_title(
        "Group-1 enrichment by source and action"
    )
    ax.grid(
        True,
        axis="x",
        alpha=0.3,
    )
    ax.legend()

    _save_figure(
        fig,
        "03_source_action_rate__"
        + _rq2_context_safe(
            "__".join(map(str, config_key))
        )
        + ".png",
        "Group-1 enrichment by source and action",
        info["main_context"],
        config_key,
    )


# ------------------------------------------------------------
# 4. Numeric effect sizes
# ------------------------------------------------------------

_section(
    "4. Numeric graph-statistic differences",
    (
        "Hedges' g standardizes group 1 minus group 0. Positive values mean "
        "the feature is larger in group 1; negative values mean it is smaller. "
        "For endpoint mode this is endpoint label 1 minus label 0. For subset "
        "mode this is top-score minus bottom-score. The score variables used "
        "to create subset groups were excluded from these feature tests."
    ),
)

for config_key, frame in numeric_effect_summary.groupby(
    base_group_cols,
    dropna=False,
):
    meta = _config_meta(config_key)
    info = _mode_info(
        meta["scoring_mode"],
        meta["comparison_rule"],
    )

    frame = (
        frame.replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["hedges_g_mean"])
        .copy()
    )
    frame = frame.sort_values(
        "abs_hedges_g",
        ascending=False,
    )

    frame["direction"] = np.where(
        frame["hedges_g_mean"] > 0,
        f"higher in {info['short_1']}",
        np.where(
            frame["hedges_g_mean"] < 0,
            f"lower in {info['short_1']}",
            "no difference",
        ),
    )
    frame["effect_magnitude"] = (
        frame["hedges_g_mean"].map(
            _effect_label
        )
    )

    _subsection(
        "Complete numeric comparison table",
        (
            f"{_config_title(config_key)}. The table is sorted by "
            "absolute standardized difference. Direction consistency "
            "is the fraction of seeds agreeing with the majority sign."
        ),
    )

    display_columns = [
        "feature",
        "group_1_mean",
        "group_0_mean",
        "mean_difference",
        "hedges_g_mean",
        "hedges_g_sd",
        "direction",
        "effect_magnitude",
        "direction_consistency",
        "mann_whitney_fdr_significant_fraction",
        "n_seeds",
    ]
    available_columns = [
        column
        for column in display_columns
        if column in frame.columns
    ]
    display(frame[available_columns])

    top = (
        frame.head(
            RQ2_CONTEXT_TOP_NUMERIC
        )
        .sort_values("hedges_g_mean")
    )

    print(
        "\nLargest numeric differences with contextual interpretation:"
    )

    for _, row in frame.head(
        min(15, len(frame))
    ).iterrows():
        print(
            f"- {_pretty_feature(row['feature'])}: "
            f"g={row['hedges_g_mean']:.3f} "
            f"({_effect_label(row['hedges_g_mean'])}; "
            f"{row['direction']}), "
            f"group-1 mean={row['group_1_mean']:.4g}, "
            f"group-0 mean={row['group_0_mean']:.4g}, "
            f"seed direction consistency="
            f"{row['direction_consistency']:.2f}."
        )

    if not top.empty:
        fig, ax = plt.subplots(
            figsize=(12, max(7, 0.34 * len(top)))
        )
        y = np.arange(len(top))

        ax.barh(
            y,
            top["hedges_g_mean"],
            xerr=top["hedges_g_sd"],
            capsize=3,
        )
        ax.axvline(0.0, linewidth=1.0)

        for reference in [-0.5, -0.2, 0.2, 0.5]:
            ax.axvline(
                reference,
                linestyle=":",
                linewidth=0.8,
            )

        ax.set_yticks(y)
        ax.set_yticklabels(
            [
                _pretty_feature(value)
                for value in top["feature"]
            ]
        )
        ax.set_xlabel(
            "Hedges' g: "
            + info["contrast"]
            + " (error bars: SD across seeds)"
        )
        ax.set_title(
            "Largest numeric differences"
        )
        ax.grid(
            True,
            axis="x",
            alpha=0.3,
        )

        _save_figure(
            fig,
            "04_top_numeric_effects__"
            + _rq2_context_safe(
                "__".join(map(str, config_key))
            )
            + ".png",
            "Largest numeric differences",
            (
                f"Top {len(top)} standardized differences for "
                f"{_config_title(config_key)}."
            ),
            config_key,
        )

    for page_index, start in enumerate(
        range(
            0,
            len(frame),
            RQ2_CONTEXT_EFFECTS_PER_PAGE,
        ),
        start=1,
    ):
        page = (
            frame.iloc[
                start:
                start + RQ2_CONTEXT_EFFECTS_PER_PAGE
            ]
            .sort_values("hedges_g_mean")
        )

        if page.empty:
            continue

        _subsection(
            f"All numeric effects — page {page_index}",
            (
                f"Features {start + 1}–{start + len(page)} "
                "after sorting by absolute Hedges' g."
            ),
        )
        display(page[available_columns])

        fig, ax = plt.subplots(
            figsize=(
                12,
                max(7, 0.32 * len(page)),
            )
        )
        y = np.arange(len(page))

        ax.barh(
            y,
            page["hedges_g_mean"],
            xerr=page["hedges_g_sd"],
            capsize=2,
        )
        ax.axvline(0.0, linewidth=1.0)
        ax.set_yticks(y)
        ax.set_yticklabels(
            [
                _pretty_feature(value)
                for value in page["feature"]
            ]
        )
        ax.set_xlabel(
            "Hedges' g: " + info["contrast"]
        )
        ax.set_title(
            f"All numeric effects — page {page_index}"
        )
        ax.grid(
            True,
            axis="x",
            alpha=0.3,
        )

        _save_figure(
            fig,
            f"04_all_numeric_effects_page-{page_index}__"
            + _rq2_context_safe(
                "__".join(map(str, config_key))
            )
            + ".png",
            f"All numeric effects — page {page_index}",
            (
                "Paginated complete numeric effect output for "
                + _config_title(config_key)
            ),
            config_key,
        )


# ------------------------------------------------------------
# 5. Seed-level effect stability
# ------------------------------------------------------------

_section(
    "5. Stability of numeric effects across victim seeds",
    (
        "The heatmaps show Hedges' g separately for every seed. A feature "
        "is more robust when its sign and approximate magnitude remain "
        "consistent across seeds."
    ),
)

for config_key, summary_frame in numeric_effect_summary.groupby(
    base_group_cols,
    dropna=False,
):
    meta = _config_meta(config_key)
    info = _mode_info(
        meta["scoring_mode"],
        meta["comparison_rule"],
    )

    top_features = (
        summary_frame
        .dropna(subset=["hedges_g_mean"])
        .nlargest(
            min(
                RQ2_CONTEXT_TOP_NUMERIC,
                len(summary_frame),
            ),
            "abs_hedges_g",
        )["feature"]
        .tolist()
    )

    mask = np.ones(
        len(numeric_comparison_raw_df),
        dtype=bool,
    )

    for column, value in meta.items():
        mask &= (
            numeric_comparison_raw_df[column]
            == value
        )

    raw = numeric_comparison_raw_df[
        mask
        & numeric_comparison_raw_df[
            "feature"
        ].isin(top_features)
    ]

    if raw.empty:
        continue

    pivot = raw.pivot_table(
        index="feature",
        columns="seed",
        values="hedges_g",
        aggfunc="mean",
    )
    pivot = pivot.reindex(top_features)

    _subsection(
        "Seed-by-feature Hedges' g matrix",
        (
            f"{_config_title(config_key)}. Rows are ordered by "
            "absolute mean effect."
        ),
    )
    display(pivot.reset_index())

    matrix = pivot.to_numpy(dtype=float)
    finite = matrix[np.isfinite(matrix)]
    max_abs = (
        max(
            0.2,
            float(np.max(np.abs(finite))),
        )
        if finite.size
        else 1.0
    )

    fig, ax = plt.subplots(
        figsize=(
            max(7, 1.0 * pivot.shape[1] + 4),
            max(7, 0.34 * pivot.shape[0]),
        )
    )
    image = ax.imshow(
        matrix,
        aspect="auto",
        vmin=-max_abs,
        vmax=max_abs,
    )
    ax.set_xticks(
        np.arange(pivot.shape[1])
    )
    ax.set_xticklabels(
        [str(value) for value in pivot.columns]
    )
    ax.set_yticks(
        np.arange(pivot.shape[0])
    )
    ax.set_yticklabels(
        [
            _pretty_feature(value)
            for value in pivot.index
        ]
    )
    ax.set_xlabel("Victim-model seed")
    ax.set_title(
        "Numeric effect stability across seeds"
    )
    fig.colorbar(
        image,
        ax=ax,
        label="Hedges' g: " + info["contrast"],
    )

    _save_figure(
        fig,
        "05_numeric_seed_stability__"
        + _rq2_context_safe(
            "__".join(map(str, config_key))
        )
        + ".png",
        "Numeric effect stability across seeds",
        (
            "Seed-level effect matrix for "
            + _config_title(config_key)
        ),
        config_key,
    )


# ------------------------------------------------------------
# 6. Raw distributions
# ------------------------------------------------------------

_section(
    "6. Raw distributions of the strongest numeric differences",
    (
        "These boxplots show the raw feature distributions for group 0 and "
        "group 1. They expose skewness, zero inflation, and outliers that "
        "can be hidden by a single effect-size summary."
    ),
)

for config_key, summary_frame in numeric_effect_summary.groupby(
    base_group_cols,
    dropna=False,
):
    meta = _config_meta(config_key)
    info = _mode_info(
        meta["scoring_mode"],
        meta["comparison_rule"],
    )

    mask = np.ones(
        len(edge_stats_df),
        dtype=bool,
    )
    for column, value in meta.items():
        mask &= edge_stats_df[column] == value
    config_edges = edge_stats_df[mask]

    candidates = []

    for _, row in summary_frame.sort_values(
        "abs_hedges_g",
        ascending=False,
    ).iterrows():
        feature = row["feature"]

        if feature not in config_edges.columns:
            continue

        numeric = pd.to_numeric(
            config_edges[feature],
            errors="coerce",
        )

        if numeric.nunique(dropna=True) <= 4:
            continue

        candidates.append(feature)

        if (
            len(candidates)
            >= RQ2_CONTEXT_DISTRIBUTION_FEATURES
        ):
            break

    for rank, feature in enumerate(
        candidates,
        start=1,
    ):
        values_0 = pd.to_numeric(
            config_edges.loc[
                config_edges["label_group"] == 0,
                feature,
            ],
            errors="coerce",
        ).dropna()

        values_1 = pd.to_numeric(
            config_edges.loc[
                config_edges["label_group"] == 1,
                feature,
            ],
            errors="coerce",
        ).dropna()

        if values_0.empty or values_1.empty:
            continue

        summary_table = pd.DataFrame([
            {
                "label_group": 0,
                "group_name": info["group_0"],
                "n": len(values_0),
                "mean": values_0.mean(),
                "sd": values_0.std(ddof=1),
                "median": values_0.median(),
                "q1": values_0.quantile(0.25),
                "q3": values_0.quantile(0.75),
            },
            {
                "label_group": 1,
                "group_name": info["group_1"],
                "n": len(values_1),
                "mean": values_1.mean(),
                "sd": values_1.std(ddof=1),
                "median": values_1.median(),
                "q1": values_1.quantile(0.25),
                "q3": values_1.quantile(0.75),
            },
        ])

        effect_row = summary_frame[
            summary_frame["feature"] == feature
        ].iloc[0]

        context = (
            f"{_pretty_feature(feature)} is ranked {rank} for "
            f"{_config_title(config_key)}. Mean Hedges' g is "
            f"{effect_row['hedges_g_mean']:.3f}."
        )

        _subsection(
            f"Distribution: {_pretty_feature(feature)}",
            context,
        )
        display(summary_table)

        fig, ax = plt.subplots(
            figsize=(7.5, 5.0)
        )
        ax.boxplot(
            [
                values_0.to_numpy(),
                values_1.to_numpy(),
            ],
            labels=[
                info["short_0"],
                info["short_1"],
            ],
            showfliers=False,
        )
        ax.set_ylabel(
            _pretty_feature(feature)
        )
        ax.set_title(
            _pretty_feature(feature)
            + " by comparison group"
        )
        ax.grid(
            True,
            axis="y",
            alpha=0.3,
        )

        _save_figure(
            fig,
            f"06_distribution_{rank:02d}_"
            + _rq2_context_safe(feature)
            + "__"
            + _rq2_context_safe(
                "__".join(map(str, config_key))
            )
            + ".png",
            (
                "Distribution of "
                + _pretty_feature(feature)
            ),
            context,
            config_key,
        )


# ------------------------------------------------------------
# 7. Categorical associations
# ------------------------------------------------------------

_section(
    "7. Categorical associations",
    (
        "Cramér's V measures association strength between a categorical "
        "edge property and comparison-group membership. It is non-"
        "directional, so level-proportion plots are needed to identify "
        "which categories are enriched in group 1."
    ),
)

for config_key, frame in categorical_effect_summary.groupby(
    base_group_cols,
    dropna=False,
):
    meta = _config_meta(config_key)
    info = _mode_info(
        meta["scoring_mode"],
        meta["comparison_rule"],
    )

    frame = (
        frame.replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["cramers_v_mean"])
        .copy()
    )
    frame = frame.sort_values(
        "cramers_v_mean",
        ascending=False,
    )
    frame["association_strength"] = (
        frame["cramers_v_mean"].map(
            _association_label
        )
    )

    _subsection(
        "Complete categorical association table",
        (
            f"{_config_title(config_key)}. The table is sorted by "
            "mean Cramér's V across victim seeds."
        ),
    )
    display(frame)

    print("\nStrongest categorical associations:")
    for _, row in frame.head(
        min(15, len(frame))
    ).iterrows():
        print(
            f"- {_pretty_feature(row['feature'])}: "
            f"Cramér's V={row['cramers_v_mean']:.3f} "
            f"({_association_label(row['cramers_v_mean'])}), "
            f"chi-square FDR-significant in "
            f"{row['chi_square_fdr_significant_fraction']:.2f} "
            "of seeds."
        )

    top = (
        frame.head(
            RQ2_CONTEXT_TOP_CATEGORICAL
        )
        .sort_values("cramers_v_mean")
    )

    if top.empty:
        continue

    fig, ax = plt.subplots(
        figsize=(
            12,
            max(6, 0.35 * len(top)),
        )
    )
    y = np.arange(len(top))
    ax.barh(
        y,
        top["cramers_v_mean"],
        xerr=top["cramers_v_sd"],
        capsize=3,
    )
    ax.set_yticks(y)
    ax.set_yticklabels(
        [
            _pretty_feature(value)
            for value in top["feature"]
        ]
    )
    ax.set_xlabel(
        "Cramér's V (error bars: SD across seeds)"
    )
    ax.set_xlim(left=0)
    ax.set_title(
        "Largest categorical associations"
    )
    ax.grid(
        True,
        axis="x",
        alpha=0.3,
    )

    _save_figure(
        fig,
        "07_categorical_associations__"
        + _rq2_context_safe(
            "__".join(map(str, config_key))
        )
        + ".png",
        "Largest categorical associations",
        info["main_context"],
        config_key,
    )


def _categorical_level_summary(frame, feature):
    raw_rows = []

    for group_key, group in frame.groupby(
        raw_group_cols,
        dropna=False,
    ):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        meta = dict(
            zip(raw_group_cols, group_key)
        )

        for label_group, label_frame in group.groupby(
            "label_group"
        ):
            counts = (
                label_frame[feature]
                .fillna("<NA>")
                .astype(str)
                .value_counts(dropna=False)
            )
            total = counts.sum()

            for level, count in counts.items():
                raw_rows.append({
                    **meta,
                    "feature": feature,
                    "level": str(level),
                    "label_group": int(label_group),
                    "count": int(count),
                    "fraction": float(
                        count / max(1, total)
                    ),
                })

    raw = pd.DataFrame(raw_rows)

    if raw.empty:
        return raw, raw

    aggregate = (
        raw.groupby(
            base_group_cols
            + [
                "feature",
                "level",
                "label_group",
            ],
            dropna=False,
        )
        .agg(
            n_seeds=("seed", "nunique"),
            count_mean=("count", "mean"),
            fraction_mean=("fraction", "mean"),
            fraction_sd=("fraction", "std"),
        )
        .reset_index()
    )
    aggregate["fraction_sd"] = (
        aggregate["fraction_sd"].fillna(0.0)
    )

    return raw, aggregate


for feature in RQ2_CONTEXT_CATEGORICAL_FEATURES:
    if feature not in edge_stats_df.columns:
        continue

    raw, aggregate = _categorical_level_summary(
        edge_stats_df,
        feature,
    )

    if aggregate.empty:
        continue

    raw.to_csv(
        RQ2_CONTEXT_TABLE_DIR
        / (
            "categorical_levels_"
            + _rq2_context_safe(feature)
            + "_per_seed.csv"
        ),
        index=False,
    )
    aggregate.to_csv(
        RQ2_CONTEXT_TABLE_DIR
        / (
            "categorical_levels_"
            + _rq2_context_safe(feature)
            + "_averaged.csv"
        ),
        index=False,
    )

    for config_key, frame in aggregate.groupby(
        base_group_cols,
        dropna=False,
    ):
        meta = _config_meta(config_key)
        info = _mode_info(
            meta["scoring_mode"],
            meta["comparison_rule"],
        )

        if frame["level"].nunique() > 18:
            print(
                f"Skipping direct level plot for {feature} in "
                f"{_config_title(config_key)} because it has "
                f"{frame['level'].nunique()} levels. CSV saved."
            )
            continue

        levels = (
            frame["level"]
            .drop_duplicates()
            .tolist()
        )

        _subsection(
            f"Level proportions: {_pretty_feature(feature)}",
            (
                f"{_config_title(config_key)}. Each pair of bars "
                "compares the within-group proportion."
            ),
        )
        display(
            frame.sort_values(
                ["level", "label_group"]
            )
        )

        x = np.arange(len(levels))
        width = 0.38
        fig, ax = plt.subplots(
            figsize=(
                max(9, 1.1 * len(levels)),
                5.2,
            )
        )

        for offset_index, label_group in enumerate([0, 1]):
            label_frame = (
                frame[
                    frame["label_group"]
                    == label_group
                ]
                .set_index("level")
                .reindex(levels)
                .reset_index()
            )
            offset = (
                offset_index - 0.5
            ) * width

            ax.bar(
                x + offset,
                label_frame["fraction_mean"],
                width=width,
                yerr=label_frame[
                    "fraction_sd"
                ],
                capsize=3,
                label=_label_name(
                    label_group,
                    meta["scoring_mode"],
                    meta["comparison_rule"],
                ),
            )

        ax.set_xticks(x)
        ax.set_xticklabels(
            levels,
            rotation=25,
            ha="right",
        )
        ax.set_ylabel("Within-group proportion")
        ax.set_title(
            _pretty_feature(feature)
        )
        ax.grid(
            True,
            axis="y",
            alpha=0.3,
        )
        ax.legend()

        _save_figure(
            fig,
            "07_levels_"
            + _rq2_context_safe(feature)
            + "__"
            + _rq2_context_safe(
                "__".join(map(str, config_key))
            )
            + ".png",
            (
                "Level proportions for "
                + _pretty_feature(feature)
            ),
            (
                "Direction of categorical association for "
                + _config_title(config_key)
            ),
            config_key,
        )


# ------------------------------------------------------------
# 8. True-class pair structure
# ------------------------------------------------------------

_section(
    "8. True-class pair structure",
    (
        "The matrix shows the conditional group-1 rate for each unordered "
        "pair of endpoint classes. For endpoint mode this is endpoint-label-1 "
        "yield. For subset mode it indicates enrichment in the top-score "
        "extreme relative to the bottom-score extreme."
    ),
)

for config_key, frame in edge_stats_df.groupby(
    base_group_cols,
    dropna=False,
):
    meta = _config_meta(config_key)
    info = _mode_info(
        meta["scoring_mode"],
        meta["comparison_rule"],
    )

    class_rate_rows = []

    for (
        seed,
        class_pair,
    ), pair_frame in frame.groupby(
        ["seed", "canonical_true_label_pair"],
        dropna=False,
    ):
        class_rate_rows.append({
            "seed": int(seed),
            "canonical_true_label_pair": str(class_pair),
            "n_candidates": int(len(pair_frame)),
            "group_1_rate": float(
                pair_frame["label_group"].mean()
            ),
        })

    class_rates_raw = pd.DataFrame(
        class_rate_rows
    )

    class_rates = (
        class_rates_raw
        .groupby(
            "canonical_true_label_pair",
            dropna=False,
        )
        .agg(
            n_seeds=("seed", "nunique"),
            n_candidates_mean=(
                "n_candidates",
                "mean",
            ),
            group_1_rate_mean=(
                "group_1_rate",
                "mean",
            ),
            group_1_rate_sd=(
                "group_1_rate",
                "std",
            ),
        )
        .reset_index()
    )
    class_rates["group_1_rate_sd"] = (
        class_rates["group_1_rate_sd"]
        .fillna(0.0)
    )
    class_rates = class_rates.sort_values(
        [
            "group_1_rate_mean",
            "n_candidates_mean",
        ],
        ascending=[False, False],
    )

    _subsection(
        "Group-1 rate by true-class pair",
        (
            f"{_config_title(config_key)}. Higher values indicate "
            f"greater enrichment in {info['group_1']}."
        ),
    )
    display(class_rates)

    class_rates.to_csv(
        RQ2_CONTEXT_TABLE_DIR
        / (
            "class_pair_group1_rates__"
            + _rq2_context_safe(
                "__".join(map(str, config_key))
            )
            + ".csv"
        ),
        index=False,
    )

    parsed = []

    for _, row in class_rates.iterrows():
        numbers = re.findall(
            r"\d+",
            str(
                row[
                    "canonical_true_label_pair"
                ]
            ),
        )

        if len(numbers) < 2:
            continue

        a, b = int(numbers[0]), int(numbers[1])
        parsed.append((
            a,
            b,
            row["group_1_rate_mean"],
            row["n_candidates_mean"],
        ))

    if not parsed:
        continue

    classes = sorted({
        value
        for row in parsed
        for value in row[:2]
    })
    class_to_index = {
        value: index
        for index, value in enumerate(classes)
    }
    rate_matrix = np.full(
        (len(classes), len(classes)),
        np.nan,
    )
    count_matrix = np.full(
        (len(classes), len(classes)),
        np.nan,
    )

    for a, b, rate, count in parsed:
        i = class_to_index[a]
        j = class_to_index[b]
        rate_matrix[i, j] = rate
        rate_matrix[j, i] = rate
        count_matrix[i, j] = count
        count_matrix[j, i] = count

    fig, ax = plt.subplots(
        figsize=(7.5, 6.5)
    )
    image = ax.imshow(
        rate_matrix,
        vmin=0.0,
        vmax=max(
            0.01,
            np.nanmax(rate_matrix),
        ),
    )
    ax.set_xticks(
        np.arange(len(classes))
    )
    ax.set_xticklabels(classes)
    ax.set_yticks(
        np.arange(len(classes))
    )
    ax.set_yticklabels(classes)
    ax.set_xlabel("Endpoint class")
    ax.set_ylabel("Endpoint class")
    ax.set_title(
        "Mean group-1 rate by true-class pair"
    )

    for i in range(len(classes)):
        for j in range(len(classes)):
            if np.isfinite(rate_matrix[i, j]):
                ax.text(
                    j,
                    i,
                    (
                        f"{rate_matrix[i, j]:.2f}\n"
                        f"n≈{count_matrix[i, j]:.0f}"
                    ),
                    ha="center",
                    va="center",
                    fontsize=8,
                )

    fig.colorbar(
        image,
        ax=ax,
        label=info["group_1_rate"],
    )

    _save_figure(
        fig,
        "08_true_class_pair_matrix__"
        + _rq2_context_safe(
            "__".join(map(str, config_key))
        )
        + ".png",
        "True-class pair group-1 matrix",
        info["main_context"],
        config_key,
    )


# ------------------------------------------------------------
# 9. Group-1 rate along numeric gradients
# ------------------------------------------------------------

_section(
    "9. Group-1 rates along numeric gradients",
    (
        "Candidates are binned by an interpretable feature and the conditional "
        "group-1 rate is plotted. Endpoint mode estimates endpoint-label-1 "
        "yield. Subset mode estimates top-score enrichment among only the "
        "selected top/bottom extremes; it is not a population probability "
        "over all candidates."
    ),
)


def _quantile_bin_rates(frame, feature, q=5):
    raw_parts = []

    for config_key, config_frame in frame.groupby(
        base_group_cols,
        dropna=False,
    ):
        working = config_frame[
            [
                *raw_group_cols,
                "label_group",
                feature,
            ]
        ].copy()

        working[feature] = pd.to_numeric(
            working[feature],
            errors="coerce",
        )
        working = working.dropna(
            subset=[feature]
        )

        if (
            working.empty
            or working[feature].nunique() < 2
        ):
            continue

        try:
            working["bin"] = pd.qcut(
                working[feature],
                q=min(
                    q,
                    working[feature].nunique(),
                ),
                duplicates="drop",
            )
        except ValueError:
            continue

        working["bin_label"] = (
            working["bin"].astype(str)
        )

        raw_part = (
            working.groupby(
                raw_group_cols + ["bin_label"],
                dropna=False,
            )
            .agg(
                n_candidates=(
                    "label_group",
                    "size",
                ),
                feature_mean=(feature, "mean"),
                group_1_rate=(
                    "label_group",
                    "mean",
                ),
            )
            .reset_index()
        )

        raw_parts.append(raw_part)

    if not raw_parts:
        return pd.DataFrame(), pd.DataFrame()

    raw = pd.concat(
        raw_parts,
        ignore_index=True,
    )

    aggregate = (
        raw.groupby(
            base_group_cols + ["bin_label"],
            dropna=False,
        )
        .agg(
            n_seeds=("seed", "nunique"),
            n_candidates_mean=(
                "n_candidates",
                "mean",
            ),
            feature_mean=(
                "feature_mean",
                "mean",
            ),
            group_1_rate_mean=(
                "group_1_rate",
                "mean",
            ),
            group_1_rate_sd=(
                "group_1_rate",
                "std",
            ),
        )
        .reset_index()
    )
    aggregate["group_1_rate_sd"] = (
        aggregate["group_1_rate_sd"]
        .fillna(0.0)
    )

    return raw, aggregate


for feature in RQ2_CONTEXT_BINNED_FEATURES:
    if feature not in edge_stats_df.columns:
        continue

    raw, aggregate = _quantile_bin_rates(
        edge_stats_df,
        feature,
    )

    if aggregate.empty:
        continue

    raw.to_csv(
        RQ2_CONTEXT_TABLE_DIR
        / (
            "binned_"
            + _rq2_context_safe(feature)
            + "_per_seed.csv"
        ),
        index=False,
    )
    aggregate.to_csv(
        RQ2_CONTEXT_TABLE_DIR
        / (
            "binned_"
            + _rq2_context_safe(feature)
            + "_averaged.csv"
        ),
        index=False,
    )

    for config_key, frame in aggregate.groupby(
        base_group_cols,
        dropna=False,
    ):
        meta = _config_meta(config_key)
        info = _mode_info(
            meta["scoring_mode"],
            meta["comparison_rule"],
        )

        frame = frame.sort_values(
            "feature_mean"
        )

        _subsection(
            f"Binned group-1 rate: {_pretty_feature(feature)}",
            (
                f"{_config_title(config_key)}. Bins run from lower "
                "to higher feature values."
            ),
        )
        display(frame)

        fig, ax = plt.subplots(
            figsize=(9.5, 5.0)
        )
        x = np.arange(len(frame))

        ax.errorbar(
            x,
            frame["group_1_rate_mean"],
            yerr=frame["group_1_rate_sd"],
            marker="o",
            capsize=3,
        )
        ax.axhline(
            0.5,
            linestyle=":",
            linewidth=0.9,
        )
        ax.set_xticks(x)
        ax.set_xticklabels(
            frame["bin_label"],
            rotation=25,
            ha="right",
        )
        ax.set_xlabel(
            _pretty_feature(feature)
            + " quantile bin"
        )
        ax.set_ylabel(
            "Mean "
            + info["group_1_rate"]
            + " across victim seeds"
        )
        ax.set_ylim(0, 1)
        ax.set_title(
            "Group-1 rate across "
            + _pretty_feature(feature)
        )
        ax.grid(
            True,
            alpha=0.3,
        )

        _save_figure(
            fig,
            "09_binned_"
            + _rq2_context_safe(feature)
            + "__"
            + _rq2_context_safe(
                "__".join(map(str, config_key))
            )
            + ".png",
            (
                "Binned group-1 rate for "
                + _pretty_feature(feature)
            ),
            info["main_context"],
            config_key,
        )


# ------------------------------------------------------------
# 10. Mode-specific mechanism diagnostics
# ------------------------------------------------------------

_section(
    "10. Mode-specific mechanism diagnostics",
    (
        "Endpoint-hit patterns exist only for endpoint mining. Subset mining "
        "instead reports score separation, inclusion counts, and selection "
        "cutoffs. No endpoint-hit values are invented for subset candidates."
    ),
)

endpoint_edges = edge_stats_df[
    edge_stats_df["scoring_mode"] == "endpoint"
].copy()

if endpoint_edges.empty:
    print("No endpoint runs are present; endpoint-hit mechanism plots skipped.")
else:
    endpoint_eligibility_raw, endpoint_eligibility_rates = (
        _category_group_1_rate_table(
            endpoint_edges,
            ["n_clean_correct_endpoints"],
        )
    )

    _display_and_save(
        endpoint_eligibility_rates.sort_values(
            base_group_cols
            + ["n_clean_correct_endpoints"]
        ),
        "endpoint_eligibility_rates.csv",
    )

    endpoint_edges["hit_pattern"] = np.select(
        [
            endpoint_edges[
                "both_hit"
            ].fillna(False).astype(bool),
            endpoint_edges[
                "u_hit"
            ].fillna(False).astype(bool)
            & ~endpoint_edges[
                "v_hit"
            ].fillna(False).astype(bool),
            ~endpoint_edges[
                "u_hit"
            ].fillna(False).astype(bool)
            & endpoint_edges[
                "v_hit"
            ].fillna(False).astype(bool),
        ],
        ["both", "u_only", "v_only"],
        default="neither",
    )

    for config_key, frame in endpoint_edges.groupby(
        base_group_cols,
        dropna=False,
    ):
        meta = _config_meta(config_key)
        info = _mode_info(
            meta["scoring_mode"],
            meta["comparison_rule"],
        )

        eligibility = endpoint_eligibility_rates[
            (
                endpoint_eligibility_rates[
                    "candidate_config_id"
                ]
                == meta["candidate_config_id"]
            )
            & (
                endpoint_eligibility_rates[
                    "scoring_mode"
                ]
                == meta["scoring_mode"]
            )
            & (
                endpoint_eligibility_rates[
                    "comparison_rule"
                ]
                == meta["comparison_rule"]
            )
            & (
                endpoint_eligibility_rates[
                    "endpoint_mining_hop"
                ]
                == meta["endpoint_mining_hop"]
            )
        ].sort_values(
            "n_clean_correct_endpoints"
        )

        _subsection(
            "Endpoint eligibility-conditioned harmful rate",
            (
                f"{_config_title(config_key)}. Candidates with zero "
                "clean-correct endpoints are ineligible for endpoint "
                "label 1."
            ),
        )
        display(eligibility)

        fig, ax = plt.subplots(
            figsize=(7.5, 4.8)
        )
        ax.bar(
            eligibility[
                "n_clean_correct_endpoints"
            ].astype(str),
            eligibility["group_1_rate_mean"],
            yerr=eligibility[
                "group_1_rate_sd"
            ],
            capsize=3,
        )
        ax.set_xlabel(
            "Number of clean-correct endpoints"
        )
        ax.set_ylabel(
            "Mean endpoint-label-1 rate"
        )
        ax.set_ylim(0, 1)
        ax.set_title(
            "Endpoint-label eligibility"
        )
        ax.grid(
            True,
            axis="y",
            alpha=0.3,
        )

        _save_figure(
            fig,
            "10_endpoint_eligibility__"
            + _rq2_context_safe(
                "__".join(map(str, config_key))
            )
            + ".png",
            "Endpoint-label eligibility",
            info["main_context"],
            config_key,
        )

        hit_rows = []

        for (
            seed,
            label_group,
        ), label_frame in frame.groupby(
            ["seed", "label_group"]
        ):
            proportions = (
                label_frame["hit_pattern"]
                .value_counts(normalize=True)
            )

            for pattern in [
                "neither",
                "u_only",
                "v_only",
                "both",
            ]:
                hit_rows.append({
                    "seed": int(seed),
                    "label_group": int(label_group),
                    "hit_pattern": pattern,
                    "fraction": float(
                        proportions.get(
                            pattern,
                            0.0,
                        )
                    ),
                })

        hit_raw = pd.DataFrame(hit_rows)
        hit_summary = (
            hit_raw.groupby(
                ["label_group", "hit_pattern"],
                dropna=False,
            )
            .agg(
                fraction_mean=(
                    "fraction",
                    "mean",
                ),
                fraction_sd=(
                    "fraction",
                    "std",
                ),
                n_seeds=("seed", "nunique"),
            )
            .reset_index()
        )
        hit_summary["fraction_sd"] = (
            hit_summary["fraction_sd"]
            .fillna(0.0)
        )

        _subsection(
            "Endpoint hit-pattern composition",
            (
                "This decomposition is endpoint-specific. "
                "Label 0 should be dominated by neither."
            ),
        )
        display(hit_summary)

        patterns = [
            "neither",
            "u_only",
            "v_only",
            "both",
        ]
        x = np.arange(len(patterns))
        width = 0.38
        fig, ax = plt.subplots(
            figsize=(8.5, 4.8)
        )

        for offset_index, label_group in enumerate([0, 1]):
            label_summary = (
                hit_summary[
                    hit_summary["label_group"]
                    == label_group
                ]
                .set_index("hit_pattern")
                .reindex(patterns)
                .reset_index()
            )
            offset = (
                offset_index - 0.5
            ) * width

            ax.bar(
                x + offset,
                label_summary["fraction_mean"],
                width=width,
                yerr=label_summary["fraction_sd"],
                capsize=3,
                label=_label_name(
                    label_group,
                    meta["scoring_mode"],
                    meta["comparison_rule"],
                ),
            )

        ax.set_xticks(x)
        ax.set_xticklabels(patterns)
        ax.set_ylabel("Within-group fraction")
        ax.set_title(
            "Endpoint hit patterns"
        )
        ax.grid(
            True,
            axis="y",
            alpha=0.3,
        )
        ax.legend()

        _save_figure(
            fig,
            "10_endpoint_hit_patterns__"
            + _rq2_context_safe(
                "__".join(map(str, config_key))
            )
            + ".png",
            "Endpoint hit-pattern composition",
            info["main_context"],
            config_key,
        )

subset_edges = edge_stats_df[
    edge_stats_df["scoring_mode"]
    == "subset_accuracy_drop"
].copy()

if subset_edges.empty:
    print("No subset-accuracy-drop runs are present; subset diagnostics skipped.")
else:
    subset_diagnostics = (
        subset_edges
        .groupby(
            raw_group_cols + ["label_group"],
            dropna=False,
        )
        .agg(
            n_selected=("score_raw", "size"),
            score_raw_mean=("score_raw", "mean"),
            score_raw_std=("score_raw", "std"),
            score_raw_min=("score_raw", "min"),
            score_raw_max=("score_raw", "max"),
            score_percentile_mean=(
                "score_percentile",
                "mean",
            ),
            inclusion_count_mean=(
                "inclusion_count",
                "mean",
            ),
            inclusion_count_std=(
                "inclusion_count",
                "std",
            ),
        )
        .reset_index()
    )

    subset_diagnostics["group_name"] = [
        _label_name(
            row.label_group,
            row.scoring_mode,
            row.comparison_rule,
        )
        for row in subset_diagnostics.itertuples()
    ]

    _display_and_save(
        subset_diagnostics.sort_values(
            raw_group_cols + ["label_group"]
        ),
        "subset_score_group_diagnostics.csv",
    )

    if not comparison_selection_df.empty:
        subset_selection = comparison_selection_df[
            comparison_selection_df["scoring_mode"]
            == "subset_accuracy_drop"
        ].copy()

        if not subset_selection.empty:
            _subsection(
                "Subset top/bottom selection cutoffs",
                (
                    "The table exposes score range, top and bottom "
                    "cutoffs, tie counts at each boundary, observation "
                    "counts, and whether a run had constant scores."
                ),
            )
            display(
                subset_selection.sort_values(
                    raw_group_cols
                )
            )


# ------------------------------------------------------------
# 11. Node involvement
# ------------------------------------------------------------

_section(
    "11. Node involvement",
    (
        "Incidence is normalized within each seed and comparison group. "
        "This reveals whether group-1 edges repeatedly involve a small "
        "set of nodes or whether their endpoint coverage is diffuse."
    ),
)

node_frame = node_involvement_df.copy()

node_totals = (
    node_frame.groupby(
        raw_group_cols + ["label_group"],
        dropna=False,
    )["candidate_incidence"]
    .sum()
    .rename("total_incidence")
    .reset_index()
)

node_frame = node_frame.merge(
    node_totals,
    on=raw_group_cols + ["label_group"],
    how="left",
)

node_frame["incidence_share"] = (
    node_frame["candidate_incidence"]
    / node_frame["total_incidence"].clip(lower=1)
)

node_frame.to_csv(
    RQ2_CONTEXT_TABLE_DIR
    / "node_involvement_normalized.csv",
    index=False,
)

for config_key, frame in node_frame.groupby(
    base_group_cols,
    dropna=False,
):
    meta = _config_meta(config_key)
    info = _mode_info(
        meta["scoring_mode"],
        meta["comparison_rule"],
    )

    node_summary = (
        frame.groupby(
            ["node", "label_group"],
            dropna=False,
        )
        .agg(
            n_seeds=("seed", "nunique"),
            incidence_share_mean=(
                "incidence_share",
                "mean",
            ),
            incidence_share_sd=(
                "incidence_share",
                "std",
            ),
            candidate_incidence_mean=(
                "candidate_incidence",
                "mean",
            ),
            degree_mean=("degree", "mean"),
            pagerank_mean=("pagerank", "mean"),
            betweenness_mean=(
                "betweenness",
                "mean",
            ),
            clean_confidence_mean=(
                "clean_confidence",
                "mean",
            ),
        )
        .reset_index()
    )

    node_summary["incidence_share_sd"] = (
        node_summary["incidence_share_sd"]
        .fillna(0.0)
    )

    group_1_nodes = (
        node_summary[
            node_summary["label_group"] == 1
        ]
        .nlargest(
            RQ2_CONTEXT_TOP_NODES,
            "incidence_share_mean",
        )["node"]
        .tolist()
    )

    plotted = node_summary[
        node_summary["node"].isin(
            group_1_nodes
        )
    ].copy()

    pivot = (
        plotted.pivot_table(
            index="node",
            columns="label_group",
            values="incidence_share_mean",
            aggfunc="mean",
        )
        .fillna(0.0)
        .reindex(group_1_nodes)
    )

    _subsection(
        "Nodes most involved in group-1 candidate edges",
        (
            f"{_config_title(config_key)}. Nodes are ranked by "
            f"their mean share of {info['group_1']} endpoint "
            "incidences."
        ),
    )
    display(
        plotted.sort_values(
            ["node", "label_group"]
        )
    )

    y = np.arange(len(group_1_nodes))
    height = 0.38
    fig, ax = plt.subplots(
        figsize=(
            11,
            max(6, 0.34 * len(group_1_nodes)),
        )
    )

    values_0 = (
        pivot[0]
        if 0 in pivot.columns
        else pd.Series(
            0.0,
            index=pivot.index,
        )
    )
    values_1 = (
        pivot[1]
        if 1 in pivot.columns
        else pd.Series(
            0.0,
            index=pivot.index,
        )
    )

    ax.barh(
        y - height / 2,
        values_0,
        height=height,
        label=info["group_0"],
    )
    ax.barh(
        y + height / 2,
        values_1,
        height=height,
        label=info["group_1"],
    )
    ax.set_yticks(y)
    ax.set_yticklabels(
        [str(node) for node in group_1_nodes]
    )
    ax.invert_yaxis()
    ax.set_xlabel(
        "Mean share of endpoint incidences"
    )
    ax.set_ylabel("Node ID")
    ax.set_title(
        "Nodes concentrated in group-1 candidate edges"
    )
    ax.grid(
        True,
        axis="x",
        alpha=0.3,
    )
    ax.legend()

    _save_figure(
        fig,
        "11_top_group1_nodes__"
        + _rq2_context_safe(
            "__".join(map(str, config_key))
        )
        + ".png",
        "Nodes concentrated in group-1 edges",
        info["main_context"],
        config_key,
    )

    fig, ax = plt.subplots(
        figsize=(8.5, 5.5)
    )

    for label_group, label_frame in frame.groupby(
        "label_group"
    ):
        ax.scatter(
            label_frame["degree"],
            label_frame["incidence_share"],
            alpha=0.35,
            label=_label_name(
                label_group,
                meta["scoring_mode"],
                meta["comparison_rule"],
            ),
        )

    ax.set_xlabel("Clean node degree")
    ax.set_ylabel(
        "Node share of candidate endpoint incidences"
    )
    ax.set_title(
        "Node degree versus candidate-set involvement"
    )
    ax.grid(
        True,
        alpha=0.3,
    )
    ax.legend()

    _save_figure(
        fig,
        "11_degree_vs_incidence__"
        + _rq2_context_safe(
            "__".join(map(str, config_key))
        )
        + ".png",
        "Node degree versus candidate-set involvement",
        info["main_context"],
        config_key,
    )


# ------------------------------------------------------------
# 12. Paired seed-level differences
# ------------------------------------------------------------

_section(
    "12. Paired victim-seed comparisons",
    (
        "The victim seed is the observational unit. Endpoint differences "
        "are label 1 minus label 0; subset differences are top-score minus "
        "bottom-score. With one seed, these outputs remain descriptive."
    ),
)

if seed_level_comparison_df.empty:
    print(
        "No seed-level paired comparison table is available."
    )
else:
    paired = seed_level_comparison_df.copy()
    paired["abs_seed_mean_difference"] = (
        paired[
            "seed_mean_difference_1_minus_0"
        ].abs()
    )

    for config_key, frame in paired.groupby(
        base_group_cols,
        dropna=False,
    ):
        meta = _config_meta(config_key)
        info = _mode_info(
            meta["scoring_mode"],
            meta["comparison_rule"],
        )

        frame = frame.sort_values(
            "abs_seed_mean_difference",
            ascending=False,
        )

        _subsection(
            "Paired group-1 minus group-0 feature differences",
            (
                f"{_config_title(config_key)}. The contrast is "
                f"{info['contrast']}. Differences remain in each "
                "feature's original units."
            ),
        )
        display(frame)

        frame.to_csv(
            RQ2_CONTEXT_TABLE_DIR
            / (
                "paired_seed_differences__"
                + _rq2_context_safe(
                    "__".join(map(str, config_key))
                )
                + ".csv"
            ),
            index=False,
        )


# ------------------------------------------------------------
# 13. Thesis-oriented interpretation prompts
# ------------------------------------------------------------

_section(
    "13. Automatically generated interpretation summary",
    (
        "These statements use association language. Subset statements refer "
        "to top-versus-bottom score extremes and must not be rewritten as "
        "individual causal edge effects."
    ),
)

thesis_summary_rows = []

for config_key, config_edges in edge_stats_df.groupby(
    base_group_cols,
    dropna=False,
):
    meta = _config_meta(config_key)
    info = _mode_info(
        meta["scoring_mode"],
        meta["comparison_rule"],
    )

    print("\n" + _config_title(config_key))
    print("- Comparison:", info["main_context"])

    if meta["scoring_mode"] == "endpoint":
        group_1_rate_by_seed = (
            config_edges
            .groupby("seed")["label_group"]
            .mean()
        )
        rate_mean, rate_sd, _, n_seeds = _mean_sd_sem(
            group_1_rate_by_seed
        )
        print(
            f"- Endpoint-label prevalence: {rate_mean:.3f} of "
            f"candidates had endpoint label 1 on average across "
            f"{n_seeds} seed(s) (SD={rate_sd:.3f})."
        )
    else:
        score_means = (
            config_edges
            .groupby(
                ["seed", "label_group"]
            )["score_raw"]
            .mean()
            .unstack("label_group")
        )
        score_gap = (
            score_means[1] - score_means[0]
            if 0 in score_means.columns
            and 1 in score_means.columns
            else pd.Series(dtype=float)
        )
        gap_mean, gap_sd, _, n_seeds = _mean_sd_sem(
            score_gap
        )
        print(
            f"- Score separation: the top-minus-bottom mean raw "
            f"score gap was {gap_mean:.6g} across {n_seeds} "
            f"seed(s) (SD={gap_sd:.6g})."
        )

    effect_mask = np.ones(
        len(numeric_effect_summary),
        dtype=bool,
    )
    category_mask = np.ones(
        len(categorical_effect_summary),
        dtype=bool,
    )

    for column, value in meta.items():
        effect_mask &= (
            numeric_effect_summary[column]
            == value
        )
        category_mask &= (
            categorical_effect_summary[column]
            == value
        )

    effects = (
        numeric_effect_summary[
            effect_mask
        ]
        .sort_values(
            "abs_hedges_g",
            ascending=False,
        )
    )

    for _, row in effects.head(8).iterrows():
        direction = (
            "higher"
            if row["hedges_g_mean"] > 0
            else "lower"
        )

        statement = (
            f"{_pretty_feature(row['feature'])} was {direction} "
            f"in {info['group_1']} relative to {info['group_0']} "
            f"(mean g={row['hedges_g_mean']:.3f}, "
            f"{_effect_label(row['hedges_g_mean'])}; direction "
            f"consistency={row['direction_consistency']:.2f})."
        )

        print("- Numeric:", statement)
        thesis_summary_rows.append({
            **meta,
            "type": "numeric",
            "feature": row["feature"],
            "statement": statement,
        })

    cat_effects = (
        categorical_effect_summary[
            category_mask
        ]
        .sort_values(
            "cramers_v_mean",
            ascending=False,
        )
    )

    for _, row in cat_effects.head(5).iterrows():
        statement = (
            f"{_pretty_feature(row['feature'])} was associated "
            f"with {info['group_1']} versus {info['group_0']} "
            f"(mean Cramér's V="
            f"{row['cramers_v_mean']:.3f}, "
            f"{_association_label(row['cramers_v_mean'])})."
        )

        print("- Categorical:", statement)
        thesis_summary_rows.append({
            **meta,
            "type": "categorical",
            "feature": row["feature"],
            "statement": statement,
        })

    if meta["scoring_mode"] == "endpoint":
        print(
            "- Caution: these patterns characterize the local "
            "endpoint label. Multi-edge attack usefulness must be "
            "evaluated separately."
        )
    else:
        print(
            "- Caution: these patterns distinguish score extremes "
            "among observed candidates. The score is assigned from "
            "joint subset outcomes and is not an isolated edge effect."
        )


thesis_summary_df = pd.DataFrame(
    thesis_summary_rows
)
thesis_summary_df.to_csv(
    RQ2_CONTEXT_TABLE_DIR
    / "thesis_interpretation_prompts.csv",
    index=False,
)


# ------------------------------------------------------------
# Save manifest and output index
# ------------------------------------------------------------

plot_manifest_df = pd.DataFrame(
    plot_manifest_rows
)
plot_manifest_df.to_csv(
    RQ2_CONTEXT_TABLE_DIR
    / "plot_manifest.csv",
    index=False,
)

context_report_lines.extend([
    "",
    "Interpretation cautions",
    "-----------------------",
    (
        "1. Endpoint group 1 is a local correct-to-incorrect endpoint "
        "criterion, not a direct global multi-edge attack label."
    ),
    (
        "2. Subset group 1 and group 0 are equal-sized score extremes; "
        "the middle candidates are excluded."
    ),
    (
        "3. The subset score is an edge-associated mean subset drop, "
        "not an individual causal effect."
    ),
    (
        "4. Candidate edges share endpoints, so edge-level observations "
        "are dependent."
    ),
    (
        "5. The victim seed should be the primary unit for robustness "
        "and inference."
    ),
    (
        "6. The analyzed candidate sets are deliberately enriched and "
        "do not estimate graph-wide harmful-edge prevalence."
    ),
])

(
    RQ2_CONTEXT_TABLE_DIR
    / "contextual_output_report.txt"
).write_text(
    "\n".join(context_report_lines),
    encoding="utf-8",
)

output_index = {
    "source_statistics_directory": str(
        RQ2_GRAPH_STATS_OUT_DIR.resolve()
    ),
    "plot_directory": str(
        RQ2_CONTEXT_PLOT_DIR.resolve()
    ),
    "table_directory": str(
        RQ2_CONTEXT_TABLE_DIR.resolve()
    ),
    "n_plots": int(len(plot_manifest_df)),
    "n_edge_rows": int(len(edge_stats_df)),
    "n_victim_seeds": int(
        edge_stats_df["seed"].nunique()
    ),
    "candidate_configurations": sorted(
        edge_stats_df[
            "candidate_config_id"
        ].astype(str).unique().tolist()
    ),
    "scoring_modes": sorted(
        edge_stats_df[
            "scoring_mode"
        ].astype(str).unique().tolist()
    ),
    "comparison_rules": sorted(
        edge_stats_df[
            "comparison_rule"
        ].astype(str).unique().tolist()
    ),
    "endpoint_hops": sorted(
        pd.to_numeric(
            edge_stats_df.loc[
                edge_stats_df["scoring_mode"]
                == "endpoint",
                "endpoint_mining_hop",
            ],
            errors="coerce",
        )
        .dropna()
        .unique()
        .tolist()
    ),
}

(
    RQ2_CONTEXT_TABLE_DIR
    / "contextual_output_index.json"
).write_text(
    json.dumps(
        output_index,
        indent=2,
    ),
    encoding="utf-8",
)

_section(
    "Completed contextual graph-statistics output",
    (
        "All generated figures and tables are separated by scoring mode "
        "and comparison rule. Endpoint behavior remains binary; subset "
        "behavior is reported as top-versus-bottom score-extreme behavior."
    ),
)

print(
    "Number of figures:",
    len(plot_manifest_df),
)
print(
    "Figures:",
    RQ2_CONTEXT_PLOT_DIR.resolve(),
)
print(
    "Tables and report:",
    RQ2_CONTEXT_TABLE_DIR.resolve(),
)
print(
    "Plot manifest:",
    (
        RQ2_CONTEXT_TABLE_DIR
        / "plot_manifest.csv"
    ).resolve(),
)
print(
    "Interpretation prompts:",
    (
        RQ2_CONTEXT_TABLE_DIR
        / "thesis_interpretation_prompts.csv"
    ).resolve(),
)
print(
    "Context report:",
    (
        RQ2_CONTEXT_TABLE_DIR
        / "contextual_output_report.txt"
    ).resolve(),
)

display(plot_manifest_df)


---

## Experiment 2 — Single-Edge Injection into PR-BCD

### Research objective

The mining procedure evaluates candidates individually on the clean graph. However, PR-BCD operates on a changing candidate block and continuously updates relaxed perturbation weights.

This experiment tests whether a mined edge remains useful when it is inserted into an independently evolving PR-BCD optimization state.

The compared conditions are:

1. no-injection baseline;
2. endpoint label-1 candidate;
3. endpoint label-0 candidate;
4. candidate with the highest measured accuracy drop;
5. random unseen candidate.

All paired conditions use:

- the same victim-model seed;
- the same random initial block;
- the same PR-BCD sampling seed;
- the same attack budget;
- the same injection epoch;
- the same optimization configuration.

At the injection epoch, the selected edge replaces the currently lowest-weight coordinate in the PR-BCD block.

### Conditional interpretation

Let \(S_t\) denote the PR-BCD state at the injection epoch. The conditional injection effect is measured relative to the paired no-injection run:

\[
H_{\mathrm{inj}}(e\mid S_t)
=
A_{\mathrm{baseline}}
-
A_{\mathrm{injected}}.
\]

A positive value indicates that injecting the edge led to a stronger final attack.

### Execution block

The following cell:

- activates the in-memory injection extension;
- selects one candidate from each edge category;
- constructs paired initial blocks;
- executes the baseline and injection conditions;
- records the complete PR-BCD trajectories;
- records whether the injected edge survives and is selected in the final attack.

In [ ]:
# %% [EXECUTION CELL]
# ============================================================
# RQ2 multi-edge missed-candidate injection experiment
# ============================================================


from datetime import datetime
from pathlib import Path
from timeit import default_timer as timer
from IPython.display import display

import ast
import gc
import importlib
import inspect
import json
import random
import re
import textwrap
import warnings

import numpy as np
import pandas as pd
import torch

from experiments import experiment_global_attack_direct
import rgnn_at_scale.attacks as _rq2_attacks_package
import rgnn_at_scale.attacks.prbcd as _rq2_prbcd_module


# ============================================================
# In-memory PRBCD injection overlay
#
# The overlay supports one or several linear edge IDs. At the
# selected epoch, it replaces the same number of lowest-weight
# coordinates and initializes the inserted coordinates at eps.
# ============================================================

if (
    getattr(
        _rq2_prbcd_module.PRBCD,
        "_rq2_multi_edge_injection_overlay",
        False,
    )
    or getattr(
        _rq2_prbcd_module.PRBCD,
        "_rq2_single_edge_injection_overlay",
        False,
    )
):
    # The earlier single-edge overlay already accepts a list of IDs,
    # so it can be reused directly for the multi-edge intervention.
    RQ2InjectionPRBCD = _rq2_prbcd_module.PRBCD
    RQ2InjectionPRBCD._rq2_multi_edge_injection_overlay = True

else:
    _RQ2BasePRBCD = _rq2_prbcd_module.PRBCD

    _required_base_args = {
        "initial_block_path",
        "initial_block_label",
        "resampling_enabled",
        "block_diagnostics_enabled",
        "attack_sampling_seed",
    }
    _base_args = set(
        inspect.signature(_RQ2BasePRBCD.__init__).parameters
    )
    _missing_base_args = _required_base_args - _base_args

    if _missing_base_args:
        raise RuntimeError(
            "The injection overlay must be applied on top of the "
            "RQ2 custom-block PRBCD class. Missing base arguments: "
            f"{sorted(_missing_base_args)}"
        )

    def _compile_attack_with_rq2_injection(base_cls):
        """Insert the injection hook at the start of the epoch loop."""
        source = textwrap.dedent(
            inspect.getsource(base_cls._attack)
        )
        tree = ast.parse(source)

        function = next(
            node
            for node in tree.body
            if isinstance(
                node,
                (ast.FunctionDef, ast.AsyncFunctionDef),
            )
            and node.name == "_attack"
        )

        inserted = False

        for node in ast.walk(function):
            if not isinstance(node, ast.For):
                continue

            has_epoch_range = any(
                isinstance(child, ast.Attribute)
                and isinstance(child.value, ast.Name)
                and child.value.id == "self"
                and child.attr == "epochs"
                for child in ast.walk(node.iter)
            )

            if not has_epoch_range:
                continue

            if not isinstance(node.target, ast.Name):
                continue

            hook = ast.Expr(
                value=ast.Call(
                    func=ast.Attribute(
                        value=ast.Name(
                            id="self",
                            ctx=ast.Load(),
                        ),
                        attr="_rq2_apply_edge_injection",
                        ctx=ast.Load(),
                    ),
                    args=[],
                    keywords=[
                        ast.keyword(
                            arg="epoch",
                            value=ast.Name(
                                id=node.target.id,
                                ctx=ast.Load(),
                            ),
                        )
                    ],
                )
            )

            node.body.insert(0, hook)
            inserted = True
            break

        if not inserted:
            raise RuntimeError(
                "Could not identify the PRBCD epoch loop for "
                "the RQ2 injection hook."
            )

        ast.fix_missing_locations(tree)

        module_globals = dict(
            vars(importlib.import_module(base_cls.__module__))
        )
        namespace = {}

        exec(
            compile(
                tree,
                filename=(
                    inspect.getsourcefile(base_cls)
                    or "<rq2_multi_edge_injection>"
                ),
                mode="exec",
            ),
            module_globals,
            namespace,
        )

        return namespace["_attack"]

    class RQ2InjectionPRBCD(_RQ2BasePRBCD):
        _rq2_multi_edge_injection_overlay = True
        _rq2_single_edge_injection_overlay = True

        def __init__(
            self,
            *args,
            initial_block_path=None,
            initial_block_label=None,
            resampling_enabled=True,
            block_diagnostics_enabled=False,
            attack_sampling_seed=None,
            rq2_injection_linear_ids=None,
            rq2_injection_epoch=-1,
            rq2_injection_initial_weight=None,
            **kwargs,
        ):
            super().__init__(
                *args,
                initial_block_path=initial_block_path,
                initial_block_label=initial_block_label,
                resampling_enabled=resampling_enabled,
                block_diagnostics_enabled=block_diagnostics_enabled,
                attack_sampling_seed=attack_sampling_seed,
                **kwargs,
            )

            self.rq2_injection_linear_ids = sorted({
                int(edge_id)
                for edge_id in (
                    rq2_injection_linear_ids or []
                )
            })
            self.rq2_injection_epoch = int(
                rq2_injection_epoch
            )
            self.rq2_injection_initial_weight = (
                None
                if rq2_injection_initial_weight is None
                else float(rq2_injection_initial_weight)
            )
            self._rq2_injection_applied = False

        @torch.no_grad()
        def _rq2_apply_edge_injection(self, epoch):
            if (
                self._rq2_injection_applied
                or not self.rq2_injection_linear_ids
                or int(epoch) != self.rq2_injection_epoch
            ):
                return

            current_ids = (
                self.current_search_space
                .detach()
                .to(self.device)
                .long()
            )
            current_weights = (
                self.perturbed_edge_weight
                .detach()
                .to(self.device)
                .float()
            )

            requested_ids = torch.tensor(
                self.rq2_injection_linear_ids,
                device=self.device,
                dtype=torch.long,
            )

            valid = (
                (requested_ids >= 0)
                & (
                    requested_ids
                    < int(self.n_possible_edges)
                )
            )
            requested_ids = torch.unique(
                requested_ids[valid],
                sorted=True,
            )

            present_mask = torch.isin(
                requested_ids,
                current_ids,
            )
            already_present = requested_ids[present_mask]
            insert_ids = requested_ids[~present_mask]

            n_replace = min(
                int(insert_ids.numel()),
                int(current_ids.numel()),
            )
            insert_ids = insert_ids[:n_replace]

            initial_weight = (
                float(self.eps)
                if self.rq2_injection_initial_weight is None
                else float(
                    self.rq2_injection_initial_weight
                )
            )

            if n_replace > 0:
                lowest_idx = torch.argsort(
                    current_weights
                )[:n_replace]

                keep_mask = torch.ones(
                    current_ids.numel(),
                    dtype=torch.bool,
                    device=self.device,
                )
                keep_mask[lowest_idx] = False

                dropped_ids = current_ids[lowest_idx]

                new_ids = torch.cat([
                    current_ids[keep_mask],
                    insert_ids,
                ])
                new_weights = torch.cat([
                    current_weights[keep_mask],
                    torch.full(
                        (insert_ids.numel(),),
                        initial_weight,
                        device=self.device,
                        dtype=torch.float32,
                    ),
                ])

                new_ids, order = torch.sort(new_ids)
                new_weights = new_weights[order]

                self.current_search_space = new_ids
                self.perturbed_edge_weight = new_weights

                if self.make_undirected:
                    self.modified_edge_index = (
                        self.linear_to_triu_idx(
                            self.n,
                            self.current_search_space,
                        )
                    )
                else:
                    self.modified_edge_index = (
                        self.linear_to_full_idx(
                            self.n,
                            self.current_search_space,
                        )
                    )
                    keep = (
                        self.modified_edge_index[0]
                        != self.modified_edge_index[1]
                    )
                    self.current_search_space = (
                        self.current_search_space[keep]
                    )
                    self.modified_edge_index = (
                        self.modified_edge_index[:, keep]
                    )
                    self.perturbed_edge_weight = (
                        self.perturbed_edge_weight[keep]
                    )

                if getattr(self, "tried_mask", None) is not None:
                    self.tried_mask[insert_ids] = True

            else:
                dropped_ids = torch.empty(
                    0,
                    device=self.device,
                    dtype=torch.long,
                )

            diagnostics = self.attack_statistics.setdefault(
                "block_diagnostics",
                {},
            )
            diagnostics.setdefault(
                "injection_events",
                [],
            ).append({
                "epoch": int(epoch),
                "requested_ids": (
                    requested_ids.detach().cpu().clone()
                ),
                "inserted_ids": (
                    insert_ids.detach().cpu().clone()
                ),
                "already_present_ids": (
                    already_present.detach().cpu().clone()
                ),
                "dropped_ids": (
                    dropped_ids.detach().cpu().clone()
                ),
                "initial_weight": float(initial_weight),
                "block_size_after": int(
                    self.current_search_space.numel()
                ),
            })

            diagnostics[
                "injection_requested_linear_ids"
            ] = requested_ids.detach().cpu().clone()
            diagnostics["injection_epoch"] = int(epoch)

            self._rq2_injection_applied = True

    RQ2InjectionPRBCD._attack = (
        _compile_attack_with_rq2_injection(
            _RQ2BasePRBCD
        )
    )
    RQ2InjectionPRBCD._attack.__qualname__ = (
        "RQ2InjectionPRBCD._attack"
    )

    if not hasattr(
        _rq2_attacks_package,
        "_rq2_injection_original_create_attack",
    ):
        _rq2_attacks_package._rq2_injection_original_create_attack = (
            _rq2_attacks_package.create_attack
        )

    _rq2_original_create_attack = (
        _rq2_attacks_package
        ._rq2_injection_original_create_attack
    )

    def _rq2_injection_create_attack(attack, **kwargs):
        if str(attack).lower() == "prbcd":
            return RQ2InjectionPRBCD(**kwargs)
        return _rq2_original_create_attack(
            attack,
            **kwargs,
        )

    _rq2_prbcd_module.PRBCD = RQ2InjectionPRBCD
    _rq2_attacks_package.PRBCD = RQ2InjectionPRBCD
    _rq2_attacks_package.create_attack = (
        _rq2_injection_create_attack
    )
    experiment_global_attack_direct.create_attack = (
        _rq2_injection_create_attack
    )


PRBCD = RQ2InjectionPRBCD

_required_injection_args = {
    "rq2_injection_linear_ids",
    "rq2_injection_epoch",
    "rq2_injection_initial_weight",
}
_missing_injection_args = (
    _required_injection_args
    - set(inspect.signature(PRBCD.__init__).parameters)
)

if _missing_injection_args:
    raise RuntimeError(
        "RQ2 injection overlay is incomplete: "
        f"{sorted(_missing_injection_args)}"
    )

print("RQ2 multi-edge injection overlay active:", PRBCD)


# ============================================================
# Standalone injection-experiment configuration
#
# These values are intentionally independent of the balanced
# fixed-block experiment. The only prerequisite experiment is the
# direct-mining stage (`mining_runs`). The graph and victim-model
# identifiers come from the common RQ2 dataset setup.
# ============================================================

_required_rq2_injection_context = {
    "DATASET",
    "MODEL_LABEL",
    "graph_sparse_rq2",
    "mining_runs",
}
_missing_rq2_injection_context = [
    name
    for name in sorted(_required_rq2_injection_context)
    if name not in globals()
]
if _missing_rq2_injection_context:
    raise RuntimeError(
        "Run the common RQ2 dataset/model setup and direct-mining "
        "cells first. Missing: "
        f"{_missing_rq2_injection_context}"
    )

RQ2_INJECTION_RUN_PHASE = True

# Dataset/model execution settings owned by this experiment.
RQ2_INJECTION_DATASET = DATASET
RQ2_INJECTION_MODEL_LABEL = MODEL_LABEL
RQ2_INJECTION_DATA_DIR = "./data"
RQ2_INJECTION_ARTIFACT_DIR = "cache"
RQ2_INJECTION_MODEL_STORAGE_TYPE = "demo_custom_split"
RQ2_INJECTION_PERT_ADJ_STORAGE_TYPE = "evasion_global_adj"
RQ2_INJECTION_PERT_ATTR_STORAGE_TYPE = "evasion_global_attr"
RQ2_INJECTION_DEVICE = "cpu"
RQ2_INJECTION_DATA_DEVICE = "cpu"
RQ2_INJECTION_DEBUG_LEVEL = "info"
RQ2_INJECTION_USE_CERT = "none"
RQ2_INJECTION_BINARY_ATTR = False
RQ2_INJECTION_MAKE_UNDIRECTED = True
RQ2_INJECTION_SEMI = True

# PRBCD hyperparameters owned by this experiment.
RQ2_INJECTION_EPSILON = 0.05
RQ2_INJECTION_EPOCHS = 100
RQ2_INJECTION_RESAMPLING_EPOCHS = 50
RQ2_INJECTION_FINE_TUNE_EPOCHS = (
    RQ2_INJECTION_EPOCHS
    - RQ2_INJECTION_RESAMPLING_EPOCHS
)
RQ2_INJECTION_WITH_EARLY_STOPPING = False
RQ2_INJECTION_BLOCK_SIZE = 1_000
RQ2_INJECTION_REPEATS = 3
RQ2_INJECTION_RESAMPLING_ENABLED = True
RQ2_INJECTION_INITIAL_WEIGHT = None  # None -> PRBCD epsilon

# Same intervention size as the RQ1 experiment, but defined
# locally so this cell does not depend on an RQ1 variable.
RQ2_INJECTION_N_EDGES = 50

# Inject halfway through this experiment's own resampling phase.
RQ2_INJECTION_EPOCH = max(
    0,
    RQ2_INJECTION_RESAMPLING_EPOCHS // 2,
)

# Direct-mining filters owned by this experiment.
RQ2_INJECTION_LABEL_THRESHOLD = 0.5
RQ2_INJECTION_MIN_ACCURACY_DROP = 0.0
RQ2_INJECTION_HOPS = None  # e.g. {2}; None uses all mined hops
RQ2_INJECTION_SEEDS = None  # e.g. {0, 1, 2}; None uses all mined seeds
RQ2_INJECTION_CANDIDATE_CONFIG_IDS = None  # e.g. {"cfg_name"}

# Random controls are unseen by the baseline and optionally also
# excluded from every directly mined candidate set.
RQ2_INJECTION_RANDOM_EXCLUDE_ALL_MINED = True

# Require the complete, equally sized intervention for every
# retained condition. Incomplete paired groups are skipped.
RQ2_INJECTION_REQUIRE_EXACT_N = True

# Output owned by this experiment.
RQ2_INJECTION_BASE_OUT_DIR = (
    Path("extendedPlotting")
    / "rq2_missed_multi_edge_injection"
)
RQ2_INJECTION_RUN_ID = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)
RQ2_INJECTION_OUT_DIR = (
    RQ2_INJECTION_BASE_OUT_DIR
    / f"{RQ2_INJECTION_DATASET}__{RQ2_INJECTION_RUN_ID}"
)
RQ2_INJECTION_BLOCK_DIR = (
    RQ2_INJECTION_OUT_DIR / "initial_blocks"
)

RQ2_INJECTION_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
RQ2_INJECTION_BLOCK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
(
    RQ2_INJECTION_BASE_OUT_DIR
    / "latest_run.txt"
).write_text(
    str(RQ2_INJECTION_OUT_DIR.resolve()),
    encoding="utf-8",
)

# Graph-size and attack-budget constants are computed locally.
RQ2_INJECTION_N_NODES = int(
    graph_sparse_rq2.attr_matrix.shape[0]
)
RQ2_INJECTION_N_POSSIBLE = (
    RQ2_INJECTION_N_NODES
    * (RQ2_INJECTION_N_NODES - 1)
    // 2
)
RQ2_INJECTION_N_UNDIRECTED = int(
    graph_sparse_rq2.adj_matrix.nnz // 2
)
RQ2_INJECTION_ATTACK_BUDGET = max(
    1,
    round(
        RQ2_INJECTION_EPSILON
        * RQ2_INJECTION_N_UNDIRECTED
    ),
)

# Validate only this experiment's configuration.
if not 0 <= RQ2_INJECTION_RESAMPLING_EPOCHS <= RQ2_INJECTION_EPOCHS:
    raise ValueError(
        "RQ2_INJECTION_RESAMPLING_EPOCHS must be in "
        "[0, RQ2_INJECTION_EPOCHS]."
    )
if RQ2_INJECTION_FINE_TUNE_EPOCHS < 0:
    raise ValueError(
        "RQ2_INJECTION_FINE_TUNE_EPOCHS must be non-negative."
    )
if RQ2_INJECTION_REPEATS < 1:
    raise ValueError(
        "RQ2_INJECTION_REPEATS must be at least 1."
    )
if RQ2_INJECTION_N_EDGES <= 0:
    raise ValueError(
        "RQ2_INJECTION_N_EDGES must be positive."
    )
if RQ2_INJECTION_BLOCK_SIZE <= RQ2_INJECTION_ATTACK_BUDGET:
    raise ValueError(
        "RQ2_INJECTION_BLOCK_SIZE must exceed this experiment's "
        f"attack budget ({RQ2_INJECTION_ATTACK_BUDGET})."
    )
if not 0 <= RQ2_INJECTION_EPOCH < RQ2_INJECTION_EPOCHS:
    raise ValueError(
        "RQ2_INJECTION_EPOCH must be a valid PRBCD epoch."
    )
if (
    RQ2_INJECTION_RESAMPLING_ENABLED
    and RQ2_INJECTION_EPOCH
    >= RQ2_INJECTION_RESAMPLING_EPOCHS
):
    warnings.warn(
        "The injection epoch lies outside the nominal resampling "
        "phase. The intervention will occur during fine tuning."
    )

print(
    "Standalone RQ2 injection configuration:",
    f"epsilon={RQ2_INJECTION_EPSILON}",
    f"attack_budget={RQ2_INJECTION_ATTACK_BUDGET}",
    f"block_size={RQ2_INJECTION_BLOCK_SIZE}",
    f"epochs={RQ2_INJECTION_EPOCHS}",
    f"resampling_epochs={RQ2_INJECTION_RESAMPLING_EPOCHS}",
    f"injection_epoch={RQ2_INJECTION_EPOCH}",
    f"n_injected={RQ2_INJECTION_N_EDGES}",
    f"repeats={RQ2_INJECTION_REPEATS}",
)


# ============================================================
# Standalone utility helpers
# ============================================================


def _rq2_injection_set_global_seed(
    seed,
    deterministic=True,
):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic and hasattr(torch, "use_deterministic_algorithms"):
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = bool(deterministic)
        torch.backends.cudnn.benchmark = not bool(deterministic)


def _rq2_safe(value):
    return re.sub(
        r"[^a-zA-Z0-9_.=-]+",
        "_",
        str(value),
    ).strip("_")


def _as_list(value):
    if value is None:
        return []
    if torch.is_tensor(value):
        value = value.detach().cpu()
        return (
            [value.item()]
            if value.ndim == 0
            else value.reshape(-1).tolist()
        )
    if isinstance(value, np.ndarray):
        return value.reshape(-1).tolist()
    if isinstance(value, (list, tuple)):
        return list(value)
    return [value]


def _find_attack_statistics(obj, seen=None):
    if seen is None:
        seen = set()

    obj_id = id(obj)
    if obj_id in seen:
        return None
    seen.add(obj_id)

    if isinstance(obj, dict):
        for key in (
            "attack_statistics",
            "attack_stats",
            "stats",
        ):
            candidate = obj.get(key)
            if isinstance(candidate, dict) and (
                isinstance(
                    candidate.get("accuracy"),
                    (list, tuple, np.ndarray, torch.Tensor),
                )
                or isinstance(
                    candidate.get("loss"),
                    (list, tuple, np.ndarray, torch.Tensor),
                )
            ):
                return candidate

        if isinstance(
            obj.get("accuracy"),
            (list, tuple, np.ndarray, torch.Tensor),
        ):
            return obj

        for value in obj.values():
            found = _find_attack_statistics(value, seen)
            if found is not None:
                return found

    elif isinstance(obj, (list, tuple)):
        for value in obj:
            found = _find_attack_statistics(value, seen)
            if found is not None:
                return found

    return None


def _extract_final_accuracy(result):
    if isinstance(result, dict):
        rows = result.get("results", []) or []
        if (
            rows
            and isinstance(rows[0], dict)
            and "accuracy" in rows[0]
        ):
            return float(
                torch.as_tensor(rows[0]["accuracy"])
                .detach()
                .cpu()
                .item()
            )

    raise KeyError(
        "Could not extract final attacked accuracy from result."
    )


def _sample_random_linear_block(
    n_possible,
    count,
    *,
    forbidden,
    seed,
):
    """Sample unique linear IDs without materializing all pairs."""
    forbidden = {int(value) for value in forbidden}

    if int(count) > int(n_possible) - len(forbidden):
        raise ValueError(
            "Not enough non-forbidden pairs for random sampling."
        )

    rng = np.random.default_rng(int(seed))
    selected = set()

    while len(selected) < int(count):
        remaining = int(count) - len(selected)
        batch_size = max(4096, 3 * remaining)
        draws = rng.integers(
            0,
            int(n_possible),
            size=batch_size,
            endpoint=False,
        )

        for value in draws.tolist():
            value = int(value)
            if value in forbidden or value in selected:
                continue
            selected.add(value)
            if len(selected) == int(count):
                break

    return torch.tensor(
        sorted(selected),
        dtype=torch.long,
    )


def _save_block(path, linear_ids, metadata):
    linear_ids = torch.unique(
        torch.as_tensor(
            linear_ids,
            dtype=torch.long,
        ),
        sorted=True,
    )
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "linear_ids": linear_ids,
            "metadata": dict(metadata),
        },
        path,
    )
    return path.resolve()


def _two_stage_summary(frame, group_cols, metric_cols):
    """Average repeats per seed, then aggregate across seeds."""
    if frame is None or frame.empty:
        return pd.DataFrame()

    available_metrics = [
        metric
        for metric in metric_cols
        if metric in frame.columns
    ]
    if not available_metrics:
        return pd.DataFrame()

    work = frame.copy()
    for metric in available_metrics:
        work[metric] = pd.to_numeric(
            work[metric],
            errors="coerce",
        )

    seed_level = (
        work
        .groupby(
            group_cols + ["seed"],
            dropna=False,
        )[available_metrics]
        .mean()
        .reset_index()
    )

    grouped = seed_level.groupby(
        group_cols,
        dropna=False,
    )
    summary = grouped[available_metrics].agg(
        ["mean", "std", "count"]
    )
    summary.columns = [
        f"{metric}_{stat}"
        for metric, stat in summary.columns
    ]
    summary = summary.reset_index()

    for metric in available_metrics:
        std_column = f"{metric}_std"
        count_column = f"{metric}_count"
        summary[std_column] = summary[std_column].fillna(0.0)
        summary[f"{metric}_sem"] = (
            summary[std_column]
            / np.sqrt(
                summary[count_column].clip(lower=1)
            )
        )

    n_seeds = (
        seed_level
        .groupby(group_cols, dropna=False)["seed"]
        .nunique()
        .rename("n_seeds")
        .reset_index()
    )

    return summary.merge(
        n_seeds,
        on=group_cols,
        how="left",
        validate="one_to_one",
    )


# ============================================================
# Helpers
# ============================================================


def _rq2_to_long_tensor(value):
    if value is None:
        return torch.empty(0, dtype=torch.long)
    return (
        torch.as_tensor(value, dtype=torch.long)
        .detach()
        .cpu()
        .flatten()
    )


def _rq2_get_epoch_block(diagnostics, epoch):
    epoch_blocks = diagnostics.get("epoch_blocks", {}) or {}

    if isinstance(epoch_blocks, dict):
        block = epoch_blocks.get(epoch)
        if block is None:
            block = epoch_blocks.get(str(epoch))
    else:
        block = (
            epoch_blocks[epoch]
            if 0 <= epoch < len(epoch_blocks)
            else None
        )

    return _rq2_to_long_tensor(block)


def _rq2_ever_seen_edges(diagnostics, *, fallback_initial=None):
    """Union of every edge that appeared in the baseline block."""
    seen = set()

    initial = diagnostics.get("initial_block")
    if initial is None:
        initial = fallback_initial
    seen.update(_rq2_to_long_tensor(initial).tolist())

    epoch_blocks = diagnostics.get("epoch_blocks", {}) or {}
    blocks = (
        epoch_blocks.values()
        if isinstance(epoch_blocks, dict)
        else epoch_blocks
    )
    for block in blocks:
        seen.update(_rq2_to_long_tensor(block).tolist())

    for event_key in [
        "resample_events",
        "resampling_events",
    ]:
        for event in diagnostics.get(event_key, []) or []:
            if not isinstance(event, dict):
                continue
            for after_key in [
                "after",
                "after_ids",
                "block_after",
            ]:
                if event.get(after_key) is not None:
                    seen.update(
                        _rq2_to_long_tensor(
                            event[after_key]
                        ).tolist()
                    )

    final_block = diagnostics.get("final_block")
    seen.update(_rq2_to_long_tensor(final_block).tolist())

    return {int(edge_id) for edge_id in seen}


def _rq2_score_table(run):
    """Return one maximum mining score per undirected linear ID."""
    src = run["src"].detach().cpu().long().flatten()
    dst = run["dst"].detach().cpu().long().flatten()
    scores = (
        run["labels_changed"]
        .detach()
        .cpu()
        .float()
        .flatten()
    )

    if not (
        src.numel() == dst.numel() == scores.numel()
    ):
        raise ValueError(
            "src, dst and labels_changed must have equal length."
        )

    u = torch.minimum(src, dst)
    v = torch.maximum(src, dst)
    valid = u < v

    ids = PRBCD.triu_idx_to_linear_idx(
        RQ2_INJECTION_N_NODES,
        torch.stack([u[valid], v[valid]], dim=0),
    ).detach().cpu().long()

    table = pd.DataFrame({
        "linear_id": ids.numpy(),
        "mining_score": scores[valid].numpy(),
    })

    if table.empty:
        return table

    return (
        table
        .groupby("linear_id", as_index=False)
        .agg(mining_score=("mining_score", "max"))
        .sort_values(
            ["mining_score", "linear_id"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )


def _rq2_filter_missed(table, baseline_seen):
    if table is None or table.empty:
        return pd.DataFrame(
            columns=["linear_id", "mining_score"]
        )

    return table[
        ~table["linear_id"].astype(int).isin(
            baseline_seen
        )
    ].copy()


def _rq2_sample_exact_ids(table, n, *, seed):
    if len(table) < n:
        return []

    rng = np.random.default_rng(int(seed))
    chosen_positions = rng.choice(
        len(table),
        size=int(n),
        replace=False,
    )

    return (
        table.iloc[chosen_positions]["linear_id"]
        .astype(int)
        .tolist()
    )


def _rq2_top_exact_ids(table, n):
    if len(table) < n:
        return []

    return (
        table
        .sort_values(
            ["mining_score", "linear_id"],
            ascending=[False, True],
        )
        .head(int(n))["linear_id"]
        .astype(int)
        .tolist()
    )


def _rq2_score_lookup(table):
    if table is None or table.empty:
        return {}

    return {
        int(row.linear_id): float(row.mining_score)
        for row in table.itertuples(index=False)
    }


def _rq2_injection_diagnostics(result):
    stats = _find_attack_statistics(result)
    if not stats:
        raise RuntimeError(
            "PRBCD attack statistics were not found."
        )

    diagnostics = (
        stats.get("block_diagnostics", {}) or {}
    )
    return stats, diagnostics


def _rq2_run_injection_attack(
    *,
    victim_seed,
    condition,
    initial_block_path,
    attack_sampling_seed,
    injection_ids,
):
    attack_params = {
        "block_size": int(RQ2_INJECTION_BLOCK_SIZE),
        "epochs": int(RQ2_INJECTION_EPOCHS),
        "fine_tune_epochs": int(
            RQ2_INJECTION_FINE_TUNE_EPOCHS
        ),
        "with_early_stopping": bool(
            RQ2_INJECTION_WITH_EARLY_STOPPING
        ),
        "keep_heuristic": "WeightOnly",
        "do_synchronize": True,
        "loss_type": "tanhMargin",
        "initial_block_path": str(initial_block_path),
        "initial_block_label": (
            f"rq2_missed_injection__{condition}"
        ),
        "resampling_enabled": bool(
            RQ2_INJECTION_RESAMPLING_ENABLED
        ),
        "block_diagnostics_enabled": True,
        "attack_sampling_seed": int(
            attack_sampling_seed
        ),
        "rq2_injection_linear_ids": [
            int(edge_id)
            for edge_id in injection_ids
        ],
        "rq2_injection_epoch": int(
            RQ2_INJECTION_EPOCH
        ),
        "rq2_injection_initial_weight": (
            RQ2_INJECTION_INITIAL_WEIGHT
        ),
    }

    _rq2_injection_set_global_seed(int(attack_sampling_seed))
    started = timer()

    result = experiment_global_attack_direct.run(
        graph=graph_sparse_rq2,
        data_dir=RQ2_INJECTION_DATA_DIR,
        dataset=RQ2_INJECTION_DATASET,
        attack="PRBCD",
        attack_params=attack_params,
        selector_params={},
        epsilons=[RQ2_INJECTION_EPSILON],
        binary_attr=RQ2_INJECTION_BINARY_ATTR,
        make_undirected=RQ2_INJECTION_MAKE_UNDIRECTED,
        seed=int(victim_seed),
        artifact_dir=RQ2_INJECTION_ARTIFACT_DIR,
        pert_adj_storage_type=(
            RQ2_INJECTION_PERT_ADJ_STORAGE_TYPE
        ),
        pert_attr_storage_type=(
            RQ2_INJECTION_PERT_ATTR_STORAGE_TYPE
        ),
        model_label=RQ2_INJECTION_MODEL_LABEL,
        model_storage_type=(
            RQ2_INJECTION_MODEL_STORAGE_TYPE
        ),
        device=RQ2_INJECTION_DEVICE,
        data_device=RQ2_INJECTION_DEVICE,
        debug_level=RQ2_INJECTION_DEBUG_LEVEL,
        semi=RQ2_INJECTION_SEMI,
        use_cert=RQ2_INJECTION_USE_CERT,
    )

    runtime_seconds = timer() - started
    return result, float(runtime_seconds)


def _rq2_extract_run_payload(
    result,
    *,
    runtime_seconds,
    injection_ids,
):
    stats, diagnostics = _rq2_injection_diagnostics(result)

    accuracies = np.asarray(
        _as_list(stats.get("accuracy")),
        dtype=float,
    )
    losses = np.asarray(
        _as_list(stats.get("loss")),
        dtype=float,
    )

    if accuracies.size == 0:
        raise RuntimeError(
            "Injection run contains no accuracy trajectory."
        )

    clean_accuracy = float(accuracies[0])
    final_accuracy = float(
        _extract_final_accuracy(result)
    )

    relaxed_accuracies = (
        accuracies[1:]
        if accuracies.size > 1
        else accuracies
    )

    integrated_drop = float(
        np.trapz(
            clean_accuracy - accuracies,
            dx=1.0,
        )
        / max(1, accuracies.size - 1)
    )

    injection_events = (
        diagnostics.get("injection_events", []) or []
    )
    event = injection_events[0] if injection_events else {}

    requested = set(
        _rq2_to_long_tensor(
            event.get("requested_ids", injection_ids)
        ).tolist()
    )
    inserted = set(
        _rq2_to_long_tensor(
            event.get("inserted_ids")
        ).tolist()
    )
    already_present = set(
        _rq2_to_long_tensor(
            event.get("already_present_ids")
        ).tolist()
    )
    dropped = _rq2_to_long_tensor(
        event.get("dropped_ids")
    )

    final_ids = set(
        _rq2_to_long_tensor(
            diagnostics.get("final_linear_ids")
        ).tolist()
    )
    final_block = set(
        _rq2_to_long_tensor(
            diagnostics.get("final_block")
        ).tolist()
    )

    requested_ids = {
        int(edge_id) for edge_id in injection_ids
    }

    return {
        "stats": stats,
        "diagnostics": diagnostics,
        "accuracies": accuracies,
        "losses": losses,
        "clean_accuracy": clean_accuracy,
        "final_accuracy": final_accuracy,
        "final_accuracy_drop": (
            clean_accuracy - final_accuracy
        ),
        "minimum_relaxed_accuracy": float(
            np.nanmin(relaxed_accuracies)
        ),
        "maximum_relaxed_accuracy_drop": (
            clean_accuracy
            - float(np.nanmin(relaxed_accuracies))
        ),
        "integrated_accuracy_drop": integrated_drop,
        "runtime_seconds": float(runtime_seconds),
        "injection_event_recorded": bool(
            injection_events
        ),
        "requested_ids": requested,
        "inserted_ids": inserted,
        "already_present_ids": already_present,
        "dropped_ids": dropped,
        "final_ids": final_ids,
        "final_block": final_block,
        "n_requested": len(requested_ids),
        "n_inserted": len(requested_ids & inserted),
        "n_already_present": len(
            requested_ids & already_present
        ),
        "n_in_final_block": len(
            requested_ids & final_block
        ),
        "n_finally_selected": len(
            requested_ids & final_ids
        ),
    }


def _rq2_append_result_rows(
    *,
    payload,
    common,
    injection_ids,
    mining_score_lookup,
    run_rows,
    epoch_rows,
    edge_rows,
):
    injection_ids = [int(x) for x in injection_ids]
    injection_set = set(injection_ids)
    n_injected = len(injection_ids)

    run_rows.append({
        **common,
        "n_injected": int(n_injected),
        "clean_accuracy": payload["clean_accuracy"],
        "final_accuracy": payload["final_accuracy"],
        "final_accuracy_drop": (
            payload["final_accuracy_drop"]
        ),
        "minimum_relaxed_accuracy": (
            payload["minimum_relaxed_accuracy"]
        ),
        "maximum_relaxed_accuracy_drop": (
            payload["maximum_relaxed_accuracy_drop"]
        ),
        "integrated_accuracy_drop": (
            payload["integrated_accuracy_drop"]
        ),
        "runtime_seconds": payload["runtime_seconds"],
        "injection_event_recorded": (
            payload["injection_event_recorded"]
        ),
        "n_inserted": int(payload["n_inserted"]),
        "n_already_present": int(
            payload["n_already_present"]
        ),
        "n_dropped_for_injection": int(
            payload["dropped_ids"].numel()
        ),
        "all_requested_inserted": bool(
            n_injected == 0
            or payload["n_inserted"] == n_injected
        ),
        "n_injected_in_final_block": int(
            payload["n_in_final_block"]
        ),
        "injected_fraction_in_final_block": (
            payload["n_in_final_block"] / n_injected
            if n_injected > 0
            else np.nan
        ),
        "n_injected_finally_selected": int(
            payload["n_finally_selected"]
        ),
        "injected_fraction_finally_selected": (
            payload["n_finally_selected"] / n_injected
            if n_injected > 0
            else np.nan
        ),
    })

    diagnostics = payload["diagnostics"]
    accuracies = payload["accuracies"]
    losses = payload["losses"]
    clean_accuracy = payload["clean_accuracy"]

    for stat_index, accuracy_value in enumerate(
        accuracies
    ):
        prbcd_epoch = int(stat_index - 1)
        current_block = _rq2_get_epoch_block(
            diagnostics,
            prbcd_epoch,
        )
        current_set = set(current_block.tolist())

        n_present = len(injection_set & current_set)

        epoch_rows.append({
            **common,
            "n_injected": int(n_injected),
            "stat_index": int(stat_index),
            "prbcd_epoch": prbcd_epoch,
            "is_clean_baseline": bool(
                stat_index == 0
            ),
            "accuracy": float(accuracy_value),
            "accuracy_drop": (
                clean_accuracy - float(accuracy_value)
            ),
            "loss": (
                float(losses[stat_index])
                if stat_index < losses.size
                else np.nan
            ),
            "n_injected_edges_present": int(n_present),
            "injected_edge_retention_fraction": (
                n_present / n_injected
                if n_injected > 0
                else np.nan
            ),
            "all_injected_edges_present": (
                float(n_present == n_injected)
                if n_injected > 0
                else np.nan
            ),
        })

    for edge_id in injection_ids:
        pair = PRBCD.linear_to_triu_idx(
            RQ2_INJECTION_N_NODES,
            torch.tensor(
                [int(edge_id)],
                dtype=torch.long,
            ),
        ).cpu()

        edge_rows.append({
            **common,
            "linear_id": int(edge_id),
            "u": int(pair[0, 0].item()),
            "v": int(pair[1, 0].item()),
            "mining_score": float(
                mining_score_lookup.get(
                    int(edge_id),
                    np.nan,
                )
            ),
            "edge_inserted": bool(
                int(edge_id)
                in payload["inserted_ids"]
            ),
            "edge_already_present": bool(
                int(edge_id)
                in payload["already_present_ids"]
            ),
            "edge_in_final_block": bool(
                int(edge_id)
                in payload["final_block"]
            ),
            "edge_finally_selected": bool(
                int(edge_id)
                in payload["final_ids"]
            ),
        })


# ============================================================
# Mining runs used for the intervention pools
#
# Works with:
#   - endpoint-only mining runs
#   - subset_accuracy_drop-only mining runs
#   - both modes together
# ============================================================

def _rq2_injection_run_is_selected(run):
    if (
        RQ2_INJECTION_SEEDS is not None
        and int(run["seed"]) not in RQ2_INJECTION_SEEDS
    ):
        return False

    if (
        RQ2_INJECTION_CANDIDATE_CONFIG_IDS is not None
        and run["candidate_config_id"]
        not in RQ2_INJECTION_CANDIDATE_CONFIG_IDS
    ):
        return False

    # Hop filtering belongs only to endpoint mining. Subset mining has
    # no meaningful endpoint hop and must remain usable on its own.
    if str(run["scoring_mode"]) == "endpoint":
        hop = run.get("endpoint_mining_hop", 0)
        if (
            RQ2_INJECTION_HOPS is not None
            and hop not in RQ2_INJECTION_HOPS
        ):
            return False

    return True


def _rq2_injection_endpoint_hop(run):
    if run is None or str(run.get("scoring_mode")) != "endpoint":
        return 0

    value = run.get("endpoint_mining_hop", 0)
    if value is None or pd.isna(value):
        return 0
    return int(value)


endpoint_runs = {
    (int(run["seed"]), run["candidate_config_id"]): run
    for run in mining_runs
    if str(run["scoring_mode"]) == "endpoint"
    and _rq2_injection_run_is_selected(run)
}

accuracy_drop_runs = {
    (int(run["seed"]), run["candidate_config_id"]): run
    for run in mining_runs
    if str(run["scoring_mode"]) == "subset_accuracy_drop"
    and _rq2_injection_run_is_selected(run)
}

source_run_keys = sorted(
    set(endpoint_runs) | set(accuracy_drop_runs)
)

if not source_run_keys:
    available_modes = sorted({
        str(run.get("scoring_mode"))
        for run in mining_runs
    })
    raise RuntimeError(
        "No endpoint or subset_accuracy_drop mining runs matched the "
        "injection filters. Available modes: "
        f"{available_modes}"
    )

if not endpoint_runs:
    warnings.warn(
        "No endpoint mining runs are loaded. Endpoint-label injection "
        "conditions will be skipped; subset_accuracy_drop conditions "
        "and the random unseen control will still run."
    )

if not accuracy_drop_runs:
    warnings.warn(
        "No subset_accuracy_drop mining runs are loaded. "
        "The accuracy-drop injection condition will be skipped."
    )


# ============================================================
# Execute baseline first, then select only baseline-missed edges
# ============================================================

rq2_injection_run_rows = []
rq2_injection_epoch_rows = []
rq2_injection_edge_rows = []
rq2_injection_selection_rows = []
rq2_injection_skip_rows = []

if RQ2_INJECTION_RUN_PHASE:
    for source_index, key in enumerate(source_run_keys):
        victim_seed, candidate_config_id = key
        endpoint_run = endpoint_runs.get(key)
        accuracy_run = accuracy_drop_runs.get(key)

        # Any available mode can anchor the baseline run metadata.
        reference_run = (
            endpoint_run
            if endpoint_run is not None
            else accuracy_run
        )
        if reference_run is None:
            continue

        endpoint_table = (
            _rq2_score_table(endpoint_run)
            if endpoint_run is not None
            else pd.DataFrame(
                columns=["linear_id", "mining_score"]
            )
        )
        accuracy_table = (
            _rq2_score_table(accuracy_run)
            if accuracy_run is not None
            else pd.DataFrame(
                columns=["linear_id", "mining_score"]
            )
        )

        all_mined_ids = set(
            endpoint_table["linear_id"]
            .astype(int)
            .tolist()
        )
        all_mined_ids.update(
            accuracy_table["linear_id"]
            .astype(int)
            .tolist()
        )

        for repeat in range(RQ2_INJECTION_REPEATS):
            selection_seed = (
                810_000
                + int(victim_seed) * 10_000
                + source_index * 100
                + int(repeat) * 10
            )
            initial_block_seed = selection_seed + 3
            attack_sampling_seed = (
                910_000
                + int(victim_seed) * 10_000
                + source_index * 100
                + int(repeat)
            )

            # The baseline initial block is generated without knowing
            # the intervention set. Any later selected edge must be
            # absent from the complete baseline ever-seen union.
            initial_ids = _sample_random_linear_block(
                RQ2_INJECTION_N_POSSIBLE,
                RQ2_INJECTION_BLOCK_SIZE,
                forbidden=set(),
                seed=initial_block_seed,
            )

            initial_block_path = _save_block(
                RQ2_INJECTION_BLOCK_DIR / (
                    f"baseline_initial__seed-{victim_seed}__"
                    f"{_rq2_safe(candidate_config_id)}__"
                    f"repeat-{repeat}.pt"
                ),
                initial_ids,
                {
                    "dataset": RQ2_INJECTION_DATASET,
                    "victim_seed": int(victim_seed),
                    "candidate_config_id": candidate_config_id,
                    "repeat": int(repeat),
                    "block_size": int(
                        RQ2_INJECTION_BLOCK_SIZE
                    ),
                    "block_seed": int(initial_block_seed),
                    "purpose": (
                        "paired baseline and missed-edge injection"
                    ),
                },
            )

            print(
                "\nRQ2 BASELINE",
                f"seed={victim_seed} | cfg={candidate_config_id} | "
                f"repeat={repeat} | epoch={RQ2_INJECTION_EPOCH}",
            )

            baseline_result, baseline_runtime = (
                _rq2_run_injection_attack(
                    victim_seed=victim_seed,
                    condition="baseline",
                    initial_block_path=initial_block_path,
                    attack_sampling_seed=attack_sampling_seed,
                    injection_ids=[],
                )
            )
            baseline_payload = _rq2_extract_run_payload(
                baseline_result,
                runtime_seconds=baseline_runtime,
                injection_ids=[],
            )

            baseline_seen = _rq2_ever_seen_edges(
                baseline_payload["diagnostics"],
                fallback_initial=initial_ids,
            )

            endpoint_missed = _rq2_filter_missed(
                endpoint_table,
                baseline_seen,
            )
            endpoint_positive_pool = endpoint_missed[
                endpoint_missed["mining_score"]
                > float(RQ2_INJECTION_LABEL_THRESHOLD)
            ].copy()
            endpoint_zero_pool = endpoint_missed[
                endpoint_missed["mining_score"]
                <= float(RQ2_INJECTION_LABEL_THRESHOLD)
            ].copy()

            accuracy_missed = _rq2_filter_missed(
                accuracy_table,
                baseline_seen,
            )
            accuracy_positive_pool = accuracy_missed[
                accuracy_missed["mining_score"]
                > float(RQ2_INJECTION_MIN_ACCURACY_DROP)
            ].copy()

            # Only validate pools for mining modes that actually exist.
            pool_counts = {}
            if endpoint_run is not None:
                pool_counts.update({
                    "endpoint_label_1_missed": int(
                        len(endpoint_positive_pool)
                    ),
                    "endpoint_label_0_missed": int(
                        len(endpoint_zero_pool)
                    ),
                })
            if accuracy_run is not None:
                pool_counts[
                    "accuracy_drop_highest_missed"
                ] = int(len(accuracy_positive_pool))

            missing_pools = [
                condition
                for condition, count in pool_counts.items()
                if count < RQ2_INJECTION_N_EDGES
            ]

            if missing_pools and RQ2_INJECTION_REQUIRE_EXACT_N:
                message = (
                    "Insufficient baseline-missed candidates for "
                    f"seed={victim_seed}, cfg={candidate_config_id}, "
                    f"repeat={repeat}: {pool_counts}; required="
                    f"{RQ2_INJECTION_N_EDGES}."
                )
                warnings.warn(message)

                rq2_injection_skip_rows.append({
                    "seed": int(victim_seed),
                    "candidate_config_id": candidate_config_id,
                    "endpoint_mining_hop": (
                        _rq2_injection_endpoint_hop(
                            endpoint_run
                        )
                    ),
                    "repeat": int(repeat),
                    "reason": message,
                    "n_baseline_seen": int(
                        len(baseline_seen)
                    ),
                    **{
                        f"pool_{name}": int(count)
                        for name, count in pool_counts.items()
                    },
                })

                del baseline_result
                gc.collect()
                continue

            endpoint_positive_ids = (
                _rq2_sample_exact_ids(
                    endpoint_positive_pool,
                    RQ2_INJECTION_N_EDGES,
                    seed=selection_seed,
                )
                if endpoint_run is not None
                else []
            )
            endpoint_zero_ids = (
                _rq2_sample_exact_ids(
                    endpoint_zero_pool,
                    RQ2_INJECTION_N_EDGES,
                    seed=selection_seed + 1,
                )
                if endpoint_run is not None
                else []
            )
            accuracy_high_ids = (
                _rq2_top_exact_ids(
                    accuracy_positive_pool,
                    RQ2_INJECTION_N_EDGES,
                )
                if accuracy_run is not None
                else []
            )

            random_forbidden = set(baseline_seen)
            if RQ2_INJECTION_RANDOM_EXCLUDE_ALL_MINED:
                random_forbidden.update(all_mined_ids)
            random_forbidden.update(endpoint_positive_ids)
            random_forbidden.update(endpoint_zero_ids)
            random_forbidden.update(accuracy_high_ids)

            random_unseen_ids = (
                _sample_random_linear_block(
                    RQ2_INJECTION_N_POSSIBLE,
                    RQ2_INJECTION_N_EDGES,
                    forbidden=random_forbidden,
                    seed=selection_seed + 2,
                )
                .detach()
                .cpu()
                .long()
                .tolist()
            )

            conditions = [
                {
                    "condition": "baseline",
                    "candidate_type": "none",
                    "ids": [],
                    "score_table": pd.DataFrame(
                        columns=[
                            "linear_id",
                            "mining_score",
                        ]
                    ),
                },
            ]

            if endpoint_run is not None:
                conditions.extend([
                    {
                        "condition": "endpoint_label_1_missed",
                        "candidate_type": (
                            "baseline_missed_local_harmful"
                        ),
                        "ids": endpoint_positive_ids,
                        "score_table": endpoint_positive_pool,
                    },
                    {
                        "condition": "endpoint_label_0_missed",
                        "candidate_type": (
                            "baseline_missed_local_nonharmful_control"
                        ),
                        "ids": endpoint_zero_ids,
                        "score_table": endpoint_zero_pool,
                    },
                ])

            if accuracy_run is not None:
                conditions.append({
                    "condition": (
                        "accuracy_drop_highest_missed"
                    ),
                    "candidate_type": (
                        "baseline_missed_highest_global_drop"
                    ),
                    "ids": accuracy_high_ids,
                    "score_table": accuracy_positive_pool,
                })

            conditions.append({
                "condition": "random_unseen_control",
                "candidate_type": (
                    "baseline_missed_random_control"
                ),
                "ids": random_unseen_ids,
                "score_table": pd.DataFrame({
                    "linear_id": random_unseen_ids,
                    "mining_score": np.nan,
                }),
            })

            common_base = {
                "seed": int(victim_seed),
                "candidate_config_id": candidate_config_id,
                "source_modes": "+".join(
                    mode
                    for mode, run_value in (
                        ("endpoint", endpoint_run),
                        ("subset_accuracy_drop", accuracy_run),
                    )
                    if run_value is not None
                ),
                "candidate_set_size": int(
                    reference_run["candidate_set_size"]
                ),
                # Neutral value 0 keeps existing grouping/pairing code
                # compatible for subset-only experiments.
                "endpoint_mining_hop": (
                    _rq2_injection_endpoint_hop(
                        endpoint_run
                    )
                ),
                "repeat": int(repeat),
                "injection_epoch": int(
                    RQ2_INJECTION_EPOCH
                ),
                "block_size": int(
                    RQ2_INJECTION_BLOCK_SIZE
                ),
                "attack_budget": int(
                    RQ2_INJECTION_ATTACK_BUDGET
                ),
                "attack_sampling_seed": int(
                    attack_sampling_seed
                ),
                "initial_block_seed": int(
                    initial_block_seed
                ),
                "initial_block_path": str(
                    initial_block_path
                ),
                "n_baseline_seen": int(
                    len(baseline_seen)
                ),
                "baseline_coverage_fraction": float(
                    len(baseline_seen)
                    / max(1, RQ2_INJECTION_N_POSSIBLE)
                ),
                "n_endpoint_positive_missed_available": int(
                    len(endpoint_positive_pool)
                ),
                "n_endpoint_zero_missed_available": int(
                    len(endpoint_zero_pool)
                ),
                "n_accuracy_positive_missed_available": int(
                    len(accuracy_positive_pool)
                ),
            }

            # Record the already executed baseline only after the exact
            # intervention pools have passed validation.
            baseline_common = {
                **common_base,
                "condition": "baseline",
                "candidate_type": "none",
            }
            _rq2_append_result_rows(
                payload=baseline_payload,
                common=baseline_common,
                injection_ids=[],
                mining_score_lookup={},
                run_rows=rq2_injection_run_rows,
                epoch_rows=rq2_injection_epoch_rows,
                edge_rows=rq2_injection_edge_rows,
            )

            rq2_injection_selection_rows.append({
                **baseline_common,
                "linear_id": np.nan,
                "mining_score": np.nan,
                "was_missed_by_baseline": True,
                "selection_seed": int(selection_seed),
            })

            for condition_spec in conditions:
                condition = condition_spec["condition"]
                if condition == "baseline":
                    continue

                injection_ids = [
                    int(edge_id)
                    for edge_id in condition_spec["ids"]
                ]

                if (
                    RQ2_INJECTION_REQUIRE_EXACT_N
                    and len(injection_ids)
                    != RQ2_INJECTION_N_EDGES
                ):
                    raise RuntimeError(
                        f"Condition {condition} selected "
                        f"{len(injection_ids)} edges, expected "
                        f"{RQ2_INJECTION_N_EDGES}."
                    )

                if any(
                    edge_id in baseline_seen
                    for edge_id in injection_ids
                ):
                    raise RuntimeError(
                        f"Condition {condition} contains an edge "
                        "that was seen by the baseline."
                    )

                score_lookup = _rq2_score_lookup(
                    condition_spec["score_table"]
                )

                condition_common = {
                    **common_base,
                    "condition": condition,
                    "candidate_type": (
                        condition_spec["candidate_type"]
                    ),
                }

                for rank, edge_id in enumerate(
                    injection_ids,
                    start=1,
                ):
                    rq2_injection_selection_rows.append({
                        **condition_common,
                        "selection_rank": int(rank),
                        "linear_id": int(edge_id),
                        "mining_score": float(
                            score_lookup.get(
                                int(edge_id),
                                np.nan,
                            )
                        ),
                        "was_missed_by_baseline": bool(
                            edge_id not in baseline_seen
                        ),
                        "selection_seed": int(
                            selection_seed
                        ),
                    })

                print(
                    "\nRQ2 MISSED-EDGE INJECTION",
                    f"seed={victim_seed} | cfg={candidate_config_id} | "
                    f"repeat={repeat} | condition={condition} | "
                    f"epoch={RQ2_INJECTION_EPOCH} | "
                    f"n={len(injection_ids)}",
                )

                result, runtime_seconds = (
                    _rq2_run_injection_attack(
                        victim_seed=victim_seed,
                        condition=condition,
                        initial_block_path=(
                            initial_block_path
                        ),
                        attack_sampling_seed=(
                            attack_sampling_seed
                        ),
                        injection_ids=injection_ids,
                    )
                )

                payload = _rq2_extract_run_payload(
                    result,
                    runtime_seconds=runtime_seconds,
                    injection_ids=injection_ids,
                )

                if payload["n_already_present"] > 0:
                    warnings.warn(
                        f"{condition}: "
                        f"{payload['n_already_present']} selected "
                        "baseline-missed edges were already present "
                        "at the intervention epoch. Check replay "
                        "determinism."
                    )

                if (
                    payload["n_inserted"]
                    != len(injection_ids)
                ):
                    warnings.warn(
                        f"{condition}: inserted "
                        f"{payload['n_inserted']} of "
                        f"{len(injection_ids)} requested edges."
                    )

                _rq2_append_result_rows(
                    payload=payload,
                    common=condition_common,
                    injection_ids=injection_ids,
                    mining_score_lookup=score_lookup,
                    run_rows=rq2_injection_run_rows,
                    epoch_rows=rq2_injection_epoch_rows,
                    edge_rows=rq2_injection_edge_rows,
                )

                del result
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            del baseline_result
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


# ============================================================
# Build data frames and paired effects
# ============================================================

rq2_injection_runs_raw_df = pd.DataFrame(
    rq2_injection_run_rows
)
rq2_injection_epochs_raw_df = pd.DataFrame(
    rq2_injection_epoch_rows
)
rq2_injection_edges_raw_df = pd.DataFrame(
    rq2_injection_edge_rows
)
rq2_injection_selection_df = pd.DataFrame(
    rq2_injection_selection_rows
)
rq2_injection_skips_df = pd.DataFrame(
    rq2_injection_skip_rows
)

if rq2_injection_runs_raw_df.empty:
    raise RuntimeError(
        "No complete RQ2 missed-edge injection groups were "
        "completed. Inspect rq2_injection_skips_df."
    )

rq2_injection_group_cols = [
    "candidate_config_id",
    "endpoint_mining_hop",
    "condition",
    "candidate_type",
]

rq2_injection_runs_summary_df = _two_stage_summary(
    rq2_injection_runs_raw_df,
    rq2_injection_group_cols,
    [
        "final_accuracy",
        "final_accuracy_drop",
        "maximum_relaxed_accuracy_drop",
        "integrated_accuracy_drop",
        "n_inserted",
        "injected_fraction_in_final_block",
        "injected_fraction_finally_selected",
        "runtime_seconds",
    ],
)

_pair_index = [
    "seed",
    "candidate_config_id",
    "endpoint_mining_hop",
    "repeat",
]

_baseline = rq2_injection_runs_raw_df[
    rq2_injection_runs_raw_df["condition"]
    == "baseline"
][
    _pair_index
    + [
        "final_accuracy",
        "final_accuracy_drop",
        "integrated_accuracy_drop",
        "maximum_relaxed_accuracy_drop",
    ]
].rename(columns={
    "final_accuracy": "baseline_final_accuracy",
    "final_accuracy_drop": (
        "baseline_final_accuracy_drop"
    ),
    "integrated_accuracy_drop": (
        "baseline_integrated_accuracy_drop"
    ),
    "maximum_relaxed_accuracy_drop": (
        "baseline_maximum_relaxed_accuracy_drop"
    ),
})

rq2_injection_paired_raw_df = (
    rq2_injection_runs_raw_df[
        rq2_injection_runs_raw_df["condition"]
        != "baseline"
    ]
    .merge(
        _baseline,
        on=_pair_index,
        how="inner",
        validate="many_to_one",
    )
)

rq2_injection_paired_raw_df[
    "conditional_accuracy_reduction"
] = (
    rq2_injection_paired_raw_df[
        "baseline_final_accuracy"
    ]
    - rq2_injection_paired_raw_df["final_accuracy"]
)

rq2_injection_paired_raw_df[
    "conditional_accuracy_drop_gain"
] = (
    rq2_injection_paired_raw_df["final_accuracy_drop"]
    - rq2_injection_paired_raw_df[
        "baseline_final_accuracy_drop"
    ]
)

rq2_injection_paired_raw_df[
    "conditional_integrated_drop_gain"
] = (
    rq2_injection_paired_raw_df[
        "integrated_accuracy_drop"
    ]
    - rq2_injection_paired_raw_df[
        "baseline_integrated_accuracy_drop"
    ]
)

rq2_injection_paired_raw_df[
    "conditional_maximum_drop_gain"
] = (
    rq2_injection_paired_raw_df[
        "maximum_relaxed_accuracy_drop"
    ]
    - rq2_injection_paired_raw_df[
        "baseline_maximum_relaxed_accuracy_drop"
    ]
)

rq2_injection_paired_summary_df = _two_stage_summary(
    rq2_injection_paired_raw_df,
    [
        "candidate_config_id",
        "endpoint_mining_hop",
        "condition",
        "candidate_type",
    ],
    [
        "conditional_accuracy_reduction",
        "conditional_accuracy_drop_gain",
        "conditional_integrated_drop_gain",
        "conditional_maximum_drop_gain",
    ],
)


# ============================================================
# Save
# ============================================================

rq2_injection_runs_raw_df.to_csv(
    RQ2_INJECTION_OUT_DIR / "injection_runs_raw.csv",
    index=False,
)
rq2_injection_runs_summary_df.to_csv(
    RQ2_INJECTION_OUT_DIR / "injection_runs_summary.csv",
    index=False,
)
rq2_injection_epochs_raw_df.to_csv(
    RQ2_INJECTION_OUT_DIR
    / "injection_epoch_metrics_raw.csv",
    index=False,
)
rq2_injection_edges_raw_df.to_csv(
    RQ2_INJECTION_OUT_DIR / "injection_edges_raw.csv",
    index=False,
)
rq2_injection_selection_df.to_csv(
    RQ2_INJECTION_OUT_DIR / "injection_selections.csv",
    index=False,
)
rq2_injection_skips_df.to_csv(
    RQ2_INJECTION_OUT_DIR / "injection_skipped_groups.csv",
    index=False,
)
rq2_injection_paired_raw_df.to_csv(
    RQ2_INJECTION_OUT_DIR
    / "injection_paired_effects_raw.csv",
    index=False,
)
rq2_injection_paired_summary_df.to_csv(
    RQ2_INJECTION_OUT_DIR
    / "injection_paired_effects_summary.csv",
    index=False,
)

(
    RQ2_INJECTION_OUT_DIR / "experiment_config.json"
).write_text(
    json.dumps(
        {
            "dataset": RQ2_INJECTION_DATASET,
            "model_label": RQ2_INJECTION_MODEL_LABEL,
            "repeats": RQ2_INJECTION_REPEATS,
            "nominal_resampling_epochs": (
                RQ2_INJECTION_RESAMPLING_EPOCHS
            ),
            "with_early_stopping": (
                RQ2_INJECTION_WITH_EARLY_STOPPING
            ),
            "label_threshold": (
                RQ2_INJECTION_LABEL_THRESHOLD
            ),
            "minimum_accuracy_drop": (
                RQ2_INJECTION_MIN_ACCURACY_DROP
            ),
            "selected_hops": (
                None
                if RQ2_INJECTION_HOPS is None
                else sorted(RQ2_INJECTION_HOPS)
            ),
            "selected_seeds": (
                None
                if RQ2_INJECTION_SEEDS is None
                else sorted(RQ2_INJECTION_SEEDS)
            ),
            "selected_candidate_config_ids": (
                None
                if RQ2_INJECTION_CANDIDATE_CONFIG_IDS is None
                else sorted(RQ2_INJECTION_CANDIDATE_CONFIG_IDS)
            ),
            "epsilon": RQ2_INJECTION_EPSILON,
            "attack_budget": RQ2_INJECTION_ATTACK_BUDGET,
            "block_size": RQ2_INJECTION_BLOCK_SIZE,
            "epochs": RQ2_INJECTION_EPOCHS,
            "fine_tune_epochs": (
                RQ2_INJECTION_FINE_TUNE_EPOCHS
            ),
            "injection_epoch": RQ2_INJECTION_EPOCH,
            "n_injection_edges": (
                RQ2_INJECTION_N_EDGES
            ),
            "injection_initial_weight": (
                RQ2_INJECTION_INITIAL_WEIGHT
            ),
            "resampling_enabled": (
                RQ2_INJECTION_RESAMPLING_ENABLED
            ),
            "missed_definition": (
                "not present in baseline initial block, any "
                "recorded epoch block, any recorded post-"
                "resampling block, or final block"
            ),
            "available_source_modes": sorted(
                {
                    str(run["scoring_mode"])
                    for run in mining_runs
                    if _rq2_injection_run_is_selected(run)
                    and str(run["scoring_mode"])
                    in {"endpoint", "subset_accuracy_drop"}
                }
            ),
            "conditions": sorted(
                rq2_injection_runs_raw_df[
                    "condition"
                ].unique()
            ),
        },
        indent=2,
    ),
    encoding="utf-8",
)

print("\nRQ2 missed-edge injection run summary")
display(rq2_injection_runs_summary_df)

print("\nRQ2 paired conditional-effect summary")
display(rq2_injection_paired_summary_df)

if not rq2_injection_skips_df.empty:
    print("\nSkipped groups")
    display(rq2_injection_skips_df)

print(
    "\nSaved RQ2 missed-edge injection experiment to:",
    RQ2_INJECTION_OUT_DIR.resolve(),
)


### Analysis and plotting block

The following cell compares every injection condition with its paired no-injection baseline.

The primary outcome is the conditional reduction in final accuracy:

\[
\Delta A_{\mathrm{inj}}
=
A_{\mathrm{baseline}}
-
A_{\mathrm{injected}}.
\]

Additional outcomes include:

- change in final accuracy drop;
- change in integrated accuracy drop;
- accuracy trajectory before and after injection;
- retention of the injected edge in later candidate blocks;
- presence of the edge in the final PR-BCD block;
- selection of the edge as a final discrete perturbation.

The experiment distinguishes between several possible outcomes:

- the edge is inserted but immediately discarded;
- the edge survives but is never selected;
- the edge changes the optimization trajectory indirectly;
- the edge is retained and selected as a final perturbation;
- an individually harmful edge becomes conditionally neutral or cancelling.

This experiment therefore evaluates conditional harmfulness relative to an algorithmic optimization state rather than only relative to a fixed perturbation set.

In [ ]:
# %% [PLOTTING CELL]
# ============================================================
# Standalone RQ2 missed multi-edge injection plots
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


# ============================================================
# Select and load an injection run
# ============================================================

RQ2_INJECTION_BASE_OUT_DIR = (
    Path("extendedPlotting")
    / "rq2_missed_multi_edge_injection"
)

# Set to a concrete directory to plot a specific run. With None,
# the cell loads the run stored in latest_run.txt.
RQ2_INJECTION_PLOT_RUN_DIR = None

if RQ2_INJECTION_PLOT_RUN_DIR is None:
    latest_pointer = (
        RQ2_INJECTION_BASE_OUT_DIR
        / "latest_run.txt"
    )
    if not latest_pointer.exists():
        raise FileNotFoundError(
            "No standalone injection run was found. Run the "
            "injection execution cell first."
        )
    RQ2_INJECTION_OUT_DIR = Path(
        latest_pointer.read_text(encoding="utf-8").strip()
    )
else:
    RQ2_INJECTION_OUT_DIR = Path(
        RQ2_INJECTION_PLOT_RUN_DIR
    )

if not RQ2_INJECTION_OUT_DIR.exists():
    raise FileNotFoundError(
        f"Injection run directory does not exist: "
        f"{RQ2_INJECTION_OUT_DIR}"
    )


def _rq2_injection_read_csv(filename):
    path = RQ2_INJECTION_OUT_DIR / filename

    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()

    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        # An empty DataFrame without predefined columns can produce
        # a CSV containing only a newline. Treat that file as empty.
        return pd.DataFrame()


rq2_injection_runs_raw_df = _rq2_injection_read_csv(
    "injection_runs_raw.csv"
)
rq2_injection_epochs_raw_df = _rq2_injection_read_csv(
    "injection_epoch_metrics_raw.csv"
)
rq2_injection_edges_raw_df = _rq2_injection_read_csv(
    "injection_edges_raw.csv"
)
rq2_injection_selection_df = _rq2_injection_read_csv(
    "injection_selections.csv"
)
rq2_injection_skips_df = _rq2_injection_read_csv(
    "injection_skipped_groups.csv"
)
rq2_injection_paired_raw_df = _rq2_injection_read_csv(
    "injection_paired_effects_raw.csv"
)

if rq2_injection_runs_raw_df.empty:
    raise RuntimeError(
        "The selected injection run contains no completed runs."
    )

config_path = RQ2_INJECTION_OUT_DIR / "experiment_config.json"
if config_path.exists():
    RQ2_INJECTION_CONFIG = json.loads(
        config_path.read_text(encoding="utf-8")
    )
else:
    RQ2_INJECTION_CONFIG = {}

RQ2_INJECTION_EPOCH = int(
    RQ2_INJECTION_CONFIG.get("injection_epoch", 0)
)
RQ2_INJECTION_N_EDGES = int(
    RQ2_INJECTION_CONFIG.get(
        "n_injection_edges",
        pd.to_numeric(
            rq2_injection_runs_raw_df.get(
                "n_injected",
                pd.Series([0]),
            ),
            errors="coerce",
        ).max(),
    )
)

RQ2_INJECTION_PLOT_DIR = RUN_PLOTS_DIR
RQ2_INJECTION_PLOT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RQ2_INJECTION_ACCURACY_YMIN = 0.70
RQ2_INJECTION_JITTER = 0.055

print(
    "Plotting standalone injection run:",
    RQ2_INJECTION_OUT_DIR.resolve(),
)


_rq2_condition_order = [
    "baseline",
    "random_unseen_control",
    "endpoint_label_0_missed",
    "endpoint_label_1_missed",
    "accuracy_drop_highest_missed",
]
_rq2_condition_order = [
    condition
    for condition in _rq2_condition_order
    if condition
    in set(rq2_injection_runs_raw_df["condition"])
]

_rq2_condition_labels = {
    "baseline": "Baseline",
    "random_unseen_control": "Random unseen",
    "endpoint_label_0_missed": "Missed label 0",
    "endpoint_label_1_missed": "Missed label 1",
    "accuracy_drop_highest_missed": (
        "Missed highest accuracy drop"
    ),
}


# Match the RQ1 aggregation logic: average repeats within each
# victim seed before constructing distribution plots.
_rq2_box_run_df = (
    rq2_injection_runs_raw_df
    .groupby(
        [
            "seed",
            "candidate_config_id",
            "endpoint_mining_hop",
            "condition",
        ],
        as_index=False,
    )
    .agg(
        final_accuracy=("final_accuracy", "mean"),
        final_accuracy_drop=("final_accuracy_drop", "mean"),
    )
)

_rq2_box_paired_df = (
    rq2_injection_paired_raw_df
    .groupby(
        [
            "seed",
            "candidate_config_id",
            "endpoint_mining_hop",
            "condition",
        ],
        as_index=False,
    )
    .agg(
        conditional_accuracy_reduction=(
            "conditional_accuracy_reduction",
            "mean",
        ),
        conditional_integrated_drop_gain=(
            "conditional_integrated_drop_gain",
            "mean",
        ),
    )
)


def _rq2_boxplot_with_points(
    ax,
    frame,
    *,
    value_column,
    conditions,
    ylabel,
    title,
    zero_line=False,
):
    plot_data = []
    used_conditions = []

    for condition in conditions:
        values = (
            frame.loc[
                frame["condition"] == condition,
                value_column,
            ]
            .dropna()
            .astype(float)
            .to_numpy()
        )

        if values.size == 0:
            continue

        plot_data.append(values)
        used_conditions.append(condition)

    if not plot_data:
        return []

    positions = np.arange(1, len(plot_data) + 1)

    ax.boxplot(
        plot_data,
        positions=positions,
        widths=0.55,
        showmeans=True,
    )

    rng = np.random.default_rng(12345)
    for position, values in zip(positions, plot_data):
        jitter = rng.uniform(
            -RQ2_INJECTION_JITTER,
            RQ2_INJECTION_JITTER,
            size=len(values),
        )
        ax.scatter(
            np.full(len(values), position) + jitter,
            values,
            alpha=0.65,
            s=24,
            zorder=3,
        )

    if zero_line:
        ax.axhline(
            0.0,
            linestyle="--",
            linewidth=1.0,
        )

    ax.set_xticks(positions)
    ax.set_xticklabels(
        [
            _rq2_condition_labels.get(c, c)
            for c in used_conditions
        ],
        rotation=18,
        ha="right",
    )
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)

    return used_conditions


# ============================================================
# 1. Final accuracy reduction from each seed-specific baseline
# ============================================================

fig, ax = plt.subplots(figsize=(11, 5.8))

_rq2_boxplot_with_points(
    ax,
    _rq2_box_run_df,
    value_column="final_accuracy_drop",
    conditions=_rq2_condition_order,
    ylabel="Clean accuracy − final attacked accuracy",
    title="Final accuracy reduction after missed-edge injection",
    zero_line=True,
)

fig.tight_layout()
fig.savefig(
    RQ2_INJECTION_PLOT_DIR
    / "01_final_accuracy_drop_boxplot.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


# ============================================================
# 2. RQ1-style boxplot: paired causal accuracy reduction
# ============================================================

if not rq2_injection_paired_raw_df.empty:
    _effect_order = [
        condition
        for condition in _rq2_condition_order
        if condition != "baseline"
    ]

    fig, ax = plt.subplots(figsize=(10.5, 5.5))

    _rq2_boxplot_with_points(
        ax,
        _rq2_box_paired_df,
        value_column="conditional_accuracy_reduction",
        conditions=_effect_order,
        ylabel="Baseline accuracy − injected accuracy",
        title=(
            "Paired causal accuracy benefit of injecting "
            "baseline-missed candidates"
        ),
        zero_line=True,
    )

    fig.tight_layout()
    fig.savefig(
        RQ2_INJECTION_PLOT_DIR
        / "02_paired_accuracy_reduction_boxplot.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()


# ============================================================
# 3. RQ1-style boxplot: paired integrated-drop gain
# ============================================================

if not rq2_injection_paired_raw_df.empty:
    fig, ax = plt.subplots(figsize=(10.5, 5.5))

    _rq2_boxplot_with_points(
        ax,
        _rq2_box_paired_df,
        value_column="conditional_integrated_drop_gain",
        conditions=_effect_order,
        ylabel="Injected integrated drop − baseline integrated drop",
        title=(
            "Paired trajectory benefit of injecting "
            "baseline-missed candidates"
        ),
        zero_line=True,
    )

    fig.tight_layout()
    fig.savefig(
        RQ2_INJECTION_PLOT_DIR
        / "03_paired_integrated_drop_gain_boxplot.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()


# ============================================================
# 4. RQ1-style paired comparison against random injection
# ============================================================

if not rq2_injection_paired_raw_df.empty:
    paired_wide = (
        _rq2_box_paired_df
        .pivot_table(
            index=[
                "seed",
                "candidate_config_id",
                "endpoint_mining_hop",
            ],
            columns="condition",
            values="conditional_accuracy_reduction",
            aggfunc="mean",
        )
        .reset_index()
    )

    harmful_conditions = [
        condition
        for condition in [
            "endpoint_label_1_missed",
            "accuracy_drop_highest_missed",
        ]
        if condition in paired_wide.columns
    ]

    for harmful_condition in harmful_conditions:
        required = {
            "random_unseen_control",
            harmful_condition,
        }
        if not required.issubset(paired_wide.columns):
            continue

        part = paired_wide.dropna(
            subset=list(required)
        )
        if part.empty:
            continue

        fig, ax = plt.subplots(figsize=(7.5, 6))

        ax.scatter(
            part["random_unseen_control"],
            part[harmful_condition],
            alpha=0.75,
        )

        x_limits = ax.get_xlim()
        y_limits = ax.get_ylim()
        low = min(x_limits[0], y_limits[0])
        high = max(x_limits[1], y_limits[1])

        ax.plot(
            [low, high],
            [low, high],
            linestyle="--",
        )
        ax.set_xlim(low, high)
        ax.set_ylim(low, high)

        ax.set_xlabel(
            "Accuracy reduction from random unseen injection"
        )
        ax.set_ylabel(
            "Accuracy reduction from "
            f"{_rq2_condition_labels[harmful_condition]} injection"
        )
        ax.set_title(
            "Paired causal benefit relative to random injection"
        )
        ax.grid(alpha=0.3)

        fig.tight_layout()
        fig.savefig(
            RQ2_INJECTION_PLOT_DIR
            / (
                "04_paired_vs_random__"
                f"{harmful_condition}.png"
            ),
            dpi=200,
            bbox_inches="tight",
        )
        plt.show()


# ============================================================
# 5. Full accuracy trajectory
# ============================================================

_trajectory = (
    rq2_injection_epochs_raw_df[
        rq2_injection_epochs_raw_df["prbcd_epoch"] >= 0
    ]
    .groupby(
        ["condition", "prbcd_epoch"],
        as_index=False,
    )
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        n_runs=("accuracy", "size"),
    )
)
_trajectory["accuracy_std"] = (
    _trajectory["accuracy_std"].fillna(0.0)
)

fig, ax = plt.subplots(figsize=(10.5, 6))

for condition in _rq2_condition_order:
    part = _trajectory[
        _trajectory["condition"] == condition
    ].sort_values("prbcd_epoch")

    if part.empty:
        continue

    line = ax.plot(
        part["prbcd_epoch"],
        part["accuracy_mean"],
        label=_rq2_condition_labels.get(
            condition,
            condition,
        ),
    )[0]

    ax.fill_between(
        part["prbcd_epoch"],
        part["accuracy_mean"] - part["accuracy_std"],
        part["accuracy_mean"] + part["accuracy_std"],
        alpha=0.12,
        color=line.get_color(),
    )

ax.axvline(
    RQ2_INJECTION_EPOCH,
    linestyle="--",
    linewidth=1.2,
    label="Injection epoch",
)

ax.set_xlabel("PRBCD epoch")
ax.set_ylabel("Victim accuracy")
ax.set_title(
    "PRBCD trajectory before and after multi-edge injection"
)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(
    RQ2_INJECTION_PLOT_DIR
    / "05_accuracy_trajectory.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


# ============================================================
# 6. Mean retention fraction after intervention
# ============================================================

_retention = rq2_injection_epochs_raw_df[
    (rq2_injection_epochs_raw_df["condition"] != "baseline")
    & (
        rq2_injection_epochs_raw_df["prbcd_epoch"]
        >= RQ2_INJECTION_EPOCH
    )
].copy()

if not _retention.empty:
    _retention["epochs_since_injection"] = (
        _retention["prbcd_epoch"]
        - RQ2_INJECTION_EPOCH
    )

    _retention_summary = (
        _retention
        .groupby(
            ["condition", "epochs_since_injection"],
            as_index=False,
        )
        .agg(
            retention_mean=(
                "injected_edge_retention_fraction",
                "mean",
            ),
            retention_std=(
                "injected_edge_retention_fraction",
                "std",
            ),
        )
    )
    _retention_summary["retention_std"] = (
        _retention_summary["retention_std"]
        .fillna(0.0)
    )

    fig, ax = plt.subplots(figsize=(10.5, 5.5))

    for condition in [
        c for c in _rq2_condition_order
        if c != "baseline"
    ]:
        part = _retention_summary[
            _retention_summary["condition"] == condition
        ].sort_values("epochs_since_injection")

        if part.empty:
            continue

        line = ax.plot(
            part["epochs_since_injection"],
            part["retention_mean"],
            label=_rq2_condition_labels.get(
                condition,
                condition,
            ),
        )[0]

        ax.fill_between(
            part["epochs_since_injection"],
            part["retention_mean"] - part["retention_std"],
            part["retention_mean"] + part["retention_std"],
            alpha=0.12,
            color=line.get_color(),
        )

    ax.set_xlabel("Epochs since injection")
    ax.set_ylabel(
        "Mean fraction of injected edges remaining in block"
    )
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(
        "Retention of baseline-missed injected edges"
    )
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    fig.tight_layout()
    fig.savefig(
        RQ2_INJECTION_PLOT_DIR
        / "06_injected_edge_retention.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()


# ============================================================
# 7. Mechanical insertion check
# ============================================================

_mechanical = (
    rq2_injection_runs_raw_df[
        rq2_injection_runs_raw_df["condition"]
        != "baseline"
    ]
    .groupby("condition", as_index=False)
    .agg(
        requested_mean=("n_injected", "mean"),
        inserted_mean=("n_inserted", "mean"),
        already_present_mean=(
            "n_already_present",
            "mean",
        ),
    )
    .set_index("condition")
    .reindex([
        c for c in _rq2_condition_order
        if c != "baseline"
    ])
    .dropna(how="all")
    .reset_index()
)

if not _mechanical.empty:
    x = np.arange(len(_mechanical))
    width = 0.25

    fig, ax = plt.subplots(figsize=(10.5, 5.2))

    ax.bar(
        x - width,
        _mechanical["requested_mean"],
        width,
        label="Requested",
    )
    ax.bar(
        x,
        _mechanical["inserted_mean"],
        width,
        label="Inserted",
    )
    ax.bar(
        x + width,
        _mechanical["already_present_mean"],
        width,
        label="Already present",
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [
            _rq2_condition_labels.get(c, c)
            for c in _mechanical["condition"]
        ],
        rotation=18,
        ha="right",
    )
    ax.set_ylabel("Mean number of edges")
    ax.set_title("Mechanical verification of the injection")
    ax.grid(axis="y", alpha=0.3)
    ax.legend()

    fig.tight_layout()
    fig.savefig(
        RQ2_INJECTION_PLOT_DIR
        / "07_injection_mechanical_check.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()


# ============================================================
# Compact reports
# ============================================================

_plot_run_report = (
    rq2_injection_runs_raw_df
    .groupby("condition", as_index=False)
    .agg(
        n_runs=("final_accuracy", "size"),
        final_accuracy_mean=("final_accuracy", "mean"),
        final_accuracy_std=("final_accuracy", "std"),
        n_inserted_mean=("n_inserted", "mean"),
        final_block_retention_mean=(
            "injected_fraction_in_final_block",
            "mean",
        ),
        final_selection_fraction_mean=(
            "injected_fraction_finally_selected",
            "mean",
        ),
    )
)

_plot_paired_report = (
    rq2_injection_paired_raw_df
    .groupby("condition", as_index=False)
    .agg(
        n_pairs=(
            "conditional_accuracy_reduction",
            "size",
        ),
        accuracy_reduction_mean=(
            "conditional_accuracy_reduction",
            "mean",
        ),
        accuracy_reduction_std=(
            "conditional_accuracy_reduction",
            "std",
        ),
        integrated_drop_gain_mean=(
            "conditional_integrated_drop_gain",
            "mean",
        ),
        integrated_drop_gain_std=(
            "conditional_integrated_drop_gain",
            "std",
        ),
    )
)

_plot_run_report.to_csv(
    RQ2_INJECTION_OUT_DIR
    / "injection_plot_run_report.csv",
    index=False,
)
_plot_paired_report.to_csv(
    RQ2_INJECTION_OUT_DIR
    / "injection_plot_paired_report.csv",
    index=False,
)

print("RQ2 injection run report")
display(_plot_run_report)

print("\nRQ2 paired injection report")
display(_plot_paired_report)

print(
    "\nInjection plots saved to:",
    RQ2_INJECTION_PLOT_DIR.resolve(),
)


---

## Experiment 3 — Oracle Evaluation of Joint and Conditional Harmfulness

### Research objective

An edge that is harmful when evaluated individually does not necessarily remain harmful when combined with other perturbations.

The oracle experiment applies candidate perturbations cumulatively and compares two orderings:

1. candidates ordered by their mined harmfulness score;
2. candidates placed in a random order.

For an ordering \(e_1,\ldots,e_n\), define the cumulative perturbation block

\[
B_k=\{e_1,\ldots,e_k\}.
\]

The test accuracy after applying the first \(k\) candidates is

\[
A_k=A(G\oplus B_k).
\]

The experiment first determines whether mined candidates construct a stronger standalone attack than random candidates. It then examines whether each newly added flip block remains harmful under the perturbations already present.

### Execution block

Candidates are added cumulatively in steps of `BLOCK_STEP_ORACLE`.

For every step, the experiment records:

- the previous cumulative block size;
- the new cumulative block size;
- the number of newly added edges;
- accuracy before adding the new flip block;
- accuracy after adding the new flip block;
- cumulative accuracy drop;
- maximum accuracy drop observed so far.

The marginal conditional effect of the newly added flip block is

\[
\Delta A_k
=
A(B_{k-1})-A(B_k).
\]

The newly added block is classified as:

- **conditionally harmful** when \(\Delta A_k>0\);
- **neutral** when \(\Delta A_k\approx 0\);
- **cancelling** when \(\Delta A_k<0\).

With `BLOCK_STEP_ORACLE = 10`, the conditional effect refers to a block of up to ten newly added perturbations. Setting the value to one produces an individual-edge conditional analysis.

In [ ]:
##%%
# ============================================================
# RQ2: Mode-aware oracle evaluation
#
# endpoint:
#   - preserves the original binary-label behavior
#   - descending binary labels versus one random ordering
#   - preserves the exact n_positive prefix analysis
#
# subset_accuracy_drop:
#   - ranks observed candidates by continuous score_raw
#   - compares descending, ascending, and repeated random orderings
#   - evaluates exact fixed budget fractions and a focused top-10% budget
#
# Random repetitions are averaged within each victim seed before results
# are aggregated across seeds. The victim seed therefore remains the
# principal experimental unit.
# ============================================================

from helpers.selector_pipeline_helpers import accuracy
from pathlib import Path
from tqdm.auto import tqdm
from IPython.display import display

import pandas as pd
import numpy as np
import re
import torch


# ============================================================
# Configuration
# ============================================================

# Set to 1 for an individual-edge conditional analysis.
# With 10, each delta describes a block of up to ten new flips.
BLOCK_STEP_ORACLE = 10

CONDITIONAL_EFFECT_TOL = 1e-10

# Fractions of total accuracy:
# 0.01 corresponds to one percentage point.
FIXED_ACCURACY_REDUCTIONS = [0.01, 0.02, 0.05]

# Exact candidate-budget fractions evaluated for every ordering.
# These are especially important for continuous subset scores, where there
# is no natural binary positive-prefix length.
ORACLE_FIXED_BUDGET_FRACTIONS = [
    0.01,
    0.02,
    0.05,
    0.10,
    0.20,
    0.50,
    1.00,
]

# For subset scoring, the focused prefix is an equal 10% candidate budget:
# descending = highest-scored 10%, ascending = lowest-scored 10%, and random
# = a random 10% control.
ORACLE_SUBSET_FOCUS_FRACTION = 0.10

# Endpoint behavior remains one random ordering, as in the original cell.
ORACLE_ENDPOINT_RANDOM_REPEATS = 1

# Continuous-score oracle uses repeated random controls. These repetitions
# are averaged within seed before aggregation across victim seeds.
ORACLE_SUBSET_RANDOM_REPEATS = 10

ORACLE_OUT_DIR = (
    Path("extendedPlotting")
    / "oracle_checks_multiseed"
)
PARTS_DIR = ORACLE_OUT_DIR / "parts"
PLOTS_DIR = RUN_PLOTS_DIR

ORACLE_OUT_DIR.mkdir(parents=True, exist_ok=True)
PARTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

RESUME_FROM_PARTS = True

# Old binary-positive-prefix parts must not be loaded into this mode-aware
# experiment.
ORACLE_SCHEMA_VERSION = "mode_aware_oracle_v4"


# ============================================================
# Helpers
# ============================================================

def _oracle_safe(value):
    return re.sub(
        r"[^a-zA-Z0-9_.=-]+",
        "_",
        str(value),
    ).strip("_")


def _flip_candidate_inplace(adj, run, candidate_index):
    u = int(run["cand_src"][candidate_index])
    v = int(run["cand_dst"][candidate_index])
    exists = float(run["exists_list"][candidate_index])

    adj[u, v] = 0.0 if exists else 1.0
    adj[v, u] = 0.0 if exists else 1.0


def _classify_conditional_effect(
    delta_accuracy,
    tolerance=CONDITIONAL_EFFECT_TOL,
):
    """
    delta_accuracy = previous_accuracy - current_accuracy

    Positive: conditionally harmful.
    Approximately zero: neutral.
    Negative: cancelling.
    """
    if pd.isna(delta_accuracy):
        return "baseline"

    if delta_accuracy > tolerance:
        return "conditionally_harmful"

    if delta_accuracy < -tolerance:
        return "cancelling"

    return "neutral"


def _threshold_column_name(threshold):
    percentage_points = float(threshold) * 100
    threshold_text = f"{percentage_points:g}".replace(".", "p")
    return f"edges_to_{threshold_text}pp_drop"


def _fraction_column_name(fraction):
    percentage = float(fraction) * 100
    text = f"{percentage:g}".replace(".", "p")
    return f"at_{text}pct_budget"


def _fraction_to_k(fraction, n_candidates):
    """Convert a positive candidate fraction to a reproducible exact budget."""
    n_candidates = int(n_candidates)
    if n_candidates <= 0:
        return 0

    fraction = float(fraction)
    if not 0.0 < fraction <= 1.0:
        raise ValueError(
            f"Budget fractions must lie in (0, 1], got {fraction}."
        )

    return min(
        n_candidates,
        max(1, int(np.ceil(fraction * n_candidates))),
    )


def _deterministic_score_order(scores, *, descending):
    """Sort by score with candidate index as a deterministic tie-break."""
    scores = np.asarray(scores, dtype=np.float64).reshape(-1)
    if not np.isfinite(scores).all():
        raise ValueError("Oracle ranking scores must all be finite.")

    candidate_index = np.arange(scores.size, dtype=np.int64)
    primary = -scores if descending else scores
    return np.lexsort((candidate_index, primary)).astype(np.int64)


def _resolve_oracle_target(run):
    """Return mode-specific ranking data and focused-prefix metadata."""
    scoring_mode = str(run["scoring_mode"])
    n_cands = int(run["n_cands"])

    if scoring_mode == "endpoint":
        labels = (
            torch.as_tensor(run["labels_changed"])
            .detach()
            .cpu()
            .float()
            .flatten()
            .numpy()
        )

        if labels.size != n_cands:
            raise ValueError(
                "Endpoint labels_changed and n_cands are inconsistent for "
                f"{run['candidate_config_id']}."
            )

        # Endpoint labels must stay binary. No continuous adaptation is applied.
        if not np.all(np.isclose(labels, 0.0) | np.isclose(labels, 1.0)):
            raise ValueError(
                "Endpoint oracle expected binary labels in {0, 1}."
            )

        n_positive = int(np.sum(labels > 0.5))
        focus_limit = n_positive

        return {
            "target_kind": "binary",
            "ranking_score_name": "endpoint_label",
            "ranking_scores": labels.astype(np.float64),
            "n_positive_examples": n_positive,
            "positive_rate": n_positive / max(1, n_cands),
            "focus_prefix_name": "positive_label_prefix",
            "focus_prefix_limit": focus_limit,
            "focus_prefix_fraction": focus_limit / max(1, n_cands),
            "random_repeats": int(ORACLE_ENDPOINT_RANDOM_REPEATS),
            "constant_score_run": bool(np.ptp(labels) <= 1e-12),
        }

    if scoring_mode == "subset_accuracy_drop":
        scores = np.asarray(
            run.get("score_raw", run.get("labels_raw")),
            dtype=np.float64,
        ).reshape(-1)

        if scores.size != n_cands:
            raise ValueError(
                "subset score_raw and n_cands are inconsistent for "
                f"{run['candidate_config_id']}."
            )

        if not np.isfinite(scores).all():
            raise ValueError(
                "subset oracle requires finite score_raw values. "
                "Unobserved candidates should already have been removed."
            )

        focus_limit = _fraction_to_k(
            ORACLE_SUBSET_FOCUS_FRACTION,
            n_cands,
        )

        return {
            "target_kind": "continuous",
            "ranking_score_name": "mean_harmful_drop_when_selected",
            "ranking_scores": scores,
            # Deliberately not defined for continuous labels.
            "n_positive_examples": np.nan,
            "positive_rate": np.nan,
            "focus_prefix_name": (
                f"top_{100 * ORACLE_SUBSET_FOCUS_FRACTION:g}pct_budget"
            ),
            "focus_prefix_limit": focus_limit,
            "focus_prefix_fraction": focus_limit / max(1, n_cands),
            "random_repeats": int(ORACLE_SUBSET_RANDOM_REPEATS),
            "constant_score_run": bool(np.ptp(scores) <= 1e-12),
        }

    raise ValueError(
        f"Unsupported scoring_mode={scoring_mode!r}. "
        "Expected 'endpoint' or 'subset_accuracy_drop'."
    )


def _build_attack_orders(run, target_info, run_index):
    """Construct mode-specific deterministic and random oracle orderings."""
    scoring_mode = str(run["scoring_mode"])
    scores = target_info["ranking_scores"]
    n_cands = int(run["n_cands"])
    seed = int(run["seed"])
    hop = int(run.get("endpoint_mining_hop", 0))

    orders = []

    if scoring_mode == "endpoint":
        # Preserve the original endpoint ordering exactly, including the
        # existing tie behavior of torch.argsort for binary labels.
        order_top = (
            torch.as_tensor(scores, dtype=torch.float32)
            .argsort(descending=True)
            .cpu()
            .numpy()
            .astype(np.int64)
        )
        orders.append({
            "attack": "top-k by mined label",
            "attack_family": "descending",
            "random_repeat": 0,
            "order": order_top,
        })

    else:
        order_top = _deterministic_score_order(
            scores,
            descending=True,
        )
        order_bottom = _deterministic_score_order(
            scores,
            descending=False,
        )

        orders.extend([
            {
                "attack": "top-k by subset score",
                "attack_family": "descending",
                "random_repeat": 0,
                "order": order_top,
            },
            {
                "attack": "bottom-k by subset score",
                "attack_family": "ascending",
                "random_repeat": 0,
                "order": order_bottom,
            },
        ])

    random_repeats = int(target_info["random_repeats"])
    if random_repeats <= 0:
        raise ValueError("The number of random oracle repetitions must be positive.")

    for random_repeat in range(random_repeats):
        rng_seed = (
            seed
            + 1009 * int(run_index)
            + 31 * hop
            + n_cands
            + 104729 * random_repeat
        )
        rng = np.random.default_rng(rng_seed)
        orders.append({
            "attack": "random candidates",
            "attack_family": "random",
            "random_repeat": int(random_repeat),
            "order": rng.permutation(n_cands).astype(np.int64),
        })

    return orders


def _build_block_sizes(n_cands, target_info):
    """Include regular steps and all exact mode-aware evaluation budgets."""
    n_cands = int(n_cands)

    sizes = set(
        range(
            0,
            n_cands + 1,
            int(BLOCK_STEP_ORACLE),
        )
    )
    sizes.update({0, n_cands})

    focus_limit = int(target_info["focus_prefix_limit"])
    if 0 <= focus_limit <= n_cands:
        sizes.add(focus_limit)

    # Do not insert new evaluation points into endpoint trajectories: doing so
    # would split the original blocks and change endpoint conditional-effect
    # statistics. Exact fixed-fraction budgets are added only for the
    # continuous subset-score oracle.
    if target_info["target_kind"] == "continuous":
        for fraction in ORACLE_FIXED_BUDGET_FRACTIONS:
            sizes.add(_fraction_to_k(fraction, n_cands))

    return sorted(
        int(k)
        for k in sizes
        if 0 <= int(k) <= n_cands
    )


def _build_oracle_run_metrics(step_frame):
    """
    Convert step trajectories into one row per seed, candidate configuration,
    ordering, and random repetition.
    """
    if step_frame is None or step_frame.empty:
        return pd.DataFrame()

    run_rows = []

    run_group_cols = [
        "seed",
        "candidate_config_id",
        "candidate_set_size",
        "prbcd_candidate_fraction",
        "actual_prbcd_fraction",
        "scoring_mode",
        "target_kind",
        "ranking_score_name",
        "endpoint_mining_hop",
        "attack",
        "attack_family",
        "random_repeat",
    ]

    for key, group in step_frame.groupby(
        run_group_cols,
        dropna=False,
    ):
        group = (
            group
            .sort_values("k")
            .reset_index(drop=True)
        )

        addition_steps = group[
            group["n_edges_added"] > 0
        ].copy()

        harmful_steps = addition_steps[
            addition_steps["conditional_effect"]
            == "conditionally_harmful"
        ]

        neutral_steps = addition_steps[
            addition_steps["conditional_effect"]
            == "neutral"
        ]

        cancelling_steps = addition_steps[
            addition_steps["conditional_effect"]
            == "cancelling"
        ]

        n_steps = int(len(addition_steps))
        n_harmful = int(len(harmful_steps))
        n_neutral = int(len(neutral_steps))
        n_cancelling = int(len(cancelling_steps))

        final_row = group.iloc[-1]

        max_drop_position = (
            group["cumulative_accuracy_drop"]
            .astype(float)
            .idxmax()
        )
        max_drop_row = group.loc[max_drop_position]

        row = {
            **dict(zip(run_group_cols, key)),

            "clean_accuracy": float(
                group["clean_accuracy"].iloc[0]
            ),
            "n_candidates": int(
                group["n_candidates"].iloc[0]
            ),
            "n_positive_examples": float(
                group["n_positive_examples"].iloc[0]
            ),
            "positive_rate": float(
                group["positive_rate"].iloc[0]
            ),
            "focus_prefix_name": str(
                group["focus_prefix_name"].iloc[0]
            ),
            "focus_prefix_limit": int(
                group["focus_prefix_limit"].iloc[0]
            ),
            "focus_prefix_fraction": float(
                group["focus_prefix_fraction"].iloc[0]
            ),
            "constant_score_run": bool(
                group["constant_score_run"].iloc[0]
            ),

            "final_k": int(final_row["k"]),
            "final_test_accuracy": float(
                final_row["test_accuracy"]
            ),
            "final_cumulative_accuracy_drop": float(
                final_row["cumulative_accuracy_drop"]
            ),

            "maximum_achieved_accuracy_drop": float(
                max_drop_row["cumulative_accuracy_drop"]
            ),
            "k_at_maximum_accuracy_drop": int(
                max_drop_row["k"]
            ),
            "budget_fraction_at_maximum_accuracy_drop": float(
                max_drop_row["budget_fraction"]
            ),

            "n_addition_steps": n_steps,
            "n_conditionally_harmful_steps": n_harmful,
            "n_neutral_steps": n_neutral,
            "n_cancelling_steps": n_cancelling,

            "proportion_conditionally_harmful": (
                n_harmful / n_steps
                if n_steps > 0
                else np.nan
            ),
            "proportion_neutral": (
                n_neutral / n_steps
                if n_steps > 0
                else np.nan
            ),
            "proportion_cancelling": (
                n_cancelling / n_steps
                if n_steps > 0
                else np.nan
            ),

            "largest_harmful_step": (
                float(harmful_steps["delta_accuracy"].max())
                if not harmful_steps.empty
                else np.nan
            ),
            "largest_cancelling_step": (
                float(cancelling_steps["delta_accuracy"].min())
                if not cancelling_steps.empty
                else np.nan
            ),
        }

        for threshold in FIXED_ACCURACY_REDUCTIONS:
            column = _threshold_column_name(threshold)

            reached = group[
                group["cumulative_accuracy_drop"]
                >= float(threshold) - CONDITIONAL_EFFECT_TOL
            ]

            row[column] = (
                int(reached["k"].min())
                if not reached.empty
                else np.nan
            )

        if (
            n_harmful
            + n_neutral
            + n_cancelling
            != n_steps
        ):
            raise RuntimeError(
                "Conditional-effect categories do not sum "
                "to the number of addition steps."
            )

        run_rows.append(row)

    return pd.DataFrame(run_rows)


def _average_random_repeats_within_seed(
    frame,
    *,
    id_columns,
    numeric_columns,
):
    """
    Average repeated random orderings within each victim seed.

    Deterministic attacks have one repetition, so they pass through unchanged.
    """
    if frame is None or frame.empty:
        return pd.DataFrame()

    available_numeric = [
        column
        for column in numeric_columns
        if column in frame.columns
    ]

    grouped = (
        frame
        .groupby(id_columns, dropna=False)[available_numeric]
        .mean()
        .reset_index()
    )

    repeat_counts = (
        frame
        .groupby(id_columns, dropna=False)["random_repeat"]
        .nunique()
        .rename("n_ordering_repetitions")
        .reset_index()
    )

    return grouped.merge(
        repeat_counts,
        on=id_columns,
        how="left",
    )


def _build_fixed_budget_endpoint_table(step_frame):
    """Select the exact row corresponding to every requested budget fraction."""
    if step_frame is None or step_frame.empty:
        return pd.DataFrame()

    rows = []

    group_cols = [
        "seed",
        "candidate_config_id",
        "candidate_set_size",
        "prbcd_candidate_fraction",
        "actual_prbcd_fraction",
        "scoring_mode",
        "target_kind",
        "ranking_score_name",
        "endpoint_mining_hop",
        "attack",
        "attack_family",
        "random_repeat",
    ]

    # The endpoint trajectory intentionally keeps its old block grid. Exact
    # fixed-fraction endpoints are therefore reported only for continuous
    # subset_accuracy_drop runs.
    subset_frame = step_frame[
        step_frame["scoring_mode"] == "subset_accuracy_drop"
    ].copy()

    for key, group in subset_frame.groupby(group_cols, dropna=False):
        group = group.sort_values("k")
        n_candidates = int(group["n_candidates"].iloc[0])

        for requested_fraction in ORACLE_FIXED_BUDGET_FRACTIONS:
            requested_k = _fraction_to_k(
                requested_fraction,
                n_candidates,
            )

            selected = group[group["k"] == requested_k]
            if selected.empty:
                raise RuntimeError(
                    "An exact fixed-budget oracle endpoint was not "
                    f"evaluated: fraction={requested_fraction}, k={requested_k}."
                )

            source_row = selected.iloc[0].to_dict()
            source_row.update({
                "requested_budget_fraction": float(requested_fraction),
                "requested_k": int(requested_k),
            })
            rows.append(source_row)

    return pd.DataFrame(rows)


REQUIRED_MODE_AWARE_COLUMNS = {
    "previous_k",
    "n_edges_added",
    "previous_test_accuracy",
    "delta_accuracy",
    "conditional_effect",
    "is_conditionally_harmful",
    "is_neutral",
    "is_cancelling",
    "cumulative_accuracy_drop",
    "maximum_accuracy_drop_so_far",
    "target_kind",
    "ranking_score_name",
    "focus_prefix_name",
    "focus_prefix_limit",
    "within_focus_prefix",
    "is_focus_prefix_endpoint",
    "attack_family",
    "random_repeat",
    "budget_fraction",
    "selected_score_sum",
    "selected_score_mean",
    "score_mass_captured_fraction",
}


# ============================================================
# Execute oracle trajectories
# ============================================================

if int(BLOCK_STEP_ORACLE) <= 0:
    raise ValueError("BLOCK_STEP_ORACLE must be positive.")

if int(ORACLE_ENDPOINT_RANDOM_REPEATS) <= 0:
    raise ValueError("ORACLE_ENDPOINT_RANDOM_REPEATS must be positive.")

if int(ORACLE_SUBSET_RANDOM_REPEATS) <= 0:
    raise ValueError("ORACLE_SUBSET_RANDOM_REPEATS must be positive.")

oracle_frames = []

for run_index, run in enumerate(mining_runs):
    seed = int(run["seed"])
    scoring_mode = str(run["scoring_mode"])

    part_csv = PARTS_DIR / (
        f"oracle__{ORACLE_SCHEMA_VERSION}"
        f"__seed-{seed}"
        f"__{_oracle_safe(run['candidate_config_id'])}"
        f"__mode-{_oracle_safe(scoring_mode)}"
        f"__h-{run['endpoint_mining_hop']}"
        f"__endpointRandom-{ORACLE_ENDPOINT_RANDOM_REPEATS}"
        f"__subsetRandom-{ORACLE_SUBSET_RANDOM_REPEATS}"
        f"__subsetFocus-{_oracle_safe(ORACLE_SUBSET_FOCUS_FRACTION)}"
        f"__budgets-{_oracle_safe('-'.join(map(str, ORACLE_FIXED_BUDGET_FRACTIONS)))}.csv"
    )

    if RESUME_FROM_PARTS and part_csv.exists():
        cached_df = pd.read_csv(part_csv)

        if REQUIRED_MODE_AWARE_COLUMNS.issubset(
            cached_df.columns
        ):
            oracle_frames.append(cached_df)
            print("Loaded:", part_csv)
            continue

        print(
            "Outdated cached oracle schema; recomputing:",
            part_csv,
        )

    context = run["context"]
    set_global_seed(seed)

    n_cands = int(run["n_cands"])
    if n_cands <= 0:
        raise ValueError(
            f"Oracle received no candidates for {run['candidate_config_id']}."
        )

    target_info = _resolve_oracle_target(run)
    ranking_scores = np.asarray(
        target_info["ranking_scores"],
        dtype=np.float64,
    )

    attack_orders = _build_attack_orders(
        run,
        target_info,
        run_index,
    )
    block_sizes = _build_block_sizes(
        n_cands,
        target_info,
    )

    total_score_mass = float(np.clip(ranking_scores, 0.0, None).sum())

    if target_info["constant_score_run"]:
        print(
            f"[ORACLE WARNING] Constant ranking scores for seed={seed}, "
            f"config={run['candidate_config_id']}, mode={scoring_mode}. "
            "Descending and ascending rankings contain no score information."
        )

    rows = []

    for attack_spec in attack_orders:
        attack_name = attack_spec["attack"]
        attack_family = attack_spec["attack_family"]
        random_repeat = int(attack_spec["random_repeat"])
        order = np.asarray(attack_spec["order"], dtype=np.int64)

        if order.size != n_cands or np.unique(order).size != n_cands:
            raise ValueError(
                f"Invalid oracle ordering for {attack_name}: expected a "
                "permutation of all candidate indices."
            )

        adj = run["adj_orig"].clone()

        previous_k = 0
        previous_accuracy = None
        clean_accuracy = None
        maximum_drop_so_far = 0.0

        progress = tqdm(
            block_sizes,
            desc=(
                f"oracle seed={seed} "
                f"{run_index + 1}/{len(mining_runs)} "
                f"{attack_name} r={random_repeat}"
            ),
            leave=False,
        )

        for k in progress:
            k = int(k)

            newly_added_positions = order[previous_k:k]

            for candidate_index in newly_added_positions.tolist():
                _flip_candidate_inplace(
                    adj,
                    run,
                    candidate_index,
                )

            test_accuracy = float(
                accuracy(
                    context["model"],
                    context["attr"],
                    context["labels"],
                    context["idx_test"],
                    edge_index=adj,
                )
            )

            if clean_accuracy is None:
                clean_accuracy = test_accuracy

            n_edges_added = int(k - previous_k)

            if previous_accuracy is None:
                delta_accuracy = np.nan
            else:
                delta_accuracy = float(
                    previous_accuracy - test_accuracy
                )

            conditional_effect = _classify_conditional_effect(
                delta_accuracy
            )

            cumulative_accuracy_drop = float(
                clean_accuracy - test_accuracy
            )

            maximum_drop_so_far = max(
                maximum_drop_so_far,
                cumulative_accuracy_drop,
            )

            selected_indices = order[:k]
            selected_scores = ranking_scores[selected_indices]
            selected_positive_scores = np.clip(
                selected_scores,
                0.0,
                None,
            )

            selected_score_sum = float(selected_scores.sum())
            selected_score_mean = (
                float(selected_scores.mean())
                if selected_scores.size
                else np.nan
            )
            score_mass_captured_fraction = (
                float(selected_positive_scores.sum() / total_score_mass)
                if total_score_mass > 1e-12
                else np.nan
            )

            focus_limit = int(target_info["focus_prefix_limit"])

            rows.append({
                "seed": seed,
                "candidate_config_id": run[
                    "candidate_config_id"
                ],
                "candidate_set_size": int(
                    run["candidate_set_size"]
                ),
                "prbcd_candidate_fraction": float(
                    run["prbcd_candidate_fraction"]
                ),
                "actual_prbcd_fraction": float(
                    run["actual_prbcd_fraction"]
                ),
                "scoring_mode": scoring_mode,
                "target_kind": target_info["target_kind"],
                "ranking_score_name": target_info[
                    "ranking_score_name"
                ],
                "endpoint_mining_hop": int(
                    run["endpoint_mining_hop"]
                ),

                "mining_clean_accuracy": float(
                    run["clean_accuracy"]
                ),
                "clean_accuracy": float(clean_accuracy),

                "n_candidates": n_cands,
                "n_positive_examples": target_info[
                    "n_positive_examples"
                ],
                "positive_rate": target_info["positive_rate"],

                "focus_prefix_name": target_info[
                    "focus_prefix_name"
                ],
                "focus_prefix_limit": focus_limit,
                "focus_prefix_fraction": target_info[
                    "focus_prefix_fraction"
                ],
                "within_focus_prefix": bool(
                    k <= focus_limit
                ),
                "is_focus_prefix_endpoint": bool(
                    k == focus_limit
                ),

                # Compatibility columns: meaningful only for endpoint mode.
                "positive_prefix_limit": (
                    focus_limit
                    if scoring_mode == "endpoint"
                    else np.nan
                ),
                "within_positive_prefix": (
                    bool(k <= focus_limit)
                    if scoring_mode == "endpoint"
                    else False
                ),
                "is_positive_prefix_endpoint": (
                    bool(k == focus_limit)
                    if scoring_mode == "endpoint"
                    else False
                ),

                "constant_score_run": bool(
                    target_info["constant_score_run"]
                ),

                "attack": attack_name,
                "attack_family": attack_family,
                "random_repeat": random_repeat,

                "previous_k": int(previous_k),
                "k": int(k),
                "budget_fraction": float(k / n_cands),
                "n_edges_added": n_edges_added,

                "previous_test_accuracy": (
                    float(previous_accuracy)
                    if previous_accuracy is not None
                    else np.nan
                ),
                "test_accuracy": test_accuracy,

                "delta_accuracy": delta_accuracy,
                "conditional_effect": conditional_effect,

                "is_conditionally_harmful": (
                    np.nan
                    if n_edges_added == 0
                    else float(
                        conditional_effect
                        == "conditionally_harmful"
                    )
                ),
                "is_neutral": (
                    np.nan
                    if n_edges_added == 0
                    else float(
                        conditional_effect == "neutral"
                    )
                ),
                "is_cancelling": (
                    np.nan
                    if n_edges_added == 0
                    else float(
                        conditional_effect == "cancelling"
                    )
                ),

                "cumulative_accuracy_drop": (
                    cumulative_accuracy_drop
                ),
                "maximum_accuracy_drop_so_far": (
                    maximum_drop_so_far
                ),

                "selected_score_sum": selected_score_sum,
                "selected_score_mean": selected_score_mean,
                "score_mass_captured_fraction": (
                    score_mass_captured_fraction
                ),
            })

            previous_k = k
            previous_accuracy = test_accuracy

    run_df = pd.DataFrame(rows)
    run_df.to_csv(part_csv, index=False)
    oracle_frames.append(run_df)


# ============================================================
# Combine full trajectories
# ============================================================

oracle_raw_df = (
    pd.concat(
        oracle_frames,
        ignore_index=True,
    )
    if oracle_frames
    else pd.DataFrame()
)

if oracle_raw_df.empty:
    raise RuntimeError(
        "Oracle evaluation produced no rows."
    )


# ============================================================
# Average repeated random orderings within seed
# ============================================================

oracle_step_seed_id_cols = [
    "seed",
    "candidate_config_id",
    "candidate_set_size",
    "prbcd_candidate_fraction",
    "scoring_mode",
    "target_kind",
    "ranking_score_name",
    "focus_prefix_name",
    "endpoint_mining_hop",
    "k",
    "budget_fraction",
    "attack",
    "attack_family",
]

oracle_step_metrics = [
    "actual_prbcd_fraction",
    "clean_accuracy",
    "n_candidates",
    "n_positive_examples",
    "positive_rate",
    "focus_prefix_limit",
    "focus_prefix_fraction",
    "n_edges_added",
    "test_accuracy",
    "delta_accuracy",
    "is_conditionally_harmful",
    "is_neutral",
    "is_cancelling",
    "cumulative_accuracy_drop",
    "maximum_accuracy_drop_so_far",
    "selected_score_sum",
    "selected_score_mean",
    "score_mass_captured_fraction",
]

oracle_seed_step_df = _average_random_repeats_within_seed(
    oracle_raw_df,
    id_columns=oracle_step_seed_id_cols,
    numeric_columns=oracle_step_metrics,
)

oracle_step_group_cols = [
    column
    for column in oracle_step_seed_id_cols
    if column != "seed"
]

oracle_summary_df = aggregate_over_seeds(
    oracle_seed_step_df,
    oracle_step_group_cols,
    oracle_step_metrics + ["n_ordering_repetitions"],
)


# ============================================================
# Full-trajectory run-level report
# ============================================================

oracle_run_raw_df = _build_oracle_run_metrics(
    oracle_raw_df
)

threshold_metric_columns = [
    _threshold_column_name(threshold)
    for threshold in FIXED_ACCURACY_REDUCTIONS
]

oracle_run_metrics = [
    "actual_prbcd_fraction",
    "clean_accuracy",
    "n_candidates",
    "n_positive_examples",
    "positive_rate",
    "focus_prefix_limit",
    "focus_prefix_fraction",
    "constant_score_run",
    "final_k",
    "final_test_accuracy",
    "final_cumulative_accuracy_drop",
    "maximum_achieved_accuracy_drop",
    "k_at_maximum_accuracy_drop",
    "budget_fraction_at_maximum_accuracy_drop",

    "n_addition_steps",
    "n_conditionally_harmful_steps",
    "n_neutral_steps",
    "n_cancelling_steps",

    "proportion_conditionally_harmful",
    "proportion_neutral",
    "proportion_cancelling",

    "largest_harmful_step",
    "largest_cancelling_step",

    *threshold_metric_columns,
]

oracle_run_seed_id_cols = [
    "seed",
    "candidate_config_id",
    "candidate_set_size",
    "prbcd_candidate_fraction",
    "scoring_mode",
    "target_kind",
    "ranking_score_name",
    "focus_prefix_name",
    "endpoint_mining_hop",
    "attack",
    "attack_family",
]

oracle_seed_run_df = _average_random_repeats_within_seed(
    oracle_run_raw_df,
    id_columns=oracle_run_seed_id_cols,
    numeric_columns=oracle_run_metrics,
)

oracle_run_group_cols = [
    column
    for column in oracle_run_seed_id_cols
    if column != "seed"
]

oracle_run_summary_df = aggregate_over_seeds(
    oracle_seed_run_df,
    oracle_run_group_cols,
    oracle_run_metrics + ["n_ordering_repetitions"],
)


# ============================================================
# Mode-aware focused-prefix analysis
#
# endpoint:
#   exact binary positive-label prefix.
#
# subset_accuracy_drop:
#   exact equal candidate budget given by ORACLE_SUBSET_FOCUS_FRACTION.
#   Descending, ascending, and random attacks therefore use the same k.
# ============================================================

oracle_focus_prefix_raw_df = oracle_raw_df[
    (oracle_raw_df["focus_prefix_limit"] > 0)
    & (
        oracle_raw_df["k"]
        <= oracle_raw_df["focus_prefix_limit"]
    )
].copy()

if oracle_focus_prefix_raw_df.empty:
    oracle_focus_prefix_seed_df = pd.DataFrame()
    oracle_focus_prefix_summary_df = pd.DataFrame()
    oracle_focus_prefix_run_raw_df = pd.DataFrame()
    oracle_focus_prefix_seed_run_df = pd.DataFrame()
    oracle_focus_prefix_run_summary_df = pd.DataFrame()
    oracle_focus_prefix_endpoint_raw_df = pd.DataFrame()
    oracle_focus_prefix_endpoint_seed_df = pd.DataFrame()
    oracle_focus_prefix_endpoint_summary_df = pd.DataFrame()
else:
    oracle_focus_prefix_seed_df = _average_random_repeats_within_seed(
        oracle_focus_prefix_raw_df,
        id_columns=oracle_step_seed_id_cols,
        numeric_columns=oracle_step_metrics,
    )

    oracle_focus_prefix_summary_df = aggregate_over_seeds(
        oracle_focus_prefix_seed_df,
        oracle_step_group_cols,
        oracle_step_metrics + ["n_ordering_repetitions"],
    )

    oracle_focus_prefix_run_raw_df = _build_oracle_run_metrics(
        oracle_focus_prefix_raw_df
    )

    oracle_focus_prefix_seed_run_df = _average_random_repeats_within_seed(
        oracle_focus_prefix_run_raw_df,
        id_columns=oracle_run_seed_id_cols,
        numeric_columns=oracle_run_metrics,
    )

    oracle_focus_prefix_run_summary_df = aggregate_over_seeds(
        oracle_focus_prefix_seed_run_df,
        oracle_run_group_cols,
        oracle_run_metrics + ["n_ordering_repetitions"],
    )

    oracle_focus_prefix_endpoint_raw_df = oracle_raw_df[
        (oracle_raw_df["focus_prefix_limit"] > 0)
        & (
            oracle_raw_df["k"]
            == oracle_raw_df["focus_prefix_limit"]
        )
    ].copy()

    if oracle_focus_prefix_endpoint_raw_df.empty:
        raise RuntimeError(
            "The exact mode-aware focus-prefix endpoint was not evaluated."
        )

    focus_endpoint_seed_id_cols = [
        column
        for column in oracle_step_seed_id_cols
        if column not in {"k", "budget_fraction"}
    ]

    oracle_focus_prefix_endpoint_seed_df = (
        _average_random_repeats_within_seed(
            oracle_focus_prefix_endpoint_raw_df,
            id_columns=focus_endpoint_seed_id_cols,
            numeric_columns=[
                "actual_prbcd_fraction",
                "clean_accuracy",
                "n_candidates",
                "n_positive_examples",
                "positive_rate",
                "focus_prefix_limit",
                "focus_prefix_fraction",
                "test_accuracy",
                "cumulative_accuracy_drop",
                "maximum_accuracy_drop_so_far",
                "selected_score_sum",
                "selected_score_mean",
                "score_mass_captured_fraction",
            ],
        )
    )

    focus_endpoint_group_cols = [
        column
        for column in focus_endpoint_seed_id_cols
        if column != "seed"
    ]

    oracle_focus_prefix_endpoint_summary_df = aggregate_over_seeds(
        oracle_focus_prefix_endpoint_seed_df,
        focus_endpoint_group_cols,
        [
            "actual_prbcd_fraction",
            "clean_accuracy",
            "n_candidates",
            "n_positive_examples",
            "positive_rate",
            "focus_prefix_limit",
            "focus_prefix_fraction",
            "test_accuracy",
            "cumulative_accuracy_drop",
            "maximum_accuracy_drop_so_far",
            "selected_score_sum",
            "selected_score_mean",
            "score_mass_captured_fraction",
            "n_ordering_repetitions",
        ],
    )


# ============================================================
# Preserve the original endpoint positive-prefix variables
# ============================================================

oracle_positive_prefix_raw_df = oracle_focus_prefix_raw_df[
    oracle_focus_prefix_raw_df["scoring_mode"] == "endpoint"
].copy()

oracle_positive_prefix_summary_df = oracle_focus_prefix_summary_df[
    oracle_focus_prefix_summary_df["scoring_mode"] == "endpoint"
].copy() if not oracle_focus_prefix_summary_df.empty else pd.DataFrame()

oracle_positive_prefix_run_raw_df = oracle_focus_prefix_run_raw_df[
    oracle_focus_prefix_run_raw_df["scoring_mode"] == "endpoint"
].copy() if not oracle_focus_prefix_run_raw_df.empty else pd.DataFrame()

oracle_positive_prefix_run_summary_df = oracle_focus_prefix_run_summary_df[
    oracle_focus_prefix_run_summary_df["scoring_mode"] == "endpoint"
].copy() if not oracle_focus_prefix_run_summary_df.empty else pd.DataFrame()

oracle_positive_prefix_endpoint_raw_df = oracle_focus_prefix_endpoint_raw_df[
    oracle_focus_prefix_endpoint_raw_df["scoring_mode"] == "endpoint"
].copy() if not oracle_focus_prefix_endpoint_raw_df.empty else pd.DataFrame()

oracle_positive_prefix_endpoint_summary_df = (
    oracle_focus_prefix_endpoint_summary_df[
        oracle_focus_prefix_endpoint_summary_df["scoring_mode"] == "endpoint"
    ].copy()
    if not oracle_focus_prefix_endpoint_summary_df.empty
    else pd.DataFrame()
)


# ============================================================
# Subset focused-budget variables
# ============================================================

oracle_subset_focus_raw_df = oracle_focus_prefix_raw_df[
    oracle_focus_prefix_raw_df["scoring_mode"]
    == "subset_accuracy_drop"
].copy()

oracle_subset_focus_summary_df = oracle_focus_prefix_summary_df[
    oracle_focus_prefix_summary_df["scoring_mode"]
    == "subset_accuracy_drop"
].copy() if not oracle_focus_prefix_summary_df.empty else pd.DataFrame()

oracle_subset_focus_run_raw_df = oracle_focus_prefix_run_raw_df[
    oracle_focus_prefix_run_raw_df["scoring_mode"]
    == "subset_accuracy_drop"
].copy() if not oracle_focus_prefix_run_raw_df.empty else pd.DataFrame()

oracle_subset_focus_run_summary_df = oracle_focus_prefix_run_summary_df[
    oracle_focus_prefix_run_summary_df["scoring_mode"]
    == "subset_accuracy_drop"
].copy() if not oracle_focus_prefix_run_summary_df.empty else pd.DataFrame()

oracle_subset_focus_endpoint_raw_df = oracle_focus_prefix_endpoint_raw_df[
    oracle_focus_prefix_endpoint_raw_df["scoring_mode"]
    == "subset_accuracy_drop"
].copy() if not oracle_focus_prefix_endpoint_raw_df.empty else pd.DataFrame()

oracle_subset_focus_endpoint_summary_df = (
    oracle_focus_prefix_endpoint_summary_df[
        oracle_focus_prefix_endpoint_summary_df["scoring_mode"]
        == "subset_accuracy_drop"
    ].copy()
    if not oracle_focus_prefix_endpoint_summary_df.empty
    else pd.DataFrame()
)


# ============================================================
# Exact fixed-budget endpoint report for continuous subset scoring
# ============================================================

oracle_fixed_budget_raw_df = _build_fixed_budget_endpoint_table(
    oracle_raw_df
)

fixed_budget_seed_id_cols = [
    "seed",
    "candidate_config_id",
    "candidate_set_size",
    "prbcd_candidate_fraction",
    "scoring_mode",
    "target_kind",
    "ranking_score_name",
    "endpoint_mining_hop",
    "attack",
    "attack_family",
    "requested_budget_fraction",
    "requested_k",
]

fixed_budget_metrics = [
    "actual_prbcd_fraction",
    "clean_accuracy",
    "n_candidates",
    "n_positive_examples",
    "positive_rate",
    "test_accuracy",
    "cumulative_accuracy_drop",
    "maximum_accuracy_drop_so_far",
    "selected_score_sum",
    "selected_score_mean",
    "score_mass_captured_fraction",
]

if oracle_fixed_budget_raw_df.empty:
    oracle_fixed_budget_seed_df = pd.DataFrame()
    oracle_fixed_budget_summary_df = pd.DataFrame()
else:
    oracle_fixed_budget_seed_df = _average_random_repeats_within_seed(
        oracle_fixed_budget_raw_df,
        id_columns=fixed_budget_seed_id_cols,
        numeric_columns=fixed_budget_metrics,
    )

    fixed_budget_group_cols = [
        column
        for column in fixed_budget_seed_id_cols
        if column != "seed"
    ]

    oracle_fixed_budget_summary_df = aggregate_over_seeds(
        oracle_fixed_budget_seed_df,
        fixed_budget_group_cols,
        fixed_budget_metrics + ["n_ordering_repetitions"],
    )


# ============================================================
# Save
# ============================================================

# Full trajectories.
oracle_raw_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_mode_aware_steps_raw_all_seeds.csv",
    index=False,
)

oracle_seed_step_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_mode_aware_steps_random_averaged_within_seed.csv",
    index=False,
)

oracle_summary_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_mode_aware_steps_averaged.csv",
    index=False,
)

# Compatibility filenames used by the old plotting cells.
oracle_raw_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_conditional_steps_raw_all_seeds.csv",
    index=False,
)

oracle_summary_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_conditional_steps_averaged.csv",
    index=False,
)

# Run-level metrics.
oracle_run_raw_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_mode_aware_run_metrics_raw.csv",
    index=False,
)

oracle_seed_run_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_mode_aware_run_metrics_random_averaged_within_seed.csv",
    index=False,
)

oracle_run_summary_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_mode_aware_run_metrics_averaged.csv",
    index=False,
)

oracle_run_raw_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_conditional_run_metrics_raw.csv",
    index=False,
)

oracle_run_summary_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_conditional_run_metrics_averaged.csv",
    index=False,
)

# Generic mode-aware focused prefix.
oracle_focus_prefix_raw_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_focus_prefix_steps_raw.csv",
    index=False,
)

oracle_focus_prefix_summary_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_focus_prefix_steps_averaged.csv",
    index=False,
)

oracle_focus_prefix_run_raw_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_focus_prefix_run_metrics_raw.csv",
    index=False,
)

oracle_focus_prefix_run_summary_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_focus_prefix_run_metrics_averaged.csv",
    index=False,
)

oracle_focus_prefix_endpoint_raw_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_focus_prefix_endpoint_raw.csv",
    index=False,
)

oracle_focus_prefix_endpoint_summary_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_focus_prefix_endpoint_averaged.csv",
    index=False,
)

# Original endpoint-positive-prefix outputs remain endpoint-only.
oracle_positive_prefix_raw_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_positive_prefix_steps_raw.csv",
    index=False,
)

oracle_positive_prefix_summary_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_positive_prefix_steps_averaged.csv",
    index=False,
)

oracle_positive_prefix_run_raw_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_positive_prefix_run_metrics_raw.csv",
    index=False,
)

oracle_positive_prefix_run_summary_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_positive_prefix_run_metrics_averaged.csv",
    index=False,
)

oracle_positive_prefix_endpoint_raw_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_positive_prefix_endpoint_raw.csv",
    index=False,
)

oracle_positive_prefix_endpoint_summary_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_positive_prefix_endpoint_averaged.csv",
    index=False,
)

# Continuous subset-focused budget outputs.
oracle_subset_focus_raw_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_subset_top10pct_budget_steps_raw.csv",
    index=False,
)

oracle_subset_focus_summary_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_subset_top10pct_budget_steps_averaged.csv",
    index=False,
)

oracle_subset_focus_run_raw_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_subset_top10pct_budget_run_metrics_raw.csv",
    index=False,
)

oracle_subset_focus_run_summary_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_subset_top10pct_budget_run_metrics_averaged.csv",
    index=False,
)

oracle_subset_focus_endpoint_raw_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_subset_top10pct_budget_endpoint_raw.csv",
    index=False,
)

oracle_subset_focus_endpoint_summary_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_subset_top10pct_budget_endpoint_averaged.csv",
    index=False,
)

# Fixed equal-budget comparison tables for subset_accuracy_drop.
oracle_fixed_budget_raw_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_fixed_budget_endpoints_raw.csv",
    index=False,
)

oracle_fixed_budget_seed_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_fixed_budget_endpoints_random_averaged_within_seed.csv",
    index=False,
)

oracle_fixed_budget_summary_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_fixed_budget_endpoints_averaged.csv",
    index=False,
)


# ============================================================
# Display
# ============================================================

print("\nFull mode-aware step-level oracle results")
display(oracle_summary_df.head(40))

print("\nFull mode-aware run-level conditional-harmfulness report")
display(oracle_run_summary_df.head(40))

if not oracle_positive_prefix_run_summary_df.empty:
    print("\nEndpoint positive-prefix run-level results")
    display(
        oracle_positive_prefix_run_summary_df.head(30)
    )

    print(
        "\nEndpoint results after exactly n_positive "
        "jointly applied candidates"
    )
    display(
        oracle_positive_prefix_endpoint_summary_df.head(30)
    )

if not oracle_subset_focus_run_summary_df.empty:
    print(
        f"\nSubset-score focused-budget results "
        f"({100 * ORACLE_SUBSET_FOCUS_FRACTION:g}% candidate budget)"
    )
    display(
        oracle_subset_focus_endpoint_summary_df.head(40)
    )

print("\nExact fixed-budget comparisons for subset_accuracy_drop")
display(oracle_fixed_budget_summary_df.head(60))

print(
    "\nSaved mode-aware oracle results to:",
    ORACLE_OUT_DIR.resolve(),
)


### Analysis and plotting block

The oracle results are analyzed at two levels.

#### Step-level analysis

The step-level analysis reports the evolution of:

- cumulative attacked accuracy;
- marginal conditional effect \(\Delta A_k\);
- cumulative accuracy drop;
- maximum accuracy drop reached so far.

#### Run-level analysis

For each victim seed, candidate configuration, and ordering, the following measures are computed:

- proportion of conditionally harmful additions;
- proportion of neutral additions;
- proportion of cancelling additions;
- final cumulative accuracy drop;
- maximum achieved accuracy drop;
- number of edges at which the maximum drop occurs;
- number of edges required to reach fixed accuracy reductions;
- largest harmful marginal step;
- largest cancelling marginal step.

The main comparison is

\[
\text{mined ordering}
\quad\text{versus}\quad
\text{random ordering}.
\]

This experiment provides the central evidence for conditional harmfulness. It determines whether the mined ordering produces more damaging combinations and reveals how often harmfulness persists, disappears, or reverses as the perturbation block grows.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


# ============================================================
# Plot configuration
# ============================================================

plot_group_cols = [
    "candidate_config_id",
    "candidate_set_size",
    "prbcd_candidate_fraction",
    "scoring_mode",
    "endpoint_mining_hop",
]

# Later positive-prefix k values may contain fewer seeds when
# n_positive differs between victim seeds.
PLOT_COMPLETE_SEED_SUPPORT_ONLY = True

# Conditional-effect plot styling.
ADD_CONDITIONAL_TRENDLINES = True
CONDITIONAL_DATA_ALPHA = 0.30
CONDITIONAL_DATA_LINEWIDTH = 1.2
CONDITIONAL_DATA_MARKERSIZE = 3
CONDITIONAL_BAND_ALPHA = 0.06
CONDITIONAL_TREND_ALPHA = 1.0
CONDITIONAL_TREND_LINEWIDTH = 2.3

# Seed-normalized relative-drop plot styling.
RELATIVE_DROP_BAND_ALPHA = 0.15
RELATIVE_DROP_MARKERSIZE = 3


# ============================================================
# Helpers
# ============================================================

def _complete_seed_support(frame):
    """
    Keep only k values supported by the maximum number of seeds.

    This prevents the positive-prefix mean curve from silently
    changing its seed composition at larger k values.
    """
    if (
        frame is None
        or frame.empty
        or not PLOT_COMPLETE_SEED_SUPPORT_ONLY
        or "n_seeds" not in frame.columns
    ):
        return frame

    required_n_seeds = int(frame["n_seeds"].max())

    return frame[
        frame["n_seeds"] == required_n_seeds
    ].copy()


def _add_conditional_trendline(
    ax,
    x,
    y,
    *,
    color=None,
    label="",
):
    """
    Add a linear least-squares trendline.

    Negative slope:
        marginal harmfulness tends to decrease as k increases.

    Positive slope:
        marginal harmfulness tends to increase as k increases.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if x.size < 2 or np.unique(x).size < 2:
        return np.nan

    slope, intercept = np.polyfit(
        x,
        y,
        deg=1,
    )

    x_trend = np.linspace(
        float(x.min()),
        float(x.max()),
        100,
    )
    y_trend = slope * x_trend + intercept

    ax.plot(
        x_trend,
        y_trend,
        color=color,
        linestyle="--",
        linewidth=CONDITIONAL_TREND_LINEWIDTH,
        alpha=CONDITIONAL_TREND_ALPHA,
        zorder=4,
        label=(
            f"{label} trend "
            f"(slope={slope:.2e})"
        ),
    )

    return float(slope)


def _plot_conditional_curve(
    ax,
    attack_df,
    *,
    attack_name,
):
    """
    Plot a transparent observed conditional-effect curve, its
    mean ± SD band, and an opaque linear trendline.
    """
    if attack_df is None or attack_df.empty:
        return np.nan

    x_values = attack_df[
        "k"
    ].to_numpy(dtype=float)

    mean_values = attack_df[
        "delta_accuracy_mean"
    ].to_numpy(dtype=float)

    std_values = (
        attack_df[
            "delta_accuracy_std"
        ]
        .fillna(0.0)
        .to_numpy(dtype=float)
    )

    valid = (
        np.isfinite(x_values)
        & np.isfinite(mean_values)
        & np.isfinite(std_values)
    )

    x_values = x_values[valid]
    mean_values = mean_values[valid]
    std_values = std_values[valid]

    if x_values.size == 0:
        return np.nan

    # The observed points and connecting line are deliberately
    # transparent so the trendline remains visually dominant.
    data_line = ax.plot(
        x_values,
        mean_values,
        marker="o",
        markersize=CONDITIONAL_DATA_MARKERSIZE,
        linewidth=CONDITIONAL_DATA_LINEWIDTH,
        alpha=CONDITIONAL_DATA_ALPHA,
        zorder=2,
        label=f"{attack_name} observations",
    )[0]

    curve_color = data_line.get_color()

    ax.fill_between(
        x_values,
        mean_values - std_values,
        mean_values + std_values,
        color=curve_color,
        alpha=CONDITIONAL_BAND_ALPHA,
        zorder=1,
    )

    if not ADD_CONDITIONAL_TRENDLINES:
        return np.nan

    return _add_conditional_trendline(
        ax,
        x_values,
        mean_values,
        color=curve_color,
        label=attack_name,
    )


# ============================================================
# Seed-normalized relative accuracy-drop helpers
# ============================================================

def _relative_drop_summary(
    raw_df,
    *,
    group_cols,
):
    """
    Normalize every trajectory against its own seed-specific clean
    baseline before aggregating across seeds.

    For seed s and joint perturbation step k:

        relative_accuracy_drop_percent
            = 100 * (clean_accuracy - test_accuracy)
                    / clean_accuracy

    Positive values indicate an accuracy reduction. Every seed starts
    at 0%, independent of its absolute clean accuracy.
    """
    if raw_df is None or raw_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    required_columns = {
        *group_cols,
        "seed",
        "clean_accuracy",
        "test_accuracy",
    }
    missing_columns = required_columns - set(raw_df.columns)

    if missing_columns:
        raise KeyError(
            "Cannot calculate seed-normalized relative accuracy drops. "
            f"Missing columns: {sorted(missing_columns)}"
        )

    frame = raw_df.copy()

    frame["clean_accuracy"] = pd.to_numeric(
        frame["clean_accuracy"],
        errors="coerce",
    )
    frame["test_accuracy"] = pd.to_numeric(
        frame["test_accuracy"],
        errors="coerce",
    )

    valid_clean = (
        np.isfinite(frame["clean_accuracy"])
        & (frame["clean_accuracy"] > 0.0)
    )

    frame["relative_accuracy_drop_percent"] = np.where(
        valid_clean,
        100.0
        * (
            frame["clean_accuracy"]
            - frame["test_accuracy"]
        )
        / frame["clean_accuracy"],
        np.nan,
    )

    # One value per seed, attack and k. This prevents accidental
    # duplicate rows inside a seed from changing its statistical weight.
    seed_group_cols = [
        *group_cols,
        "seed",
    ]

    seed_df = (
        frame
        .groupby(
            seed_group_cols,
            dropna=False,
            as_index=False,
        )
        .agg(
            relative_accuracy_drop_percent=(
                "relative_accuracy_drop_percent",
                "mean",
            )
        )
    )

    summary_df = (
        seed_df
        .groupby(
            group_cols,
            dropna=False,
            as_index=False,
        )
        .agg(
            relative_accuracy_drop_percent_mean=(
                "relative_accuracy_drop_percent",
                "mean",
            ),
            relative_accuracy_drop_percent_std=(
                "relative_accuracy_drop_percent",
                "std",
            ),
            n_seeds=(
                "seed",
                "nunique",
            ),
        )
    )

    summary_df[
        "relative_accuracy_drop_percent_std"
    ] = summary_df[
        "relative_accuracy_drop_percent_std"
    ].fillna(0.0)

    summary_df[
        "relative_accuracy_drop_percent_sem"
    ] = (
        summary_df[
            "relative_accuracy_drop_percent_std"
        ]
        / np.sqrt(
            summary_df["n_seeds"].clip(lower=1)
        )
    )

    summary_df[
        "relative_accuracy_drop_percent_ci95"
    ] = (
        1.96
        * summary_df[
            "relative_accuracy_drop_percent_sem"
        ]
    )

    return seed_df, summary_df


def _plot_relative_drop_curve(
    ax,
    attack_df,
    *,
    attack_name,
):
    """Plot mean ± SD of the seed-normalized relative drop."""
    if attack_df is None or attack_df.empty:
        return

    attack_df = attack_df.sort_values("k")

    x_values = attack_df[
        "k"
    ].to_numpy(dtype=float)

    mean_values = attack_df[
        "relative_accuracy_drop_percent_mean"
    ].to_numpy(dtype=float)

    std_values = (
        attack_df[
            "relative_accuracy_drop_percent_std"
        ]
        .fillna(0.0)
        .to_numpy(dtype=float)
    )

    valid = (
        np.isfinite(x_values)
        & np.isfinite(mean_values)
        & np.isfinite(std_values)
    )

    x_values = x_values[valid]
    mean_values = mean_values[valid]
    std_values = std_values[valid]

    if x_values.size == 0:
        return

    line = ax.plot(
        x_values,
        mean_values,
        marker="o",
        markersize=RELATIVE_DROP_MARKERSIZE,
        label=attack_name,
    )[0]

    ax.fill_between(
        x_values,
        mean_values - std_values,
        mean_values + std_values,
        color=line.get_color(),
        alpha=RELATIVE_DROP_BAND_ALPHA,
    )


def _matching_group_rows(
    frame,
    *,
    columns,
    key,
):
    """Select rows matching a groupby key, including NaN keys."""
    if frame is None or frame.empty:
        return pd.DataFrame()

    mask = np.ones(len(frame), dtype=bool)

    for column, value in zip(columns, key):
        if pd.isna(value):
            mask &= frame[column].isna().to_numpy()
        else:
            mask &= (
                frame[column].to_numpy()
                == value
            )

    return frame.loc[mask].copy()


# ============================================================
# Build seed-normalized relative accuracy-drop summaries
# ============================================================

relative_drop_group_cols = [
    *plot_group_cols,
    "attack",
    "k",
]

(
    oracle_relative_drop_seed_df,
    oracle_relative_drop_summary_df,
) = _relative_drop_summary(
    oracle_raw_df,
    group_cols=relative_drop_group_cols,
)

if (
    "oracle_positive_prefix_raw_df" in globals()
    and oracle_positive_prefix_raw_df is not None
    and not oracle_positive_prefix_raw_df.empty
):
    (
        oracle_positive_prefix_relative_drop_seed_df,
        oracle_positive_prefix_relative_drop_summary_df,
    ) = _relative_drop_summary(
        oracle_positive_prefix_raw_df,
        group_cols=relative_drop_group_cols,
    )
else:
    oracle_positive_prefix_relative_drop_seed_df = (
        pd.DataFrame()
    )
    oracle_positive_prefix_relative_drop_summary_df = (
        pd.DataFrame()
    )

# Save exactly the normalized values used in the figures.
oracle_relative_drop_seed_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_relative_accuracy_drop_seed_level.csv",
    index=False,
)
oracle_relative_drop_summary_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_relative_accuracy_drop_averaged.csv",
    index=False,
)

if not oracle_positive_prefix_relative_drop_seed_df.empty:
    oracle_positive_prefix_relative_drop_seed_df.to_csv(
        ORACLE_OUT_DIR
        / "oracle_positive_prefix_relative_drop_seed_level.csv",
        index=False,
    )
    oracle_positive_prefix_relative_drop_summary_df.to_csv(
        ORACLE_OUT_DIR
        / "oracle_positive_prefix_relative_drop_averaged.csv",
        index=False,
    )


# ============================================================
# 1. Full percentage accuracy drop from seed-specific baseline
# ============================================================

for key, group in (
    oracle_relative_drop_summary_df.groupby(
        plot_group_cols,
        dropna=False,
    )
):
    fig, ax = plt.subplots(figsize=(9, 4.5))

    for attack_name, attack_df in group.groupby(
        "attack",
        sort=False,
    ):
        _plot_relative_drop_curve(
            ax,
            attack_df,
            attack_name=attack_name,
        )

    ax.axhline(
        0.0,
        linestyle="--",
        linewidth=0.9,
        label="clean baseline",
    )

    ax.set_xlabel(
        "k: number of edges applied jointly"
    )
    ax.set_ylabel(
        "Relative accuracy drop from "
        "seed-specific baseline (%)"
    )

    title = " | ".join(map(str, key))
    ax.set_title(
        "Oracle relative accuracy-drop comparison"
    )

    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()

    fig.savefig(
        PLOTS_DIR
        / (
            "oracle_relative_accuracy_drop__"
            f"{_oracle_safe(title)}.png"
        ),
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()


# ============================================================
# 2. Full marginal conditional effect with trendlines
# ============================================================

for key, group in oracle_summary_df.groupby(
    plot_group_cols,
    dropna=False,
):
    fig, ax = plt.subplots(figsize=(9, 4.5))

    trend_rows = []

    for attack_name, attack_df in group.groupby(
        "attack",
        sort=False,
    ):
        attack_df = (
            attack_df[
                attack_df["k"] > 0
            ]
            .sort_values("k")
        )

        slope = _plot_conditional_curve(
            ax,
            attack_df,
            attack_name=attack_name,
        )

        trend_rows.append({
            "attack": attack_name,
            "slope": slope,
        })

    ax.axhline(
        0.0,
        linestyle=":",
        linewidth=1.0,
        alpha=0.8,
        zorder=3,
        label="neutral effect",
    )

    ax.set_xlabel(
        "k after adding the new flip block"
    )
    ax.set_ylabel(
        r"$\Delta A_k=A(B_{k-1})-A(B_k)$"
    )

    title = " | ".join(map(str, key))
    ax.set_title(
        "Conditional effect of each newly added flip block"
    )

    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()

    fig.savefig(
        PLOTS_DIR
        / (
            "oracle_delta_accuracy_with_trend__"
            f"{_oracle_safe(title)}.png"
        ),
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()


# ============================================================
# 3. Full cumulative accuracy drop in percentage points
# ============================================================

for key, group in oracle_summary_df.groupby(
    plot_group_cols,
    dropna=False,
):
    fig, ax = plt.subplots(figsize=(9, 4.5))

    for attack_name, attack_df in group.groupby(
        "attack",
        sort=False,
    ):
        attack_df = attack_df.sort_values("k")

        add_mean_std_band(
            ax,
            attack_df["k"],
            attack_df[
                "cumulative_accuracy_drop_mean"
            ],
            attack_df[
                "cumulative_accuracy_drop_std"
            ],
            marker="o",
            markersize=3,
            label=attack_name,
        )

    for threshold in FIXED_ACCURACY_REDUCTIONS:
        ax.axhline(
            float(threshold),
            linestyle=":",
            linewidth=0.8,
            label=f"{threshold * 100:g} pp reduction",
        )

    ax.set_xlabel(
        "k: number of edges applied jointly"
    )
    ax.set_ylabel(
        "Cumulative test-accuracy drop "
        "(accuracy proportion)"
    )

    title = " | ".join(map(str, key))
    ax.set_title(
        "Absolute cumulative oracle attack effect"
    )

    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()

    fig.savefig(
        PLOTS_DIR
        / (
            "oracle_cumulative_drop__"
            f"{_oracle_safe(title)}.png"
        ),
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()


# ============================================================
# 4. Full conditional-effect proportions
# ============================================================

for key, group in oracle_run_summary_df.groupby(
    plot_group_cols,
    dropna=False,
):
    group = (
        group
        .sort_values("attack")
        .reset_index(drop=True)
    )

    attack_names = group["attack"].tolist()
    x = np.arange(len(attack_names))
    width = 0.25

    metrics = [
        (
            "proportion_conditionally_harmful",
            "Conditionally harmful",
        ),
        (
            "proportion_neutral",
            "Neutral",
        ),
        (
            "proportion_cancelling",
            "Cancelling",
        ),
    ]

    fig, ax = plt.subplots(figsize=(9, 4.8))

    for metric_index, (metric, label) in enumerate(
        metrics
    ):
        offset = (
            metric_index
            - (len(metrics) - 1) / 2
        ) * width

        means = group[
            f"{metric}_mean"
        ].to_numpy(dtype=float)

        stds = (
            group[f"{metric}_std"]
            .fillna(0.0)
            .to_numpy(dtype=float)
        )

        ax.bar(
            x + offset,
            means,
            width,
            yerr=stds,
            capsize=3,
            label=label,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        attack_names,
        rotation=15,
        ha="right",
    )

    ax.set_ylim(0, 1)
    ax.set_ylabel(
        "Proportion of addition steps"
    )

    title = " | ".join(map(str, key))
    ax.set_title(
        "Conditional-effect composition over the full candidate set"
    )

    ax.grid(
        True,
        axis="y",
        alpha=0.3,
    )
    ax.legend()
    fig.tight_layout()

    fig.savefig(
        PLOTS_DIR
        / (
            "oracle_effect_proportions__"
            f"{_oracle_safe(title)}.png"
        ),
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()


# ============================================================
# Positive-prefix plots
# ============================================================

if not oracle_positive_prefix_summary_df.empty:

    # ========================================================
    # 5. Percentage drop within the positive-label budget
    # ========================================================

    if not oracle_positive_prefix_relative_drop_summary_df.empty:

        for key, group in (
            oracle_positive_prefix_relative_drop_summary_df.groupby(
                plot_group_cols,
                dropna=False,
            )
        ):
            fig, ax = plt.subplots(figsize=(9, 4.5))

            for attack_name, attack_df in group.groupby(
                "attack",
                sort=False,
            ):
                attack_df = _complete_seed_support(
                    attack_df.sort_values("k")
                )

                if attack_df.empty:
                    continue

                _plot_relative_drop_curve(
                    ax,
                    attack_df,
                    attack_name=attack_name,
                )

            ax.axhline(
                0.0,
                linestyle="--",
                linewidth=0.9,
                label="clean baseline",
            )

            source_group = _matching_group_rows(
                oracle_positive_prefix_summary_df,
                columns=plot_group_cols,
                key=key,
            )

            mean_positive_count = (
                float(
                    source_group[
                        "n_positive_examples_mean"
                    ].mean()
                )
                if (
                    not source_group.empty
                    and "n_positive_examples_mean"
                    in source_group.columns
                )
                else np.nan
            )

            positive_suffix = (
                f" | mean n_positive="
                f"{mean_positive_count:.1f}"
                if np.isfinite(mean_positive_count)
                else ""
            )

            ax.set_xlabel(
                "k: candidates applied within the "
                "positive-label budget"
            )
            ax.set_ylabel(
                "Relative accuracy drop from "
                "seed-specific baseline (%)"
            )

            title = " | ".join(map(str, key))
            ax.set_title(
                "Relative accuracy drop within the mined positive prefix"
            )

            ax.grid(True, alpha=0.3)
            ax.legend()
            fig.tight_layout()

            fig.savefig(
                PLOTS_DIR
                / (
                    "oracle_positive_prefix_relative_drop__"
                    f"{_oracle_safe(title)}.png"
                ),
                dpi=200,
                bbox_inches="tight",
            )

            plt.show()


    # ========================================================
    # 6. Marginal effects within the positive prefix
    #    with prominent trendlines
    # ========================================================

    for key, group in (
        oracle_positive_prefix_summary_df.groupby(
            plot_group_cols,
            dropna=False,
        )
    ):
        fig, ax = plt.subplots(figsize=(9, 4.5))

        for attack_name, attack_df in group.groupby(
            "attack",
            sort=False,
        ):
            attack_df = (
                attack_df[
                    attack_df["k"] > 0
                ]
                .sort_values("k")
            )

            attack_df = _complete_seed_support(
                attack_df
            )

            _plot_conditional_curve(
                ax,
                attack_df,
                attack_name=attack_name,
            )

        ax.axhline(
            0.0,
            linestyle=":",
            linewidth=1.0,
            alpha=0.8,
            zorder=3,
            label="neutral effect",
        )

        ax.set_xlabel(
            "k: candidates applied within the "
            "positive-label budget"
        )
        ax.set_ylabel(
            r"$\Delta A_k=A(B_{k-1})-A(B_k)$"
        )

        title = " | ".join(map(str, key))
        ax.set_title(
            "Conditional effects within the positive prefix"
        )

        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
        fig.tight_layout()

        fig.savefig(
            PLOTS_DIR
            / (
                "oracle_positive_prefix_delta_with_trend__"
                f"{_oracle_safe(title)}.png"
            ),
            dpi=200,
            bbox_inches="tight",
        )

        plt.show()


    # ========================================================
    # 7. Absolute cumulative drop within the positive prefix
    # ========================================================

    for key, group in (
        oracle_positive_prefix_summary_df.groupby(
            plot_group_cols,
            dropna=False,
        )
    ):
        fig, ax = plt.subplots(figsize=(9, 4.5))

        for attack_name, attack_df in group.groupby(
            "attack",
            sort=False,
        ):
            attack_df = _complete_seed_support(
                attack_df.sort_values("k")
            )

            if attack_df.empty:
                continue

            add_mean_std_band(
                ax,
                attack_df["k"],
                attack_df[
                    "cumulative_accuracy_drop_mean"
                ],
                attack_df[
                    "cumulative_accuracy_drop_std"
                ],
                marker="o",
                markersize=3,
                label=attack_name,
            )

        ax.set_xlabel(
            "k: candidates applied within the "
            "positive-label budget"
        )
        ax.set_ylabel(
            "Cumulative test-accuracy drop"
        )

        title = " | ".join(map(str, key))
        ax.set_title(
            "Cumulative effect of the mined positive prefix"
        )

        ax.grid(True, alpha=0.3)
        ax.legend()
        fig.tight_layout()

        fig.savefig(
            PLOTS_DIR
            / (
                "oracle_positive_prefix_cumulative__"
                f"{_oracle_safe(title)}.png"
            ),
            dpi=200,
            bbox_inches="tight",
        )

        plt.show()


    # ========================================================
    # 8. Conditional-effect proportions inside the prefix
    # ========================================================

    for key, group in (
        oracle_positive_prefix_run_summary_df.groupby(
            plot_group_cols,
            dropna=False,
        )
    ):
        group = (
            group
            .sort_values("attack")
            .reset_index(drop=True)
        )

        attack_names = group["attack"].tolist()
        x = np.arange(len(attack_names))
        width = 0.25

        metrics = [
            (
                "proportion_conditionally_harmful",
                "Conditionally harmful",
            ),
            (
                "proportion_neutral",
                "Neutral",
            ),
            (
                "proportion_cancelling",
                "Cancelling",
            ),
        ]

        fig, ax = plt.subplots(
            figsize=(9, 4.8)
        )

        for metric_index, (
            metric,
            label,
        ) in enumerate(metrics):
            offset = (
                metric_index
                - (len(metrics) - 1) / 2
            ) * width

            means = group[
                f"{metric}_mean"
            ].to_numpy(dtype=float)

            stds = (
                group[f"{metric}_std"]
                .fillna(0.0)
                .to_numpy(dtype=float)
            )

            ax.bar(
                x + offset,
                means,
                width,
                yerr=stds,
                capsize=3,
                label=label,
            )

        ax.set_xticks(x)
        ax.set_xticklabels(
            attack_names,
            rotation=15,
            ha="right",
        )

        ax.set_ylim(0, 1)
        ax.set_ylabel(
            "Proportion of addition steps"
        )

        title = " | ".join(map(str, key))
        ax.set_title(
            "Conditional-effect composition within the positive prefix"
        )

        ax.grid(
            True,
            axis="y",
            alpha=0.3,
        )
        ax.legend()
        fig.tight_layout()

        fig.savefig(
            PLOTS_DIR
            / (
                "oracle_positive_prefix_proportions__"
                f"{_oracle_safe(title)}.png"
            ),
            dpi=200,
            bbox_inches="tight",
        )

        plt.show()


    # ========================================================
    # 9. Exact result after k=n_positive candidates
    # ========================================================

    for key, group in (
        oracle_positive_prefix_endpoint_summary_df.groupby(
            plot_group_cols,
            dropna=False,
        )
    ):
        group = (
            group
            .sort_values("attack")
            .reset_index(drop=True)
        )

        fig, ax = plt.subplots(
            figsize=(8, 4.8)
        )

        x = np.arange(len(group))

        means = group[
            "cumulative_accuracy_drop_mean"
        ].to_numpy(dtype=float)

        stds = (
            group[
                "cumulative_accuracy_drop_std"
            ]
            .fillna(0.0)
            .to_numpy(dtype=float)
        )

        ax.bar(
            x,
            means,
            yerr=stds,
            capsize=4,
        )

        ax.set_xticks(x)
        ax.set_xticklabels(
            group["attack"],
            rotation=15,
            ha="right",
        )

        mean_positive_count = float(
            group[
                "n_positive_examples_mean"
            ].mean()
        )

        ax.set_ylabel(
            "Cumulative test-accuracy drop"
        )

        title = " | ".join(map(str, key))
        ax.set_title(
            "Effect after applying the positive-label budget"
        )

        ax.grid(
            True,
            axis="y",
            alpha=0.3,
        )
        fig.tight_layout()

        fig.savefig(
            PLOTS_DIR
            / (
                "oracle_positive_prefix_endpoint__"
                f"{_oracle_safe(title)}.png"
            ),
            dpi=200,
            bbox_inches="tight",
        )

        plt.show()


# ============================================================
# Compact full-trajectory report
# ============================================================

full_report_columns = [
    "candidate_config_id",
    "candidate_set_size",
    "prbcd_candidate_fraction",
    "scoring_mode",
    "endpoint_mining_hop",
    "attack",

    "n_positive_examples_mean",

    "proportion_conditionally_harmful_mean",
    "proportion_neutral_mean",
    "proportion_cancelling_mean",

    "final_cumulative_accuracy_drop_mean",
    "maximum_achieved_accuracy_drop_mean",
    "k_at_maximum_accuracy_drop_mean",

    *[
        f"{_threshold_column_name(threshold)}_mean"
        for threshold in FIXED_ACCURACY_REDUCTIONS
    ],

    "n_seeds",
]

full_report_columns = [
    column
    for column in full_report_columns
    if column in oracle_run_summary_df.columns
]

oracle_conditional_report_df = (
    oracle_run_summary_df[
        full_report_columns
    ]
    .sort_values(
        [
            "candidate_config_id",
            "scoring_mode",
            "endpoint_mining_hop",
            "attack",
        ]
    )
    .reset_index(drop=True)
)

oracle_conditional_report_df.to_csv(
    ORACLE_OUT_DIR
    / "oracle_conditional_harmfulness_report.csv",
    index=False,
)


# ============================================================
# Compact positive-prefix report
# ============================================================

if not oracle_positive_prefix_run_summary_df.empty:
    positive_report_columns = [
        "candidate_config_id",
        "candidate_set_size",
        "prbcd_candidate_fraction",
        "scoring_mode",
        "endpoint_mining_hop",
        "attack",

        "n_positive_examples_mean",

        "proportion_conditionally_harmful_mean",
        "proportion_neutral_mean",
        "proportion_cancelling_mean",

        "final_test_accuracy_mean",
        "final_cumulative_accuracy_drop_mean",
        "maximum_achieved_accuracy_drop_mean",
        "k_at_maximum_accuracy_drop_mean",

        *[
            f"{_threshold_column_name(threshold)}_mean"
            for threshold in FIXED_ACCURACY_REDUCTIONS
        ],

        "n_seeds",
    ]

    positive_report_columns = [
        column
        for column in positive_report_columns
        if column
        in oracle_positive_prefix_run_summary_df.columns
    ]

    oracle_positive_prefix_report_df = (
        oracle_positive_prefix_run_summary_df[
            positive_report_columns
        ]
        .sort_values(
            [
                "candidate_config_id",
                "scoring_mode",
                "endpoint_mining_hop",
                "attack",
            ]
        )
        .reset_index(drop=True)
    )

    oracle_positive_prefix_report_df.to_csv(
        ORACLE_OUT_DIR
        / "oracle_positive_prefix_report.csv",
        index=False,
    )

else:
    oracle_positive_prefix_report_df = (
        pd.DataFrame()
    )


print("Full conditional-harmfulness report")
display(oracle_conditional_report_df)

if not oracle_positive_prefix_report_df.empty:
    print(
        "\nPositive-prefix "
        "conditional-harmfulness report"
    )
    display(
        oracle_positive_prefix_report_df
    )

print(
    "\nSaved oracle plots and reports to:",
    ORACLE_OUT_DIR.resolve(),
)

In [ ]:
# ============================================================
# RQ2: Characterize mined edge groups for endpoint and subset scoring
#
# Placement:
#   Run this cell after the direct-mining cell has created `mining_runs`.
#
# Comparison rules:
#   endpoint:
#       group 1 = endpoint label 1
#       group 0 = endpoint label 0
#
#   subset_accuracy_drop:
#       group 1 = top 10% of observed score_raw values
#       group 0 = bottom 10% of observed score_raw values
#       the middle 80% are excluded from this characterization
#
# Outputs:
#   - one row per selected candidate edge and mining run
#   - one row per endpoint occurrence and mining run
#   - descriptive statistics for every individual seed/run
#   - seed-level aggregate statistics and paired group-1 minus group-0 effects
#   - per-run comparison plots
#   - aggregate-over-seeds comparison plots
# ============================================================

from pathlib import Path
from collections import Counter
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import sparse as scipy_sparse
from scipy.stats import ttest_1samp, wilcoxon


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

RQ2_CHAR_SCORING_MODES = {
    "endpoint",
    "subset_accuracy_drop",
}

RQ2_CHAR_ENDPOINT_LABEL_THRESHOLD = 0.5
RQ2_CHAR_SUBSET_EXTREME_FRACTION = 0.10

# Add a matched control group based on the complete clean graph.
#
# Edge-level control:
#   every undirected edge in the clean graph, exactly once.
#
# Endpoint-level control:
#   both endpoint occurrences of every clean edge. This keeps the
#   observation unit comparable to the mined candidate-edge endpoints and
#   also makes partner-dependent metrics well-defined.
RQ2_CHAR_INCLUDE_CLEAN_GRAPH_CONTROL = True
RQ2_CHAR_CONTROL_GROUP = 2
RQ2_CHAR_CONTROL_NAME = "Whole clean graph"

# None means: analyze all candidate configurations.
RQ2_CHAR_CANDIDATE_CONFIG_IDS = None

# Applied only to endpoint-mining runs. Subset runs use h=0 as bookkeeping
# and are not removed by this filter.
RQ2_CHAR_ENDPOINT_HOPS = None

RQ2_CHAR_MAKE_PER_RUN_PLOTS = True
RQ2_CHAR_MAKE_AGGREGATE_PLOTS = True
RQ2_CHAR_SHOW_PLOTS = True
RQ2_CHAR_SAVE_DPI = 220

RQ2_CHAR_OUT_DIR = (
    Path("extendedPlotting")
    / "rq2_mode_aware_edge_characterization"
)
RQ2_CHAR_TABLE_DIR = RQ2_CHAR_OUT_DIR / "tables"
RQ2_CHAR_PER_RUN_PLOT_DIR = RUN_PLOTS_DIR
RQ2_CHAR_AGG_PLOT_DIR = RUN_PLOTS_DIR

for _directory in [
    RQ2_CHAR_OUT_DIR,
    RQ2_CHAR_TABLE_DIR,
    RQ2_CHAR_PER_RUN_PLOT_DIR,
    RQ2_CHAR_AGG_PLOT_DIR,
]:
    _directory.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# General helpers
# ------------------------------------------------------------


def _rq2_char_safe(value):
    return re.sub(r"[^a-zA-Z0-9_.=-]+", "_", str(value)).strip("_")


def _rq2_char_cpu_tensor(value, *, dtype=None):
    if torch.is_tensor(value):
        tensor = value.detach().cpu()
    elif scipy_sparse.issparse(value):
        tensor = torch.from_numpy(value.toarray())
    elif hasattr(value, "to_dense"):
        dense = value.to_dense()
        tensor = dense.detach().cpu() if torch.is_tensor(dense) else torch.as_tensor(dense)
    else:
        tensor = torch.as_tensor(value)

    if dtype is not None:
        tensor = tensor.to(dtype=dtype)
    return tensor


def _rq2_char_dense_features(context):
    value = context["attr"]

    if scipy_sparse.issparse(value):
        array = value.toarray()
        return np.asarray(array, dtype=np.float32)

    if torch.is_tensor(value):
        tensor = value.detach().cpu()
        if tensor.is_sparse:
            tensor = tensor.to_dense()
        return tensor.float().numpy()

    if hasattr(value, "to_dense"):
        value = value.to_dense()
        if torch.is_tensor(value):
            return value.detach().cpu().float().numpy()

    return np.asarray(value, dtype=np.float32)


def _rq2_char_clear_model_cache_and_call(model, x, adj):
    has_cache = hasattr(model, "adj_preped")
    previous_cache = getattr(model, "adj_preped", None) if has_cache else None

    try:
        if has_cache:
            model.adj_preped = None
        output = model(x, adj)
    finally:
        if has_cache:
            model.adj_preped = previous_cache

    if isinstance(output, (tuple, list)):
        output = output[0]
    return output


def _rq2_char_clean_predictions(run):
    """Return clean confidence and margin statistics for every node."""
    context = run["context"]
    model = context["model"]
    model.eval()

    try:
        model_device = next(model.parameters()).device
    except StopIteration:
        model_device = torch.device("cpu")

    x_model = context["attr"]
    if hasattr(x_model, "to"):
        x_model = x_model.to(model_device)

    adj_model = run["adj_orig"]
    if hasattr(adj_model, "to"):
        adj_model = adj_model.to(model_device)

    labels_model = context["labels"]
    if hasattr(labels_model, "to"):
        labels_model = labels_model.to(model_device, dtype=torch.long)
    else:
        labels_model = torch.as_tensor(labels_model, device=model_device, dtype=torch.long)

    with torch.no_grad():
        logits = _rq2_char_clear_model_cache_and_call(
            model,
            x_model,
            adj_model,
        )
        probabilities = torch.softmax(logits, dim=-1)
        predictions = probabilities.argmax(dim=-1)
        confidence = probabilities.max(dim=-1).values

        top_k = min(2, int(probabilities.size(1)))
        top_values = torch.topk(probabilities, k=top_k, dim=-1).values
        if top_k == 1:
            prediction_margin = top_values[:, 0]
        else:
            prediction_margin = top_values[:, 0] - top_values[:, 1]

        true_probability = probabilities.gather(
            1,
            labels_model.view(-1, 1),
        ).squeeze(1)

        other_probabilities = probabilities.clone()
        other_probabilities.scatter_(
            1,
            labels_model.view(-1, 1),
            float("-inf"),
        )
        strongest_other_probability = other_probabilities.max(dim=-1).values

        # Positive values mean that the true class is ahead of every competing
        # class. Negative values mean that another class has higher probability.
        true_class_margin = true_probability - strongest_other_probability
        correct = predictions.eq(labels_model)

    return {
        "prediction": predictions.detach().cpu().numpy().astype(int),
        "clean_correct": correct.detach().cpu().numpy().astype(bool),
        "confidence": confidence.detach().cpu().numpy().astype(float),
        "prediction_margin": prediction_margin.detach().cpu().numpy().astype(float),
        "true_class_probability": true_probability.detach().cpu().numpy().astype(float),
        "true_class_margin": true_class_margin.detach().cpu().numpy().astype(float),
    }


def _rq2_char_clean_graph(context):
    """Build clean undirected neighborhoods and node-level class composition."""
    n_nodes = int(context["n_nodes"])
    edge_index = _rq2_char_cpu_tensor(
        context["edge_index"],
        dtype=torch.long,
    ).numpy()
    labels = _rq2_char_cpu_tensor(
        context["labels"],
        dtype=torch.long,
    ).numpy().astype(int)
    features = _rq2_char_dense_features(context)

    if features.ndim != 2 or features.shape[0] != n_nodes:
        raise ValueError(
            "The feature matrix must have shape [n_nodes, n_features], "
            f"but received {features.shape}."
        )

    neighbors = [set() for _ in range(n_nodes)]
    clean_edge_set = set()

    for source, target in zip(edge_index[0], edge_index[1]):
        source = int(source)
        target = int(target)
        if source == target:
            continue
        u, v = (source, target) if source < target else (target, source)
        clean_edge_set.add((u, v))
        neighbors[u].add(v)
        neighbors[v].add(u)

    degree = np.asarray([len(values) for values in neighbors], dtype=float)
    n_classes = int(labels.max()) + 1 if labels.size else 0
    neighborhood_class_counts = np.zeros((n_nodes, n_classes), dtype=np.int64)

    for node, node_neighbors in enumerate(neighbors):
        if not node_neighbors:
            continue
        neighbor_labels = labels[np.fromiter(node_neighbors, dtype=int)]
        neighborhood_class_counts[node] = np.bincount(
            neighbor_labels,
            minlength=n_classes,
        )

    same_class_neighbor_fraction = np.full(n_nodes, np.nan, dtype=float)
    neighborhood_class_entropy = np.full(n_nodes, np.nan, dtype=float)
    neighborhood_majority_fraction = np.full(n_nodes, np.nan, dtype=float)

    for node in range(n_nodes):
        if degree[node] <= 0:
            continue

        counts = neighborhood_class_counts[node].astype(float)
        probabilities = counts / counts.sum()
        positive = probabilities > 0

        same_class_neighbor_fraction[node] = (
            neighborhood_class_counts[node, labels[node]] / degree[node]
        )
        neighborhood_majority_fraction[node] = probabilities.max()

        entropy = -float(np.sum(probabilities[positive] * np.log(probabilities[positive])))
        if n_classes > 1:
            entropy /= math.log(n_classes)
        neighborhood_class_entropy[node] = entropy

    feature_l2_norm = np.linalg.norm(features, axis=1)

    return {
        "n_nodes": n_nodes,
        "labels": labels,
        "features": features,
        "feature_l2_norm": feature_l2_norm,
        "neighbors": neighbors,
        "clean_edge_set": clean_edge_set,
        "degree": degree,
        "neighborhood_class_counts": neighborhood_class_counts,
        "same_class_neighbor_fraction": same_class_neighbor_fraction,
        "neighborhood_class_entropy": neighborhood_class_entropy,
        "neighborhood_majority_fraction": neighborhood_majority_fraction,
    }


def _rq2_char_pair_summary(first, second, prefix, output):
    output[f"pair_{prefix}_mean"] = float(np.nanmean([first, second]))
    output[f"pair_{prefix}_min"] = float(np.nanmin([first, second]))
    output[f"pair_{prefix}_max"] = float(np.nanmax([first, second]))
    output[f"pair_{prefix}_absdiff"] = float(abs(first - second))



def _rq2_char_comparison_rule(scoring_mode):
    scoring_mode = str(scoring_mode)

    if scoring_mode == "endpoint":
        return "endpoint_label_1_vs_0"

    if scoring_mode == "subset_accuracy_drop":
        percentage = 100.0 * float(
            RQ2_CHAR_SUBSET_EXTREME_FRACTION
        )
        percentage_text = f"{percentage:g}".replace(".", "p")
        return f"top_{percentage_text}pct_vs_bottom_{percentage_text}pct"

    raise ValueError(
        f"Unsupported scoring_mode={scoring_mode!r}."
    )


def _rq2_char_group_names(scoring_mode):
    scoring_mode = str(scoring_mode)

    if scoring_mode == "endpoint":
        return {
            RQ2_CHAR_CONTROL_GROUP: RQ2_CHAR_CONTROL_NAME,
            0: "Label 0 (no endpoint hit)",
            1: "Label 1 (endpoint-harmful)",
        }

    if scoring_mode == "subset_accuracy_drop":
        percentage = 100.0 * float(
            RQ2_CHAR_SUBSET_EXTREME_FRACTION
        )
        return {
            RQ2_CHAR_CONTROL_GROUP: RQ2_CHAR_CONTROL_NAME,
            0: f"Bottom {percentage:g}% score",
            1: f"Top {percentage:g}% score",
        }

    raise ValueError(
        f"Unsupported scoring_mode={scoring_mode!r}."
    )


def _rq2_char_plot_group_names(scoring_mode):
    """
    Compact multiline labels used only on plot x-axes.

    The full descriptive names from _rq2_char_group_names remain
    unchanged in tables, CSV files, and metadata.
    """
    scoring_mode = str(scoring_mode)

    if scoring_mode == "endpoint":
        return {
            RQ2_CHAR_CONTROL_GROUP: "Clean graph",
            0: "Label 0\n(no hit)",
            1: "Label 1\n(harmful)",
        }

    if scoring_mode == "subset_accuracy_drop":
        percentage = (
            100.0
            * float(RQ2_CHAR_SUBSET_EXTREME_FRACTION)
        )
        return {
            RQ2_CHAR_CONTROL_GROUP: "Clean graph",
            0: f"Bottom {percentage:g}%",
            1: f"Top {percentage:g}%",
        }

    raise ValueError(
        f"Unsupported scoring_mode={scoring_mode!r}."
    )


def _rq2_char_format_group_axis(
    axis,
    plot_group_order,
    plot_group_names,
):
    """Apply readable centered multiline labels to a subplot."""
    axis.set_xticks(
        range(len(plot_group_order))
    )
    axis.set_xticklabels(
        [
            plot_group_names[group_id]
            for group_id in plot_group_order
        ],
        rotation=0,
        ha="center",
        fontsize=10,
    )
    axis.tick_params(
        axis="x",
        pad=8,
        length=3,
    )

    # Add horizontal breathing room around the three categories.
    axis.margins(x=0.12)


def _rq2_char_plot_group_order():
    """Control first, then the two mined comparison groups."""
    return [
        RQ2_CHAR_CONTROL_GROUP,
        0,
        1,
    ]


def _rq2_char_difference_text(scoring_mode):
    names = _rq2_char_group_names(scoring_mode)
    return f"{names[1]} minus {names[0]}"


def _rq2_char_aligned_array(
    run,
    key,
    n_candidates,
    *,
    fallback=None,
    dtype=float,
):
    if key in run:
        value = run[key]
    elif key in run.get("mining_result", {}):
        value = run["mining_result"][key]
    else:
        value = fallback

    if value is None:
        return None

    values = np.asarray(
        _rq2_char_cpu_tensor(value).numpy()
        if torch.is_tensor(value)
        else value,
        dtype=dtype,
    ).reshape(-1)

    if values.size != int(n_candidates):
        raise ValueError(
            f"{key} has length {values.size}, but the run contains "
            f"{n_candidates} downstream candidates."
        )

    return values


def _rq2_char_select_groups(run, n_candidates):
    """
    Return candidate indices and group metadata for one mining run.

    Endpoint mode preserves the original binary-label comparison.

    Subset mode selects equal-sized top and bottom score deciles. Ranking is
    based on score_raw. Ties at a boundary are resolved deterministically by
    sample index so both groups always contain the same number of edges.
    """
    scoring_mode = str(run.get("scoring_mode"))
    comparison_rule = _rq2_char_comparison_rule(scoring_mode)
    sample_indices = np.arange(int(n_candidates), dtype=np.int64)

    if scoring_mode == "endpoint":
        labels = _rq2_char_aligned_array(
            run,
            "labels_changed",
            n_candidates,
            dtype=float,
        )

        if labels is None:
            labels = _rq2_char_aligned_array(
                run,
                "endpoint_labels",
                n_candidates,
                dtype=float,
            )

        if labels is None:
            raise KeyError(
                "Endpoint run does not contain labels_changed or "
                "endpoint_labels."
            )

        groups = (
            labels > float(RQ2_CHAR_ENDPOINT_LABEL_THRESHOLD)
        ).astype(np.int64)

        return {
            "selected_indices": sample_indices,
            "group_by_index": groups,
            "comparison_rule": comparison_rule,
            "score_raw": labels.astype(np.float64),
            "score_norm": labels.astype(np.float64),
            "score_percentile": labels.astype(np.float64),
            "inclusion_count": np.ones(
                int(n_candidates),
                dtype=np.int64,
            ),
            "n_available": int(n_candidates),
            "n_selected_per_group": None,
            "bottom_cutoff_score": np.nan,
            "top_cutoff_score": np.nan,
            "n_at_bottom_cutoff": np.nan,
            "n_at_top_cutoff": np.nan,
            "constant_score_run": False,
        }

    if scoring_mode != "subset_accuracy_drop":
        raise ValueError(
            f"Unsupported scoring_mode={scoring_mode!r}."
        )

    score_raw = _rq2_char_aligned_array(
        run,
        "score_raw",
        n_candidates,
        dtype=float,
    )

    if score_raw is None:
        raise KeyError(
            "subset_accuracy_drop run does not contain score_raw. "
            "Recompute mining with the mean-drop schema."
        )

    score_norm = _rq2_char_aligned_array(
        run,
        "score_norm",
        n_candidates,
        fallback=np.full(n_candidates, np.nan),
        dtype=float,
    )

    score_percentile = _rq2_char_aligned_array(
        run,
        "score_percentile",
        n_candidates,
        fallback=np.full(n_candidates, np.nan),
        dtype=float,
    )

    inclusion_count = _rq2_char_aligned_array(
        run,
        "inclusion_count",
        n_candidates,
        fallback=np.ones(n_candidates, dtype=np.int64),
        dtype=np.int64,
    )

    valid_mask = (
        np.isfinite(score_raw)
        & (inclusion_count > 0)
    )
    valid_indices = sample_indices[valid_mask]
    n_valid = int(valid_indices.size)

    if n_valid < 2:
        raise RuntimeError(
            "At least two observed subset-scored candidates are needed "
            "for a top-versus-bottom comparison."
        )

    n_per_group = int(
        np.floor(
            float(RQ2_CHAR_SUBSET_EXTREME_FRACTION)
            * n_valid
        )
    )
    n_per_group = max(1, n_per_group)
    n_per_group = min(n_per_group, n_valid // 2)

    if n_per_group <= 0:
        raise RuntimeError(
            "The subset extreme groups would be empty."
        )

    # Primary key: score_raw. Secondary key: original sample index.
    valid_order = np.lexsort(
        (
            valid_indices,
            score_raw[valid_indices],
        )
    )
    ordered_valid_indices = valid_indices[valid_order]

    bottom_indices = ordered_valid_indices[:n_per_group]
    top_indices = ordered_valid_indices[-n_per_group:]

    group_by_index = np.full(
        int(n_candidates),
        -1,
        dtype=np.int64,
    )
    group_by_index[bottom_indices] = 0
    group_by_index[top_indices] = 1

    bottom_cutoff = float(
        np.max(score_raw[bottom_indices])
    )
    top_cutoff = float(
        np.min(score_raw[top_indices])
    )

    constant_score_run = bool(
        np.allclose(
            score_raw[valid_indices],
            score_raw[valid_indices][0],
        )
    )

    if constant_score_run:
        warnings.warn(
            "All observed subset scores are identical. The selected top "
            "and bottom groups differ only through deterministic tie-breaking, "
            "so the comparison is not substantively interpretable.",
            RuntimeWarning,
        )

    return {
        "selected_indices": np.concatenate(
            [bottom_indices, top_indices]
        ),
        "group_by_index": group_by_index,
        "comparison_rule": comparison_rule,
        "score_raw": score_raw,
        "score_norm": score_norm,
        "score_percentile": score_percentile,
        "inclusion_count": inclusion_count,
        "n_available": n_valid,
        "n_selected_per_group": int(n_per_group),
        "bottom_cutoff_score": bottom_cutoff,
        "top_cutoff_score": top_cutoff,
        "n_at_bottom_cutoff": int(
            np.isclose(
                score_raw[valid_indices],
                bottom_cutoff,
            ).sum()
        ),
        "n_at_top_cutoff": int(
            np.isclose(
                score_raw[valid_indices],
                top_cutoff,
            ).sum()
        ),
        "constant_score_run": constant_score_run,
    }



def _rq2_char_build_clean_graph_control_rows(
    *,
    run,
    clean,
    prediction,
    seed,
    candidate_config_id,
    scoring_mode,
    comparison_rule,
    endpoint_hop,
):
    """
    Build a matched control group from the complete clean graph.

    Each undirected clean edge contributes:
      - one edge-level control observation;
      - two endpoint-level control observations.

    Using endpoint occurrences instead of unique nodes keeps the control
    observation unit aligned with the mined-edge endpoint table and allows
    partner-dependent quantities such as partner-class neighbor fraction to
    be computed consistently.
    """
    if not RQ2_CHAR_INCLUDE_CLEAN_GRAPH_CONTROL:
        return [], []

    group_names = _rq2_char_group_names(
        scoring_mode
    )
    control_group = int(
        RQ2_CHAR_CONTROL_GROUP
    )
    control_name = group_names[
        control_group
    ]

    control_edge_rows = []
    control_endpoint_rows = []

    clean_edges = sorted(
        clean["clean_edge_set"]
    )

    for control_index, (u, v) in enumerate(
        clean_edges
    ):
        u = int(u)
        v = int(v)

        u_label = int(
            clean["labels"][u]
        )
        v_label = int(
            clean["labels"][v]
        )
        same_true_class = bool(
            u_label == v_label
        )

        # Match the candidate-edge definitions exactly: exclude the direct
        # partner only from the overlap neighborhoods.
        u_local_neighbors = (
            clean["neighbors"][u] - {v}
        )
        v_local_neighbors = (
            clean["neighbors"][v] - {u}
        )
        common_neighbors = (
            u_local_neighbors
            & v_local_neighbors
        )
        neighbor_union = (
            u_local_neighbors
            | v_local_neighbors
        )
        smaller_neighborhood = min(
            len(u_local_neighbors),
            len(v_local_neighbors),
        )

        common_neighbor_count = float(
            len(common_neighbors)
        )
        local_neighbor_jaccard = (
            common_neighbor_count
            / len(neighbor_union)
            if neighbor_union
            else 0.0
        )
        local_overlap_coefficient = (
            common_neighbor_count
            / smaller_neighborhood
            if smaller_neighborhood > 0
            else 0.0
        )

        x_u = clean["features"][u]
        x_v = clean["features"][v]
        norm_u = float(
            clean["feature_l2_norm"][u]
        )
        norm_v = float(
            clean["feature_l2_norm"][v]
        )

        feature_cosine_similarity = (
            float(
                np.dot(x_u, x_v)
                / (norm_u * norm_v)
            )
            if norm_u > 0 and norm_v > 0
            else np.nan
        )
        feature_l2_distance = float(
            np.linalg.norm(x_u - x_v)
        )

        support_u = x_u != 0
        support_v = x_v != 0
        support_union = np.logical_or(
            support_u,
            support_v,
        ).sum()
        feature_support_jaccard = (
            float(
                np.logical_and(
                    support_u,
                    support_v,
                ).sum()
                / support_union
            )
            if support_union > 0
            else np.nan
        )

        u_degree = float(
            clean["degree"][u]
        )
        v_degree = float(
            clean["degree"][v]
        )

        u_partner_class_fraction = (
            float(
                clean[
                    "neighborhood_class_counts"
                ][u, v_label]
                / u_degree
            )
            if u_degree > 0
            else np.nan
        )
        v_partner_class_fraction = (
            float(
                clean[
                    "neighborhood_class_counts"
                ][v, u_label]
                / v_degree
            )
            if v_degree > 0
            else np.nan
        )

        edge_uid = (
            f"seed={seed}|cfg={candidate_config_id}|"
            f"mode={scoring_mode}|rule={comparison_rule}|"
            f"h={endpoint_hop}|control=clean_graph|"
            f"edge={u}-{v}"
        )

        edge_row = {
            "dataset": globals().get(
                "DATASET",
                "unknown",
            ),
            "seed": int(seed),
            "candidate_config_id": str(
                candidate_config_id
            ),
            "candidate_set_size": int(
                run["candidate_set_size"]
            ),
            "prbcd_candidate_fraction": float(
                run["prbcd_candidate_fraction"]
            ),
            "actual_prbcd_fraction": float(
                run["actual_prbcd_fraction"]
            ),
            "scoring_mode": str(
                scoring_mode
            ),
            "comparison_rule": str(
                comparison_rule
            ),
            "endpoint_mining_hop": int(
                endpoint_hop
            ),
            "sample_index": int(
                control_index
            ),
            "edge_uid": edge_uid,
            "u": u,
            "v": v,

            "is_clean_graph_control": True,
            "label_value": np.nan,
            "label_group": control_group,
            "label_name": control_name,
            "group_name": control_name,

            # The control is not selected by the mining score.
            "score_raw": np.nan,
            "score_norm": np.nan,
            "score_percentile": np.nan,
            "inclusion_count": 0,
            "n_selected_per_group": np.nan,
            "bottom_cutoff_score": np.nan,
            "top_cutoff_score": np.nan,
            "constant_score_run": False,

            "exists_clean": True,
            "action": "delete",
            "u_true_class": u_label,
            "v_true_class": v_label,
            "true_class_pair": (
                f"{min(u_label, v_label)}-"
                f"{max(u_label, v_label)}"
            ),
            "same_true_class": float(
                same_true_class
            ),
            "feature_cosine_similarity": (
                feature_cosine_similarity
            ),
            "feature_l2_distance": (
                feature_l2_distance
            ),
            "feature_support_jaccard": (
                feature_support_jaccard
            ),
            "common_neighbors": (
                common_neighbor_count
            ),
            "local_neighbor_jaccard": float(
                local_neighbor_jaccard
            ),
            "local_overlap_coefficient": float(
                local_overlap_coefficient
            ),
            "u_partner_class_neighbor_fraction": (
                u_partner_class_fraction
            ),
            "v_partner_class_neighbor_fraction": (
                v_partner_class_fraction
            ),
        }

        endpoint_values = {
            "confidence": (
                float(
                    prediction["confidence"][u]
                ),
                float(
                    prediction["confidence"][v]
                ),
            ),
            "prediction_margin": (
                float(
                    prediction[
                        "prediction_margin"
                    ][u]
                ),
                float(
                    prediction[
                        "prediction_margin"
                    ][v]
                ),
            ),
            "true_class_probability": (
                float(
                    prediction[
                        "true_class_probability"
                    ][u]
                ),
                float(
                    prediction[
                        "true_class_probability"
                    ][v]
                ),
            ),
            "true_class_margin": (
                float(
                    prediction[
                        "true_class_margin"
                    ][u]
                ),
                float(
                    prediction[
                        "true_class_margin"
                    ][v]
                ),
            ),
            "degree": (
                u_degree,
                v_degree,
            ),
            "same_class_neighbor_fraction": (
                float(
                    clean[
                        "same_class_neighbor_fraction"
                    ][u]
                ),
                float(
                    clean[
                        "same_class_neighbor_fraction"
                    ][v]
                ),
            ),
            "neighborhood_class_entropy": (
                float(
                    clean[
                        "neighborhood_class_entropy"
                    ][u]
                ),
                float(
                    clean[
                        "neighborhood_class_entropy"
                    ][v]
                ),
            ),
            "neighborhood_majority_fraction": (
                float(
                    clean[
                        "neighborhood_majority_fraction"
                    ][u]
                ),
                float(
                    clean[
                        "neighborhood_majority_fraction"
                    ][v]
                ),
            ),
            "partner_class_neighbor_fraction": (
                u_partner_class_fraction,
                v_partner_class_fraction,
            ),
        }

        for (
            metric_name,
            (u_value, v_value),
        ) in endpoint_values.items():
            edge_row[f"u_{metric_name}"] = (
                u_value
            )
            edge_row[f"v_{metric_name}"] = (
                v_value
            )
            _rq2_char_pair_summary(
                u_value,
                v_value,
                metric_name,
                edge_row,
            )

        control_edge_rows.append(
            edge_row
        )

        for (
            endpoint_role,
            node,
            partner,
            node_label,
            partner_label,
            value_index,
        ) in [
            (
                "u",
                u,
                v,
                u_label,
                v_label,
                0,
            ),
            (
                "v",
                v,
                u,
                v_label,
                u_label,
                1,
            ),
        ]:
            control_endpoint_rows.append({
                "dataset": edge_row["dataset"],
                "seed": int(seed),
                "candidate_config_id": str(
                    candidate_config_id
                ),
                "candidate_set_size": int(
                    run["candidate_set_size"]
                ),
                "prbcd_candidate_fraction": float(
                    run[
                        "prbcd_candidate_fraction"
                    ]
                ),
                "actual_prbcd_fraction": float(
                    run[
                        "actual_prbcd_fraction"
                    ]
                ),
                "scoring_mode": str(
                    scoring_mode
                ),
                "comparison_rule": str(
                    comparison_rule
                ),
                "endpoint_mining_hop": int(
                    endpoint_hop
                ),
                "sample_index": int(
                    control_index
                ),
                "edge_uid": edge_uid,
                "is_clean_graph_control": True,
                "label_value": np.nan,
                "label_group": control_group,
                "label_name": control_name,
                "group_name": control_name,
                "score_raw": np.nan,
                "score_norm": np.nan,
                "score_percentile": np.nan,
                "inclusion_count": 0,
                "endpoint_role": endpoint_role,
                "node": int(node),
                "partner_node": int(partner),
                "node_true_class": int(
                    node_label
                ),
                "partner_true_class": int(
                    partner_label
                ),
                "same_true_class": float(
                    same_true_class
                ),
                "clean_prediction": int(
                    prediction["prediction"][node]
                ),
                "clean_correct": float(
                    prediction[
                        "clean_correct"
                    ][node]
                ),
                "confidence": endpoint_values[
                    "confidence"
                ][value_index],
                "prediction_margin": (
                    endpoint_values[
                        "prediction_margin"
                    ][value_index]
                ),
                "true_class_probability": (
                    endpoint_values[
                        "true_class_probability"
                    ][value_index]
                ),
                "true_class_margin": (
                    endpoint_values[
                        "true_class_margin"
                    ][value_index]
                ),
                "degree": endpoint_values[
                    "degree"
                ][value_index],
                "same_class_neighbor_fraction": (
                    endpoint_values[
                        "same_class_neighbor_fraction"
                    ][value_index]
                ),
                "neighborhood_class_entropy": (
                    endpoint_values[
                        "neighborhood_class_entropy"
                    ][value_index]
                ),
                "neighborhood_majority_fraction": (
                    endpoint_values[
                        "neighborhood_majority_fraction"
                    ][value_index]
                ),
                "partner_class_neighbor_fraction": (
                    endpoint_values[
                        "partner_class_neighbor_fraction"
                    ][value_index]
                ),
            })

    return (
        control_edge_rows,
        control_endpoint_rows,
    )

def _rq2_char_describe_by_run(frame, metrics, level_name):
    group_columns = [
        "seed",
        "candidate_config_id",
        "scoring_mode",
        "comparison_rule",
        "endpoint_mining_hop",
        "label_group",
    ]
    rows = []

    for group_key, group in frame.groupby(group_columns, dropna=False):
        metadata = dict(zip(group_columns, group_key))

        for feature in metrics:
            values = (
                pd.to_numeric(
                    group[feature],
                    errors="coerce",
                )
                .dropna()
                .to_numpy(float)
            )

            if values.size == 0:
                continue

            q1, median, q3 = np.quantile(
                values,
                [0.25, 0.50, 0.75],
            )

            rows.append({
                **metadata,
                "analysis_level": level_name,
                "feature": feature,
                "n_observations": int(values.size),
                "mean": float(values.mean()),
                "std": (
                    float(values.std(ddof=1))
                    if values.size > 1
                    else 0.0
                ),
                "median": float(median),
                "q1": float(q1),
                "q3": float(q3),
                "iqr": float(q3 - q1),
                "minimum": float(values.min()),
                "maximum": float(values.max()),
            })

    return pd.DataFrame(rows)


def _rq2_char_aggregate_over_seeds(run_summary):
    group_columns = [
        "analysis_level",
        "candidate_config_id",
        "scoring_mode",
        "comparison_rule",
        "endpoint_mining_hop",
        "label_group",
        "feature",
    ]

    if run_summary.empty:
        return pd.DataFrame()

    aggregate = (
        run_summary
        .groupby(group_columns, dropna=False)
        .agg(
            n_seeds=("seed", "nunique"),
            seed_mean_mean=("mean", "mean"),
            seed_mean_std=("mean", "std"),
            seed_median_mean=("median", "mean"),
            mean_n_observations=("n_observations", "mean"),
        )
        .reset_index()
    )

    aggregate["seed_mean_std"] = (
        aggregate["seed_mean_std"].fillna(0.0)
    )
    aggregate["seed_mean_sem"] = (
        aggregate["seed_mean_std"]
        / np.sqrt(
            aggregate["n_seeds"].clip(lower=1)
        )
    )
    aggregate["seed_mean_ci95"] = (
        1.96 * aggregate["seed_mean_sem"]
    )
    return aggregate


def _rq2_char_paired_seed_effects(run_summary):
    rows = []
    group_columns = [
        "analysis_level",
        "candidate_config_id",
        "scoring_mode",
        "comparison_rule",
        "endpoint_mining_hop",
        "feature",
    ]

    for group_key, group in run_summary.groupby(
        group_columns,
        dropna=False,
    ):
        metadata = dict(
            zip(group_columns, group_key)
        )

        pivot = group.pivot_table(
            index="seed",
            columns="label_group",
            values="mean",
            aggfunc="first",
        )

        if 0 not in pivot.columns or 1 not in pivot.columns:
            continue

        paired = pivot[[0, 1]].dropna()
        differences = paired[1] - paired[0]
        n_seeds = int(len(differences))

        if n_seeds == 0:
            continue

        difference_std = (
            float(differences.std(ddof=1))
            if n_seeds > 1
            else 0.0
        )
        difference_sem = (
            difference_std / math.sqrt(n_seeds)
            if n_seeds > 0
            else np.nan
        )

        if (
            n_seeds >= 2
            and not np.allclose(differences, 0.0)
        ):
            try:
                t_result = ttest_1samp(
                    differences,
                    popmean=0.0,
                    nan_policy="omit",
                )
                paired_t_statistic = float(
                    t_result.statistic
                )
                paired_t_p = float(
                    t_result.pvalue
                )
            except Exception:
                paired_t_statistic = np.nan
                paired_t_p = np.nan

            try:
                wilcoxon_result = wilcoxon(
                    differences
                )
                wilcoxon_statistic = float(
                    wilcoxon_result.statistic
                )
                wilcoxon_p = float(
                    wilcoxon_result.pvalue
                )
            except Exception:
                wilcoxon_statistic = np.nan
                wilcoxon_p = np.nan
        else:
            paired_t_statistic = np.nan
            paired_t_p = np.nan
            wilcoxon_statistic = np.nan
            wilcoxon_p = np.nan

        group_names = _rq2_char_group_names(
            metadata["scoring_mode"]
        )

        rows.append({
            **metadata,
            "group_0_name": group_names[0],
            "group_1_name": group_names[1],
            "n_paired_seeds": n_seeds,
            "group_0_seed_mean": float(
                paired[0].mean()
            ),
            "group_1_seed_mean": float(
                paired[1].mean()
            ),
            "mean_difference_group1_minus_group0": float(
                differences.mean()
            ),
            # Backward-compatible aliases for endpoint-oriented code.
            "label_0_seed_mean": float(
                paired[0].mean()
            ),
            "label_1_seed_mean": float(
                paired[1].mean()
            ),
            "mean_difference_label1_minus_label0": float(
                differences.mean()
            ),
            "difference_std": difference_std,
            "difference_sem": difference_sem,
            "difference_ci95": 1.96 * difference_sem,
            "fraction_seeds_positive_difference": float(
                (differences > 0).mean()
            ),
            "paired_t_statistic": paired_t_statistic,
            "paired_t_p": paired_t_p,
            "wilcoxon_statistic": wilcoxon_statistic,
            "wilcoxon_p": wilcoxon_p,
        })

    return pd.DataFrame(rows)



# ------------------------------------------------------------
# Select mining runs
# ------------------------------------------------------------

if "mining_runs" not in globals() or not mining_runs:
    raise RuntimeError(
        "`mining_runs` is missing. Run the direct-mining execution cell first."
    )

allowed_scoring_modes = {
    str(value)
    for value in RQ2_CHAR_SCORING_MODES
}

rq2_characterization_runs = [
    run
    for run in mining_runs
    if str(run.get("scoring_mode")) in allowed_scoring_modes
]

if RQ2_CHAR_CANDIDATE_CONFIG_IDS is not None:
    allowed_configs = {
        str(value)
        for value in RQ2_CHAR_CANDIDATE_CONFIG_IDS
    }
    rq2_characterization_runs = [
        run
        for run in rq2_characterization_runs
        if str(run["candidate_config_id"]) in allowed_configs
    ]

if RQ2_CHAR_ENDPOINT_HOPS is not None:
    allowed_hops = {
        int(value)
        for value in RQ2_CHAR_ENDPOINT_HOPS
    }
    rq2_characterization_runs = [
        run
        for run in rq2_characterization_runs
        if (
            str(run.get("scoring_mode"))
            != "endpoint"
            or int(run["endpoint_mining_hop"]) in allowed_hops
        )
    ]

if not rq2_characterization_runs:
    raise RuntimeError(
        "No mining runs match the characterization filters."
    )

print(
    "Mode-aware characterization runs:",
    len(rq2_characterization_runs),
)
print(
    "Scoring modes:",
    sorted({
        str(run["scoring_mode"])
        for run in rq2_characterization_runs
    }),
)
print(
    "Seeds:",
    sorted({
        int(run["seed"])
        for run in rq2_characterization_runs
    }),
)
print(
    "Candidate configurations:",
    sorted({
        str(run["candidate_config_id"])
        for run in rq2_characterization_runs
    }),
)
print(
    "Endpoint hops:",
    sorted({
        int(run["endpoint_mining_hop"])
        for run in rq2_characterization_runs
        if str(run["scoring_mode"]) == "endpoint"
    }),
)


# ------------------------------------------------------------
# Build raw edge-level and endpoint-level tables
# ------------------------------------------------------------

edge_rows = []
endpoint_rows = []
selection_diagnostic_rows = []

graph_cache = {}
prediction_cache = {}

for run_index, run in enumerate(rq2_characterization_runs):
    seed = int(run["seed"])
    candidate_config_id = str(
        run["candidate_config_id"]
    )
    scoring_mode = str(run["scoring_mode"])
    comparison_rule = _rq2_char_comparison_rule(
        scoring_mode
    )
    endpoint_hop = int(
        run["endpoint_mining_hop"]
    )
    context = run["context"]

    edge_index_cpu = _rq2_char_cpu_tensor(
        context["edge_index"],
        dtype=torch.long,
    )
    topology_signature = (
        int(context["n_nodes"]),
        int(edge_index_cpu.shape[1]),
        int(
            edge_index_cpu[
                :,
                : min(128, edge_index_cpu.shape[1]),
            ].sum().item()
        ),
    )

    if topology_signature not in graph_cache:
        graph_cache[topology_signature] = (
            _rq2_char_clean_graph(context)
        )
    clean = graph_cache[topology_signature]

    prediction_key = (
        seed,
        id(context["model"]),
    )
    if prediction_key not in prediction_cache:
        prediction_cache[prediction_key] = (
            _rq2_char_clean_predictions(run)
        )
    prediction = prediction_cache[prediction_key]

    src = (
        _rq2_char_cpu_tensor(
            run["src"],
            dtype=torch.long,
        )
        .numpy()
        .astype(int)
        .reshape(-1)
    )
    dst = (
        _rq2_char_cpu_tensor(
            run["dst"],
            dtype=torch.long,
        )
        .numpy()
        .astype(int)
        .reshape(-1)
    )

    if len(src) != len(dst):
        raise RuntimeError(
            "Candidate source and destination arrays are not aligned for "
            f"seed={seed}, config={candidate_config_id}, "
            f"mode={scoring_mode}, h={endpoint_hop}."
        )

    n_candidates = int(len(src))

    selection = _rq2_char_select_groups(
        run,
        n_candidates,
    )
    group_names = _rq2_char_group_names(
        scoring_mode
    )

    selected_indices = np.asarray(
        selection["selected_indices"],
        dtype=np.int64,
    )

    if selected_indices.size == 0:
        raise RuntimeError(
            "No candidate indices were selected for characterization."
        )

    selected_group_values = (
        selection["group_by_index"][selected_indices]
    )

    group_0_count = int(
        np.sum(selected_group_values == 0)
    )
    group_1_count = int(
        np.sum(selected_group_values == 1)
    )

    if group_0_count == 0 or group_1_count == 0:
        raise RuntimeError(
            "Both comparison groups must be non-empty. "
            f"Got group_0={group_0_count}, "
            f"group_1={group_1_count} for "
            f"seed={seed}, config={candidate_config_id}, "
            f"mode={scoring_mode}."
        )

    selection_diagnostic_rows.append({
        "seed": seed,
        "candidate_config_id": candidate_config_id,
        "candidate_set_size": int(
            run["candidate_set_size"]
        ),
        "prbcd_candidate_fraction": float(
            run["prbcd_candidate_fraction"]
        ),
        "actual_prbcd_fraction": float(
            run["actual_prbcd_fraction"]
        ),
        "scoring_mode": scoring_mode,
        "comparison_rule": comparison_rule,
        "endpoint_mining_hop": endpoint_hop,
        "n_downstream_candidates": n_candidates,
        "n_available_for_comparison": int(
            selection["n_available"]
        ),
        "n_selected_group_0": group_0_count,
        "n_selected_group_1": group_1_count,
        "n_selected_total": int(
            selected_indices.size
        ),
        "selected_fraction_of_available": float(
            selected_indices.size
            / max(1, int(selection["n_available"]))
        ),
        "n_selected_per_group": (
            selection["n_selected_per_group"]
        ),
        "bottom_cutoff_score": (
            selection["bottom_cutoff_score"]
        ),
        "top_cutoff_score": (
            selection["top_cutoff_score"]
        ),
        "n_at_bottom_cutoff": (
            selection["n_at_bottom_cutoff"]
        ),
        "n_at_top_cutoff": (
            selection["n_at_top_cutoff"]
        ),
        "constant_score_run": bool(
            selection["constant_score_run"]
        ),
        "clean_graph_control_enabled": bool(
            RQ2_CHAR_INCLUDE_CLEAN_GRAPH_CONTROL
        ),
        "n_control_clean_edges": (
            int(len(clean["clean_edge_set"]))
            if RQ2_CHAR_INCLUDE_CLEAN_GRAPH_CONTROL
            else 0
        ),
        "n_control_endpoint_occurrences": (
            2 * int(len(clean["clean_edge_set"]))
            if RQ2_CHAR_INCLUDE_CLEAN_GRAPH_CONTROL
            else 0
        ),
    })

    for sample_index in selected_indices.tolist():
        sample_index = int(sample_index)

        u = int(src[sample_index])
        v = int(dst[sample_index])

        label_group = int(
            selection["group_by_index"][sample_index]
        )
        if label_group not in {0, 1}:
            raise RuntimeError(
                "Selected candidate does not have a valid group."
            )

        score_raw = float(
            selection["score_raw"][sample_index]
        )
        score_norm = float(
            selection["score_norm"][sample_index]
        )
        score_percentile = float(
            selection["score_percentile"][sample_index]
        )
        inclusion_count = int(
            selection["inclusion_count"][sample_index]
        )

        # Endpoint behavior remains unchanged: label_value is the binary
        # endpoint label. For subset mode it is the canonical raw score.
        label_value = score_raw

        canonical_u, canonical_v = (
            (u, v)
            if u < v
            else (v, u)
        )

        edge_uid = (
            f"seed={seed}|cfg={candidate_config_id}|"
            f"mode={scoring_mode}|rule={comparison_rule}|"
            f"h={endpoint_hop}|sample={sample_index}|"
            f"edge={canonical_u}-{canonical_v}"
        )

        u_label = int(clean["labels"][u])
        v_label = int(clean["labels"][v])
        same_true_class = bool(
            u_label == v_label
        )

        # Exclude the direct partner from each local neighborhood. This makes
        # overlap statistics comparable for existing and non-existing edges.
        u_local_neighbors = (
            clean["neighbors"][u] - {v}
        )
        v_local_neighbors = (
            clean["neighbors"][v] - {u}
        )
        common_neighbors = (
            u_local_neighbors
            & v_local_neighbors
        )
        neighbor_union = (
            u_local_neighbors
            | v_local_neighbors
        )
        smaller_neighborhood = min(
            len(u_local_neighbors),
            len(v_local_neighbors),
        )

        common_neighbor_count = float(
            len(common_neighbors)
        )
        local_neighbor_jaccard = (
            common_neighbor_count
            / len(neighbor_union)
            if neighbor_union
            else 0.0
        )
        local_overlap_coefficient = (
            common_neighbor_count
            / smaller_neighborhood
            if smaller_neighborhood > 0
            else 0.0
        )

        x_u = clean["features"][u]
        x_v = clean["features"][v]
        norm_u = float(
            clean["feature_l2_norm"][u]
        )
        norm_v = float(
            clean["feature_l2_norm"][v]
        )
        feature_cosine_similarity = (
            float(
                np.dot(x_u, x_v)
                / (norm_u * norm_v)
            )
            if norm_u > 0 and norm_v > 0
            else np.nan
        )
        feature_l2_distance = float(
            np.linalg.norm(x_u - x_v)
        )

        support_u = x_u != 0
        support_v = x_v != 0
        support_union = np.logical_or(
            support_u,
            support_v,
        ).sum()
        feature_support_jaccard = (
            float(
                np.logical_and(
                    support_u,
                    support_v,
                ).sum()
                / support_union
            )
            if support_union > 0
            else np.nan
        )

        u_degree = float(
            clean["degree"][u]
        )
        v_degree = float(
            clean["degree"][v]
        )

        u_partner_class_fraction = (
            float(
                clean[
                    "neighborhood_class_counts"
                ][u, v_label]
                / u_degree
            )
            if u_degree > 0
            else np.nan
        )
        v_partner_class_fraction = (
            float(
                clean[
                    "neighborhood_class_counts"
                ][v, u_label]
                / v_degree
            )
            if v_degree > 0
            else np.nan
        )

        exists_clean = bool(
            (canonical_u, canonical_v)
            in clean["clean_edge_set"]
        )

        edge_row = {
            "dataset": globals().get(
                "DATASET",
                "unknown",
            ),
            "seed": seed,
            "candidate_config_id": candidate_config_id,
            "candidate_set_size": int(
                run["candidate_set_size"]
            ),
            "prbcd_candidate_fraction": float(
                run["prbcd_candidate_fraction"]
            ),
            "actual_prbcd_fraction": float(
                run["actual_prbcd_fraction"]
            ),
            "scoring_mode": scoring_mode,
            "comparison_rule": comparison_rule,
            "endpoint_mining_hop": endpoint_hop,
            "sample_index": sample_index,
            "edge_uid": edge_uid,
            "u": u,
            "v": v,

            # Common internal grouping convention.
            "is_clean_graph_control": False,
            "label_value": label_value,
            "label_group": label_group,
            "label_name": group_names[label_group],
            "group_name": group_names[label_group],

            # Mining-score diagnostics. These are not included among the
            # graph-characteristic metrics, because subset groups are defined
            # using score_raw.
            "score_raw": score_raw,
            "score_norm": score_norm,
            "score_percentile": score_percentile,
            "inclusion_count": inclusion_count,
            "n_selected_per_group": (
                selection["n_selected_per_group"]
            ),
            "bottom_cutoff_score": (
                selection["bottom_cutoff_score"]
            ),
            "top_cutoff_score": (
                selection["top_cutoff_score"]
            ),
            "constant_score_run": bool(
                selection["constant_score_run"]
            ),

            "exists_clean": exists_clean,
            "action": (
                "delete"
                if exists_clean
                else "add"
            ),
            "u_true_class": u_label,
            "v_true_class": v_label,
            "true_class_pair": (
                f"{min(u_label, v_label)}-"
                f"{max(u_label, v_label)}"
            ),
            "same_true_class": float(
                same_true_class
            ),
            "feature_cosine_similarity": (
                feature_cosine_similarity
            ),
            "feature_l2_distance": (
                feature_l2_distance
            ),
            "feature_support_jaccard": (
                feature_support_jaccard
            ),
            "common_neighbors": (
                common_neighbor_count
            ),
            "local_neighbor_jaccard": float(
                local_neighbor_jaccard
            ),
            "local_overlap_coefficient": float(
                local_overlap_coefficient
            ),
            "u_partner_class_neighbor_fraction": (
                u_partner_class_fraction
            ),
            "v_partner_class_neighbor_fraction": (
                v_partner_class_fraction
            ),
        }

        endpoint_values = {
            "confidence": (
                float(
                    prediction["confidence"][u]
                ),
                float(
                    prediction["confidence"][v]
                ),
            ),
            "prediction_margin": (
                float(
                    prediction[
                        "prediction_margin"
                    ][u]
                ),
                float(
                    prediction[
                        "prediction_margin"
                    ][v]
                ),
            ),
            "true_class_probability": (
                float(
                    prediction[
                        "true_class_probability"
                    ][u]
                ),
                float(
                    prediction[
                        "true_class_probability"
                    ][v]
                ),
            ),
            "true_class_margin": (
                float(
                    prediction[
                        "true_class_margin"
                    ][u]
                ),
                float(
                    prediction[
                        "true_class_margin"
                    ][v]
                ),
            ),
            "degree": (
                u_degree,
                v_degree,
            ),
            "same_class_neighbor_fraction": (
                float(
                    clean[
                        "same_class_neighbor_fraction"
                    ][u]
                ),
                float(
                    clean[
                        "same_class_neighbor_fraction"
                    ][v]
                ),
            ),
            "neighborhood_class_entropy": (
                float(
                    clean[
                        "neighborhood_class_entropy"
                    ][u]
                ),
                float(
                    clean[
                        "neighborhood_class_entropy"
                    ][v]
                ),
            ),
            "neighborhood_majority_fraction": (
                float(
                    clean[
                        "neighborhood_majority_fraction"
                    ][u]
                ),
                float(
                    clean[
                        "neighborhood_majority_fraction"
                    ][v]
                ),
            ),
            "partner_class_neighbor_fraction": (
                u_partner_class_fraction,
                v_partner_class_fraction,
            ),
        }

        for (
            metric_name,
            (u_value, v_value),
        ) in endpoint_values.items():
            edge_row[f"u_{metric_name}"] = (
                u_value
            )
            edge_row[f"v_{metric_name}"] = (
                v_value
            )
            _rq2_char_pair_summary(
                u_value,
                v_value,
                metric_name,
                edge_row,
            )

        edge_rows.append(edge_row)

        for (
            endpoint_role,
            node,
            partner,
            node_label,
            partner_label,
            value_index,
        ) in [
            (
                "u",
                u,
                v,
                u_label,
                v_label,
                0,
            ),
            (
                "v",
                v,
                u,
                v_label,
                u_label,
                1,
            ),
        ]:
            endpoint_rows.append({
                "dataset": edge_row["dataset"],
                "seed": seed,
                "candidate_config_id": (
                    candidate_config_id
                ),
                "candidate_set_size": int(
                    run["candidate_set_size"]
                ),
                "prbcd_candidate_fraction": float(
                    run[
                        "prbcd_candidate_fraction"
                    ]
                ),
                "actual_prbcd_fraction": float(
                    run[
                        "actual_prbcd_fraction"
                    ]
                ),
                "scoring_mode": scoring_mode,
                "comparison_rule": (
                    comparison_rule
                ),
                "endpoint_mining_hop": (
                    endpoint_hop
                ),
                "sample_index": sample_index,
                "edge_uid": edge_uid,
                "is_clean_graph_control": False,
                "label_value": label_value,
                "label_group": label_group,
                "label_name": (
                    edge_row["label_name"]
                ),
                "group_name": (
                    edge_row["group_name"]
                ),
                "score_raw": score_raw,
                "score_norm": score_norm,
                "score_percentile": (
                    score_percentile
                ),
                "inclusion_count": (
                    inclusion_count
                ),
                "endpoint_role": endpoint_role,
                "node": int(node),
                "partner_node": int(partner),
                "node_true_class": int(
                    node_label
                ),
                "partner_true_class": int(
                    partner_label
                ),
                "same_true_class": float(
                    same_true_class
                ),
                "clean_prediction": int(
                    prediction["prediction"][node]
                ),
                "clean_correct": float(
                    prediction[
                        "clean_correct"
                    ][node]
                ),
                "confidence": endpoint_values[
                    "confidence"
                ][value_index],
                "prediction_margin": (
                    endpoint_values[
                        "prediction_margin"
                    ][value_index]
                ),
                "true_class_probability": (
                    endpoint_values[
                        "true_class_probability"
                    ][value_index]
                ),
                "true_class_margin": (
                    endpoint_values[
                        "true_class_margin"
                    ][value_index]
                ),
                "degree": endpoint_values[
                    "degree"
                ][value_index],
                "same_class_neighbor_fraction": (
                    endpoint_values[
                        "same_class_neighbor_fraction"
                    ][value_index]
                ),
                "neighborhood_class_entropy": (
                    endpoint_values[
                        "neighborhood_class_entropy"
                    ][value_index]
                ),
                "neighborhood_majority_fraction": (
                    endpoint_values[
                        "neighborhood_majority_fraction"
                    ][value_index]
                ),
                "partner_class_neighbor_fraction": (
                    endpoint_values[
                        "partner_class_neighbor_fraction"
                    ][value_index]
                ),
            })

    # Add a matched control based on every undirected edge in the clean
    # graph and both of its endpoint occurrences.
    (
        clean_control_edge_rows,
        clean_control_endpoint_rows,
    ) = _rq2_char_build_clean_graph_control_rows(
        run=run,
        clean=clean,
        prediction=prediction,
        seed=seed,
        candidate_config_id=candidate_config_id,
        scoring_mode=scoring_mode,
        comparison_rule=comparison_rule,
        endpoint_hop=endpoint_hop,
    )

    edge_rows.extend(
        clean_control_edge_rows
    )
    endpoint_rows.extend(
        clean_control_endpoint_rows
    )

    print(
        f"Characterized seed={seed} | "
        f"config={candidate_config_id} | "
        f"mode={scoring_mode} | "
        f"rule={comparison_rule} | "
        f"h={endpoint_hop} | "
        f"selected={selected_indices.size:,}/"
        f"{n_candidates:,} | "
        f"clean_control_edges="
        f"{len(clean_control_edge_rows):,}"
    )


rq2_edge_characteristics_df = pd.DataFrame(
    edge_rows
)
rq2_endpoint_characteristics_df = pd.DataFrame(
    endpoint_rows
)
rq2_characterization_selection_df = (
    pd.DataFrame(selection_diagnostic_rows)
)

if (
    rq2_edge_characteristics_df.empty
    or rq2_endpoint_characteristics_df.empty
):
    raise RuntimeError(
        "No characterization rows were created."
    )

rq2_edge_characteristics_df = (
    rq2_edge_characteristics_df.replace(
        [np.inf, -np.inf],
        np.nan,
    )
)
rq2_endpoint_characteristics_df = (
    rq2_endpoint_characteristics_df.replace(
        [np.inf, -np.inf],
        np.nan,
    )
)


# ------------------------------------------------------------
# Per-run and seed-aggregate summary tables
# ------------------------------------------------------------

RQ2_CHAR_ENDPOINT_METRICS = [
    "confidence",
    "prediction_margin",
    "true_class_probability",
    "true_class_margin",
    "degree",
    "same_class_neighbor_fraction",
    "neighborhood_class_entropy",
    "neighborhood_majority_fraction",
    "partner_class_neighbor_fraction",
]

RQ2_CHAR_EDGE_METRICS = [
    "same_true_class",
    "feature_cosine_similarity",
    "feature_l2_distance",
    "feature_support_jaccard",
    "common_neighbors",
    "local_neighbor_jaccard",
    "local_overlap_coefficient",
    "pair_confidence_mean",
    "pair_confidence_min",
    "pair_true_class_margin_mean",
    "pair_true_class_margin_min",
    "pair_degree_mean",
    "pair_degree_min",
    "pair_same_class_neighbor_fraction_mean",
    "pair_same_class_neighbor_fraction_min",
    "pair_partner_class_neighbor_fraction_mean",
]

rq2_endpoint_run_summary_df = (
    _rq2_char_describe_by_run(
        rq2_endpoint_characteristics_df,
        RQ2_CHAR_ENDPOINT_METRICS,
        "endpoint",
    )
)
rq2_edge_run_summary_df = (
    _rq2_char_describe_by_run(
        rq2_edge_characteristics_df,
        RQ2_CHAR_EDGE_METRICS,
        "edge",
    )
)
rq2_characteristics_run_summary_df = pd.concat(
    [
        rq2_endpoint_run_summary_df,
        rq2_edge_run_summary_df,
    ],
    ignore_index=True,
)

rq2_characteristics_aggregate_df = (
    _rq2_char_aggregate_over_seeds(
        rq2_characteristics_run_summary_df
    )
)
rq2_characteristics_paired_effects_df = (
    _rq2_char_paired_seed_effects(
        rq2_characteristics_run_summary_df
    )
)

# Detailed true-class-pair composition for each individual run.
_true_class_run_group_cols = [
    "seed",
    "candidate_config_id",
    "scoring_mode",
    "comparison_rule",
    "endpoint_mining_hop",
    "label_group",
]

_class_pair_counts = (
    rq2_edge_characteristics_df
    .groupby(
        _true_class_run_group_cols
        + ["true_class_pair"],
        dropna=False,
    )
    .size()
    .rename("n_edges")
    .reset_index()
)

_class_pair_totals = (
    _class_pair_counts
    .groupby(
        _true_class_run_group_cols,
        dropna=False,
    )["n_edges"]
    .sum()
    .rename("n_group_edges")
    .reset_index()
)

rq2_true_class_pair_run_df = (
    _class_pair_counts.merge(
        _class_pair_totals,
        on=_true_class_run_group_cols,
        how="left",
    )
)

rq2_true_class_pair_run_df[
    "fraction_within_group"
] = (
    rq2_true_class_pair_run_df["n_edges"]
    / rq2_true_class_pair_run_df[
        "n_group_edges"
    ].clip(lower=1)
)

# Backward-compatible alias.
rq2_true_class_pair_run_df[
    "fraction_within_label_group"
] = rq2_true_class_pair_run_df[
    "fraction_within_group"
]

rq2_true_class_pair_aggregate_df = (
    rq2_true_class_pair_run_df
    .groupby(
        [
            "candidate_config_id",
            "scoring_mode",
            "comparison_rule",
            "endpoint_mining_hop",
            "label_group",
            "true_class_pair",
        ],
        dropna=False,
    )
    .agg(
        n_seeds=("seed", "nunique"),
        fraction_mean=(
            "fraction_within_group",
            "mean",
        ),
        fraction_std=(
            "fraction_within_group",
            "std",
        ),
    )
    .reset_index()
)

rq2_true_class_pair_aggregate_df[
    "fraction_std"
] = (
    rq2_true_class_pair_aggregate_df[
        "fraction_std"
    ].fillna(0.0)
)
rq2_true_class_pair_aggregate_df[
    "fraction_ci95"
] = (
    1.96
    * rq2_true_class_pair_aggregate_df[
        "fraction_std"
    ]
    / np.sqrt(
        rq2_true_class_pair_aggregate_df[
            "n_seeds"
        ].clip(lower=1)
    )
)


# ------------------------------------------------------------
# Save all tables
# ------------------------------------------------------------

rq2_edge_characteristics_df.to_csv(
    RQ2_CHAR_TABLE_DIR
    / "edge_characteristics_raw.csv",
    index=False,
)
rq2_endpoint_characteristics_df.to_csv(
    RQ2_CHAR_TABLE_DIR
    / "endpoint_characteristics_raw.csv",
    index=False,
)
rq2_characterization_selection_df.to_csv(
    RQ2_CHAR_TABLE_DIR
    / "group_selection_diagnostics_per_run.csv",
    index=False,
)
rq2_characteristics_run_summary_df.to_csv(
    RQ2_CHAR_TABLE_DIR
    / "characteristics_summary_per_run.csv",
    index=False,
)
rq2_characteristics_aggregate_df.to_csv(
    RQ2_CHAR_TABLE_DIR
    / "characteristics_summary_aggregated_over_seeds.csv",
    index=False,
)
rq2_characteristics_paired_effects_df.to_csv(
    RQ2_CHAR_TABLE_DIR
    / "paired_group1_minus_group0_effects_over_seeds.csv",
    index=False,
)
rq2_true_class_pair_run_df.to_csv(
    RQ2_CHAR_TABLE_DIR
    / "true_class_pair_composition_per_run.csv",
    index=False,
)
rq2_true_class_pair_aggregate_df.to_csv(
    RQ2_CHAR_TABLE_DIR
    / "true_class_pair_composition_aggregated_over_seeds.csv",
    index=False,
)


# ------------------------------------------------------------
# Plotting function
# ------------------------------------------------------------

RQ2_CHAR_PLOT_SPECS = [
    {
        "level": "endpoint",
        "feature": "confidence",
        "title": "Endpoint maximum class confidence",
        "ylabel": "Maximum predicted probability",
    },
    {
        "level": "endpoint",
        "feature": "true_class_margin",
        "title": "Endpoint true-class classification margin",
        "ylabel": "p(true class) − max p(other class)",
    },
    {
        "level": "endpoint",
        "feature": "degree",
        "title": "Endpoint degree",
        "ylabel": "Clean undirected degree",
    },
    {
        "level": "edge",
        "feature": "same_true_class",
        "title": "Same true class",
        "ylabel": "Proportion of edges",
        "binary": True,
    },
    {
        "level": "edge",
        "feature": "feature_cosine_similarity",
        "title": "Endpoint feature similarity",
        "ylabel": "Feature cosine similarity",
    },
    {
        "level": "edge",
        "feature": "common_neighbors",
        "title": "Common neighbors",
        "ylabel": "Number of common neighbors",
    },
    {
        "level": "edge",
        "feature": "local_neighbor_jaccard",
        "title": "Local neighborhood overlap",
        "ylabel": "Jaccard overlap",
    },
    {
        "level": "endpoint",
        "feature": "same_class_neighbor_fraction",
        "title": "Endpoint neighborhood homophily",
        "ylabel": "Same-class neighbor fraction",
    },
    {
        "level": "endpoint",
        "feature": "partner_class_neighbor_fraction",
        "title": "Partner-class presence in endpoint neighborhood",
        "ylabel": "Neighbors with partner's true class",
    },
]


def plot_rq2_label_characteristics(
    edge_frame=rq2_edge_characteristics_df,
    endpoint_frame=rq2_endpoint_characteristics_df,
    run_summary=rq2_characteristics_run_summary_df,
    *,
    make_per_run=RQ2_CHAR_MAKE_PER_RUN_PLOTS,
    make_aggregate=RQ2_CHAR_MAKE_AGGREGATE_PLOTS,
    show=RQ2_CHAR_SHOW_PLOTS,
):
    """
    Compare the two mode-specific candidate groups.

    Endpoint:
        group 0 = endpoint label 0
        group 1 = endpoint label 1

    subset_accuracy_drop:
        group 0 = bottom score decile
        group 1 = top score decile

    Per-run figures use raw edge/endpoint distributions for one victim seed.
    The third group is a matched clean-graph control consisting of every
    undirected clean edge and both endpoint occurrences. Aggregate figures
    use one mean per seed and group before aggregating, so victim seeds—not
    individual candidate edges—are the replicates.
    """
    plot_manifest = []

    run_group_columns = [
        "seed",
        "candidate_config_id",
        "scoring_mode",
        "comparison_rule",
        "endpoint_mining_hop",
    ]

    if make_per_run:
        run_keys = (
            edge_frame[run_group_columns]
            .drop_duplicates()
            .sort_values(run_group_columns)
            .itertuples(
                index=False,
                name=None,
            )
        )

        for (
            seed,
            candidate_config_id,
            scoring_mode,
            comparison_rule,
            endpoint_hop,
        ) in run_keys:
            edge_part = edge_frame[
                (edge_frame["seed"] == seed)
                & (
                    edge_frame[
                        "candidate_config_id"
                    ]
                    == candidate_config_id
                )
                & (
                    edge_frame["scoring_mode"]
                    == scoring_mode
                )
                & (
                    edge_frame["comparison_rule"]
                    == comparison_rule
                )
                & (
                    edge_frame[
                        "endpoint_mining_hop"
                    ]
                    == endpoint_hop
                )
            ]

            endpoint_part = endpoint_frame[
                (endpoint_frame["seed"] == seed)
                & (
                    endpoint_frame[
                        "candidate_config_id"
                    ]
                    == candidate_config_id
                )
                & (
                    endpoint_frame["scoring_mode"]
                    == scoring_mode
                )
                & (
                    endpoint_frame[
                        "comparison_rule"
                    ]
                    == comparison_rule
                )
                & (
                    endpoint_frame[
                        "endpoint_mining_hop"
                    ]
                    == endpoint_hop
                )
            ]

            group_names = _rq2_char_group_names(
                scoring_mode
            )
            plot_group_names = (
                _rq2_char_plot_group_names(
                    scoring_mode
                )
            )

            fig, axes = plt.subplots(
                3,
                3,
                figsize=(18, 14),
            )

            for ax, spec in zip(
                axes.flat,
                RQ2_CHAR_PLOT_SPECS,
            ):
                source = (
                    endpoint_part
                    if spec["level"] == "endpoint"
                    else edge_part
                )
                feature = spec["feature"]

                plot_group_order = (
                    _rq2_char_plot_group_order()
                )
                plot_group_values = {
                    group_id: (
                        pd.to_numeric(
                            source.loc[
                                source["label_group"]
                                == group_id,
                                feature,
                            ],
                            errors="coerce",
                        )
                        .dropna()
                        .to_numpy(float)
                    )
                    for group_id
                    in plot_group_order
                }

                all_groups_available = all(
                    plot_group_values[
                        group_id
                    ].size > 0
                    for group_id
                    in plot_group_order
                )

                if (
                    spec.get("binary", False)
                    and all_groups_available
                ):
                    means = [
                        float(
                            plot_group_values[
                                group_id
                            ].mean()
                        )
                        for group_id
                        in plot_group_order
                    ]
                    ax.bar(
                        range(
                            len(plot_group_order)
                        ),
                        means,
                    )
                    ax.set_ylim(0, 1)

                elif all_groups_available:
                    ax.boxplot(
                        [
                            plot_group_values[
                                group_id
                            ]
                            for group_id
                            in plot_group_order
                        ],
                        # Matplotlib boxplots otherwise default to positions
                        # 1, 2, 3, while the shared axis formatter uses
                        # 0, 1, 2. Explicit positions keep boxes and labels
                        # perfectly aligned:
                        # clean graph -> label 0 -> label 1.
                        positions=list(
                            range(
                                len(plot_group_order)
                            )
                        ),
                        widths=0.55,
                        showfliers=False,
                        showmeans=True,
                    )

                else:
                    missing_names = [
                        group_names[
                            group_id
                        ]
                        for group_id
                        in plot_group_order
                        if (
                            plot_group_values[
                                group_id
                            ].size == 0
                        )
                    ]
                    ax.text(
                        0.5,
                        0.5,
                        "Missing group(s):\n"
                        + "\n".join(
                            missing_names
                        ),
                        ha="center",
                        va="center",
                        transform=ax.transAxes,
                    )

                if all_groups_available:
                    _rq2_char_format_group_axis(
                        ax,
                        plot_group_order,
                        plot_group_names,
                    )

                ax.set_title(spec["title"])
                ax.set_ylabel(spec["ylabel"])
                ax.grid(
                    True,
                    axis="y",
                    alpha=0.3,
                )

            fig.suptitle(
                "RQ2 mined-edge characterization",
                fontsize=14,
            )
            fig.tight_layout(
                rect=[0, 0.045, 1, 0.96],
                h_pad=2.4,
                w_pad=1.8,
            )

            filename = (
                f"seed-{seed}"
                f"__cfg-{_rq2_char_safe(candidate_config_id)}"
                f"__mode-{_rq2_char_safe(scoring_mode)}"
                f"__rule-{_rq2_char_safe(comparison_rule)}"
                f"__h-{endpoint_hop}.png"
            )
            path = (
                RQ2_CHAR_PER_RUN_PLOT_DIR
                / filename
            )
            fig.savefig(
                path,
                dpi=RQ2_CHAR_SAVE_DPI,
                bbox_inches="tight",
            )

            plot_manifest.append({
                "plot_type": (
                    "per_run_raw_distributions"
                ),
                "seed": seed,
                "candidate_config_id": (
                    candidate_config_id
                ),
                "scoring_mode": scoring_mode,
                "comparison_rule": comparison_rule,
                "endpoint_mining_hop": (
                    endpoint_hop
                ),
                "path": str(path.resolve()),
            })

            if show:
                plt.show()
            else:
                plt.close(fig)

    if make_aggregate:
        aggregate_group_columns = [
            "candidate_config_id",
            "scoring_mode",
            "comparison_rule",
            "endpoint_mining_hop",
        ]

        aggregate_keys = (
            edge_frame[
                aggregate_group_columns
            ]
            .drop_duplicates()
            .sort_values(
                aggregate_group_columns
            )
            .itertuples(
                index=False,
                name=None,
            )
        )

        for (
            candidate_config_id,
            scoring_mode,
            comparison_rule,
            endpoint_hop,
        ) in aggregate_keys:
            group_names = _rq2_char_group_names(
                scoring_mode
            )
            plot_group_names = (
                _rq2_char_plot_group_names(
                    scoring_mode
                )
            )

            fig, axes = plt.subplots(
                3,
                3,
                figsize=(18, 14),
            )

            for ax, spec in zip(
                axes.flat,
                RQ2_CHAR_PLOT_SPECS,
            ):
                summary_part = run_summary[
                    (
                        run_summary[
                            "analysis_level"
                        ]
                        == spec["level"]
                    )
                    & (
                        run_summary["feature"]
                        == spec["feature"]
                    )
                    & (
                        run_summary[
                            "candidate_config_id"
                        ]
                        == candidate_config_id
                    )
                    & (
                        run_summary[
                            "scoring_mode"
                        ]
                        == scoring_mode
                    )
                    & (
                        run_summary[
                            "comparison_rule"
                        ]
                        == comparison_rule
                    )
                    & (
                        run_summary[
                            "endpoint_mining_hop"
                        ]
                        == endpoint_hop
                    )
                ]

                pivot = (
                    summary_part.pivot_table(
                        index="seed",
                        columns="label_group",
                        values="mean",
                        aggfunc="first",
                    )
                )

                plot_group_order = (
                    _rq2_char_plot_group_order()
                )
                missing_groups = [
                    group_id
                    for group_id
                    in plot_group_order
                    if group_id not in pivot.columns
                ]

                if missing_groups:
                    ax.text(
                        0.5,
                        0.5,
                        "Missing group(s):\n"
                        + "\n".join(
                            group_names[
                                group_id
                            ]
                            for group_id
                            in missing_groups
                        ),
                        ha="center",
                        va="center",
                        transform=ax.transAxes,
                    )
                    ax.set_title(spec["title"])
                    continue

                paired = pivot[
                    plot_group_order
                ].dropna()

                if paired.empty:
                    ax.text(
                        0.5,
                        0.5,
                        "No seeds contain all three groups",
                        ha="center",
                        va="center",
                        transform=ax.transAxes,
                    )
                    ax.set_title(spec["title"])
                    continue

                x_positions = np.arange(
                    len(plot_group_order)
                )

                for _, seed_values in (
                    paired.iterrows()
                ):
                    ax.plot(
                        x_positions,
                        [
                            seed_values[
                                group_id
                            ]
                            for group_id
                            in plot_group_order
                        ],
                        marker="o",
                        linewidth=0.8,
                        alpha=0.30,
                    )

                means = (
                    paired.mean(axis=0)
                    .to_numpy(float)
                )

                if len(paired) > 1:
                    standard_deviation = (
                        paired.std(
                            axis=0,
                            ddof=1,
                        )
                        .to_numpy(float)
                    )
                    ci95 = (
                        1.96
                        * standard_deviation
                        / math.sqrt(len(paired))
                    )
                else:
                    ci95 = np.zeros(
                        len(plot_group_order),
                        dtype=float,
                    )

                ax.errorbar(
                    x_positions,
                    means,
                    yerr=ci95,
                    marker="o",
                    linewidth=2.4,
                    capsize=5,
                    label=(
                        "mean ± 95% CI across seeds"
                    ),
                )

                mean_difference = float(
                    (
                        paired[1]
                        - paired[0]
                    ).mean()
                )
                mean_group_1_vs_control = float(
                    (
                        paired[1]
                        - paired[
                            RQ2_CHAR_CONTROL_GROUP
                        ]
                    ).mean()
                )

                ax.set_title(
                    spec["title"]
                )
                _rq2_char_format_group_axis(
                    ax,
                    plot_group_order,
                    plot_group_names,
                )
                ax.set_ylabel(spec["ylabel"])

                if spec.get("binary", False):
                    ax.set_ylim(0, 1)

                ax.grid(
                    True,
                    axis="y",
                    alpha=0.3,
                )
                ax.legend(fontsize=8)

            fig.suptitle(
                "RQ2 mined-edge characterization across victim seeds",
                fontsize=14,
            )
            fig.tight_layout(
                rect=[0, 0.045, 1, 0.96],
                h_pad=2.4,
                w_pad=1.8,
            )

            filename = (
                "aggregate"
                f"__cfg-{_rq2_char_safe(candidate_config_id)}"
                f"__mode-{_rq2_char_safe(scoring_mode)}"
                f"__rule-{_rq2_char_safe(comparison_rule)}"
                f"__h-{endpoint_hop}.png"
            )
            path = (
                RQ2_CHAR_AGG_PLOT_DIR
                / filename
            )
            fig.savefig(
                path,
                dpi=RQ2_CHAR_SAVE_DPI,
                bbox_inches="tight",
            )

            plot_manifest.append({
                "plot_type": (
                    "aggregate_seed_level_means"
                ),
                "seed": None,
                "candidate_config_id": (
                    candidate_config_id
                ),
                "scoring_mode": scoring_mode,
                "comparison_rule": comparison_rule,
                "endpoint_mining_hop": (
                    endpoint_hop
                ),
                "path": str(path.resolve()),
            })

            if show:
                plt.show()
            else:
                plt.close(fig)

    plot_manifest_df = pd.DataFrame(
        plot_manifest
    )
    plot_manifest_df.to_csv(
        RQ2_CHAR_TABLE_DIR
        / "plot_manifest.csv",
        index=False,
    )
    return plot_manifest_df


rq2_characterization_plot_manifest_df = (
    plot_rq2_label_characteristics()
)


# ------------------------------------------------------------
# Thesis-ready displays
# ------------------------------------------------------------

_sort_run = [
    "candidate_config_id",
    "scoring_mode",
    "comparison_rule",
    "endpoint_mining_hop",
    "seed",
    "analysis_level",
    "feature",
    "label_group",
]

_sort_aggregate = [
    "candidate_config_id",
    "scoring_mode",
    "comparison_rule",
    "endpoint_mining_hop",
    "analysis_level",
    "feature",
    "label_group",
]

_sort_effects = [
    "candidate_config_id",
    "scoring_mode",
    "comparison_rule",
    "endpoint_mining_hop",
    "analysis_level",
    "feature",
]

print("\nGroup-selection diagnostics")
display(
    rq2_characterization_selection_df.sort_values(
        [
            "candidate_config_id",
            "scoring_mode",
            "comparison_rule",
            "endpoint_mining_hop",
            "seed",
        ]
    )
)

print("\nPer-run descriptive statistics")
display(
    rq2_characteristics_run_summary_df.sort_values(
        _sort_run
    )
)

print("\nAggregated seed-level group summaries")
display(
    rq2_characteristics_aggregate_df.sort_values(
        _sort_aggregate
    )
)

print(
    "\nPaired seed-level effects: "
    "group 1 minus group 0"
)
display(
    rq2_characteristics_paired_effects_df.sort_values(
        _sort_effects
    )
)

print(
    "\nDetailed true-class-pair composition "
    "aggregated over seeds"
)
display(
    rq2_true_class_pair_aggregate_df.sort_values(
        [
            "candidate_config_id",
            "scoring_mode",
            "comparison_rule",
            "endpoint_mining_hop",
            "label_group",
            "fraction_mean",
        ],
        ascending=[
            True,
            True,
            True,
            True,
            True,
            False,
        ],
    )
)

experiment_config = {
    "scoring_modes": sorted(
        allowed_scoring_modes
    ),
    "endpoint_label_threshold": float(
        RQ2_CHAR_ENDPOINT_LABEL_THRESHOLD
    ),
    "subset_extreme_fraction": float(
        RQ2_CHAR_SUBSET_EXTREME_FRACTION
    ),
    "endpoint_rule": (
        "label_1_vs_label_0"
    ),
    "subset_rule": (
        "top_score_extreme_vs_bottom_score_extreme"
    ),
    "subset_ranking_score": "score_raw",
    "subset_tie_breaker": (
        "original_sample_index"
    ),
    "clean_graph_control_enabled": bool(
        RQ2_CHAR_INCLUDE_CLEAN_GRAPH_CONTROL
    ),
    "clean_graph_control_group": int(
        RQ2_CHAR_CONTROL_GROUP
    ),
    "clean_graph_control_name": str(
        RQ2_CHAR_CONTROL_NAME
    ),
    "clean_graph_control_edge_unit": (
        "all_unique_undirected_clean_edges"
    ),
    "clean_graph_control_endpoint_unit": (
        "both_endpoint_occurrences_of_each_clean_edge"
    ),
    "candidate_config_ids": (
        None
        if RQ2_CHAR_CANDIDATE_CONFIG_IDS is None
        else [
            str(value)
            for value
            in RQ2_CHAR_CANDIDATE_CONFIG_IDS
        ]
    ),
    "endpoint_hops": (
        None
        if RQ2_CHAR_ENDPOINT_HOPS is None
        else [
            int(value)
            for value
            in RQ2_CHAR_ENDPOINT_HOPS
        ]
    ),
    "n_selected_edge_rows": int(
        len(rq2_edge_characteristics_df)
    ),
    "n_selected_endpoint_rows": int(
        len(rq2_endpoint_characteristics_df)
    ),
    "victim_seeds": sorted({
        int(value)
        for value
        in rq2_edge_characteristics_df[
            "seed"
        ].unique()
    }),
}

(
    RQ2_CHAR_TABLE_DIR
    / "experiment_config.json"
).write_text(
    json.dumps(
        experiment_config,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "\nInterpretation:"
)
print(
    "- endpoint: group 1 is the original endpoint-harmful label and "
    "group 0 is the original no-hit label."
)
print(
    "- subset_accuracy_drop: group 1 contains the highest-scoring "
    f"{100 * RQ2_CHAR_SUBSET_EXTREME_FRACTION:g}% and group 0 contains "
    f"the lowest-scoring {100 * RQ2_CHAR_SUBSET_EXTREME_FRACTION:g}% "
    "within each seed/run."
)
print(
    "- The clean-graph control contains every unique undirected edge in "
    "the clean graph. Endpoint-level control values contain both endpoint "
    "occurrences of every clean edge, matching the mined-edge observation "
    "unit and keeping partner-dependent metrics defined."
)
print(
    "- score_raw, score_norm, score_percentile, and inclusion_count are "
    "saved as diagnostics but are not among the characteristics compared, "
    "because subset group membership is defined from score_raw."
)

print(
    "\nSaved RQ2 edge-characterization outputs to:"
)
print(RQ2_CHAR_OUT_DIR.resolve())



---

## Experiment 4 — Balanced Label-Conditioned PR-BCD Initialization

### Research objective

RQ1 showed that PR-BCD can only optimize candidates contained in its restricted search block. This experiment tests whether endpoint label-1 candidates define a more useful PR-BCD candidate space than label-0 or random candidates.

Three initial block types are compared:

1. endpoint label-1 candidates;
2. endpoint label-0 candidates;
3. random candidates from the full edge-perturbation space.

The label-conditioned blocks are balanced to a common size:

\[
b
=
\min
\left(
n_{\mathrm{label\,1}},
n_{\mathrm{label\,0}}
\right).
\]

The larger label group is sampled without replacement to match the smaller group. The random control receives the same block size.

This balancing ensures that attack differences cannot be attributed to different numbers of available candidate coordinates.


In [ ]:
from datetime import datetime
from pathlib import Path
from timeit import default_timer as timer
import gc
import inspect
import json
import os
import re

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

from experiments import experiment_global_attack_direct
from rgnn_at_scale.attacks.prbcd import PRBCD
from sparse_smoothing.utils import load_and_standardize


# ============================================================
# Configuration
# ============================================================

RQ2_BLOCK_SCHEMA_VERSION = "mode_aware_extreme_blocks_v1"

RQ2_BLOCK_EPSILON = 0.03
RQ2_BLOCK_EPOCHS = 100
RQ2_BLOCK_RESAMPLING_EPOCHS = 50
RQ2_BLOCK_REPEATS = 3          # use >= 5 for the final thesis experiment
RQ2_BLOCK_WITH_EARLY_STOPPING = False

# Endpoint mode keeps the original binary definition.
RQ2_BLOCK_ENDPOINT_LABEL_THRESHOLD = 0.5

# subset_accuracy_drop uses equal-sized top and bottom score groups.
RQ2_BLOCK_SUBSET_EXTREME_FRACTION = 0.10

# The full-space random block excludes every candidate edge from the mined
# candidate set, not only the selected top/bottom extremes.
RQ2_BLOCK_RANDOM_EXCLUDE_MINED_CANDIDATES = True

RQ2_BLOCK_SCORING_MODES = {
    "endpoint",
    "subset_accuracy_drop",
}

# Example: {2}. None uses every endpoint hop represented in mining_runs.
# subset_accuracy_drop normally has endpoint_mining_hop == 0.
RQ2_BLOCK_HOPS = None

RQ2_BLOCK_USE_CERT = "none"
RQ2_BLOCK_MODEL_STORAGE_TYPE = "demo_custom_split"
RQ2_BLOCK_ARTIFACT_DIR = "cache"
RQ2_BLOCK_PERT_ADJ_STORAGE_TYPE = "evasion_global_adj"
RQ2_BLOCK_PERT_ATTR_STORAGE_TYPE = "evasion_global_attr"

RQ2_BLOCK_BASE_OUT_DIR = (
    Path("extendedPlotting")
    / "rq2_mode_aware_initial_block_multiseed"
)


# ============================================================
# Validation
# ============================================================

if not 0 <= RQ2_BLOCK_RESAMPLING_EPOCHS <= RQ2_BLOCK_EPOCHS:
    raise ValueError(
        "RQ2_BLOCK_RESAMPLING_EPOCHS must be in "
        "[0, RQ2_BLOCK_EPOCHS]."
    )

if RQ2_BLOCK_REPEATS < 1:
    raise ValueError("RQ2_BLOCK_REPEATS must be at least 1.")

if not 0.0 < RQ2_BLOCK_SUBSET_EXTREME_FRACTION <= 0.5:
    raise ValueError(
        "RQ2_BLOCK_SUBSET_EXTREME_FRACTION must be in (0, 0.5]."
    )

supported_modes = {
    "endpoint",
    "subset_accuracy_drop",
}
unknown_modes = set(RQ2_BLOCK_SCORING_MODES) - supported_modes
if unknown_modes:
    raise ValueError(
        f"Unsupported RQ2 block scoring modes: {sorted(unknown_modes)}"
    )

required_prbcd_args = {
    "initial_block_path",
    "initial_block_label",
    "resampling_enabled",
    "block_diagnostics_enabled",
    "attack_sampling_seed",
}

available_prbcd_args = set(
    inspect.signature(PRBCD.__init__).parameters
)

missing_prbcd_args = (
    required_prbcd_args
    - available_prbcd_args
)

if missing_prbcd_args:
    raise RuntimeError(
        "The loaded PRBCD class is not the RQ2 custom-block version. "
        f"Missing constructor arguments: {sorted(missing_prbcd_args)}. "
        "Replace rgnn_at_scale/attacks/prbcd.py with the patched file, "
        "restart the kernel, and rerun the notebook."
    )

print("RQ2 block schema:", RQ2_BLOCK_SCHEMA_VERSION)
print("Scoring modes:", sorted(RQ2_BLOCK_SCORING_MODES))
print(
    "Subset extreme fraction:",
    RQ2_BLOCK_SUBSET_EXTREME_FRACTION,
)

### Execution block

Each of the three initializations is evaluated under two optimization conditions.

#### Fixed candidate block

The candidate block remains fixed throughout optimization. PR-BCD can update the relaxed weights but cannot introduce new candidate edges.

This condition isolates candidate quality:

> Does a block of locally harmful candidates provide a better restricted optimization space than an equally sized label-0 or random block?

#### Random coordinate resampling

PR-BCD performs its standard WeightOnly retention and random refill procedure.

This condition evaluates whether the initial candidate advantage persists when the attack can introduce new coordinates.

The resulting six conditions are:

1. label 1 with a fixed block;
2. label 1 with random resampling;
3. label 0 with a fixed block;
4. label 0 with random resampling;
5. random initialization with a fixed block;
6. random initialization with random resampling.

Within each victim seed and repeat, the attack configuration and stochastic seeds are paired as closely as possible.

In [ ]:
from datetime import datetime
from pathlib import Path
from timeit import default_timer as timer
import gc
import json
import math
import os
import re

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from experiments import experiment_global_attack_direct
from rgnn_at_scale.attacks.prbcd import PRBCD
from sparse_smoothing.utils import load_and_standardize


# ============================================================
# Helpers
# ============================================================

def _rq2_safe(value):
    return re.sub(
        r"[^a-zA-Z0-9_.=-]+",
        "_",
        str(value),
    ).strip("_")


def _as_list(value):
    if value is None:
        return []

    if torch.is_tensor(value):
        value = value.detach().cpu()
        return (
            [value.item()]
            if value.ndim == 0
            else value.reshape(-1).tolist()
        )

    if isinstance(value, np.ndarray):
        return value.reshape(-1).tolist()

    if isinstance(value, (list, tuple)):
        return list(value)

    return [value]


def _find_attack_statistics(obj, seen=None):
    if seen is None:
        seen = set()

    obj_id = id(obj)
    if obj_id in seen:
        return None

    seen.add(obj_id)

    if isinstance(obj, dict):
        for key in (
            "attack_statistics",
            "attack_stats",
            "stats",
        ):
            candidate = obj.get(key)

            if isinstance(candidate, dict) and (
                isinstance(
                    candidate.get("accuracy"),
                    (
                        list,
                        tuple,
                        np.ndarray,
                        torch.Tensor,
                    ),
                )
                or isinstance(
                    candidate.get("loss"),
                    (
                        list,
                        tuple,
                        np.ndarray,
                        torch.Tensor,
                    ),
                )
            ):
                return candidate

        if isinstance(
            obj.get("accuracy"),
            (
                list,
                tuple,
                np.ndarray,
                torch.Tensor,
            ),
        ):
            return obj

        for value in obj.values():
            found = _find_attack_statistics(
                value,
                seen,
            )

            if found is not None:
                return found

    elif isinstance(obj, (list, tuple)):
        for value in obj:
            found = _find_attack_statistics(
                value,
                seen,
            )

            if found is not None:
                return found

    return None


def _extract_final_accuracy(result):
    if isinstance(result, dict):
        rows = result.get("results", []) or []

        if (
            rows
            and isinstance(rows[0], dict)
            and "accuracy" in rows[0]
        ):
            return float(
                torch.as_tensor(
                    rows[0]["accuracy"]
                )
                .detach()
                .cpu()
                .item()
            )

    raise KeyError(
        "Could not extract final attacked accuracy "
        "from experiment result."
    )


def _pairs_to_linear_ids(
    src,
    dst,
    n_nodes,
):
    src = torch.as_tensor(
        src,
        dtype=torch.long,
    ).flatten()

    dst = torch.as_tensor(
        dst,
        dtype=torch.long,
    ).flatten()

    if src.numel() != dst.numel():
        raise ValueError(
            "src and dst must have equal lengths."
        )

    if src.numel() == 0:
        return torch.empty(
            0,
            dtype=torch.long,
        )

    u = torch.minimum(src, dst)
    v = torch.maximum(src, dst)

    valid = u < v

    pairs = torch.stack(
        [
            u[valid],
            v[valid],
        ],
        dim=0,
    )

    return torch.unique(
        PRBCD.triu_idx_to_linear_idx(
            int(n_nodes),
            pairs,
        )
        .cpu()
        .long(),
        sorted=True,
    )


def _sample_random_linear_block(
    n_possible,
    count,
    *,
    forbidden,
    seed,
):
    """
    Uniform rejection sampler without materializing all possible pairs.
    """
    forbidden = {
        int(value)
        for value in forbidden
    }

    count = int(count)

    if count > int(n_possible) - len(forbidden):
        raise ValueError(
            "Not enough non-forbidden pairs "
            "for the random baseline."
        )

    rng = np.random.default_rng(
        int(seed)
    )

    selected = set()

    while len(selected) < count:
        remaining = count - len(selected)
        batch_size = max(
            4096,
            3 * remaining,
        )

        draws = rng.integers(
            0,
            int(n_possible),
            size=batch_size,
            endpoint=False,
        )

        for value in draws.tolist():
            value = int(value)

            if (
                value not in forbidden
                and value not in selected
            ):
                selected.add(value)

                if len(selected) == count:
                    break

    return torch.tensor(
        sorted(selected),
        dtype=torch.long,
    )


def _save_block(
    path,
    linear_ids,
    metadata,
):
    linear_ids = torch.unique(
        torch.as_tensor(
            linear_ids,
            dtype=torch.long,
        ),
        sorted=True,
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        {
            "linear_ids": linear_ids,
            "metadata": dict(metadata),
        },
        path,
    )

    return path.resolve()


def _epoch_to_drop(
    accuracies,
    clean_accuracy,
    drop,
):
    for stat_index, accuracy_value in enumerate(
        accuracies
    ):
        if (
            clean_accuracy
            - accuracy_value
            >= drop
        ):
            # Statistics index zero is the clean baseline.
            return stat_index - 1

    return np.nan


def _two_stage_summary(
    frame,
    group_cols,
    metric_cols,
):
    """
    Average repeats within victim seed, then aggregate victim seeds.
    """
    if frame.empty:
        return pd.DataFrame()

    available_metrics = [
        metric
        for metric in metric_cols
        if metric in frame.columns
    ]

    if not available_metrics:
        return pd.DataFrame()

    work = frame.copy()

    for metric in available_metrics:
        work[metric] = pd.to_numeric(
            work[metric],
            errors="coerce",
        )

    seed_level = (
        work
        .groupby(
            group_cols + ["seed"],
            dropna=False,
        )[available_metrics]
        .mean()
        .reset_index()
    )

    grouped = seed_level.groupby(
        group_cols,
        dropna=False,
    )

    summary = grouped[
        available_metrics
    ].agg(
        [
            "mean",
            "std",
            "count",
        ]
    )

    summary.columns = [
        f"{metric}_{statistic}"
        for metric, statistic in summary.columns
    ]

    summary = summary.reset_index()

    for metric in available_metrics:
        summary[f"{metric}_std"] = (
            summary[f"{metric}_std"]
            .fillna(0.0)
        )

        summary[f"{metric}_sem"] = (
            summary[f"{metric}_std"]
            / np.sqrt(
                summary[
                    f"{metric}_count"
                ].clip(lower=1)
            )
        )

    n_seeds = (
        seed_level
        .groupby(
            group_cols,
            dropna=False,
        )["seed"]
        .nunique()
        .rename("n_seeds")
        .reset_index()
    )

    summary = summary.merge(
        n_seeds,
        on=group_cols,
        how="left",
        validate="one_to_one",
    )

    return summary


def _sample_ids_without_replacement(
    linear_ids,
    count,
    *,
    seed,
):
    """
    Sample exactly count unique IDs without replacement.
    """
    linear_ids = torch.unique(
        torch.as_tensor(
            linear_ids,
            dtype=torch.long,
        ).flatten(),
        sorted=True,
    )

    count = int(count)

    if count < 0:
        raise ValueError(
            "count must be non-negative."
        )

    if count > int(linear_ids.numel()):
        raise ValueError(
            f"Cannot draw {count} IDs from only "
            f"{int(linear_ids.numel())} unique IDs."
        )

    if count == int(linear_ids.numel()):
        return linear_ids.clone()

    rng = np.random.default_rng(
        int(seed)
    )

    selected_positions = rng.choice(
        int(linear_ids.numel()),
        size=count,
        replace=False,
    )

    selected_positions = torch.as_tensor(
        selected_positions,
        dtype=torch.long,
    )

    return torch.sort(
        linear_ids[selected_positions]
    ).values


def _run_array(
    run,
    key,
    *,
    dtype,
    fallback_key=None,
):
    value = run.get(key)

    if value is None and fallback_key is not None:
        value = run.get(
            "mining_result",
            {},
        ).get(fallback_key)

    if value is None:
        raise KeyError(
            f"Run does not contain required field {key!r}."
        )

    if torch.is_tensor(value):
        array = (
            value
            .detach()
            .cpu()
            .numpy()
        )
    else:
        array = np.asarray(value)

    return np.asarray(
        array,
        dtype=dtype,
    ).reshape(-1)


def _build_mode_condition_groups(
    run,
    *,
    n_nodes,
):
    """
    Construct the two matched mined groups for one mining run.

    endpoint:
        high group = endpoint label 1
        low group  = endpoint label 0

    subset_accuracy_drop:
        high group = top extreme fraction by observed score_raw
        low group  = bottom extreme fraction by observed score_raw
    """
    scoring_mode = str(
        run["scoring_mode"]
    )

    src = torch.as_tensor(
        run["src"],
        dtype=torch.long,
    ).detach().cpu().flatten()

    dst = torch.as_tensor(
        run["dst"],
        dtype=torch.long,
    ).detach().cpu().flatten()

    if src.numel() != dst.numel():
        raise ValueError(
            "Run src and dst have unequal lengths."
        )

    n_rows = int(src.numel())

    if n_rows == 0:
        raise ValueError(
            "Candidate run is empty."
        )

    all_candidate_ids = _pairs_to_linear_ids(
        src,
        dst,
        n_nodes,
    )

    if int(all_candidate_ids.numel()) != n_rows:
        raise ValueError(
            "The block experiment requires one unique "
            "undirected pair per candidate row. "
            f"Rows={n_rows}, unique IDs={int(all_candidate_ids.numel())}."
        )

    if scoring_mode == "endpoint":
        labels = _run_array(
            run,
            "labels_changed",
            dtype=np.float64,
        )

        if labels.size != n_rows:
            raise ValueError(
                "Endpoint labels do not align with candidate rows."
            )

        high_mask = (
            labels
            > float(
                RQ2_BLOCK_ENDPOINT_LABEL_THRESHOLD
            )
        )

        low_mask = ~high_mask

        high_ids = _pairs_to_linear_ids(
            src[torch.as_tensor(high_mask)],
            dst[torch.as_tensor(high_mask)],
            n_nodes,
        )

        low_ids = _pairs_to_linear_ids(
            src[torch.as_tensor(low_mask)],
            dst[torch.as_tensor(low_mask)],
            n_nodes,
        )

        return {
            "comparison_rule": "endpoint_label_1_vs_0",
            "group_high_initialization": "label_positive",
            "group_low_initialization": "label_zero",
            "group_high_target": "endpoint_label_1",
            "group_low_target": "endpoint_label_0",
            "group_high_display": "Endpoint label 1",
            "group_low_display": "Endpoint label 0",
            "group_high_ids_full": high_ids,
            "group_low_ids_full": low_ids,
            "all_candidate_ids": all_candidate_ids,
            "n_candidate_rows": n_rows,
            "n_observed_scores": n_rows,
            "extreme_fraction": np.nan,
            "bottom_cutoff_score": np.nan,
            "top_cutoff_score": np.nan,
            "bottom_score_mean": np.nan,
            "top_score_mean": np.nan,
            "constant_score_run": False,
        }

    if scoring_mode == "subset_accuracy_drop":
        score_raw = _run_array(
            run,
            "score_raw",
            dtype=np.float64,
            fallback_key="score_raw",
        )

        if score_raw.size != n_rows:
            raise ValueError(
                "score_raw does not align with candidate rows. "
                f"scores={score_raw.size}, candidates={n_rows}."
            )

        observed_value = run.get(
            "observed_mask"
        )

        if observed_value is None:
            observed_mask = np.isfinite(
                score_raw
            )
        else:
            if torch.is_tensor(observed_value):
                observed_mask = (
                    observed_value
                    .detach()
                    .cpu()
                    .numpy()
                )
            else:
                observed_mask = np.asarray(
                    observed_value
                )

            observed_mask = np.asarray(
                observed_mask,
                dtype=bool,
            ).reshape(-1)

            if observed_mask.size != n_rows:
                # The notebook's schema-v2 run tensors are already filtered
                # to observed candidates. In that case all remaining rows are
                # observed even when mining_result retains a full-size mask.
                if np.isfinite(score_raw).all():
                    observed_mask = np.ones(
                        n_rows,
                        dtype=bool,
                    )
                else:
                    raise ValueError(
                        "observed_mask does not align with score_raw."
                    )

        valid_indices = np.flatnonzero(
            observed_mask
            & np.isfinite(score_raw)
        )

        n_valid = int(
            valid_indices.size
        )

        if n_valid < 2:
            raise ValueError(
                "At least two observed finite subset scores "
                "are required."
            )

        n_per_group = int(
            math.floor(
                float(
                    RQ2_BLOCK_SUBSET_EXTREME_FRACTION
                )
                * n_valid
            )
        )

        n_per_group = max(
            1,
            min(
                n_per_group,
                n_valid // 2,
            ),
        )

        # sample_index is used only as deterministic tie breaking.
        sample_index = np.arange(
            n_rows,
            dtype=np.int64,
        )

        valid_order = np.lexsort(
            (
                sample_index[valid_indices],
                score_raw[valid_indices],
            )
        )

        ordered_valid_indices = (
            valid_indices[valid_order]
        )

        low_indices = ordered_valid_indices[
            :n_per_group
        ]

        high_indices = ordered_valid_indices[
            -n_per_group:
        ]

        if set(low_indices.tolist()) & set(
            high_indices.tolist()
        ):
            raise RuntimeError(
                "Top and bottom subset groups overlap."
            )

        high_ids = _pairs_to_linear_ids(
            src[torch.as_tensor(high_indices)],
            dst[torch.as_tensor(high_indices)],
            n_nodes,
        )

        low_ids = _pairs_to_linear_ids(
            src[torch.as_tensor(low_indices)],
            dst[torch.as_tensor(low_indices)],
            n_nodes,
        )

        if (
            int(high_ids.numel()) != n_per_group
            or int(low_ids.numel()) != n_per_group
        ):
            raise ValueError(
                "Duplicate candidate pairs appeared in the "
                "top/bottom subset groups."
            )

        return {
            "comparison_rule": (
                f"top_vs_bottom_"
                f"{100 * float(RQ2_BLOCK_SUBSET_EXTREME_FRACTION):g}pct"
            ),
            "group_high_initialization": "score_top",
            "group_low_initialization": "score_bottom",
            "group_high_target": "top_subset_score",
            "group_low_target": "bottom_subset_score",
            "group_high_display": (
                f"Top "
                f"{100 * float(RQ2_BLOCK_SUBSET_EXTREME_FRACTION):g}% "
                "subset score"
            ),
            "group_low_display": (
                f"Bottom "
                f"{100 * float(RQ2_BLOCK_SUBSET_EXTREME_FRACTION):g}% "
                "subset score"
            ),
            "group_high_ids_full": high_ids,
            "group_low_ids_full": low_ids,
            "all_candidate_ids": all_candidate_ids,
            "n_candidate_rows": n_rows,
            "n_observed_scores": n_valid,
            "extreme_fraction": float(
                RQ2_BLOCK_SUBSET_EXTREME_FRACTION
            ),
            "bottom_cutoff_score": float(
                score_raw[low_indices].max()
            ),
            "top_cutoff_score": float(
                score_raw[high_indices].min()
            ),
            "bottom_score_mean": float(
                score_raw[low_indices].mean()
            ),
            "top_score_mean": float(
                score_raw[high_indices].mean()
            ),
            "constant_score_run": bool(
                np.allclose(
                    score_raw[valid_indices],
                    score_raw[valid_indices][0],
                )
            ),
        }

    raise ValueError(
        f"Unsupported scoring mode: {scoring_mode!r}"
    )


# ============================================================
# Select mining runs
# ============================================================

if "mining_runs" not in globals() or not mining_runs:
    raise RuntimeError(
        "`mining_runs` is missing. "
        "Run candidate mining first."
    )

rq2_source_runs = [
    run
    for run in mining_runs
    if str(
        run.get("scoring_mode")
    ) in RQ2_BLOCK_SCORING_MODES
    and (
        RQ2_BLOCK_HOPS is None
        or int(
            run.get(
                "endpoint_mining_hop",
                0,
            )
        ) in RQ2_BLOCK_HOPS
    )
]

if not rq2_source_runs:
    raise RuntimeError(
        "No matching mode-aware mining runs were found."
    )


# ============================================================
# Graph and output setup
# ============================================================

graph_sparse_rq2 = load_and_standardize(
    os.path.join(
        "data",
        f"{DATASET}.npz",
    )
)

N_RQ2_NODES = int(
    graph_sparse_rq2.attr_matrix.shape[0]
)

N_RQ2_UNDIRECTED = int(
    graph_sparse_rq2.adj_matrix.nnz // 2
)

N_RQ2_POSSIBLE = (
    N_RQ2_NODES
    * (N_RQ2_NODES - 1)
    // 2
)

RQ2_BLOCK_ATTACK_BUDGET = max(
    1,
    round(
        RQ2_BLOCK_EPSILON
        * N_RQ2_UNDIRECTED
    ),
)

RQ2_BLOCK_FINE_TUNE_EPOCHS = (
    RQ2_BLOCK_EPOCHS
    - RQ2_BLOCK_RESAMPLING_EPOCHS
)

RQ2_BLOCK_RUN_ID = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

RQ2_BLOCK_OUT_DIR = (
    RQ2_BLOCK_BASE_OUT_DIR
    / f"{DATASET}__{RQ2_BLOCK_RUN_ID}"
)

RQ2_BLOCK_BLOCK_DIR = (
    RQ2_BLOCK_OUT_DIR
    / "initial_blocks"
)

RQ2_BLOCK_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RQ2_BLOCK_BLOCK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

(
    RQ2_BLOCK_BASE_OUT_DIR
    / "latest_run.txt"
).write_text(
    str(
        RQ2_BLOCK_OUT_DIR.resolve()
    ),
    encoding="utf-8",
)

print(
    "RQ2 mode-aware matched initial-block experiment"
)

print(
    "scoring modes:",
    sorted({
        str(run["scoring_mode"])
        for run in rq2_source_runs
    }),
)

print(
    "mining runs:",
    len(rq2_source_runs),
)

print(
    "attack budget:",
    RQ2_BLOCK_ATTACK_BUDGET,
)

print(
    "epochs:",
    RQ2_BLOCK_EPOCHS,
)

print(
    "nominal resampling epochs:",
    RQ2_BLOCK_RESAMPLING_EPOCHS,
)

print(
    "fine-tune epochs:",
    RQ2_BLOCK_FINE_TUNE_EPOCHS,
)

print(
    "output:",
    RQ2_BLOCK_OUT_DIR.resolve(),
)


# ============================================================
# Execute six matched conditions per mining run
#
# Endpoint:
#   label_positive versus label_zero versus random_full_space
#
# subset_accuracy_drop:
#   score_top versus score_bottom versus random_full_space
#
# Every initialization is run:
#   - as a fixed block
#   - with standard PRBCD random resampling
# ============================================================

rq2_block_run_rows = []
rq2_block_epoch_rows = []
rq2_block_retention_rows = []
rq2_block_balance_rows = []


for source_index, run in enumerate(
    rq2_source_runs
):
    victim_seed = int(
        run["seed"]
    )

    scoring_mode = str(
        run["scoring_mode"]
    )

    mode_groups = _build_mode_condition_groups(
        run,
        n_nodes=N_RQ2_NODES,
    )

    high_ids_full = (
        mode_groups[
            "group_high_ids_full"
        ]
    )

    low_ids_full = (
        mode_groups[
            "group_low_ids_full"
        ]
    )

    n_high = int(
        high_ids_full.numel()
    )

    n_low = int(
        low_ids_full.numel()
    )

    if n_high == 0:
        print(
            "Skipping run without a high group:",
            run["candidate_config_id"],
            scoring_mode,
        )
        continue

    if n_low == 0:
        print(
            "Skipping run without a low group:",
            run["candidate_config_id"],
            scoring_mode,
        )
        continue

    high_set_full = set(
        high_ids_full.tolist()
    )

    low_set_full = set(
        low_ids_full.tolist()
    )

    overlap = (
        high_set_full
        & low_set_full
    )

    if overlap:
        raise ValueError(
            f"Run {run['candidate_config_id']} contains "
            f"{len(overlap)} unique IDs in both comparison groups."
        )

    balanced_block_size = min(
        n_high,
        n_low,
    )

    if (
        balanced_block_size
        <= RQ2_BLOCK_ATTACK_BUDGET
    ):
        raise ValueError(
            f"Matched block for {run['candidate_config_id']} "
            f"({scoring_mode}) contains "
            f"{balanced_block_size} edges, but attack budget is "
            f"{RQ2_BLOCK_ATTACK_BUDGET}. "
            "Lower epsilon, enlarge the candidate set, or increase "
            "the subset extreme fraction."
        )

    if n_high < n_low:
        minority_group = "high"
        majority_group = "low"
    elif n_low < n_high:
        minority_group = "low"
        majority_group = "high"
    else:
        minority_group = "tie"
        majority_group = "tie"

    all_candidate_set = set(
        mode_groups[
            "all_candidate_ids"
        ].tolist()
    )

    base_metadata = {
        "block_schema_version": (
            RQ2_BLOCK_SCHEMA_VERSION
        ),
        "dataset": DATASET,
        "victim_seed": victim_seed,
        "candidate_config_id": run[
            "candidate_config_id"
        ],
        "scoring_mode": scoring_mode,
        "comparison_rule": mode_groups[
            "comparison_rule"
        ],
        "endpoint_mining_hop": int(
            run.get(
                "endpoint_mining_hop",
                0,
            )
        ),
        "candidate_set_size": int(
            run["candidate_set_size"]
        ),
        "n_candidate_rows": int(
            mode_groups[
                "n_candidate_rows"
            ]
        ),
        "n_observed_scores": int(
            mode_groups[
                "n_observed_scores"
            ]
        ),
        "n_group_high_unique_edges": n_high,
        "n_group_low_unique_edges": n_low,
        "group_high_initialization": mode_groups[
            "group_high_initialization"
        ],
        "group_low_initialization": mode_groups[
            "group_low_initialization"
        ],
        "group_high_target": mode_groups[
            "group_high_target"
        ],
        "group_low_target": mode_groups[
            "group_low_target"
        ],
        "group_high_display": mode_groups[
            "group_high_display"
        ],
        "group_low_display": mode_groups[
            "group_low_display"
        ],
        "balanced_block_size": (
            balanced_block_size
        ),
        "minority_group": minority_group,
        "majority_group": majority_group,
        "extreme_fraction": mode_groups[
            "extreme_fraction"
        ],
        "bottom_cutoff_score": mode_groups[
            "bottom_cutoff_score"
        ],
        "top_cutoff_score": mode_groups[
            "top_cutoff_score"
        ],
        "bottom_score_mean": mode_groups[
            "bottom_score_mean"
        ],
        "top_score_mean": mode_groups[
            "top_score_mean"
        ],
        "constant_score_run": mode_groups[
            "constant_score_run"
        ],
        "candidate_hash": run.get(
            "candidate_hash",
            "",
        ),
    }

    print(
        "\nMATCHING",
        f"seed={victim_seed} | "
        f"{run['candidate_config_id']} | "
        f"mode={scoring_mode} | "
        f"h={run.get('endpoint_mining_hop', 0)} | "
        f"high={n_high} | low={n_low} | "
        f"common_block_size={balanced_block_size} | "
        f"rule={mode_groups['comparison_rule']}",
    )

    if (
        scoring_mode
        == "subset_accuracy_drop"
        and mode_groups[
            "constant_score_run"
        ]
    ):
        print(
            "[warning] All observed subset scores are equal. "
            "Top/bottom membership is determined entirely by "
            "the deterministic tie breaker."
        )

    for repeat in range(
        RQ2_BLOCK_REPEATS
    ):
        base_subset_seed = (
            300_000
            + victim_seed * 10_000
            + source_index * 100
            + repeat * 10
        )

        high_subset_seed = (
            base_subset_seed
        )

        low_subset_seed = (
            base_subset_seed
            + 1
        )

        random_seed = (
            base_subset_seed
            + 2
        )

        high_ids = (
            _sample_ids_without_replacement(
                high_ids_full,
                balanced_block_size,
                seed=high_subset_seed,
            )
        )

        low_ids = (
            _sample_ids_without_replacement(
                low_ids_full,
                balanced_block_size,
                seed=low_subset_seed,
            )
        )

        high_set = set(
            high_ids.tolist()
        )

        low_set = set(
            low_ids.tolist()
        )

        forbidden = (
            all_candidate_set
            if RQ2_BLOCK_RANDOM_EXCLUDE_MINED_CANDIDATES
            else set()
        )

        random_ids = (
            _sample_random_linear_block(
                N_RQ2_POSSIBLE,
                balanced_block_size,
                forbidden=forbidden,
                seed=random_seed,
            )
        )

        high_initialization = mode_groups[
            "group_high_initialization"
        ]

        low_initialization = mode_groups[
            "group_low_initialization"
        ]

        high_path = _save_block(
            RQ2_BLOCK_BLOCK_DIR
            / (
                f"{high_initialization}__"
                f"seed-{victim_seed}__"
                f"{_rq2_safe(run['candidate_config_id'])}__"
                f"mode-{_rq2_safe(scoring_mode)}__"
                f"h-{run.get('endpoint_mining_hop', 0)}__"
                f"repeat-{repeat}.pt"
            ),
            high_ids,
            {
                **base_metadata,
                "initialization": high_initialization,
                "block_origin": "mined",
                "target_group": "high",
                "target_group_name": mode_groups[
                    "group_high_target"
                ],
                "subset_seed": high_subset_seed,
                "was_subsampled": bool(
                    n_high
                    > balanced_block_size
                ),
                "sampling_fraction": (
                    balanced_block_size
                    / n_high
                ),
            },
        )

        low_path = _save_block(
            RQ2_BLOCK_BLOCK_DIR
            / (
                f"{low_initialization}__"
                f"seed-{victim_seed}__"
                f"{_rq2_safe(run['candidate_config_id'])}__"
                f"mode-{_rq2_safe(scoring_mode)}__"
                f"h-{run.get('endpoint_mining_hop', 0)}__"
                f"repeat-{repeat}.pt"
            ),
            low_ids,
            {
                **base_metadata,
                "initialization": low_initialization,
                "block_origin": "mined",
                "target_group": "low",
                "target_group_name": mode_groups[
                    "group_low_target"
                ],
                "subset_seed": low_subset_seed,
                "was_subsampled": bool(
                    n_low
                    > balanced_block_size
                ),
                "sampling_fraction": (
                    balanced_block_size
                    / n_low
                ),
            },
        )

        random_path = _save_block(
            RQ2_BLOCK_BLOCK_DIR
            / (
                f"random_full_space__"
                f"seed-{victim_seed}__"
                f"{_rq2_safe(run['candidate_config_id'])}__"
                f"mode-{_rq2_safe(scoring_mode)}__"
                f"h-{run.get('endpoint_mining_hop', 0)}__"
                f"repeat-{repeat}.pt"
            ),
            random_ids,
            {
                **base_metadata,
                "initialization": (
                    "random_full_space"
                ),
                "block_origin": "random",
                "target_group": "random",
                "target_group_name": (
                    "random_control"
                ),
                "random_seed": random_seed,
                "excluded_all_mined_candidate_ids": bool(
                    RQ2_BLOCK_RANDOM_EXCLUDE_MINED_CANDIDATES
                ),
            },
        )

        rq2_block_balance_rows.append({
            "seed": victim_seed,
            "candidate_config_id": run[
                "candidate_config_id"
            ],
            "scoring_mode": scoring_mode,
            "comparison_rule": mode_groups[
                "comparison_rule"
            ],
            "endpoint_mining_hop": int(
                run.get(
                    "endpoint_mining_hop",
                    0,
                )
            ),
            "repeat": int(repeat),
            "candidate_set_size": int(
                run["candidate_set_size"]
            ),
            "n_candidate_rows": int(
                mode_groups[
                    "n_candidate_rows"
                ]
            ),
            "n_observed_scores": int(
                mode_groups[
                    "n_observed_scores"
                ]
            ),
            "n_group_high_edges": n_high,
            "n_group_low_edges": n_low,
            "balanced_block_size": (
                balanced_block_size
            ),
            "group_high_initialization": (
                high_initialization
            ),
            "group_low_initialization": (
                low_initialization
            ),
            "group_high_target": mode_groups[
                "group_high_target"
            ],
            "group_low_target": mode_groups[
                "group_low_target"
            ],
            "minority_group": minority_group,
            "majority_group": majority_group,
            "high_was_subsampled": bool(
                n_high
                > balanced_block_size
            ),
            "low_was_subsampled": bool(
                n_low
                > balanced_block_size
            ),
            "high_sampling_fraction": (
                balanced_block_size
                / n_high
            ),
            "low_sampling_fraction": (
                balanced_block_size
                / n_low
            ),
            "high_subset_seed": int(
                high_subset_seed
            ),
            "low_subset_seed": int(
                low_subset_seed
            ),
            "random_block_seed": int(
                random_seed
            ),
            "extreme_fraction": (
                mode_groups[
                    "extreme_fraction"
                ]
            ),
            "bottom_cutoff_score": (
                mode_groups[
                    "bottom_cutoff_score"
                ]
            ),
            "top_cutoff_score": (
                mode_groups[
                    "top_cutoff_score"
                ]
            ),
            "bottom_score_mean": (
                mode_groups[
                    "bottom_score_mean"
                ]
            ),
            "top_score_mean": (
                mode_groups[
                    "top_score_mean"
                ]
            ),
            "constant_score_run": (
                mode_groups[
                    "constant_score_run"
                ]
            ),
            "high_block_path": str(
                high_path
            ),
            "low_block_path": str(
                low_path
            ),
            "random_block_path": str(
                random_path
            ),
        })

        initialization_specs = [
            {
                "initialization": (
                    high_initialization
                ),
                "block_origin": "mined",
                "target_group": "high",
                "target_group_name": (
                    mode_groups[
                        "group_high_target"
                    ]
                ),
                "initial_ids": high_ids,
                "block_path": high_path,
            },
            {
                "initialization": (
                    low_initialization
                ),
                "block_origin": "mined",
                "target_group": "low",
                "target_group_name": (
                    mode_groups[
                        "group_low_target"
                    ]
                ),
                "initial_ids": low_ids,
                "block_path": low_path,
            },
            {
                "initialization": (
                    "random_full_space"
                ),
                "block_origin": "random",
                "target_group": "random",
                "target_group_name": (
                    "random_control"
                ),
                "initial_ids": random_ids,
                "block_path": random_path,
            },
        ]

        for initialization_spec in (
            initialization_specs
        ):
            initialization = (
                initialization_spec[
                    "initialization"
                ]
            )

            block_origin = (
                initialization_spec[
                    "block_origin"
                ]
            )

            target_group = (
                initialization_spec[
                    "target_group"
                ]
            )

            target_group_name = (
                initialization_spec[
                    "target_group_name"
                ]
            )

            initial_ids = (
                initialization_spec[
                    "initial_ids"
                ]
            )

            block_path = (
                initialization_spec[
                    "block_path"
                ]
            )

            block_size = (
                balanced_block_size
            )

            initial_set = set(
                initial_ids.tolist()
            )

            initial_high_fraction = (
                len(
                    initial_set
                    & high_set_full
                )
                / max(
                    1,
                    len(initial_set),
                )
            )

            initial_low_fraction = (
                len(
                    initial_set
                    & low_set_full
                )
                / max(
                    1,
                    len(initial_set),
                )
            )

            for resampling_enabled in (
                False,
                True,
            ):
                condition = (
                    f"{initialization}__"
                    f"{'random_resampling' if resampling_enabled else 'fixed_block'}"
                )

                attack_sampling_seed = (
                    500_000
                    + victim_seed * 10_000
                    + source_index * 1_000
                    + repeat
                )

                attack_params = {
                    "block_size": int(
                        block_size
                    ),
                    "epochs": int(
                        RQ2_BLOCK_EPOCHS
                    ),
                    "fine_tune_epochs": int(
                        RQ2_BLOCK_FINE_TUNE_EPOCHS
                    ),
                    "with_early_stopping": bool(
                        RQ2_BLOCK_WITH_EARLY_STOPPING
                    ),
                    "keep_heuristic": (
                        "WeightOnly"
                    ),
                    "do_synchronize": True,
                    "loss_type": (
                        "tanhMargin"
                    ),
                    "initial_block_path": str(
                        block_path
                    ),
                    "initial_block_label": (
                        condition
                    ),
                    "resampling_enabled": bool(
                        resampling_enabled
                    ),
                    "block_diagnostics_enabled": (
                        True
                    ),
                    "attack_sampling_seed": int(
                        attack_sampling_seed
                    ),
                }

                print(
                    "\n",
                    f"seed={victim_seed} | "
                    f"{run['candidate_config_id']} | "
                    f"mode={scoring_mode} | "
                    f"h={run.get('endpoint_mining_hop', 0)} | "
                    f"repeat={repeat} | "
                    f"condition={condition} | "
                    f"block_size={block_size}",
                )

                set_global_seed(
                    attack_sampling_seed
                )

                started = timer()

                result = (
                    experiment_global_attack_direct
                    .run(
                        graph=graph_sparse_rq2,
                        data_dir="./data",
                        dataset=DATASET,
                        attack="PRBCD",
                        attack_params=attack_params,
                        selector_params={},
                        epsilons=[
                            RQ2_BLOCK_EPSILON
                        ],
                        binary_attr=False,
                        make_undirected=True,
                        seed=victim_seed,
                        artifact_dir=(
                            RQ2_BLOCK_ARTIFACT_DIR
                        ),
                        pert_adj_storage_type=(
                            RQ2_BLOCK_PERT_ADJ_STORAGE_TYPE
                        ),
                        pert_attr_storage_type=(
                            RQ2_BLOCK_PERT_ATTR_STORAGE_TYPE
                        ),
                        model_label=MODEL_LABEL,
                        model_storage_type=(
                            RQ2_BLOCK_MODEL_STORAGE_TYPE
                        ),
                        device="cpu",
                        data_device="cpu",
                        debug_level="info",
                        semi=True,
                        use_cert=(
                            RQ2_BLOCK_USE_CERT
                        ),
                    )
                )

                runtime_seconds = (
                    timer()
                    - started
                )

                stats = _find_attack_statistics(
                    result
                )

                if not stats:
                    raise RuntimeError(
                        "PRBCD attack statistics "
                        "were not found."
                    )

                diagnostics = (
                    stats.get(
                        "block_diagnostics",
                        {},
                    )
                    or {}
                )

                accuracies = np.asarray(
                    _as_list(
                        stats.get(
                            "accuracy"
                        )
                    ),
                    dtype=float,
                )

                losses = np.asarray(
                    _as_list(
                        stats.get(
                            "loss"
                        )
                    ),
                    dtype=float,
                )

                if accuracies.size == 0:
                    raise RuntimeError(
                        "PRBCD run contains no "
                        "accuracy trajectory."
                    )

                clean_accuracy = float(
                    accuracies[0]
                )

                final_accuracy = (
                    _extract_final_accuracy(
                        result
                    )
                )

                min_relaxed_accuracy = (
                    float(
                        np.nanmin(
                            accuracies[1:]
                        )
                    )
                    if accuracies.size > 1
                    else clean_accuracy
                )

                integrated_drop = float(
                    np.trapz(
                        clean_accuracy
                        - accuracies,
                        dx=1.0,
                    )
                    / max(
                        1,
                        accuracies.size - 1,
                    )
                )

                final_ids = torch.as_tensor(
                    diagnostics.get(
                        "final_linear_ids",
                        torch.empty(
                            0,
                            dtype=torch.long,
                        ),
                    ),
                    dtype=torch.long,
                ).flatten()

                final_set = set(
                    final_ids.tolist()
                )

                final_block = (
                    torch.as_tensor(
                        diagnostics.get(
                            "final_block",
                            torch.empty(
                                0,
                                dtype=torch.long,
                            ),
                        ),
                        dtype=torch.long,
                    )
                    .flatten()
                )

                final_block_set = set(
                    final_block.tolist()
                )

                common_row = {
                    "block_schema_version": (
                        RQ2_BLOCK_SCHEMA_VERSION
                    ),
                    "seed": victim_seed,
                    "candidate_config_id": run[
                        "candidate_config_id"
                    ],
                    "candidate_set_size": int(
                        run[
                            "candidate_set_size"
                        ]
                    ),
                    "prbcd_candidate_fraction": float(
                        run[
                            "prbcd_candidate_fraction"
                        ]
                    ),
                    "actual_prbcd_fraction": float(
                        run[
                            "actual_prbcd_fraction"
                        ]
                    ),
                    "scoring_mode": scoring_mode,
                    "comparison_rule": (
                        mode_groups[
                            "comparison_rule"
                        ]
                    ),
                    "endpoint_mining_hop": int(
                        run.get(
                            "endpoint_mining_hop",
                            0,
                        )
                    ),
                    "repeat": int(repeat),
                    "condition": condition,
                    "initialization": initialization,
                    "block_origin": block_origin,
                    "target_group": target_group,
                    "target_group_name": (
                        target_group_name
                    ),
                    "resampling_enabled": bool(
                        resampling_enabled
                    ),
                    "block_size": int(
                        block_size
                    ),
                    "balanced_block_size": int(
                        balanced_block_size
                    ),
                    "n_group_high_edges": n_high,
                    "n_group_low_edges": n_low,
                    "group_high_initialization": (
                        high_initialization
                    ),
                    "group_low_initialization": (
                        low_initialization
                    ),
                    "group_high_target": (
                        mode_groups[
                            "group_high_target"
                        ]
                    ),
                    "group_low_target": (
                        mode_groups[
                            "group_low_target"
                        ]
                    ),
                    "minority_group": (
                        minority_group
                    ),
                    "majority_group": (
                        majority_group
                    ),
                    "high_was_subsampled": bool(
                        n_high
                        > balanced_block_size
                    ),
                    "low_was_subsampled": bool(
                        n_low
                        > balanced_block_size
                    ),
                    "high_sampling_fraction": (
                        balanced_block_size
                        / n_high
                    ),
                    "low_sampling_fraction": (
                        balanced_block_size
                        / n_low
                    ),
                    "initial_high_fraction": (
                        initial_high_fraction
                    ),
                    "initial_low_fraction": (
                        initial_low_fraction
                    ),
                    "extreme_fraction": (
                        mode_groups[
                            "extreme_fraction"
                        ]
                    ),
                    "bottom_cutoff_score": (
                        mode_groups[
                            "bottom_cutoff_score"
                        ]
                    ),
                    "top_cutoff_score": (
                        mode_groups[
                            "top_cutoff_score"
                        ]
                    ),
                    "bottom_score_mean": (
                        mode_groups[
                            "bottom_score_mean"
                        ]
                    ),
                    "top_score_mean": (
                        mode_groups[
                            "top_score_mean"
                        ]
                    ),
                    "attack_budget": int(
                        RQ2_BLOCK_ATTACK_BUDGET
                    ),
                    "epsilon": float(
                        RQ2_BLOCK_EPSILON
                    ),
                    "epochs": int(
                        RQ2_BLOCK_EPOCHS
                    ),
                    "nominal_resampling_epochs": int(
                        RQ2_BLOCK_RESAMPLING_EPOCHS
                    ),
                    "fine_tune_epochs": int(
                        RQ2_BLOCK_FINE_TUNE_EPOCHS
                    ),
                }

                rq2_block_run_rows.append({
                    **common_row,
                    "clean_accuracy": (
                        clean_accuracy
                    ),
                    "final_accuracy": (
                        final_accuracy
                    ),
                    "final_accuracy_drop": (
                        clean_accuracy
                        - final_accuracy
                    ),
                    "minimum_relaxed_accuracy": (
                        min_relaxed_accuracy
                    ),
                    "maximum_relaxed_accuracy_drop": (
                        clean_accuracy
                        - min_relaxed_accuracy
                    ),
                    "integrated_accuracy_drop": (
                        integrated_drop
                    ),
                    "epoch_to_1pp_drop": (
                        _epoch_to_drop(
                            accuracies,
                            clean_accuracy,
                            0.01,
                        )
                    ),
                    "epoch_to_2pp_drop": (
                        _epoch_to_drop(
                            accuracies,
                            clean_accuracy,
                            0.02,
                        )
                    ),
                    "n_final_selected": len(
                        final_set
                    ),
                    "n_final_selected_from_initial": len(
                        final_set
                        & initial_set
                    ),
                    "fraction_final_selected_from_initial": (
                        len(
                            final_set
                            & initial_set
                        )
                        / max(
                            1,
                            len(final_set),
                        )
                    ),
                    "n_final_selected_group_high": len(
                        final_set
                        & high_set_full
                    ),
                    "fraction_final_selected_group_high": (
                        len(
                            final_set
                            & high_set_full
                        )
                        / max(
                            1,
                            len(final_set),
                        )
                    ),
                    "n_final_selected_group_low": len(
                        final_set
                        & low_set_full
                    ),
                    "fraction_final_selected_group_low": (
                        len(
                            final_set
                            & low_set_full
                        )
                        / max(
                            1,
                            len(final_set),
                        )
                    ),
                    "final_block_initial_retention": (
                        len(
                            final_block_set
                            & initial_set
                        )
                        / max(
                            1,
                            len(initial_set),
                        )
                    ),
                    "final_block_group_high_fraction": (
                        len(
                            final_block_set
                            & high_set_full
                        )
                        / max(
                            1,
                            len(final_block_set),
                        )
                    ),
                    "final_block_group_low_fraction": (
                        len(
                            final_block_set
                            & low_set_full
                        )
                        / max(
                            1,
                            len(final_block_set),
                        )
                    ),
                    "runtime_seconds": float(
                        runtime_seconds
                    ),
                    "initial_block_path": str(
                        block_path
                    ),
                })

                for (
                    stat_index,
                    accuracy_value,
                ) in enumerate(
                    accuracies
                ):
                    prbcd_epoch = (
                        stat_index
                        - 1
                    )

                    rq2_block_epoch_rows.append({
                        **common_row,
                        "stat_index": int(
                            stat_index
                        ),
                        "prbcd_epoch": int(
                            prbcd_epoch
                        ),
                        "is_clean_baseline": bool(
                            stat_index == 0
                        ),
                        "accuracy": float(
                            accuracy_value
                        ),
                        "accuracy_drop": (
                            clean_accuracy
                            - float(
                                accuracy_value
                            )
                        ),
                        "loss": (
                            float(
                                losses[
                                    stat_index
                                ]
                            )
                            if stat_index
                            < losses.size
                            else np.nan
                        ),
                    })

                epoch_blocks = (
                    diagnostics.get(
                        "epoch_blocks",
                        {},
                    )
                    or {}
                )

                for (
                    epoch_key,
                    current_block,
                ) in epoch_blocks.items():
                    current_set = set(
                        torch.as_tensor(
                            current_block,
                            dtype=torch.long,
                        )
                        .flatten()
                        .tolist()
                    )

                    rq2_block_retention_rows.append({
                        **common_row,
                        "prbcd_epoch": int(
                            epoch_key
                        ),
                        "initial_retention": (
                            len(
                                current_set
                                & initial_set
                            )
                            / max(
                                1,
                                len(initial_set),
                            )
                        ),
                        "group_high_fraction_in_current_block": (
                            len(
                                current_set
                                & high_set_full
                            )
                            / max(
                                1,
                                len(current_set),
                            )
                        ),
                        "group_low_fraction_in_current_block": (
                            len(
                                current_set
                                & low_set_full
                            )
                            / max(
                                1,
                                len(current_set),
                            )
                        ),
                    })

                print(
                    f"final_accuracy={final_accuracy:.6f} | "
                    f"drop={clean_accuracy-final_accuracy:.6f} | "
                    f"runtime={runtime_seconds:.1f}s"
                )

                del result
                gc.collect()

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()


# ============================================================
# Save and aggregate
# ============================================================

rq2_block_runs_raw_df = pd.DataFrame(
    rq2_block_run_rows
)

rq2_block_epochs_raw_df = pd.DataFrame(
    rq2_block_epoch_rows
)

rq2_block_retention_raw_df = pd.DataFrame(
    rq2_block_retention_rows
)

rq2_block_balance_raw_df = pd.DataFrame(
    rq2_block_balance_rows
)

if rq2_block_runs_raw_df.empty:
    raise RuntimeError(
        "No RQ2 matched block attacks were completed."
    )

rq2_group_cols = [
    "candidate_config_id",
    "scoring_mode",
    "comparison_rule",
    "endpoint_mining_hop",
    "initialization",
    "block_origin",
    "target_group",
    "target_group_name",
    "resampling_enabled",
    "condition",
]

rq2_run_metrics = [
    "block_size",
    "balanced_block_size",
    "n_group_high_edges",
    "n_group_low_edges",
    "high_sampling_fraction",
    "low_sampling_fraction",
    "clean_accuracy",
    "final_accuracy",
    "final_accuracy_drop",
    "minimum_relaxed_accuracy",
    "maximum_relaxed_accuracy_drop",
    "integrated_accuracy_drop",
    "epoch_to_1pp_drop",
    "epoch_to_2pp_drop",
    "fraction_final_selected_from_initial",
    "fraction_final_selected_group_high",
    "fraction_final_selected_group_low",
    "final_block_initial_retention",
    "final_block_group_high_fraction",
    "final_block_group_low_fraction",
    "runtime_seconds",
]

rq2_block_runs_summary_df = (
    _two_stage_summary(
        rq2_block_runs_raw_df,
        rq2_group_cols,
        rq2_run_metrics,
    )
)

rq2_epoch_group_cols = (
    rq2_group_cols
    + ["prbcd_epoch"]
)

rq2_block_epochs_summary_df = (
    _two_stage_summary(
        rq2_block_epochs_raw_df,
        rq2_epoch_group_cols,
        [
            "block_size",
            "accuracy",
            "accuracy_drop",
            "loss",
        ],
    )
)

if not rq2_block_retention_raw_df.empty:
    rq2_block_retention_summary_df = (
        _two_stage_summary(
            rq2_block_retention_raw_df,
            rq2_epoch_group_cols,
            [
                "block_size",
                "initial_retention",
                "group_high_fraction_in_current_block",
                "group_low_fraction_in_current_block",
            ],
        )
    )
else:
    rq2_block_retention_summary_df = (
        pd.DataFrame()
    )

for frame in (
    rq2_block_runs_summary_df,
    rq2_block_epochs_summary_df,
    rq2_block_retention_summary_df,
):
    if (
        not frame.empty
        and "block_size_mean"
        in frame.columns
    ):
        frame["block_size"] = (
            frame["block_size_mean"]
        )


# ============================================================
# Paired initialization differences
# ============================================================

def _paired_condition_difference(
    frame,
    condition_a,
    condition_b,
):
    index_cols = [
        "seed",
        "candidate_config_id",
        "scoring_mode",
        "comparison_rule",
        "endpoint_mining_hop",
        "repeat",
        "resampling_enabled",
    ]

    metrics = [
        "final_accuracy",
        "final_accuracy_drop",
        "integrated_accuracy_drop",
        "minimum_relaxed_accuracy",
        "maximum_relaxed_accuracy_drop",
    ]

    left = frame[
        frame["initialization"]
        == condition_a
    ][
        index_cols
        + metrics
    ].copy()

    right = frame[
        frame["initialization"]
        == condition_b
    ][
        index_cols
        + metrics
    ].copy()

    paired = left.merge(
        right,
        on=index_cols,
        how="inner",
        suffixes=(
            "_a",
            "_b",
        ),
        validate="one_to_one",
    )

    paired["condition_a"] = (
        condition_a
    )

    paired["condition_b"] = (
        condition_b
    )

    for metric in metrics:
        paired[
            f"{metric}_difference"
        ] = (
            paired[
                f"{metric}_a"
            ]
            - paired[
                f"{metric}_b"
            ]
        )

    return paired


pair_frames = []

for (
    scoring_mode,
    comparison_rule,
), group in rq2_block_runs_raw_df.groupby(
    [
        "scoring_mode",
        "comparison_rule",
    ],
    dropna=False,
):
    high_initializations = (
        group.loc[
            group["target_group"]
            == "high",
            "initialization",
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    low_initializations = (
        group.loc[
            group["target_group"]
            == "low",
            "initialization",
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    if (
        len(high_initializations) != 1
        or len(low_initializations) != 1
    ):
        raise RuntimeError(
            "Expected exactly one high and one low "
            f"initialization for {scoring_mode}, {comparison_rule}."
        )

    high_initialization = (
        high_initializations[0]
    )

    low_initialization = (
        low_initializations[0]
    )

    pair_frames.extend([
        _paired_condition_difference(
            group,
            high_initialization,
            low_initialization,
        ),
        _paired_condition_difference(
            group,
            high_initialization,
            "random_full_space",
        ),
        _paired_condition_difference(
            group,
            low_initialization,
            "random_full_space",
        ),
    ])

rq2_block_paired_raw_df = (
    pd.concat(
        [
            frame
            for frame in pair_frames
            if not frame.empty
        ],
        ignore_index=True,
    )
    if any(
        not frame.empty
        for frame in pair_frames
    )
    else pd.DataFrame()
)

if not rq2_block_paired_raw_df.empty:
    paired_group_cols = [
        "candidate_config_id",
        "scoring_mode",
        "comparison_rule",
        "endpoint_mining_hop",
        "resampling_enabled",
        "condition_a",
        "condition_b",
    ]

    paired_metrics = [
        column
        for column in rq2_block_paired_raw_df.columns
        if column.endswith(
            "_difference"
        )
    ]

    rq2_block_paired_summary_df = (
        _two_stage_summary(
            rq2_block_paired_raw_df,
            paired_group_cols,
            paired_metrics,
        )
    )
else:
    rq2_block_paired_summary_df = (
        pd.DataFrame()
    )


# ============================================================
# Write outputs
# ============================================================

rq2_block_runs_raw_df.to_csv(
    RQ2_BLOCK_OUT_DIR
    / "runs_raw.csv",
    index=False,
)

rq2_block_runs_summary_df.to_csv(
    RQ2_BLOCK_OUT_DIR
    / "runs_summary.csv",
    index=False,
)

rq2_block_epochs_raw_df.to_csv(
    RQ2_BLOCK_OUT_DIR
    / "epoch_metrics_raw.csv",
    index=False,
)

rq2_block_epochs_summary_df.to_csv(
    RQ2_BLOCK_OUT_DIR
    / "epoch_metrics_summary.csv",
    index=False,
)

rq2_block_retention_raw_df.to_csv(
    RQ2_BLOCK_OUT_DIR
    / "block_retention_raw.csv",
    index=False,
)

rq2_block_retention_summary_df.to_csv(
    RQ2_BLOCK_OUT_DIR
    / "block_retention_summary.csv",
    index=False,
)

rq2_block_balance_raw_df.to_csv(
    RQ2_BLOCK_OUT_DIR
    / "block_balance_raw.csv",
    index=False,
)

rq2_block_paired_raw_df.to_csv(
    RQ2_BLOCK_OUT_DIR
    / "paired_differences_raw.csv",
    index=False,
)

rq2_block_paired_summary_df.to_csv(
    RQ2_BLOCK_OUT_DIR
    / "paired_differences_summary.csv",
    index=False,
)

experiment_config = {
    "block_schema_version": (
        RQ2_BLOCK_SCHEMA_VERSION
    ),
    "dataset": DATASET,
    "epsilon": RQ2_BLOCK_EPSILON,
    "attack_budget": (
        RQ2_BLOCK_ATTACK_BUDGET
    ),
    "epochs": RQ2_BLOCK_EPOCHS,
    "nominal_resampling_epochs": (
        RQ2_BLOCK_RESAMPLING_EPOCHS
    ),
    "fine_tune_epochs": (
        RQ2_BLOCK_FINE_TUNE_EPOCHS
    ),
    "repeats": RQ2_BLOCK_REPEATS,
    "with_early_stopping": (
        RQ2_BLOCK_WITH_EARLY_STOPPING
    ),
    "scoring_modes": sorted(
        RQ2_BLOCK_SCORING_MODES
    ),
    "endpoint_label_threshold": (
        RQ2_BLOCK_ENDPOINT_LABEL_THRESHOLD
    ),
    "subset_extreme_fraction": (
        RQ2_BLOCK_SUBSET_EXTREME_FRACTION
    ),
    "endpoint_matching_strategy": (
        "common block size=min(n_label_1,n_label_0); "
        "larger group sampled without replacement per repeat"
    ),
    "subset_matching_strategy": (
        "equal top and bottom observed-score extremes; "
        "score_raw ranking with deterministic sample-index tie break"
    ),
    "random_excludes_all_mined_candidate_ids": bool(
        RQ2_BLOCK_RANDOM_EXCLUDE_MINED_CANDIDATES
    ),
    "victim_seeds": sorted({
        int(run["seed"])
        for run in rq2_source_runs
    }),
}

(
    RQ2_BLOCK_OUT_DIR
    / "experiment_config.json"
).write_text(
    json.dumps(
        experiment_config,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "\nMatched block construction"
)

display(
    rq2_block_balance_raw_df
)

print(
    "\nSeed-averaged attack results"
)

display(
    rq2_block_runs_summary_df
)

print(
    "\nPaired condition differences"
)

display(
    rq2_block_paired_summary_df
)

print(
    "Saved mode-aware RQ2 block experiment to:",
    RQ2_BLOCK_OUT_DIR.resolve(),
)


### Analysis and plotting block

The following cell evaluates attack strength, optimization dynamics, and block retention.

#### Attack outcomes

The analysis reports:

- final attacked accuracy;
- final accuracy drop;
- minimum relaxed accuracy;
- maximum relaxed accuracy drop;
- integrated accuracy drop;
- number of epochs required to reach fixed accuracy reductions.

#### Candidate utilization

The analysis also reports:

- fraction of final perturbations originating from the initial block;
- fraction of final perturbations carrying label 1;
- fraction of final perturbations carrying label 0;
- retention of the original block over optimization epochs;
- label composition of the final candidate block;
- label composition of the final discrete perturbation set.

#### Main comparisons

The paired comparisons are:

\[
\text{label 1}
\quad\text{versus}\quad
\text{label 0},
\]

\[
\text{label 1}
\quad\text{versus}\quad
\text{random},
\]

and

\[
\text{fixed block}
\quad\text{versus}\quad
\text{random resampling}.
\]

This experiment connects local harmfulness directly to the candidate-coverage limitation identified in RQ1.

In [ ]:
# ============================================================
# Plot an existing complete or partially completed mode-aware
# RQ2 initial-block run.
# ============================================================

from pathlib import Path
from itertools import combinations
from statistics import NormalDist
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from scipy.stats import t as student_t
    from scipy.stats import ttest_1samp
except ImportError:
    student_t = None
    ttest_1samp = None


# ============================================================
# Select the existing run
# ============================================================

def _resolve_rq2_block_output_directory():
    current = globals().get(
        "RQ2_BLOCK_OUT_DIR"
    )

    if current is not None:
        current = Path(current)

        if current.exists():
            return current

    base = Path(
        globals().get(
            "RQ2_BLOCK_BASE_OUT_DIR",
            Path("extendedPlotting")
            / "rq2_mode_aware_initial_block_multiseed",
        )
    )

    latest_file = (
        base
        / "latest_run.txt"
    )

    if latest_file.exists():
        latest_path = Path(
            latest_file
            .read_text(
                encoding="utf-8"
            )
            .strip()
        )

        if latest_path.exists():
            return latest_path

    completed = [
        path
        for path in base.glob("*")
        if path.is_dir()
        and (
            path
            / "runs_raw.csv"
        ).exists()
    ]

    if completed:
        return max(
            completed,
            key=lambda path: (
                path.stat().st_mtime,
                path.name,
            ),
        )

    raise FileNotFoundError(
        "Could not find an RQ2 block output directory. "
        "Run the mode-aware execution cell first."
    )


RQ2_BLOCK_OUT_DIR = (
    _resolve_rq2_block_output_directory()
)

RQ2_BLOCK_PLOT_DIR = RUN_PLOTS_DIR

RQ2_BLOCK_PLOT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RQ2_BLOCK_REPORT_DIR = (
    RQ2_BLOCK_OUT_DIR
    / "thesis_reports"
)

RQ2_BLOCK_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Statistical reporting configuration.
RQ2_CONFIDENCE_LEVEL = float(
    globals().get(
        "RQ2_CONFIDENCE_LEVEL",
        0.95,
    )
)

# Accuracy values are stored as fractions. Thesis-facing tables and plots use
# percentage points for accuracy changes and percentages for accuracies.
RQ2_ACCURACY_SCALE = 100.0


# ============================================================
# General helpers
# ============================================================

def _rq2_safe(value):
    return re.sub(
        r"[^a-zA-Z0-9_.=-]+",
        "_",
        str(value),
    ).strip("_")


def _rq2_read_csv(
    out_dir,
    filename,
):
    path = (
        Path(out_dir)
        / filename
    )

    if not path.exists():
        print(
            f"[missing] {filename}"
        )
        return pd.DataFrame()

    if path.stat().st_size == 0:
        print(
            f"[empty]   {filename}"
        )
        return pd.DataFrame()

    try:
        frame = pd.read_csv(
            path,
            on_bad_lines="skip",
        )
    except pd.errors.EmptyDataError:
        print(
            f"[empty]   {filename}"
        )
        return pd.DataFrame()

    print(
        f"[loaded]  {filename}: "
        f"{len(frame):,} rows"
    )

    return frame


def _rq2_t_critical(
    count,
    confidence_level=RQ2_CONFIDENCE_LEVEL,
):
    """Return a two-sided t critical value, or NaN when it is undefined."""
    count = pd.to_numeric(
        count,
        errors="coerce",
    )

    alpha = 1.0 - float(confidence_level)
    result = pd.Series(
        np.nan,
        index=count.index,
        dtype=float,
    )

    valid = count >= 2

    if not valid.any():
        return result

    if student_t is not None:
        result.loc[valid] = student_t.ppf(
            1.0 - alpha / 2.0,
            count.loc[valid] - 1,
        )
    else:
        # Fallback only. Installing SciPy is recommended because a normal
        # critical value is optimistic for small numbers of seeds.
        result.loc[valid] = NormalDist().inv_cdf(
            1.0 - alpha / 2.0
        )

    return result


def _rq2_two_stage_summary(
    frame,
    group_cols,
    metric_cols,
):
    """
    Average repeats within each seed, then summarize across independent seeds.

    The seed is the inferential unit. Standard deviations and confidence
    intervals are therefore calculated across seed-level means, not across
    epochs or repeated attack executions sharing the same victim-model seed.
    """
    if frame.empty:
        return pd.DataFrame()

    missing_group_cols = [
        column
        for column in (
            group_cols
            + ["seed"]
        )
        if column
        not in frame.columns
    ]

    if missing_group_cols:
        raise ValueError(
            "Cannot rebuild summary. "
            "Missing grouping columns: "
            f"{missing_group_cols}"
        )

    available_metrics = [
        column
        for column in metric_cols
        if column in frame.columns
    ]

    if not available_metrics:
        return pd.DataFrame()

    work = frame.copy()

    for metric in available_metrics:
        work[metric] = pd.to_numeric(
            work[metric],
            errors="coerce",
        )

    # Stage 1: collapse stochastic repeats belonging to the same seed.
    seed_level = (
        work
        .groupby(
            group_cols + ["seed"],
            dropna=False,
        )[available_metrics]
        .mean()
        .reset_index()
    )

    # Stage 2: describe variability across independent seeds.
    grouped = seed_level.groupby(
        group_cols,
        dropna=False,
    )

    summary = grouped[available_metrics].agg(
        [
            "mean",
            "std",
            "count",
        ]
    )

    summary.columns = [
        f"{metric}_{statistic}"
        for metric, statistic
        in summary.columns
    ]

    summary = summary.reset_index()

    n_seeds = (
        seed_level
        .groupby(
            group_cols,
            dropna=False,
        )["seed"]
        .nunique()
        .rename("n_seeds")
        .reset_index()
    )

    summary = summary.merge(
        n_seeds,
        on=group_cols,
        how="left",
        validate="one_to_one",
    )

    for metric in available_metrics:
        mean_column = f"{metric}_mean"
        std_column = f"{metric}_std"
        count_column = f"{metric}_count"
        sem_column = f"{metric}_sem"
        low_column = f"{metric}_ci_low"
        high_column = f"{metric}_ci_high"

        # Important: do not replace a one-seed SD with zero. With n=1,
        # between-seed variability and a confidence interval are undefined.
        summary[sem_column] = (
            summary[std_column]
            / np.sqrt(summary[count_column])
        )

        critical = _rq2_t_critical(
            summary[count_column],
            RQ2_CONFIDENCE_LEVEL,
        )

        half_width = critical * summary[sem_column]
        summary[low_column] = summary[mean_column] - half_width
        summary[high_column] = summary[mean_column] + half_width

    return summary


def _rq2_seed_level_frame(
    frame,
    group_cols,
    metric_cols,
):
    """Return one row per condition and seed after averaging repeats."""
    if frame.empty:
        return pd.DataFrame()

    available_metrics = [
        metric
        for metric in metric_cols
        if metric in frame.columns
    ]

    if not available_metrics:
        return pd.DataFrame()

    work = frame.copy()
    for metric in available_metrics:
        work[metric] = pd.to_numeric(
            work[metric],
            errors="coerce",
        )

    return (
        work
        .groupby(
            group_cols + ["seed"],
            dropna=False,
        )[available_metrics]
        .mean()
        .reset_index()
    )

def _rq2_add_block_size_alias(
    frame,
):
    frame = frame.copy()

    if (
        not frame.empty
        and "block_size"
        not in frame.columns
        and "block_size_mean"
        in frame.columns
    ):
        frame["block_size"] = (
            frame[
                "block_size_mean"
            ]
        )

    return frame


def _rq2_is_true(series):
    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .isin({
            "true",
            "1",
            "yes",
        })
    )


def _rq2_add_mean_ci_band(
    ax,
    x,
    mean,
    ci_low,
    ci_high,
    *,
    label,
    scale=1.0,
):
    """Plot a mean line and its confidence interval."""
    plot_frame = pd.DataFrame({
        "x": pd.to_numeric(x, errors="coerce"),
        "mean": pd.to_numeric(mean, errors="coerce"),
        "ci_low": pd.to_numeric(ci_low, errors="coerce"),
        "ci_high": pd.to_numeric(ci_high, errors="coerce"),
    }).dropna(
        subset=["x", "mean"]
    )

    if plot_frame.empty:
        return False

    plot_frame = plot_frame.sort_values("x")

    x_values = plot_frame["x"].to_numpy(dtype=float)
    mean_values = plot_frame["mean"].to_numpy(dtype=float) * scale
    low_values = plot_frame["ci_low"].to_numpy(dtype=float) * scale
    high_values = plot_frame["ci_high"].to_numpy(dtype=float) * scale

    line = ax.plot(
        x_values,
        mean_values,
        label=label,
    )[0]

    finite_interval = np.isfinite(low_values) & np.isfinite(high_values)
    if finite_interval.any():
        ax.fill_between(
            x_values,
            low_values,
            high_values,
            where=finite_interval,
            interpolate=True,
            alpha=0.16,
            color=line.get_color(),
        )

    return True


def _rq2_ci_yerr(
    mean,
    ci_low,
    ci_high,
    *,
    scale=1.0,
):
    """Return Matplotlib-compatible asymmetric confidence-interval errors."""
    mean_values = pd.to_numeric(mean, errors="coerce").to_numpy(dtype=float)
    low_values = pd.to_numeric(ci_low, errors="coerce").to_numpy(dtype=float)
    high_values = pd.to_numeric(ci_high, errors="coerce").to_numpy(dtype=float)

    lower = (mean_values - low_values) * scale
    upper = (high_values - mean_values) * scale

    lower[~np.isfinite(lower)] = 0.0
    upper[~np.isfinite(upper)] = 0.0

    return np.vstack([lower, upper])

def _add_mode_aware_columns(
    frame,
):
    """
    Add missing mode-aware columns and support older endpoint-only outputs.
    """
    if frame.empty:
        return frame

    frame = frame.copy()

    if "scoring_mode" not in frame.columns:
        frame[
            "scoring_mode"
        ] = "endpoint"

    if "comparison_rule" not in frame.columns:
        frame[
            "comparison_rule"
        ] = np.where(
            frame[
                "scoring_mode"
            ].astype(str)
            == "subset_accuracy_drop",
            "top_vs_bottom_10pct",
            "endpoint_label_1_vs_0",
        )

    if "target_group" not in frame.columns:
        initialization = (
            frame.get(
                "initialization",
                pd.Series(
                    "",
                    index=frame.index,
                ),
            )
            .astype(str)
        )

        frame[
            "target_group"
        ] = np.select(
            [
                initialization.isin({
                    "label_positive",
                    "score_top",
                }),
                initialization.isin({
                    "label_zero",
                    "score_bottom",
                }),
                initialization.eq(
                    "random_full_space"
                ),
            ],
            [
                "high",
                "low",
                "random",
            ],
            default="unknown",
        )

    legacy_map = {
        "fraction_final_selected_group_high": (
            "fraction_final_selected_label_positive"
        ),
        "fraction_final_selected_group_low": (
            "fraction_final_selected_label_zero"
        ),
        "final_block_group_high_fraction": (
            "final_block_positive_fraction"
        ),
        "final_block_group_low_fraction": (
            "final_block_zero_fraction"
        ),
        "group_high_fraction_in_current_block": (
            "positive_fraction_in_current_block"
        ),
        "group_low_fraction_in_current_block": (
            "zero_fraction_in_current_block"
        ),
        "n_group_high_edges": (
            "n_positive_labels"
        ),
        "n_group_low_edges": (
            "n_zero_labels"
        ),
        "high_sampling_fraction": (
            "positive_sampling_fraction"
        ),
        "low_sampling_fraction": (
            "zero_sampling_fraction"
        ),
    }

    for (
        new_column,
        old_column,
    ) in legacy_map.items():
        if (
            new_column
            not in frame.columns
            and old_column
            in frame.columns
        ):
            frame[
                new_column
            ] = frame[
                old_column
            ]

    return frame


def _rq2_attach_group_lookup(
    target,
    source,
    *,
    value_column,
    output_column,
):
    """Attach a grouped value using the most specific shared run keys."""
    if (
        target.empty
        or source.empty
        or value_column not in source.columns
    ):
        target = target.copy()
        target[output_column] = np.nan
        return target

    key_candidates = [
        "candidate_config_id",
        "scoring_mode",
        "comparison_rule",
        "endpoint_mining_hop",
        "seed",
        "repeat",
        "condition",
    ]

    keys = [
        column
        for column in key_candidates
        if column in target.columns
        and column in source.columns
    ]

    # A seed key is essential; otherwise the lookup would mix baselines.
    if "seed" not in keys:
        target = target.copy()
        target[output_column] = np.nan
        return target

    lookup = (
        source
        .dropna(subset=[value_column])
        .groupby(
            keys,
            dropna=False,
        )[value_column]
        .mean()
        .rename(output_column)
        .reset_index()
    )

    return target.merge(
        lookup,
        on=keys,
        how="left",
        validate="many_to_one",
    )


def _rq2_add_seed_specific_accuracy_drops(
    runs_frame,
    epochs_frame,
):
    """
    Compute accuracy drops before any cross-seed aggregation.

    Every row is normalized against the clean baseline belonging to the same
    seed (and, where available, the same repeat/configuration/condition).
    """
    runs = runs_frame.copy()
    epochs = epochs_frame.copy()

    if not runs.empty:
        clean = pd.to_numeric(
            runs.get(
                "clean_accuracy",
                pd.Series(np.nan, index=runs.index),
            ),
            errors="coerce",
        )

        final = pd.to_numeric(
            runs.get(
                "final_accuracy",
                pd.Series(np.nan, index=runs.index),
            ),
            errors="coerce",
        )

        runs[
            "final_accuracy_drop_seed_baseline"
        ] = clean - final

        # Older raw files may already contain a correctly computed row-level
        # drop even when one of the two source columns is unavailable.
        if "final_accuracy_drop" in runs.columns:
            existing_drop = pd.to_numeric(
                runs["final_accuracy_drop"],
                errors="coerce",
            )
            runs[
                "final_accuracy_drop_seed_baseline"
            ] = runs[
                "final_accuracy_drop_seed_baseline"
            ].fillna(existing_drop)

    if not epochs.empty and "accuracy" in epochs.columns:
        epochs["accuracy"] = pd.to_numeric(
            epochs["accuracy"],
            errors="coerce",
        )

        baseline = pd.Series(
            np.nan,
            index=epochs.index,
            dtype=float,
        )

        # Prefer an explicit clean-accuracy column when available.
        if "clean_accuracy" in epochs.columns:
            baseline = pd.to_numeric(
                epochs["clean_accuracy"],
                errors="coerce",
            )

        # Next prefer the epoch -1 row from the same seed/run/condition.
        if "prbcd_epoch" in epochs.columns:
            epoch_number = pd.to_numeric(
                epochs["prbcd_epoch"],
                errors="coerce",
            )
            baseline_rows = epochs.loc[
                epoch_number.eq(-1)
            ].copy()

            if not baseline_rows.empty:
                epochs = _rq2_attach_group_lookup(
                    epochs,
                    baseline_rows,
                    value_column="accuracy",
                    output_column="__baseline_from_epoch_minus_one",
                )
                baseline = baseline.fillna(
                    pd.to_numeric(
                        epochs.pop(
                            "__baseline_from_epoch_minus_one"
                        ),
                        errors="coerce",
                    )
                )

        # Finally use the clean run-level accuracy from the matching seed/run.
        if not runs.empty and "clean_accuracy" in runs.columns:
            epochs = _rq2_attach_group_lookup(
                epochs,
                runs,
                value_column="clean_accuracy",
                output_column="__baseline_from_runs",
            )
            baseline = baseline.fillna(
                pd.to_numeric(
                    epochs.pop(
                        "__baseline_from_runs"
                    ),
                    errors="coerce",
                )
            )

        epochs[
            "seed_baseline_accuracy"
        ] = baseline

        epochs[
            "accuracy_drop_seed_baseline"
        ] = (
            epochs["seed_baseline_accuracy"]
            - epochs["accuracy"]
        )

        if "accuracy_drop" in epochs.columns:
            existing_drop = pd.to_numeric(
                epochs["accuracy_drop"],
                errors="coerce",
            )
            epochs[
                "accuracy_drop_seed_baseline"
            ] = epochs[
                "accuracy_drop_seed_baseline"
            ].fillna(existing_drop)

    return runs, epochs


def _mode_group_labels(
    scoring_mode,
    comparison_rule,
):
    scoring_mode = str(
        scoring_mode
    )

    if (
        scoring_mode
        == "subset_accuracy_drop"
    ):
        fraction_match = re.search(
            r"([0-9]+(?:p[0-9]+)?)pct",
            str(comparison_rule),
        )

        fraction_text = (
            fraction_match
            .group(1)
            .replace(
                "p",
                ".",
            )
            if fraction_match
            else "10"
        )

        return (
            f"Top {fraction_text}% score",
            f"Bottom {fraction_text}% score",
        )

    return (
        "Endpoint label 1",
        "Endpoint label 0",
    )


def _config_description(
    config_key,
):
    (
        candidate_config_id,
        scoring_mode,
        comparison_rule,
        endpoint_hop,
    ) = config_key

    return (
        f"config={candidate_config_id} | "
        f"mode={scoring_mode} | "
        f"comparison={comparison_rule} | "
        f"h={endpoint_hop}"
    )


# ============================================================
# Load available data
# ============================================================

print(
    "Loading RQ2 run:"
)

print(
    RQ2_BLOCK_OUT_DIR.resolve()
)

print()

rq2_block_runs_raw_df = (
    _add_mode_aware_columns(
        _rq2_read_csv(
            RQ2_BLOCK_OUT_DIR,
            "runs_raw.csv",
        )
    )
)

rq2_block_epochs_raw_df = (
    _add_mode_aware_columns(
        _rq2_read_csv(
            RQ2_BLOCK_OUT_DIR,
            "epoch_metrics_raw.csv",
        )
    )
)

rq2_block_retention_raw_df = (
    _add_mode_aware_columns(
        _rq2_read_csv(
            RQ2_BLOCK_OUT_DIR,
            "block_retention_raw.csv",
        )
    )
)

rq2_block_runs_summary_df = (
    _add_mode_aware_columns(
        _rq2_read_csv(
            RQ2_BLOCK_OUT_DIR,
            "runs_summary.csv",
        )
    )
)

rq2_block_epochs_summary_df = (
    _add_mode_aware_columns(
        _rq2_read_csv(
            RQ2_BLOCK_OUT_DIR,
            "epoch_metrics_summary.csv",
        )
    )
)

rq2_block_retention_summary_df = (
    _add_mode_aware_columns(
        _rq2_read_csv(
            RQ2_BLOCK_OUT_DIR,
            "block_retention_summary.csv",
        )
    )
)

rq2_block_paired_summary_df = (
    _add_mode_aware_columns(
        _rq2_read_csv(
            RQ2_BLOCK_OUT_DIR,
            "paired_differences_summary.csv",
        )
    )
)


(
    rq2_block_runs_raw_df,
    rq2_block_epochs_raw_df,
) = _rq2_add_seed_specific_accuracy_drops(
    rq2_block_runs_raw_df,
    rq2_block_epochs_raw_df,
)


# ============================================================
# Rebuild summaries from raw data where available
# ============================================================

rq2_group_cols = [
    "candidate_config_id",
    "scoring_mode",
    "comparison_rule",
    "endpoint_mining_hop",
    "initialization",
    "block_origin",
    "target_group",
    "target_group_name",
    "resampling_enabled",
    "condition",
]

rq2_run_metrics = [
    "block_size",
    "balanced_block_size",
    "n_group_high_edges",
    "n_group_low_edges",
    "high_sampling_fraction",
    "low_sampling_fraction",
    "clean_accuracy",
    "final_accuracy",
    "final_accuracy_drop",
    "final_accuracy_drop_seed_baseline",
    "minimum_relaxed_accuracy",
    "maximum_relaxed_accuracy_drop",
    "integrated_accuracy_drop",
    "epoch_to_1pp_drop",
    "epoch_to_2pp_drop",
    "fraction_final_selected_from_initial",
    "fraction_final_selected_group_high",
    "fraction_final_selected_group_low",
    "final_block_initial_retention",
    "final_block_group_high_fraction",
    "final_block_group_low_fraction",
    "runtime_seconds",
]

rq2_epoch_group_cols = (
    rq2_group_cols
    + ["prbcd_epoch"]
)

if not rq2_block_runs_raw_df.empty:
    print(
        "\nRebuilding runs summary from runs_raw.csv "
        "using seed-specific baselines"
    )

    rq2_block_runs_summary_df = (
        _rq2_two_stage_summary(
            rq2_block_runs_raw_df,
            rq2_group_cols,
            rq2_run_metrics,
        )
    )

if not rq2_block_epochs_raw_df.empty:
    print(
        "Rebuilding epoch summary from epoch_metrics_raw.csv "
        "using seed-specific baselines"
    )

    rq2_block_epochs_summary_df = (
        _rq2_two_stage_summary(
            rq2_block_epochs_raw_df,
            rq2_epoch_group_cols,
            [
                "block_size",
                "accuracy",
                "accuracy_drop",
                "accuracy_drop_seed_baseline",
                "seed_baseline_accuracy",
                "loss",
            ],
        )
    )

if not rq2_block_retention_raw_df.empty:
    print(
        "Rebuilding retention summary "
        "from block_retention_raw.csv"
    )

    rq2_block_retention_summary_df = (
        _rq2_two_stage_summary(
            rq2_block_retention_raw_df,
            rq2_epoch_group_cols,
            [
                "block_size",
                "initial_retention",
                "group_high_fraction_in_current_block",
                "group_low_fraction_in_current_block",
            ],
        )
    )

rq2_block_runs_summary_df = (
    _rq2_add_block_size_alias(
        rq2_block_runs_summary_df
    )
)

rq2_block_epochs_summary_df = (
    _rq2_add_block_size_alias(
        rq2_block_epochs_summary_df
    )
)

rq2_block_retention_summary_df = (
    _rq2_add_block_size_alias(
        rq2_block_retention_summary_df
    )
)


# ============================================================
# Stop when no metric data were saved
# ============================================================

if (
    rq2_block_runs_summary_df.empty
    and rq2_block_epochs_summary_df.empty
    and rq2_block_retention_summary_df.empty
):
    available_files = sorted(
        path.name
        for path in RQ2_BLOCK_OUT_DIR.iterdir()
    )

    raise RuntimeError(
        "No saved RQ2 metric CSV data were found.\n\n"
        "The initial block files alone cannot reconstruct "
        "accuracy trajectories when execution was interrupted "
        "before the CSV-writing stage.\n\n"
        f"Files currently present:\n{available_files}"
    )


# ============================================================
# Experiment configuration
# ============================================================

config_path = (
    RQ2_BLOCK_OUT_DIR
    / "experiment_config.json"
)

experiment_config = {}

if config_path.exists():
    with config_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        experiment_config = (
            json.load(file)
        )

resampling_boundary = int(
    experiment_config.get(
        "nominal_resampling_epochs",
        globals().get(
            "RQ2_BLOCK_RESAMPLING_EPOCHS",
            50,
        ),
    )
)


# ============================================================
# Thesis-ready result tables and paired inference
# ============================================================

def _rq2_holm_adjust(p_values):
    """Holm-adjust a sequence of p-values while preserving NaNs."""
    p_values = pd.to_numeric(
        pd.Series(p_values),
        errors="coerce",
    )
    adjusted = pd.Series(
        np.nan,
        index=p_values.index,
        dtype=float,
    )

    valid = p_values.dropna().sort_values()
    m = len(valid)
    running_max = 0.0

    for rank, (index, value) in enumerate(valid.items()):
        candidate = min(1.0, (m - rank) * float(value))
        running_max = max(running_max, candidate)
        adjusted.loc[index] = running_max

    return adjusted


def _rq2_build_paired_comparisons(
    runs_raw,
    *,
    metric="final_accuracy_drop_seed_baseline",
):
    """
    Compare conditions using complete seed pairs.

    Repeats are averaged within seed before pairing. Positive differences mean
    condition A caused a larger accuracy reduction than condition B.
    """
    if runs_raw.empty or metric not in runs_raw.columns:
        return pd.DataFrame(), pd.DataFrame()

    pair_strata = [
        column
        for column in [
            "candidate_config_id",
            "scoring_mode",
            "comparison_rule",
            "endpoint_mining_hop",
            "resampling_enabled",
            "block_size",
        ]
        if column in runs_raw.columns
    ]

    seed_condition = (
        runs_raw
        .assign(
            **{
                metric: pd.to_numeric(
                    runs_raw[metric],
                    errors="coerce",
                )
            }
        )
        .groupby(
            pair_strata + ["condition", "seed"],
            dropna=False,
        )[metric]
        .mean()
        .reset_index()
    )

    summary_rows = []
    difference_rows = []

    grouped = (
        [((), seed_condition)]
        if not pair_strata
        else seed_condition.groupby(
            pair_strata,
            dropna=False,
        )
    )

    for stratum_key, frame in grouped:
        if pair_strata:
            if not isinstance(stratum_key, tuple):
                stratum_key = (stratum_key,)
            stratum = dict(zip(pair_strata, stratum_key))
        else:
            stratum = {}

        pivot = frame.pivot_table(
            index="seed",
            columns="condition",
            values=metric,
            aggfunc="mean",
        )

        conditions = sorted(
            pivot.columns.astype(str),
            key=lambda value: (
                RQ2_CONDITION_ORDER_MAP.get(
                    value,
                    len(RQ2_CONDITION_ORDER_MAP),
                ),
                value,
            ),
        )

        for condition_a, condition_b in combinations(conditions, 2):
            differences = (
                pivot[condition_a]
                - pivot[condition_b]
            ).dropna()

            n_pairs = int(differences.size)
            mean_difference = (
                float(differences.mean())
                if n_pairs > 0
                else np.nan
            )
            std_difference = (
                float(differences.std(ddof=1))
                if n_pairs >= 2
                else np.nan
            )
            sem_difference = (
                std_difference / np.sqrt(n_pairs)
                if n_pairs >= 2
                else np.nan
            )

            if n_pairs >= 2:
                if student_t is not None:
                    critical = float(
                        student_t.ppf(
                            1.0 - (1.0 - RQ2_CONFIDENCE_LEVEL) / 2.0,
                            n_pairs - 1,
                        )
                    )
                else:
                    critical = NormalDist().inv_cdf(
                        1.0 - (1.0 - RQ2_CONFIDENCE_LEVEL) / 2.0
                    )
                ci_low = mean_difference - critical * sem_difference
                ci_high = mean_difference + critical * sem_difference
            else:
                ci_low = np.nan
                ci_high = np.nan

            if (
                ttest_1samp is not None
                and n_pairs >= 2
                and np.isfinite(std_difference)
                and std_difference > 0
            ):
                p_value = float(
                    ttest_1samp(
                        differences.to_numpy(dtype=float),
                        popmean=0.0,
                        nan_policy="omit",
                    ).pvalue
                )
            elif n_pairs >= 2 and std_difference == 0:
                p_value = 0.0 if mean_difference != 0 else 1.0
            else:
                p_value = np.nan

            cohen_dz = (
                mean_difference / std_difference
                if n_pairs >= 2
                and np.isfinite(std_difference)
                and std_difference > 0
                else np.nan
            )

            summary_rows.append({
                **stratum,
                "condition_a": condition_a,
                "condition_b": condition_b,
                "difference_definition": "A minus B",
                "n_paired_seeds": n_pairs,
                "final_accuracy_drop_difference_mean": mean_difference,
                "final_accuracy_drop_difference_std": std_difference,
                "final_accuracy_drop_difference_sem": sem_difference,
                "final_accuracy_drop_difference_ci_low": ci_low,
                "final_accuracy_drop_difference_ci_high": ci_high,
                "paired_t_test_p_value": p_value,
                "cohen_dz": cohen_dz,
            })

            for seed, difference in differences.items():
                difference_rows.append({
                    **stratum,
                    "condition_a": condition_a,
                    "condition_b": condition_b,
                    "seed": seed,
                    "final_accuracy_drop_difference": float(difference),
                })

    summary = pd.DataFrame(summary_rows)
    differences = pd.DataFrame(difference_rows)

    if not summary.empty:
        summary["paired_t_test_p_value_holm"] = _rq2_holm_adjust(
            summary["paired_t_test_p_value"]
        )

    return summary, differences


def _rq2_thesis_primary_table(summary):
    if summary.empty:
        return pd.DataFrame()

    table = summary.copy()

    scaled_columns = {
        "clean_accuracy_mean": "clean_accuracy_mean_percent",
        "clean_accuracy_std": "clean_accuracy_std_percent",
        "clean_accuracy_ci_low": "clean_accuracy_ci_low_percent",
        "clean_accuracy_ci_high": "clean_accuracy_ci_high_percent",
        "final_accuracy_drop_seed_baseline_mean": "final_drop_mean_pp",
        "final_accuracy_drop_seed_baseline_std": "final_drop_std_pp",
        "final_accuracy_drop_seed_baseline_ci_low": "final_drop_ci_low_pp",
        "final_accuracy_drop_seed_baseline_ci_high": "final_drop_ci_high_pp",
        "maximum_relaxed_accuracy_drop_mean": "maximum_drop_mean_pp",
        "maximum_relaxed_accuracy_drop_std": "maximum_drop_std_pp",
        "maximum_relaxed_accuracy_drop_ci_low": "maximum_drop_ci_low_pp",
        "maximum_relaxed_accuracy_drop_ci_high": "maximum_drop_ci_high_pp",
    }

    for source, destination in scaled_columns.items():
        if source in table.columns:
            table[destination] = (
                pd.to_numeric(table[source], errors="coerce")
                * RQ2_ACCURACY_SCALE
            )

    preferred = [
        "candidate_config_id",
        "scoring_mode",
        "comparison_rule",
        "endpoint_mining_hop",
        "condition",
        "initialization",
        "resampling_enabled",
        "block_size",
        "n_seeds",
        "final_accuracy_drop_seed_baseline_count",
        "clean_accuracy_mean_percent",
        "clean_accuracy_std_percent",
        "clean_accuracy_ci_low_percent",
        "clean_accuracy_ci_high_percent",
        "final_drop_mean_pp",
        "final_drop_std_pp",
        "final_drop_ci_low_pp",
        "final_drop_ci_high_pp",
        "maximum_drop_mean_pp",
        "maximum_drop_std_pp",
        "maximum_drop_ci_low_pp",
        "maximum_drop_ci_high_pp",
        "integrated_accuracy_drop_mean",
        "integrated_accuracy_drop_std",
        "runtime_seconds_mean",
        "runtime_seconds_std",
    ]

    return table[
        [column for column in preferred if column in table.columns]
    ]

# ============================================================
# Condition ordering and display labels
# ============================================================

RQ2_CONDITION_ORDER = [
    "label_positive__fixed_block",
    "label_positive__random_resampling",
    "label_zero__fixed_block",
    "label_zero__random_resampling",
    "score_top__fixed_block",
    "score_top__random_resampling",
    "score_bottom__fixed_block",
    "score_bottom__random_resampling",
    "random_full_space__fixed_block",
    "random_full_space__random_resampling",
]

RQ2_CONDITION_ORDER_MAP = {
    condition: index
    for index, condition
    in enumerate(
        RQ2_CONDITION_ORDER
    )
}

CONDITION_DISPLAY = {
    "label_positive__fixed_block": (
        "Endpoint label 1 — fixed"
    ),
    "label_positive__random_resampling": (
        "Endpoint label 1 — resampling"
    ),
    "label_zero__fixed_block": (
        "Endpoint label 0 — fixed"
    ),
    "label_zero__random_resampling": (
        "Endpoint label 0 — resampling"
    ),
    "score_top__fixed_block": (
        "Top score — fixed"
    ),
    "score_top__random_resampling": (
        "Top score — resampling"
    ),
    "score_bottom__fixed_block": (
        "Bottom score — fixed"
    ),
    "score_bottom__random_resampling": (
        "Bottom score — resampling"
    ),
    "random_full_space__fixed_block": (
        "Random full space — fixed"
    ),
    "random_full_space__random_resampling": (
        "Random full space — resampling"
    ),
}

plot_config_cols = [
    "candidate_config_id",
    "scoring_mode",
    "comparison_rule",
    "endpoint_mining_hop",
]


def _rq2_order_frame(
    frame,
):
    frame = frame.copy()

    frame[
        "_condition_order"
    ] = (
        frame["condition"]
        .map(
            RQ2_CONDITION_ORDER_MAP
        )
        .fillna(
            len(
                RQ2_CONDITION_ORDER_MAP
            )
        )
    )

    return frame.sort_values(
        [
            "_condition_order",
            "block_size",
            "condition",
        ]
    )


def _rq2_plot_label(
    row,
):
    condition = str(
        row["condition"]
    )

    display_name = (
        CONDITION_DISPLAY.get(
            condition,
            condition,
        )
    )

    block_size = row.get(
        "block_size",
        np.nan,
    )

    block_label = (
        "B=?"
        if pd.isna(
            block_size
        )
        else (
            f"B={int(round(float(block_size)))}"
        )
    )

    n_seeds = row.get(
        "n_seeds",
        np.nan,
    )

    if pd.isna(n_seeds):
        return (
            f"{display_name} "
            f"({block_label})"
        )

    return (
        f"{display_name} "
        f"({block_label}, "
        f"n={int(n_seeds)})"
    )


# ============================================================
# Build and export thesis reporting tables
# ============================================================

rq2_seed_level_runs_df = _rq2_seed_level_frame(
    rq2_block_runs_raw_df,
    rq2_group_cols,
    rq2_run_metrics,
)

rq2_thesis_primary_results_df = _rq2_thesis_primary_table(
    rq2_block_runs_summary_df
)

(
    rq2_block_paired_summary_df,
    rq2_block_paired_seed_differences_df,
) = _rq2_build_paired_comparisons(
    rq2_block_runs_raw_df,
)

if not rq2_block_paired_summary_df.empty:
    for column in [
        "final_accuracy_drop_difference_mean",
        "final_accuracy_drop_difference_std",
        "final_accuracy_drop_difference_sem",
        "final_accuracy_drop_difference_ci_low",
        "final_accuracy_drop_difference_ci_high",
    ]:
        rq2_block_paired_summary_df[
            f"{column}_pp"
        ] = (
            pd.to_numeric(
                rq2_block_paired_summary_df[column],
                errors="coerce",
            )
            * RQ2_ACCURACY_SCALE
        )

# Completeness is exported even for partial runs.
if not rq2_block_runs_raw_df.empty:
    rq2_run_completeness_df = (
        rq2_block_runs_raw_df
        .groupby(
            plot_config_cols + ["condition"],
            dropna=False,
        )
        .agg(
            completed_rows=("condition", "size"),
            completed_seeds=("seed", "nunique"),
            completed_repeats=("repeat", "nunique"),
        )
        .reset_index()
        .sort_values(plot_config_cols + ["condition"])
    )
else:
    rq2_run_completeness_df = pd.DataFrame()

report_exports = {
    "thesis_primary_results.csv": rq2_thesis_primary_results_df,
    "thesis_seed_level_results.csv": rq2_seed_level_runs_df,
    "thesis_paired_comparisons.csv": rq2_block_paired_summary_df,
    "thesis_paired_seed_differences.csv": rq2_block_paired_seed_differences_df,
    "thesis_run_completeness.csv": rq2_run_completeness_df,
}

for filename, frame in report_exports.items():
    if not frame.empty:
        frame.to_csv(
            RQ2_BLOCK_REPORT_DIR / filename,
            index=False,
        )

methodology_summary = {
    "primary_outcome": (
        "Final test-accuracy drop from the clean baseline belonging to the "
        "same seed, reported in percentage points."
    ),
    "inferential_unit": "seed",
    "repeat_handling": (
        "Repeated attack executions are averaged within seed before "
        "between-seed statistics are calculated."
    ),
    "variability": "Sample standard deviation across seed-level means.",
    "confidence_interval": (
        f"Two-sided {RQ2_CONFIDENCE_LEVEL:.0%} Student-t confidence interval "
        "across seed-level means; undefined for fewer than two seeds."
    ),
    "paired_comparisons": (
        "Conditions are paired by seed. Positive A-minus-B values mean "
        "condition A produced the stronger accuracy reduction."
    ),
    "multiple_testing": "Holm adjustment across exported paired comparisons.",
    "accuracy_units": (
        "Accuracies are shown as percentages; accuracy changes are shown "
        "as percentage points."
    ),
}

with (
    RQ2_BLOCK_REPORT_DIR / "thesis_reporting_method.json"
).open("w", encoding="utf-8") as file:
    json.dump(
        methodology_summary,
        file,
        indent=2,
    )

reporting_note = f"""Statistical reporting note
==========================

Primary outcome
---------------
Final test-accuracy reduction relative to the clean baseline of the same seed.
Accuracy reductions are reported in percentage points.

Aggregation
-----------
Repeated attack executions were first averaged within each seed. Means,
sample standard deviations, and two-sided {RQ2_CONFIDENCE_LEVEL:.0%} Student-t
confidence intervals were then calculated across seed-level means. The seed,
not the epoch or repeated attack execution, was treated as the independent
statistical unit. Standard deviations and confidence intervals are reported as
not estimable when fewer than two seeds are available.

Paired comparisons
------------------
Conditions were compared within the same seeds. The reported paired effect is
the mean seed-level difference in accuracy reduction (condition A minus
condition B), together with its standard deviation, {RQ2_CONFIDENCE_LEVEL:.0%}
confidence interval, paired t-test p-value, Holm-adjusted p-value, and Cohen's
dz. Positive differences indicate a stronger attack under condition A.

Figure conventions
------------------
Accuracy-drop trajectory bands and final-result error bars show
{RQ2_CONFIDENCE_LEVEL:.0%} confidence intervals. Final-result plots additionally
show the individual seed-level observations.
"""

(
    RQ2_BLOCK_REPORT_DIR / "thesis_reporting_note.txt"
).write_text(
    reporting_note,
    encoding="utf-8",
)

print("\nThesis-ready primary results")
if not rq2_thesis_primary_results_df.empty:
    display(rq2_thesis_primary_results_df)

print("\nThesis-ready paired comparisons")
if not rq2_block_paired_summary_df.empty:
    display(rq2_block_paired_summary_df)

# ============================================================
# Display partial-run completeness
# ============================================================

print(
    "\nAvailable completed conditions"
)

if not rq2_run_completeness_df.empty:
    display(
        rq2_run_completeness_df
    )

elif not rq2_block_runs_summary_df.empty:
    available_columns = (
        plot_config_cols
        + [
            "condition",
            "n_seeds",
        ]
    )

    available_columns = [
        column
        for column in available_columns
        if column
        in rq2_block_runs_summary_df.columns
    ]

    display(
        rq2_block_runs_summary_df[
            available_columns
        ].sort_values(
            [
                column
                for column in (
                    plot_config_cols
                    + ["condition"]
                )
                if column
                in available_columns
            ]
        )
    )


# ============================================================
# 1. Accuracy trajectories
# ============================================================

required_accuracy_columns = {
    "accuracy_drop_seed_baseline_mean",
    "accuracy_drop_seed_baseline_ci_low",
    "accuracy_drop_seed_baseline_ci_high",
    "prbcd_epoch",
    "condition",
    "block_size",
}

if (
    not rq2_block_epochs_summary_df.empty
    and required_accuracy_columns.issubset(
        rq2_block_epochs_summary_df.columns
    )
):
    for (
        config_key,
        frame,
    ) in (
        rq2_block_epochs_summary_df
        .groupby(
            plot_config_cols,
            dropna=False,
        )
    ):
        frame = (
            _rq2_order_frame(
                frame
            )
        )

        fig, ax = plt.subplots(
            figsize=(12, 6)
        )

        plotted = 0

        for (
            condition,
            condition_df,
        ) in frame.groupby(
            "condition",
            sort=False,
        ):
            condition_df = (
                condition_df
                .sort_values(
                    "prbcd_epoch"
                )
            )

            representative_row = (
                condition_df.iloc[0]
            )

            plotted += int(
                _rq2_add_mean_ci_band(
                    ax,
                    condition_df["prbcd_epoch"],
                    condition_df["accuracy_drop_seed_baseline_mean"],
                    condition_df["accuracy_drop_seed_baseline_ci_low"],
                    condition_df["accuracy_drop_seed_baseline_ci_high"],
                    label=_rq2_plot_label(representative_row),
                    scale=RQ2_ACCURACY_SCALE,
                )
            )

        if plotted == 0:
            plt.close(fig)
            continue

        ax.axvline(
            resampling_boundary - 1,
            linestyle="--",
            linewidth=0.9,
            label=(
                "fine-tuning boundary"
            ),
        )

        ax.set_xlabel(
            "PRBCD epoch "
            "(-1 = clean baseline)"
        )

        ax.axhline(
            0.0,
            linewidth=0.9,
        )

        ax.set_ylabel(
            "accuracy drop from seed-specific baseline (percentage points)"
        )

        ax.set_title(
            f"Accuracy-drop trajectories (mean and {RQ2_CONFIDENCE_LEVEL:.0%} CI)"
        )

        ax.grid(
            True,
            alpha=0.3,
        )

        ax.legend(
            bbox_to_anchor=(
                1.02,
                1,
            ),
            loc="upper left",
        )

        fig.tight_layout()

        output_path = (
            RQ2_BLOCK_PLOT_DIR
            / (
                "accuracy_drop_partial__"
                f"{_rq2_safe('__'.join(map(str, config_key)))}"
                ".png"
            )
        )

        fig.savefig(
            output_path,
            dpi=200,
            bbox_inches="tight",
        )

        plt.show()

else:
    print(
        "\nSkipping accuracy trajectories: "
        "no epoch summary data are available."
    )


# ============================================================
# 2. Final accuracy drop: raw seeds, mean, and confidence interval
# ============================================================

required_final_columns = {
    "final_accuracy_drop_seed_baseline_mean",
    "final_accuracy_drop_seed_baseline_ci_low",
    "final_accuracy_drop_seed_baseline_ci_high",
    "condition",
    "block_size",
}

if (
    not rq2_block_runs_summary_df.empty
    and required_final_columns.issubset(
        rq2_block_runs_summary_df.columns
    )
):
    for config_key, frame in (
        rq2_block_runs_summary_df
        .groupby(
            plot_config_cols,
            dropna=False,
        )
    ):
        frame = _rq2_order_frame(frame)
        frame = frame.dropna(
            subset=["final_accuracy_drop_seed_baseline_mean"]
        )

        if frame.empty:
            continue

        labels = [
            _rq2_plot_label(row)
            for _, row in frame.iterrows()
        ]
        x = np.arange(len(frame))

        fig, ax = plt.subplots(figsize=(12, 5.8))

        # Individual seed-level observations. Repeats have already been
        # averaged within seed in rq2_seed_level_runs_df.
        if not rq2_seed_level_runs_df.empty:
            seed_config = rq2_seed_level_runs_df.copy()
            for column, value in zip(plot_config_cols, config_key):
                if pd.isna(value):
                    seed_config = seed_config[seed_config[column].isna()]
                else:
                    seed_config = seed_config[seed_config[column] == value]

            for position, (_, row) in enumerate(frame.iterrows()):
                points = seed_config[
                    seed_config["condition"].astype(str)
                    == str(row["condition"])
                ].copy()

                if "block_size" in points.columns and pd.notna(row["block_size"]):
                    points = points[
                        np.isclose(
                            pd.to_numeric(points["block_size"], errors="coerce"),
                            float(row["block_size"]),
                            equal_nan=False,
                        )
                    ]

                values = (
                    pd.to_numeric(
                        points["final_accuracy_drop_seed_baseline"],
                        errors="coerce",
                    )
                    .dropna()
                    .to_numpy(dtype=float)
                    * RQ2_ACCURACY_SCALE
                )

                if values.size:
                    offsets = np.linspace(-0.09, 0.09, values.size)
                    ax.scatter(
                        position + offsets,
                        values,
                        alpha=0.58,
                        s=28,
                        zorder=2,
                    )

        means = (
            pd.to_numeric(
                frame["final_accuracy_drop_seed_baseline_mean"],
                errors="coerce",
            ).to_numpy(dtype=float)
            * RQ2_ACCURACY_SCALE
        )
        errors = _rq2_ci_yerr(
            frame["final_accuracy_drop_seed_baseline_mean"],
            frame["final_accuracy_drop_seed_baseline_ci_low"],
            frame["final_accuracy_drop_seed_baseline_ci_high"],
            scale=RQ2_ACCURACY_SCALE,
        )

        ax.errorbar(
            x,
            means,
            yerr=errors,
            marker="D",
            linestyle="none",
            capsize=5,
            markersize=6,
            linewidth=1.4,
            zorder=3,
            label=(
                f"mean and {RQ2_CONFIDENCE_LEVEL:.0%} CI; "
                "circles = seed-level results"
            ),
        )

        ax.axhline(0.0, linewidth=0.9)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=28, ha="right")
        ax.set_ylabel(
            "seed-specific clean accuracy minus final accuracy "
            "(percentage points)"
        )
        ax.set_title(
            "Final PRBCD accuracy reduction"
        )
        ax.grid(True, axis="y", alpha=0.3)
        ax.legend()
        fig.tight_layout()

        output_path = (
            RQ2_BLOCK_PLOT_DIR
            / (
                "final_accuracy_drop_thesis__"
                f"{_rq2_safe('__'.join(map(str, config_key)))}"
                ".png"
            )
        )
        fig.savefig(output_path, dpi=200, bbox_inches="tight")
        plt.show()
else:
    print(
        "\nSkipping final accuracy-drop plots: "
        "no completed run-level results are available."
    )


# ============================================================
# 3. Evolution of current search block
# ============================================================

retention_columns = {
    "condition",
    "block_size",
    "prbcd_epoch",
    "resampling_enabled",
    "initial_retention_mean",
    "initial_retention_ci_low",
    "initial_retention_ci_high",
    "group_high_fraction_in_current_block_mean",
    "group_high_fraction_in_current_block_ci_low",
    "group_high_fraction_in_current_block_ci_high",
    "group_low_fraction_in_current_block_mean",
    "group_low_fraction_in_current_block_ci_low",
    "group_low_fraction_in_current_block_ci_high",
}

if (
    not rq2_block_retention_summary_df.empty
    and retention_columns.issubset(
        rq2_block_retention_summary_df.columns
    )
):
    for (
        config_key,
        frame,
    ) in (
        rq2_block_retention_summary_df
        .groupby(
            plot_config_cols,
            dropna=False,
        )
    ):
        frame = frame[
            _rq2_is_true(
                frame[
                    "resampling_enabled"
                ]
            )
        ].copy()

        if frame.empty:
            continue

        frame = (
            _rq2_order_frame(
                frame
            )
        )

        high_label, low_label = (
            _mode_group_labels(
                config_key[1],
                config_key[2],
            )
        )

        fig, axes = plt.subplots(
            1,
            3,
            figsize=(18, 4.8),
        )

        plotted = 0

        for (
            condition,
            condition_df,
        ) in frame.groupby(
            "condition",
            sort=False,
        ):
            condition_df = (
                condition_df
                .sort_values(
                    "prbcd_epoch"
                )
            )

            label = (
                _rq2_plot_label(
                    condition_df.iloc[0]
                )
            )

            plotted += int(
                _rq2_add_mean_ci_band(
                    axes[0],
                    condition_df["prbcd_epoch"],
                    condition_df["initial_retention_mean"],
                    condition_df["initial_retention_ci_low"],
                    condition_df["initial_retention_ci_high"],
                    label=label,
                )
            )

            _rq2_add_mean_ci_band(
                axes[1],
                condition_df["prbcd_epoch"],
                condition_df["group_high_fraction_in_current_block_mean"],
                condition_df["group_high_fraction_in_current_block_ci_low"],
                condition_df["group_high_fraction_in_current_block_ci_high"],
                label=label,
            )

            _rq2_add_mean_ci_band(
                axes[2],
                condition_df["prbcd_epoch"],
                condition_df["group_low_fraction_in_current_block_mean"],
                condition_df["group_low_fraction_in_current_block_ci_low"],
                condition_df["group_low_fraction_in_current_block_ci_high"],
                label=label,
            )

        if plotted == 0:
            plt.close(fig)
            continue

        axes[0].set_title(
            "Fraction of initial block retained"
        )

        axes[1].set_title(
            f"{high_label} fraction "
            "in current block"
        )

        axes[2].set_title(
            f"{low_label} fraction "
            "in current block"
        )

        for ax in axes:
            ax.set_xlabel(
                "PRBCD epoch"
            )
            ax.set_ylim(
                -0.02,
                1.02,
            )
            ax.grid(
                True,
                alpha=0.3,
            )
            ax.legend(
                fontsize=8,
            )

        fig.suptitle(
            f"Block evolution (mean and {RQ2_CONFIDENCE_LEVEL:.0%} CI)"
        )

        fig.tight_layout()

        output_path = (
            RQ2_BLOCK_PLOT_DIR
            / (
                "retention_partial__"
                f"{_rq2_safe('__'.join(map(str, config_key)))}"
                ".png"
            )
        )

        fig.savefig(
            output_path,
            dpi=200,
            bbox_inches="tight",
        )

        plt.show()

else:
    print(
        "\nSkipping retention plots: "
        "no block-retention data are available."
    )


# ============================================================
# 4. Composition of final perturbation set
# ============================================================

composition_columns = {
    "condition",
    "block_size",
    "fraction_final_selected_group_high_mean",
    "fraction_final_selected_group_high_ci_low",
    "fraction_final_selected_group_high_ci_high",
    "fraction_final_selected_group_low_mean",
    "fraction_final_selected_group_low_ci_low",
    "fraction_final_selected_group_low_ci_high",
}

if (
    not rq2_block_runs_summary_df.empty
    and composition_columns.issubset(
        rq2_block_runs_summary_df.columns
    )
):
    for (
        config_key,
        frame,
    ) in (
        rq2_block_runs_summary_df
        .groupby(
            plot_config_cols,
            dropna=False,
        )
    ):
        frame = (
            _rq2_order_frame(
                frame
            )
        )

        frame = frame.dropna(
            subset=[
                "fraction_final_selected_group_high_mean",
                "fraction_final_selected_group_low_mean",
            ],
            how="all",
        )

        if frame.empty:
            continue

        high_label, low_label = (
            _mode_group_labels(
                config_key[1],
                config_key[2],
            )
        )

        labels = [
            _rq2_plot_label(row)
            for _, row
            in frame.iterrows()
        ]

        x = np.arange(
            len(frame)
        )

        fig, ax = plt.subplots(
            figsize=(12, 5.2)
        )

        ax.errorbar(
            x - 0.08,
            frame[
                "fraction_final_selected_group_high_mean"
            ],
            yerr=_rq2_ci_yerr(
                frame["fraction_final_selected_group_high_mean"],
                frame["fraction_final_selected_group_high_ci_low"],
                frame["fraction_final_selected_group_high_ci_high"],
            ),
            marker="o",
            linestyle="none",
            capsize=4,
            label=(
                f"final fraction: {high_label}"
            ),
        )

        ax.errorbar(
            x + 0.08,
            frame[
                "fraction_final_selected_group_low_mean"
            ],
            yerr=_rq2_ci_yerr(
                frame["fraction_final_selected_group_low_mean"],
                frame["fraction_final_selected_group_low_ci_low"],
                frame["fraction_final_selected_group_low_ci_high"],
            ),
            marker="s",
            linestyle="none",
            capsize=4,
            label=(
                f"final fraction: {low_label}"
            ),
        )

        ax.set_xticks(x)

        ax.set_xticklabels(
            labels,
            rotation=28,
            ha="right",
        )

        ax.set_ylim(
            -0.02,
            1.02,
        )

        ax.set_ylabel(
            "fraction of final selected perturbations"
        )

        ax.set_title(
            f"Final perturbation composition (mean and {RQ2_CONFIDENCE_LEVEL:.0%} CI)"
        )

        ax.grid(
            True,
            axis="y",
            alpha=0.3,
        )

        ax.legend()

        fig.tight_layout()

        output_path = (
            RQ2_BLOCK_PLOT_DIR
            / (
                "final_composition_partial__"
                f"{_rq2_safe('__'.join(map(str, config_key)))}"
                ".png"
            )
        )

        fig.savefig(
            output_path,
            dpi=200,
            bbox_inches="tight",
        )

        plt.show()

else:
    print(
        "\nSkipping composition plots: "
        "no final-composition results are available."
    )


# ============================================================
# 5. Paired condition differences
# ============================================================

if not rq2_block_paired_summary_df.empty:
    print("\nPaired condition differences")
    display(rq2_block_paired_summary_df)

    required_paired = {
        "condition_a",
        "condition_b",
        "final_accuracy_drop_difference_mean",
        "final_accuracy_drop_difference_ci_low",
        "final_accuracy_drop_difference_ci_high",
    }

    if required_paired.issubset(rq2_block_paired_summary_df.columns):
        paired_plot_config_cols = [
            column
            for column in plot_config_cols
            if column in rq2_block_paired_summary_df.columns
        ]

        grouped = (
            [((), rq2_block_paired_summary_df)]
            if not paired_plot_config_cols
            else rq2_block_paired_summary_df.groupby(
                paired_plot_config_cols,
                dropna=False,
            )
        )

        for config_key, frame in grouped:
            frame = frame.copy()
            if not isinstance(config_key, tuple):
                config_key = (config_key,)

            frame["comparison"] = (
                frame["condition_a"].astype(str)
                + " − "
                + frame["condition_b"].astype(str)
                + np.where(
                    _rq2_is_true(frame["resampling_enabled"])
                    if "resampling_enabled" in frame.columns
                    else False,
                    " | resampling",
                    " | fixed",
                )
                + " | n="
                + frame["n_paired_seeds"].astype(str)
            )

            x = np.arange(len(frame))
            means = (
                pd.to_numeric(
                    frame["final_accuracy_drop_difference_mean"],
                    errors="coerce",
                ).to_numpy(dtype=float)
                * RQ2_ACCURACY_SCALE
            )
            errors = _rq2_ci_yerr(
                frame["final_accuracy_drop_difference_mean"],
                frame["final_accuracy_drop_difference_ci_low"],
                frame["final_accuracy_drop_difference_ci_high"],
                scale=RQ2_ACCURACY_SCALE,
            )

            fig, ax = plt.subplots(
                figsize=(
                    11,
                    max(4.5, 0.6 * len(frame)),
                )
            )

            ax.errorbar(
                means,
                x,
                xerr=errors,
                marker="o",
                linestyle="none",
                capsize=4,
            )
            ax.axvline(0.0, linewidth=0.9)
            ax.set_yticks(x)
            ax.set_yticklabels(frame["comparison"])
            ax.set_xlabel(
                "paired difference in final accuracy reduction "
                "(percentage points; positive = condition A is stronger)"
            )

            if len(config_key) == 4:
                description = _config_description(config_key)
            else:
                description = "paired available configurations"

            ax.set_title(
                f"Paired condition differences (mean and "
                f"{RQ2_CONFIDENCE_LEVEL:.0%} CI)"
            )
            ax.grid(True, axis="x", alpha=0.3)
            fig.tight_layout()

            output_path = (
                RQ2_BLOCK_PLOT_DIR
                / (
                    "paired_differences_thesis__"
                    f"{_rq2_safe('__'.join(map(str, config_key)))}"
                    ".png"
                )
            )
            fig.savefig(output_path, dpi=200, bbox_inches="tight")
            plt.show()


print()

print(
    "Plots saved to:"
)

print(
    RQ2_BLOCK_PLOT_DIR.resolve()
)

print()
print(
    "Thesis tables and reporting notes saved to:"
)
print(
    RQ2_BLOCK_REPORT_DIR.resolve()
)


---

# RQ2 Synthesis — From Individual Harmfulness to PR-BCD Usefulness

The RQ2 experiments investigate progressively stronger notions of harmfulness.

## 1. Individual harmfulness

Direct mining determines whether an edge causes observable damage when evaluated individually.

Endpoint mining captures local harmfulness:

\[
e
\rightarrow
H_{\mathrm{loc}}(e;G).
\]

Accuracy-drop mining captures an individual global effect:

\[
e
\rightarrow
A(G)-A(G\oplus e).
\]

## 2. Conditional harmfulness under joint perturbations

The oracle experiment determines whether mined candidates continue to contribute damage after other perturbations have already been applied:

\[
e
\rightarrow
H_{\mathrm{cond}}(e\mid B;G).
\]

This separates conditionally harmful, neutral, and cancelling additions.

## 3. Conditional harmfulness during optimization

The injection experiment evaluates whether a mined candidate remains useful inside an evolving PR-BCD state:

\[
e
\rightarrow
H_{\mathrm{inj}}(e\mid S_t).
\]

This measures harmfulness under the current block composition, relaxed weights, and optimization history.

## 4. Candidate-space usefulness

The balanced initialization experiment evaluates whether local harmfulness information improves the restricted PR-BCD search space:

\[
H_{\mathrm{loc}}
\rightarrow
\text{label-informed block}
\rightarrow
\text{PR-BCD attack effectiveness}.
\]

Together, the experiments test the chain

\[
\boxed{
\text{individually harmful}
\rightarrow
\text{conditionally harmful}
\rightarrow
\text{useful as a PR-BCD candidate}
}
\]

The results also decompose candidate-block quality into two components:

\[
\boxed{
\text{candidate coverage}
+
\text{candidate compatibility}
}
\]

Endpoint label-1 initialization primarily addresses candidate coverage. The oracle and injection experiments examine whether the selected candidates remain compatible and harmful under joint or algorithmic conditions.

## Construct and persist the RQ3 source-subgraph candidate-mining sweep

This block now uses two independent cache layers:

1. **Candidate-pool cache:** stores the selected source nodes and the maximum
   sampled candidate pool once per victim seed, subgraph method, subgraph seed,
   and subgraph fraction.
2. **Label cache:** stores the expensive victim-query labels once per exact
   candidate size and scoring mode.

The sweep is resumable at configuration level: existing artifacts are loaded
individually, and only missing or explicitly invalidated configurations are
computed again. Progress is written continuously to a CSV manifest.


### Whole-graph selector condition (`subgraph_fraction = 1.0`)

When the requested fraction is `1.0`, the pipeline uses every graph node and every structural edge for selector message passing. Candidate pairs are still sampled from the complete upper-triangular pair space and labeled with the same resumable per-configuration cache. `subgraph_seed` remains useful as an independent candidate-pool sampling replicate; the graph itself is not resampled.


In [ ]:
# ============================================================
# RQ3: deterministic source-subgraph construction + cached labeling
#
# Stage A: construct source subgraphs only.
# Stage B: sample candidates, load cached labels or label and save them.
#
# Supported source-graph methods:
#   - "stratified_context": task-aware stratified core-node selection,
#                           budgeted neighborhood context, and induction
#   - "forest_fire"       : Forest Fire baseline followed by graph induction
#   - fraction 1.0         : canonical "whole_graph" condition using every node
#
# For stratified_context, source nodes are ordered as
#   [core candidate-endpoint nodes | context-only nodes].
# Candidate pairs are sampled only from the core-node prefix, while the GCN
# and local victim see the complete induced graph on all source nodes.
#
# Produces every combination of:
#   victim seed × subgraph method × subgraph seed × subgraph fraction
#   × training candidate size × scoring mode
#
# Important isolation rule:
#   - RQ3_SUBGRAPH_MINING_RUNS contains the complete RQ3 sweep.
#   - `mining_runs` remains the original RQ2 full-graph collection.
# ============================================================

from collections import defaultdict, deque
import copy
from datetime import datetime, timezone
import hashlib
import json
import math
import os
from pathlib import Path
import random
import warnings

import networkx as nx
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from helpers.selector_pipeline_helpers import (
    EndpointPRBCDV4Scorer,
    mine_candidate_edge_scores,
    _dense_adj,
)


# ------------------------------------------------------------
# User-facing configuration
# ------------------------------------------------------------

# Select the task-aware method and, optionally, the Forest Fire baseline.
_raw_methods = globals().get(
    "RQ3_SUBGRAPH_METHODS",
    globals().get(
        "RQ3_SUBGRAPH_METHOD",
        ["stratified_context", "forest_fire"],
    ),
)
if isinstance(_raw_methods, str):
    _raw_methods = [_raw_methods]

RQ3_SUBGRAPH_METHODS = list(_raw_methods)
RQ3_PRIMARY_SUBGRAPH_METHOD = str(
    globals().get(
        "RQ3_PRIMARY_SUBGRAPH_METHOD",
        "stratified_context",
    )
)

# Task-aware sampler configuration.
# The core fraction is a lower bound. It is automatically increased when more
# core nodes are required to provide the largest configured candidate pool.
RQ3_STRATIFIED_CORE_SHARE = float(
    globals().get("RQ3_STRATIFIED_CORE_SHARE", 0.50)
)
RQ3_STRATIFIED_CONTEXT_HOPS = int(
    globals().get("RQ3_STRATIFIED_CONTEXT_HOPS", 2)
)
RQ3_STRATIFIED_DEGREE_BINS = int(
    globals().get("RQ3_STRATIFIED_DEGREE_BINS", 3)
)
RQ3_STRATIFIED_CORE_BINS = int(
    globals().get("RQ3_STRATIFIED_CORE_BINS", 3)
)
RQ3_STRATIFIED_MAX_COMMUNITIES = int(
    globals().get("RQ3_STRATIFIED_MAX_COMMUNITIES", 12)
)
RQ3_STRATIFIED_COMMUNITY_SEED = int(
    globals().get("RQ3_STRATIFIED_COMMUNITY_SEED", 0)
)

# Forest Fire baseline parameter.
RQ3_FOREST_FIRE_BURN_PROBABILITY = float(
    globals().get("RQ3_FOREST_FIRE_BURN_PROBABILITY", 0.7)
)

# Persistent RQ3 mining artifacts.
#
# Layer 1 stores the selected source nodes and the maximum sampled candidate
# pool for one victim/method/subgraph-seed/fraction configuration. This means
# rerunning the notebook does not resample the candidate block.
#
# Layer 2 stores the expensive victim-query labels separately for every exact
# candidate-size/scoring-mode configuration. Existing files are loaded one by
# one, so an interrupted sweep resumes from the first missing configuration.
RQ3_SUBGRAPH_BUNDLE_CACHE_DIR = Path(
    globals().get(
        "RQ3_SUBGRAPH_BUNDLE_CACHE_DIR",
        "cache/rq3_subgraph_candidate_pools",
    )
)
RQ3_LABEL_CACHE_DIR = Path(
    globals().get(
        "RQ3_LABEL_CACHE_DIR",
        "cache/rq3_subgraph_candidate_labels",
    )
)
RQ3_MINING_MANIFEST_DIR = Path(
    globals().get(
        "RQ3_MINING_MANIFEST_DIR",
        "extendedPlotting/rq3_subgraph_candidate_mining",
    )
)

RQ3_FORCE_RESAMPLE_CANDIDATES = bool(
    globals().get("RQ3_FORCE_RESAMPLE_CANDIDATES", False)
)
RQ3_FORCE_RELABEL = bool(globals().get("RQ3_FORCE_RELABEL", False))

RQ3_SUBGRAPH_BUNDLE_CACHE_VERSION = str(
    globals().get(
        "RQ3_SUBGRAPH_BUNDLE_CACHE_VERSION",
        "rq3-subgraph-candidate-pool-v2-core-context",
    )
)

# Increment this value whenever the meaning/schema of cached labels changes.
RQ3_LABEL_CACHE_VERSION = str(
    globals().get("RQ3_LABEL_CACHE_VERSION", "rq3-label-cache-v3-core-context")
)

RQ3_SUBGRAPH_CONSTRUCTION_SUMMARY_CSV = (
    RQ3_MINING_MANIFEST_DIR / "subgraph_construction_summary.csv"
)
RQ3_LABEL_MANIFEST_CSV = (
    RQ3_MINING_MANIFEST_DIR / "candidate_label_manifest.csv"
)


# ------------------------------------------------------------
# Generic helpers and configuration validation
# ------------------------------------------------------------

def _unique_preserve_order(values):
    output = []
    seen = set()
    for value in values:
        key = value.item() if isinstance(value, np.generic) else value
        if key not in seen:
            seen.add(key)
            output.append(key)
    return output


def _normalise_subgraph_method(method: str) -> str:
    method = str(method).strip().lower().replace("-", "_")
    aliases = {
        "stratified": "stratified_context",
        "task_aware": "stratified_context",
        "core_context": "stratified_context",
        "stratified_core_context": "stratified_context",
        "forestfire": "forest_fire",
        "ff": "forest_fire",
    }
    return aliases.get(method, method)


RQ3_SUBGRAPH_FRACTIONS = [
    float(value)
    for value in _unique_preserve_order(RQ3_SUBGRAPH_FRACTIONS)
]
RQ3_SUBGRAPH_TRAINING_CANDIDATE_SIZES = [
    int(value)
    for value in _unique_preserve_order(
        RQ3_SUBGRAPH_TRAINING_CANDIDATE_SIZES
    )
]
RQ3_SUBGRAPH_SEEDS = [
    int(value)
    for value in _unique_preserve_order(RQ3_SUBGRAPH_SEEDS)
]
RQ3_SUBGRAPH_SCORING_MODES = [
    str(value)
    for value in _unique_preserve_order(RQ3_SUBGRAPH_SCORING_MODES)
]
RQ3_SUBGRAPH_METHODS = [
    _normalise_subgraph_method(value)
    for value in _unique_preserve_order(RQ3_SUBGRAPH_METHODS)
]
RQ3_PRIMARY_SUBGRAPH_METHOD = _normalise_subgraph_method(
    RQ3_PRIMARY_SUBGRAPH_METHOD
)
RQ3_PRIMARY_EFFECTIVE_SUBGRAPH_METHOD = (
    "whole_graph"
    if np.isclose(float(RQ3_PRIMARY_SUBGRAPH_FRACTION), 1.0)
    else RQ3_PRIMARY_SUBGRAPH_METHOD
)

if not RQ3_SUBGRAPH_FRACTIONS:
    raise ValueError("RQ3_SUBGRAPH_FRACTIONS must not be empty.")
if not RQ3_SUBGRAPH_TRAINING_CANDIDATE_SIZES:
    raise ValueError(
        "RQ3_SUBGRAPH_TRAINING_CANDIDATE_SIZES must not be empty."
    )
if not RQ3_SUBGRAPH_SEEDS:
    raise ValueError("RQ3_SUBGRAPH_SEEDS must not be empty.")
if not RQ3_SUBGRAPH_SCORING_MODES:
    raise ValueError("RQ3_SUBGRAPH_SCORING_MODES must not be empty.")
if not RQ3_SUBGRAPH_METHODS:
    raise ValueError("RQ3_SUBGRAPH_METHODS must not be empty.")

supported_methods = {"stratified_context", "forest_fire"}
unsupported_methods = sorted(
    set(RQ3_SUBGRAPH_METHODS) - supported_methods
)
if unsupported_methods:
    raise ValueError(
        f"Unsupported RQ3 subgraph methods: {unsupported_methods}. "
        f"Supported methods are {sorted(supported_methods)}."
    )

for fraction in RQ3_SUBGRAPH_FRACTIONS:
    if not 0.0 < fraction <= 1.0:
        raise ValueError(
            "Every RQ3_SUBGRAPH_FRACTIONS value must be in (0, 1]. "
            "Use 1.0 for the whole-graph selector condition. "
            f"Found {fraction}."
        )

for size in RQ3_SUBGRAPH_TRAINING_CANDIDATE_SIZES:
    if size <= 0:
        raise ValueError(
            "Every RQ3_SUBGRAPH_TRAINING_CANDIDATE_SIZES value "
            f"must be positive. Found {size}."
        )

unsupported_modes = sorted(
    set(RQ3_SUBGRAPH_SCORING_MODES)
    - {"endpoint", "subset_accuracy_drop"}
)
if unsupported_modes:
    raise ValueError(
        "This cell supports endpoint and subset_accuracy_drop only. "
        f"Unsupported modes: {unsupported_modes}."
    )

if not 0.0 < RQ3_FOREST_FIRE_BURN_PROBABILITY < 1.0:
    raise ValueError(
        "RQ3_FOREST_FIRE_BURN_PROBABILITY must be in (0, 1)."
    )

if not 0.0 < RQ3_STRATIFIED_CORE_SHARE <= 1.0:
    raise ValueError("RQ3_STRATIFIED_CORE_SHARE must be in (0, 1].")
if RQ3_STRATIFIED_CONTEXT_HOPS < 0:
    raise ValueError("RQ3_STRATIFIED_CONTEXT_HOPS must be non-negative.")
if RQ3_STRATIFIED_DEGREE_BINS <= 0:
    raise ValueError("RQ3_STRATIFIED_DEGREE_BINS must be positive.")
if RQ3_STRATIFIED_CORE_BINS <= 0:
    raise ValueError("RQ3_STRATIFIED_CORE_BINS must be positive.")
if RQ3_STRATIFIED_MAX_COMMUNITIES <= 0:
    raise ValueError("RQ3_STRATIFIED_MAX_COMMUNITIES must be positive.")

if (
    not np.isclose(float(RQ3_PRIMARY_SUBGRAPH_FRACTION), 1.0)
    and RQ3_PRIMARY_SUBGRAPH_METHOD not in RQ3_SUBGRAPH_METHODS
):
    raise ValueError(
        "RQ3_PRIMARY_SUBGRAPH_METHOD must be contained in "
        "RQ3_SUBGRAPH_METHODS unless the primary fraction is 1.0."
    )
if not any(
    np.isclose(float(value), float(RQ3_PRIMARY_SUBGRAPH_FRACTION))
    for value in RQ3_SUBGRAPH_FRACTIONS
):
    raise ValueError(
        "RQ3_PRIMARY_SUBGRAPH_FRACTION must be contained in "
        "RQ3_SUBGRAPH_FRACTIONS."
    )
if int(RQ3_PRIMARY_TRAINING_CANDIDATE_SIZE) not in (
    RQ3_SUBGRAPH_TRAINING_CANDIDATE_SIZES
):
    raise ValueError(
        "RQ3_PRIMARY_TRAINING_CANDIDATE_SIZE must be contained in "
        "RQ3_SUBGRAPH_TRAINING_CANDIDATE_SIZES."
    )
if int(RQ3_PRIMARY_SUBGRAPH_SEED) not in RQ3_SUBGRAPH_SEEDS:
    raise ValueError(
        "RQ3_PRIMARY_SUBGRAPH_SEED must be contained in "
        "RQ3_SUBGRAPH_SEEDS."
    )

# Refresh the RQ2 snapshot on every execution.  In a notebook, guarding this
# assignment with ``if ... not in globals()`` can retain a stale/empty snapshot
# from an earlier failed RQ3 execution even though ``mining_runs`` now contains
# the successfully loaded RQ2 runs.
RQ2_FULL_GRAPH_MINING_RUNS = list(globals().get("mining_runs", []))
if not RQ2_FULL_GRAPH_MINING_RUNS:
    raise RuntimeError(
        "RQ2 full-graph reuse was requested, but `mining_runs` is empty. "
        "Execute the RQ2 candidate-mining cell before this RQ3 cell."
    )

RQ3_SUBGRAPH_BUNDLE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
RQ3_LABEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
RQ3_MINING_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)


def _rq3_safe_value(value):
    helper = globals().get("_safe_config_value")
    if callable(helper):
        return helper(value)
    return str(value).replace(".", "p").replace("-", "m")


def _rq3_stable_seed(*parts) -> int:
    payload = "|".join(str(part) for part in parts).encode("utf-8")
    return int.from_bytes(
        hashlib.sha256(payload).digest()[:4],
        byteorder="little",
        signed=False,
    )


def _rq3_as_numpy(value, dtype=None) -> np.ndarray:
    if torch.is_tensor(value):
        value = value.detach().cpu().numpy()
    return np.asarray(value, dtype=dtype)


# ------------------------------------------------------------
# Reuse RQ2 labels for the canonical whole-graph condition
# ------------------------------------------------------------

# When enabled, fraction=1.0 first looks for an exact RQ2 mining run whose
# labels were loaded from / written to the RQ2 cache. Reduced subgraphs keep
# using the ordinary RQ3 cache and local victim-query path.
RQ3_REUSE_RQ2_WHOLE_GRAPH = bool(
    globals().get("RQ3_REUSE_RQ2_WHOLE_GRAPH", True)
)

# Optional disambiguation when RQ2 contains multiple candidate configurations
# with the same candidate-set size. Accepted forms:
#   None
#   "candN-10000__prbcdFrac-0"
#   {10000: "candN-10000__prbcdFrac-0"}
RQ3_RQ2_CANDIDATE_CONFIG_ID = globals().get(
    "RQ3_RQ2_CANDIDATE_CONFIG_ID",
    None,
)

# Optional endpoint-hop selector for reused RQ2 endpoint labels.
# Leave as None to infer the hop automatically when exactly one matching RQ2
# endpoint run exists. Set an integer (for example 2) if RQ2 contains several
# endpoint-hop variants for the same seed/configuration/candidate size.
RQ3_RQ2_ENDPOINT_MINING_HOP = globals().get(
    "RQ3_RQ2_ENDPOINT_MINING_HOP",
    None,
)
if RQ3_RQ2_ENDPOINT_MINING_HOP is not None:
    RQ3_RQ2_ENDPOINT_MINING_HOP = int(
        RQ3_RQ2_ENDPOINT_MINING_HOP
    )

# True means that the RQ2 run must point to an existing .pt cache file.
# Set to False only when reusing an RQ2 result computed in the current kernel
# before it has been persisted.
RQ3_REQUIRE_RQ2_DISK_CACHE = bool(
    globals().get("RQ3_REQUIRE_RQ2_DISK_CACHE", True)
)


def _rq3_requested_rq2_candidate_config_id(
    training_candidate_size: int,
):
    configured = RQ3_RQ2_CANDIDATE_CONFIG_ID
    if isinstance(configured, dict):
        return configured.get(
            int(training_candidate_size),
            configured.get(str(int(training_candidate_size))),
        )
    return configured


def _rq3_rq2_cache_path(run: dict) -> Path | None:
    """Resolve the RQ2 cache path without assuming one working directory."""
    raw_path = str(run.get("cache_path", "")).strip()
    if not raw_path:
        return None

    path = Path(raw_path).expanduser()
    candidates = [path]
    if not path.is_absolute():
        candidates.extend([
            Path.cwd() / path,
            Path.cwd().parent / path,
        ])

    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()

    # Retain the original path for metadata/debug output even if it cannot be
    # resolved from the current process working directory.
    return path


def _rq3_rq2_candidate_pool(run: dict) -> list[tuple[int, int]]:
    """Return the complete RQ2 candidate pool in its original ordering.

    Important: ``mining_result['sampled_u']``/``sampled_v`` are not the full
    candidate pool.  They may contain only observed/scored rows.  RQ2 stores
    the complete pool explicitly in ``run['candidate_pool']`` and documents
    the downstream ``candidates``/score arrays as observed-only.
    """
    pairs = run.get("candidate_pool")
    if pairs is None:
        raise KeyError(
            "The RQ2 run has no `candidate_pool`; it cannot be reused as an "
            "exact whole-graph training condition."
        )

    canonical_pairs = []
    for pair in pairs:
        if len(pair) != 2:
            raise ValueError(f"Invalid RQ2 candidate pair: {pair!r}")
        u, v = int(pair[0]), int(pair[1])
        if u == v:
            raise ValueError("An RQ2 candidate contains a self-loop.")
        canonical_pairs.append((min(u, v), max(u, v)))

    if len(set(canonical_pairs)) != len(canonical_pairs):
        raise ValueError("The reused RQ2 candidate pool contains duplicates.")

    declared_size = int(
        run.get("candidate_set_size", len(canonical_pairs))
    )
    if len(canonical_pairs) != declared_size:
        raise ValueError(
            "RQ2 candidate_pool length disagrees with candidate_set_size: "
            f"{len(canonical_pairs)} != {declared_size}."
        )

    return canonical_pairs


def _rq3_rq2_run_is_cache_backed(run: dict) -> bool:
    """Use RQ2's own validated cache status as the primary signal."""
    status = str(run.get("cache_status", "")).strip().lower()
    has_result = isinstance(run.get("mining_result"), dict)

    # RQ2 sets `loaded` after validating metadata/candidates from the .pt file,
    # and `computed` immediately after saving the newly computed result.
    if status in {"loaded", "computed"} and has_result:
        return True

    path = _rq3_rq2_cache_path(run)
    return bool(path is not None and path.is_file() and has_result)


def _find_cached_rq2_full_graph_run(
    *,
    victim_seed: int,
    training_candidate_size: int,
    scoring_mode: str,
    endpoint_h: int | None = None,
) -> dict | None:
    """Find one exact, cache-backed RQ2 run or return None.

    ``endpoint_h=None`` deliberately does not assume h=0. It first matches
    seed, candidate configuration, candidate size, scoring mode, and cache
    availability. If several endpoint-hop variants remain, the caller must
    specify ``RQ3_RQ2_ENDPOINT_MINING_HOP``.
    """
    requested_config_id = _rq3_requested_rq2_candidate_config_id(
        training_candidate_size
    )

    matches = []
    for run in RQ2_FULL_GRAPH_MINING_RUNS:
        if int(run.get("seed", -1)) != int(victim_seed):
            continue
        if int(run.get("candidate_set_size", -1)) != int(
            training_candidate_size
        ):
            continue
        if str(run.get("scoring_mode", "")) != str(scoring_mode):
            continue
        if (
            str(scoring_mode) == "endpoint"
            and endpoint_h is not None
            and int(run.get("endpoint_mining_hop", 0)) != int(endpoint_h)
        ):
            continue
        if (
            requested_config_id is not None
            and str(run.get("candidate_config_id", ""))
            != str(requested_config_id)
        ):
            continue

        if (
            RQ3_REQUIRE_RQ2_DISK_CACHE
            and not _rq3_rq2_run_is_cache_backed(run)
        ):
            continue

        candidate_pool = _rq3_rq2_candidate_pool(run)
        if len(candidate_pool) != int(training_candidate_size):
            continue

        matches.append(run)

    if not matches:
        return None

    if len(matches) > 1:
        available_ids = sorted({
            str(run.get("candidate_config_id", ""))
            for run in matches
        })
        available_hops = sorted({
            int(run.get("endpoint_mining_hop", 0))
            for run in matches
        })

        if len(available_ids) > 1:
            raise RuntimeError(
                "Several RQ2 candidate configurations match the requested "
                "RQ3 whole-graph condition. Set "
                "RQ3_RQ2_CANDIDATE_CONFIG_ID to one exact configuration. "
                f"seed={victim_seed}, size={training_candidate_size}, "
                f"mode={scoring_mode}, matching configs={available_ids}."
            )

        if str(scoring_mode) == "endpoint" and len(available_hops) > 1:
            raise RuntimeError(
                "Several RQ2 endpoint-hop variants match the requested RQ3 "
                "whole-graph condition. Set "
                "RQ3_RQ2_ENDPOINT_MINING_HOP to one exact hop value. "
                f"seed={victim_seed}, size={training_candidate_size}, "
                f"config={available_ids[0]}, matching hops={available_hops}."
            )

        raise RuntimeError(
            "Several indistinguishable RQ2 full-graph runs match the requested "
            "RQ3 condition. Remove duplicate RQ2 runs or select a unique "
            "candidate configuration/hop. "
            f"seed={victim_seed}, size={training_candidate_size}, "
            f"mode={scoring_mode}, configs={available_ids}, "
            f"hops={available_hops}."
        )

    return matches[0]


# ------------------------------------------------------------
# Graph representation helpers
# ------------------------------------------------------------

def _canonical_undirected_pairs(
    edge_index: torch.Tensor,
) -> torch.Tensor:
    """Return unique non-self-loop undirected pairs as a CPU (2, M) tensor."""
    edge_index = edge_index.detach().cpu().long()
    if edge_index.numel() == 0:
        return torch.empty((2, 0), dtype=torch.long)

    u = torch.minimum(edge_index[0], edge_index[1])
    v = torch.maximum(edge_index[0], edge_index[1])
    keep = u < v
    if not bool(keep.any()):
        return torch.empty((2, 0), dtype=torch.long)

    pairs = torch.stack([u[keep], v[keep]], dim=0)
    return torch.unique(pairs, dim=1, sorted=True)


def _adjacency_lists(
    edge_index: torch.Tensor,
    n_nodes: int,
) -> list[list[int]]:
    adjacency = [set() for _ in range(int(n_nodes))]
    for u, v in _canonical_undirected_pairs(edge_index).t().tolist():
        u = int(u)
        v = int(v)
        adjacency[u].add(v)
        adjacency[v].add(u)
    return [sorted(neighbors) for neighbors in adjacency]


def _largest_connected_component(
    adjacency: list[list[int]],
) -> list[int]:
    n_nodes = len(adjacency)
    visited = [False] * n_nodes
    largest_component = []

    for start in range(n_nodes):
        if visited[start]:
            continue
        queue = deque([start])
        visited[start] = True
        component = []

        while queue:
            node = queue.popleft()
            component.append(node)
            for neighbor in adjacency[node]:
                if not visited[neighbor]:
                    visited[neighbor] = True
                    queue.append(neighbor)

        if len(component) > len(largest_component):
            largest_component = component

    return largest_component


# ------------------------------------------------------------
# Stage A1: subgraph node-selection methods
# ------------------------------------------------------------

def _minimum_nodes_for_pair_count(pair_count: int) -> int:
    """Smallest n with n(n-1)/2 >= pair_count."""
    pair_count = int(pair_count)
    if pair_count <= 0:
        return 2
    return int(
        math.ceil(
            (1.0 + math.sqrt(1.0 + 8.0 * pair_count)) / 2.0
        )
    )


def _quantile_bin_ids(values: np.ndarray, n_bins: int) -> np.ndarray:
    """Assign values to quantile bins while tolerating tied cut points."""
    values = np.asarray(values)
    n_bins = max(1, int(n_bins))
    if values.size == 0 or n_bins == 1:
        return np.zeros(values.size, dtype=np.int64)

    cut_points = np.quantile(
        values,
        np.linspace(0.0, 1.0, n_bins + 1)[1:-1],
    )
    cut_points = np.unique(cut_points)
    return np.searchsorted(
        cut_points,
        values,
        side="right",
    ).astype(np.int64)


def _structural_community_labels(
    graph: nx.Graph,
    *,
    seed: int,
    max_communities: int,
) -> np.ndarray:
    """
    Detect structural communities and merge small communities into one
    residual group when the requested maximum is exceeded.
    """
    n_nodes = int(graph.number_of_nodes())
    max_communities = max(1, int(max_communities))

    if graph.number_of_edges() == 0:
        return np.zeros(n_nodes, dtype=np.int64)

    try:
        communities = list(
            nx.community.louvain_communities(
                graph,
                seed=int(seed),
            )
        )
    except (AttributeError, TypeError):
        communities = list(
            nx.community.greedy_modularity_communities(graph)
        )

    communities = [
        set(int(node) for node in community)
        for community in communities
        if community
    ]
    communities.sort(
        key=lambda community: (-len(community), min(community))
    )

    labels = np.zeros(n_nodes, dtype=np.int64)
    if len(communities) <= max_communities:
        for community_id, community in enumerate(communities):
            labels[
                np.fromiter(community, dtype=np.int64)
            ] = int(community_id)
        return labels

    retained = communities[: max_communities - 1]
    residual_label = max_communities - 1
    labels.fill(residual_label)
    for community_id, community in enumerate(retained):
        labels[
            np.fromiter(community, dtype=np.int64)
        ] = int(community_id)
    return labels


def _build_stratification_profile(
    edge_index_global: torch.Tensor,
    n_nodes_global: int,
) -> dict:
    """Compute full-graph structural variables used for stratification."""
    adjacency = _adjacency_lists(edge_index_global, n_nodes_global)
    undirected_pairs = _canonical_undirected_pairs(edge_index_global)

    graph = nx.Graph()
    graph.add_nodes_from(range(int(n_nodes_global)))
    graph.add_edges_from(
        (int(u), int(v))
        for u, v in undirected_pairs.t().tolist()
    )

    degree_values = np.asarray(
        [len(adjacency[node]) for node in range(int(n_nodes_global))],
        dtype=np.int64,
    )
    core_number_map = nx.core_number(graph)
    core_values = np.asarray(
        [
            int(core_number_map.get(node, 0))
            for node in range(int(n_nodes_global))
        ],
        dtype=np.int64,
    )
    degree_bins = _quantile_bin_ids(
        degree_values,
        RQ3_STRATIFIED_DEGREE_BINS,
    )
    core_bins = _quantile_bin_ids(
        core_values,
        RQ3_STRATIFIED_CORE_BINS,
    )
    community_labels = _structural_community_labels(
        graph,
        seed=RQ3_STRATIFIED_COMMUNITY_SEED,
        max_communities=RQ3_STRATIFIED_MAX_COMMUNITIES,
    )

    strata = defaultdict(list)
    for node in range(int(n_nodes_global)):
        strata[
            (
                int(community_labels[node]),
                int(degree_bins[node]),
                int(core_bins[node]),
            )
        ].append(int(node))

    return {
        "adjacency": adjacency,
        "degree_values": degree_values,
        "core_values": core_values,
        "degree_bins": degree_bins,
        "core_bins": core_bins,
        "community_labels": community_labels,
        "strata": dict(strata),
    }


def _sample_nodes_from_strata(
    strata: dict,
    *,
    target_size: int,
    rng: random.Random,
) -> tuple[list[int], int]:
    """
    Sample across non-empty strata.

    When the budget permits, one node is first selected from every stratum.
    Remaining positions are drawn proportionally to the number of unselected
    nodes in each stratum.
    """
    target_size = int(target_size)
    pools = {
        key: list(nodes)
        for key, nodes in strata.items()
        if nodes
    }
    for nodes in pools.values():
        rng.shuffle(nodes)

    keys = list(pools)
    rng.shuffle(keys)
    available_count = sum(len(nodes) for nodes in pools.values())
    if target_size > available_count:
        raise ValueError(
            f"Requested {target_size} stratified nodes, but only "
            f"{available_count} are available."
        )

    selected = []
    covered_strata = 0

    if target_size >= len(keys):
        for key in keys:
            selected.append(pools[key].pop())
            covered_strata += 1
    else:
        for key in keys[:target_size]:
            selected.append(pools[key].pop())
            covered_strata += 1
        return selected, covered_strata

    while len(selected) < target_size:
        active_keys = [key for key in keys if pools[key]]
        total_remaining = sum(len(pools[key]) for key in active_keys)
        ticket = rng.randrange(total_remaining)
        cumulative = 0
        chosen_key = None
        for key in active_keys:
            cumulative += len(pools[key])
            if ticket < cumulative:
                chosen_key = key
                break
        if chosen_key is None:
            raise RuntimeError("Could not select a structural stratum.")
        selected.append(pools[chosen_key].pop())

    return selected, covered_strata


def _expand_core_with_context(
    adjacency: list[list[int]],
    core_nodes: list[int],
    *,
    target_size: int,
    max_hops: int,
    rng: random.Random,
) -> tuple[list[int], dict]:
    """
    Fill the remaining node budget through randomized multi-source BFS.

    The returned ordering always begins with every core node. Nodes added by
    expansion are context-only nodes and cannot become candidate endpoints.
    """
    selected_nodes = list(core_nodes)
    selected_set = set(selected_nodes)

    core_order = list(core_nodes)
    rng.shuffle(core_order)
    queue = deque((node, 0) for node in core_order)

    context_by_depth = defaultdict(int)
    while queue and len(selected_nodes) < int(target_size):
        node, depth = queue.popleft()
        if depth >= int(max_hops):
            continue

        neighbors = list(adjacency[node])
        rng.shuffle(neighbors)
        for neighbor in neighbors:
            if neighbor in selected_set:
                continue
            selected_set.add(neighbor)
            selected_nodes.append(neighbor)
            context_by_depth[int(depth + 1)] += 1
            queue.append((neighbor, depth + 1))
            if len(selected_nodes) == int(target_size):
                break

    fallback_nodes = 0
    if len(selected_nodes) < int(target_size):
        remaining = [
            node
            for node in range(len(adjacency))
            if node not in selected_set
        ]
        rng.shuffle(remaining)
        need = int(target_size) - len(selected_nodes)
        fallback = remaining[:need]
        selected_nodes.extend(fallback)
        fallback_nodes = len(fallback)

    if len(selected_nodes) != int(target_size):
        raise RuntimeError(
            "Context expansion did not produce the requested node count."
        )

    return selected_nodes, {
        "context_nodes_by_depth": dict(context_by_depth),
        "n_context_fallback_nodes": int(fallback_nodes),
    }


def _sample_source_nodes_stratified_context(
    edge_index_global: torch.Tensor,
    n_nodes_global: int,
    target_size: int,
    seed: int,
    *,
    max_candidate_pool_size: int,
    structural_profile: dict,
) -> tuple[torch.Tensor, dict]:
    """
    Stratified core-node selection followed by budgeted neighborhood context.

    Output ordering:
        [core candidate-endpoint nodes | context-only nodes]
    """
    n_nodes_global = int(n_nodes_global)
    target_size = int(target_size)
    max_candidate_pool_size = int(max_candidate_pool_size)

    if target_size > n_nodes_global:
        raise ValueError("target_size exceeds the complete graph size.")

    minimum_core_nodes = _minimum_nodes_for_pair_count(
        max_candidate_pool_size
    )
    requested_core_nodes = max(
        2,
        minimum_core_nodes,
        int(math.ceil(RQ3_STRATIFIED_CORE_SHARE * target_size)),
    )
    if requested_core_nodes > target_size:
        possible_pairs = target_size * (target_size - 1) // 2
        raise ValueError(
            "The requested source graph is too small for the configured "
            "candidate pool. "
            f"source nodes={target_size}, possible pairs={possible_pairs}, "
            f"requested candidates={max_candidate_pool_size}."
        )

    rng = random.Random(int(seed))
    core_nodes, covered_strata = _sample_nodes_from_strata(
        structural_profile["strata"],
        target_size=requested_core_nodes,
        rng=rng,
    )
    source_nodes, expansion_metadata = _expand_core_with_context(
        structural_profile["adjacency"],
        core_nodes,
        target_size=target_size,
        max_hops=RQ3_STRATIFIED_CONTEXT_HOPS,
        rng=rng,
    )

    source_set = set(source_nodes)
    one_hop_retention = []
    for node in core_nodes:
        neighbors = structural_profile["adjacency"][node]
        if not neighbors:
            one_hop_retention.append(1.0)
        else:
            retained = sum(
                neighbor in source_set
                for neighbor in neighbors
            )
            one_hop_retention.append(retained / len(neighbors))

    core_idx = np.asarray(core_nodes, dtype=np.int64)
    community_labels = structural_profile["community_labels"]
    degree_values = structural_profile["degree_values"]
    core_values = structural_profile["core_values"]

    metadata = {
        "selection_mode": "stratified_core_plus_budgeted_context",
        "n_core_nodes": int(len(core_nodes)),
        "n_context_only_nodes": int(len(source_nodes) - len(core_nodes)),
        "core_share_requested": float(RQ3_STRATIFIED_CORE_SHARE),
        "core_share_actual": float(len(core_nodes) / max(1, len(source_nodes))),
        "minimum_core_nodes_for_candidate_pool": int(minimum_core_nodes),
        "context_hops": int(RQ3_STRATIFIED_CONTEXT_HOPS),
        "degree_bins_requested": int(RQ3_STRATIFIED_DEGREE_BINS),
        "core_bins_requested": int(RQ3_STRATIFIED_CORE_BINS),
        "n_structural_communities": int(
            np.unique(community_labels).size
        ),
        "n_nonempty_strata": int(len(structural_profile["strata"])),
        "n_covered_strata": int(covered_strata),
        "covered_strata_fraction": float(
            covered_strata / max(1, len(structural_profile["strata"]))
        ),
        "n_core_communities": int(
            np.unique(community_labels[core_idx]).size
        ),
        "mean_core_degree_global": float(degree_values[core_idx].mean()),
        "median_core_degree_global": float(np.median(degree_values[core_idx])),
        "mean_core_number_global": float(core_values[core_idx].mean()),
        "median_core_number_global": float(np.median(core_values[core_idx])),
        "mean_core_one_hop_retention": float(np.mean(one_hop_retention)),
        "complete_core_one_hop_fraction": float(
            np.mean(np.isclose(one_hop_retention, 1.0))
        ),
        "core_nodes_are_local_prefix": True,
        **expansion_metadata,
    }

    return torch.tensor(source_nodes, dtype=torch.long), metadata


def _sample_source_nodes_forest_fire(
    edge_index_global: torch.Tensor,
    n_nodes_global: int,
    target_size: int,
    seed: int,
    burn_probability: float,
) -> tuple[torch.Tensor, dict]:
    """Forest Fire baseline with deterministic restarts."""
    adjacency = _adjacency_lists(edge_index_global, n_nodes_global)
    rng = random.Random(int(seed))

    selected = set()
    ordered_nodes = []
    queue = deque()
    n_fire_starts = 0
    n_burned_tree_edges = 0

    def start_new_fire() -> bool:
        nonisolated = [
            node
            for node in range(n_nodes_global)
            if node not in selected and adjacency[node]
        ]
        candidates = nonisolated or [
            node
            for node in range(n_nodes_global)
            if node not in selected
        ]
        if not candidates:
            return False
        start = rng.choice(candidates)
        selected.add(start)
        ordered_nodes.append(start)
        queue.append(start)
        return True

    if start_new_fire():
        n_fire_starts += 1

    while len(ordered_nodes) < target_size:
        if not queue:
            if not start_new_fire():
                break
            n_fire_starts += 1
            if len(ordered_nodes) >= target_size:
                break

        node = queue.popleft()
        neighbors = [
            neighbor
            for neighbor in adjacency[node]
            if neighbor not in selected
        ]
        rng.shuffle(neighbors)

        burn_count = 0
        while (
            burn_count < len(neighbors)
            and rng.random() < float(burn_probability)
        ):
            burn_count += 1

        for neighbor in neighbors[:burn_count]:
            if neighbor in selected:
                continue
            selected.add(neighbor)
            ordered_nodes.append(neighbor)
            queue.append(neighbor)
            n_burned_tree_edges += 1
            if len(ordered_nodes) == target_size:
                break

    if len(ordered_nodes) != target_size:
        raise RuntimeError(
            f"Forest Fire produced {len(ordered_nodes)} nodes; "
            f"expected {target_size}."
        )

    return torch.tensor(ordered_nodes, dtype=torch.long), {
        "selection_mode": "forest_fire_all_nodes_candidate_eligible",
        "burn_probability": float(burn_probability),
        "n_fire_starts": int(n_fire_starts),
        "n_restarts": int(max(0, n_fire_starts - 1)),
        "n_burned_tree_edges": int(n_burned_tree_edges),
        "n_core_nodes": int(target_size),
        "n_context_only_nodes": 0,
        "core_nodes_are_local_prefix": True,
    }


def _select_source_nodes(
    method: str,
    edge_index_global: torch.Tensor,
    n_nodes_global: int,
    target_size: int,
    seed: int,
    *,
    max_candidate_pool_size: int,
    structural_profile: dict | None,
) -> tuple[torch.Tensor, dict]:
    method = _normalise_subgraph_method(method)

    if method == "whole_graph":
        if int(target_size) != int(n_nodes_global):
            raise ValueError(
                "The whole_graph condition must select exactly all nodes."
            )
        return torch.arange(int(n_nodes_global), dtype=torch.long), {
            "selection_mode": "identity_all_nodes",
            "whole_graph": True,
            "n_restarts": 0,
            "n_core_nodes": int(n_nodes_global),
            "n_context_only_nodes": 0,
            "core_nodes_are_local_prefix": True,
        }

    if method == "stratified_context":
        if structural_profile is None:
            raise ValueError(
                "stratified_context requires a structural profile."
            )
        return _sample_source_nodes_stratified_context(
            edge_index_global,
            n_nodes_global,
            target_size,
            seed,
            max_candidate_pool_size=max_candidate_pool_size,
            structural_profile=structural_profile,
        )

    if method == "forest_fire":
        return _sample_source_nodes_forest_fire(
            edge_index_global,
            n_nodes_global,
            target_size,
            seed,
            burn_probability=RQ3_FOREST_FIRE_BURN_PROBABILITY,
        )

    raise ValueError(f"Unknown subgraph construction method: {method}.")


# ------------------------------------------------------------
# Stage A2: graph induction and local remapping
# ------------------------------------------------------------

def _induced_local_graph(
    context: dict,
    source_nodes_global: torch.Tensor,
) -> dict:
    """Construct the induced graph and remap nodes to 0, ..., m-1."""
    n_global = int(context["n_nodes"])
    source_nodes_global = source_nodes_global.detach().cpu().long()
    n_local = int(source_nodes_global.numel())

    global_to_local = torch.full((n_global,), -1, dtype=torch.long)
    global_to_local[source_nodes_global] = torch.arange(
        n_local,
        dtype=torch.long,
    )

    edge_global = context["edge_index"].detach().cpu().long()
    src_local = global_to_local[edge_global[0]]
    dst_local = global_to_local[edge_global[1]]
    inside_source = (src_local >= 0) & (dst_local >= 0)

    edge_index_local = torch.stack(
        [src_local[inside_source], dst_local[inside_source]],
        dim=0,
    ).to(context["device"])

    source_nodes_attr_device = source_nodes_global.to(
        context["attr"].device
    )
    source_nodes_label_device = source_nodes_global.to(
        context["labels"].device
    )

    attr_local = context["attr"][source_nodes_attr_device].to(
        context["device"]
    )
    labels_local = context["labels"][source_nodes_label_device].to(
        context["device"]
    )

    idx_test_global = torch.as_tensor(
        context["idx_test"],
        dtype=torch.long,
    ).detach().cpu()
    eval_idx_local = global_to_local[idx_test_global]
    eval_idx_local = eval_idx_local[eval_idx_local >= 0].to(
        context["device"]
    )

    return {
        "n_local": n_local,
        "source_nodes_global": source_nodes_global,
        "global_to_local": global_to_local,
        "edge_index": edge_index_local,
        "attr": attr_local,
        "labels": labels_local,
        "eval_idx": eval_idx_local,
    }


def _sample_local_candidate_pairs(
    n_endpoint_nodes: int,
    count: int,
    seed: int,
) -> list[tuple[int, int]]:
    """Uniformly sample pairs from the core-node local-ID prefix only."""
    n_endpoint_nodes = int(n_endpoint_nodes)
    n_possible = n_endpoint_nodes * (n_endpoint_nodes - 1) // 2
    if count > n_possible:
        raise ValueError(
            f"{count} candidates requested, but {n_endpoint_nodes} "
            f"candidate-eligible core nodes provide only {n_possible} "
            "unique undirected pairs."
        )

    rng = random.Random(int(seed))
    linear_ids = torch.tensor(
        rng.sample(range(n_possible), int(count)),
        dtype=torch.long,
    )
    pairs = EndpointPRBCDV4Scorer.linear_to_triu_idx(
        n_endpoint_nodes,
        linear_ids,
    ).cpu()
    return [(int(u), int(v)) for u, v in pairs.t().tolist()]


# ------------------------------------------------------------
# Stage A3: subgraph statistics
# ------------------------------------------------------------

def _connected_component_sizes_from_pairs(
    n_nodes: int,
    pairs: torch.Tensor,
) -> list[int]:
    adjacency = [[] for _ in range(int(n_nodes))]
    for u, v in pairs.t().tolist():
        u = int(u)
        v = int(v)
        adjacency[u].append(v)
        adjacency[v].append(u)

    visited = [False] * int(n_nodes)
    component_sizes = []
    for start in range(int(n_nodes)):
        if visited[start]:
            continue
        queue = deque([start])
        visited[start] = True
        size = 0
        while queue:
            node = queue.popleft()
            size += 1
            for neighbor in adjacency[node]:
                if not visited[neighbor]:
                    visited[neighbor] = True
                    queue.append(neighbor)
        component_sizes.append(size)
    return component_sizes


def _compute_subgraph_stats(
    context: dict,
    local: dict,
    method: str,
    victim_seed: int,
    subgraph_seed: int,
    construction_seed: int,
    requested_fraction: float,
    sampler_metadata: dict,
) -> dict:
    n_global = int(context["n_nodes"])
    n_local = int(local["n_local"])

    global_pairs = _canonical_undirected_pairs(context["edge_index"])
    local_pairs = _canonical_undirected_pairs(local["edge_index"])

    n_global_edges = int(global_pairs.size(1))
    n_local_edges = int(local_pairs.size(1))
    n_possible_local = n_local * (n_local - 1) // 2

    degrees = torch.zeros(n_local, dtype=torch.long)
    if local_pairs.numel() > 0:
        degrees += torch.bincount(
            local_pairs[0], minlength=n_local
        )
        degrees += torch.bincount(
            local_pairs[1], minlength=n_local
        )

    component_sizes = _connected_component_sizes_from_pairs(
        n_local,
        local_pairs,
    )
    largest_component_size = max(component_sizes, default=0)

    stats = {
        "victim_seed": int(victim_seed),
        "subgraph_method": str(method),
        "subgraph_seed": int(subgraph_seed),
        "construction_seed": int(construction_seed),
        "subgraph_fraction_requested": float(requested_fraction),
        "subgraph_fraction_actual": float(n_local / n_global),
        "n_nodes_global": int(n_global),
        "n_source_nodes": int(n_local),
        "n_test_nodes_local": int(local["eval_idx"].numel()),
        "n_edges_global_undirected": int(n_global_edges),
        "n_edges_local_undirected": int(n_local_edges),
        "global_edge_coverage": float(
            n_local_edges / max(1, n_global_edges)
        ),
        "density_local": float(
            n_local_edges / max(1, n_possible_local)
        ),
        "average_degree_local": float(
            degrees.float().mean().item() if n_local else 0.0
        ),
        "minimum_degree_local": int(
            degrees.min().item() if n_local else 0
        ),
        "median_degree_local": float(
            degrees.float().median().item() if n_local else 0.0
        ),
        "maximum_degree_local": int(
            degrees.max().item() if n_local else 0
        ),
        "n_isolated_nodes_local": int((degrees == 0).sum().item()),
        "n_connected_components_local": int(len(component_sizes)),
        "largest_component_size_local": int(largest_component_size),
        "largest_component_fraction_local": float(
            largest_component_size / max(1, n_local)
        ),
        "n_possible_additions_local": int(
            max(0, n_possible_local - n_local_edges)
        ),
        "n_possible_deletions_local": int(n_local_edges),
    }

    for key, value in sampler_metadata.items():
        stats[f"method_{key}"] = value

    return stats


def _print_subgraph_stats(stats: dict) -> None:
    print("=" * 80)
    print(
        "RQ3 source subgraph constructed\n"
        f"  victim seed:              {stats['victim_seed']}\n"
        f"  construction method:      {stats['subgraph_method']}\n"
        f"  subgraph seed:            {stats['subgraph_seed']}\n"
        f"  effective seed:           {stats['construction_seed']}\n"
        f"  requested node fraction:  {stats['subgraph_fraction_requested']:.6f}\n"
        f"  actual nodes:             {stats['n_source_nodes']}/{stats['n_nodes_global']}\n"
        f"  undirected edges:         {stats['n_edges_local_undirected']}\n"
        f"  graph density:            {stats['density_local']:.6f}\n"
        f"  average degree:           {stats['average_degree_local']:.3f}\n"
        f"  degree min/median/max:    "
        f"{stats['minimum_degree_local']}/"
        f"{stats['median_degree_local']:.1f}/"
        f"{stats['maximum_degree_local']}\n"
        f"  connected components:     {stats['n_connected_components_local']}\n"
        f"  largest component share:  {stats['largest_component_fraction_local']:.3f}\n"
        f"  isolated nodes:           {stats['n_isolated_nodes_local']}\n"
        f"  local test nodes:         {stats['n_test_nodes_local']}\n"
        f"  global edge coverage:     {stats['global_edge_coverage']:.6f}"
    )


# ------------------------------------------------------------
# Cache fingerprints and serialization
# ------------------------------------------------------------

def _hash_tensor(hasher, tensor: torch.Tensor) -> None:
    tensor = tensor.detach()
    hasher.update(str(tensor.dtype).encode("utf-8"))
    hasher.update(str(tuple(tensor.shape)).encode("utf-8"))
    hasher.update(str(tensor.layout).encode("utf-8"))

    if tensor.layout != torch.strided:
        tensor = tensor.to_sparse_coo().coalesce().cpu()
        _hash_tensor(hasher, tensor.indices())
        _hash_tensor(hasher, tensor.values())
        return

    array = tensor.cpu().contiguous().numpy()
    hasher.update(array.tobytes(order="C"))


def _context_fingerprint(context: dict) -> str:
    """Fingerprint all inputs that materially affect local victim labels."""
    hasher = hashlib.sha256()
    hasher.update(str(int(context["n_nodes"])).encode("utf-8"))
    _hash_tensor(hasher, context["edge_index"])
    _hash_tensor(hasher, context["attr"])
    _hash_tensor(hasher, context["labels"])
    _hash_tensor(
        hasher,
        torch.as_tensor(context["idx_test"], dtype=torch.long),
    )

    model = context["model"]
    for key, value in sorted(model.state_dict().items()):
        hasher.update(str(key).encode("utf-8"))
        if torch.is_tensor(value):
            _hash_tensor(hasher, value)
        else:
            hasher.update(repr(value).encode("utf-8"))

    return hasher.hexdigest()


def _pairs_fingerprint(pairs: list[tuple[int, int]]) -> str:
    array = np.asarray(pairs, dtype=np.int64).reshape(-1, 2)
    return hashlib.sha256(array.tobytes(order="C")).hexdigest()


def _nodes_fingerprint(nodes: torch.Tensor) -> str:
    array = nodes.detach().cpu().long().contiguous().numpy()
    return hashlib.sha256(array.tobytes(order="C")).hexdigest()


def _sampler_cache_configuration(method: str) -> dict:
    method = _normalise_subgraph_method(method)
    if method == "stratified_context":
        return {
            "core_share": float(RQ3_STRATIFIED_CORE_SHARE),
            "context_hops": int(RQ3_STRATIFIED_CONTEXT_HOPS),
            "degree_bins": int(RQ3_STRATIFIED_DEGREE_BINS),
            "core_bins": int(RQ3_STRATIFIED_CORE_BINS),
            "max_communities": int(RQ3_STRATIFIED_MAX_COMMUNITIES),
            "community_seed": int(RQ3_STRATIFIED_COMMUNITY_SEED),
        }
    if method == "forest_fire":
        return {
            "burn_probability": float(
                RQ3_FOREST_FIRE_BURN_PROBABILITY
            )
        }
    return {"whole_graph": True}


def _make_subgraph_bundle_spec(
    *,
    context_fingerprint: str,
    victim_seed: int,
    subgraph_method: str,
    subgraph_seed: int,
    construction_seed: int,
    subgraph_fraction_requested: float,
    n_source_nodes: int,
    max_training_candidate_size: int,
    candidate_pool_seed: int,
    sampler_configuration: dict,
) -> dict:
    """Stable identity for one reusable source-subgraph candidate pool."""
    return {
        "cache_version": RQ3_SUBGRAPH_BUNDLE_CACHE_VERSION,
        "context_fingerprint": str(context_fingerprint),
        "victim_seed": int(victim_seed),
        "subgraph_method": str(subgraph_method),
        "subgraph_seed": int(subgraph_seed),
        "construction_seed": int(construction_seed),
        "subgraph_fraction_requested": float(
            subgraph_fraction_requested
        ),
        "n_source_nodes": int(n_source_nodes),
        "max_training_candidate_size": int(
            max_training_candidate_size
        ),
        "candidate_pool_seed": int(candidate_pool_seed),
        "sampler_configuration": dict(sampler_configuration),
    }


def _subgraph_bundle_path(spec: dict, cache_key: str) -> Path:
    return RQ3_SUBGRAPH_BUNDLE_CACHE_DIR / (
        f"victim-{_rq3_safe_value(spec['victim_seed'])}"
        f"__method-{_rq3_safe_value(spec['subgraph_method'])}"
        f"__subseed-{_rq3_safe_value(spec['subgraph_seed'])}"
        f"__frac-{_rq3_safe_value(spec['subgraph_fraction_requested'])}"
        f"__maxCand-{_rq3_safe_value(spec['max_training_candidate_size'])}"
        f"__{cache_key[:20]}.pt"
    )


def _try_load_subgraph_bundle(
    *,
    cache_path: Path,
    cache_key: str,
    expected_n_source_nodes: int,
    expected_max_candidate_size: int,
):
    if RQ3_FORCE_RESAMPLE_CANDIDATES or not cache_path.exists():
        return None

    try:
        payload = _torch_load_unrestricted(cache_path)
    except Exception as exc:
        warnings.warn(
            f"Could not load RQ3 candidate-pool cache {cache_path}: "
            f"{exc}. The pool will be reconstructed."
        )
        return None

    if payload.get("cache_version") != RQ3_SUBGRAPH_BUNDLE_CACHE_VERSION:
        return None
    if payload.get("cache_key") != cache_key:
        return None

    source_nodes = torch.as_tensor(
        payload.get("source_nodes_global", []),
        dtype=torch.long,
    ).view(-1)
    core_nodes = torch.as_tensor(
        payload.get("core_nodes_global", []),
        dtype=torch.long,
    ).view(-1)
    candidate_pool_local = [
        (int(u), int(v))
        for u, v in payload.get("candidate_pool_local", [])
    ]

    if source_nodes.numel() != int(expected_n_source_nodes):
        return None
    if core_nodes.numel() < 2 or core_nodes.numel() > source_nodes.numel():
        return None
    if not torch.equal(
        source_nodes[: core_nodes.numel()],
        core_nodes,
    ):
        return None
    if len(candidate_pool_local) != int(expected_max_candidate_size):
        return None

    n_core = int(core_nodes.numel())
    if any(not (0 <= u < v < n_core) for u, v in candidate_pool_local):
        return None
    if len(set(candidate_pool_local)) != len(candidate_pool_local):
        return None

    return {
        "source_nodes_global": source_nodes,
        "core_nodes_global": core_nodes,
        "sampler_metadata": dict(payload.get("sampler_metadata", {})),
        "candidate_pool_seed": int(payload["candidate_pool_seed"]),
        "candidate_pool_local": candidate_pool_local,
        "created_utc": payload.get("created_utc", ""),
    }


def _save_subgraph_bundle(
    *,
    cache_path: Path,
    cache_key: str,
    cache_spec: dict,
    source_nodes_global: torch.Tensor,
    core_nodes_global: torch.Tensor,
    sampler_metadata: dict,
    candidate_pool_seed: int,
    candidate_pool_local: list[tuple[int, int]],
) -> None:
    payload = {
        "cache_version": RQ3_SUBGRAPH_BUNDLE_CACHE_VERSION,
        "cache_key": cache_key,
        "cache_spec": cache_spec,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "source_nodes_global": source_nodes_global.detach().cpu().long(),
        "core_nodes_global": core_nodes_global.detach().cpu().long(),
        "sampler_metadata": dict(sampler_metadata),
        "candidate_pool_seed": int(candidate_pool_seed),
        "candidate_pool_local": list(candidate_pool_local),
    }
    _atomic_torch_save(payload, cache_path)


def _cache_key_from_spec(spec: dict) -> str:
    payload = json.dumps(
        spec,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def _cache_path_for_spec(spec: dict, cache_key: str) -> Path:
    return RQ3_LABEL_CACHE_DIR / (
        f"victim-{_rq3_safe_value(spec['victim_seed'])}"
        f"__method-{_rq3_safe_value(spec['subgraph_method'])}"
        f"__subseed-{_rq3_safe_value(spec['subgraph_seed'])}"
        f"__frac-{_rq3_safe_value(spec['subgraph_fraction_requested'])}"
        f"__cand-{_rq3_safe_value(spec['training_candidate_size'])}"
        f"__mode-{_rq3_safe_value(spec['scoring_mode'])}"
        f"__{cache_key[:20]}.pt"
    )


def _to_cpu_serializable(value):
    if torch.is_tensor(value):
        return value.detach().cpu()
    if isinstance(value, np.ndarray):
        return value.copy()
    if isinstance(value, dict):
        return {
            key: _to_cpu_serializable(item)
            for key, item in value.items()
        }
    if isinstance(value, tuple):
        return tuple(_to_cpu_serializable(item) for item in value)
    if isinstance(value, list):
        return [_to_cpu_serializable(item) for item in value]
    if isinstance(value, set):
        return {_to_cpu_serializable(item) for item in value}
    return value


def _atomic_torch_save(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(
        path.suffix + f".tmp-{os.getpid()}"
    )
    torch.save(_to_cpu_serializable(payload), temporary_path)
    os.replace(temporary_path, path)


def _atomic_write_dataframe(
    dataframe: pd.DataFrame,
    path: Path,
) -> None:
    """Persist partial sweep progress without leaving a half-written CSV."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(
        path.suffix + f".tmp-{os.getpid()}"
    )
    dataframe.to_csv(temporary_path, index=False)
    os.replace(temporary_path, path)


def _torch_load_unrestricted(path: Path):
    # PyTorch 2.6 changed the default of weights_only. The cache is local and
    # created by this notebook, so explicitly allow its ordinary Python values.
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def _try_load_label_cache(
    cache_path: Path,
    cache_key: str,
    source_nodes_global: torch.Tensor,
    candidates_local: list[tuple[int, int]],
    candidates_global: list[tuple[int, int]],
):
    if RQ3_FORCE_RELABEL or not cache_path.exists():
        return None

    try:
        payload = _torch_load_unrestricted(cache_path)
    except Exception as exc:
        warnings.warn(
            f"Could not load RQ3 label cache {cache_path}: {exc}. "
            "The configuration will be relabeled."
        )
        return None

    if payload.get("cache_version") != RQ3_LABEL_CACHE_VERSION:
        return None
    if payload.get("cache_key") != cache_key:
        return None

    cached_nodes = torch.as_tensor(
        payload.get("source_nodes_global", []),
        dtype=torch.long,
    )
    if not torch.equal(
        cached_nodes,
        source_nodes_global.detach().cpu().long(),
    ):
        return None

    if payload.get("candidate_pairs_local") != candidates_local:
        return None
    if payload.get("candidate_pairs_global") != candidates_global:
        return None

    result_local = payload.get("result_local")
    if not isinstance(result_local, dict):
        return None

    return result_local


def _save_label_cache(
    cache_path: Path,
    cache_key: str,
    cache_spec: dict,
    source_nodes_global: torch.Tensor,
    candidates_local: list[tuple[int, int]],
    candidates_global: list[tuple[int, int]],
    result_local: dict,
) -> None:
    payload = {
        "cache_version": RQ3_LABEL_CACHE_VERSION,
        "cache_key": cache_key,
        "cache_spec": cache_spec,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "source_nodes_global": source_nodes_global.detach().cpu().long(),
        "candidate_pairs_local": candidates_local,
        "candidate_pairs_global": candidates_global,
        "result_local": result_local,
    }
    _atomic_torch_save(payload, cache_path)


# ------------------------------------------------------------
# Victim wrapper: same weights, local tensors only
# ------------------------------------------------------------

class _FreshLocalVictim(torch.nn.Module):
    """Force the victim to use the supplied local adjacency."""

    def __init__(self, victim):
        super().__init__()
        self.victim = victim

    def forward(self, data: torch.Tensor, adj: torch.Tensor):
        victim = self.victim
        old_cache = getattr(victim, "adj_preped", None)
        old_gdc = copy.deepcopy(getattr(victim, "gdc_params", None))

        try:
            if hasattr(victim, "adj_preped"):
                victim.adj_preped = None

            if isinstance(old_gdc, dict):
                safe_gdc = dict(old_gdc)
                if "k" in safe_gdc:
                    safe_gdc["k"] = min(
                        int(safe_gdc["k"]),
                        int(data.size(0)),
                    )
                victim.gdc_params = safe_gdc

            return victim(data=data, adj=adj)

        finally:
            if hasattr(victim, "adj_preped"):
                victim.adj_preped = old_cache
            if old_gdc is not None:
                victim.gdc_params = old_gdc


# ------------------------------------------------------------
# Stage A: construct the complete subgraph sweep only
# ------------------------------------------------------------

def construct_rq3_subgraph_sweep(
    seed_contexts: list[dict],
) -> tuple[list[dict], list[dict]]:
    subgraph_configs = []
    summary_rows = []
    fingerprint_by_context_id = {}
    structural_profile_by_context_id = {}

    for context in seed_contexts:
        activate_seed_context(context)
        victim_seed = int(context["seed"])
        set_global_seed(victim_seed)

        context_id = id(context)
        if context_id not in fingerprint_by_context_id:
            print(
                f"Computing cache fingerprint for victim seed "
                f"{victim_seed} ..."
            )
            fingerprint_by_context_id[context_id] = _context_fingerprint(
                context
            )
        context_fingerprint = fingerprint_by_context_id[context_id]

        n_global = int(context["n_nodes"])

        if (
            "stratified_context" in RQ3_SUBGRAPH_METHODS
            and context_id not in structural_profile_by_context_id
        ):
            print(
                f"Computing structural stratification profile for "
                f"victim seed {victim_seed} ..."
            )
            structural_profile_by_context_id[context_id] = (
                _build_stratification_profile(
                    context["edge_index"],
                    n_global,
                )
            )

        for method_index, method in enumerate(RQ3_SUBGRAPH_METHODS):
            for subgraph_seed in RQ3_SUBGRAPH_SEEDS:
                for fraction in RQ3_SUBGRAPH_FRACTIONS:
                    is_whole_graph = bool(
                        np.isclose(float(fraction), 1.0)
                    )

                    # The whole graph has neither a construction-method
                    # replicate nor a subgraph-sampling replicate. Keep exactly
                    # one canonical whole-graph configuration per victim seed.
                    if is_whole_graph and (
                        method_index > 0
                        or int(subgraph_seed)
                        != int(RQ3_PRIMARY_SUBGRAPH_SEED)
                    ):
                        continue

                    effective_method = (
                        "whole_graph" if is_whole_graph else str(method)
                    )

                    # For fractions below one, this remains the subgraph
                    # construction seed. At fraction one, subgraph_seed acts as
                    # an independent candidate-pool replicate.
                    construction_seed = _rq3_stable_seed(
                        "rq3_subgraph_construction",
                        victim_seed,
                        effective_method,
                        subgraph_seed,
                    )

                    n_source = (
                        n_global
                        if is_whole_graph
                        else max(
                            2,
                            math.ceil(float(fraction) * n_global),
                        )
                    )

                    # The maximum pool is persisted once. Smaller requested
                    # training sets remain deterministic prefixes.
                    max_training_candidate_size = max(
                        RQ3_SUBGRAPH_TRAINING_CANDIDATE_SIZES
                    )
                    candidate_pool_seed = _rq3_stable_seed(
                        "rq3_candidate_pool",
                        victim_seed,
                        effective_method,
                        subgraph_seed,
                        f"{float(fraction):.12g}",
                    )
                    bundle_spec = _make_subgraph_bundle_spec(
                        context_fingerprint=context_fingerprint,
                        victim_seed=victim_seed,
                        subgraph_method=effective_method,
                        subgraph_seed=subgraph_seed,
                        construction_seed=construction_seed,
                        subgraph_fraction_requested=fraction,
                        n_source_nodes=n_source,
                        max_training_candidate_size=(
                            max_training_candidate_size
                        ),
                        candidate_pool_seed=candidate_pool_seed,
                        sampler_configuration=(
                            _sampler_cache_configuration(effective_method)
                        ),
                    )
                    bundle_key = _cache_key_from_spec(bundle_spec)
                    bundle_path = _subgraph_bundle_path(
                        bundle_spec,
                        bundle_key,
                    )
                    bundle = _try_load_subgraph_bundle(
                        cache_path=bundle_path,
                        cache_key=bundle_key,
                        expected_n_source_nodes=n_source,
                        expected_max_candidate_size=(
                            max_training_candidate_size
                        ),
                    )
                    candidate_pool_cache_hit = bundle is not None

                    if candidate_pool_cache_hit:
                        source_nodes_global = bundle[
                            "source_nodes_global"
                        ]
                        core_nodes_global = bundle[
                            "core_nodes_global"
                        ]
                        sampler_metadata = bundle["sampler_metadata"]
                        candidate_pool_seed = int(
                            bundle["candidate_pool_seed"]
                        )
                        candidate_pool_local = list(
                            bundle["candidate_pool_local"]
                        )
                        print(
                            "[CACHE] Loaded source subgraph and candidate "
                            f"pool: {bundle_path}"
                        )
                    else:
                        source_nodes_global, sampler_metadata = (
                            _select_source_nodes(
                                method=effective_method,
                                edge_index_global=context["edge_index"],
                                n_nodes_global=n_global,
                                target_size=n_source,
                                seed=construction_seed,
                                max_candidate_pool_size=(
                                    max_training_candidate_size
                                ),
                                structural_profile=(
                                    structural_profile_by_context_id.get(
                                        context_id
                                    )
                                    if effective_method
                                    == "stratified_context"
                                    else None
                                ),
                            )
                        )
                        n_core_nodes = int(
                            sampler_metadata.get(
                                "n_core_nodes",
                                source_nodes_global.numel(),
                            )
                        )
                        core_nodes_global = source_nodes_global[
                            :n_core_nodes
                        ].clone()
                        candidate_pool_local = (
                            _sample_local_candidate_pairs(
                                n_core_nodes,
                                max_training_candidate_size,
                                candidate_pool_seed,
                            )
                        )
                        _save_subgraph_bundle(
                            cache_path=bundle_path,
                            cache_key=bundle_key,
                            cache_spec=bundle_spec,
                            source_nodes_global=source_nodes_global,
                            core_nodes_global=core_nodes_global,
                            sampler_metadata=sampler_metadata,
                            candidate_pool_seed=candidate_pool_seed,
                            candidate_pool_local=candidate_pool_local,
                        )
                        print(
                            "[CACHE] Saved source subgraph and candidate "
                            f"pool: {bundle_path}"
                        )

                    n_core_nodes = int(core_nodes_global.numel())
                    if not torch.equal(
                        source_nodes_global[:n_core_nodes],
                        core_nodes_global,
                    ):
                        raise AssertionError(
                            "Core nodes must occupy the local-ID prefix."
                        )
                    if any(
                        not (0 <= u < v < n_core_nodes)
                        for u, v in candidate_pool_local
                    ):
                        raise AssertionError(
                            "A cached candidate pair uses a context-only node."
                        )

                    local = _induced_local_graph(
                        context,
                        source_nodes_global,
                    )

                    source_mask = torch.zeros(
                        n_global,
                        dtype=torch.bool,
                    )
                    source_mask[source_nodes_global] = True
                    target_nodes_global = torch.nonzero(
                        ~source_mask,
                        as_tuple=True,
                    )[0]

                    config_key = (
                        victim_seed,
                        str(effective_method),
                        int(subgraph_seed),
                        float(fraction),
                    )

                    stats = _compute_subgraph_stats(
                        context=context,
                        local=local,
                        method=effective_method,
                        victim_seed=victim_seed,
                        subgraph_seed=subgraph_seed,
                        construction_seed=construction_seed,
                        requested_fraction=fraction,
                        sampler_metadata=sampler_metadata,
                    )
                    stats = dict(stats)
                    stats.update({
                        "candidate_pool_cache_hit": bool(
                            candidate_pool_cache_hit
                        ),
                        "candidate_pool_cache_path": str(bundle_path),
                        "candidate_pool_cache_key": str(bundle_key),
                        "candidate_pool_seed": int(candidate_pool_seed),
                        "max_training_candidate_size": int(
                            max_training_candidate_size
                        ),
                        "n_core_nodes": int(n_core_nodes),
                        "n_context_only_nodes": int(
                            source_nodes_global.numel() - n_core_nodes
                        ),
                    })
                    _print_subgraph_stats(stats)
                    summary_rows.append(stats)

                    subgraph_configs.append({
                        "config_key": config_key,
                        "context": context,
                        "victim_seed": victim_seed,
                        "subgraph_method": str(effective_method),
                        "subgraph_seed": int(subgraph_seed),
                        "construction_seed": int(construction_seed),
                        "subgraph_fraction_requested": float(fraction),
                        "subgraph_fraction_actual": float(
                            local["n_local"] / n_global
                        ),
                        "source_nodes_global": source_nodes_global,
                        "core_nodes_global": core_nodes_global,
                        "n_core_nodes": int(n_core_nodes),
                        "n_context_only_nodes": int(
                            source_nodes_global.numel() - n_core_nodes
                        ),
                        "target_nodes_global": target_nodes_global,
                        "local": local,
                        "sampler_metadata": sampler_metadata,
                        "subgraph_stats": stats,
                        "context_fingerprint": context_fingerprint,
                        "candidate_pool_seed": int(
                            candidate_pool_seed
                        ),
                        "candidate_pool_local": list(
                            candidate_pool_local
                        ),
                        "candidate_pool_cache_hit": bool(
                            candidate_pool_cache_hit
                        ),
                        "candidate_pool_cache_path": str(bundle_path),
                        "candidate_pool_cache_key": str(bundle_key),
                    })

    return subgraph_configs, summary_rows


# ------------------------------------------------------------
# Stage B helpers: one cached labeling configuration
# ------------------------------------------------------------

def _make_label_cache_spec(
    subgraph_config: dict,
    training_candidate_size: int,
    scoring_mode: str,
    candidate_pool_seed: int,
    mining_seed: int,
    candidates_local: list[tuple[int, int]],
) -> dict:
    return {
        "cache_version": RQ3_LABEL_CACHE_VERSION,
        "context_fingerprint": subgraph_config[
            "context_fingerprint"
        ],
        "victim_seed": int(subgraph_config["victim_seed"]),
        "subgraph_method": str(
            subgraph_config["subgraph_method"]
        ),
        "subgraph_seed": int(subgraph_config["subgraph_seed"]),
        "construction_seed": int(
            subgraph_config["construction_seed"]
        ),
        "subgraph_fraction_requested": float(
            subgraph_config["subgraph_fraction_requested"]
        ),
        "subgraph_fraction_actual": float(
            subgraph_config["subgraph_fraction_actual"]
        ),
        "source_nodes_sha256": _nodes_fingerprint(
            subgraph_config["source_nodes_global"]
        ),
        "core_nodes_sha256": _nodes_fingerprint(
            subgraph_config["core_nodes_global"]
        ),
        "n_core_nodes": int(subgraph_config["n_core_nodes"]),
        "training_candidate_size": int(training_candidate_size),
        "candidate_pool_seed": int(candidate_pool_seed),
        "candidate_pairs_sha256": _pairs_fingerprint(
            candidates_local
        ),
        "scoring_mode": str(scoring_mode),
        "mining_seed": int(mining_seed),
        "subset_fraction": float(SUBSET_FRACTION),
        "n_subsets": int(N_SUBSETS),
        "endpoint_k_samples": None,
        "endpoint_require_correct_to_incorrect": True,
        "two_hop_k_samples": None,
        "h": 0,
    }


def _rq3_scope_labels(subgraph_config: dict) -> tuple[str, str]:
    """Return victim-query and selector-training scope labels."""
    n_global = int(subgraph_config["context"]["n_nodes"])
    n_source = int(
        subgraph_config["source_nodes_global"].numel()
    )
    fraction = float(
        subgraph_config["subgraph_fraction_requested"]
    )
    is_whole_graph = (
        n_source == n_global
        and np.isclose(fraction, 1.0)
    )
    if is_whole_graph:
        return (
            "fixed_full_graph",
            "whole_graph_sampled_candidates",
        )
    return (
        "fixed_induced_source_subgraph",
        "induced_source_subgraph_only",
    )


def _build_run_from_result(
    *,
    subgraph_config: dict,
    local_context: dict,
    local_victim: torch.nn.Module,
    local_adj: torch.Tensor,
    full_adj: torch.Tensor,
    candidates_local: list[tuple[int, int]],
    candidates_global: list[tuple[int, int]],
    training_candidate_size: int,
    scoring_mode: str,
    candidate_pool_seed: int,
    mining_seed: int,
    candidate_config_id: str,
    result_local: dict,
    cache_hit: bool,
    cache_path: Path,
    cache_key: str,
) -> tuple[dict, dict]:
    context = subgraph_config["context"]
    local = subgraph_config["local"]
    source_nodes_global = subgraph_config["source_nodes_global"]
    core_nodes_global = subgraph_config["core_nodes_global"]
    n_core_nodes = int(subgraph_config["n_core_nodes"])
    target_nodes_global = subgraph_config["target_nodes_global"]
    victim_seed = int(subgraph_config["victim_seed"])
    subgraph_seed = int(subgraph_config["subgraph_seed"])
    subgraph_method = str(subgraph_config["subgraph_method"])
    fraction = float(
        subgraph_config["subgraph_fraction_requested"]
    )
    (
        victim_query_scope,
        selector_training_scope,
    ) = _rq3_scope_labels(subgraph_config)

    observed_mask = _rq3_as_numpy(
        result_local["observed_mask"],
        dtype=bool,
    )
    observed_indices = np.flatnonzero(observed_mask)
    if observed_indices.size == 0:
        raise RuntimeError(
            "No source-subgraph candidate received a label."
        )

    scored_candidates_local = [
        candidates_local[index]
        for index in observed_indices
    ]
    scored_candidates_global = [
        candidates_global[index]
        for index in observed_indices
    ]

    src_local = np.asarray(
        [u for u, _ in scored_candidates_local],
        dtype=np.int64,
    )
    dst_local = np.asarray(
        [v for _, v in scored_candidates_local],
        dtype=np.int64,
    )
    src_global = np.asarray(
        [u for u, _ in scored_candidates_global],
        dtype=np.int64,
    )
    dst_global = np.asarray(
        [v for _, v in scored_candidates_global],
        dtype=np.int64,
    )

    score_raw = _rq3_as_numpy(
        result_local["score_raw"],
        dtype=np.float64,
    )[observed_indices]
    score_norm = _rq3_as_numpy(
        result_local["score_norm"],
        dtype=np.float32,
    )[observed_indices]
    score_percentile = _rq3_as_numpy(
        result_local["score_percentile"],
        dtype=np.float32,
    )[observed_indices]
    exists = _rq3_as_numpy(
        result_local["exists"],
        dtype=np.float32,
    )[observed_indices]
    inclusion_count = _rq3_as_numpy(
        result_local.get(
            "inclusion_count",
            np.ones(
                int(training_candidate_size),
                dtype=np.int64,
            ),
        ),
        dtype=np.int64,
    )[observed_indices]

    n_delete_candidates = sum(
        bool(local_adj[u, v].item() > 0.5)
        for u, v in candidates_local
    )
    n_add_candidates = (
        int(training_candidate_size)
        - int(n_delete_candidates)
    )
    n_positive_labels = int((score_norm > 0).sum())

    result_local = dict(result_local)
    result_local.update({
        "victim_query_scope": victim_query_scope,
        "selector_training_scope": selector_training_scope,
        "source_nodes_global": source_nodes_global.clone(),
        "core_nodes_global": core_nodes_global.clone(),
        "n_core_nodes": int(n_core_nodes),
        "n_context_only_nodes": int(
            source_nodes_global.numel() - n_core_nodes
        ),
        "candidate_pairs_local": candidates_local,
        "candidate_pairs_global": candidates_global,
        "subgraph_method": subgraph_method,
        "subgraph_seed": subgraph_seed,
        "construction_seed": int(
            subgraph_config["construction_seed"]
        ),
        "subgraph_fraction_requested": fraction,
        "subgraph_fraction_actual": float(
            local["n_local"] / int(context["n_nodes"])
        ),
        "training_candidate_size": int(training_candidate_size),
        "candidate_pool_seed": int(candidate_pool_seed),
        "mining_seed": int(mining_seed),
        "label_cache_hit": bool(cache_hit),
        "label_cache_path": str(cache_path),
        "label_cache_key": str(cache_key),
        "candidate_pool_cache_hit": bool(
            subgraph_config["candidate_pool_cache_hit"]
        ),
        "candidate_pool_cache_path": str(
            subgraph_config["candidate_pool_cache_path"]
        ),
        "candidate_pool_cache_key": str(
            subgraph_config["candidate_pool_cache_key"]
        ),
    })

    run = {
        "seed": victim_seed,
        "context": local_context,
        "full_context": context,
        "local_victim": local_victim,
        "adj_orig": local_adj,
        "adj_orig_full": full_adj,

        "candidate_config_id": candidate_config_id,
        "candidate_set_size": int(training_candidate_size),
        "training_candidate_size": int(training_candidate_size),
        "subgraph_method": subgraph_method,
        "subgraph_seed": subgraph_seed,
        "construction_seed": int(
            subgraph_config["construction_seed"]
        ),
        "subgraph_fraction_requested": fraction,
        "subgraph_fraction_actual": float(
            local["n_local"] / int(context["n_nodes"])
        ),
        "candidate_pool_seed": int(candidate_pool_seed),
        "mining_seed": int(mining_seed),
        "label_cache_hit": bool(cache_hit),
        "label_cache_path": str(cache_path),
        "label_cache_key": str(cache_key),
        "candidate_pool_cache_hit": bool(
            subgraph_config["candidate_pool_cache_hit"]
        ),
        "candidate_pool_cache_path": str(
            subgraph_config["candidate_pool_cache_path"]
        ),
        "candidate_pool_cache_key": str(
            subgraph_config["candidate_pool_cache_key"]
        ),

        "prbcd_candidate_fraction": 0.0,
        "actual_prbcd_fraction": 0.0,
        "budget": 0,
        "prbcd_budget_fraction": 0.0,
        "n_prbcd_requested": 0,
        "n_prbcd_mined_unique": 0,
        "n_prbcd_selected": 0,
        "n_random_candidates": int(training_candidate_size),
        "n_add_candidates": int(n_add_candidates),
        "n_delete_candidates": int(n_delete_candidates),

        "schema_version": result_local["schema_version"],
        "score_schema_version": result_local[
            "score_schema_version"
        ],
        "target_kind": result_local["target_kind"],
        "score_definition": result_local["score_definition"],
        "score_aggregation": result_local["score_aggregation"],
        "score_normalization": result_local[
            "score_normalization"
        ],
        "score_clipping": result_local.get("score_clipping"),

        "scoring_mode": scoring_mode,
        "endpoint_mining_hop": 0,
        "victim_query_scope": victim_query_scope,
        "selector_training_scope": selector_training_scope,

        "source_nodes_global": source_nodes_global,
        "core_nodes_global": core_nodes_global,
        "n_core_nodes": int(n_core_nodes),
        "n_context_only_nodes": int(
            source_nodes_global.numel() - n_core_nodes
        ),
        "target_nodes_global": target_nodes_global,
        "global_to_local": local["global_to_local"],
        "n_source_nodes": int(source_nodes_global.numel()),
        "n_target_nodes": int(target_nodes_global.numel()),
        "subgraph_stats": subgraph_config["subgraph_stats"],
        "sampler_metadata": subgraph_config["sampler_metadata"],

        "candidate_pool": candidates_local,
        "candidate_pool_local": candidates_local,
        "candidate_pool_global": candidates_global,
        "candidates": scored_candidates_local,
        "candidates_local": scored_candidates_local,
        "candidates_global": scored_candidates_global,

        "n_candidates_total": int(training_candidate_size),
        "n_observed_candidates": int(observed_indices.size),
        "n_unobserved_candidates": int(
            training_candidate_size - observed_indices.size
        ),
        "observed_fraction": float(
            observed_indices.size / training_candidate_size
        ),

        "mining_result": result_local,
        "observed_mask_all": observed_mask,
        "observed_indices": observed_indices,
        "score_raw": score_raw,
        "score_norm": score_norm,
        "score_percentile": score_percentile,
        "inclusion_count": inclusion_count,
        "labels_raw": score_raw,
        "labels_norm": score_norm,
        "labels_changed": torch.tensor(
            score_norm,
            dtype=torch.float32,
            device=context["device"],
        ),

        # Local IDs are used by local diagnostics/training.
        "src": torch.tensor(
            src_local,
            dtype=torch.long,
            device=context["device"],
        ),
        "dst": torch.tensor(
            dst_local,
            dtype=torch.long,
            device=context["device"],
        ),
        # Global IDs are retained for whole-graph transfer.
        "src_global": torch.tensor(
            src_global,
            dtype=torch.long,
            device=context["device"],
        ),
        "dst_global": torch.tensor(
            dst_global,
            dtype=torch.long,
            device=context["device"],
        ),
        "exists": torch.tensor(
            exists,
            dtype=torch.float32,
            device=context["device"],
        ),
        "clean_accuracy": float(result_local["clean_accuracy"]),
        "n_positive_labels": n_positive_labels,
    }

    summary_row = {
        "seed": victim_seed,
        "candidate_config_id": candidate_config_id,
        "subgraph_method": subgraph_method,
        "scoring_mode": scoring_mode,
        "subgraph_seed": subgraph_seed,
        "construction_seed": int(
            subgraph_config["construction_seed"]
        ),
        "subgraph_fraction_requested": fraction,
        "subgraph_fraction_actual": run[
            "subgraph_fraction_actual"
        ],
        "n_nodes_global": int(context["n_nodes"]),
        "n_source_nodes": run["n_source_nodes"],
        "n_core_nodes": run["n_core_nodes"],
        "n_context_only_nodes": run["n_context_only_nodes"],
        "n_target_nodes": run["n_target_nodes"],
        "training_candidate_size": int(training_candidate_size),
        "candidate_pool_seed": int(candidate_pool_seed),
        "mining_seed": int(mining_seed),
        "n_observed_candidates": run["n_observed_candidates"],
        "n_add_candidates": int(n_add_candidates),
        "n_delete_candidates": int(n_delete_candidates),
        "n_positive_labels": n_positive_labels,
        "positive_fraction": n_positive_labels
        / max(1, run["n_observed_candidates"]),
        "local_clean_accuracy": run["clean_accuracy"],
        "victim_query_scope": victim_query_scope,
        "selector_training_scope": selector_training_scope,
        "label_cache_hit": bool(cache_hit),
        "label_cache_path": str(cache_path),
        "candidate_pool_cache_hit": bool(
            subgraph_config["candidate_pool_cache_hit"]
        ),
        "candidate_pool_cache_path": str(
            subgraph_config["candidate_pool_cache_path"]
        ),
    }

    return run, summary_row



# ------------------------------------------------------------
# RQ2 -> RQ3 whole-graph adapter
# ------------------------------------------------------------

def _reuse_rq2_full_graph_run(
    *,
    rq2_run: dict,
    subgraph_config: dict,
    local_context: dict,
    local_victim: torch.nn.Module,
    local_adj: torch.Tensor,
    full_adj: torch.Tensor,
    training_candidate_size: int,
    scoring_mode: str,
) -> tuple[dict, dict]:
    """Adapt one cached RQ2 mining result to the RQ3 run schema."""
    context = subgraph_config["context"]
    victim_seed = int(subgraph_config["victim_seed"])
    rq2_config_id = str(rq2_run["candidate_config_id"])
    rq2_cache_path = _rq3_rq2_cache_path(rq2_run)

    candidates = _rq3_rq2_candidate_pool(rq2_run)
    if len(candidates) != int(training_candidate_size):
        raise RuntimeError(
            "The selected RQ2 candidate pool does not match the requested "
            f"training size: {len(candidates)} != {training_candidate_size}."
        )

    result_local = rq2_run.get("mining_result")
    if not isinstance(result_local, dict):
        raise RuntimeError(
            "The matching RQ2 run has no mining_result dictionary."
        )

    rq3_candidate_config_id = (
        "rq3WholeGraph"
        f"__rq2Cfg-{_rq3_safe_value(rq2_config_id)}"
        f"__trainCandN-{_rq3_safe_value(training_candidate_size)}"
    )
    reuse_key = hashlib.sha256(
        (
            f"rq2|{victim_seed}|{rq2_config_id}|{scoring_mode}|"
            f"{training_candidate_size}|{rq2_run.get('candidate_hash', '')}"
        ).encode("utf-8")
    ).hexdigest()

    run, summary_row = _build_run_from_result(
        subgraph_config=subgraph_config,
        local_context=local_context,
        local_victim=local_victim,
        local_adj=local_adj,
        full_adj=full_adj,
        candidates_local=candidates,
        candidates_global=candidates,
        training_candidate_size=training_candidate_size,
        scoring_mode=scoring_mode,
        candidate_pool_seed=_rq3_stable_seed(
            "rq2_reused_candidate_pool",
            victim_seed,
            rq2_config_id,
        ),
        mining_seed=int(rq2_run.get("seed", victim_seed)),
        candidate_config_id=rq3_candidate_config_id,
        result_local=result_local,
        cache_hit=True,
        cache_path=(
            rq2_cache_path
            if rq2_cache_path is not None
            else Path("<in-memory-rq2-result>")
        ),
        cache_key=reuse_key,
    )

    # Restore RQ2 candidate-construction metadata that the generic RQ3 builder
    # intentionally initializes to the reduced-subgraph defaults.
    rq2_metadata_keys = [
        "prbcd_candidate_fraction",
        "actual_prbcd_fraction",
        "budget",
        "prbcd_budget_fraction",
        "n_prbcd_requested",
        "n_prbcd_mined_unique",
        "n_prbcd_selected",
        "n_random_candidates",
        "n_add_candidates",
        "n_delete_candidates",
        "candidate_hash",
    ]
    for key in rq2_metadata_keys:
        if key in rq2_run:
            run[key] = rq2_run[key]

    run.update({
        "label_source": "rq2_full_graph_cache",
        "cache_source": "rq2",
        "rq2_candidate_config_id": rq2_config_id,
        "rq2_cache_path": (
            str(rq2_cache_path) if rq2_cache_path is not None else ""
        ),
        "rq2_cache_status": str(rq2_run.get("cache_status", "")),
        "rq2_candidate_pool": list(rq2_run.get("candidate_pool", [])),
        # Preserve the exact RQ2 endpoint-label definition. In the current
        # RQ2 caches endpoint labels use h=2, whereas subset_accuracy_drop
        # uses h=0.
        "endpoint_mining_hop": int(
            rq2_run.get("endpoint_mining_hop", 0)
        ),
        "subgraph_method": "whole_graph",
        "subgraph_fraction_requested": 1.0,
        "subgraph_fraction_actual": 1.0,
        "victim_query_scope": "fixed_full_graph",
        "selector_training_scope": "whole_graph_sampled_candidates",
    })

    summary_row.update({
        "label_source": "rq2_full_graph_cache",
        "cache_source": "rq2",
        "rq2_candidate_config_id": rq2_config_id,
        "rq2_cache_path": (
            str(rq2_cache_path) if rq2_cache_path is not None else ""
        ),
        "rq2_cache_status": str(rq2_run.get("cache_status", "")),
        "endpoint_mining_hop": int(
            rq2_run.get("endpoint_mining_hop", 0)
        ),
        "prbcd_candidate_fraction": float(
            rq2_run.get("prbcd_candidate_fraction", 0.0)
        ),
        "actual_prbcd_fraction": float(
            rq2_run.get("actual_prbcd_fraction", 0.0)
        ),
    })

    return run, summary_row


# ------------------------------------------------------------
# Stage B: label the complete sweep, using cache where possible
# ------------------------------------------------------------

def label_rq3_subgraph_sweep(
    subgraph_configs: list[dict],
) -> tuple[list[dict], list[dict], dict]:
    mining_runs_output = []
    summary_rows = []
    local_contexts_by_config = {}
    full_adj_by_context_id = {}

    max_training_candidate_size = max(
        RQ3_SUBGRAPH_TRAINING_CANDIDATE_SIZES
    )

    for subgraph_config in subgraph_configs:
        context = subgraph_config["context"]
        local = subgraph_config["local"]
        victim_seed = int(subgraph_config["victim_seed"])
        subgraph_seed = int(subgraph_config["subgraph_seed"])
        subgraph_method = str(
            subgraph_config["subgraph_method"]
        )
        fraction = float(
            subgraph_config["subgraph_fraction_requested"]
        )
        (
            victim_query_scope,
            selector_training_scope,
        ) = _rq3_scope_labels(subgraph_config)
        source_nodes_global = subgraph_config[
            "source_nodes_global"
        ]
        core_nodes_global = subgraph_config[
            "core_nodes_global"
        ]
        n_core_nodes = int(subgraph_config["n_core_nodes"])
        target_nodes_global = subgraph_config[
            "target_nodes_global"
        ]

        activate_seed_context(context)
        set_global_seed(victim_seed)

        if local["eval_idx"].numel() == 0:
            raise RuntimeError(
                "The source subgraph contains no test nodes, so candidate "
                "labeling cannot run. Configuration: "
                f"victim_seed={victim_seed}, method={subgraph_method}, "
                f"subgraph_seed={subgraph_seed}, fraction={fraction}."
            )

        local_victim = _FreshLocalVictim(
            context["model"]
        ).to(context["device"]).eval()

        local_adj = _dense_adj(
            local["edge_index"],
            local["n_local"],
            device=context["device"],
        )

        context_id = id(context)
        if context_id not in full_adj_by_context_id:
            full_adj_by_context_id[context_id] = _dense_adj(
                context["edge_index"],
                int(context["n_nodes"]),
                device=context["device"],
            )
        full_adj = full_adj_by_context_id[context_id]

        local_context = {
            "seed": victim_seed,
            "device": context["device"],
            "n_nodes": int(local["n_local"]),
            "attr": local["attr"],
            "edge_index": local["edge_index"],
            "labels": local["labels"],
            "idx_test": local["eval_idx"],
            "model": local_victim,
            "source_nodes_global": source_nodes_global,
            "core_nodes_global": core_nodes_global,
            "n_core_nodes": int(n_core_nodes),
            "n_context_only_nodes": int(
                source_nodes_global.numel() - n_core_nodes
            ),
            "global_to_local": local["global_to_local"],
            "target_nodes_global": target_nodes_global,
            "full_context": context,
            "subgraph_method": subgraph_method,
            "subgraph_seed": subgraph_seed,
            "construction_seed": int(
                subgraph_config["construction_seed"]
            ),
            "subgraph_fraction_requested": fraction,
            "subgraph_fraction_actual": float(
                local["n_local"] / int(context["n_nodes"])
            ),
            "sampler_metadata": subgraph_config[
                "sampler_metadata"
            ],
            "subgraph_stats": subgraph_config["subgraph_stats"],
            "selector_training_scope": selector_training_scope,
            "victim_query_scope": victim_query_scope,
        }
        local_contexts_by_config[
            subgraph_config["config_key"]
        ] = local_context

        # Stage A already loaded or created the maximum candidate pool.
        # Smaller training sets are deterministic prefixes, so no candidate
        # resampling occurs here.
        candidate_pool_seed = int(
            subgraph_config["candidate_pool_seed"]
        )
        candidate_pool_local = list(
            subgraph_config["candidate_pool_local"]
        )
        if len(candidate_pool_local) < int(
            max_training_candidate_size
        ):
            raise RuntimeError(
                "Cached candidate pool is smaller than the configured "
                "maximum training-candidate size."
            )

        for training_candidate_size in (
            RQ3_SUBGRAPH_TRAINING_CANDIDATE_SIZES
        ):
            training_candidate_size = int(training_candidate_size)
            candidates_local = candidate_pool_local[
                :training_candidate_size
            ]
            if any(
                not (0 <= u < v < n_core_nodes)
                for u, v in candidates_local
            ):
                raise AssertionError(
                    "Candidate pool contains a context-only endpoint."
                )
            candidates_global = [
                (
                    int(source_nodes_global[u_local]),
                    int(source_nodes_global[v_local]),
                )
                for u_local, v_local in candidates_local
            ]

            candidate_config_id = (
                "rq3Subgraph"
                f"__method-{_rq3_safe_value(subgraph_method)}"
                f"__nodeFrac-{_rq3_safe_value(fraction)}"
                f"__trainCandN-{_rq3_safe_value(training_candidate_size)}"
                f"__subgraphSeed-{_rq3_safe_value(subgraph_seed)}"
            )

            for scoring_mode in RQ3_SUBGRAPH_SCORING_MODES:
                # The canonical full-graph condition is identical to RQ2.
                # Reuse its exact candidate ordering and labels when an
                # unambiguous cache-backed run is available.
                if (
                    RQ3_REUSE_RQ2_WHOLE_GRAPH
                    and subgraph_method == "whole_graph"
                    and np.isclose(fraction, 1.0)
                ):
                    rq2_run = _find_cached_rq2_full_graph_run(
                        victim_seed=victim_seed,
                        training_candidate_size=training_candidate_size,
                        scoring_mode=scoring_mode,
                        endpoint_h=(
                            RQ3_RQ2_ENDPOINT_MINING_HOP
                            if scoring_mode == "endpoint"
                            else None
                        ),
                    )

                    if rq2_run is not None:
                        run, summary_row = _reuse_rq2_full_graph_run(
                            rq2_run=rq2_run,
                            subgraph_config=subgraph_config,
                            local_context=local_context,
                            local_victim=local_victim,
                            local_adj=local_adj,
                            full_adj=full_adj,
                            training_candidate_size=(
                                training_candidate_size
                            ),
                            scoring_mode=scoring_mode,
                        )
                        mining_runs_output.append(run)
                        summary_rows.append(summary_row)

                        print("=" * 80)
                        print(
                            "RQ3 whole-graph labels reused from RQ2\n"
                            f"  victim seed:          {victim_seed}\n"
                            f"  RQ2 candidate config: "
                            f"{rq2_run['candidate_config_id']}\n"
                            f"  training candidates:  "
                            f"{training_candidate_size}\n"
                            f"  scoring mode:         {scoring_mode}\n"
                            f"  endpoint hop:         "
                            f"{rq2_run.get('endpoint_mining_hop', 0)}\n"
                            f"  RQ2 cache path:       "
                            f"{rq2_run.get('cache_path', '')}"
                        )

                        _atomic_write_dataframe(
                            pd.DataFrame(summary_rows),
                            RQ3_LABEL_MANIFEST_CSV,
                        )
                        continue

                    available = [
                        {
                            "seed": run.get("seed"),
                            "size": run.get("candidate_set_size"),
                            "mode": run.get("scoring_mode"),
                            "h": run.get("endpoint_mining_hop"),
                            "config": run.get("candidate_config_id"),
                            "cache_status": run.get("cache_status"),
                            "pool_len": len(run.get("candidate_pool", [])),
                        }
                        for run in RQ2_FULL_GRAPH_MINING_RUNS
                        if int(run.get("seed", -1)) == int(victim_seed)
                        and int(run.get("candidate_set_size", -1))
                        == int(training_candidate_size)
                    ]
                    warnings.warn(
                        "No unambiguous cache-backed RQ2 run matched the "
                        "whole-graph RQ3 condition. Falling back to the "
                        "ordinary RQ3 candidate pool and labeling cache. "
                        f"seed={victim_seed}, "
                        f"size={training_candidate_size}, "
                        f"mode={scoring_mode}. "
                        f"RQ2 candidates at this seed/size: {available}"
                    )

                mining_seed = _rq3_stable_seed(
                    "rq3_label_mining",
                    victim_seed,
                    subgraph_method,
                    subgraph_seed,
                    f"{fraction:.12g}",
                    training_candidate_size,
                    scoring_mode,
                )

                cache_spec = _make_label_cache_spec(
                    subgraph_config=subgraph_config,
                    training_candidate_size=training_candidate_size,
                    scoring_mode=scoring_mode,
                    candidate_pool_seed=candidate_pool_seed,
                    mining_seed=mining_seed,
                    candidates_local=candidates_local,
                )
                cache_key = _cache_key_from_spec(cache_spec)
                cache_path = _cache_path_for_spec(
                    cache_spec,
                    cache_key,
                )

                result_local = _try_load_label_cache(
                    cache_path=cache_path,
                    cache_key=cache_key,
                    source_nodes_global=source_nodes_global,
                    candidates_local=candidates_local,
                    candidates_global=candidates_global,
                )
                cache_hit = result_local is not None

                print("=" * 80)
                print(
                    "RQ3 candidate labeling\n"
                    f"  victim seed:             {victim_seed}\n"
                    f"  subgraph method:         {subgraph_method}\n"
                    f"  subgraph seed:           {subgraph_seed}\n"
                    f"  requested fraction:      {fraction}\n"
                    f"  source nodes:            "
                    f"{local['n_local']}/{context['n_nodes']}\n"
                    f"  training candidates:     "
                    f"{training_candidate_size}\n"
                    f"  scoring mode:            {scoring_mode}\n"
                    f"  local test nodes:        "
                    f"{local['eval_idx'].numel()}\n"
                    f"  label cache:             "
                    f"{'HIT' if cache_hit else 'MISS'}\n"
                    f"  cache path:              {cache_path}"
                )

                if not cache_hit:
                    result_local = mine_candidate_edge_scores(
                        model=local_victim,
                        attr=local["attr"],
                        edge_index=local["edge_index"],
                        labels=local["labels"],
                        eval_idx=local["eval_idx"],
                        candidates=candidates_local,
                        n_nodes=local["n_local"],
                        mode=scoring_mode,
                        subset_fraction=SUBSET_FRACTION,
                        n_subsets=N_SUBSETS,
                        endpoint_k_samples=None,
                        endpoint_require_correct_to_incorrect=True,
                        two_hop_k_samples=None,
                        h=0,
                        seed=mining_seed,
                        device=context["device"],
                        verbose=True,
                    )
                    _save_label_cache(
                        cache_path=cache_path,
                        cache_key=cache_key,
                        cache_spec=cache_spec,
                        source_nodes_global=source_nodes_global,
                        candidates_local=candidates_local,
                        candidates_global=candidates_global,
                        result_local=result_local,
                    )
                    print(f"[CACHE] Saved labeled candidates: {cache_path}")
                else:
                    print(f"[CACHE] Loaded labeled candidates: {cache_path}")

                run, summary_row = _build_run_from_result(
                    subgraph_config=subgraph_config,
                    local_context=local_context,
                    local_victim=local_victim,
                    local_adj=local_adj,
                    full_adj=full_adj,
                    candidates_local=candidates_local,
                    candidates_global=candidates_global,
                    training_candidate_size=training_candidate_size,
                    scoring_mode=scoring_mode,
                    candidate_pool_seed=candidate_pool_seed,
                    mining_seed=mining_seed,
                    candidate_config_id=candidate_config_id,
                    result_local=result_local,
                    cache_hit=cache_hit,
                    cache_path=cache_path,
                    cache_key=cache_key,
                )
                mining_runs_output.append(run)
                summary_rows.append(summary_row)

                # Persist partial progress after every exact configuration.
                # If the kernel or Slurm job stops, the manifest still records
                # every completed/cache-loaded candidate set.
                _atomic_write_dataframe(
                    pd.DataFrame(summary_rows),
                    RQ3_LABEL_MANIFEST_CSV,
                )

    return mining_runs_output, summary_rows, local_contexts_by_config


# ============================================================
# Execute Stage A: subgraph construction only
# ============================================================

RQ3_SUBGRAPH_CONFIGS, rq3_subgraph_construction_summary_rows = (
    construct_rq3_subgraph_sweep(SEED_CONTEXTS)
)

if not RQ3_SUBGRAPH_CONFIGS:
    raise RuntimeError("No RQ3 source subgraphs were constructed.")

RQ3_SOURCE_NODES_BY_CONFIG = {
    config["config_key"]: config["source_nodes_global"].clone()
    for config in RQ3_SUBGRAPH_CONFIGS
}
RQ3_CORE_NODES_BY_CONFIG = {
    config["config_key"]: config["core_nodes_global"].clone()
    for config in RQ3_SUBGRAPH_CONFIGS
}
RQ3_TARGET_NODES_BY_CONFIG = {
    config["config_key"]: config["target_nodes_global"].clone()
    for config in RQ3_SUBGRAPH_CONFIGS
}

rq3_subgraph_construction_summary_df = pd.DataFrame(
    rq3_subgraph_construction_summary_rows
)
display(rq3_subgraph_construction_summary_df)
_atomic_write_dataframe(
    rq3_subgraph_construction_summary_df,
    RQ3_SUBGRAPH_CONSTRUCTION_SUMMARY_CSV,
)

_rq3_constructed_methods = sorted({
    config["subgraph_method"]
    for config in RQ3_SUBGRAPH_CONFIGS
})
print(
    f"Constructed {len(RQ3_SUBGRAPH_CONFIGS)} RQ3 source graphs "
    f"across methods {_rq3_constructed_methods}."
)


# ============================================================
# Execute Stage B: cached candidate labeling only
# ============================================================

(
    RQ3_SUBGRAPH_MINING_RUNS,
    rq3_subgraph_mining_summary_rows,
    RQ3_LOCAL_CONTEXTS_BY_CONFIG,
) = label_rq3_subgraph_sweep(RQ3_SUBGRAPH_CONFIGS)

if not RQ3_SUBGRAPH_MINING_RUNS:
    raise RuntimeError("No RQ3 subgraph mining runs were created.")

RQ3_PRIMARY_SUBGRAPH_MINING_RUNS = [
    run
    for run in RQ3_SUBGRAPH_MINING_RUNS
    if run["subgraph_method"] == RQ3_PRIMARY_EFFECTIVE_SUBGRAPH_METHOD
    and np.isclose(
        float(run["subgraph_fraction_requested"]),
        float(RQ3_PRIMARY_SUBGRAPH_FRACTION),
    )
    and int(run["training_candidate_size"])
    == int(RQ3_PRIMARY_TRAINING_CANDIDATE_SIZE)
    and int(run["subgraph_seed"])
    == int(RQ3_PRIMARY_SUBGRAPH_SEED)
]

if not RQ3_PRIMARY_SUBGRAPH_MINING_RUNS:
    raise RuntimeError(
        "The configured primary RQ3 subgraph combination produced no runs."
    )

rq3_subgraph_mining_summary_df = pd.DataFrame(
    rq3_subgraph_mining_summary_rows
)
display(rq3_subgraph_mining_summary_df)
_atomic_write_dataframe(
    rq3_subgraph_mining_summary_df,
    RQ3_LABEL_MANIFEST_CSV,
)

n_cache_hits = int(
    rq3_subgraph_mining_summary_df["label_cache_hit"].sum()
)
print(
    f"Created {len(RQ3_SUBGRAPH_MINING_RUNS)} total RQ3 subgraph "
    f"mining runs and {len(RQ3_PRIMARY_SUBGRAPH_MINING_RUNS)} primary "
    f"runs. Label cache hits: {n_cache_hits}/"
    f"{len(RQ3_SUBGRAPH_MINING_RUNS)}.\n"
    f"The original {len(RQ2_FULL_GRAPH_MINING_RUNS)} RQ2 mining runs "
    "remain in `mining_runs` and `RQ2_FULL_GRAPH_MINING_RUNS`.\n"
    f"Candidate-pool cache: {RQ3_SUBGRAPH_BUNDLE_CACHE_DIR}\n"
    f"Label cache:          {RQ3_LABEL_CACHE_DIR}\n"
    f"Progress manifest:    {RQ3_LABEL_MANIFEST_CSV}"
)


In [ ]:
# ============================================================
# RQ3 persistent mining-cache status
#
# Normal restart:
#   RQ3_FORCE_RESAMPLE_CANDIDATES = False
#   RQ3_FORCE_RELABEL = False
#
# Selective invalidation:
#   - set RQ3_FORCE_RESAMPLE_CANDIDATES=True to rebuild source nodes and
#     candidate pools;
#   - set RQ3_FORCE_RELABEL=True to keep the candidate pools but recompute
#     all victim-query labels.
#
# Run the construction/mining cell above again after changing either flag.
# ============================================================

from pathlib import Path
from IPython.display import display
import pandas as pd

print("RQ3_FORCE_RESAMPLE_CANDIDATES:", RQ3_FORCE_RESAMPLE_CANDIDATES)
print("RQ3_FORCE_RELABEL:", RQ3_FORCE_RELABEL)
print("Candidate-pool cache:", RQ3_SUBGRAPH_BUNDLE_CACHE_DIR.resolve())
print("Label cache:", RQ3_LABEL_CACHE_DIR.resolve())
print("Construction summary:", RQ3_SUBGRAPH_CONSTRUCTION_SUMMARY_CSV.resolve())
print("Label manifest:", RQ3_LABEL_MANIFEST_CSV.resolve())

pool_files = sorted(Path(RQ3_SUBGRAPH_BUNDLE_CACHE_DIR).glob("*.pt"))
label_files = sorted(Path(RQ3_LABEL_CACHE_DIR).glob("*.pt"))

print(f"Cached source-subgraph candidate pools: {len(pool_files)}")
print(f"Cached labeled candidate sets:          {len(label_files)}")

if Path(RQ3_SUBGRAPH_CONSTRUCTION_SUMMARY_CSV).exists():
    construction_manifest = pd.read_csv(
        RQ3_SUBGRAPH_CONSTRUCTION_SUMMARY_CSV
    )
    display(construction_manifest)

if Path(RQ3_LABEL_MANIFEST_CSV).exists():
    label_manifest = pd.read_csv(RQ3_LABEL_MANIFEST_CSV)
    display(label_manifest)


In [ ]:
# ============================================================
# Train selector GNN exclusively on the induced source subgraph
#
# Training visibility:
#   - node features: source nodes only
#   - structural edges: induced source-subgraph edges only
#   - labeled pairs: source-subgraph candidates only
#
# Retained for later transfer:
#   - full_context
#   - source_nodes_global
#   - edge_index_lab_global
# ============================================================

from helpers.selector_pipeline_helpers import train_selector

from IPython.display import display
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ============================================================
# Configuration
# ============================================================

LP_ROC_DATA_OUT_DIR = (
    Path("extendedPlotting")
    / "lp_training_roc_data"
)
LP_ROC_DATA_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# A continuous candidate is considered harmful whenever its
# normalized score is strictly greater than zero.
ROC_POSITIVE_THRESHOLD = 0.0

ROC_PRED_BATCH_SIZE = 200_000


# Fallback in case this helper has not already been defined.
if "_safe_config_value" not in globals():

    def _safe_config_value(value):
        text = str(value)
        return (
            text.replace(" ", "-")
            .replace("/", "-")
            .replace("\\", "-")
            .replace(":", "-")
            .replace(".", "p")
        )


# ============================================================
# Label-configuration helpers
# ============================================================

def _resolve_label_modes():
    configured = globals().get(
        "LABEL_MODES",
        [
            globals().get(
                "LABEL_MODE",
                "continuous",
            )
        ],
    )

    if isinstance(configured, str):
        configured = [configured]

    label_modes = [
        str(mode)
        for mode in configured
    ]

    if not label_modes:
        raise ValueError(
            "LABEL_MODES must contain at least one "
            "label mode."
        )

    allowed = {
        "continuous",
        "projected",
    }

    unknown = sorted(
        set(label_modes) - allowed
    )

    if unknown:
        raise ValueError(
            f"Unknown LABEL_MODES value(s): {unknown}. "
            f"Allowed: {sorted(allowed)}"
        )

    return label_modes


def _resolve_extreme_fractions():
    configured = globals().get(
        "EXTREME_FRACTIONS",
        [
            globals().get(
                "EXTREME_FRACTION",
                0.2,
            )
        ],
    )

    if isinstance(
        configured,
        (int, float),
    ):
        configured = [configured]

    fractions = [
        float(value)
        for value in configured
    ]

    if not fractions:
        raise ValueError(
            "EXTREME_FRACTIONS must contain "
            "at least one value."
        )

    for fraction in fractions:
        if not 0.0 < fraction <= 0.5:
            raise ValueError(
                "Every EXTREME_FRACTIONS value "
                "must be in (0, 0.5]."
            )

    return fractions


def _label_training_configs_for_scoring_run(
    scoring_mode: str,
):
    """
    Endpoint scoring:
        one binary/continuous-label selector.

    Subset-accuracy-drop scoring:
        all configured continuous/projected variants.
    """
    if scoring_mode == "endpoint":
        return [{
            "label_mode": "continuous",
            "extreme_fraction": None,
        }]

    if scoring_mode == "subset_accuracy_drop":
        configs = []

        for label_mode in _resolve_label_modes():

            if label_mode == "continuous":
                configs.append({
                    "label_mode": "continuous",
                    "extreme_fraction": None,
                })

            elif label_mode == "projected":
                for fraction in (
                    _resolve_extreme_fractions()
                ):
                    configs.append({
                        "label_mode": "projected",
                        "extreme_fraction": float(
                            fraction
                        ),
                    })

            else:
                raise ValueError(
                    f"Unknown label mode: "
                    f"{label_mode}"
                )

        return configs

    raise ValueError(
        f"Unknown scoring_mode: {scoring_mode}"
    )


# ============================================================
# Subgraph reconstruction
# ============================================================

def _resolve_full_context(run: dict) -> dict:
    """
    Recover the original complete-graph context.

    This supports both versions of the source-mining cell:

    1. run["context"] is still the full graph;
    2. run["context"] is local and contains "full_context".
    """
    if isinstance(
        run.get("full_context"),
        dict,
    ):
        return run["full_context"]

    context = run["context"]

    if isinstance(
        context.get("full_context"),
        dict,
    ):
        return context["full_context"]

    return context


def _resolve_source_nodes(
    run: dict,
) -> torch.Tensor:
    """
    Return source-subgraph nodes as global node IDs.
    """
    source_nodes = run.get(
        "source_nodes_global"
    )

    if source_nodes is None:
        source_nodes = run["context"].get(
            "source_nodes_global"
        )

    if source_nodes is None:
        source_nodes = run["context"].get(
            "rq3_source_nodes_global"
        )

    if source_nodes is None:
        raise KeyError(
            "No source-node mapping found. Expected "
            "'source_nodes_global' in the run or context."
        )

    source_nodes = torch.as_tensor(
        source_nodes,
        dtype=torch.long,
    ).detach().cpu().view(-1)

    if source_nodes.numel() < 2:
        raise ValueError(
            "The selector source subgraph must "
            "contain at least two nodes."
        )

    if torch.unique(source_nodes).numel() != (
        source_nodes.numel()
    ):
        raise ValueError(
            "source_nodes_global contains duplicates."
        )

    return source_nodes


def _build_selector_subgraph_context(
    run: dict,
):
    """
    Reconstruct exactly the induced source graph used as the
    selector's training graph.

    Returns
    -------
    selector_context
        Local source-subgraph context.

    full_context
        Complete graph retained for later selector transfer.

    global_to_local
        Mapping from complete-graph IDs to source-local IDs.

    source_nodes_global
        Reverse mapping: local ID -> global ID.
    """
    full_context = _resolve_full_context(
        run
    )

    source_nodes_global = (
        _resolve_source_nodes(run)
    )

    full_attr = torch.as_tensor(
        full_context["attr"]
    )

    full_edge_index = torch.as_tensor(
        full_context["edge_index"],
        dtype=torch.long,
    )

    n_global = int(
        full_context.get(
            "n_nodes",
            full_attr.size(0),
        )
    )

    if full_attr.size(0) != n_global:
        raise ValueError(
            "Full feature matrix and n_nodes do "
            "not agree."
        )

    if (
        source_nodes_global.min() < 0
        or source_nodes_global.max() >= n_global
    ):
        raise ValueError(
            "source_nodes_global contains an "
            "invalid complete-graph node ID."
        )

    n_local = int(
        source_nodes_global.numel()
    )
    selector_training_scope = str(
        run.get(
            "selector_training_scope",
            (
                "whole_graph_sampled_candidates"
                if n_local == n_global
                else "induced_source_subgraph_only"
            ),
        )
    )

    global_to_local = torch.full(
        (n_global,),
        -1,
        dtype=torch.long,
    )

    global_to_local[
        source_nodes_global
    ] = torch.arange(
        n_local,
        dtype=torch.long,
    )

    # Work on CPU for reliable indexing, then move the
    # reconstructed local graph to the training device.
    edge_global_cpu = (
        full_edge_index
        .detach()
        .cpu()
        .long()
    )

    src_local_all = global_to_local[
        edge_global_cpu[0]
    ]

    dst_local_all = global_to_local[
        edge_global_cpu[1]
    ]

    inside_source = (
        (src_local_all >= 0)
        & (dst_local_all >= 0)
    )

    edge_index_local = torch.stack(
        [
            src_local_all[inside_source],
            dst_local_all[inside_source],
        ],
        dim=0,
    )

    device = full_context.get(
        "device",
        full_attr.device,
    )

    source_nodes_on_attr_device = (
        source_nodes_global.to(
            full_attr.device
        )
    )

    attr_local = full_attr[
        source_nodes_on_attr_device
    ].to(device)

    edge_index_local = (
        edge_index_local
        .long()
        .to(device)
        .contiguous()
    )

    labels_local = None

    if full_context.get("labels") is not None:
        full_labels = torch.as_tensor(
            full_context["labels"]
        )

        labels_local = full_labels[
            source_nodes_global.to(
                full_labels.device
            )
        ].to(device)

    # Map full-graph evaluation nodes to local IDs for
    # diagnostics only.
    idx_test_global = full_context.get(
        "idx_test",
        full_context.get(
            "test_idx",
            None,
        ),
    )

    if idx_test_global is None:
        idx_test_local = torch.empty(
            0,
            dtype=torch.long,
            device=device,
        )

    else:
        idx_test_global = torch.as_tensor(
            idx_test_global,
            dtype=torch.long,
        ).detach().cpu().view(-1)

        idx_test_local = global_to_local[
            idx_test_global
        ]

        idx_test_local = idx_test_local[
            idx_test_local >= 0
        ].to(device)

    selector_context = {
        "seed": int(run["seed"]),
        "device": device,
        "n_nodes": n_local,
        "attr": attr_local,
        "edge_index": edge_index_local,
        "labels": labels_local,
        "idx_test": idx_test_local,
        "model": run.get(
            "local_victim",
            run.get("context", {}).get("model"),
        ),

        # Mapping information.
        "source_nodes_global": (
            source_nodes_global
        ),
        "global_to_local": (
            global_to_local
        ),

        # Retained exclusively for later transfer/evaluation.
        "full_context": full_context,

        "selector_training_scope": (
            selector_training_scope
        ),
    }

    # Hard guarantees that the selector structure contains
    # no node outside the source subgraph.
    if edge_index_local.numel():
        if int(edge_index_local.min()) < 0:
            raise AssertionError(
                "Local structural graph contains a "
                "negative node ID."
            )

        if int(edge_index_local.max()) >= n_local:
            raise AssertionError(
                "Local structural graph contains a "
                "node outside the source subgraph."
            )

    if attr_local.size(0) != n_local:
        raise AssertionError(
            "Local feature matrix has the wrong "
            "number of nodes."
        )

    return (
        selector_context,
        full_context,
        global_to_local,
        source_nodes_global,
    )


def _resolve_local_and_global_candidate_ids(
    run: dict,
    *,
    source_nodes_global: torch.Tensor,
    global_to_local: torch.Tensor,
    n_global: int,
    n_local: int,
    device,
):
    """
    Return candidate endpoints in both ID systems.

    The original supplied source-mining cell stores run["src"] and
    run["dst"] as global IDs. This also supports a later version
    that explicitly stores src_global/dst_global while keeping
    run["src"]/run["dst"] local.
    """
    if (
        "src_global" in run
        and "dst_global" in run
    ):
        src_global = torch.as_tensor(
            run["src_global"],
            dtype=torch.long,
        ).detach().cpu().view(-1)

        dst_global = torch.as_tensor(
            run["dst_global"],
            dtype=torch.long,
        ).detach().cpu().view(-1)

        src_local = global_to_local[
            src_global
        ]

        dst_local = global_to_local[
            dst_global
        ]

    else:
        context_n_nodes = int(
            run["context"].get(
                "n_nodes",
                n_global,
            )
        )

        run_src = torch.as_tensor(
            run["src"],
            dtype=torch.long,
        ).detach().cpu().view(-1)

        run_dst = torch.as_tensor(
            run["dst"],
            dtype=torch.long,
        ).detach().cpu().view(-1)

        # A local context indicates that run["src"]/run["dst"]
        # are already local IDs.
        ids_are_local = (
            context_n_nodes == n_local
            and context_n_nodes != n_global
        )

        if ids_are_local:
            src_local = run_src
            dst_local = run_dst

            if (
                src_local.min() < 0
                or src_local.max() >= n_local
                or dst_local.min() < 0
                or dst_local.max() >= n_local
            ):
                raise ValueError(
                    "Local candidate endpoint outside "
                    "the source-subgraph range."
                )

            src_global = source_nodes_global[
                src_local
            ]

            dst_global = source_nodes_global[
                dst_local
            ]

        else:
            # The original source-mining cell stores these
            # candidate endpoints in complete-graph ID space.
            src_global = run_src
            dst_global = run_dst

            if (
                src_global.min() < 0
                or src_global.max() >= n_global
                or dst_global.min() < 0
                or dst_global.max() >= n_global
            ):
                raise ValueError(
                    "Global candidate endpoint outside "
                    "the complete-graph range."
                )

            src_local = global_to_local[
                src_global
            ]

            dst_local = global_to_local[
                dst_global
            ]

    if (
        (src_local < 0).any()
        or (dst_local < 0).any()
    ):
        raise ValueError(
            "At least one labeled candidate is not fully "
            "contained in the source subgraph."
        )

    if (
        src_local.numel()
        != dst_local.numel()
    ):
        raise ValueError(
            "Candidate source and destination arrays "
            "have unequal lengths."
        )

    return (
        src_local.to(device),
        dst_local.to(device),
        src_global.to(device),
        dst_global.to(device),
    )


# ============================================================
# Prediction and metric helpers
# ============================================================

def _predict_lp_probs(
    lp_model,
    x,
    edge_index_struct,
    edge_index_lab,
    device,
    batch_size=200_000,
):
    """
    Return selector probabilities for labeled local pairs.
    """
    lp_model.eval()

    x = x.to(device)
    edge_index_struct = (
        edge_index_struct
        .long()
        .to(device)
    )
    edge_index_lab = (
        edge_index_lab
        .long()
        .to(device)
    )

    probability_parts = []

    with torch.no_grad():
        for start in range(
            0,
            edge_index_lab.size(1),
            int(batch_size),
        ):
            end = min(
                start + int(batch_size),
                edge_index_lab.size(1),
            )

            logits = lp_model(
                x,
                edge_index_struct,
                edge_index_lab[:, start:end],
            ).view(-1)

            probability_parts.append(
                torch.sigmoid(
                    logits
                ).detach().cpu()
            )

    if not probability_parts:
        return np.empty(
            0,
            dtype=np.float32,
        )

    return torch.cat(
        probability_parts,
        dim=0,
    ).numpy()


def _safe_auc_ap(
    y_true,
    y_score,
):
    """
    ROC-AUC and AP are undefined when only one class exists.
    """
    y_true = np.asarray(
        y_true
    ).astype(int)

    y_score = np.asarray(
        y_score
    ).astype(float)

    if np.unique(y_true).size < 2:
        return np.nan, np.nan

    return (
        float(
            roc_auc_score(
                y_true,
                y_score,
            )
        ),
        float(
            average_precision_score(
                y_true,
                y_score,
            )
        ),
    )


def _aligned_endpoint_binary_labels(
    run: dict,
    scores: torch.Tensor,
) -> torch.Tensor:
    """
    Recover endpoint labels and align them with the candidate arrays
    retained in the mining run.
    """
    endpoint_labels = torch.as_tensor(
        run["mining_result"][
            "endpoint_labels"
        ],
        dtype=torch.float32,
        device=scores.device,
    ).view(-1)

    if endpoint_labels.numel() == scores.numel():
        return endpoint_labels

    observed_indices = run.get(
        "observed_indices"
    )

    if observed_indices is not None:
        observed_indices = torch.as_tensor(
            observed_indices,
            dtype=torch.long,
            device=scores.device,
        ).view(-1)

        if (
            observed_indices.numel()
            == scores.numel()
            and endpoint_labels.numel()
            > int(observed_indices.max())
        ):
            return endpoint_labels[
                observed_indices
            ]

    raise ValueError(
        "Endpoint-label array cannot be aligned with "
        "the retained candidate arrays."
    )


# ============================================================
# Plan selector-training runs
# ============================================================

RQ3_SUBGRAPH_TRAINING_RUNS = []
RQ3_SUBGRAPH_LP_MODEL_VARIANTS = {}
planned_training_configs = []

for scoring_run in RQ3_SUBGRAPH_MINING_RUNS:
    configurations = (
        _label_training_configs_for_scoring_run(
            scoring_run["scoring_mode"]
        )
    )

    for label_cfg in configurations:
        planned_training_configs.append(
            (
                scoring_run,
                label_cfg,
            )
        )


# ============================================================
# Train selectors
# ============================================================

for run_index, (
    run,
    label_cfg,
) in enumerate(
    planned_training_configs,
    start=1,
):
    label_mode = label_cfg[
        "label_mode"
    ]

    extreme_fraction = label_cfg[
        "extreme_fraction"
    ]

    (
        selector_context,
        full_context,
        global_to_local,
        source_nodes_global,
    ) = _build_selector_subgraph_context(
        run
    )

    device = selector_context[
        "device"
    ]

    n_local = int(
        selector_context["n_nodes"]
    )

    n_global = int(
        full_context.get(
            "n_nodes",
            full_context["attr"].size(0),
        )
    )
    selector_training_scope = str(
        selector_context.get(
            "selector_training_scope",
            run.get(
                "selector_training_scope",
                "induced_source_subgraph_only",
            ),
        )
    )

    # These are the only graph inputs visible to the selector
    # during training.
    selector_x = (
        selector_context["attr"]
        .to(device)
    )

    selector_edge_index_struct = (
        selector_context["edge_index"]
        .long()
        .to(device)
    )

    (
        src_local_run,
        dst_local_run,
        src_global_run,
        dst_global_run,
    ) = _resolve_local_and_global_candidate_ids(
        run,
        source_nodes_global=source_nodes_global,
        global_to_local=global_to_local,
        n_global=n_global,
        n_local=n_local,
        device=device,
    )

    scores = torch.as_tensor(
        run["labels_changed"],
        dtype=torch.float32,
        device=device,
    ).flatten()

    exists_run = torch.as_tensor(
        run["exists"],
        dtype=torch.float32,
        device=device,
    ).flatten()

    n_pairs = int(
        src_local_run.numel()
    )

    if not (
        dst_local_run.numel()
        == scores.numel()
        == exists_run.numel()
        == n_pairs
    ):
        raise ValueError(
            "Candidate endpoints, labels and existence "
            "flags do not have equal lengths."
        )

    # Final visibility assertions.
    if n_pairs == 0:
        raise ValueError(
            "No labeled source-subgraph candidate pairs "
            "are available for training."
        )

    if (
        int(src_local_run.min()) < 0
        or int(dst_local_run.min()) < 0
        or int(src_local_run.max()) >= n_local
        or int(dst_local_run.max()) >= n_local
    ):
        raise AssertionError(
            "The selector candidate labels contain a node "
            "outside the source subgraph."
        )

    print("=" * 80)
    print(
        f"Training LP-GNN "
        f"{run_index}/{len(planned_training_configs)}\n"
        f"  seed:                    {run['seed']}\n"
        f"  configuration:           {run['candidate_config_id']}\n"
        f"  scoring mode:            {run['scoring_mode']}\n"
        f"  label mode:              {label_mode}\n"
        f"  extreme fraction:        "
        f"{extreme_fraction if extreme_fraction is not None else 'na'}\n"
        f"  visible selector nodes:  {n_local}/{n_global}\n"
        f"  visible structural edges:"
        f"  {selector_edge_index_struct.size(1)}\n"
        f"  available labeled pairs: {n_pairs}\n"
        f"  selector scope:          {selector_training_scope}"
    )

    # --------------------------------------------------------
    # Remove unobserved/non-finite labels
    # --------------------------------------------------------

    valid_mask = torch.isfinite(
        scores
    )

    src_local_valid = src_local_run[
        valid_mask
    ]
    dst_local_valid = dst_local_run[
        valid_mask
    ]

    src_global_valid = src_global_run[
        valid_mask
    ]
    dst_global_valid = dst_global_run[
        valid_mask
    ]

    scores_valid = scores[
        valid_mask
    ]

    exists_valid = exists_run[
        valid_mask
    ]

    if scores_valid.numel() < 3:
        raise ValueError(
            "Too few finite candidate labels for "
            "train/validation/test splitting."
        )

    # --------------------------------------------------------
    # Explicit harmful/non-harmful labels
    # --------------------------------------------------------

    if (
        run["scoring_mode"] == "endpoint"
        and "endpoint_labels"
        in run["mining_result"]
    ):
        endpoint_binary_all = (
            _aligned_endpoint_binary_labels(
                run,
                scores,
            )
        )

        roc_binary_valid = (
            endpoint_binary_all[
                valid_mask
            ] > 0
        )

    else:
        roc_binary_valid = (
            scores_valid
            > ROC_POSITIVE_THRESHOLD
        )

    # --------------------------------------------------------
    # Build continuous or projected training labels
    # --------------------------------------------------------

    if label_mode == "continuous":
        src_local_bal = (
            src_local_valid
        )
        dst_local_bal = (
            dst_local_valid
        )

        src_global_bal = (
            src_global_valid
        )
        dst_global_bal = (
            dst_global_valid
        )

        lbl_bal = scores_valid
        ex_bal = exists_valid

        roc_y_true_bal = (
            roc_binary_valid.float()
        )

    elif label_mode == "projected":
        if extreme_fraction is None:
            raise ValueError(
                "extreme_fraction is required "
                "for projected labels."
            )

        n_valid = int(
            scores_valid.numel()
        )

        if n_valid < 2:
            raise ValueError(
                "Projected training requires at "
                "least two finite labels."
            )

        n_per_class = max(
            1,
            min(
                int(
                    n_valid
                    * float(extreme_fraction)
                ),
                n_valid // 2,
            ),
        )

        order = torch.argsort(
            scores_valid
        )

        low_idx = order[
            :n_per_class
        ]

        high_idx = order[
            -n_per_class:
        ]

        selected_idx = torch.cat(
            [
                low_idx,
                high_idx,
            ]
        )

        src_local_bal = (
            src_local_valid[
                selected_idx
            ]
        )

        dst_local_bal = (
            dst_local_valid[
                selected_idx
            ]
        )

        src_global_bal = (
            src_global_valid[
                selected_idx
            ]
        )

        dst_global_bal = (
            dst_global_valid[
                selected_idx
            ]
        )

        ex_bal = exists_valid[
            selected_idx
        ]

        lbl_bal = torch.cat(
            [
                torch.zeros(
                    n_per_class,
                    device=device,
                    dtype=torch.float32,
                ),
                torch.ones(
                    n_per_class,
                    device=device,
                    dtype=torch.float32,
                ),
            ]
        )

        # The projected labels themselves define
        # harmful/non-harmful membership.
        roc_y_true_bal = (
            lbl_bal.clone()
        )

    else:
        raise ValueError(
            "label_mode must be "
            "'continuous' or 'projected'."
        )

    edge_index_bal_local = torch.stack(
        [
            src_local_bal.long(),
            dst_local_bal.long(),
        ],
        dim=0,
    )

    edge_index_bal_global = torch.stack(
        [
            src_global_bal.long(),
            dst_global_bal.long(),
        ],
        dim=0,
    )

    lbl_bal = lbl_bal.float()
    roc_y_true_bal = (
        roc_y_true_bal.float()
    )

    if edge_index_bal_local.numel():
        if (
            int(
                edge_index_bal_local.min()
            ) < 0
            or int(
                edge_index_bal_local.max()
            ) >= n_local
        ):
            raise AssertionError(
                "Training pair contains a node "
                "outside the source subgraph."
            )

    # --------------------------------------------------------
    # Train using ONLY local source-subgraph tensors
    # --------------------------------------------------------

    trained_lp_model = (
        train_selector(
            attr=selector_x,
            edge_index_struct=(
                selector_edge_index_struct
            ),
            edge_index_lab=(
                edge_index_bal_local
            ),
            y_label=lbl_bal,

            # Used for stratification and classification
            # metrics, while y_label may remain continuous.
            binary_y_label=(
                roc_y_true_bal
            ),

            device=device,
            num_epochs=1000,
            hidden_dim=64,
            out_dim=64,
            lr=5e-4,
            weight_decay=5e-4,
            use_tqdm=True,
            verbose=True,

            early_stop=True,
            early_stop_metric="val_loss",
            min_epochs_before_early_stop=300,
            patience=15,
            early_stop_min_delta=1e-4,
            restore_best=True,

            train_ratio=0.70,
            val_ratio=0.15,
            test_ratio=0.15,
            split_seed=42,

            threshold=0.5,
            evaluate_test_each_epoch=False,
        )
    )

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    lp_model_group_label = (
        f"{run['candidate_config_id']}"
        f"__score-"
        f"{_safe_config_value(run['scoring_mode'])}"
        f"__label-"
        f"{_safe_config_value(label_mode)}"
    )

    if extreme_fraction is not None:
        lp_model_group_label += (
            f"__extremeFrac-"
            f"{_safe_config_value(extreme_fraction)}"
        )

    # Unique label prevents multiseed output overwrites.
    variant_label = (
        f"{lp_model_group_label}"
        f"__seed-{int(run['seed'])}"
    )

    # --------------------------------------------------------
    # ROC data on all locally labeled selector pairs
    # --------------------------------------------------------

    roc_y_score = _predict_lp_probs(
        lp_model=trained_lp_model,
        x=selector_x,
        edge_index_struct=(
            selector_edge_index_struct
        ),
        edge_index_lab=(
            edge_index_bal_local
        ),
        device=device,
        batch_size=ROC_PRED_BATCH_SIZE,
    )

    roc_y_true = (
        roc_y_true_bal
        .detach()
        .cpu()
        .numpy()
        .astype(int)
        .reshape(-1)
    )

    if (
        roc_y_true.shape[0]
        != roc_y_score.shape[0]
    ):
        raise RuntimeError(
            f"ROC label/score length mismatch "
            f"for {variant_label}: "
            f"{roc_y_true.shape[0]} labels versus "
            f"{roc_y_score.shape[0]} scores."
        )

    roc_auc_all, roc_ap_all = (
        _safe_auc_ap(
            roc_y_true,
            roc_y_score,
        )
    )

    roc_data_path = (
        LP_ROC_DATA_OUT_DIR
        / (
            "roc_data__"
            f"{_safe_config_value(variant_label)}"
            ".npz"
        )
    )

    np.savez_compressed(
        roc_data_path,

        y_true=roc_y_true,
        y_score=roc_y_score,

        # Local pair IDs used for selector training.
        edge_index_lab_local=(
            edge_index_bal_local
            .detach()
            .cpu()
            .numpy()
        ),

        # Matching complete-graph IDs retained for transfer.
        edge_index_lab_global=(
            edge_index_bal_global
            .detach()
            .cpu()
            .numpy()
        ),

        source_nodes_global=(
            source_nodes_global
            .detach()
            .cpu()
            .numpy()
        ),

        y_label_training=(
            lbl_bal
            .detach()
            .cpu()
            .numpy()
        ),

        exists_training=(
            ex_bal
            .detach()
            .cpu()
            .numpy()
        ),

        label_mode=np.array(
            label_mode
        ),

        scoring_mode=np.array(
            run["scoring_mode"]
        ),

        candidate_config_id=np.array(
            run["candidate_config_id"]
        ),

        subgraph_seed=np.array(
            int(run["subgraph_seed"])
        ),

        subgraph_fraction_requested=np.array(
            float(run["subgraph_fraction_requested"])
        ),

        subgraph_fraction_actual=np.array(
            float(run["subgraph_fraction_actual"])
        ),

        training_candidate_size=np.array(
            int(run["training_candidate_size"])
        ),

        selector_training_scope=np.array(
            selector_training_scope
        ),

        n_selector_nodes=np.array(
            n_local
        ),

        n_global_nodes=np.array(
            n_global
        ),
    )

    # --------------------------------------------------------
    # Attach selector diagnostics
    # --------------------------------------------------------

    if (
        not hasattr(
            trained_lp_model,
            "training_history",
        )
        or trained_lp_model.training_history
        is None
    ):
        trained_lp_model.training_history = {}

    trained_lp_model.training_history.update({
        "all_labels": roc_y_true,
        "all_probs": roc_y_score,
        "roc_data_path": str(
            roc_data_path
        ),

        "selector_training_scope": (
            selector_training_scope
        ),

        "n_selector_nodes": n_local,
        "n_global_nodes": n_global,

        "source_nodes_global": (
            source_nodes_global.clone()
        ),

        "edge_index_lab_global": (
            edge_index_bal_global
            .detach()
            .cpu()
        ),
    })

    # --------------------------------------------------------
    # Store trained run
    # --------------------------------------------------------

    trained_run = {
        **run,

        # Important: diagnostic functions operating on the
        # training graph must now see the local context.
        "context": selector_context,

        # Whole graph retained for later RQ3 transfer/attack.
        "full_context": full_context,

        "lp_model": trained_lp_model,
        "lp_model_label": variant_label,
        "lp_model_group_label": (
            lp_model_group_label
        ),

        "label_mode": label_mode,

        "extreme_fraction": (
            ""
            if extreme_fraction is None
            else float(extreme_fraction)
        ),

        "selector_training_scope": (
            selector_training_scope
        ),

        "n_selector_nodes": n_local,
        "n_global_nodes": n_global,

        "n_training_edges": int(
            edge_index_bal_local.size(1)
        ),

        "n_training_positive": int(
            roc_y_true.sum()
        ),

        "n_training_negative": int(
            (roc_y_true == 0).sum()
        ),

        # Local tensors actually shown to the selector.
        "x": (
            selector_x
            .detach()
            .cpu()
        ),

        "edge_index_struct": (
            selector_edge_index_struct
            .detach()
            .cpu()
        ),

        "edge_index_lab": (
            edge_index_bal_local
            .detach()
            .cpu()
        ),

        "y_label": (
            lbl_bal
            .detach()
            .cpu()
        ),

        "binary_labels": (
            roc_y_true_bal
            .detach()
            .cpu()
            .long()
        ),

        "roc_y_true_tensor": (
            roc_y_true_bal
            .detach()
            .cpu()
            .long()
        ),

        # Global mapping retained for transfer and bookkeeping.
        "source_nodes_global": (
            source_nodes_global
            .detach()
            .cpu()
        ),

        "global_to_local": (
            global_to_local
            .detach()
            .cpu()
        ),

        "edge_index_lab_global": (
            edge_index_bal_global
            .detach()
            .cpu()
        ),

        # ROC-ready data.
        "roc_split": "all",
        "roc_y_true": roc_y_true,
        "roc_y_score": roc_y_score,
        "roc_auc_all": roc_auc_all,
        "roc_ap_all": roc_ap_all,

        "roc_n_positive": int(
            roc_y_true.sum()
        ),

        "roc_n_negative": int(
            (roc_y_true == 0).sum()
        ),

        "roc_data_path": str(
            roc_data_path
        ),
    }

    RQ3_SUBGRAPH_TRAINING_RUNS.append(
        trained_run
    )

    # --------------------------------------------------------
    # Store sweep-ready selector variant
    # --------------------------------------------------------

    RQ3_SUBGRAPH_LP_MODEL_VARIANTS[
        variant_label
    ] = {
        "model": trained_lp_model,

        "lp_model_label": (
            variant_label
        ),

        "lp_model_group_label": (
            lp_model_group_label
        ),

        "seed": int(
            run["seed"]
        ),

        "candidate_config_id": (
            run["candidate_config_id"]
        ),

        "candidate_set_size": (
            run["candidate_set_size"]
        ),

        "training_candidate_size": int(
            run["training_candidate_size"]
        ),

        "subgraph_seed": int(
            run["subgraph_seed"]
        ),

        "subgraph_fraction_requested": float(
            run["subgraph_fraction_requested"]
        ),

        "subgraph_fraction_actual": float(
            run["subgraph_fraction_actual"]
        ),

        "n_source_nodes": int(
            run["n_source_nodes"]
        ),

        "n_target_nodes": int(
            run["n_target_nodes"]
        ),

        "prbcd_candidate_fraction": (
            run.get(
                "prbcd_candidate_fraction",
                0.0,
            )
        ),

        "actual_prbcd_fraction": (
            run.get(
                "actual_prbcd_fraction",
                0.0,
            )
        ),

        "scoring_mode": (
            run["scoring_mode"]
        ),

        "endpoint_mining_hop": (
            run.get(
                "endpoint_mining_hop",
                0,
            )
        ),

        "n_prbcd_requested": (
            run.get(
                "n_prbcd_requested",
                0,
            )
        ),

        "n_prbcd_mined_unique": (
            run.get(
                "n_prbcd_mined_unique",
                0,
            )
        ),

        "n_prbcd_selected": (
            run.get(
                "n_prbcd_selected",
                0,
            )
        ),

        "n_random_candidates": (
            run.get(
                "n_random_candidates",
                run["candidate_set_size"],
            )
        ),

        "n_positive_labels": (
            run.get(
                "n_positive_labels",
                int(roc_y_true.sum()),
            )
        ),

        "label_mode": label_mode,

        "extreme_fraction": (
            ""
            if extreme_fraction is None
            else float(extreme_fraction)
        ),

        "selector_training_scope": (
            selector_training_scope
        ),

        "n_selector_nodes": n_local,
        "n_global_nodes": n_global,

        "source_nodes_global": (
            source_nodes_global
            .detach()
            .cpu()
        ),

        # ROC metadata.
        "roc_auc_all": roc_auc_all,
        "roc_ap_all": roc_ap_all,

        "roc_n_positive": int(
            roc_y_true.sum()
        ),

        "roc_n_negative": int(
            (roc_y_true == 0).sum()
        ),

        "roc_data_path": str(
            roc_data_path
        ),
    }

    print(
        f"Training complete | "
        f"selector nodes={n_local}/{n_global} | "
        f"training pairs={edge_index_bal_local.size(1)} | "
        f"AUC={roc_auc_all:.4f} | "
        f"AP={roc_ap_all:.4f} | "
        f"positives={int(roc_y_true.sum())}/"
        f"{len(roc_y_true)}\n"
        f"ROC data: {roc_data_path}"
    )


# ============================================================
# Expose trained selector collections
# ============================================================
# ============================================================
# Expose the complete sweep and one primary legacy combination
# ============================================================

if not RQ3_SUBGRAPH_TRAINING_RUNS:
    raise RuntimeError("No RQ3 subgraph LP-GNN models were trained.")

RQ3_PRIMARY_LP_TRAINING_RUNS = [
    run
    for run in RQ3_SUBGRAPH_TRAINING_RUNS
    if np.isclose(
        float(run["subgraph_fraction_requested"]),
        float(RQ3_PRIMARY_SUBGRAPH_FRACTION),
    )
    and int(run["training_candidate_size"])
    == int(RQ3_PRIMARY_TRAINING_CANDIDATE_SIZE)
    and int(run["subgraph_seed"])
    == int(RQ3_PRIMARY_SUBGRAPH_SEED)
]

if not RQ3_PRIMARY_LP_TRAINING_RUNS:
    raise RuntimeError(
        "No trained selectors match the configured primary subgraph "
        "fraction, training candidate size, and subgraph seed."
    )

# Legacy downstream diagnostics see only the primary combination.
lp_training_runs = list(RQ3_PRIMARY_LP_TRAINING_RUNS)
lp_model = lp_training_runs[0]["lp_model"]

lp_model_variants = {
    run["lp_model_label"]: RQ3_SUBGRAPH_LP_MODEL_VARIANTS[
        run["lp_model_label"]
    ]
    for run in lp_training_runs
}
LP_MODEL_VARIANTS = lp_model_variants

# §8 explicitly uses this complete registry.
MODEL_GUIDED_PRBCD_LP_MODEL_VARIANTS = (
    RQ3_SUBGRAPH_LP_MODEL_VARIANTS
)


# ============================================================
# Summaries
# ============================================================

def _rq3_training_summary_row(run):
    return {
        "lp_model_label": run["lp_model_label"],
        "lp_model_group_label": run["lp_model_group_label"],
        "seed": int(run["seed"]),
        "candidate_config_id": run["candidate_config_id"],
        "candidate_set_size": run["candidate_set_size"],
        "training_candidate_size": int(run["training_candidate_size"]),
        "subgraph_seed": int(run["subgraph_seed"]),
        "subgraph_fraction_requested": float(
            run["subgraph_fraction_requested"]
        ),
        "subgraph_fraction_actual": float(
            run["subgraph_fraction_actual"]
        ),
        "n_source_nodes": int(run["n_source_nodes"]),
        "n_target_nodes": int(run["n_target_nodes"]),
        "scoring_mode": run["scoring_mode"],
        "label_mode": run["label_mode"],
        "extreme_fraction": run["extreme_fraction"],
        "selector_training_scope": run["selector_training_scope"],
        "n_selector_nodes": run["n_selector_nodes"],
        "n_global_nodes": run["n_global_nodes"],
        "n_visible_structural_edges": int(
            run["edge_index_struct"].size(1)
        ),
        "n_training_edges": run["n_training_edges"],
        "n_training_positive": run["n_training_positive"],
        "n_training_negative": run["n_training_negative"],
        "roc_split": run["roc_split"],
        "roc_auc_all": run["roc_auc_all"],
        "roc_ap_all": run["roc_ap_all"],
        "roc_n_positive": run["roc_n_positive"],
        "roc_n_negative": run["roc_n_negative"],
        "roc_data_path": run["roc_data_path"],
    }


rq3_subgraph_training_summary_df = pd.DataFrame([
    _rq3_training_summary_row(run)
    for run in RQ3_SUBGRAPH_TRAINING_RUNS
])

lp_training_summary_df = pd.DataFrame([
    _rq3_training_summary_row(run)
    for run in lp_training_runs
])

display(rq3_subgraph_training_summary_df)

print(
    f"Trained {len(RQ3_SUBGRAPH_TRAINING_RUNS)} selector runs across the "
    "complete subgraph sweep.\n"
    f"Legacy diagnostics receive {len(lp_training_runs)} primary runs.\n"
    f"§8 receives {len(MODEL_GUIDED_PRBCD_LP_MODEL_VARIANTS)} selector "
    "variants."
)


### RQ3 selector-training statistics across the complete subgraph sweep

The following diagnostic cells evaluate every trained selector in `RQ3_SUBGRAPH_TRAINING_RUNS`. The later fixed-block experiment still uses only the configured primary selector through `lp_training_runs`, while §8 uses the complete selector registry.


In [ ]:
from pathlib import Path
from IPython.display import display
import hashlib
import json
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# Output and plot configuration
# ============================================================

LP_HISTORY_OUT_DIR = Path("extendedPlotting") / "lp_training_dynamics_multiseed"
LP_HISTORY_OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAINING_PLOT_DIR = Path(
    globals().get(
        "RUN_PLOTS_DIR",
        LP_HISTORY_OUT_DIR,
    )
)
TRAINING_PLOT_DIR.mkdir(parents=True, exist_ok=True)

# PNG is convenient for inspection; PDF and SVG preserve vector text and
# lines for direct inclusion in the thesis.
TRAINING_EXPORT_FORMATS = tuple(
    globals().get(
        "TRAINING_EXPORT_FORMATS",
        ("png", "pdf", "svg"),
    )
)
TRAINING_PNG_DPI = int(globals().get("TRAINING_PNG_DPI", 300))
TRAINING_SHOW_PLOTS = bool(globals().get("TRAINING_SHOW_PLOTS", True))

THESIS_RC = {
    "font.size": 10.0,
    "axes.titlesize": 12.0,
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9.0,
    "ytick.labelsize": 9.0,
    "legend.fontsize": 8.6,
    "legend.title_fontsize": 8.6,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.8,
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
}


# ============================================================
# Diagnostic scope
# ============================================================

if "RQ3_SUBGRAPH_TRAINING_RUNS" not in globals():
    raise RuntimeError(
        "RQ3_SUBGRAPH_TRAINING_RUNS is unavailable. Run the RQ3 "
        "subgraph construction and selector-training cells first."
    )

RQ3_TRAINING_STAT_RUNS = list(RQ3_SUBGRAPH_TRAINING_RUNS)

if not RQ3_TRAINING_STAT_RUNS:
    raise RuntimeError(
        "No trained RQ3 subgraph selectors are available for diagnostics."
    )


# ============================================================
# Configuration metadata
# ============================================================

# These fields define a like-for-like training configuration. Victim/model
# seed is deliberately excluded because means and standard deviations are
# computed across victim seeds.
TRAINING_CONFIG_ID_FIELDS = [
    "lp_model_group_label",
    "candidate_config_id",
    "subgraph_method",
    "subgraph_seed",
    "subgraph_fraction_requested",
    "training_candidate_size",
    "scoring_mode",
    "endpoint_mining_hop",
    "label_mode",
    "extreme_fraction",
    "selector_training_scope",
    "victim_query_scope",
    "prbcd_candidate_fraction",
]

# These fields are retained for traceability and displayed when available.
TRAINING_CONFIG_METADATA_FIELDS = [
    "lp_model_label",
    "candidate_set_size",
    "subgraph_fraction_actual",
    "construction_seed",
    "candidate_pool_seed",
    "mining_seed",
    "n_source_nodes",
    "n_target_nodes",
    "n_selector_nodes",
    "n_global_nodes",
    "n_nodes_global",
    "n_test_nodes_local",
    "n_edges_local_undirected",
    "density_local",
    "global_edge_coverage",
    "n_training_edges",
    "n_training_positive",
    "n_training_negative",
    "n_positive_labels",
    "actual_prbcd_fraction",
    "n_prbcd_requested",
    "n_prbcd_mined_unique",
    "n_prbcd_selected",
    "n_random_candidates",
]


def _clean_config_value(value):
    """Return a stable scalar suitable for IDs, tables, and JSON output."""
    if value is None:
        return ""

    if isinstance(value, np.generic):
        value = value.item()

    if isinstance(value, float):
        if not np.isfinite(value):
            return ""
        return float(value)

    if isinstance(value, (int, bool, str)):
        return value

    return str(value)


def _get_run_value(run, key, default=""):
    value = run.get(key, default)

    # Graph-construction diagnostics can be nested in subgraph_stats or in
    # sampler_metadata, depending on the upstream notebook version.
    if value in (None, ""):
        for nested_key in ("subgraph_stats", "sampler_metadata"):
            nested = run.get(nested_key, {})
            if isinstance(nested, dict) and key in nested:
                value = nested.get(key, default)
                break

    return _clean_config_value(value)


def _build_training_config_record(run):
    record = {
        "seed": int(run["seed"]),
        "lp_model_group_label": str(run.get("lp_model_group_label", "")),
        "lp_model_label": str(run.get("lp_model_label", "")),
    }

    for key in TRAINING_CONFIG_ID_FIELDS + TRAINING_CONFIG_METADATA_FIELDS:
        if key not in record:
            record[key] = _get_run_value(run, key)

    requested_fraction = record.get("subgraph_fraction_requested", "")
    method = str(record.get("subgraph_method", "")).strip()

    if not method and requested_fraction != "":
        try:
            if np.isclose(float(requested_fraction), 1.0):
                method = "whole_graph"
        except (TypeError, ValueError):
            pass

    record["subgraph_method"] = method or "unknown"

    # Normalize common node-count aliases without imposing a single upstream
    # schema on the training cell.
    if record.get("n_global_nodes", "") == "":
        record["n_global_nodes"] = record.get("n_nodes_global", "")
    if record.get("n_nodes_global", "") == "":
        record["n_nodes_global"] = record.get("n_global_nodes", "")
    if record.get("n_selector_nodes", "") == "":
        record["n_selector_nodes"] = record.get("n_source_nodes", "")
    if record.get("n_source_nodes", "") == "":
        record["n_source_nodes"] = record.get("n_selector_nodes", "")

    identity_payload = {
        key: record.get(key, "")
        for key in TRAINING_CONFIG_ID_FIELDS
    }
    canonical_payload = json.dumps(
        identity_payload,
        sort_keys=True,
        ensure_ascii=True,
        separators=(",", ":"),
    )
    config_hash = hashlib.sha1(
        canonical_payload.encode("utf-8")
    ).hexdigest()[:10]

    record["training_config_hash"] = config_hash
    record["training_config_id"] = canonical_payload
    return record


def _history_safe_file(value, max_length=155):
    helper = globals().get("_safe_config_value")
    if callable(helper):
        safe = str(helper(value))
    else:
        safe = (
            str(value)
            .replace("/", "-")
            .replace("\\", "-")
            .replace(" ", "_")
            .replace(":", "-")
            .replace(".", "p")
        )

    if len(safe) <= max_length:
        return safe

    digest = hashlib.sha1(safe.encode("utf-8")).hexdigest()[:10]
    prefix_length = max(20, max_length - len(digest) - 2)
    return f"{safe[:prefix_length].rstrip('._-')}__{digest}"


def _pretty_method(value):
    mapping = {
        "bfs": "BFS",
        "ties": "TIES",
        "forest_fire": "Forest Fire",
        "forestfire": "Forest Fire",
        "whole_graph": "Whole graph",
    }
    value = str(value).strip()
    return mapping.get(value, value.replace("_", " ").title() or "Unknown")


def _pretty_token(value):
    value = str(value).strip()
    if not value:
        return "not specified"
    return value.replace("_", " ")


def _format_fraction(value, decimals=1):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return "not specified"

    if not np.isfinite(number):
        return "not specified"

    percentage = 100.0 * number
    if np.isclose(percentage, round(percentage)):
        return f"{percentage:.0f}%"
    return f"{percentage:.{decimals}f}%"


def _format_number(value, decimals=0):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return "not specified"

    if not np.isfinite(number):
        return "not specified"

    if decimals == 0 and np.isclose(number, round(number)):
        return f"{int(round(number)):,}"
    return f"{number:,.{decimals}f}"


def _is_present(value):
    if value is None:
        return False
    if isinstance(value, float) and not np.isfinite(value):
        return False
    return str(value).strip().lower() not in {"", "nan", "none"}


def _compact_text(value, max_chars=54):
    text = str(value).strip()
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 1].rstrip() + "…"


def _configuration_slug(config):
    parts = [
        f"method-{_history_safe_file(config.get('subgraph_method', 'unknown'), 24)}",
        f"frac-{_history_safe_file(config.get('subgraph_fraction_requested', 'na'), 18)}",
        f"subseed-{_history_safe_file(config.get('subgraph_seed', 'na'), 18)}",
        f"cand-{_history_safe_file(config.get('training_candidate_size', 'na'), 20)}",
        f"score-{_history_safe_file(config.get('scoring_mode', 'na'), 28)}",
        f"label-{_history_safe_file(config.get('label_mode', 'na'), 22)}",
        f"cfg-{config.get('training_config_hash', 'unknown')}",
    ]
    return _history_safe_file("__".join(parts), max_length=175)


def _configuration_text(config):
    method = _pretty_method(config.get("subgraph_method", "unknown"))
    requested = _format_fraction(config.get("subgraph_fraction_requested", ""))
    actual = _format_fraction(
        config.get(
            "subgraph_fraction_actual_mean",
            config.get("subgraph_fraction_actual", ""),
        ),
        decimals=2,
    )
    training_candidates = _format_number(
        config.get("training_candidate_size", "")
    )

    if method == "Whole graph":
        subtitle = (
            f"Whole-graph LP-GNN training · "
            f"{training_candidates} candidate pairs"
        )
    else:
        subtitle = (
            f"{method} source subgraph · requested {requested} "
            f"(actual {actual}) · {training_candidates} candidate pairs"
        )

    details = [
        f"Scoring: {_pretty_token(config.get('scoring_mode', ''))}",
        f"Labels: {_pretty_token(config.get('label_mode', ''))}",
        f"Subgraph seed: {config.get('subgraph_seed', 'not specified')}",
    ]

    endpoint_hop = config.get("endpoint_mining_hop", "")
    if _is_present(endpoint_hop):
        details.append(f"Endpoint hop: {endpoint_hop}")

    extreme_fraction = config.get("extreme_fraction", "")
    if _is_present(extreme_fraction):
        details.append(
            f"Extreme fraction: {_format_fraction(extreme_fraction)}"
        )

    selector_scope = config.get("selector_training_scope", "")
    if _is_present(selector_scope):
        details.append(f"Selector scope: {_pretty_token(selector_scope)}")

    query_scope = config.get("victim_query_scope", "")
    if _is_present(query_scope):
        details.append(f"Victim-query scope: {_pretty_token(query_scope)}")

    source_nodes = config.get(
        "n_source_nodes_mean",
        config.get(
            "n_selector_nodes_mean",
            config.get("n_source_nodes", config.get("n_selector_nodes", "")),
        ),
    )
    global_nodes = config.get(
        "n_global_nodes_mean",
        config.get(
            "n_nodes_global_mean",
            config.get("n_global_nodes", config.get("n_nodes_global", "")),
        ),
    )
    if _is_present(source_nodes) and _is_present(global_nodes):
        details.append(
            f"Selector nodes: {_format_number(source_nodes)}/"
            f"{_format_number(global_nodes)}"
        )

    training_edges = config.get("n_training_edges_mean", "")
    if _is_present(training_edges):
        details.append(f"Training pairs used: {_format_number(training_edges)}")

    positive = config.get("n_training_positive_mean", "")
    negative = config.get("n_training_negative_mean", "")
    if _is_present(positive) and _is_present(negative):
        details.append(
            f"Class counts: {_format_number(positive)} positive / "
            f"{_format_number(negative)} negative"
        )

    n_seeds = config.get("n_seeds", config.get("seed_count", ""))
    if _is_present(n_seeds):
        details.append(f"Victim seeds: {_format_number(n_seeds)}")

    candidate_config = config.get("candidate_config_id", "")
    if _is_present(candidate_config):
        details.append(
            f"Candidate config: {_compact_text(candidate_config, 46)}"
        )

    details.append(
        f"Configuration: {config.get('training_config_hash', 'unknown')}"
    )

    detail_text = " · ".join(details)
    detail_text = "\n".join(
        textwrap.wrap(
            detail_text,
            width=122,
            break_long_words=False,
            break_on_hyphens=False,
        )
    )
    return subtitle, detail_text


def _apply_thesis_layout(
    fig,
    ax,
    title,
    config,
    *,
    top=0.78,
    bottom=0.22,
    left=0.13,
    right=0.97,
):
    subtitle, detail_text = _configuration_text(config)

    fig.suptitle(
        title,
        x=left,
        y=0.975,
        ha="left",
        va="top",
        fontsize=14,
        fontweight="semibold",
    )
    fig.text(
        left,
        0.915,
        subtitle,
        ha="left",
        va="top",
        fontsize=9.6,
    )
    fig.text(
        left,
        0.025,
        detail_text,
        ha="left",
        va="bottom",
        fontsize=8.0,
        color="0.35",
        linespacing=1.3,
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out")
    fig.subplots_adjust(
        top=top,
        bottom=bottom,
        left=left,
        right=right,
    )


def _save_thesis_figure(fig, stem):
    for extension in TRAINING_EXPORT_FORMATS:
        extension = str(extension).lower().lstrip(".")
        output_path = TRAINING_PLOT_DIR / f"{stem}.{extension}"
        save_kwargs = {
            "bbox_inches": "tight",
            "facecolor": "white",
        }
        if extension == "png":
            save_kwargs["dpi"] = TRAINING_PNG_DPI

        fig.savefig(output_path, **save_kwargs)
        print("Saved:", output_path)

    if TRAINING_SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)


def _mean_std_frame(frame, group_cols, value_cols):
    aggregations = {}
    for column in value_cols:
        aggregations[f"{column}_mean"] = (column, "mean")
        aggregations[f"{column}_std"] = (column, "std")

    result = frame.groupby(group_cols, as_index=False).agg(
        **aggregations,
        n_seeds=("seed", "nunique"),
    )

    for column in result.columns:
        if column.endswith("_std"):
            result[column] = result[column].fillna(0.0)

    return result


def _plot_band(ax, x, mean, std, label, *, clip=None):
    x = np.asarray(x, dtype=float)
    mean = np.asarray(mean, dtype=float)
    std = np.asarray(std, dtype=float)

    lower = mean - std
    upper = mean + std
    if clip is not None:
        lower = np.clip(lower, clip[0], clip[1])
        upper = np.clip(upper, clip[0], clip[1])

    line = ax.plot(x, mean, label=label)[0]
    ax.fill_between(
        x,
        lower,
        upper,
        alpha=0.16,
        color=line.get_color(),
        linewidth=0,
    )


# ============================================================
# Build complete configuration registry
# ============================================================

run_config_records = [
    _build_training_config_record(run)
    for run in RQ3_TRAINING_STAT_RUNS
]
run_config_df = pd.DataFrame(run_config_records)

# Guard against the unlikely event of a hash collision.
config_id_counts = run_config_df.groupby(
    "training_config_hash"
)["training_config_id"].nunique()
if (config_id_counts > 1).any():
    raise RuntimeError(
        "A training-configuration hash collision occurred. Increase the "
        "configuration hash length before aggregating results."
    )

# Fields whose across-seed means and standard deviations are useful in the
# manifest and in figure annotations.
manifest_numeric_fields = [
    key
    for key in [
        "subgraph_fraction_actual",
        "candidate_set_size",
        "n_source_nodes",
        "n_target_nodes",
        "n_selector_nodes",
        "n_global_nodes",
        "n_nodes_global",
        "n_test_nodes_local",
        "n_edges_local_undirected",
        "density_local",
        "global_edge_coverage",
        "n_training_edges",
        "n_training_positive",
        "n_training_negative",
        "n_positive_labels",
        "actual_prbcd_fraction",
        "n_prbcd_requested",
        "n_prbcd_mined_unique",
        "n_prbcd_selected",
        "n_random_candidates",
    ]
    if key in run_config_df.columns
]

for column in manifest_numeric_fields:
    run_config_df[column] = pd.to_numeric(
        run_config_df[column],
        errors="coerce",
    )

manifest_aggregations = {
    key: (key, "first")
    for key in TRAINING_CONFIG_ID_FIELDS
    if key in run_config_df.columns
}
manifest_aggregations.update({
    "training_config_id": ("training_config_id", "first"),
    "lp_model_label_example": ("lp_model_label", "first"),
    "n_seeds": ("seed", "nunique"),
    "victim_seeds": (
        "seed",
        lambda values: ",".join(
            str(int(value))
            for value in sorted(pd.unique(values))
        ),
    ),
})

for column in manifest_numeric_fields:
    manifest_aggregations[f"{column}_mean"] = (column, "mean")
    manifest_aggregations[f"{column}_std"] = (column, "std")

# Preserve seed-like provenance fields as readable lists because they may
# legitimately differ across victim seeds.
for column in ["construction_seed", "candidate_pool_seed", "mining_seed"]:
    if column in run_config_df.columns:
        manifest_aggregations[f"{column}_values"] = (
            column,
            lambda values: ",".join(
                str(value)
                for value in sorted(
                    {
                        str(value)
                        for value in values
                        if _is_present(value)
                    }
                )
            ),
        )

training_configuration_manifest_df = (
    run_config_df
    .groupby("training_config_hash", as_index=False)
    .agg(**manifest_aggregations)
)

for column in training_configuration_manifest_df.columns:
    if column.endswith("_std"):
        training_configuration_manifest_df[column] = (
            training_configuration_manifest_df[column].fillna(0.0)
        )

training_configuration_manifest_df["plot_slug"] = (
    training_configuration_manifest_df.apply(
        _configuration_slug,
        axis=1,
    )
)

manifest_csv_path = (
    LP_HISTORY_OUT_DIR / "lp_training_configuration_manifest.csv"
)
manifest_json_path = (
    LP_HISTORY_OUT_DIR / "lp_training_configuration_manifest.json"
)
training_configuration_manifest_df.to_csv(manifest_csv_path, index=False)
with open(manifest_json_path, "w", encoding="utf-8") as handle:
    json.dump(
        training_configuration_manifest_df.to_dict(orient="records"),
        handle,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

print("Saved:", manifest_csv_path)
print("Saved:", manifest_json_path)

rq3_training_stat_scope_df = (
    run_config_df[
        [
            column
            for column in [
                "training_config_hash",
                "seed",
                "subgraph_method",
                "subgraph_seed",
                "subgraph_fraction_requested",
                "subgraph_fraction_actual",
                "training_candidate_size",
                "scoring_mode",
                "endpoint_mining_hop",
                "label_mode",
                "extreme_fraction",
                "selector_training_scope",
                "n_source_nodes",
                "n_global_nodes",
                "candidate_config_id",
                "lp_model_group_label",
            ]
            if column in run_config_df.columns
        ]
    ]
    .sort_values(
        [
            column
            for column in [
                "subgraph_method",
                "subgraph_fraction_requested",
                "training_candidate_size",
                "subgraph_seed",
                "scoring_mode",
                "label_mode",
                "seed",
            ]
            if column in run_config_df.columns
        ]
    )
    .reset_index(drop=True)
)

print(
    f"[RQ3 diagnostics] processing {len(RQ3_TRAINING_STAT_RUNS)} "
    f"trained selector runs across "
    f"{run_config_df['training_config_hash'].nunique()} complete "
    "configuration groups."
)
display(rq3_training_stat_scope_df)


# ============================================================
# Collect epoch and stopping statistics
# ============================================================

history_rows = []
stopping_rows = []

for run, config_record in zip(
    RQ3_TRAINING_STAT_RUNS,
    run_config_records,
):
    history = run["lp_model"].training_history

    for row in history.get("epoch_metrics", []):
        history_rows.append({
            **config_record,
            **row,
        })

    stopping_rows.append({
        **config_record,
        "best_epoch": history.get("best_epoch"),
        "stopped_at": history.get("stopped_at"),
        "early_stopped": int(bool(history.get("early_stopped"))),
        "best_train_loss": history.get("best_train_loss"),
        "best_val_loss": history.get("best_val_loss"),
        "generalization_gap_at_best": history.get(
            "generalization_gap_at_best"
        ),
        "training_runtime_seconds": history.get(
            "training_runtime_seconds"
        ),
        "training_seconds_per_epoch": history.get(
            "training_seconds_per_epoch"
        ),
        "mean_grad_norm": (
            float(np.mean(history.get("grad_norms", [])))
            if history.get("grad_norms") else np.nan
        ),
        "max_grad_norm": (
            float(np.max(history.get("grad_norms", [])))
            if history.get("grad_norms") else np.nan
        ),
        "final_grad_norm": (
            float(history.get("grad_norms", [np.nan])[-1])
            if history.get("grad_norms") else np.nan
        ),
    })

lp_history_raw_df = pd.DataFrame(history_rows)
lp_stopping_raw_df = pd.DataFrame(stopping_rows)

if lp_history_raw_df.empty:
    raise RuntimeError(
        "No epoch histories found. Use the updated "
        "selector_pipeline_helpers.py and rerun the LP training cell."
    )

expected_n_seeds = int(
    globals().get(
        "N_SEEDS",
        max(
            1,
            lp_history_raw_df.groupby(
                "training_config_hash"
            )["seed"].nunique().max(),
        ),
    )
)

seed_coverage = lp_history_raw_df.groupby(
    "training_config_hash"
)["seed"].nunique()
missing_seed_coverage = seed_coverage[
    seed_coverage != expected_n_seeds
]
if not missing_seed_coverage.empty:
    raise RuntimeError(
        "LP-training diagnostics are missing configured victim seeds:\n"
        + missing_seed_coverage.to_string()
    )

epoch_numeric_columns = [
    column
    for column in [
        "loss",
        "bce_unweighted",
        "mae",
        "mse",
        "rmse",
        "r2",
        "pearson",
        "spearman",
        "acc",
        "balanced_accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "mcc",
        "auc",
        "ap",
        "brier_soft",
        "brier_binary",
        "ece",
        "mce",
        "positive_rate",
        "predicted_positive_rate",
        "probability_mean",
        "probability_std",
        "logit_mean",
        "logit_std",
        "grad_norm",
        "learning_rate",
        "epoch_runtime_seconds",
    ]
    if column in lp_history_raw_df.columns
]

lp_history_summary_df = _mean_std_frame(
    lp_history_raw_df,
    ["training_config_hash", "split", "epoch"],
    epoch_numeric_columns,
)

# Add complete configuration columns to every exported averaged row.
lp_history_summary_df = lp_history_summary_df.merge(
    training_configuration_manifest_df,
    on="training_config_hash",
    how="left",
    validate="many_to_one",
)

# Curves are restricted to epochs represented by every configured seed.
lp_history_common_df = lp_history_summary_df[
    lp_history_summary_df["n_seeds_x"] == expected_n_seeds
].copy()

if lp_history_common_df.empty:
    raise RuntimeError(
        "No LP-training epochs are shared by every configured seed. "
        "Check seed coverage or reduce the expected N_SEEDS value."
    )

# Per-seed generalization gap, then average the gap itself.
gap_source = (
    lp_history_raw_df[
        lp_history_raw_df["split"].isin(["train", "validation"])
    ]
    .pivot_table(
        index=["seed", "training_config_hash", "epoch"],
        columns="split",
        values="loss",
        aggfunc="first",
    )
    .reset_index()
)

gap_source["validation_minus_train"] = (
    gap_source["validation"] - gap_source["train"]
)

lp_gap_summary_df = _mean_std_frame(
    gap_source.dropna(subset=["validation_minus_train"]),
    ["training_config_hash", "epoch"],
    ["validation_minus_train"],
).merge(
    training_configuration_manifest_df,
    on="training_config_hash",
    how="left",
    validate="many_to_one",
)

lp_gap_common_df = lp_gap_summary_df[
    lp_gap_summary_df["n_seeds_x"] == expected_n_seeds
].copy()

stopping_numeric = [
    "best_epoch",
    "stopped_at",
    "early_stopped",
    "best_train_loss",
    "best_val_loss",
    "generalization_gap_at_best",
    "training_runtime_seconds",
    "training_seconds_per_epoch",
    "mean_grad_norm",
    "max_grad_norm",
    "final_grad_norm",
]

lp_stopping_summary_df = _mean_std_frame(
    lp_stopping_raw_df,
    ["training_config_hash"],
    stopping_numeric,
).merge(
    training_configuration_manifest_df,
    on="training_config_hash",
    how="left",
    validate="one_to_one",
)

# Pandas adds suffixes because both the statistical summaries and manifest
# contain n_seeds. Give the statistical coverage column the clear name used
# below and retain manifest seed count as configuration_n_seeds.
for frame in [
    lp_history_summary_df,
    lp_history_common_df,
    lp_gap_summary_df,
    lp_gap_common_df,
    lp_stopping_summary_df,
]:
    if "n_seeds_x" in frame.columns:
        frame.rename(
            columns={
                "n_seeds_x": "n_seeds",
                "n_seeds_y": "configuration_n_seeds",
            },
            inplace=True,
        )

# Reapply the common-epoch filtering after the normalized column names.
lp_history_common_df = lp_history_summary_df[
    lp_history_summary_df["n_seeds"] == expected_n_seeds
].copy()
lp_gap_common_df = lp_gap_summary_df[
    lp_gap_summary_df["n_seeds"] == expected_n_seeds
].copy()


# ============================================================
# Export numerical results
# ============================================================

lp_history_raw_df.to_csv(
    LP_HISTORY_OUT_DIR / "lp_epoch_metrics_raw.csv",
    index=False,
)
lp_history_summary_df.to_csv(
    LP_HISTORY_OUT_DIR / "lp_epoch_metrics_averaged.csv",
    index=False,
)
lp_gap_summary_df.to_csv(
    LP_HISTORY_OUT_DIR / "lp_generalization_gap_averaged.csv",
    index=False,
)
lp_stopping_raw_df.to_csv(
    LP_HISTORY_OUT_DIR / "lp_stopping_summary_raw.csv",
    index=False,
)
lp_stopping_summary_df.to_csv(
    LP_HISTORY_OUT_DIR / "lp_stopping_summary_averaged.csv",
    index=False,
)


# ============================================================
# Thesis-ready plots
# ============================================================

with plt.rc_context(THESIS_RC):
    for config_hash, group in lp_history_common_df.groupby(
        "training_config_hash",
        sort=False,
    ):
        config = training_configuration_manifest_df[
            training_configuration_manifest_df["training_config_hash"]
            == config_hash
        ].iloc[0].to_dict()

        # Ensure figure text uses the seed coverage actually represented by the
        # common epoch curves.
        config["n_seeds"] = expected_n_seeds
        slug = _configuration_slug(config)

        stopping = lp_stopping_summary_df[
            lp_stopping_summary_df["training_config_hash"] == config_hash
        ].iloc[0]

        # ------------------------------------------------------
        # 1. Optimization and generalization: train/validation BCE
        # ------------------------------------------------------
        fig, ax = plt.subplots(figsize=(8.6, 5.4))
        for split_name, split_display in [
            ("train", "Training"),
            ("validation", "Validation"),
        ]:
            split_df = group[
                group["split"] == split_name
            ].sort_values("epoch")
            if split_df.empty:
                continue

            _plot_band(
                ax,
                split_df["epoch"],
                split_df["loss_mean"],
                split_df["loss_std"],
                f"{split_display} weighted BCE",
            )

        if pd.notna(stopping.get("best_epoch_mean")):
            ax.axvline(
                stopping["best_epoch_mean"],
                linestyle="--",
                linewidth=1.2,
                label=(
                    f"Mean best epoch "
                    f"({stopping['best_epoch_mean']:.1f})"
                ),
            )
        if pd.notna(stopping.get("stopped_at_mean")):
            ax.axvline(
                stopping["stopped_at_mean"],
                linestyle=":",
                linewidth=1.2,
                label=(
                    f"Mean stopping epoch "
                    f"({stopping['stopped_at_mean']:.1f})"
                ),
            )

        ax.set_xlabel("Epoch")
        ax.set_ylabel("Weighted BCE loss")
        ax.grid(True, alpha=0.25)
        ax.legend(frameon=False)
        _apply_thesis_layout(
            fig,
            ax,
            "LP-GNN optimization",
            config,
        )
        _save_thesis_figure(fig, f"loss_curves__{slug}")

        # ------------------------------------------------------
        # 2. Generalization gap
        # ------------------------------------------------------
        gap_df = lp_gap_common_df[
            lp_gap_common_df["training_config_hash"] == config_hash
        ].sort_values("epoch")

        if not gap_df.empty:
            fig, ax = plt.subplots(figsize=(8.6, 5.0))
            _plot_band(
                ax,
                gap_df["epoch"],
                gap_df["validation_minus_train_mean"],
                gap_df["validation_minus_train_std"],
                "Validation BCE − training BCE",
            )
            ax.axhline(0.0, linestyle="--", linewidth=1)
            ax.set_xlabel("Epoch")
            ax.set_ylabel("Generalization gap")
            ax.grid(True, alpha=0.25)
            ax.legend(frameon=False)
            _apply_thesis_layout(
                fig,
                ax,
                "LP-GNN generalization gap",
                config,
            )
            _save_thesis_figure(
                fig,
                f"generalization_gap__{slug}",
            )

        # ------------------------------------------------------
        # 3. Gradient norm
        # ------------------------------------------------------
        train_df = group[group["split"] == "train"].sort_values("epoch")
        if (
            not train_df.empty
            and "grad_norm_mean" in train_df.columns
            and train_df["grad_norm_mean"].notna().any()
        ):
            fig, ax = plt.subplots(figsize=(8.6, 5.0))
            _plot_band(
                ax,
                train_df["epoch"],
                train_df["grad_norm_mean"],
                train_df["grad_norm_std"],
                "Global L2 gradient norm",
            )
            if (train_df["grad_norm_mean"].dropna() > 0).all():
                ax.set_yscale("log")
            ax.set_xlabel("Epoch")
            ax.set_ylabel("Gradient norm")
            ax.grid(True, alpha=0.25)
            ax.legend(frameon=False)
            _apply_thesis_layout(
                fig,
                ax,
                "LP-GNN gradient stability",
                config,
            )
            _save_thesis_figure(fig, f"gradient_norm__{slug}")

        # ------------------------------------------------------
        # 4. Validation discrimination metrics
        # ------------------------------------------------------
        val_df = group[
            group["split"] == "validation"
        ].sort_values("epoch")

        discrimination_metrics = [
            ("auc", "ROC-AUC"),
            ("ap", "Average precision"),
            ("f1", "F1"),
            ("mcc", "MCC"),
            ("balanced_accuracy", "Balanced accuracy"),
        ]
        available = [
            (column, display_name)
            for column, display_name in discrimination_metrics
            if f"{column}_mean" in val_df.columns
            and val_df[f"{column}_mean"].notna().any()
        ]

        if available:
            fig, ax = plt.subplots(figsize=(8.6, 5.4))
            for column, display_name in available:
                _plot_band(
                    ax,
                    val_df["epoch"],
                    val_df[f"{column}_mean"],
                    val_df[f"{column}_std"],
                    display_name,
                    clip=(-1.0, 1.0),
                )
            ax.set_xlabel("Epoch")
            ax.set_ylabel("Validation metric")
            ax.set_ylim(-1.05, 1.05)
            ax.grid(True, alpha=0.25)
            ax.legend(frameon=False, ncol=2)
            _apply_thesis_layout(
                fig,
                ax,
                "LP-GNN validation discrimination",
                config,
            )
            _save_thesis_figure(
                fig,
                f"validation_discrimination__{slug}",
            )

        # ------------------------------------------------------
        # 5. Continuous-target fit metrics
        # ------------------------------------------------------
        continuous_metrics = [
            ("mae", "MAE"),
            ("rmse", "RMSE"),
        ]
        available_error = [
            (column, display_name)
            for column, display_name in continuous_metrics
            if f"{column}_mean" in val_df.columns
            and val_df[f"{column}_mean"].notna().any()
        ]

        if available_error:
            fig, ax = plt.subplots(figsize=(8.6, 5.1))
            for column, display_name in available_error:
                _plot_band(
                    ax,
                    val_df["epoch"],
                    val_df[f"{column}_mean"],
                    val_df[f"{column}_std"],
                    display_name,
                    clip=(0.0, np.inf),
                )
            ax.set_xlabel("Epoch")
            ax.set_ylabel("Validation error")
            ax.grid(True, alpha=0.25)
            ax.legend(frameon=False)
            _apply_thesis_layout(
                fig,
                ax,
                "LP-GNN continuous-target fit",
                config,
            )
            _save_thesis_figure(
                fig,
                f"validation_continuous_error__{slug}",
            )

        correlation_metrics = [
            ("pearson", "Pearson"),
            ("spearman", "Spearman"),
            ("r2", "R²"),
        ]
        available_corr = [
            (column, display_name)
            for column, display_name in correlation_metrics
            if f"{column}_mean" in val_df.columns
            and val_df[f"{column}_mean"].notna().any()
        ]

        if available_corr:
            fig, ax = plt.subplots(figsize=(8.6, 5.1))
            for column, display_name in available_corr:
                _plot_band(
                    ax,
                    val_df["epoch"],
                    val_df[f"{column}_mean"],
                    val_df[f"{column}_std"],
                    display_name,
                    clip=(-1.0, 1.0),
                )
            ax.axhline(0.0, linestyle="--", linewidth=1)
            ax.set_xlabel("Epoch")
            ax.set_ylabel("Validation association")
            ax.set_ylim(-1.05, 1.05)
            ax.grid(True, alpha=0.25)
            ax.legend(frameon=False)
            _apply_thesis_layout(
                fig,
                ax,
                "LP-GNN target-score association",
                config,
            )
            _save_thesis_figure(
                fig,
                f"validation_correlations__{slug}",
            )


# ============================================================
# Notebook summaries
# ============================================================

stopping_display_columns = [
    column
    for column in [
        "training_config_hash",
        "subgraph_method",
        "subgraph_fraction_requested",
        "subgraph_fraction_actual_mean",
        "subgraph_seed",
        "training_candidate_size",
        "scoring_mode",
        "label_mode",
        "best_epoch_mean",
        "best_epoch_std",
        "stopped_at_mean",
        "stopped_at_std",
        "best_train_loss_mean",
        "best_val_loss_mean",
        "generalization_gap_at_best_mean",
        "training_runtime_seconds_mean",
        "n_seeds",
    ]
    if column in lp_stopping_summary_df.columns
]

display(lp_stopping_summary_df[stopping_display_columns])

In [ ]:
from pathlib import Path
from IPython.display import display
import hashlib
import json
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
)


# ============================================================
# Output and plot configuration
# ============================================================

LP_HELDOUT_OUT_DIR = Path("extendedPlotting") / "lp_training_heldout_multiseed"
LP_HELDOUT_OUT_DIR.mkdir(parents=True, exist_ok=True)

HELDOUT_PLOT_DIR = Path(
    globals().get(
        "RUN_PLOTS_DIR",
        LP_HELDOUT_OUT_DIR,
    )
)
HELDOUT_PLOT_DIR.mkdir(parents=True, exist_ok=True)

# PNG is convenient for inspection; PDF and SVG retain vector text/lines for
# direct inclusion in a thesis.
HELDOUT_EXPORT_FORMATS = tuple(
    globals().get(
        "HELDOUT_EXPORT_FORMATS",
        ("png", "pdf", "svg"),
    )
)
HELDOUT_PNG_DPI = int(globals().get("HELDOUT_PNG_DPI", 300))
HELDOUT_SHOW_PLOTS = bool(globals().get("HELDOUT_SHOW_PLOTS", True))

FPR_GRID = np.linspace(0.0, 1.0, 501)
RECALL_GRID = np.linspace(0.0, 1.0, 501)

THESIS_RC = {
    "font.size": 10.0,
    "axes.titlesize": 12.0,
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9.0,
    "ytick.labelsize": 9.0,
    "legend.fontsize": 8.8,
    "legend.title_fontsize": 8.8,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.8,
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
}


# ============================================================
# Configuration metadata
# ============================================================

# Fields that define a like-for-like held-out configuration. In particular,
# subgraph seed is deliberately kept separate from the victim seed over which
# means and standard deviations are computed.
HELDOUT_CONFIG_ID_FIELDS = [
    "lp_model_group_label",
    "candidate_config_id",
    "subgraph_method",
    "subgraph_seed",
    "subgraph_fraction_requested",
    "training_candidate_size",
    "scoring_mode",
    "endpoint_mining_hop",
    "label_mode",
    "extreme_fraction",
    "selector_training_scope",
    "prbcd_candidate_fraction",
]

# Additional fields that are exported and displayed when available.
HELDOUT_CONFIG_METADATA_FIELDS = [
    "candidate_set_size",
    "subgraph_fraction_actual",
    "construction_seed",
    "candidate_pool_seed",
    "mining_seed",
    "n_source_nodes",
    "n_target_nodes",
    "n_selector_nodes",
    "n_global_nodes",
    "n_nodes_global",
    "n_test_nodes_local",
    "n_edges_local_undirected",
    "density_local",
    "global_edge_coverage",
    "actual_prbcd_fraction",
    "n_prbcd_requested",
    "n_prbcd_mined_unique",
    "n_prbcd_selected",
    "n_random_candidates",
]


def _heldout_safe_file(value, max_length=150):
    helper = globals().get("_safe_config_value")
    if callable(helper):
        safe = str(helper(value))
    else:
        safe = (
            str(value)
            .replace("/", "-")
            .replace("\\", "-")
            .replace(" ", "_")
            .replace(":", "-")
            .replace(".", "p")
        )

    if len(safe) <= max_length:
        return safe

    digest = hashlib.sha1(safe.encode("utf-8")).hexdigest()[:10]
    prefix_length = max(20, max_length - len(digest) - 2)
    return f"{safe[:prefix_length].rstrip('._-')}__{digest}"


def _clean_config_value(value):
    """Return a stable scalar suitable for IDs, tables, and JSON output."""
    if value is None:
        return ""

    if isinstance(value, np.generic):
        value = value.item()

    if isinstance(value, float):
        if not np.isfinite(value):
            return ""
        return float(value)

    if isinstance(value, (int, bool, str)):
        return value

    return str(value)


def _get_run_value(run, key, default=""):
    value = run.get(key, default)

    # Some graph-construction diagnostics are nested in subgraph_stats.
    if value in (None, ""):
        subgraph_stats = run.get("subgraph_stats", {})
        if isinstance(subgraph_stats, dict):
            value = subgraph_stats.get(key, default)

    return _clean_config_value(value)


def _build_heldout_config_record(run):
    record = {
        "seed": int(run["seed"]),
        "lp_model_group_label": str(run.get("lp_model_group_label", "")),
        "lp_model_label": str(run.get("lp_model_label", "")),
    }

    for key in HELDOUT_CONFIG_ID_FIELDS + HELDOUT_CONFIG_METADATA_FIELDS:
        if key not in record:
            record[key] = _get_run_value(run, key)

    requested_fraction = record.get("subgraph_fraction_requested", "")
    method = str(record.get("subgraph_method", "")).strip()

    if not method and requested_fraction != "":
        try:
            if np.isclose(float(requested_fraction), 1.0):
                method = "whole_graph"
        except (TypeError, ValueError):
            pass

    record["subgraph_method"] = method or "unknown"

    # Fill common node-count aliases without forcing the upstream run schema.
    if record.get("n_global_nodes", "") == "":
        record["n_global_nodes"] = record.get("n_nodes_global", "")
    if record.get("n_nodes_global", "") == "":
        record["n_nodes_global"] = record.get("n_global_nodes", "")
    if record.get("n_selector_nodes", "") == "":
        record["n_selector_nodes"] = record.get("n_source_nodes", "")
    if record.get("n_source_nodes", "") == "":
        record["n_source_nodes"] = record.get("n_selector_nodes", "")

    identity_payload = {
        key: record.get(key, "")
        for key in HELDOUT_CONFIG_ID_FIELDS
    }
    canonical_payload = json.dumps(
        identity_payload,
        sort_keys=True,
        ensure_ascii=True,
        separators=(",", ":"),
    )
    config_hash = hashlib.sha1(
        canonical_payload.encode("utf-8")
    ).hexdigest()[:10]

    record["heldout_config_hash"] = config_hash
    record["heldout_config_id"] = canonical_payload
    return record


def _curve_band(ax, x, mean, std, label):
    x = np.asarray(x, dtype=float)
    mean = np.asarray(mean, dtype=float)
    std = np.asarray(std, dtype=float)

    lower = np.clip(mean - std, 0.0, 1.0)
    upper = np.clip(mean + std, 0.0, 1.0)

    line = ax.plot(x, mean, label=label)[0]
    ax.fill_between(
        x,
        lower,
        upper,
        alpha=0.18,
        color=line.get_color(),
        linewidth=0,
        label="±1 SD across victim seeds",
    )


def _pretty_method(value):
    mapping = {
        "bfs": "BFS",
        "ties": "TIES",
        "forest_fire": "Forest Fire",
        "whole_graph": "Whole graph",
    }
    value = str(value).strip()
    return mapping.get(value, value.replace("_", " ").title() or "Unknown")


def _pretty_token(value):
    value = str(value).strip()
    if not value:
        return "not specified"
    return value.replace("_", " ")


def _format_fraction(value, decimals=1):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return "not specified"

    if not np.isfinite(number):
        return "not specified"

    percentage = 100.0 * number
    if np.isclose(percentage, round(percentage)):
        return f"{percentage:.0f}%"
    return f"{percentage:.{decimals}f}%"


def _format_number(value, decimals=0):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return "not specified"

    if not np.isfinite(number):
        return "not specified"

    if decimals == 0 and np.isclose(number, round(number)):
        return f"{int(round(number)):,}"
    return f"{number:,.{decimals}f}"


def _compact_text(value, max_chars=58):
    text = str(value).strip()
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 1].rstrip() + "…"


def _configuration_slug(config):
    parts = [
        f"method-{_heldout_safe_file(config.get('subgraph_method', 'unknown'), 24)}",
        f"frac-{_heldout_safe_file(config.get('subgraph_fraction_requested', 'na'), 18)}",
        f"subseed-{_heldout_safe_file(config.get('subgraph_seed', 'na'), 18)}",
        f"cand-{_heldout_safe_file(config.get('training_candidate_size', 'na'), 20)}",
        f"score-{_heldout_safe_file(config.get('scoring_mode', 'na'), 28)}",
        f"label-{_heldout_safe_file(config.get('label_mode', 'na'), 22)}",
        f"cfg-{config.get('heldout_config_hash', 'unknown')}",
    ]
    return _heldout_safe_file("__".join(parts), max_length=170)


def _is_present(value):
    if value is None:
        return False
    if isinstance(value, float) and not np.isfinite(value):
        return False
    text = str(value).strip().lower()
    return text not in {"", "nan", "none"}


def _configuration_text(config):
    method = _pretty_method(config.get("subgraph_method", "unknown"))
    requested = _format_fraction(config.get("subgraph_fraction_requested", ""))
    actual_mean = _format_fraction(
        config.get(
            "subgraph_fraction_actual_mean",
            config.get("subgraph_fraction_actual", ""),
        ),
        decimals=2,
    )
    training_candidates = _format_number(
        config.get("training_candidate_size", "")
    )

    if method == "Whole graph":
        subtitle = (
            f"Whole-graph selector · {training_candidates} training candidates"
        )
    else:
        subtitle = (
            f"{method} source subgraph · requested {requested} "
            f"(actual {actual_mean}) · {training_candidates} training candidates"
        )

    details = [
        f"Scoring: {_pretty_token(config.get('scoring_mode', ''))}",
        f"Labels: {_pretty_token(config.get('label_mode', ''))}",
        f"Subgraph seed: {config.get('subgraph_seed', 'not specified')}",
    ]

    endpoint_hop = config.get("endpoint_mining_hop", "")
    if _is_present(endpoint_hop):
        details.append(f"Endpoint hop: {endpoint_hop}")

    extreme_fraction = config.get("extreme_fraction", "")
    if _is_present(extreme_fraction):
        details.append(
            f"Extreme fraction: {_format_fraction(extreme_fraction)}"
        )

    selector_scope = config.get("selector_training_scope", "")
    if _is_present(selector_scope):
        details.append(f"Scope: {_pretty_token(selector_scope)}")

    prbcd_fraction = config.get("prbcd_candidate_fraction", "")
    if _is_present(prbcd_fraction):
        try:
            if float(prbcd_fraction) > 0:
                details.append(
                    f"Requested PRBCD share: {_format_fraction(prbcd_fraction)}"
                )
        except (TypeError, ValueError):
            pass

    actual_prbcd = config.get(
        "actual_prbcd_fraction_mean",
        config.get("actual_prbcd_fraction", ""),
    )
    if _is_present(actual_prbcd):
        try:
            if float(actual_prbcd) > 0:
                details.append(
                    f"Actual PRBCD share: {_format_fraction(actual_prbcd)}"
                )
        except (TypeError, ValueError):
            pass

    source_nodes = config.get(
        "n_source_nodes_mean",
        config.get(
            "n_selector_nodes_mean",
            config.get("n_source_nodes", config.get("n_selector_nodes", "")),
        ),
    )
    global_nodes = config.get(
        "n_global_nodes_mean",
        config.get(
            "n_nodes_global_mean",
            config.get("n_global_nodes", config.get("n_nodes_global", "")),
        ),
    )
    if _is_present(source_nodes) and _is_present(global_nodes):
        details.append(
            f"Selector nodes: {_format_number(source_nodes)}/{_format_number(global_nodes)}"
        )

    n_seeds = config.get("n_seeds", config.get("seed_count", ""))
    if _is_present(n_seeds):
        details.append(f"Victim seeds: {_format_number(n_seeds)}")

    candidate_config = config.get("candidate_config_id", "")
    if _is_present(candidate_config):
        details.append(
            f"Candidate config: {_compact_text(candidate_config, 46)}"
        )

    details.append(f"Configuration: {config.get('heldout_config_hash', 'unknown')}")

    detail_text = " · ".join(details)
    detail_text = "\n".join(
        textwrap.wrap(
            detail_text,
            width=118,
            break_long_words=False,
            break_on_hyphens=False,
        )
    )
    return subtitle, detail_text


def _apply_thesis_layout(
    fig,
    ax,
    title,
    config,
    *,
    top=0.78,
    bottom=0.21,
    left=0.14,
    right=0.97,
):
    subtitle, detail_text = _configuration_text(config)

    fig.suptitle(
        title,
        x=left,
        y=0.975,
        ha="left",
        va="top",
        fontsize=14,
        fontweight="semibold",
    )
    fig.text(
        left,
        0.915,
        subtitle,
        ha="left",
        va="top",
        fontsize=9.6,
    )
    fig.text(
        left,
        0.025,
        detail_text,
        ha="left",
        va="bottom",
        fontsize=8.1,
        color="0.35",
        linespacing=1.3,
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out")
    fig.subplots_adjust(
        top=top,
        bottom=bottom,
        left=left,
        right=right,
    )


def _save_thesis_figure(fig, stem):
    for extension in HELDOUT_EXPORT_FORMATS:
        extension = str(extension).lower().lstrip(".")
        output_path = HELDOUT_PLOT_DIR / f"{stem}.{extension}"
        save_kwargs = {
            "bbox_inches": "tight",
            "facecolor": "white",
        }
        if extension == "png":
            save_kwargs["dpi"] = HELDOUT_PNG_DPI

        fig.savefig(output_path, **save_kwargs)
        print("Saved:", output_path)

    if HELDOUT_SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)


# ============================================================
# Collect held-out predictions and metrics
# ============================================================

roc_curve_rows = []
pr_curve_rows = []
calibration_rows = []
roc_metric_rows = []
confusion_rows = []

# Binary endpoint targets and continuous subset-accuracy-drop targets require
# different held-out evaluations. Endpoint runs continue through the original
# ROC/PR/calibration/confusion calculation below without any change. The
# continuous subset runs are collected for a separate regression/ranking
# evaluation appended at the end of this cell.
eligible_lp_training_runs = []
subset_accuracy_drop_runs = []

for run in RQ3_TRAINING_STAT_RUNS:
    scoring_mode = str(run.get("scoring_mode", ""))
    model_label = str(run.get("lp_model_label", ""))

    is_subset_accuracy_drop = (
        scoring_mode == "subset_accuracy_drop"
        or "score-subset_accuracy_drop" in model_label
    )

    if is_subset_accuracy_drop:
        subset_accuracy_drop_runs.append(run)
        print(
            "[CONTINUOUS HELD-OUT] queued subset_accuracy_drop: "
            f"{model_label}"
        )
        continue

    eligible_lp_training_runs.append(run)

if not eligible_lp_training_runs:
    raise RuntimeError(
        "No endpoint LP-GNN runs are available for the existing binary "
        "held-out evaluation."
    )

print(
    f"[HELD-OUT] binary endpoint evaluation: "
    f"{len(eligible_lp_training_runs)} runs; "
    f"continuous subset evaluation: {len(subset_accuracy_drop_runs)} runs."
)

for run in eligible_lp_training_runs:
    history = run["lp_model"].training_history
    outputs = history["final_split_outputs"]["test"]
    metrics = history["final_split_metrics"]["test"]

    y_true = np.asarray(outputs["binary_targets"]).astype(int).reshape(-1)
    y_score = np.asarray(outputs["probabilities"]).astype(float).reshape(-1)

    if np.unique(y_true).size < 2:
        raise RuntimeError(
            f"Held-out ROC/AP undefined for {run['lp_model_label']}: "
            "test split contains only one class."
        )

    fpr, tpr, _ = roc_curve(y_true, y_score)
    tpr_interp = np.interp(FPR_GRID, fpr, tpr)
    tpr_interp[0] = 0.0
    tpr_interp[-1] = 1.0

    precision, recall, _ = precision_recall_curve(y_true, y_score)
    order = np.argsort(recall)
    recall_sorted = recall[order]
    precision_sorted = precision[order]
    precision_interp = np.interp(
        RECALL_GRID,
        recall_sorted,
        precision_sorted,
    )

    base = _build_heldout_config_record(run)

    roc_curve_rows.extend({
        **base,
        "fpr": float(x),
        "tpr": float(y),
    } for x, y in zip(FPR_GRID, tpr_interp))

    pr_curve_rows.extend({
        **base,
        "recall": float(x),
        "precision": float(y),
    } for x, y in zip(RECALL_GRID, precision_interp))

    roc_metric_rows.append({
        **base,
        "auc": float(roc_auc_score(y_true, y_score)),
        "ap": float(average_precision_score(y_true, y_score)),
        "n_samples": int(y_true.size),
        "n_positive": int(y_true.sum()),
        "n_negative": int((y_true == 0).sum()),
        "positive_rate": float(y_true.mean()),
        **{
            key: value
            for key, value in metrics.items()
            if key != "calibration_bins"
        },
    })

    for calibration_bin in metrics.get("calibration_bins", []):
        calibration_rows.append({**base, **calibration_bin})

    tn = int(metrics["tn"])
    fp = int(metrics["fp"])
    fn = int(metrics["fn"])
    tp = int(metrics["tp"])
    negative_total = max(1, tn + fp)
    positive_total = max(1, tp + fn)
    confusion_rows.extend([
        {
            **base,
            "true_class": "negative",
            "predicted_class": "negative",
            "count": tn,
            "row_fraction": tn / negative_total,
        },
        {
            **base,
            "true_class": "negative",
            "predicted_class": "positive",
            "count": fp,
            "row_fraction": fp / negative_total,
        },
        {
            **base,
            "true_class": "positive",
            "predicted_class": "negative",
            "count": fn,
            "row_fraction": fn / positive_total,
        },
        {
            **base,
            "true_class": "positive",
            "predicted_class": "positive",
            "count": tp,
            "row_fraction": tp / positive_total,
        },
    ])

roc_curve_raw_df = pd.DataFrame(roc_curve_rows)
pr_curve_raw_df = pd.DataFrame(pr_curve_rows)
calibration_raw_df = pd.DataFrame(calibration_rows)
roc_metrics_raw_df = pd.DataFrame(roc_metric_rows)
confusion_raw_df = pd.DataFrame(confusion_rows)


# ============================================================
# Validate seed coverage per complete configuration
# ============================================================

expected_n_seeds = int(
    globals().get(
        "N_SEEDS",
        max(
            1,
            roc_metrics_raw_df
            .groupby("heldout_config_id")["seed"]
            .nunique()
            .max(),
        ),
    )
)

seed_coverage = (
    roc_metrics_raw_df
    .groupby("heldout_config_id")["seed"]
    .nunique()
)
missing_seed_coverage = seed_coverage[seed_coverage != expected_n_seeds]
if not missing_seed_coverage.empty:
    readable_missing = (
        roc_metrics_raw_df[
            roc_metrics_raw_df["heldout_config_id"].isin(
                missing_seed_coverage.index
            )
        ][
            [
                "heldout_config_id",
                "heldout_config_hash",
                "subgraph_method",
                "subgraph_fraction_requested",
                "subgraph_seed",
                "training_candidate_size",
                "scoring_mode",
                "label_mode",
            ]
        ]
        .drop_duplicates()
        .merge(
            missing_seed_coverage.rename("n_seeds"),
            left_on="heldout_config_id",
            right_index=True,
            how="left",
        )
        if "heldout_config_id" in roc_metrics_raw_df.columns
        else missing_seed_coverage
    )
    raise RuntimeError(
        "Held-out evaluation is missing configured victim seeds for at least "
        "one complete configuration:\n"
        + str(readable_missing)
    )


# ============================================================
# Configuration manifest and seed aggregation
# ============================================================

stable_manifest_fields = [
    "heldout_config_id",
    "heldout_config_hash",
    "lp_model_group_label",
    "candidate_config_id",
    "subgraph_method",
    "subgraph_seed",
    "subgraph_fraction_requested",
    "training_candidate_size",
    "candidate_set_size",
    "scoring_mode",
    "endpoint_mining_hop",
    "label_mode",
    "extreme_fraction",
    "selector_training_scope",
    "prbcd_candidate_fraction",
]
stable_manifest_fields = [
    column
    for column in stable_manifest_fields
    if column in roc_metrics_raw_df.columns
]

# Fail loudly if a supposedly stable field differs across victim seeds within
# a complete configuration. This protects the thesis figures from mislabeled
# or accidentally pooled runs.
for column in stable_manifest_fields:
    if column in {"heldout_config_id", "heldout_config_hash"}:
        continue
    unique_counts = (
        roc_metrics_raw_df
        .groupby("heldout_config_id")[column]
        .nunique(dropna=False)
    )
    inconsistent = unique_counts[unique_counts > 1]
    if not inconsistent.empty:
        raise RuntimeError(
            f"Configuration field {column!r} varies across victim seeds "
            "inside one held-out configuration. Refusing to aggregate."
        )

configuration_manifest_df = (
    roc_metrics_raw_df[stable_manifest_fields]
    .drop_duplicates(subset=["heldout_config_id"])
    .copy()
)

varying_metadata_fields = [
    column
    for column in [
        "subgraph_fraction_actual",
        "n_source_nodes",
        "n_target_nodes",
        "n_selector_nodes",
        "n_global_nodes",
        "n_nodes_global",
        "n_test_nodes_local",
        "n_edges_local_undirected",
        "density_local",
        "global_edge_coverage",
        "actual_prbcd_fraction",
        "n_prbcd_requested",
        "n_prbcd_mined_unique",
        "n_prbcd_selected",
        "n_random_candidates",
    ]
    if column in roc_metrics_raw_df.columns
]

if varying_metadata_fields:
    numeric_metadata = roc_metrics_raw_df[
        ["heldout_config_id"] + varying_metadata_fields
    ].copy()

    for column in varying_metadata_fields:
        numeric_metadata[column] = pd.to_numeric(
            numeric_metadata[column],
            errors="coerce",
        )

    metadata_aggregation = {}
    for column in varying_metadata_fields:
        metadata_aggregation[f"{column}_mean"] = (column, "mean")
        metadata_aggregation[f"{column}_std"] = (column, "std")
        metadata_aggregation[f"{column}_min"] = (column, "min")
        metadata_aggregation[f"{column}_max"] = (column, "max")

    varying_manifest_df = (
        numeric_metadata
        .groupby("heldout_config_id", as_index=False)
        .agg(**metadata_aggregation)
    )

    std_columns = [
        column
        for column in varying_manifest_df.columns
        if column.endswith("_std")
    ]
    varying_manifest_df[std_columns] = (
        varying_manifest_df[std_columns].fillna(0.0)
    )

    configuration_manifest_df = configuration_manifest_df.merge(
        varying_manifest_df,
        on="heldout_config_id",
        how="left",
        validate="one_to_one",
    )

seed_count_df = (
    roc_metrics_raw_df
    .groupby("heldout_config_id", as_index=False)
    .agg(n_seeds=("seed", "nunique"))
)
configuration_manifest_df = configuration_manifest_df.merge(
    seed_count_df,
    on="heldout_config_id",
    how="left",
    validate="one_to_one",
)

heldout_metric_columns = [
    column for column in [
        "auc", "ap", "n_samples", "n_positive", "n_negative",
        "positive_rate", "loss", "bce_unweighted",
        "acc", "balanced_accuracy", "precision", "recall",
        "specificity", "f1", "mcc",
        "mae", "mse", "rmse", "r2", "pearson", "spearman",
        "brier_soft", "brier_binary", "ece", "mce",
        "predicted_positive_rate",
        "probability_mean", "probability_std",
    ]
    if column in roc_metrics_raw_df.columns
]

roc_metrics_df = aggregate_over_seeds(
    roc_metrics_raw_df,
    ["heldout_config_id"],
    heldout_metric_columns,
).merge(
    configuration_manifest_df,
    on="heldout_config_id",
    how="left",
    validate="one_to_one",
)

roc_curve_df = (
    roc_curve_raw_df
    .groupby(["heldout_config_id", "fpr"], as_index=False)
    .agg(
        tpr_mean=("tpr", "mean"),
        tpr_std=("tpr", "std"),
        n_seeds=("seed", "nunique"),
    )
)
roc_curve_df["tpr_std"] = roc_curve_df["tpr_std"].fillna(0.0)

pr_curve_df = (
    pr_curve_raw_df
    .groupby(["heldout_config_id", "recall"], as_index=False)
    .agg(
        precision_mean=("precision", "mean"),
        precision_std=("precision", "std"),
        n_seeds=("seed", "nunique"),
    )
)
pr_curve_df["precision_std"] = pr_curve_df["precision_std"].fillna(0.0)

calibration_df = (
    calibration_raw_df
    .dropna(subset=["mean_probability", "observed_positive_fraction"])
    .groupby(
        ["heldout_config_id", "bin", "lower", "upper"],
        as_index=False,
    )
    .agg(
        mean_probability_mean=("mean_probability", "mean"),
        mean_probability_std=("mean_probability", "std"),
        observed_fraction_mean=("observed_positive_fraction", "mean"),
        observed_fraction_std=("observed_positive_fraction", "std"),
        count_mean=("count", "mean"),
        n_seeds=("seed", "nunique"),
    )
)
for column in ["mean_probability_std", "observed_fraction_std"]:
    calibration_df[column] = calibration_df[column].fillna(0.0)

confusion_df = (
    confusion_raw_df
    .groupby(
        ["heldout_config_id", "true_class", "predicted_class"],
        as_index=False,
    )
    .agg(
        row_fraction_mean=("row_fraction", "mean"),
        row_fraction_std=("row_fraction", "std"),
        count_mean=("count", "mean"),
        n_seeds=("seed", "nunique"),
    )
)
confusion_df["row_fraction_std"] = (
    confusion_df["row_fraction_std"].fillna(0.0)
)


# ============================================================
# Export numerical results and complete configuration manifest
# ============================================================

roc_curve_raw_df.to_csv(
    LP_HELDOUT_OUT_DIR / "heldout_roc_curve_raw.csv", index=False
)
roc_curve_df.to_csv(
    LP_HELDOUT_OUT_DIR / "heldout_roc_curve_averaged.csv", index=False
)
pr_curve_raw_df.to_csv(
    LP_HELDOUT_OUT_DIR / "heldout_pr_curve_raw.csv", index=False
)
pr_curve_df.to_csv(
    LP_HELDOUT_OUT_DIR / "heldout_pr_curve_averaged.csv", index=False
)
calibration_raw_df.to_csv(
    LP_HELDOUT_OUT_DIR / "heldout_calibration_raw.csv", index=False
)
calibration_df.to_csv(
    LP_HELDOUT_OUT_DIR / "heldout_calibration_averaged.csv", index=False
)
roc_metrics_raw_df.to_csv(
    LP_HELDOUT_OUT_DIR / "heldout_metrics_raw.csv", index=False
)
roc_metrics_df.to_csv(
    LP_HELDOUT_OUT_DIR / "heldout_metrics_averaged.csv", index=False
)
confusion_raw_df.to_csv(
    LP_HELDOUT_OUT_DIR / "heldout_confusion_raw.csv", index=False
)
confusion_df.to_csv(
    LP_HELDOUT_OUT_DIR / "heldout_confusion_averaged.csv", index=False
)
configuration_manifest_df.to_csv(
    LP_HELDOUT_OUT_DIR / "heldout_configuration_manifest.csv", index=False
)

with open(
    LP_HELDOUT_OUT_DIR / "heldout_configuration_manifest.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        configuration_manifest_df.to_dict(orient="records"),
        handle,
        indent=2,
        ensure_ascii=False,
        default=str,
    )


# ============================================================
# Thesis-ready held-out plots
# ============================================================

with plt.rc_context(THESIS_RC):
    for config_id in roc_metrics_df["heldout_config_id"]:
        metric = roc_metrics_df[
            roc_metrics_df["heldout_config_id"] == config_id
        ].iloc[0]
        config = metric.to_dict()
        safe_config = _configuration_slug(config)

        # ------------------------------------------------------
        # 1. Held-out ROC
        # ------------------------------------------------------
        group = roc_curve_df[
            roc_curve_df["heldout_config_id"] == config_id
        ].sort_values("fpr")

        fig, ax = plt.subplots(figsize=(6.6, 6.0))
        _curve_band(
            ax,
            group["fpr"],
            group["tpr_mean"],
            group["tpr_std"],
            (
                f"Mean ROC: AUC {metric['auc_mean']:.3f} "
                f"± {metric['auc_std']:.3f}"
            ),
        )
        ax.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            linewidth=1.0,
            label="Random classifier",
        )
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("False-positive rate")
        ax.set_ylabel("True-positive rate")
        ax.grid(True, alpha=0.22, linewidth=0.7)
        ax.legend(loc="lower right", frameon=False)
        _apply_thesis_layout(
            fig,
            ax,
            "Held-out ROC curve",
            config,
            top=0.77,
            bottom=0.24,
        )
        _save_thesis_figure(
            fig,
            f"heldout_roc__{safe_config}",
        )

        # ------------------------------------------------------
        # 2. Held-out precision-recall
        # ------------------------------------------------------
        group = pr_curve_df[
            pr_curve_df["heldout_config_id"] == config_id
        ].sort_values("recall")

        fig, ax = plt.subplots(figsize=(6.6, 6.0))
        _curve_band(
            ax,
            group["recall"],
            group["precision_mean"],
            group["precision_std"],
            (
                f"Mean PR: AP {metric['ap_mean']:.3f} "
                f"± {metric['ap_std']:.3f}"
            ),
        )
        ax.axhline(
            metric["positive_rate_mean"],
            linestyle="--",
            linewidth=1.0,
            label=(
                "Positive-rate baseline "
                f"({metric['positive_rate_mean']:.3f})"
            ),
        )
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
        ax.grid(True, alpha=0.22, linewidth=0.7)
        ax.legend(loc="best", frameon=False)
        _apply_thesis_layout(
            fig,
            ax,
            "Held-out precision–recall curve",
            config,
            top=0.77,
            bottom=0.24,
        )
        _save_thesis_figure(
            fig,
            f"heldout_pr__{safe_config}",
        )

        # ------------------------------------------------------
        # 3. Calibration
        # ------------------------------------------------------
        group = calibration_df[
            calibration_df["heldout_config_id"] == config_id
        ].sort_values("bin")

        if not group.empty:
            fig, ax = plt.subplots(figsize=(6.6, 6.0))
            ax.errorbar(
                group["mean_probability_mean"],
                group["observed_fraction_mean"],
                xerr=group["mean_probability_std"],
                yerr=group["observed_fraction_std"],
                marker="o",
                markersize=4.5,
                capsize=3,
                linewidth=1.5,
                label=(
                    f"Mean calibration: ECE {metric['ece_mean']:.3f} "
                    f"± {metric['ece_std']:.3f}"
                ),
            )
            ax.plot(
                [0, 1],
                [0, 1],
                linestyle="--",
                linewidth=1.0,
                label="Perfect calibration",
            )
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.set_aspect("equal", adjustable="box")
            ax.set_xlabel("Mean predicted probability")
            ax.set_ylabel("Observed harmful-edge fraction")
            ax.grid(True, alpha=0.22, linewidth=0.7)
            ax.legend(loc="best", frameon=False)
            _apply_thesis_layout(
                fig,
                ax,
                "Held-out calibration",
                config,
                top=0.77,
                bottom=0.24,
            )
            _save_thesis_figure(
                fig,
                f"heldout_calibration__{safe_config}",
            )

        # ------------------------------------------------------
        # 4. Row-normalized confusion matrix
        # ------------------------------------------------------
        group = confusion_df[
            confusion_df["heldout_config_id"] == config_id
        ]
        matrix = np.zeros((2, 2), dtype=float)
        matrix_std = np.zeros((2, 2), dtype=float)
        class_to_index = {"negative": 0, "positive": 1}

        for row in group.itertuples():
            i = class_to_index[row.true_class]
            j = class_to_index[row.predicted_class]
            matrix[i, j] = row.row_fraction_mean
            matrix_std[i, j] = row.row_fraction_std

        fig, ax = plt.subplots(figsize=(6.5, 5.8))
        image = ax.imshow(matrix, vmin=0.0, vmax=1.0)

        for i in range(2):
            for j in range(2):
                text_color = "white" if matrix[i, j] >= 0.55 else "black"
                ax.text(
                    j,
                    i,
                    f"{matrix[i, j]:.3f}\n± {matrix_std[i, j]:.3f}",
                    ha="center",
                    va="center",
                    color=text_color,
                    fontsize=10,
                )

        ax.set_xticks(
            [0, 1],
            ["Predicted\nnon-harmful", "Predicted\nharmful"],
        )
        ax.set_yticks(
            [0, 1],
            ["True non-harmful", "True harmful"],
        )
        ax.set_xlabel("Predicted class")
        ax.set_ylabel("True class")
        colorbar = fig.colorbar(
            image,
            ax=ax,
            fraction=0.046,
            pad=0.04,
        )
        colorbar.set_label("Mean row fraction")
        _apply_thesis_layout(
            fig,
            ax,
            "Held-out confusion matrix",
            config,
            top=0.75,
            bottom=0.27,
            left=0.18,
            right=0.90,
        )
        _save_thesis_figure(
            fig,
            f"heldout_confusion__{safe_config}",
        )

        # ------------------------------------------------------
        # 5a. Higher-is-better metrics
        # ------------------------------------------------------
        higher_better = [
            ("auc", "ROC-AUC"),
            ("ap", "Average precision"),
            ("balanced_accuracy", "Balanced accuracy"),
            ("f1", "F1"),
            ("mcc", "MCC"),
        ]
        higher_better = [
            (key, name)
            for key, name in higher_better
            if f"{key}_mean" in metric.index
        ]

        if higher_better:
            fig, ax = plt.subplots(figsize=(7.4, 5.2))
            names = [name for _, name in higher_better]
            means = [metric[f"{key}_mean"] for key, _ in higher_better]
            stds = [metric[f"{key}_std"] for key, _ in higher_better]
            bars = ax.bar(
                names,
                means,
                yerr=stds,
                capsize=4,
                width=0.68,
            )
            ax.axhline(0.0, linewidth=0.9)
            ax.set_ylim(-1.05, 1.05)
            ax.set_ylabel("Held-out metric")
            ax.tick_params(axis="x", rotation=18)
            ax.grid(True, axis="y", alpha=0.22, linewidth=0.7)

            for bar, mean, std in zip(bars, means, stds):
                y = mean + (0.05 if mean >= 0 else -0.08)
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    y,
                    f"{mean:.3f}\n± {std:.3f}",
                    ha="center",
                    va="bottom" if mean >= 0 else "top",
                    fontsize=8.2,
                )

            _apply_thesis_layout(
                fig,
                ax,
                "Held-out discrimination summary",
                config,
                top=0.74,
                bottom=0.28,
                left=0.12,
            )
            _save_thesis_figure(
                fig,
                f"heldout_metric_summary__{safe_config}",
            )

        # ------------------------------------------------------
        # 5b. Lower-is-better metrics
        # ------------------------------------------------------
        lower_better = [
            ("loss", "Weighted BCE"),
            ("rmse", "RMSE"),
            ("brier_binary", "Binary Brier"),
            ("ece", "ECE"),
        ]
        lower_better = [
            (key, name)
            for key, name in lower_better
            if f"{key}_mean" in metric.index
        ]

        if lower_better:
            fig, ax = plt.subplots(figsize=(7.4, 5.2))
            names = [name for _, name in lower_better]
            means = [metric[f"{key}_mean"] for key, _ in lower_better]
            stds = [metric[f"{key}_std"] for key, _ in lower_better]
            bars = ax.bar(
                names,
                means,
                yerr=stds,
                capsize=4,
                width=0.68,
            )
            ax.set_ylabel("Held-out error")
            ax.tick_params(axis="x", rotation=18)
            ax.grid(True, axis="y", alpha=0.22, linewidth=0.7)

            upper_limit = ax.get_ylim()[1]
            offset = max(upper_limit * 0.025, 0.002)
            for bar, mean, std in zip(bars, means, stds):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    mean + std + offset,
                    f"{mean:.3f}\n± {std:.3f}",
                    ha="center",
                    va="bottom",
                    fontsize=8.2,
                )

            _apply_thesis_layout(
                fig,
                ax,
                "Held-out error and calibration",
                config,
                top=0.74,
                bottom=0.28,
                left=0.12,
            )
            _save_thesis_figure(
                fig,
                f"heldout_error_summary__{safe_config}",
            )


# ============================================================
# Notebook summary table
# ============================================================

display_columns = [
    column for column in [
        "heldout_config_hash",
        "subgraph_method",
        "subgraph_fraction_requested",
        "subgraph_fraction_actual_mean",
        "subgraph_seed",
        "training_candidate_size",
        "scoring_mode",
        "label_mode",
        "endpoint_mining_hop",
        "selector_training_scope",
        "n_source_nodes_mean",
        "n_global_nodes_mean",
        "n_seeds",
        "auc_mean", "auc_std", "ap_mean", "ap_std",
        "balanced_accuracy_mean", "f1_mean", "mcc_mean",
        "rmse_mean", "pearson_mean", "spearman_mean",
        "brier_binary_mean", "ece_mean",
        "n_samples_mean", "positive_rate_mean",
        "lp_model_group_label",
    ]
    if column in roc_metrics_df.columns
]

display(
    roc_metrics_df[display_columns]
    .sort_values(
        [
            column
            for column in [
                "subgraph_method",
                "subgraph_fraction_requested",
                "subgraph_seed",
                "training_candidate_size",
                "scoring_mode",
                "label_mode",
            ]
            if column in roc_metrics_df.columns
        ]
    )
)

# ============================================================
# Continuous held-out evaluation for subset_accuracy_drop
# ============================================================
#
# subset_accuracy_drop uses normalized continuous targets in [0, 1]. ROC,
# average precision, precision, recall, F1, calibration curves, and confusion
# matrices require a binary reference label and are therefore deliberately not
# calculated here. Binarizing the targets merely to obtain those metrics would
# introduce an additional, arbitrary target threshold.
#
# Instead, this section evaluates:
#   - absolute and squared prediction error: MAE, MSE, RMSE;
#   - explained variation: R²;
#   - linear association: Pearson correlation;
#   - ranking agreement: Spearman correlation;
#   - prediction spread, which reveals collapse to nearly constant scores.
#
# All metrics are calculated on the untouched test split after restoration of
# the validation-selected checkpoint. Means and sample standard deviations are
# then calculated across victim-model seeds, exactly as for the endpoint runs.


def _heldout_numpy_1d(values, *, dtype=float):
    """Convert tensors, arrays, or lists to a one-dimensional NumPy array."""
    if values is None:
        return np.asarray([], dtype=dtype)
    if hasattr(values, "detach"):
        values = values.detach().cpu().numpy()
    return np.asarray(values, dtype=dtype).reshape(-1)


def _continuous_targets_from_test_split(run, history, outputs):
    """
    Return the original continuous targets belonging to the frozen test split.

    Newer helper versions expose them directly in final_split_outputs. The
    fallback reconstructs them from run['y_label'] and training_history['test_idx'],
    which are the exact tensors used during selector training.
    """
    for key in (
        "soft_targets",
        "continuous_targets",
        "targets",
    ):
        if key in outputs:
            candidate = _heldout_numpy_1d(outputs[key], dtype=float)
            if candidate.size:
                return candidate, f"final_split_outputs[{key!r}]"

    if "y_label" not in run:
        raise KeyError(
            f"Continuous held-out targets are unavailable for "
            f"{run.get('lp_model_label', 'unknown model')}: neither a target "
            "array in final_split_outputs nor run['y_label'] is present."
        )

    if "test_idx" not in history:
        raise KeyError(
            f"training_history['test_idx'] is missing for "
            f"{run.get('lp_model_label', 'unknown model')}."
        )

    all_targets = _heldout_numpy_1d(run["y_label"], dtype=float)
    test_idx = _heldout_numpy_1d(history["test_idx"], dtype=int)

    if test_idx.size == 0:
        raise RuntimeError(
            f"The held-out test split is empty for "
            f"{run.get('lp_model_label', 'unknown model')}."
        )
    if test_idx.min() < 0 or test_idx.max() >= all_targets.size:
        raise IndexError(
            f"Test indices are incompatible with run['y_label'] for "
            f"{run.get('lp_model_label', 'unknown model')}."
        )

    return all_targets[test_idx], "run['y_label'][training_history['test_idx']]"


def _safe_pearson_continuous(targets, predictions):
    targets = np.asarray(targets, dtype=float)
    predictions = np.asarray(predictions, dtype=float)
    if targets.size < 2:
        return np.nan
    if np.std(targets, ddof=0) <= 1e-12:
        return np.nan
    if np.std(predictions, ddof=0) <= 1e-12:
        return np.nan
    return float(np.corrcoef(targets, predictions)[0, 1])


def _safe_spearman_continuous(targets, predictions):
    targets = np.asarray(targets, dtype=float)
    predictions = np.asarray(predictions, dtype=float)
    if targets.size < 2:
        return np.nan

    target_ranks = (
        pd.Series(targets)
        .rank(method="average")
        .to_numpy(dtype=float)
    )
    prediction_ranks = (
        pd.Series(predictions)
        .rank(method="average")
        .to_numpy(dtype=float)
    )
    return _safe_pearson_continuous(target_ranks, prediction_ranks)


def _continuous_metrics(targets, predictions):
    targets = np.asarray(targets, dtype=float).reshape(-1)
    predictions = np.asarray(predictions, dtype=float).reshape(-1)

    if targets.size != predictions.size:
        raise ValueError(
            "Continuous target/prediction length mismatch: "
            f"{targets.size} targets versus {predictions.size} predictions."
        )
    if targets.size == 0:
        raise ValueError("The continuous held-out split is empty.")

    finite = np.isfinite(targets) & np.isfinite(predictions)
    if not finite.all():
        raise RuntimeError(
            f"Continuous held-out arrays contain "
            f"{int((~finite).sum())} non-finite pairs."
        )

    if targets.min() < -1e-7 or targets.max() > 1.0 + 1e-7:
        raise ValueError(
            "subset_accuracy_drop targets must lie in [0, 1], got "
            f"[{targets.min()}, {targets.max()}]."
        )

    targets = np.clip(targets, 0.0, 1.0)
    predictions = np.clip(predictions, 0.0, 1.0)

    residuals = predictions - targets
    absolute_errors = np.abs(residuals)
    squared_errors = residuals ** 2

    mae = float(np.mean(absolute_errors))
    mse = float(np.mean(squared_errors))
    rmse = float(np.sqrt(mse))

    target_centered = targets - targets.mean()
    ss_res = float(np.sum(squared_errors))
    ss_tot = float(np.sum(target_centered ** 2))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 1e-12 else np.nan

    # The training objective is BCE with continuous targets. Reporting this
    # value is optional for the thesis table, but retaining it makes the test
    # evaluation directly comparable with the optimized objective.
    epsilon = 1e-7
    clipped_predictions = np.clip(predictions, epsilon, 1.0 - epsilon)
    bce_soft = float(
        -np.mean(
            targets * np.log(clipped_predictions)
            + (1.0 - targets) * np.log(1.0 - clipped_predictions)
        )
    )

    return {
        "n_samples": int(targets.size),
        "bce_soft": bce_soft,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
        "pearson": _safe_pearson_continuous(targets, predictions),
        "spearman": _safe_spearman_continuous(targets, predictions),
        "mean_error": float(np.mean(residuals)),
        "target_mean": float(np.mean(targets)),
        "target_std": float(np.std(targets, ddof=0)),
        "target_min": float(np.min(targets)),
        "target_max": float(np.max(targets)),
        "target_unique_count": int(np.unique(targets).size),
        "probability_mean": float(np.mean(predictions)),
        "probability_std": float(np.std(predictions, ddof=0)),
        "probability_min": float(np.min(predictions)),
        "probability_max": float(np.max(predictions)),
    }


continuous_metric_rows = []
continuous_prediction_rows = []

for run in subset_accuracy_drop_runs:
    history = run["lp_model"].training_history
    outputs = history["final_split_outputs"]["test"]

    predictions = _heldout_numpy_1d(
        outputs["probabilities"],
        dtype=float,
    )
    targets, target_source = _continuous_targets_from_test_split(
        run,
        history,
        outputs,
    )

    if targets.size != predictions.size:
        raise RuntimeError(
            f"Held-out target/prediction mismatch for "
            f"{run['lp_model_label']}: {targets.size} versus "
            f"{predictions.size}. Target source: {target_source}."
        )

    base = _build_heldout_config_record(run)
    calculated = _continuous_metrics(targets, predictions)

    continuous_metric_rows.append({
        **base,
        **calculated,
        "target_source": target_source,
        "evaluation_kind": "continuous_target",
        "classification_metrics_calculated": 0,
    })

    continuous_prediction_rows.extend({
        **base,
        "test_position": int(position),
        "continuous_target": float(target),
        "predicted_score": float(prediction),
        "residual": float(prediction - target),
    } for position, (target, prediction) in enumerate(
        zip(targets, predictions)
    ))

continuous_metrics_raw_df = pd.DataFrame(continuous_metric_rows)
continuous_predictions_raw_df = pd.DataFrame(continuous_prediction_rows)

if subset_accuracy_drop_runs and continuous_metrics_raw_df.empty:
    raise RuntimeError(
        "subset_accuracy_drop runs were found, but no continuous held-out "
        "metrics were produced."
    )

if not continuous_metrics_raw_df.empty:
    # --------------------------------------------------------
    # Validate complete seed coverage independently of endpoint
    # --------------------------------------------------------
    continuous_expected_n_seeds = int(
        globals().get(
            "N_SEEDS",
            max(
                1,
                continuous_metrics_raw_df
                .groupby("heldout_config_id")["seed"]
                .nunique()
                .max(),
            ),
        )
    )

    continuous_seed_coverage = (
        continuous_metrics_raw_df
        .groupby("heldout_config_id")["seed"]
        .nunique()
    )
    incomplete_continuous = continuous_seed_coverage[
        continuous_seed_coverage != continuous_expected_n_seeds
    ]
    if not incomplete_continuous.empty:
        raise RuntimeError(
            "Continuous held-out evaluation is missing configured victim "
            "seeds for at least one complete configuration:\n"
            + incomplete_continuous.to_string()
        )

    # --------------------------------------------------------
    # Configuration manifest
    # --------------------------------------------------------
    continuous_stable_fields = [
        column
        for column in [
            "heldout_config_id",
            "heldout_config_hash",
            "lp_model_group_label",
            "candidate_config_id",
            "subgraph_method",
            "subgraph_seed",
            # construction_seed is victim-seed-specific provenance metadata
            # and must not be required to remain constant across seeds.
            "subgraph_fraction_requested",
            "training_candidate_size",
            "candidate_set_size",
            "scoring_mode",
            "endpoint_mining_hop",
            "label_mode",
            "extreme_fraction",
            "selector_training_scope",
            "prbcd_candidate_fraction",
            "target_source",
            "evaluation_kind",
            "classification_metrics_calculated",
        ]
        if column in continuous_metrics_raw_df.columns
    ]

    for column in continuous_stable_fields:
        if column in {"heldout_config_id", "heldout_config_hash"}:
            continue
        unique_counts = (
            continuous_metrics_raw_df
            .groupby("heldout_config_id")[column]
            .nunique(dropna=False)
        )
        inconsistent = unique_counts[unique_counts > 1]
        if not inconsistent.empty:
            raise RuntimeError(
                f"Continuous configuration field {column!r} varies across "
                "victim seeds inside one held-out configuration."
            )

    continuous_configuration_manifest_df = (
        continuous_metrics_raw_df[continuous_stable_fields]
        .drop_duplicates(subset=["heldout_config_id"])
        .copy()
    )

    continuous_varying_metadata_fields = [
        column
        for column in [
            "subgraph_fraction_actual",
            "n_source_nodes",
            "n_target_nodes",
            "n_selector_nodes",
            "n_global_nodes",
            "n_nodes_global",
            "n_test_nodes_local",
            "n_edges_local_undirected",
            "density_local",
            "global_edge_coverage",
            "actual_prbcd_fraction",
            "n_prbcd_requested",
            "n_prbcd_mined_unique",
            "n_prbcd_selected",
            "n_random_candidates",
        ]
        if column in continuous_metrics_raw_df.columns
    ]

    if continuous_varying_metadata_fields:
        continuous_metadata = continuous_metrics_raw_df[
            ["heldout_config_id"] + continuous_varying_metadata_fields
        ].copy()

        for column in continuous_varying_metadata_fields:
            continuous_metadata[column] = pd.to_numeric(
                continuous_metadata[column],
                errors="coerce",
            )

        metadata_aggregation = {}
        for column in continuous_varying_metadata_fields:
            metadata_aggregation[f"{column}_mean"] = (column, "mean")
            metadata_aggregation[f"{column}_std"] = (column, "std")
            metadata_aggregation[f"{column}_min"] = (column, "min")
            metadata_aggregation[f"{column}_max"] = (column, "max")

        continuous_varying_manifest_df = (
            continuous_metadata
            .groupby("heldout_config_id", as_index=False)
            .agg(**metadata_aggregation)
        )
        std_columns = [
            column
            for column in continuous_varying_manifest_df.columns
            if column.endswith("_std")
        ]
        continuous_varying_manifest_df[std_columns] = (
            continuous_varying_manifest_df[std_columns].fillna(0.0)
        )

        continuous_configuration_manifest_df = (
            continuous_configuration_manifest_df.merge(
                continuous_varying_manifest_df,
                on="heldout_config_id",
                how="left",
                validate="one_to_one",
            )
        )

    continuous_seed_count_df = (
        continuous_metrics_raw_df
        .groupby("heldout_config_id", as_index=False)
        .agg(n_seeds=("seed", "nunique"))
    )
    continuous_configuration_manifest_df = (
        continuous_configuration_manifest_df.merge(
            continuous_seed_count_df,
            on="heldout_config_id",
            how="left",
            validate="one_to_one",
        )
    )

    # --------------------------------------------------------
    # Mean ± sample SD across victim seeds
    # --------------------------------------------------------
    continuous_metric_columns = [
        "n_samples",
        "bce_soft",
        "mae",
        "mse",
        "rmse",
        "r2",
        "pearson",
        "spearman",
        "mean_error",
        "target_mean",
        "target_std",
        "target_min",
        "target_max",
        "target_unique_count",
        "probability_mean",
        "probability_std",
        "probability_min",
        "probability_max",
    ]

    continuous_metrics_df = aggregate_over_seeds(
        continuous_metrics_raw_df,
        ["heldout_config_id"],
        continuous_metric_columns,
    ).merge(
        continuous_configuration_manifest_df,
        on="heldout_config_id",
        how="left",
        validate="one_to_one",
    )

    # --------------------------------------------------------
    # Export raw and averaged numerical results
    # --------------------------------------------------------
    continuous_predictions_raw_df.to_csv(
        LP_HELDOUT_OUT_DIR / "heldout_continuous_predictions_raw.csv",
        index=False,
    )
    continuous_metrics_raw_df.to_csv(
        LP_HELDOUT_OUT_DIR / "heldout_continuous_metrics_raw.csv",
        index=False,
    )
    continuous_metrics_df.to_csv(
        LP_HELDOUT_OUT_DIR / "heldout_continuous_metrics_averaged.csv",
        index=False,
    )
    continuous_configuration_manifest_df.to_csv(
        LP_HELDOUT_OUT_DIR
        / "heldout_continuous_configuration_manifest.csv",
        index=False,
    )

    with open(
        LP_HELDOUT_OUT_DIR
        / "heldout_continuous_configuration_manifest.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            continuous_configuration_manifest_df.to_dict(
                orient="records"
            ),
            handle,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

    # --------------------------------------------------------
    # Copy-ready thesis table
    # --------------------------------------------------------
    def _mean_sd_cell(row, metric, decimals=3):
        mean_value = row.get(f"{metric}_mean", np.nan)
        std_value = row.get(f"{metric}_std", np.nan)
        if not np.isfinite(mean_value):
            return "n/a"
        if not np.isfinite(std_value):
            std_value = 0.0
        return f"{mean_value:.{decimals}f} ± {std_value:.{decimals}f}"

    continuous_thesis_rows = []
    for _, row in continuous_metrics_df.iterrows():
        continuous_thesis_rows.append({
            "Configuration": row.get("heldout_config_hash", ""),
            "Subgraph method": _pretty_method(
                row.get("subgraph_method", "unknown")
            ),
            "Source fraction": _format_fraction(
                row.get("subgraph_fraction_requested", "")
            ),
            "Training candidates": _format_number(
                row.get("training_candidate_size", "")
            ),
            "Test pairs": _mean_sd_cell(row, "n_samples", decimals=1),
            "MAE": _mean_sd_cell(row, "mae"),
            "RMSE": _mean_sd_cell(row, "rmse"),
            "Pearson": _mean_sd_cell(row, "pearson"),
            "Spearman": _mean_sd_cell(row, "spearman"),
            "R²": _mean_sd_cell(row, "r2"),
            "Prediction SD": _mean_sd_cell(row, "probability_std"),
            "Victim seeds": int(row.get("n_seeds", 0)),
        })

    continuous_thesis_table_df = pd.DataFrame(
        continuous_thesis_rows
    )
    continuous_thesis_table_df.to_csv(
        LP_HELDOUT_OUT_DIR / "heldout_continuous_thesis_table.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Thesis-ready continuous-target plots (no ROC/PR/confusion)
    # --------------------------------------------------------
    with plt.rc_context(THESIS_RC):
        for config_id in continuous_metrics_df["heldout_config_id"]:
            metric = continuous_metrics_df[
                continuous_metrics_df["heldout_config_id"] == config_id
            ].iloc[0]
            config = metric.to_dict()
            safe_config = _configuration_slug(config)

            prediction_group = continuous_predictions_raw_df[
                continuous_predictions_raw_df["heldout_config_id"]
                == config_id
            ]

            # 1. Continuous target against predicted selector score.
            fig, ax = plt.subplots(figsize=(6.8, 6.0))
            ax.scatter(
                prediction_group["continuous_target"],
                prediction_group["predicted_score"],
                s=11,
                alpha=0.28,
                edgecolors="none",
            )
            ax.plot(
                [0, 1],
                [0, 1],
                linestyle="--",
                linewidth=1.0,
                label="Perfect agreement",
            )
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.set_aspect("equal", adjustable="box")
            ax.set_xlabel("Continuous held-out target")
            ax.set_ylabel("Predicted selector score")
            ax.grid(True, alpha=0.22, linewidth=0.7)
            ax.legend(loc="best", frameon=False)
            _apply_thesis_layout(
                fig,
                ax,
                "Held-out continuous target fit",
                config,
                top=0.77,
                bottom=0.24,
            )
            _save_thesis_figure(
                fig,
                f"heldout_continuous_fit__{safe_config}",
            )

            # 2. Lower-is-better continuous prediction errors.
            fig, ax = plt.subplots(figsize=(7.0, 5.2))
            error_metrics = [
                ("mae", "MAE"),
                ("rmse", "RMSE"),
                ("bce_soft", "Soft-target BCE"),
            ]
            names = [display_name for _, display_name in error_metrics]
            means = [metric[f"{key}_mean"] for key, _ in error_metrics]
            stds = [metric[f"{key}_std"] for key, _ in error_metrics]
            bars = ax.bar(
                names,
                means,
                yerr=stds,
                capsize=4,
                width=0.65,
            )
            ax.set_ylabel("Held-out error")
            ax.tick_params(axis="x", rotation=15)
            ax.grid(True, axis="y", alpha=0.22, linewidth=0.7)
            upper_limit = ax.get_ylim()[1]
            offset = max(upper_limit * 0.025, 0.002)
            for bar, mean_value, std_value in zip(bars, means, stds):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    mean_value + std_value + offset,
                    f"{mean_value:.3f}\n± {std_value:.3f}",
                    ha="center",
                    va="bottom",
                    fontsize=8.2,
                )
            _apply_thesis_layout(
                fig,
                ax,
                "Held-out continuous prediction error",
                config,
                top=0.74,
                bottom=0.28,
                left=0.12,
            )
            _save_thesis_figure(
                fig,
                f"heldout_continuous_error__{safe_config}",
            )

            # 3. Association and ranking metrics.
            fig, ax = plt.subplots(figsize=(7.0, 5.2))
            association_metrics = [
                ("pearson", "Pearson"),
                ("spearman", "Spearman"),
                ("r2", "R²"),
            ]
            names = [display_name for _, display_name in association_metrics]
            means = [
                metric[f"{key}_mean"]
                for key, _ in association_metrics
            ]
            stds = [
                metric[f"{key}_std"]
                for key, _ in association_metrics
            ]
            bars = ax.bar(
                names,
                means,
                yerr=stds,
                capsize=4,
                width=0.65,
            )
            ax.axhline(0.0, linewidth=0.9)
            ax.set_ylim(-1.05, 1.05)
            ax.set_ylabel("Held-out association")
            ax.grid(True, axis="y", alpha=0.22, linewidth=0.7)
            for bar, mean_value, std_value in zip(bars, means, stds):
                if not np.isfinite(mean_value):
                    label = "n/a"
                    y_position = 0.04
                    vertical_alignment = "bottom"
                else:
                    label = f"{mean_value:.3f}\n± {std_value:.3f}"
                    y_position = mean_value + (
                        0.05 if mean_value >= 0 else -0.08
                    )
                    vertical_alignment = (
                        "bottom" if mean_value >= 0 else "top"
                    )
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    y_position,
                    label,
                    ha="center",
                    va=vertical_alignment,
                    fontsize=8.2,
                )
            _apply_thesis_layout(
                fig,
                ax,
                "Held-out continuous target association",
                config,
                top=0.74,
                bottom=0.25,
                left=0.12,
            )
            _save_thesis_figure(
                fig,
                f"heldout_continuous_association__{safe_config}",
            )

    print(
        "[CONTINUOUS HELD-OUT] ROC/PR, threshold metrics, calibration, "
        "and confusion matrices were intentionally not calculated for "
        "subset_accuracy_drop targets."
    )
    display(continuous_thesis_table_df)
else:
    continuous_configuration_manifest_df = pd.DataFrame()
    continuous_metrics_df = pd.DataFrame()
    continuous_thesis_table_df = pd.DataFrame()
    print(
        "[CONTINUOUS HELD-OUT] No subset_accuracy_drop runs were found."
    )



In [ ]:
from helpers.selector_pipeline_helpers import accuracy
from pathlib import Path
from IPython.display import display
import hashlib
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch


# ============================================================
# Fixed shared-oracle setup
# ============================================================

BLOCK_STEP_SELECTOR = 10
SCORE_BATCH_SIZE = 100_000
ORACLE_CANDIDATE_SIZE = 10_000
ORACLE_CANDIDATE_SEED = 123
ORACLE_RANDOM_RANKING_SEED = 42

OUT_DIR = (
    Path("extendedPlotting")
    / "selector_evaluation_fixed_shared_oracle"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

PLOT_DIR = Path(globals().get("RUN_PLOTS_DIR", OUT_DIR))
PLOT_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_FORMATS = tuple(
    globals().get("SELECTOR_EVAL_EXPORT_FORMATS", ("png", "pdf", "svg"))
)
PNG_DPI = int(globals().get("SELECTOR_EVAL_PNG_DPI", 300))
SHOW_PLOTS = bool(globals().get("SELECTOR_EVAL_SHOW_PLOTS", True))

if "RQ3_TRAINING_STAT_RUNS" not in globals():
    raise RuntimeError(
        "RQ3_TRAINING_STAT_RUNS is unavailable. Run the selector-training "
        "cells first."
    )

RUNS = list(RQ3_TRAINING_STAT_RUNS)
if not RUNS:
    raise RuntimeError("No trained selectors are available.")


# ============================================================
# Metadata
# ============================================================

CONFIG_FIELDS = [
    "lp_model_group_label",
    "candidate_config_id",
    "subgraph_method",
    "subgraph_seed",
    "subgraph_fraction_requested",
    "training_candidate_size",
    "prbcd_candidate_fraction",
    "scoring_mode",
    "endpoint_mining_hop",
    "label_mode",
    "extreme_fraction",
    "selector_training_scope",
    "victim_query_scope",
]

COMPARISON_FIELDS = [
    field
    for field in CONFIG_FIELDS
    if field not in {
        "lp_model_group_label",
        "candidate_config_id",
        "training_candidate_size",
    }
]


def _value(run, key, default=""):
    value = run.get(key, default)
    if value in (None, ""):
        for nested_key in (
            "subgraph_stats",
            "sampler_metadata",
            "construction_metadata",
        ):
            nested = run.get(nested_key, {})
            if isinstance(nested, dict) and key in nested:
                value = nested.get(key, default)
                break
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return ""
    return value if isinstance(value, (str, int, float, bool)) else str(value)


def _hash_record(record, fields, length=12):
    payload = {field: record.get(field, "") for field in fields}
    text = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:length]


def _metadata(run):
    record = {
        "seed": int(run["seed"]),
        "lp_model_label": str(run.get("lp_model_label", "")),
    }
    for field in CONFIG_FIELDS:
        record[field] = _value(run, field)

    if record.get("training_candidate_size", "") in (None, ""):
        record["training_candidate_size"] = _value(
            run,
            "candidate_set_size",
        )

    if not str(record.get("subgraph_method", "")).strip():
        try:
            if np.isclose(
                float(record.get("subgraph_fraction_requested", "")),
                1.0,
            ):
                record["subgraph_method"] = "whole_graph"
        except (TypeError, ValueError):
            record["subgraph_method"] = "unknown"

    record["selector_eval_config_id"] = _hash_record(
        record,
        CONFIG_FIELDS,
    )
    record["selector_comparison_id"] = _hash_record(
        record,
        COMPARISON_FIELDS,
    )
    return record


# ============================================================
# Shared oracle candidate pool
# ============================================================


def sample_undirected_pairs(num_nodes, num_candidates, seed):
    total = num_nodes * (num_nodes - 1) // 2
    count = min(int(num_candidates), total)
    linear_ids = torch.tensor(
        random.Random(int(seed)).sample(range(total), count),
        dtype=torch.long,
    )

    row_lengths = torch.arange(num_nodes - 1, 0, -1, dtype=torch.long)
    row_starts = torch.zeros(num_nodes, dtype=torch.long)
    row_starts[1:] = torch.cumsum(row_lengths, dim=0)
    src = torch.searchsorted(row_starts, linear_ids, right=True) - 1
    dst = src + 1 + linear_ids - row_starts[src]
    return linear_ids, src, dst


N_NODES = int(RUNS[0]["context"]["n_nodes"])
if any(int(run["context"]["n_nodes"]) != N_NODES for run in RUNS):
    raise RuntimeError("The selector runs do not use the same graph size.")

ORACLE_LINEAR_IDS, ORACLE_SRC, ORACLE_DST = sample_undirected_pairs(
    N_NODES,
    ORACLE_CANDIDATE_SIZE,
    ORACLE_CANDIDATE_SEED,
)
ORACLE_SIZE = int(ORACLE_LINEAR_IDS.numel())
ORACLE_POOL_HASH = hashlib.sha1(
    ORACLE_LINEAR_IDS.numpy().tobytes()
).hexdigest()[:12]
ORACLE_RANDOM_ORDER = np.random.default_rng(
    ORACLE_RANDOM_RANKING_SEED
).permutation(ORACLE_SIZE)

reference_adj = RUNS[0]["adj_orig"]
REFERENCE_EXISTS = (
    reference_adj[
        ORACLE_SRC.to(reference_adj.device),
        ORACLE_DST.to(reference_adj.device),
    ]
    != 0
).detach().cpu().bool()

print(
    f"Shared oracle pool: n={ORACLE_SIZE:,}, "
    f"candidate_seed={ORACLE_CANDIDATE_SEED}, "
    f"random_seed={ORACLE_RANDOM_RANKING_SEED}, "
    f"hash={ORACLE_POOL_HASH}"
)


# ============================================================
# Evaluation
# ============================================================


def evaluate_run(run, run_index):
    seed = int(run["seed"])
    context = run["context"]
    selector = run["lp_model"]
    adj_base = run["adj_orig"]
    metadata = _metadata(run)

    set_global_seed(seed)

    exists = (
        adj_base[
            ORACLE_SRC.to(adj_base.device),
            ORACLE_DST.to(adj_base.device),
        ]
        != 0
    ).detach().cpu().bool()
    if not torch.equal(exists, REFERENCE_EXISTS):
        raise RuntimeError(
            "The supposedly shared oracle pool has different edge states "
            "across runs."
        )

    device = next(selector.parameters()).device
    edge_lab = torch.stack([ORACLE_SRC, ORACLE_DST]).to(device)

    selector.eval()
    with torch.no_grad():
        h = selector.encoder(
            context["attr"].to(device),
            context["edge_index"].to(device),
        )
        scores = torch.cat([
            torch.sigmoid(
                selector.edge_head(
                    h,
                    edge_lab[:, start:start + SCORE_BATCH_SIZE],
                ).view(-1)
            ).cpu()
            for start in range(0, ORACLE_SIZE, SCORE_BATCH_SIZE)
        ])

    if scores.numel() != ORACLE_SIZE or not torch.isfinite(scores).all():
        raise RuntimeError("Invalid selector scores in oracle evaluation.")

    orders = {
        "LP-GNN ranking": torch.argsort(scores, descending=True).numpy(),
        "Random ranking": ORACLE_RANDOM_ORDER,
    }

    k_values = list(range(0, ORACLE_SIZE, BLOCK_STEP_SELECTOR))
    if k_values[-1] != ORACLE_SIZE:
        k_values.append(ORACLE_SIZE)

    rows = []
    for attack_name, order in orders.items():
        adj = adj_base.clone()
        previous_k = 0

        for k in k_values:
            for position in order[previous_k:k].tolist():
                u = int(ORACLE_SRC[position])
                v = int(ORACLE_DST[position])
                value = 1.0 - float(exists[position])
                adj[u, v] = value
                adj[v, u] = value

            test_acc = float(
                accuracy(
                    context["model"],
                    context["attr"],
                    context["labels"],
                    context["idx_test"],
                    edge_index=adj,
                )
            )
            clean_acc = float(run["clean_accuracy"])

            rows.append({
                **metadata,
                "run_index": int(run_index),
                "oracle_candidate_size": ORACLE_SIZE,
                "oracle_candidate_seed": ORACLE_CANDIDATE_SEED,
                "oracle_random_ranking_seed": ORACLE_RANDOM_RANKING_SEED,
                "oracle_pool_hash": ORACLE_POOL_HASH,
                "k": int(k),
                "k_fraction": float(k) / float(ORACLE_SIZE),
                "addition_step": int(np.ceil(k / BLOCK_STEP_SELECTOR)),
                "attack": attack_name,
                "test_accuracy": test_acc,
                "clean_accuracy": clean_acc,
                "accuracy_drop": clean_acc - test_acc,
            })
            previous_k = k

    return pd.DataFrame(rows)


selector_eval_df = pd.concat(
    [evaluate_run(run, i) for i, run in enumerate(RUNS)],
    ignore_index=True,
)

if selector_eval_df["oracle_pool_hash"].nunique() != 1:
    raise RuntimeError("More than one oracle pool was used.")

expected_n_seeds = int(
    globals().get(
        "N_SEEDS",
        selector_eval_df.groupby("selector_eval_config_id")["seed"]
        .nunique()
        .max(),
    )
)
coverage = selector_eval_df.groupby("selector_eval_config_id")["seed"].nunique()
if (coverage != expected_n_seeds).any():
    raise RuntimeError(
        "Missing victim seeds:\n" + coverage.to_string()
    )


# ============================================================
# Manifest and trajectory CSVs
# ============================================================

manifest_fields = [
    "selector_eval_config_id",
    "selector_comparison_id",
    "lp_model_group_label",
    "lp_model_label",
    "candidate_config_id",
    "subgraph_method",
    "subgraph_seed",
    "subgraph_fraction_requested",
    "training_candidate_size",
    "prbcd_candidate_fraction",
    "scoring_mode",
    "endpoint_mining_hop",
    "label_mode",
    "extreme_fraction",
    "selector_training_scope",
    "victim_query_scope",
]
manifest_fields = [c for c in manifest_fields if c in selector_eval_df.columns]

manifest_agg = {
    c: (c, "first")
    for c in manifest_fields
    if c != "selector_eval_config_id"
}
manifest_agg["n_seeds"] = ("seed", "nunique")

manifest_df = (
    selector_eval_df
    .groupby("selector_eval_config_id", as_index=False)
    .agg(**manifest_agg)
)
manifest_df["oracle_candidate_size"] = ORACLE_SIZE
manifest_df["oracle_candidate_seed"] = ORACLE_CANDIDATE_SEED
manifest_df["oracle_random_ranking_seed"] = ORACLE_RANDOM_RANKING_SEED
manifest_df["oracle_pool_hash"] = ORACLE_POOL_HASH
manifest_df.to_csv(
    OUT_DIR / "selector_evaluation_configuration_manifest.csv",
    index=False,
)

agg_spec = {
    "test_accuracy_mean": ("test_accuracy", "mean"),
    "test_accuracy_std": ("test_accuracy", "std"),
    "clean_accuracy_mean": ("clean_accuracy", "mean"),
    "clean_accuracy_std": ("clean_accuracy", "std"),
    "accuracy_drop_mean": ("accuracy_drop", "mean"),
    "accuracy_drop_std": ("accuracy_drop", "std"),
    "k_fraction_mean": ("k_fraction", "mean"),
    "addition_step_mean": ("addition_step", "mean"),
    "n_seeds_at_k": ("seed", "nunique"),
}

selector_eval_averaged_df = (
    selector_eval_df
    .groupby(["selector_eval_config_id", "k", "attack"], as_index=False)
    .agg(**agg_spec)
    .merge(manifest_df, on="selector_eval_config_id", how="left")
)
selector_eval_averaged_df[
    [c for c in selector_eval_averaged_df if c.endswith("_std")]
] = selector_eval_averaged_df[
    [c for c in selector_eval_averaged_df if c.endswith("_std")]
].fillna(0.0)

selector_eval_df.to_csv(
    OUT_DIR / "selector_eval_raw_all_seeds.csv",
    index=False,
)
selector_eval_averaged_df.to_csv(
    OUT_DIR / "selector_eval_averaged.csv",
    index=False,
)


# ============================================================
# Paired selector-versus-random CSVs
# ============================================================

keys = [
    "selector_eval_config_id",
    "seed",
    "k",
    "k_fraction",
    "addition_step",
]

lp = selector_eval_df[selector_eval_df["attack"] == "LP-GNN ranking"][
    keys + ["test_accuracy", "accuracy_drop"]
].rename(columns={
    "test_accuracy": "lp_gnn_accuracy",
    "accuracy_drop": "lp_gnn_accuracy_drop",
})

rnd = selector_eval_df[selector_eval_df["attack"] == "Random ranking"][
    keys + ["test_accuracy", "accuracy_drop"]
].rename(columns={
    "test_accuracy": "random_accuracy",
    "accuracy_drop": "random_accuracy_drop",
})

paired_df = lp.merge(rnd, on=keys, validate="one_to_one")
paired_df["lp_gnn_advantage"] = (
    paired_df["random_accuracy"] - paired_df["lp_gnn_accuracy"]
)
paired_df["accuracy_drop_advantage"] = (
    paired_df["lp_gnn_accuracy_drop"] - paired_df["random_accuracy_drop"]
)
paired_df = paired_df.merge(manifest_df, on="selector_eval_config_id", how="left")

paired_avg_df = (
    paired_df
    .groupby(["selector_eval_config_id", "k"], as_index=False)
    .agg(
        k_fraction_mean=("k_fraction", "mean"),
        addition_step_mean=("addition_step", "mean"),
        lp_gnn_accuracy_mean=("lp_gnn_accuracy", "mean"),
        lp_gnn_accuracy_std=("lp_gnn_accuracy", "std"),
        random_accuracy_mean=("random_accuracy", "mean"),
        random_accuracy_std=("random_accuracy", "std"),
        lp_gnn_accuracy_drop_mean=("lp_gnn_accuracy_drop", "mean"),
        lp_gnn_accuracy_drop_std=("lp_gnn_accuracy_drop", "std"),
        lp_gnn_advantage_mean=("lp_gnn_advantage", "mean"),
        lp_gnn_advantage_std=("lp_gnn_advantage", "std"),
        n_seeds_at_k=("seed", "nunique"),
    )
    .merge(manifest_df, on="selector_eval_config_id", how="left")
)
paired_avg_df[
    [c for c in paired_avg_df if c.endswith("_std")]
] = paired_avg_df[
    [c for c in paired_avg_df if c.endswith("_std")]
].fillna(0.0)

paired_df.to_csv(
    OUT_DIR / "selector_eval_paired_all_seeds.csv",
    index=False,
)
paired_avg_df.to_csv(
    OUT_DIR / "selector_eval_paired_averaged.csv",
    index=False,
)


# ============================================================
# Standalone-attack summary
# ============================================================


def integrate(y, x):
    if hasattr(np, "trapezoid"):
        return float(np.trapezoid(y, x))
    return float(np.trapz(y, x))


summary_rows = []
for (config_id, seed), group in paired_df.groupby(
    ["selector_eval_config_id", "seed"]
):
    group = group.sort_values("k").reset_index(drop=True)
    optimum = group.loc[group["lp_gnn_accuracy"].idxmin()]

    row = {
        "selector_eval_config_id": config_id,
        "seed": int(seed),
        "minimum_lp_gnn_accuracy": float(optimum["lp_gnn_accuracy"]),
        "maximum_lp_gnn_accuracy_drop": float(
            optimum["lp_gnn_accuracy_drop"]
        ),
        "k_star": int(optimum["k"]),
        "k_star_fraction": float(optimum["k_fraction"]),
        "addition_step_star": int(optimum["addition_step"]),
        "lp_gnn_advantage_at_k_star": float(
            optimum["lp_gnn_advantage"]
        ),
        "integrated_lp_gnn_advantage": integrate(
            group["lp_gnn_advantage"].to_numpy(float),
            group["k_fraction"].to_numpy(float),
        ),
    }

    for budget in (100, 500, 1_000, 2_000):
        budget_row = group.iloc[
            (group["k"] - min(budget, ORACLE_SIZE)).abs().argsort()[:1]
        ].iloc[0]
        row[f"accuracy_drop_at_{budget}"] = float(
            budget_row["lp_gnn_accuracy_drop"]
        )
        row[f"advantage_at_{budget}"] = float(
            budget_row["lp_gnn_advantage"]
        )

    summary_rows.append(row)

summary_seed_df = pd.DataFrame(summary_rows).merge(
    manifest_df,
    on="selector_eval_config_id",
    how="left",
)

numeric_summary_cols = [
    c
    for c in summary_seed_df.columns
    if c not in set(manifest_df.columns) | {"seed"}
    and pd.api.types.is_numeric_dtype(summary_seed_df[c])
]
summary_agg = {}
for c in numeric_summary_cols:
    summary_agg[f"{c}_mean"] = (c, "mean")
    summary_agg[f"{c}_std"] = (c, "std")
summary_agg["n_seeds_summary"] = ("seed", "nunique")

summary_avg_df = (
    summary_seed_df
    .groupby("selector_eval_config_id", as_index=False)
    .agg(**summary_agg)
    .merge(manifest_df, on="selector_eval_config_id", how="left")
)
summary_avg_df[
    [c for c in summary_avg_df if c.endswith("_std")]
] = summary_avg_df[
    [c for c in summary_avg_df if c.endswith("_std")]
].fillna(0.0)

summary_seed_df.to_csv(
    OUT_DIR / "selector_eval_standalone_summary_all_seeds.csv",
    index=False,
)
summary_avg_df.to_csv(
    OUT_DIR / "selector_eval_standalone_summary_averaged.csv",
    index=False,
)


# ============================================================
# Training-size comparison plots
# ============================================================


def plot_band(ax, frame, label):
    frame = frame.sort_values("k")
    x = frame["k"].to_numpy(float)
    mean = frame["accuracy_drop_mean"].to_numpy(float)
    std = frame["accuracy_drop_std"].to_numpy(float)
    line = ax.plot(x, mean, label=label)[0]
    ax.fill_between(
        x,
        mean - std,
        mean + std,
        color=line.get_color(),
        alpha=0.15,
        linewidth=0,
    )


def save_plot(fig, stem):
    for extension in EXPORT_FORMATS:
        extension = str(extension).lower().lstrip(".")
        path = PLOT_DIR / f"{stem}.{extension}"
        kwargs = {"bbox_inches": "tight", "facecolor": "white"}
        if extension == "png":
            kwargs["dpi"] = PNG_DPI
        fig.savefig(path, **kwargs)
        print("Saved:", path)
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)


for comparison_id, configs in manifest_df.groupby("selector_comparison_id"):
    sizes = sorted(
        pd.to_numeric(configs["training_candidate_size"], errors="coerce")
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    if len(sizes) < 2:
        continue

    config_ids = configs["selector_eval_config_id"].tolist()
    plot_df = selector_eval_averaged_df[
        selector_eval_averaged_df["selector_eval_config_id"].isin(config_ids)
    ]

    fig, ax = plt.subplots(figsize=(9.2, 5.4))

    # One shared random baseline.
    reference_id = config_ids[0]
    random_curve = plot_df[
        (plot_df["selector_eval_config_id"] == reference_id)
        & (plot_df["attack"] == "Random ranking")
    ]
    plot_band(ax, random_curve, "Shared random ranking")

    for size in sizes:
        matching = configs[
            pd.to_numeric(
                configs["training_candidate_size"],
                errors="coerce",
            ) == size
        ]
        if len(matching) != 1:
            raise RuntimeError(
                "Ambiguous comparison group for "
                f"training size {size}: {len(matching)} matches."
            )

        config_id = matching.iloc[0]["selector_eval_config_id"]
        curve = plot_df[
            (plot_df["selector_eval_config_id"] == config_id)
            & (plot_df["attack"] == "LP-GNN ranking")
        ]
        plot_band(ax, curve, f"LP-GNN, $N_{{train}}={size:,}$")

    example = configs.iloc[0]
    scoring = str(example.get("scoring_mode", "")).replace("_", " ")
    labels = str(example.get("label_mode", "")).replace("_", " ")

    ax.axhline(0.0, linestyle="--", linewidth=0.9, color="0.35")
    ax.set_xlabel("Candidate edges applied jointly, $k$")
    ax.set_ylabel("Accuracy drop from the clean graph")
    ax.set_title(
        "Standalone selector comparison on a shared oracle pool\n"
        f"scoring={scoring or 'not specified'}, "
        f"labels={labels or 'not specified'}, "
        f"oracle size={ORACLE_SIZE:,}"
    )
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()

    save_plot(
        fig,
        f"selector_training_size_comparison__{comparison_id}"
        f"__oracle-{ORACLE_POOL_HASH}",
    )


# ============================================================
# Notebook summaries
# ============================================================

print("\nConfiguration manifest")
display(manifest_df)

print("\nStandalone-attack summary")
summary_display_cols = [
    c
    for c in [
        "training_candidate_size",
        "scoring_mode",
        "label_mode",
        "minimum_lp_gnn_accuracy_mean",
        "maximum_lp_gnn_accuracy_drop_mean",
        "k_star_mean",
        "addition_step_star_mean",
        "lp_gnn_advantage_at_k_star_mean",
        "integrated_lp_gnn_advantage_mean",
        "accuracy_drop_at_1000_mean",
        "advantage_at_1000_mean",
        "n_seeds_summary",
    ]
    if c in summary_avg_df.columns
]
display(
    summary_avg_df[summary_display_cols].sort_values(
        [
            c
            for c in [
                "scoring_mode",
                "label_mode",
                "training_candidate_size",
            ]
            if c in summary_display_cols
        ]
    )
)

print("\nSaved output directory:", OUT_DIR.resolve())


In [ ]:
# ============================================================
# Thesis plots: selector training dynamics and model selection
# Place this cell after the training-dynamics export cell.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


# ============================================================
# Configuration
# ============================================================

TRAINING_METRICS_DIR = Path(
    globals().get(
        "LP_TRAINING_DYNAMICS_OUT_DIR",
        Path("extendedPlotting")
        / "lp_training_dynamics_multiseed",
    )
)

TRAINING_PLOT_DIR = (
    TRAINING_METRICS_DIR
    / "thesis_training_diagnostics"
)
TRAINING_PLOT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SHOW_PLOTS = True
SAVE_PLOTS = True
EXPORT_FORMATS = ("png", "pdf", "svg")
PNG_DPI = 300

# The main-text plots are the training/validation trajectories.
# Turn this on when separate generalization-gap figures are wanted.
PLOT_GENERALIZATION_GAP = False

# Keep the y-axis identical across candidate sizes belonging to the
# same target construction. This makes the figures comparable.
COMMON_Y_AXIS_WITHIN_TARGET = True

# Require every plotted epoch to contain all configured seeds.
STRICT_COMPLETE_SEED_COVERAGE = True

TARGET_DISPLAY_NAMES = {
    "endpoint": "Endpoint harmfulness",
    "subset_accuracy_drop": "Subset accuracy drop",
}


# ============================================================
# Input files
# ============================================================

EPOCH_AVERAGED_PATH = (
    TRAINING_METRICS_DIR
    / "lp_epoch_metrics_averaged.csv"
)
GAP_AVERAGED_PATH = (
    TRAINING_METRICS_DIR
    / "lp_generalization_gap_averaged.csv"
)
STOPPING_AVERAGED_PATH = (
    TRAINING_METRICS_DIR
    / "lp_stopping_summary_averaged.csv"
)
STOPPING_RAW_PATH = (
    TRAINING_METRICS_DIR
    / "lp_stopping_summary_raw.csv"
)

required_paths = [
    EPOCH_AVERAGED_PATH,
    GAP_AVERAGED_PATH,
    STOPPING_AVERAGED_PATH,
    STOPPING_RAW_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "The following training-dynamics files are missing:\n"
        + "\n".join(f"  {path}" for path in missing_paths)
        + "\nRun the training-dynamics export cell first."
    )

epoch_df = pd.read_csv(EPOCH_AVERAGED_PATH)
gap_df = pd.read_csv(GAP_AVERAGED_PATH)
stopping_df = pd.read_csv(STOPPING_AVERAGED_PATH)
stopping_raw_df = pd.read_csv(STOPPING_RAW_PATH)


# ============================================================
# Select whole-graph configurations
# ============================================================

def select_whole_graph_rows(dataframe):
    """
    Select the initial selector-training condition and exclude the later
    subgraph experiments.
    """
    if "subgraph_method" in dataframe.columns:
        method_mask = (
            dataframe["subgraph_method"]
            .astype(str)
            .str.strip()
            .str.lower()
            .eq("whole_graph")
        )
    else:
        method_mask = pd.Series(
            False,
            index=dataframe.index,
        )

    if "subgraph_fraction_requested" in dataframe.columns:
        fraction = pd.to_numeric(
            dataframe["subgraph_fraction_requested"],
            errors="coerce",
        )
        fraction_mask = np.isclose(
            fraction,
            1.0,
            equal_nan=False,
        )
    else:
        fraction_mask = pd.Series(
            False,
            index=dataframe.index,
        )

    selected = dataframe.loc[
        method_mask | fraction_mask
    ].copy()

    if selected.empty:
        raise RuntimeError(
            "No whole-graph selector rows were found."
        )

    return selected


epoch_df = select_whole_graph_rows(epoch_df)
gap_df = select_whole_graph_rows(gap_df)
stopping_df = select_whole_graph_rows(stopping_df)
stopping_raw_df = select_whole_graph_rows(
    stopping_raw_df
)


# ============================================================
# Validation and normalization
# ============================================================

numeric_columns = {
    "epoch",
    "training_candidate_size",
    "loss_mean",
    "loss_std",
    "n_seeds",
}

missing_columns = numeric_columns.difference(
    epoch_df.columns
)

if missing_columns:
    raise RuntimeError(
        "lp_epoch_metrics_averaged.csv is missing columns: "
        + ", ".join(sorted(missing_columns))
    )

for column in numeric_columns:
    epoch_df[column] = pd.to_numeric(
        epoch_df[column],
        errors="coerce",
    )

for column in [
    "training_candidate_size",
    "best_epoch_mean",
    "best_epoch_std",
    "stopped_at_mean",
    "stopped_at_std",
    "best_train_loss_mean",
    "best_train_loss_std",
    "best_val_loss_mean",
    "best_val_loss_std",
    "generalization_gap_at_best_mean",
    "generalization_gap_at_best_std",
    "training_runtime_seconds_mean",
    "training_runtime_seconds_std",
    "n_seeds",
]:
    if column in stopping_df.columns:
        stopping_df[column] = pd.to_numeric(
            stopping_df[column],
            errors="coerce",
        )

observed_seeds = sorted(
    pd.to_numeric(
        stopping_raw_df["seed"],
        errors="coerce",
    )
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

expected_seed_count = len(observed_seeds)

if expected_seed_count < 2:
    raise RuntimeError(
        "At least two seeds are required to calculate "
        "a meaningful standard deviation."
    )

if STRICT_COMPLETE_SEED_COVERAGE:
    incomplete_epoch_rows = epoch_df.loc[
        epoch_df["n_seeds"] != expected_seed_count
    ]

    if not incomplete_epoch_rows.empty:
        raise RuntimeError(
            "Some epoch-level averages do not contain all "
            f"{expected_seed_count} seeds. Inspect these rows:\n"
            + incomplete_epoch_rows[
                [
                    "scoring_mode",
                    "training_candidate_size",
                    "split",
                    "epoch",
                    "n_seeds",
                ]
            ].head(20).to_string(index=False)
        )

print(
    "Training metrics folder:",
    TRAINING_METRICS_DIR.resolve(),
)
print(
    "Victim-model seeds:",
    observed_seeds,
)
print(
    "Plot output folder:",
    TRAINING_PLOT_DIR.resolve(),
)


# ============================================================
# Plot helpers
# ============================================================

def safe_token(value):
    return (
        str(value)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("/", "-")
    )


def save_figure(figure, stem):
    if SAVE_PLOTS:
        for extension in EXPORT_FORMATS:
            extension = str(extension).lower().lstrip(".")
            output_path = (
                TRAINING_PLOT_DIR
                / f"{stem}.{extension}"
            )

            save_kwargs = {
                "bbox_inches": "tight",
                "facecolor": "white",
            }

            if extension == "png":
                save_kwargs["dpi"] = PNG_DPI

            figure.savefig(
                output_path,
                **save_kwargs,
            )
            print("Saved:", output_path)

    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(figure)


def clean_axis(axis):
    axis.grid(
        visible=True,
        axis="y",
        alpha=0.20,
    )
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.tick_params(direction="out")


def get_target_y_limits(target_frame):
    lower = (
        target_frame["loss_mean"]
        - target_frame["loss_std"]
    ).min()
    upper = (
        target_frame["loss_mean"]
        + target_frame["loss_std"]
    ).max()

    if not np.isfinite(lower) or not np.isfinite(upper):
        return None

    span = max(
        float(upper - lower),
        1e-6,
    )
    padding = 0.06 * span

    return (
        float(lower - padding),
        float(upper + padding),
    )


def lookup_stopping_row(
        scoring_mode,
        candidate_size,
):
    rows = stopping_df.loc[
        (
            stopping_df["scoring_mode"]
            .astype(str)
            .eq(str(scoring_mode))
        )
        & (
            stopping_df[
                "training_candidate_size"
            ]
            .eq(candidate_size)
        )
    ]

    if len(rows) != 1:
        raise RuntimeError(
            "Expected exactly one stopping summary for "
            f"scoring_mode={scoring_mode!r}, "
            f"training_candidate_size={candidate_size}, "
            f"but found {len(rows)}."
        )

    return rows.iloc[0]


def plot_training_configuration(
        scoring_mode,
        candidate_size,
        common_y_limits=None,
):
    frame = epoch_df.loc[
        (
            epoch_df["scoring_mode"]
            .astype(str)
            .eq(str(scoring_mode))
        )
        & (
            epoch_df[
                "training_candidate_size"
            ]
            .eq(candidate_size)
        )
        & (
            epoch_df["split"]
            .isin(["train", "validation"])
        )
    ].copy()

    train = (
        frame.loc[frame["split"] == "train"]
        .sort_values("epoch")
    )
    validation = (
        frame.loc[frame["split"] == "validation"]
        .sort_values("epoch")
    )

    if train.empty or validation.empty:
        raise RuntimeError(
            "Training or validation trajectory is missing for "
            f"{scoring_mode}, {candidate_size} candidates."
        )

    stopping = lookup_stopping_row(
        scoring_mode=scoring_mode,
        candidate_size=candidate_size,
    )

    best_epoch = float(
        stopping["best_epoch_mean"]
    )
    best_epoch_std = float(
        stopping["best_epoch_std"]
    )
    best_val_loss = float(
        stopping["best_val_loss_mean"]
    )
    best_val_loss_std = float(
        stopping["best_val_loss_std"]
    )
    gap = float(
        stopping[
            "generalization_gap_at_best_mean"
        ]
    )
    gap_std = float(
        stopping[
            "generalization_gap_at_best_std"
        ]
    )

    figure, axis = plt.subplots(
        figsize=(8.6, 5.3)
    )

    train_line = axis.plot(
        train["epoch"],
        train["loss_mean"],
        linewidth=2.1,
        label="Training loss",
    )[0]

    axis.fill_between(
        train["epoch"].to_numpy(),
        (
            train["loss_mean"]
            - train["loss_std"]
        ).to_numpy(),
        (
            train["loss_mean"]
            + train["loss_std"]
        ).to_numpy(),
        color=train_line.get_color(),
        alpha=0.16,
        linewidth=0,
        label="_nolegend_",
    )

    validation_line = axis.plot(
        validation["epoch"],
        validation["loss_mean"],
        linewidth=2.1,
        linestyle="--",
        label="Validation loss",
    )[0]

    axis.fill_between(
        validation["epoch"].to_numpy(),
        (
            validation["loss_mean"]
            - validation["loss_std"]
        ).to_numpy(),
        (
            validation["loss_mean"]
            + validation["loss_std"]
        ).to_numpy(),
        color=validation_line.get_color(),
        alpha=0.16,
        linewidth=0,
        label="_nolegend_",
    )

    if np.isfinite(best_epoch_std) and best_epoch_std > 0:
        axis.axvspan(
            best_epoch - best_epoch_std,
            best_epoch + best_epoch_std,
            color=validation_line.get_color(),
            alpha=0.08,
            linewidth=0,
        )

    axis.axvline(
        best_epoch,
        color=validation_line.get_color(),
        linestyle=":",
        linewidth=1.5,
        label=(
            "Selected epoch "
            f"({best_epoch:.1f} ± "
            f"{best_epoch_std:.1f})"
        ),
    )

    target_name = TARGET_DISPLAY_NAMES.get(
        scoring_mode,
        str(scoring_mode).replace("_", " ").title(),
    )

    axis.set_title(
        f"{target_name}: "
        f"{int(candidate_size):,} training candidates"
    )
    axis.set_xlabel("Training epoch")
    axis.set_ylabel(
        "Binary cross-entropy loss"
    )

    if (
        COMMON_Y_AXIS_WITHIN_TARGET
        and common_y_limits is not None
    ):
        axis.set_ylim(*common_y_limits)

    metric_text = (
        f"Best validation loss: "
        f"{best_val_loss:.3f} ± "
        f"{best_val_loss_std:.3f}\n"
        f"Gap at selected checkpoint: "
        f"{gap:.3f} ± {gap_std:.3f}\n"
        f"Seeds: {expected_seed_count}"
    )

    axis.text(
        0.98,
        0.97,
        metric_text,
        transform=axis.transAxes,
        ha="right",
        va="top",
        fontsize=9,
        bbox={
            "boxstyle": "round,pad=0.4",
            "facecolor": "white",
            "edgecolor": "0.80",
            "alpha": 0.92,
        },
    )

    clean_axis(axis)
    axis.legend(
        frameon=False,
        loc="best",
    )

    figure.tight_layout()

    stem = (
        "training_dynamics"
        f"__target-{safe_token(scoring_mode)}"
        f"__candidates-{int(candidate_size)}"
    )

    save_figure(
        figure,
        stem,
    )


def plot_gap_configuration(
        scoring_mode,
        candidate_size,
):
    frame = gap_df.loc[
        (
            gap_df["scoring_mode"]
            .astype(str)
            .eq(str(scoring_mode))
        )
        & (
            gap_df[
                "training_candidate_size"
            ]
            .eq(candidate_size)
        )
    ].copy()

    frame["epoch"] = pd.to_numeric(
        frame["epoch"],
        errors="coerce",
    )
    frame["validation_minus_train_mean"] = (
        pd.to_numeric(
            frame[
                "validation_minus_train_mean"
            ],
            errors="coerce",
        )
    )
    frame["validation_minus_train_std"] = (
        pd.to_numeric(
            frame[
                "validation_minus_train_std"
            ],
            errors="coerce",
        )
    )

    frame = frame.dropna(
        subset=[
            "epoch",
            "validation_minus_train_mean",
            "validation_minus_train_std",
        ]
    ).sort_values("epoch")

    if frame.empty:
        return

    figure, axis = plt.subplots(
        figsize=(8.6, 4.9)
    )

    line = axis.plot(
        frame["epoch"],
        frame["validation_minus_train_mean"],
        linewidth=2.0,
        label="Validation minus training loss",
    )[0]

    axis.fill_between(
        frame["epoch"].to_numpy(),
        (
            frame["validation_minus_train_mean"]
            - frame["validation_minus_train_std"]
        ).to_numpy(),
        (
            frame["validation_minus_train_mean"]
            + frame["validation_minus_train_std"]
        ).to_numpy(),
        color=line.get_color(),
        alpha=0.16,
        linewidth=0,
    )

    axis.axhline(
        0.0,
        linestyle="--",
        linewidth=1.0,
    )

    target_name = TARGET_DISPLAY_NAMES.get(
        scoring_mode,
        str(scoring_mode).replace("_", " ").title(),
    )

    axis.set_title(
        f"{target_name}: train–validation gap "
        f"with {int(candidate_size):,} candidates"
    )
    axis.set_xlabel("Training epoch")
    axis.set_ylabel(
        "Validation loss minus training loss"
    )

    clean_axis(axis)
    figure.tight_layout()

    stem = (
        "generalization_gap"
        f"__target-{safe_token(scoring_mode)}"
        f"__candidates-{int(candidate_size)}"
    )

    save_figure(
        figure,
        stem,
    )


# ============================================================
# Generate the figures
# ============================================================

configuration_pairs = (
    epoch_df[
        [
            "scoring_mode",
            "training_candidate_size",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "scoring_mode",
            "training_candidate_size",
        ]
    )
)

for scoring_mode in configuration_pairs[
    "scoring_mode"
].unique():
    target_frame = epoch_df.loc[
        epoch_df["scoring_mode"]
        .astype(str)
        .eq(str(scoring_mode))
    ]

    common_y_limits = (
        get_target_y_limits(target_frame)
        if COMMON_Y_AXIS_WITHIN_TARGET
        else None
    )

    target_candidate_sizes = (
        configuration_pairs.loc[
            configuration_pairs[
                "scoring_mode"
            ].astype(str).eq(str(scoring_mode)),
            "training_candidate_size",
        ]
        .sort_values()
        .tolist()
    )

    for candidate_size in target_candidate_sizes:
        plot_training_configuration(
            scoring_mode=scoring_mode,
            candidate_size=candidate_size,
            common_y_limits=common_y_limits,
        )

        if PLOT_GENERALIZATION_GAP:
            plot_gap_configuration(
                scoring_mode=scoring_mode,
                candidate_size=candidate_size,
            )


# ============================================================
# Comparative model-selection table
# ============================================================

comparison_columns = [
    "scoring_mode",
    "training_candidate_size",
    "best_epoch_mean",
    "best_epoch_std",
    "stopped_at_mean",
    "stopped_at_std",
    "best_train_loss_mean",
    "best_train_loss_std",
    "best_val_loss_mean",
    "best_val_loss_std",
    "generalization_gap_at_best_mean",
    "generalization_gap_at_best_std",
    "training_runtime_seconds_mean",
    "training_runtime_seconds_std",
    "n_seeds",
]

comparison_table = (
    stopping_df[comparison_columns]
    .sort_values(
        [
            "scoring_mode",
            "training_candidate_size",
        ]
    )
    .reset_index(drop=True)
)

comparison_table.insert(
    0,
    "target",
    comparison_table["scoring_mode"].map(
        TARGET_DISPLAY_NAMES
    ),
)

comparison_csv_path = (
    TRAINING_PLOT_DIR
    / "training_model_selection_summary.csv"
)
comparison_table.to_csv(
    comparison_csv_path,
    index=False,
)

print("Saved:", comparison_csv_path)
display(comparison_table)

print()
print(
    "Finished. Every trajectory shows mean ± one sample "
    f"standard deviation across {expected_seed_count} seeds."
)

In [ ]:
# ============================================================
# RQ3: WHOLE-GRAPH TRANSFER EVALUATION
#
# Train scope:
#   Each selector remains trained on its original source subgraph.
#
# Test scope:
#   A new, independently mined candidate set from the COMPLETE graph.
#
# Fairness / leakage control:
#   - The same whole-graph test candidates are used for every selector
#     belonging to the same victim seed.
#   - Test candidates are excluded if they appeared in ANY selector's
#     training pair set for that victim seed.
#   - Classification thresholds are selected on each selector's LOCAL
#     validation split and then frozen before whole-graph testing.
#
# Outputs:
#   extendedPlotting/whole_graph_transfer_multiseed/
#       whole_graph_transfer_predictions_raw.csv
#       whole_graph_transfer_metrics_raw.csv
#       whole_graph_transfer_metrics_averaged.csv
#       whole_graph_transfer_test_manifest.csv
#
# Cache:
#   cache/rq3_whole_graph_transfer_test/
# ============================================================

from pathlib import Path
from collections import defaultdict
import hashlib
import json
import random

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

# ----------------------------
# Configuration
# ----------------------------

WG_TRANSFER_TEST_SIZE = 5_000
WG_TRANSFER_TEST_SEED = 73_911
WG_TRANSFER_TOP_K = (100, 500, 1_000)
WG_TRANSFER_SCORE_BATCH_SIZE = 100_000

WG_TRANSFER_CACHE_DIR = Path("cache") / "rq3_whole_graph_transfer_test"
WG_TRANSFER_OUT_DIR = (
    Path("extendedPlotting") / "whole_graph_transfer_multiseed"
)
WG_TRANSFER_CACHE_DIR.mkdir(parents=True, exist_ok=True)
WG_TRANSFER_OUT_DIR.mkdir(parents=True, exist_ok=True)

required_names = [
    "RQ3_SUBGRAPH_TRAINING_RUNS",
    "mine_candidate_edge_scores",
    "_predict_lp_probs",
    "SUBSET_FRACTION",
    "N_SUBSETS",
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the RQ3 mining and selector-training cells first. "
        f"Missing: {missing}"
    )

WG_TRANSFER_RUNS = list(RQ3_SUBGRAPH_TRAINING_RUNS)
if not WG_TRANSFER_RUNS:
    raise RuntimeError("No trained RQ3 selectors are available.")


# ----------------------------
# Small helpers
# ----------------------------

def _canonical_pair(u, v):
    u, v = int(u), int(v)
    return (u, v) if u < v else (v, u)


def _training_pairs_global(run):
    edge_index = torch.as_tensor(
        run["edge_index_lab_global"],
        dtype=torch.long,
    ).detach().cpu()

    if edge_index.ndim != 2 or edge_index.size(0) != 2:
        raise ValueError(
            f"edge_index_lab_global has invalid shape for "
            f"{run['lp_model_label']}: {tuple(edge_index.shape)}"
        )

    return {
        _canonical_pair(u, v)
        for u, v in edge_index.t().tolist()
        if int(u) != int(v)
    }


def _sample_unseen_whole_graph_pairs(
    n_nodes,
    count,
    forbidden,
    seed,
):
    """Uniformly sample unique unordered pairs outside all training pools."""
    n_nodes = int(n_nodes)
    count = int(count)
    forbidden = set(forbidden)

    n_possible = n_nodes * (n_nodes - 1) // 2
    n_available = n_possible - len(forbidden)
    if count > n_available:
        raise ValueError(
            f"Requested {count} whole-graph test candidates, "
            f"but only {n_available} unseen pairs remain."
        )

    rng = random.Random(int(seed))
    selected = set()

    while len(selected) < count:
        u = rng.randrange(n_nodes)
        v = rng.randrange(n_nodes)
        if u == v:
            continue

        pair = _canonical_pair(u, v)
        if pair in forbidden or pair in selected:
            continue

        selected.add(pair)

    return sorted(selected)


def _validation_threshold(run, fallback=0.5):
    """
    Select the probability threshold that maximizes F1 on the model's
    LOCAL validation split. The whole-graph test labels are never used.
    """
    history = getattr(run["lp_model"], "training_history", {}) or {}
    split_outputs = history.get("final_split_outputs", {})

    validation = (
        split_outputs.get("validation")
        or split_outputs.get("val")
    )
    if not validation:
        return float(fallback), "fixed_0.5_no_validation_outputs"

    y_true = np.asarray(
        validation.get("binary_targets", [])
    ).astype(int).reshape(-1)

    y_score = np.asarray(
        validation.get("probabilities", [])
    ).astype(float).reshape(-1)

    finite = np.isfinite(y_score)
    y_true = y_true[finite]
    y_score = y_score[finite]

    if (
        y_true.size == 0
        or y_true.size != y_score.size
        or np.unique(y_true).size < 2
    ):
        return float(fallback), "fixed_0.5_invalid_validation_outputs"

    precision, recall, thresholds = precision_recall_curve(
        y_true,
        y_score,
    )

    if thresholds.size == 0:
        return float(fallback), "fixed_0.5_no_validation_thresholds"

    f1_values = (
        2.0 * precision[:-1] * recall[:-1]
        / np.maximum(
            precision[:-1] + recall[:-1],
            1e-12,
        )
    )
    best_index = int(np.nanargmax(f1_values))
    return float(thresholds[best_index]), "local_validation_f1"


def _safe_auc(y_true, y_score):
    return (
        float(roc_auc_score(y_true, y_score))
        if np.unique(y_true).size == 2
        else np.nan
    )


def _safe_ap(y_true, y_score):
    return (
        float(average_precision_score(y_true, y_score))
        if np.any(y_true == 1)
        else np.nan
    )


def _classification_metrics(y_true, y_score, threshold):
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_score = np.asarray(y_score).astype(float).reshape(-1)
    y_pred = (y_score >= float(threshold)).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    return {
        "n_test_candidates": int(y_true.size),
        "n_positive_labels": int(y_true.sum()),
        "positive_prevalence": (
            float(y_true.mean()) if y_true.size else np.nan
        ),
        "average_precision": _safe_ap(y_true, y_score),
        "roc_auc": _safe_auc(y_true, y_score),
        "threshold": float(threshold),
        "precision": float(
            precision_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "mcc": float(
            matthews_corrcoef(
                y_true,
                y_pred,
            )
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def _top_k_metrics(y_true, y_score, k_values):
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_score = np.asarray(y_score).astype(float).reshape(-1)

    order = np.argsort(-y_score)
    n_positive = int(y_true.sum())
    prevalence = float(y_true.mean()) if y_true.size else np.nan

    output = {}
    for requested_k in k_values:
        k = min(int(requested_k), int(y_true.size))
        if k <= 0:
            continue

        positives_at_k = int(y_true[order[:k]].sum())
        precision_at_k = positives_at_k / k
        recall_at_k = (
            positives_at_k / n_positive
            if n_positive > 0
            else np.nan
        )
        lift_at_k = (
            precision_at_k / prevalence
            if prevalence > 0
            else np.nan
        )

        output[f"precision_at_{requested_k}"] = float(
            precision_at_k
        )
        output[f"recall_at_{requested_k}"] = float(
            recall_at_k
        )
        output[f"lift_at_{requested_k}"] = float(
            lift_at_k
        )

    return output


def _region_labels(source_nodes, src, dst, n_nodes):
    source_mask = np.zeros(int(n_nodes), dtype=bool)
    source_mask[
        torch.as_tensor(
            source_nodes,
            dtype=torch.long,
        ).detach().cpu().numpy()
    ] = True

    src_inside = source_mask[src]
    dst_inside = source_mask[dst]

    return np.where(
        src_inside & dst_inside,
        "inside_inside",
        np.where(
            src_inside ^ dst_inside,
            "inside_outside",
            "outside_outside",
        ),
    )


def _aggregate_mean_sd(frame, group_cols):
    metric_cols = [
        column
        for column in frame.columns
        if column not in group_cols
        and column not in {
            "seed",
            "lp_model_label",
            "threshold_source",
        }
        and pd.api.types.is_numeric_dtype(frame[column])
    ]

    grouped = (
        frame
        .groupby(group_cols, dropna=False)[metric_cols]
        .agg(["mean", "std"])
        .reset_index()
    )
    grouped.columns = [
        column
        if isinstance(column, str)
        else "_".join(
            str(part)
            for part in column
            if str(part)
        )
        for column in grouped.columns
    ]

    n_seeds = (
        frame
        .groupby(group_cols, dropna=False)["seed"]
        .nunique()
        .reset_index(name="n_seeds")
    )
    return grouped.merge(
        n_seeds,
        on=group_cols,
        how="left",
    )


# ----------------------------
# Group trained selectors by victim seed
# ----------------------------

runs_by_seed = defaultdict(list)
for run in WG_TRANSFER_RUNS:
    runs_by_seed[int(run["seed"])].append(run)


# ----------------------------
# Build or load one independent whole-graph test pool per seed
# ----------------------------

test_sets_by_seed = {}
manifest_rows = []

for victim_seed, seed_runs in sorted(runs_by_seed.items()):
    full_context = seed_runs[0].get(
        "full_context",
        seed_runs[0]["context"].get("full_context"),
    )
    if full_context is None:
        raise KeyError(
            f"No full_context found for victim seed {victim_seed}."
        )

    n_nodes = int(
        full_context.get(
            "n_nodes",
            full_context["attr"].size(0),
        )
    )

    # Exclude the union of all candidate pairs shown to any selector
    # for this victim seed.
    forbidden_pairs = set()
    for run in seed_runs:
        forbidden_pairs.update(
            _training_pairs_global(run)
        )

    context_fingerprint = (
        _context_fingerprint(full_context)
        if "_context_fingerprint" in globals()
        else f"seed-{victim_seed}-nodes-{n_nodes}"
    )

    fingerprint_payload = {
        "victim_seed": int(victim_seed),
        "context_fingerprint": str(context_fingerprint),
        "n_nodes": int(n_nodes),
        "test_size": int(WG_TRANSFER_TEST_SIZE),
        "test_seed": int(WG_TRANSFER_TEST_SEED),
        "n_forbidden": int(len(forbidden_pairs)),
        "forbidden_sha1": hashlib.sha1(
            json.dumps(
                sorted(forbidden_pairs),
                separators=(",", ":"),
            ).encode("utf-8")
        ).hexdigest()[:12],
    }
    cache_hash = hashlib.sha1(
        json.dumps(
            fingerprint_payload,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()[:12]

    cache_path = (
        WG_TRANSFER_CACHE_DIR
        / (
            f"whole_graph_transfer_test"
            f"__seed-{victim_seed}"
            f"__n-{WG_TRANSFER_TEST_SIZE}"
            f"__{cache_hash}.pt"
        )
    )

    if cache_path.exists():
        try:
            payload = torch.load(
                cache_path,
                map_location="cpu",
                weights_only=False,
            )
        except TypeError:
            payload = torch.load(
                cache_path,
                map_location="cpu",
            )
        cache_hit = True

    else:
        candidates = _sample_unseen_whole_graph_pairs(
            n_nodes=n_nodes,
            count=WG_TRANSFER_TEST_SIZE,
            forbidden=forbidden_pairs,
            seed=WG_TRANSFER_TEST_SEED + victim_seed,
        )

        # Label candidates on the complete victim graph using exactly the
        # endpoint criterion already used by the RQ3 mining pipeline.
        result = mine_candidate_edge_scores(
            model=full_context["model"].eval(),
            attr=full_context["attr"],
            edge_index=full_context["edge_index"],
            labels=full_context["labels"],
            eval_idx=full_context["idx_test"],
            candidates=candidates,
            n_nodes=n_nodes,
            mode="endpoint",
            subset_fraction=SUBSET_FRACTION,
            n_subsets=N_SUBSETS,
            endpoint_k_samples=None,
            endpoint_require_correct_to_incorrect=True,
            two_hop_k_samples=None,
            h=0,
            seed=WG_TRANSFER_TEST_SEED + victim_seed,
            device=full_context["device"],
            verbose=True,
        )

        observed_mask = np.asarray(
            result["observed_mask"],
            dtype=bool,
        ).reshape(-1)

        endpoint_labels = np.asarray(
            result["endpoint_labels"],
            dtype=np.int64,
        ).reshape(-1)

        if (
            observed_mask.size != len(candidates)
            or endpoint_labels.size != len(candidates)
        ):
            raise RuntimeError(
                "Whole-graph label output does not align with "
                "the sampled test candidate pool."
            )

        observed_indices = np.flatnonzero(observed_mask)
        observed_pairs = [
            candidates[index]
            for index in observed_indices
        ]

        exists_all = np.asarray(
            result.get(
                "exists",
                np.zeros(len(candidates), dtype=np.float32),
            ),
            dtype=np.float32,
        ).reshape(-1)

        payload = {
            "metadata": fingerprint_payload,
            "candidates": observed_pairs,
            "labels": endpoint_labels[observed_indices],
            "exists": exists_all[observed_indices],
            "clean_accuracy": float(result["clean_accuracy"]),
        }
        torch.save(payload, cache_path)
        cache_hit = False

    candidates = [
        _canonical_pair(u, v)
        for u, v in payload["candidates"]
    ]
    labels = np.asarray(
        payload["labels"],
        dtype=np.int64,
    ).reshape(-1)
    exists = np.asarray(
        payload["exists"],
        dtype=np.float32,
    ).reshape(-1)

    if not (
        len(candidates) == labels.size == exists.size
    ):
        raise RuntimeError(
            f"Cached whole-graph test arrays are misaligned "
            f"for victim seed {victim_seed}."
        )

    # Final leakage assertion.
    overlap = set(candidates).intersection(forbidden_pairs)
    if overlap:
        raise AssertionError(
            f"Whole-graph test leakage detected for seed "
            f"{victim_seed}: {len(overlap)} overlapping pairs."
        )

    edge_index_lab_global = torch.tensor(
        candidates,
        dtype=torch.long,
    ).t().contiguous()

    test_sets_by_seed[victim_seed] = {
        "full_context": full_context,
        "candidates": candidates,
        "edge_index_lab_global": edge_index_lab_global,
        "labels": labels,
        "exists": exists,
        "cache_path": str(cache_path),
    }

    manifest_rows.append({
        "seed": int(victim_seed),
        "n_nodes": int(n_nodes),
        "n_test_candidates": int(labels.size),
        "n_positive_labels": int(labels.sum()),
        "positive_prevalence": float(labels.mean()),
        "n_forbidden_training_pairs": int(len(forbidden_pairs)),
        "training_overlap": int(len(overlap)),
        "cache_hit": bool(cache_hit),
        "cache_path": str(cache_path),
    })


# ----------------------------
# Score every frozen selector on the complete graph
# ----------------------------

prediction_frames = []
metric_rows = []

for run in WG_TRANSFER_RUNS:
    victim_seed = int(run["seed"])
    test_data = test_sets_by_seed[victim_seed]
    full_context = test_data["full_context"]

    model = run["lp_model"]
    model_device = next(model.parameters()).device

    y_score = _predict_lp_probs(
        lp_model=model,
        x=full_context["attr"],
        edge_index_struct=full_context["edge_index"],
        edge_index_lab=test_data["edge_index_lab_global"],
        device=model_device,
        batch_size=WG_TRANSFER_SCORE_BATCH_SIZE,
    )

    y_true = test_data["labels"].copy()
    src = test_data["edge_index_lab_global"][0].numpy()
    dst = test_data["edge_index_lab_global"][1].numpy()
    exists = test_data["exists"].copy()

    if y_score.size != y_true.size:
        raise RuntimeError(
            f"Prediction/label mismatch for "
            f"{run['lp_model_label']}: "
            f"{y_score.size} versus {y_true.size}."
        )

    threshold, threshold_source = _validation_threshold(run)

    n_nodes = int(
        full_context.get(
            "n_nodes",
            full_context["attr"].size(0),
        )
    )
    regions = _region_labels(
        source_nodes=run["source_nodes_global"],
        src=src,
        dst=dst,
        n_nodes=n_nodes,
    )

    method = str(run.get("subgraph_method", "unknown"))
    fraction = float(run["subgraph_fraction_requested"])

    prediction_frames.append(pd.DataFrame({
        "seed": victim_seed,
        "lp_model_label": run["lp_model_label"],
        "lp_model_group_label": run["lp_model_group_label"],
        "subgraph_method": method,
        "subgraph_fraction_requested": fraction,
        "training_candidate_size": int(
            run["training_candidate_size"]
        ),
        "scoring_mode": run["scoring_mode"],
        "label_mode": run["label_mode"],
        "extreme_fraction": run.get("extreme_fraction", ""),
        "src": src,
        "dst": dst,
        "exists": exists,
        "evaluation_region": regions,
        "y_true": y_true,
        "y_score": y_score,
        "threshold": float(threshold),
        "y_pred": (y_score >= threshold).astype(int),
    }))

    region_masks = {
        "all": np.ones(y_true.size, dtype=bool),
        "inside_inside": regions == "inside_inside",
        "inside_outside": regions == "inside_outside",
        "outside_outside": regions == "outside_outside",
    }

    for region_name, mask in region_masks.items():
        if int(mask.sum()) == 0:
            continue

        y_true_region = y_true[mask]
        y_score_region = y_score[mask]

        row = {
            "seed": victim_seed,
            "lp_model_label": run["lp_model_label"],
            "lp_model_group_label": run["lp_model_group_label"],
            "subgraph_method": method,
            "subgraph_fraction_requested": fraction,
            "subgraph_fraction_actual": float(
                run["subgraph_fraction_actual"]
            ),
            "n_source_nodes": int(run["n_source_nodes"]),
            "n_global_nodes": int(run["n_global_nodes"]),
            "training_candidate_size": int(
                run["training_candidate_size"]
            ),
            "scoring_mode": run["scoring_mode"],
            "label_mode": run["label_mode"],
            "extreme_fraction": run.get("extreme_fraction", ""),
            "evaluation_region": region_name,
            "threshold_source": threshold_source,
        }

        row.update(
            _classification_metrics(
                y_true_region,
                y_score_region,
                threshold,
            )
        )
        row.update(
            _top_k_metrics(
                y_true_region,
                y_score_region,
                WG_TRANSFER_TOP_K,
            )
        )
        metric_rows.append(row)


# ----------------------------
# Export
# ----------------------------

whole_graph_transfer_predictions_df = pd.concat(
    prediction_frames,
    ignore_index=True,
)
whole_graph_transfer_metrics_raw_df = pd.DataFrame(
    metric_rows
)
whole_graph_transfer_manifest_df = pd.DataFrame(
    manifest_rows
)

group_cols = [
    "lp_model_group_label",
    "subgraph_method",
    "subgraph_fraction_requested",
    "training_candidate_size",
    "scoring_mode",
    "label_mode",
    "extreme_fraction",
    "evaluation_region",
]

whole_graph_transfer_metrics_averaged_df = (
    _aggregate_mean_sd(
        whole_graph_transfer_metrics_raw_df,
        group_cols=group_cols,
    )
)

whole_graph_transfer_predictions_df.to_csv(
    WG_TRANSFER_OUT_DIR
    / "whole_graph_transfer_predictions_raw.csv",
    index=False,
)
whole_graph_transfer_metrics_raw_df.to_csv(
    WG_TRANSFER_OUT_DIR
    / "whole_graph_transfer_metrics_raw.csv",
    index=False,
)
whole_graph_transfer_metrics_averaged_df.to_csv(
    WG_TRANSFER_OUT_DIR
    / "whole_graph_transfer_metrics_averaged.csv",
    index=False,
)
whole_graph_transfer_manifest_df.to_csv(
    WG_TRANSFER_OUT_DIR
    / "whole_graph_transfer_test_manifest.csv",
    index=False,
)

print(
    "Whole-graph transfer evaluation complete.\n"
    f"Output directory: {WG_TRANSFER_OUT_DIR.resolve()}\n"
    f"Raw metric rows: {len(whole_graph_transfer_metrics_raw_df)}\n"
    f"Prediction rows: {len(whole_graph_transfer_predictions_df)}"
)

display(
    whole_graph_transfer_manifest_df
)

display_columns = [
    "subgraph_method",
    "subgraph_fraction_requested",
    "evaluation_region",
    "n_test_candidates_mean",
    "positive_prevalence_mean",
    "average_precision_mean",
    "precision_mean",
    "recall_mean",
    "f1_mean",
    "mcc_mean",
]

for k in WG_TRANSFER_TOP_K:
    for prefix in ("precision", "recall", "lift"):
        column = f"{prefix}_at_{k}_mean"
        if column in whole_graph_transfer_metrics_averaged_df:
            display_columns.append(column)

display(
    whole_graph_transfer_metrics_averaged_df[
        [
            column
            for column in display_columns
            if column in whole_graph_transfer_metrics_averaged_df
        ]
    ].sort_values(
        [
            "evaluation_region",
            "subgraph_method",
            "subgraph_fraction_requested",
        ]
    )
)


---
## §6  Score Distributions & Ranking Quality

We score every candidate and ask:
- Do edges with V4 label > 0 (appeared in at least one harmful subset) receive higher logits?
- Does the predicted score *correlate* with the true continuous label?
- AUC-ROC (binary: scored vs unscored) and precision@k are computed as proxy metrics.

In [ ]:
from pathlib import Path
import torch
import numpy as np
import pandas as pd
from IPython.display import display

SCORE_DIAGNOSTIC_OUT_DIR = Path("extendedPlotting") / "score_diagnostics_multiseed"
SCORE_DIAGNOSTIC_OUT_DIR.mkdir(parents=True, exist_ok=True)


def _tensor_stats(values):
    values = values.detach().cpu().float().flatten()
    if not values.numel(): return {k: np.nan for k in ["mean","std","min","max"]} | {"count": 0}
    return {"count": values.numel(), "mean": values.mean().item(), "std": values.std(unbiased=False).item(),
            "min": values.min().item(), "max": values.max().item()}


def score_candidate_run_with_lp_gnn(run):
    scorer, context = run["lp_model"], run["context"]
    dev = next(scorer.parameters()).device; scorer.eval()
    edge_lab = torch.stack([run["src"].long(), run["dst"].long()]).to(dev)
    with torch.no_grad():
        logits = scorer(context["attr"].to(dev), context["edge_index"].to(dev), edge_lab).view(-1)
        probs = torch.sigmoid(logits)
    logits, probs = logits.cpu(), probs.cpu()
    labels_cont = run["labels_changed"].cpu().float().view(-1); labels_binary = labels_cont > 0
    pos, neg = labels_binary, ~labels_binary
    output_df = pd.DataFrame({
        "seed": run["seed"], "candidate_idx": np.arange(len(labels_cont)),
        "src": run["src"].cpu().numpy(), "dst": run["dst"].cpu().numpy(),
        "exists": run["exists"].cpu().numpy(), "label": labels_cont.numpy(),
        "label_binary": labels_binary.numpy().astype(int), "logit": logits.numpy(), "probability": probs.numpy(),
        "lp_model_group_label": run["lp_model_group_label"],
    })
    safe = _safe_config_value(run["lp_model_label"]); output_df.to_csv(SCORE_DIAGNOSTIC_OUT_DIR / f"score_outputs__{safe}.csv", index=False)
    ps, ns, pps, nps = _tensor_stats(logits[pos]), _tensor_stats(logits[neg]), _tensor_stats(probs[pos]), _tensor_stats(probs[neg])
    summary = {
        "seed": run["seed"], "lp_model_group_label": run["lp_model_group_label"],
        **{k: run.get(k, "") for k in ["candidate_config_id","candidate_set_size","prbcd_candidate_fraction","actual_prbcd_fraction","scoring_mode","endpoint_mining_hop","label_mode","extreme_fraction"]},
        "n_candidates": len(labels_cont), "n_positive_labels": labels_binary.sum().item(),
        "positive_fraction": labels_binary.float().mean().item(), "fraction_predicted_positive_at_0_5": (probs >= .5).float().mean().item(),
        "pos_logit_mean": ps["mean"], "pos_logit_std_within": ps["std"], "neg_logit_mean": ns["mean"], "neg_logit_std_within": ns["std"],
        "pos_prob_mean": pps["mean"], "pos_prob_std_within": pps["std"], "neg_prob_mean": nps["mean"], "neg_prob_std_within": nps["std"],
    }
    return {"run": run, "labels_cont": labels_cont, "labels_binary": labels_binary, "all_logits": logits,
            "all_probs": probs, "pos_logits": logits[pos], "neg_logits": logits[neg], "output_df": output_df, "summary_row": summary}

score_diagnostic_runs = [
    score_candidate_run_with_lp_gnn(run)
    for run in RQ3_TRAINING_STAT_RUNS
]
score_diagnostics_raw_df = pd.DataFrame([item["summary_row"] for item in score_diagnostic_runs])
score_group_cols = ["lp_model_group_label","candidate_config_id","candidate_set_size","prbcd_candidate_fraction","scoring_mode","endpoint_mining_hop","label_mode","extreme_fraction"]
score_diagnostics_df = aggregate_over_seeds(score_diagnostics_raw_df, score_group_cols, [
    "actual_prbcd_fraction","n_candidates","n_positive_labels","positive_fraction","fraction_predicted_positive_at_0_5",
    "pos_logit_mean","pos_logit_std_within","neg_logit_mean","neg_logit_std_within","pos_prob_mean","pos_prob_std_within","neg_prob_mean","neg_prob_std_within",
])
score_diagnostics_raw_df.to_csv(SCORE_DIAGNOSTIC_OUT_DIR / "score_diagnostics_raw.csv", index=False)
score_diagnostics_df.to_csv(SCORE_DIAGNOSTIC_OUT_DIR / "score_diagnostics_averaged.csv", index=False)
display(score_diagnostics_df)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

for label, group_items in pd.DataFrame([{"label": s["run"]["lp_model_group_label"], "scored": s} for s in score_diagnostic_runs]).groupby("label"):
    items = group_items.scored.tolist()
    all_logits = np.concatenate([s["all_logits"].numpy() for s in items])
    bins = np.linspace(all_logits.min()-.1, all_logits.max()+.1, 40)
    neg_counts = np.vstack([np.histogram(s["neg_logits"].numpy(), bins=bins)[0] for s in items])
    pos_counts = np.vstack([np.histogram(s["pos_logits"].numpy(), bins=bins)[0] for s in items])
    precision_rows, calibration_rows = [], []
    for s in items:
        logits = s["all_logits"].numpy(); binary = s["labels_binary"].numpy().astype(float); cont = s["labels_cont"].numpy()
        order = np.argsort(-logits); precision = binary[order].cumsum() / np.arange(1, len(binary)+1)
        precision_rows.extend({"seed": s["run"]["seed"], "k": k+1, "precision": value} for k, value in enumerate(precision))
        quantiles = np.linspace(0, 1, 11); edges = np.unique(np.quantile(cont, quantiles))
        if len(edges) < 2: edges = np.array([cont.min()-.5, cont.max()+.5])
        bins_id = np.clip(np.digitize(cont, edges[1:-1]), 0, len(edges)-2)
        for b in range(len(edges)-1):
            mask = bins_id == b
            if mask.any(): calibration_rows.append({"seed": s["run"]["seed"], "label_bin": b, "true_label": cont[mask].mean(), "logit": logits[mask].mean()})
    precision_df = pd.DataFrame(precision_rows).groupby("k", as_index=False).agg(mean=("precision","mean"), std=("precision","std")); precision_df["std"] = precision_df["std"].fillna(0)
    calibration_df = pd.DataFrame(calibration_rows).groupby("label_bin", as_index=False).agg(x=("true_label","mean"), y=("logit","mean"), ystd=("logit","std")); calibration_df["ystd"] = calibration_df.ystd.fillna(0)
    fig, axes = plt.subplots(1,3,figsize=(15,4)); centers=(bins[:-1]+bins[1:])/2
    for counts, name in [(neg_counts,"label = 0"),(pos_counts,"label > 0")]:
        mean,std=counts.mean(0),counts.std(0); axes[0].step(centers,mean,where="mid",label=name); axes[0].fill_between(centers,mean-std,mean+std,step="mid",alpha=.15)
    axes[0].set_xlabel("raw logit"); axes[0].set_ylabel("mean bin count ± SD"); axes[0].legend(); axes[0].grid(True,alpha=.3)
    add_mean_std_band(axes[1], precision_df.k, precision_df["mean"], precision_df["std"], label="precision@k")
    baseline=np.mean([s["labels_binary"].float().mean().item() for s in items]); axes[1].axhline(baseline,linestyle="--",label=f"baseline {baseline:.3f}")
    axes[1].set_xlabel("k"); axes[1].set_ylabel("precision"); axes[1].legend(); axes[1].grid(True,alpha=.3)
    axes[2].errorbar(calibration_df.x,calibration_df.y,yerr=calibration_df.ystd,marker="o",capsize=3)
    axes[2].set_xlabel("mean true mined label in bin"); axes[2].set_ylabel("mean predicted logit ± SD"); axes[2].grid(True,alpha=.3)
    fig.suptitle("Scorer quality"); fig.tight_layout(); fig.savefig(RUN_PLOTS_DIR / f"score_quality__{_safe_config_value(label)}.png",dpi=200,bbox_inches="tight"); plt.show()


---
## §7  Per-Node Analysis

For each node in the candidate set, we check:
- How many of its incident candidate edges have a non-zero V4 label?
- What is the mean predicted logit for scored vs unscored incident edges?
- A positive score gap means the scorer correctly prefers harmful incident edges for this node.

In [ ]:
from collections import defaultdict
import pandas as pd
import numpy as np
from IPython.display import display

node_frames=[]
for scored in score_diagnostic_runs:
    run=scored["run"]; src=run["src"].cpu().numpy(); dst=run["dst"].cpu().numpy(); labels_np=scored["labels_cont"].numpy(); logits=scored["all_logits"].numpy()
    degrees_run=np.bincount(run["context"]["edge_index"][0].cpu().numpy(),minlength=run["context"]["n_nodes"])
    mapping=defaultdict(list)
    for i,(u,v) in enumerate(zip(src,dst)):
        item=(labels_np[i]>0,labels_np[i],logits[i]); mapping[int(u)].append(item); mapping[int(v)].append(item)
    rows=[]
    for node,cands in mapping.items():
        pos=[lg for is_pos,_,lg in cands if is_pos]; neg=[lg for is_pos,_,lg in cands if not is_pos]
        rows.append({"seed":run["seed"],"lp_model_group_label":run["lp_model_group_label"],"node":node,"degree":degrees_run[node],
                     "n_incident":len(cands),"n_scored":sum(x[0] for x in cands),"scored_rate":np.mean([x[0] for x in cands]),
                     "mean_score_pos":np.mean(pos) if pos else np.nan,"mean_score_neg":np.mean(neg) if neg else np.nan})
    node_frames.append(pd.DataFrame(rows))
node_analysis_df=pd.concat(node_frames,ignore_index=True) if node_frames else pd.DataFrame()
node_seed_summary=(node_analysis_df.groupby(["seed","lp_model_group_label"],as_index=False).agg(nodes_with_incident_candidates=("node","count"),mean_scored_rate=("scored_rate","mean"),median_scored_rate=("scored_rate","median"),mean_score_pos=("mean_score_pos","mean"),mean_score_neg=("mean_score_neg","mean")))
node_summary_df=aggregate_over_seeds(node_seed_summary,["lp_model_group_label"],["nodes_with_incident_candidates","mean_scored_rate","median_scored_rate","mean_score_pos","mean_score_neg"])
node_analysis_df.to_csv(SCORE_DIAGNOSTIC_OUT_DIR / "node_analysis_raw.csv",index=False); node_summary_df.to_csv(SCORE_DIAGNOSTIC_OUT_DIR / "node_analysis_averaged.csv",index=False)
display(node_summary_df)


In [ ]:
for label, raw in node_analysis_df.groupby("lp_model_group_label"):
    node_avg=(raw.groupby("node",as_index=False).agg(scored_rate=("scored_rate","mean"),mean_score_pos=("mean_score_pos","mean"),mean_score_neg=("mean_score_neg","mean"),degree=("degree","mean")))
    fig,axes=plt.subplots(1,3,figsize=(14,4)); axes[0].hist(node_avg.scored_rate,bins=20,edgecolor="white")
    axes[0].set_xlabel("seed-averaged incident positive-label rate"); axes[0].set_ylabel("nodes"); axes[0].grid(True,alpha=.3)
    both=node_avg.dropna(subset=["mean_score_pos","mean_score_neg"])
    if not both.empty:
        axes[1].scatter(both.mean_score_neg,both.mean_score_pos,alpha=.5,s=20); lo=min(both.mean_score_neg.min(),both.mean_score_pos.min()); hi=max(both.mean_score_neg.max(),both.mean_score_pos.max()); axes[1].plot([lo,hi],[lo,hi],linestyle="--")
        gap=both.mean_score_pos-both.mean_score_neg; axes[2].hist(gap,bins=20,edgecolor="white"); axes[2].axvline(0,linestyle="--"); axes[2].axvline(gap.mean(),label=f"mean={gap.mean():.3f}"); axes[2].legend()
    axes[1].set_xlabel("mean logit label=0"); axes[1].set_ylabel("mean logit label>0"); axes[1].grid(True,alpha=.3)
    axes[2].set_xlabel("seed-averaged score gap"); axes[2].set_ylabel("nodes"); axes[2].grid(True,alpha=.3)
    fig.suptitle("Per-node scorer analysis"); fig.tight_layout(); fig.savefig(RUN_PLOTS_DIR / f"node_analysis__{_safe_config_value(label)}.png",dpi=200,bbox_inches="tight"); plt.show()


## Fixed block selector informed amendment

In [ ]:
# ============================================================
# RQ3 MATCHED SELECTOR BENCHMARK
# CELL 1: configuration and matched experiment plan
#
# Purpose
# -------
# Compare every completed RQ2 balanced-block configuration with the
# ONE primary subgraph-trained selector exposed through `lp_training_runs`.
#
# Important separation
# --------------------
# - RQ2 matching uses the RQ2 mining configuration.
# - Selector matching uses only the victim seed plus the configured
#   primary selector metadata.
# - The selector's candidate_config_id and endpoint hop are never used
#   as RQ2 matching keys.
# - This experiment does NOT loop over the complete RQ3 subgraph sweep.
# ============================================================

from datetime import datetime
from pathlib import Path

import inspect
import json
import re

import numpy as np
import pandas as pd
import torch

from IPython.display import display
from experiments import experiment_global_attack_direct
from rgnn_at_scale.attacks.prbcd import PRBCD


# ============================================================
# Configuration
# ============================================================

# Selector to use. `lp_training_runs` already contains only the primary
# subgraph fraction, candidate size, and subgraph seed selected at the
# beginning of the notebook.
RQ3M_SELECTOR_SCORING_MODE = "endpoint"
RQ3M_SELECTOR_LABEL_MODE = "continuous"

# RQ2 comparison scope. None means every value represented in the
# completed RQ2 balanced-block experiment.
RQ3M_RQ2_SCORING_MODES = None
RQ3M_RQ2_ENDPOINT_HOPS = None
RQ3M_CANDIDATE_CONFIG_IDS = None
RQ3M_REPEATS = None

RQ3M_SELECTOR_PARAMS = {
    "accuracy_drop_selector_mode": "one_sample",
    "n_candidates_k_sample": 2_000,
    "n_candidates_one_sample": 5_000,
    "drop_mode": "endpoint",
    "acc_drop_threshold_k_samples": 1e-3,
    "loss_drop_threshold_k_samples": 1e-3,
    "k_samples_batch": 10,
    "training_data_node_cap": 15,
    "tau": 0.7,
    "score_batch_size": 1_000,
    "max_sampling_tries": 2_000_000,
    "exclude_tried": True,
    "resampling_mode": "topk",
    "top_k_per_batch": 100,
    "lp_hit_rate_detour": False,
}

# Two selector conditions, directly analogous to the two optimization
# regimes used for every RQ2 initialization.
RQ3M_CONDITIONS = [
    {
        "condition": "selector__fixed_block",
        "initialization": "selector",
        "resampling_enabled": False,
        "resampling_policy": "none",
        "use_cert": "accuracy_drop_selector",
    },
    {
        "condition": "selector__selector_resampling",
        "initialization": "selector",
        "resampling_enabled": True,
        "resampling_policy": "selector",
        "use_cert": "accuracy_drop_selector_with_resampling",
    },
]

RQ3M_BASE_OUT_DIR = Path("extendedPlotting") / "rq3_matched_selector_vs_rq2"
RQ3M_RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RQ3M_OUT_DIR = RQ3M_BASE_OUT_DIR / f"{DATASET}__{RQ3M_RUN_ID}"
RQ3M_INITIAL_BLOCK_DIR = RQ3M_OUT_DIR / "selector_initial_blocks"
RQ3M_OUT_DIR.mkdir(parents=True, exist_ok=True)
RQ3M_INITIAL_BLOCK_DIR.mkdir(parents=True, exist_ok=True)
RQ3M_BASE_OUT_DIR.mkdir(parents=True, exist_ok=True)
(RQ3M_BASE_OUT_DIR / "latest_run.txt").write_text(
    str(RQ3M_OUT_DIR.resolve()),
    encoding="utf-8",
)


# ============================================================
# Validate required notebook objects
# ============================================================

required_globals = [
    "lp_training_runs",
    "rq2_block_balance_raw_df",
    "rq2_block_runs_raw_df",
    "rq2_block_epochs_raw_df",
    "rq2_block_retention_raw_df",
    "rq2_source_runs",
    "graph_sparse_rq2",
    "N_RQ2_NODES",
    "_build_mode_condition_groups",
    "RQ2_BLOCK_EPSILON",
    "RQ2_BLOCK_EPOCHS",
    "RQ2_BLOCK_FINE_TUNE_EPOCHS",
    "RQ2_BLOCK_RESAMPLING_EPOCHS",
    "RQ2_BLOCK_WITH_EARLY_STOPPING",
    "RQ2_BLOCK_ATTACK_BUDGET",
    "RQ2_BLOCK_ARTIFACT_DIR",
    "RQ2_BLOCK_PERT_ADJ_STORAGE_TYPE",
    "RQ2_BLOCK_PERT_ATTR_STORAGE_TYPE",
    "RQ2_BLOCK_MODEL_STORAGE_TYPE",
    "MODEL_LABEL",
    "DATASET",
]

missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(
        "Run the complete RQ2 balanced-block experiment and the primary "
        f"RQ3 selector-training cells first. Missing: {missing_globals}"
    )

required_prbcd_args = {
    "lp_model",
    "resampling_enabled",
    "block_diagnostics_enabled",
    "attack_sampling_seed",
}
available_prbcd_args = set(inspect.signature(PRBCD.__init__).parameters)
missing_prbcd_args = required_prbcd_args - available_prbcd_args
if missing_prbcd_args:
    raise RuntimeError(
        "The loaded PRBCD implementation is missing matched-benchmark "
        f"arguments: {sorted(missing_prbcd_args)}"
    )


# ============================================================
# RQ2 keys and source-run metadata
# ============================================================

RQ3M_RQ2_SOURCE_KEY_COLUMNS = [
    "seed",
    "candidate_config_id",
    "scoring_mode",
    "endpoint_mining_hop",
]

RQ3M_RQ2_UNIT_KEY_COLUMNS = [
    "seed",
    "repeat",
    "candidate_config_id",
    "scoring_mode",
    "endpoint_mining_hop",
    "comparison_rule",
]


def _rq3m_rq2_source_key(mapping):
    return (
        int(mapping["seed"]),
        str(mapping["candidate_config_id"]),
        str(mapping["scoring_mode"]),
        int(mapping.get("endpoint_mining_hop", 0)),
    )


def _rq3m_rq2_unit_key(mapping):
    return (
        int(mapping["seed"]),
        int(mapping["repeat"]),
        str(mapping["candidate_config_id"]),
        str(mapping["scoring_mode"]),
        int(mapping.get("endpoint_mining_hop", 0)),
        str(mapping["comparison_rule"]),
    )


rq3m_source_index_lookup = {}
rq3m_source_run_lookup = {}
rq3m_mode_groups_lookup = {}

for source_index, source_run in enumerate(rq2_source_runs):
    source_key = _rq3m_rq2_source_key(source_run)
    if source_key in rq3m_source_index_lookup:
        raise RuntimeError(f"Duplicate RQ2 source-run key: {source_key}")

    rq3m_source_index_lookup[source_key] = int(source_index)
    rq3m_source_run_lookup[source_key] = source_run
    rq3m_mode_groups_lookup[source_key] = _build_mode_condition_groups(
        source_run,
        n_nodes=int(N_RQ2_NODES),
    )


# ============================================================
# Select exactly one primary selector per victim seed
# ============================================================


def _rq3m_matches_primary_selector(run):
    if str(run.get("scoring_mode")) != RQ3M_SELECTOR_SCORING_MODE:
        return False
    if str(run.get("label_mode")) != RQ3M_SELECTOR_LABEL_MODE:
        return False

    # These checks guarantee that this fixed-block experiment does not
    # accidentally expand over the complete subgraph sweep.
    if "RQ3_PRIMARY_SUBGRAPH_FRACTION" in globals():
        if not np.isclose(
            float(run.get("subgraph_fraction_requested", np.nan)),
            float(RQ3_PRIMARY_SUBGRAPH_FRACTION),
        ):
            return False
    if "RQ3_PRIMARY_TRAINING_CANDIDATE_SIZE" in globals():
        if int(run.get("training_candidate_size", -1)) != int(
            RQ3_PRIMARY_TRAINING_CANDIDATE_SIZE
        ):
            return False
    if "RQ3_PRIMARY_SUBGRAPH_SEED" in globals():
        if int(run.get("subgraph_seed", -1)) != int(RQ3_PRIMARY_SUBGRAPH_SEED):
            return False

    return True


def _rq3m_test_metric(trained_run, metric):
    direct = trained_run.get(f"test_{metric}")
    if direct is not None:
        try:
            return float(direct)
        except (TypeError, ValueError):
            pass

    history = getattr(trained_run.get("lp_model"), "training_history", {}) or {}
    test_metrics = (history.get("final_split_metrics", {}) or {}).get("test", {}) or {}
    value = test_metrics.get(metric)
    return float(value) if value is not None else np.nan


rq3m_primary_selector_by_seed = {}

for trained_run in lp_training_runs:
    if not _rq3m_matches_primary_selector(trained_run):
        continue

    victim_seed = int(trained_run["seed"])
    if victim_seed in rq3m_primary_selector_by_seed:
        first = rq3m_primary_selector_by_seed[victim_seed]
        raise RuntimeError(
            "More than one primary selector was found for victim seed "
            f"{victim_seed}: {first.get('lp_model_label')} and "
            f"{trained_run.get('lp_model_label')}"
        )

    rq3m_primary_selector_by_seed[victim_seed] = trained_run

if not rq3m_primary_selector_by_seed:
    raise RuntimeError(
        "No primary subgraph selector matched the configured selector mode."
    )


# ============================================================
# Build the plan from ALL selected RQ2 balanced-block rows
# ============================================================

rq3m_plan_df = rq2_block_balance_raw_df.copy()

if RQ3M_RQ2_SCORING_MODES is not None:
    allowed = {str(value) for value in RQ3M_RQ2_SCORING_MODES}
    rq3m_plan_df = rq3m_plan_df[
        rq3m_plan_df["scoring_mode"].astype(str).isin(allowed)
    ].copy()

if RQ3M_RQ2_ENDPOINT_HOPS is not None:
    allowed = {int(value) for value in RQ3M_RQ2_ENDPOINT_HOPS}
    hops = pd.to_numeric(
        rq3m_plan_df["endpoint_mining_hop"],
        errors="coerce",
    ).fillna(0).astype(int)
    rq3m_plan_df = rq3m_plan_df[hops.isin(allowed)].copy()

if RQ3M_CANDIDATE_CONFIG_IDS is not None:
    allowed = {str(value) for value in RQ3M_CANDIDATE_CONFIG_IDS}
    rq3m_plan_df = rq3m_plan_df[
        rq3m_plan_df["candidate_config_id"].astype(str).isin(allowed)
    ].copy()

if RQ3M_REPEATS is not None:
    allowed = {int(value) for value in RQ3M_REPEATS}
    rq3m_plan_df = rq3m_plan_df[
        rq3m_plan_df["repeat"].astype(int).isin(allowed)
    ].copy()

plan_columns = [
    "seed",
    "repeat",
    "candidate_config_id",
    "candidate_set_size",
    "scoring_mode",
    "comparison_rule",
    "endpoint_mining_hop",
    "balanced_block_size",
    "n_group_high_edges",
    "n_group_low_edges",
    "group_high_initialization",
    "group_low_initialization",
    "group_high_target",
    "group_low_target",
    "high_block_path",
    "low_block_path",
    "random_block_path",
]
missing_plan_columns = [column for column in plan_columns if column not in rq3m_plan_df]
if missing_plan_columns:
    raise KeyError(f"RQ2 balance table is missing: {missing_plan_columns}")

rq3m_plan_df = (
    rq3m_plan_df[plan_columns]
    .drop_duplicates()
    .sort_values(
        [
            "scoring_mode",
            "candidate_config_id",
            "seed",
            "repeat",
        ]
    )
    .reset_index(drop=True)
)

if rq3m_plan_df.empty:
    raise RuntimeError("No RQ2 balanced-block rows remain after filtering.")

plan_rows = []

for row in rq3m_plan_df.to_dict(orient="records"):
    source_key = _rq3m_rq2_source_key(row)
    victim_seed = int(row["seed"])

    if source_key not in rq3m_source_index_lookup:
        raise RuntimeError(f"No RQ2 source run matches {source_key}.")
    if victim_seed not in rq3m_primary_selector_by_seed:
        raise RuntimeError(
            "No primary subgraph selector exists for victim seed "
            f"{victim_seed}."
        )

    trained_run = rq3m_primary_selector_by_seed[victim_seed]
    mode_groups = rq3m_mode_groups_lookup[source_key]
    block_size = int(row["balanced_block_size"])

    if block_size <= int(RQ2_BLOCK_ATTACK_BUDGET):
        raise ValueError(
            f"Block size {block_size} must exceed attack budget "
            f"{RQ2_BLOCK_ATTACK_BUDGET}."
        )

    source_index = int(rq3m_source_index_lookup[source_key])
    attack_sampling_seed = (
        500_000
        + victim_seed * 10_000
        + source_index * 1_000
        + int(row["repeat"])
    )

    high_initialization = str(row["group_high_initialization"])
    low_initialization = str(row["group_low_initialization"])

    plan_rows.append({
        **row,
        "block_size": block_size,
        "source_index": source_index,
        "attack_sampling_seed": int(attack_sampling_seed),
        "group_high_display": mode_groups.get(
            "group_high_display",
            row["group_high_target"],
        ),
        "group_low_display": mode_groups.get(
            "group_low_display",
            row["group_low_target"],
        ),
        "rq2_high_fixed_condition": f"{high_initialization}__fixed_block",
        "rq2_low_fixed_condition": f"{low_initialization}__fixed_block",
        "rq2_random_fixed_condition": "random_full_space__fixed_block",
        "rq2_high_resampling_condition": (
            f"{high_initialization}__random_resampling"
        ),
        "rq2_low_resampling_condition": (
            f"{low_initialization}__random_resampling"
        ),
        "rq2_random_resampling_condition": (
            "random_full_space__random_resampling"
        ),
        "selector_lp_model_label": trained_run.get("lp_model_label", ""),
        "selector_lp_model_group_label": trained_run.get(
            "lp_model_group_label", ""
        ),
        "selector_scoring_mode": str(trained_run.get("scoring_mode", "")),
        "selector_endpoint_mining_hop": int(
            trained_run.get("endpoint_mining_hop", 0)
        ),
        "selector_label_mode": str(trained_run.get("label_mode", "")),
        "selector_training_candidate_size": int(
            trained_run.get("training_candidate_size", 0)
        ),
        "selector_subgraph_seed": int(trained_run.get("subgraph_seed", 0)),
        "selector_subgraph_fraction_requested": float(
            trained_run.get("subgraph_fraction_requested", np.nan)
        ),
        "selector_subgraph_fraction_actual": float(
            trained_run.get("subgraph_fraction_actual", np.nan)
        ),
        "selector_n_source_nodes": int(
            trained_run.get("n_source_nodes", trained_run.get("n_selector_nodes", 0))
        ),
        "selector_n_target_nodes": int(trained_run.get("n_target_nodes", 0)),
        "selector_training_scope": str(
            trained_run.get("selector_training_scope", "")
        ),
        "selector_test_auc": _rq3m_test_metric(trained_run, "auc"),
        "selector_test_ap": _rq3m_test_metric(trained_run, "ap"),
    })

rq3m_plan_df = pd.DataFrame(plan_rows)


# ============================================================
# Mechanical pairing checks
# ============================================================

rq3m_size_check = rq2_block_runs_raw_df.merge(
    rq3m_plan_df[
        RQ3M_RQ2_UNIT_KEY_COLUMNS + ["block_size"]
    ].drop_duplicates(),
    on=RQ3M_RQ2_UNIT_KEY_COLUMNS,
    how="inner",
    suffixes=("_rq2", "_plan"),
    validate="many_to_one",
)

if rq3m_size_check.empty:
    raise RuntimeError("The RQ3 plan did not match any completed RQ2 runs.")

size_mismatch = rq3m_size_check[
    rq3m_size_check["block_size_rq2"].astype(int)
    != rq3m_size_check["block_size_plan"].astype(int)
]
if not size_mismatch.empty:
    display(size_mismatch)
    raise AssertionError("A selector run would use a different RQ2 block size.")

# Exactly one selector label per victim seed is expected.
selector_count_check = rq3m_plan_df.groupby("seed")[
    "selector_lp_model_label"
].nunique()
if not (selector_count_check == 1).all():
    raise AssertionError("The plan contains multiple primary selectors per seed.")


# ============================================================
# Save plan and configuration
# ============================================================

rq3m_plan_df.to_csv(RQ3M_OUT_DIR / "matched_experiment_plan.csv", index=False)

rq3m_config = {
    "dataset": DATASET,
    "selector_scoring_mode": RQ3M_SELECTOR_SCORING_MODE,
    "selector_label_mode": RQ3M_SELECTOR_LABEL_MODE,
    "rq2_scoring_modes": (
        None
        if RQ3M_RQ2_SCORING_MODES is None
        else list(RQ3M_RQ2_SCORING_MODES)
    ),
    "rq2_endpoint_hops": (
        None
        if RQ3M_RQ2_ENDPOINT_HOPS is None
        else list(RQ3M_RQ2_ENDPOINT_HOPS)
    ),
    "epsilon": float(RQ2_BLOCK_EPSILON),
    "epochs": int(RQ2_BLOCK_EPOCHS),
    "fine_tune_epochs": int(RQ2_BLOCK_FINE_TUNE_EPOCHS),
    "nominal_resampling_epochs": int(RQ2_BLOCK_RESAMPLING_EPOCHS),
    "attack_budget": int(RQ2_BLOCK_ATTACK_BUDGET),
    "selector_params": RQ3M_SELECTOR_PARAMS,
    "conditions": RQ3M_CONDITIONS,
}
with open(RQ3M_OUT_DIR / "experiment_config.json", "w", encoding="utf-8") as handle:
    json.dump(rq3m_config, handle, indent=2)

print("Matched RQ2 units:", len(rq3m_plan_df))
print("RQ2 scoring modes:", sorted(rq3m_plan_df["scoring_mode"].astype(str).unique()))
print("Primary selector labels:", sorted(rq3m_plan_df["selector_lp_model_label"].unique()))
print("Block sizes:", sorted(rq3m_plan_df["block_size"].astype(int).unique()))
print("Output directory:", RQ3M_OUT_DIR.resolve())
display(rq3m_plan_df)


In [ ]:
# ============================================================
# RQ3 MATCHED SELECTOR BENCHMARK
# CELL 2: result, block-composition, and diagnostics helpers
# ============================================================

import hashlib
import random

import numpy as np
import torch


def _rq3m_set_global_seed(seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _rq3m_as_list(value):
    if value is None:
        return []
    if torch.is_tensor(value):
        value = value.detach().cpu()
        return [value.item()] if value.ndim == 0 else value.reshape(-1).tolist()
    if isinstance(value, np.ndarray):
        return value.reshape(-1).tolist()
    if isinstance(value, (list, tuple)):
        return list(value)
    return [value]


def _rq3m_find_attack_statistics(obj, seen=None):
    if seen is None:
        seen = set()
    obj_id = id(obj)
    if obj_id in seen:
        return None
    seen.add(obj_id)

    if isinstance(obj, dict):
        for key in ("attack_statistics", "attack_stats", "stats"):
            candidate = obj.get(key)
            if isinstance(candidate, dict) and (
                "accuracy" in candidate or "loss" in candidate
            ):
                return candidate
        if "accuracy" in obj and isinstance(
            obj["accuracy"], (list, tuple, np.ndarray, torch.Tensor)
        ):
            return obj
        for value in obj.values():
            found = _rq3m_find_attack_statistics(value, seen)
            if found is not None:
                return found
    elif isinstance(obj, (list, tuple)):
        for value in obj:
            found = _rq3m_find_attack_statistics(value, seen)
            if found is not None:
                return found
    return None


def _rq3m_extract_final_accuracy(result):
    if isinstance(result, dict):
        result_rows = result.get("results", []) or []
        if result_rows and isinstance(result_rows[0], dict) and "accuracy" in result_rows[0]:
            return float(torch.as_tensor(result_rows[0]["accuracy"]).detach().cpu().item())
    raise KeyError("Could not extract final attacked accuracy.")


def _rq3m_ids_tensor(value):
    if value is None:
        return torch.empty(0, dtype=torch.long)
    return torch.as_tensor(value, dtype=torch.long).detach().cpu().flatten()


def _rq3m_id_set(value):
    return set(_rq3m_ids_tensor(value).tolist())


def _rq3m_ids_hash(value):
    ids = torch.unique(_rq3m_ids_tensor(value), sorted=True)
    array = ids.numpy().astype(np.int64, copy=False)
    return hashlib.sha1(array.tobytes()).hexdigest()


def _rq3m_fraction(numerator, denominator):
    return float(numerator / denominator) if int(denominator) > 0 else np.nan


def _rq3m_epoch_to_drop(accuracies, clean_accuracy, required_drop):
    for stat_index, accuracy in enumerate(accuracies):
        if clean_accuracy - float(accuracy) >= float(required_drop):
            return int(stat_index - 1)
    return np.nan


def _rq3m_integrated_drop(accuracies, clean_accuracy):
    values = float(clean_accuracy) - np.asarray(accuracies, dtype=float)
    integrator = getattr(np, "trapezoid", np.trapz)
    return float(integrator(values, dx=1.0) / max(1, len(values) - 1))


def _rq3m_validate_block_diagnostics(stats, *, expected_block_size, resampling_enabled):
    diagnostics = stats.get("block_diagnostics", {}) or {}
    initial_block = _rq3m_ids_tensor(diagnostics.get("initial_block"))
    if initial_block.numel() == 0:
        raise RuntimeError("No initial selector block was recorded.")
    if int(initial_block.numel()) != int(expected_block_size):
        raise AssertionError(
            "Selector initialization returned the wrong block size: "
            f"expected={expected_block_size}, actual={initial_block.numel()}."
        )

    initial_hash = _rq3m_ids_hash(initial_block)
    epoch_blocks = diagnostics.get("epoch_blocks", {}) or {}
    changed_fixed_epochs = []

    for epoch, block in epoch_blocks.items():
        block = _rq3m_ids_tensor(block)
        if int(block.numel()) != int(expected_block_size):
            raise AssertionError(
                "Candidate block changed size: "
                f"epoch={epoch}, expected={expected_block_size}, actual={block.numel()}."
            )
        if not resampling_enabled and _rq3m_ids_hash(block) != initial_hash:
            changed_fixed_epochs.append(int(epoch))

    if changed_fixed_epochs:
        raise AssertionError(
            "Fixed selector block changed candidate IDs at epochs "
            f"{changed_fixed_epochs[:10]}."
        )

    final_block = _rq3m_ids_tensor(diagnostics.get("final_block"))
    if final_block.numel() > 0 and int(final_block.numel()) != int(expected_block_size):
        raise AssertionError(
            "Final block has the wrong size: "
            f"expected={expected_block_size}, actual={final_block.numel()}."
        )

    final_linear_ids = _rq3m_ids_tensor(diagnostics.get("final_linear_ids"))

    return {
        "diagnostics": diagnostics,
        "initial_block": initial_block,
        "initial_block_hash": initial_hash,
        "epoch_blocks": epoch_blocks,
        "final_block": final_block,
        "final_linear_ids": final_linear_ids,
        "n_resample_events": int(len(diagnostics.get("resample_events", []) or [])),
    }


def _rq3m_block_composition_metrics(
    *,
    initial_block,
    final_block,
    final_selected,
    group_high_ids,
    group_low_ids,
):
    initial_set = _rq3m_id_set(initial_block)
    final_block_set = _rq3m_id_set(final_block)
    final_selected_set = _rq3m_id_set(final_selected)
    high_set = _rq3m_id_set(group_high_ids)
    low_set = _rq3m_id_set(group_low_ids)

    return {
        "initial_high_count": len(initial_set & high_set),
        "initial_low_count": len(initial_set & low_set),
        "initial_high_fraction": _rq3m_fraction(len(initial_set & high_set), len(initial_set)),
        "initial_low_fraction": _rq3m_fraction(len(initial_set & low_set), len(initial_set)),
        "n_final_selected": len(final_selected_set),
        "n_final_selected_from_initial": len(final_selected_set & initial_set),
        "fraction_final_selected_from_initial": _rq3m_fraction(
            len(final_selected_set & initial_set),
            len(final_selected_set),
        ),
        "n_final_selected_group_high": len(final_selected_set & high_set),
        "fraction_final_selected_group_high": _rq3m_fraction(
            len(final_selected_set & high_set),
            len(final_selected_set),
        ),
        "n_final_selected_group_low": len(final_selected_set & low_set),
        "fraction_final_selected_group_low": _rq3m_fraction(
            len(final_selected_set & low_set),
            len(final_selected_set),
        ),
        "final_block_initial_retention": _rq3m_fraction(
            len(final_block_set & initial_set),
            len(initial_set),
        ),
        "final_block_group_high_fraction": _rq3m_fraction(
            len(final_block_set & high_set),
            len(final_block_set),
        ),
        "final_block_group_low_fraction": _rq3m_fraction(
            len(final_block_set & low_set),
            len(final_block_set),
        ),
    }


In [ ]:
# ============================================================
# RQ3 MATCHED SELECTOR BENCHMARK
# CELL 3: execute primary-selector fixed and resampling runs
# ============================================================

from timeit import default_timer as timer

import gc
import traceback

import numpy as np
import pandas as pd
import torch


RQ3M_STOP_ON_ERROR = False

rq3m_selector_run_rows = []
rq3m_selector_epoch_rows = []
rq3m_selector_retention_rows = []
rq3m_error_rows = []

for plan_index, plan_row in enumerate(
    rq3m_plan_df.to_dict(orient="records"),
    start=1,
):
    victim_seed = int(plan_row["seed"])
    repeat = int(plan_row["repeat"])
    block_size = int(plan_row["block_size"])
    attack_sampling_seed = int(plan_row["attack_sampling_seed"])
    source_key = _rq3m_rq2_source_key(plan_row)

    trained_run = rq3m_primary_selector_by_seed[victim_seed]
    selector_model = trained_run["lp_model"]
    selector_model.eval()

    mode_groups = rq3m_mode_groups_lookup[source_key]
    group_high_ids = mode_groups["group_high_ids_full"]
    group_low_ids = mode_groups["group_low_ids_full"]

    for condition_spec in RQ3M_CONDITIONS:
        condition = str(condition_spec["condition"])
        use_cert = str(condition_spec["use_cert"])
        resampling_enabled = bool(condition_spec["resampling_enabled"])

        print("\n" + "=" * 90)
        print(f"Plan {plan_index}/{len(rq3m_plan_df)}")
        print(
            f"seed={victim_seed} | repeat={repeat} | "
            f"RQ2 config={plan_row['candidate_config_id']} | "
            f"RQ2 mode={plan_row['scoring_mode']} | B={block_size} | "
            f"condition={condition}"
        )
        print("primary selector:", plan_row["selector_lp_model_label"])
        print("attack_sampling_seed:", attack_sampling_seed)

        attack_params = {
            "block_size": block_size,
            "epochs": int(RQ2_BLOCK_EPOCHS),
            "fine_tune_epochs": int(RQ2_BLOCK_FINE_TUNE_EPOCHS),
            "with_early_stopping": bool(RQ2_BLOCK_WITH_EARLY_STOPPING),
            "keep_heuristic": "WeightOnly",
            "do_synchronize": True,
            "loss_type": "tanhMargin",
            "lp_model": selector_model,
            "resampling_enabled": resampling_enabled,
            "block_diagnostics_enabled": True,
            "initial_block_label": condition,
            "attack_sampling_seed": attack_sampling_seed,
        }

        selector_params = dict(RQ3M_SELECTOR_PARAMS)
        _rq3m_set_global_seed(attack_sampling_seed)

        started = timer()
        result = None

        try:
            result = experiment_global_attack_direct.run(
                graph=graph_sparse_rq2,
                data_dir="./data",
                dataset=DATASET,
                attack="PRBCD",
                attack_params=attack_params,
                selector_params=selector_params,
                epsilons=[RQ2_BLOCK_EPSILON],
                binary_attr=False,
                make_undirected=True,
                seed=victim_seed,
                artifact_dir=RQ2_BLOCK_ARTIFACT_DIR,
                pert_adj_storage_type=RQ2_BLOCK_PERT_ADJ_STORAGE_TYPE,
                pert_attr_storage_type=RQ2_BLOCK_PERT_ATTR_STORAGE_TYPE,
                model_label=MODEL_LABEL,
                model_storage_type=RQ2_BLOCK_MODEL_STORAGE_TYPE,
                device="cpu",
                data_device="cpu",
                debug_level="info",
                semi=True,
                use_cert=use_cert,
            )

            runtime_seconds = timer() - started
            stats = _rq3m_find_attack_statistics(result)
            if stats is None:
                raise RuntimeError("No PRBCD attack statistics were returned.")

            accuracies = np.asarray(_rq3m_as_list(stats.get("accuracy")), dtype=float)
            losses = np.asarray(_rq3m_as_list(stats.get("loss")), dtype=float)
            if accuracies.size == 0:
                raise RuntimeError("Empty PRBCD accuracy trajectory.")

            clean_accuracy = float(accuracies[0])
            final_accuracy = _rq3m_extract_final_accuracy(result)
            minimum_relaxed_accuracy = (
                float(np.nanmin(accuracies[1:]))
                if accuracies.size > 1
                else clean_accuracy
            )

            block_info = _rq3m_validate_block_diagnostics(
                stats,
                expected_block_size=block_size,
                resampling_enabled=resampling_enabled,
            )

            composition = _rq3m_block_composition_metrics(
                initial_block=block_info["initial_block"],
                final_block=block_info["final_block"],
                final_selected=block_info["final_linear_ids"],
                group_high_ids=group_high_ids,
                group_low_ids=group_low_ids,
            )

            safe_selector = re.sub(
                r"[^A-Za-z0-9_.-]+",
                "_",
                str(plan_row["selector_lp_model_label"]),
            )
            safe_config = re.sub(
                r"[^A-Za-z0-9_.-]+",
                "_",
                str(plan_row["candidate_config_id"]),
            )
            initial_block_path = RQ3M_INITIAL_BLOCK_DIR / (
                f"{condition}__selector-{safe_selector}__seed-{victim_seed}"
                f"__repeat-{repeat}__rq2-{safe_config}__B-{block_size}.pt"
            )

            torch.save(
                {
                    "linear_ids": block_info["initial_block"],
                    "metadata": {
                        "condition": condition,
                        "seed": victim_seed,
                        "repeat": repeat,
                        "rq2_candidate_config_id": plan_row["candidate_config_id"],
                        "rq2_scoring_mode": plan_row["scoring_mode"],
                        "rq2_endpoint_mining_hop": int(plan_row["endpoint_mining_hop"]),
                        "block_size": block_size,
                        "attack_sampling_seed": attack_sampling_seed,
                        "selector_lp_model_label": plan_row["selector_lp_model_label"],
                    },
                },
                initial_block_path,
            )

            common_row = {
                # RQ2 comparison identity. These values intentionally describe
                # the RQ2 unit, not the selector's training run.
                "seed": victim_seed,
                "repeat": repeat,
                "candidate_config_id": plan_row["candidate_config_id"],
                "candidate_set_size": int(plan_row["candidate_set_size"]),
                "scoring_mode": str(plan_row["scoring_mode"]),
                "comparison_rule": str(plan_row["comparison_rule"]),
                "endpoint_mining_hop": int(plan_row["endpoint_mining_hop"]),
                "group_high_initialization": plan_row["group_high_initialization"],
                "group_low_initialization": plan_row["group_low_initialization"],
                "group_high_target": plan_row["group_high_target"],
                "group_low_target": plan_row["group_low_target"],
                "group_high_display": plan_row["group_high_display"],
                "group_low_display": plan_row["group_low_display"],
                "n_group_high_edges": int(plan_row["n_group_high_edges"]),
                "n_group_low_edges": int(plan_row["n_group_low_edges"]),

                "condition": condition,
                "initialization": "selector",
                "block_origin": "learned_selector",
                "target_group": "selector",
                "target_group_name": "learned_selector",
                "resampling_enabled": resampling_enabled,
                "resampling_policy": condition_spec["resampling_policy"],
                "use_cert": use_cert,

                "block_size": block_size,
                "balanced_block_size": block_size,
                "attack_budget": int(RQ2_BLOCK_ATTACK_BUDGET),
                "epsilon": float(RQ2_BLOCK_EPSILON),
                "epochs": int(RQ2_BLOCK_EPOCHS),
                "nominal_resampling_epochs": int(RQ2_BLOCK_RESAMPLING_EPOCHS),
                "fine_tune_epochs": int(RQ2_BLOCK_FINE_TUNE_EPOCHS),
                "attack_sampling_seed": attack_sampling_seed,

                # Selector identity kept in separate columns.
                "lp_model_label": plan_row["selector_lp_model_label"],
                "lp_model_group_label": plan_row["selector_lp_model_group_label"],
                "selector_scoring_mode": plan_row["selector_scoring_mode"],
                "selector_endpoint_mining_hop": int(
                    plan_row["selector_endpoint_mining_hop"]
                ),
                "selector_label_mode": plan_row["selector_label_mode"],
                "selector_training_candidate_size": int(
                    plan_row["selector_training_candidate_size"]
                ),
                "selector_subgraph_seed": int(plan_row["selector_subgraph_seed"]),
                "selector_subgraph_fraction_requested": float(
                    plan_row["selector_subgraph_fraction_requested"]
                ),
                "selector_subgraph_fraction_actual": float(
                    plan_row["selector_subgraph_fraction_actual"]
                ),
                "selector_n_source_nodes": int(plan_row["selector_n_source_nodes"]),
                "selector_n_target_nodes": int(plan_row["selector_n_target_nodes"]),
                "selector_training_scope": plan_row["selector_training_scope"],
                "selector_test_auc": plan_row["selector_test_auc"],
                "selector_test_ap": plan_row["selector_test_ap"],
            }

            rq3m_selector_run_rows.append({
                **common_row,
                **composition,
                "status": "ok",
                "clean_accuracy": clean_accuracy,
                "final_accuracy": final_accuracy,
                "final_accuracy_drop": clean_accuracy - final_accuracy,
                "minimum_relaxed_accuracy": minimum_relaxed_accuracy,
                "maximum_relaxed_accuracy_drop": (
                    clean_accuracy - minimum_relaxed_accuracy
                ),
                "integrated_accuracy_drop": _rq3m_integrated_drop(
                    accuracies,
                    clean_accuracy,
                ),
                "epoch_to_1pp_drop": _rq3m_epoch_to_drop(
                    accuracies,
                    clean_accuracy,
                    0.01,
                ),
                "epoch_to_2pp_drop": _rq3m_epoch_to_drop(
                    accuracies,
                    clean_accuracy,
                    0.02,
                ),
                "n_resample_events": block_info["n_resample_events"],
                "initial_block_hash": block_info["initial_block_hash"],
                "initial_block_path": str(initial_block_path.resolve()),
                "runtime_seconds": float(runtime_seconds),
            })

            selector_score_mean = _rq3m_as_list(stats.get("selector_score_mean"))
            selector_score_max = _rq3m_as_list(stats.get("selector_score_max"))
            selector_num_tries = _rq3m_as_list(stats.get("selector_score_num_tries"))
            selector_score_mode = _rq3m_as_list(stats.get("selector_score_mode"))

            for stat_index, accuracy in enumerate(accuracies):
                rq3m_selector_epoch_rows.append({
                    **common_row,
                    "status": "ok",
                    "stat_index": int(stat_index),
                    "prbcd_epoch": int(stat_index - 1),
                    "is_clean_baseline": bool(stat_index == 0),
                    "accuracy": float(accuracy),
                    "accuracy_drop": clean_accuracy - float(accuracy),
                    "loss": (
                        float(losses[stat_index])
                        if stat_index < losses.size
                        else np.nan
                    ),
                    "selector_score_mean": (
                        selector_score_mean[stat_index]
                        if stat_index < len(selector_score_mean)
                        else np.nan
                    ),
                    "selector_score_max": (
                        selector_score_max[stat_index]
                        if stat_index < len(selector_score_max)
                        else np.nan
                    ),
                    "selector_score_num_tries": (
                        selector_num_tries[stat_index]
                        if stat_index < len(selector_num_tries)
                        else 0
                    ),
                    "selector_score_mode": (
                        selector_score_mode[stat_index]
                        if stat_index < len(selector_score_mode)
                        else None
                    ),
                })

            initial_set = _rq3m_id_set(block_info["initial_block"])
            high_set = _rq3m_id_set(group_high_ids)
            low_set = _rq3m_id_set(group_low_ids)
            for epoch_key, current_block in block_info["epoch_blocks"].items():
                current_set = _rq3m_id_set(current_block)
                rq3m_selector_retention_rows.append({
                    **common_row,
                    "prbcd_epoch": int(epoch_key),
                    "initial_retention": _rq3m_fraction(
                        len(current_set & initial_set),
                        len(initial_set),
                    ),
                    "group_high_fraction_in_current_block": _rq3m_fraction(
                        len(current_set & high_set),
                        len(current_set),
                    ),
                    "group_low_fraction_in_current_block": _rq3m_fraction(
                        len(current_set & low_set),
                        len(current_set),
                    ),
                })

            print(
                f"final_accuracy={final_accuracy:.6f} | "
                f"drop={clean_accuracy-final_accuracy:.6f} | "
                f"initial-high={composition['initial_high_fraction']:.3f} | "
                f"runtime={runtime_seconds:.1f}s"
            )

        except Exception as error:
            runtime_seconds = timer() - started
            error_text = "".join(
                traceback.format_exception_only(type(error), error)
            ).strip()
            rq3m_error_rows.append({
                "seed": victim_seed,
                "repeat": repeat,
                "candidate_config_id": plan_row["candidate_config_id"],
                "scoring_mode": plan_row["scoring_mode"],
                "endpoint_mining_hop": int(plan_row["endpoint_mining_hop"]),
                "condition": condition,
                "block_size": block_size,
                "attack_sampling_seed": attack_sampling_seed,
                "selector_lp_model_label": plan_row["selector_lp_model_label"],
                "error_type": type(error).__name__,
                "error_message": error_text,
                "runtime_seconds": runtime_seconds,
            })
            print("[ERROR]", error_text)
            if RQ3M_STOP_ON_ERROR:
                raise
        finally:
            result = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # Persist after each condition.
        pd.DataFrame(rq3m_selector_run_rows).to_csv(
            RQ3M_OUT_DIR / "selector_runs_raw.csv",
            index=False,
        )
        pd.DataFrame(rq3m_selector_epoch_rows).to_csv(
            RQ3M_OUT_DIR / "selector_epochs_raw.csv",
            index=False,
        )
        pd.DataFrame(rq3m_selector_retention_rows).to_csv(
            RQ3M_OUT_DIR / "selector_block_retention_raw.csv",
            index=False,
        )
        pd.DataFrame(rq3m_error_rows).to_csv(
            RQ3M_OUT_DIR / "errors.csv",
            index=False,
        )

rq3m_selector_runs_raw_df = pd.DataFrame(rq3m_selector_run_rows)
rq3m_selector_epochs_raw_df = pd.DataFrame(rq3m_selector_epoch_rows)
rq3m_selector_retention_raw_df = pd.DataFrame(rq3m_selector_retention_rows)
rq3m_errors_df = pd.DataFrame(rq3m_error_rows)

if rq3m_selector_runs_raw_df.empty:
    raise RuntimeError("No matched selector PRBCD runs completed.")

# The fixed and selector-resampling run must begin from exactly the same
# selector-generated block within each matched RQ2 unit.
hash_index = RQ3M_RQ2_UNIT_KEY_COLUMNS + [
    "block_size",
    "attack_sampling_seed",
    "lp_model_label",
]
initial_hash_check = (
    rq3m_selector_runs_raw_df
    .pivot_table(
        index=hash_index,
        columns="condition",
        values="initial_block_hash",
        aggfunc="first",
    )
    .reset_index()
)
required_hash_columns = {
    "selector__fixed_block",
    "selector__selector_resampling",
}
missing_hash_columns = required_hash_columns - set(initial_hash_check.columns)
if missing_hash_columns:
    raise RuntimeError(
        "Missing selector conditions in initial-block pairing check: "
        f"{sorted(missing_hash_columns)}"
    )

initial_hash_check["same_initial_block"] = (
    initial_hash_check["selector__fixed_block"]
    == initial_hash_check["selector__selector_resampling"]
)
if not initial_hash_check["same_initial_block"].all():
    display(initial_hash_check[~initial_hash_check["same_initial_block"]])
    raise AssertionError(
        "Fixed and selector-resampling runs did not begin from the same block."
    )

initial_hash_check.to_csv(
    RQ3M_OUT_DIR / "initial_block_pairing_check.csv",
    index=False,
)

print("Completed selector runs:", len(rq3m_selector_runs_raw_df))
print("Selector retention rows:", len(rq3m_selector_retention_raw_df))
print("Errors:", len(rq3m_errors_df))
print("All selector condition pairs use the same initial block.")
display(rq3m_selector_runs_raw_df)


In [ ]:
# ============================================================
# RQ3 MATCHED SELECTOR BENCHMARK
# CELL 4: combine the selector with every matched RQ2 condition
# ============================================================

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# Select exactly the RQ2 units and condition names in the plan
# ============================================================

rq3m_plan_unit_df = rq3m_plan_df[
    RQ3M_RQ2_UNIT_KEY_COLUMNS
    + [
        "block_size",
        "group_high_initialization",
        "group_low_initialization",
        "group_high_target",
        "group_low_target",
        "group_high_display",
        "group_low_display",
        "rq2_high_fixed_condition",
        "rq2_low_fixed_condition",
        "rq2_random_fixed_condition",
        "rq2_high_resampling_condition",
        "rq2_low_resampling_condition",
        "rq2_random_resampling_condition",
        "selector_lp_model_label",
        "selector_training_candidate_size",
        "selector_subgraph_seed",
        "selector_subgraph_fraction_requested",
        "selector_subgraph_fraction_actual",
        "selector_n_source_nodes",
        "selector_n_target_nodes",
    ]
].drop_duplicates()

expected_rq2_rows = []
for row in rq3m_plan_unit_df.to_dict(orient="records"):
    for condition_column in (
        "rq2_high_fixed_condition",
        "rq2_low_fixed_condition",
        "rq2_random_fixed_condition",
        "rq2_high_resampling_condition",
        "rq2_low_resampling_condition",
        "rq2_random_resampling_condition",
    ):
        expected_rq2_rows.append({
            **{column: row[column] for column in RQ3M_RQ2_UNIT_KEY_COLUMNS},
            "condition": row[condition_column],
        })

rq3m_expected_rq2_conditions_df = pd.DataFrame(expected_rq2_rows).drop_duplicates()

rq3m_rq2_runs_df = rq2_block_runs_raw_df.merge(
    rq3m_expected_rq2_conditions_df,
    on=RQ3M_RQ2_UNIT_KEY_COLUMNS + ["condition"],
    how="inner",
    validate="many_to_one",
).copy()

rq3m_rq2_epochs_df = rq2_block_epochs_raw_df.merge(
    rq3m_expected_rq2_conditions_df,
    on=RQ3M_RQ2_UNIT_KEY_COLUMNS + ["condition"],
    how="inner",
    validate="many_to_one",
).copy()

rq3m_rq2_retention_df = rq2_block_retention_raw_df.merge(
    rq3m_expected_rq2_conditions_df,
    on=RQ3M_RQ2_UNIT_KEY_COLUMNS + ["condition"],
    how="inner",
    validate="many_to_one",
).copy()

# Add display metadata to all RQ2 rows. Existing RQ2 columns are preserved.
plan_metadata_columns = RQ3M_RQ2_UNIT_KEY_COLUMNS + [
    "group_high_display",
    "group_low_display",
    "selector_lp_model_label",
    "selector_training_candidate_size",
    "selector_subgraph_seed",
    "selector_subgraph_fraction_requested",
    "selector_subgraph_fraction_actual",
    "selector_n_source_nodes",
    "selector_n_target_nodes",
]
plan_metadata_df = rq3m_plan_unit_df[plan_metadata_columns].drop_duplicates()

for frame_name in (
    "rq3m_rq2_runs_df",
    "rq3m_rq2_epochs_df",
    "rq3m_rq2_retention_df",
):
    frame = globals()[frame_name]
    frame = frame.merge(
        plan_metadata_df,
        on=RQ3M_RQ2_UNIT_KEY_COLUMNS,
        how="left",
        validate="many_to_one",
        suffixes=("", "_plan"),
    )
    globals()[frame_name] = frame

rq3m_rq2_runs_df["comparison_source"] = "RQ2 mined-label block"
rq3m_rq2_epochs_df["comparison_source"] = "RQ2 mined-label block"
rq3m_rq2_retention_df["comparison_source"] = "RQ2 mined-label block"

rq3m_selector_runs_raw_df["comparison_source"] = "RQ3 learned selector"
rq3m_selector_epochs_raw_df["comparison_source"] = "RQ3 learned selector"
rq3m_selector_retention_raw_df["comparison_source"] = "RQ3 learned selector"

rq3m_all_runs_df = pd.concat(
    [rq3m_rq2_runs_df, rq3m_selector_runs_raw_df],
    ignore_index=True,
    sort=False,
)
rq3m_all_epochs_df = pd.concat(
    [rq3m_rq2_epochs_df, rq3m_selector_epochs_raw_df],
    ignore_index=True,
    sort=False,
)
rq3m_all_retention_df = pd.concat(
    [rq3m_rq2_retention_df, rq3m_selector_retention_raw_df],
    ignore_index=True,
    sort=False,
)


# ============================================================
# Mechanical completeness and block-size checks
# ============================================================

block_size_check = rq3m_all_runs_df.groupby(
    RQ3M_RQ2_UNIT_KEY_COLUMNS,
    dropna=False,
)["block_size"].nunique()
if not (block_size_check == 1).all():
    display(block_size_check[block_size_check != 1])
    raise AssertionError("A matched comparison contains different block sizes.")

expected_condition_lookup = {}
for row in rq3m_plan_df.to_dict(orient="records"):
    key = _rq3m_rq2_unit_key(row)
    expected_condition_lookup[key] = {
        row["rq2_high_fixed_condition"],
        row["rq2_low_fixed_condition"],
        row["rq2_random_fixed_condition"],
        row["rq2_high_resampling_condition"],
        row["rq2_low_resampling_condition"],
        row["rq2_random_resampling_condition"],
        "selector__fixed_block",
        "selector__selector_resampling",
    }

condition_check = rq3m_all_runs_df.groupby(
    RQ3M_RQ2_UNIT_KEY_COLUMNS,
    dropna=False,
)["condition"].agg(lambda values: set(values.astype(str)))

incomplete_rows = []
for key, observed in condition_check.items():
    mapping = dict(zip(RQ3M_RQ2_UNIT_KEY_COLUMNS, key))
    expected = expected_condition_lookup.get(_rq3m_rq2_unit_key(mapping), set())
    missing = expected - observed
    unexpected = observed - expected
    if missing or unexpected:
        incomplete_rows.append({
            **mapping,
            "missing_conditions": sorted(missing),
            "unexpected_conditions": sorted(unexpected),
        })

rq3m_incomplete_df = pd.DataFrame(incomplete_rows)
if not rq3m_incomplete_df.empty:
    print("Warning: some matched RQ2-selector units are incomplete.")
    display(rq3m_incomplete_df)


# ============================================================
# Two-stage summaries: repeats within seed, then victim seeds
# ============================================================


def _rq3m_two_stage_summary(frame, group_columns, metric_columns):
    if frame.empty:
        return pd.DataFrame()

    metrics = [metric for metric in metric_columns if metric in frame.columns]
    work = frame.copy()
    for metric in metrics:
        work[metric] = pd.to_numeric(work[metric], errors="coerce")

    seed_level = (
        work.groupby(group_columns + ["seed"], dropna=False)[metrics]
        .mean()
        .reset_index()
    )
    summary = seed_level.groupby(group_columns, dropna=False)[metrics].agg(
        ["mean", "std", "count"]
    )
    summary.columns = [
        f"{metric}_{statistic}" for metric, statistic in summary.columns
    ]
    summary = summary.reset_index()

    for metric in metrics:
        summary[f"{metric}_std"] = summary[f"{metric}_std"].fillna(0.0)
        summary[f"{metric}_sem"] = summary[f"{metric}_std"] / np.sqrt(
            summary[f"{metric}_count"].clip(lower=1)
        )
    return summary


summary_group_columns = [
    "candidate_config_id",
    "scoring_mode",
    "endpoint_mining_hop",
    "comparison_rule",
    "condition",
    "initialization",
    "resampling_enabled",
    "comparison_source",
]

rq3m_all_runs_summary_df = _rq3m_two_stage_summary(
    rq3m_all_runs_df,
    summary_group_columns,
    [
        "block_size",
        "clean_accuracy",
        "final_accuracy",
        "final_accuracy_drop",
        "minimum_relaxed_accuracy",
        "maximum_relaxed_accuracy_drop",
        "integrated_accuracy_drop",
        "epoch_to_1pp_drop",
        "epoch_to_2pp_drop",
        "initial_high_fraction",
        "initial_low_fraction",
        "fraction_final_selected_from_initial",
        "fraction_final_selected_group_high",
        "fraction_final_selected_group_low",
        "final_block_initial_retention",
        "final_block_group_high_fraction",
        "final_block_group_low_fraction",
        "runtime_seconds",
    ],
)

rq3m_all_retention_summary_df = _rq3m_two_stage_summary(
    rq3m_all_retention_df,
    summary_group_columns + ["prbcd_epoch"],
    [
        "initial_retention",
        "group_high_fraction_in_current_block",
        "group_low_fraction_in_current_block",
    ],
)


# ============================================================
# Dynamic paired comparisons
# ============================================================

paired_index = RQ3M_RQ2_UNIT_KEY_COLUMNS + [
    "block_size",
    "group_high_initialization",
    "group_low_initialization",
    "group_high_display",
    "group_low_display",
]

rq3m_paired_wide_df = (
    rq3m_all_runs_df
    .pivot_table(
        index=paired_index,
        columns="condition",
        values="final_accuracy_drop",
        aggfunc="first",
    )
    .reset_index()
)


def _rq3m_row_value(row, condition):
    value = row.get(condition, np.nan)
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


paired_rows = []
for row in rq3m_paired_wide_df.to_dict(orient="records"):
    high_fixed = f"{row['group_high_initialization']}__fixed_block"
    low_fixed = f"{row['group_low_initialization']}__fixed_block"
    random_fixed = "random_full_space__fixed_block"
    high_resampling = f"{row['group_high_initialization']}__random_resampling"
    low_resampling = f"{row['group_low_initialization']}__random_resampling"
    random_resampling = "random_full_space__random_resampling"

    selector_fixed = _rq3m_row_value(row, "selector__fixed_block")
    selector_resampling = _rq3m_row_value(row, "selector__selector_resampling")
    high_fixed_value = _rq3m_row_value(row, high_fixed)
    low_fixed_value = _rq3m_row_value(row, low_fixed)
    random_fixed_value = _rq3m_row_value(row, random_fixed)
    high_resampling_value = _rq3m_row_value(row, high_resampling)
    low_resampling_value = _rq3m_row_value(row, low_resampling)
    random_resampling_value = _rq3m_row_value(row, random_resampling)

    fixed_denominator = high_fixed_value - random_fixed_value
    fixed_numerator = selector_fixed - random_fixed_value
    resampling_denominator = high_resampling_value - random_resampling_value
    resampling_numerator = selector_resampling - random_resampling_value

    row.update({
        "selector_fixed_minus_high_fixed": selector_fixed - high_fixed_value,
        "selector_fixed_minus_low_fixed": selector_fixed - low_fixed_value,
        "selector_fixed_minus_random_fixed": selector_fixed - random_fixed_value,
        "selector_resampling_minus_high_resampling": (
            selector_resampling - high_resampling_value
        ),
        "selector_resampling_minus_low_resampling": (
            selector_resampling - low_resampling_value
        ),
        "selector_resampling_minus_random_resampling": (
            selector_resampling - random_resampling_value
        ),
        "selector_resampling_minus_selector_fixed": (
            selector_resampling - selector_fixed
        ),
        "fixed_high_advantage_recovered": (
            fixed_numerator / fixed_denominator
            if np.isfinite(fixed_denominator) and fixed_denominator > 0
            else np.nan
        ),
        "resampling_high_advantage_recovered": (
            resampling_numerator / resampling_denominator
            if np.isfinite(resampling_denominator) and resampling_denominator > 0
            else np.nan
        ),
    })
    paired_rows.append(row)

rq3m_paired_wide_df = pd.DataFrame(paired_rows)


# ============================================================
# Save all comparison data
# ============================================================

rq3m_all_runs_df.to_csv(RQ3M_OUT_DIR / "all_conditions_runs_raw.csv", index=False)
rq3m_all_epochs_df.to_csv(RQ3M_OUT_DIR / "all_conditions_epochs_raw.csv", index=False)
rq3m_all_retention_df.to_csv(
    RQ3M_OUT_DIR / "all_conditions_block_retention_raw.csv",
    index=False,
)
rq3m_all_runs_summary_df.to_csv(
    RQ3M_OUT_DIR / "all_conditions_runs_summary.csv",
    index=False,
)
rq3m_all_retention_summary_df.to_csv(
    RQ3M_OUT_DIR / "all_conditions_block_retention_summary.csv",
    index=False,
)
rq3m_paired_wide_df.to_csv(
    RQ3M_OUT_DIR / "paired_selector_comparisons.csv",
    index=False,
)
rq3m_incomplete_df.to_csv(
    RQ3M_OUT_DIR / "incomplete_comparisons.csv",
    index=False,
)

print("Combined run rows:", len(rq3m_all_runs_df))
print("Combined epoch rows:", len(rq3m_all_epochs_df))
print("Combined retention rows:", len(rq3m_all_retention_df))
print("Conditions:", sorted(rq3m_all_runs_df["condition"].astype(str).unique()))
display(rq3m_all_runs_summary_df)
display(rq3m_paired_wide_df)


In [ ]:
# ============================================================
# RQ3 MATCHED SELECTOR BENCHMARK
# CELL 5: comprehensive RQ2-versus-selector plots
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re


RQ3M_PLOT_CONFIG_COLUMNS = [
    "candidate_config_id",
    "scoring_mode",
    "endpoint_mining_hop",
    "comparison_rule",
]


def _rq3m_safe(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))


def _rq3m_condition_layout(frame):
    row = frame.iloc[0]
    high_initialization = str(row["group_high_initialization"])
    low_initialization = str(row["group_low_initialization"])
    high_display = str(row.get("group_high_display", row.get("group_high_target", "high")))
    low_display = str(row.get("group_low_display", row.get("group_low_target", "low")))

    fixed_order = [
        f"{high_initialization}__fixed_block",
        "selector__fixed_block",
        f"{low_initialization}__fixed_block",
        "random_full_space__fixed_block",
    ]
    resampling_order = [
        f"{high_initialization}__random_resampling",
        "selector__selector_resampling",
        f"{low_initialization}__random_resampling",
        "random_full_space__random_resampling",
    ]
    labels = {
        fixed_order[0]: f"{high_display}\nfixed",
        fixed_order[1]: "Selector\nfixed",
        fixed_order[2]: f"{low_display}\nfixed",
        fixed_order[3]: "Random\nfixed",
        resampling_order[0]: f"{high_display} init\nrandom refill",
        resampling_order[1]: "Selector init\nselector refill",
        resampling_order[2]: f"{low_display} init\nrandom refill",
        resampling_order[3]: "Random init\nrandom refill",
    }
    return fixed_order, resampling_order, labels


def _rq3m_selector_description(frame):
    selector_rows = frame[frame["comparison_source"] == "RQ3 learned selector"]
    if selector_rows.empty:
        return "selector metadata unavailable"
    row = selector_rows.iloc[0]
    return (
        f"selector source={float(row['selector_subgraph_fraction_requested']):g} "
        f"({int(row['selector_n_source_nodes'])} nodes), "
        f"train candidates={int(row['selector_training_candidate_size'])}, "
        f"subgraph seed={int(row['selector_subgraph_seed'])}"
    )


def _rq3m_curve_summary(frame):
    seed_level = (
        frame.groupby(["condition", "prbcd_epoch", "seed"], dropna=False)["accuracy"]
        .mean()
        .reset_index()
    )
    summary = (
        seed_level.groupby(["condition", "prbcd_epoch"], dropna=False)["accuracy"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    summary["std"] = summary["std"].fillna(0.0)
    return summary


def _rq3m_retention_curve_summary(frame):
    metrics = [
        "initial_retention",
        "group_high_fraction_in_current_block",
        "group_low_fraction_in_current_block",
    ]
    work = frame.copy()
    for metric in metrics:
        work[metric] = pd.to_numeric(work[metric], errors="coerce")
    seed_level = (
        work.groupby(["condition", "prbcd_epoch", "seed"], dropna=False)[metrics]
        .mean()
        .reset_index()
    )
    rows = []
    for keys, part in seed_level.groupby(["condition", "prbcd_epoch"], dropna=False):
        condition, epoch = keys
        row = {"condition": condition, "prbcd_epoch": epoch}
        for metric in metrics:
            row[f"{metric}_mean"] = part[metric].mean()
            row[f"{metric}_std"] = part[metric].std(ddof=1) if len(part) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)


for config_key, run_frame in rq3m_all_runs_df.groupby(
    RQ3M_PLOT_CONFIG_COLUMNS,
    dropna=False,
):
    config_mapping = dict(zip(RQ3M_PLOT_CONFIG_COLUMNS, config_key))
    mask = np.ones(len(rq3m_all_epochs_df), dtype=bool)
    retention_mask = np.ones(len(rq3m_all_retention_df), dtype=bool)
    for column, value in config_mapping.items():
        if pd.isna(value):
            mask &= rq3m_all_epochs_df[column].isna().to_numpy()
            retention_mask &= rq3m_all_retention_df[column].isna().to_numpy()
        else:
            mask &= (rq3m_all_epochs_df[column] == value).to_numpy()
            retention_mask &= (rq3m_all_retention_df[column] == value).to_numpy()

    epoch_frame = rq3m_all_epochs_df.loc[mask].copy()
    retention_frame = rq3m_all_retention_df.loc[retention_mask].copy()

    fixed_order, resampling_order, condition_labels = _rq3m_condition_layout(run_frame)
    full_order = fixed_order + resampling_order
    safe_config = _rq3m_safe("__".join(map(str, config_key)))
    block_sizes = sorted(run_frame["block_size"].dropna().astype(int).unique())
    block_size_text = str(block_sizes[0]) if len(block_sizes) == 1 else str(block_sizes)
    selector_description = _rq3m_selector_description(run_frame)
    config_title = (
        f"{config_mapping['candidate_config_id']} | "
        f"mode={config_mapping['scoring_mode']} | "
        f"h={config_mapping['endpoint_mining_hop']} | B={block_size_text}\n"
        f"{selector_description}"
    )

    # --------------------------------------------------------
    # 1. Final accuracy reduction from each seed-specific baseline
    # --------------------------------------------------------
    figure, axes = plt.subplots(1, 2, figsize=(17, 6))

    for axis, conditions, panel_title in (
        (axes[0], fixed_order, "Fixed candidate blocks"),
        (axes[1], resampling_order, "Evolving candidate blocks"),
    ):
        drop_values = [
            run_frame.loc[
                run_frame["condition"] == condition,
                "final_accuracy_drop",
            ]
            .dropna()
            .to_numpy(float)
            for condition in conditions
        ]
        panel_labels = [
            condition_labels[condition]
            for condition in conditions
        ]

        axis.boxplot(
            drop_values,
            showmeans=True,
            labels=panel_labels,
        )
        axis.axhline(0, linestyle="--", linewidth=1)
        axis.set_ylabel(
            r"Final accuracy reduction $A_{clean}-A_{final}$"
        )
        axis.set_title(panel_title)
        axis.grid(True, axis="y", alpha=0.3)
        axis.tick_params(axis="x", rotation=25)

    figure.suptitle(
        "Final accuracy reduction by candidate-block strategy"
    )
    figure.tight_layout()
    figure.savefig(
        RUN_PLOTS_DIR / f"final_results__{safe_config}.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

    # --------------------------------------------------------
    # 2. Complete PRBCD accuracy trajectories
    # --------------------------------------------------------
    curve_summary = _rq3m_curve_summary(epoch_frame)
    figure, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    for axis, conditions, title in (
        (axes[0], fixed_order, "Fixed candidate blocks"),
        (axes[1], resampling_order, "Evolving candidate blocks"),
    ):
        for condition in conditions:
            part = curve_summary[curve_summary["condition"] == condition].sort_values(
                "prbcd_epoch"
            )
            if part.empty:
                continue
            x = part["prbcd_epoch"].to_numpy(float)
            y = part["mean"].to_numpy(float)
            std = part["std"].to_numpy(float)
            axis.plot(x, y, label=condition_labels[condition].replace("\n", " "))
            axis.fill_between(x, y - std, y + std, alpha=0.12)
        axis.axvline(
            int(RQ2_BLOCK_RESAMPLING_EPOCHS) - 1,
            linestyle="--",
            linewidth=1,
            label="fine-tuning boundary",
        )
        axis.set_xlabel("PRBCD epoch (-1 = clean baseline)")
        axis.set_title(title)
        axis.grid(True, alpha=0.3)
        axis.legend(fontsize=8)
    axes[0].set_ylabel("Victim test accuracy")
    figure.suptitle("Optimization trajectories")
    figure.tight_layout()
    figure.savefig(
        RUN_PLOTS_DIR / f"accuracy_trajectories__{safe_config}.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

    # --------------------------------------------------------
    # 3. Attack efficiency: integrated drop and epochs to drop
    # --------------------------------------------------------
    figure, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    metrics = [
        ("integrated_accuracy_drop", "Integrated accuracy reduction"),
        ("epoch_to_1pp_drop", "Epoch to 1 percentage-point drop"),
        ("epoch_to_2pp_drop", "Epoch to 2 percentage-point drop"),
    ]
    for axis, (metric, title) in zip(axes, metrics):
        values = [
            pd.to_numeric(
                run_frame.loc[run_frame["condition"] == condition, metric],
                errors="coerce",
            ).dropna().to_numpy(float)
            for condition in full_order
        ]
        axis.boxplot(values, showmeans=True, labels=plot_labels)
        axis.set_title(title)
        axis.grid(True, axis="y", alpha=0.3)
        axis.tick_params(axis="x", rotation=25)
    figure.suptitle("Attack speed and optimization benefit")
    figure.tight_layout()
    figure.savefig(
        RUN_PLOTS_DIR / f"attack_efficiency__{safe_config}.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

    # --------------------------------------------------------
    # 4. Evolution of resampled candidate blocks
    # --------------------------------------------------------
    if not retention_frame.empty:
        retention_summary = _rq3m_retention_curve_summary(retention_frame)
        figure, axes = plt.subplots(1, 3, figsize=(18, 5.2))
        retention_metrics = [
            ("initial_retention", "Fraction of initial block retained"),
            (
                "group_high_fraction_in_current_block",
                f"{run_frame.iloc[0]['group_high_display']} fraction in block",
            ),
            (
                "group_low_fraction_in_current_block",
                f"{run_frame.iloc[0]['group_low_display']} fraction in block",
            ),
        ]
        for condition in resampling_order:
            part = retention_summary[
                retention_summary["condition"] == condition
            ].sort_values("prbcd_epoch")
            if part.empty:
                continue
            x = part["prbcd_epoch"].to_numpy(float)
            for axis, (metric, title) in zip(axes, retention_metrics):
                y = part[f"{metric}_mean"].to_numpy(float)
                std = part[f"{metric}_std"].to_numpy(float)
                axis.plot(x, y, label=condition_labels[condition].replace("\n", " "))
                axis.fill_between(x, y - std, y + std, alpha=0.10)
                axis.set_title(title)
                axis.set_xlabel("PRBCD epoch")
                axis.set_ylim(-0.02, 1.02)
                axis.grid(True, alpha=0.3)
        for axis in axes:
            axis.legend(fontsize=8)
        figure.suptitle("Candidate-block evolution")
        figure.tight_layout()
        figure.savefig(
            RUN_PLOTS_DIR / f"block_evolution__{safe_config}.png",
            dpi=220,
            bbox_inches="tight",
        )
        plt.show()

    # --------------------------------------------------------
    # 5. Initial and final composition relative to RQ2 groups
    # --------------------------------------------------------
    figure, axes = plt.subplots(1, 2, figsize=(17, 5.5))
    composition_specs = [
        (
            axes[0],
            "initial_high_fraction",
            "initial_low_fraction",
            "Initial candidate-block composition",
        ),
        (
            axes[1],
            "fraction_final_selected_group_high",
            "fraction_final_selected_group_low",
            "Final discrete perturbation composition",
        ),
    ]
    x = np.arange(len(full_order))
    for axis, high_metric, low_metric, title in composition_specs:
        high_means = []
        low_means = []
        high_stds = []
        low_stds = []
        for condition in full_order:
            high_values = pd.to_numeric(
                run_frame.loc[run_frame["condition"] == condition, high_metric],
                errors="coerce",
            ).dropna().to_numpy(float)
            low_values = pd.to_numeric(
                run_frame.loc[run_frame["condition"] == condition, low_metric],
                errors="coerce",
            ).dropna().to_numpy(float)
            high_means.append(np.nanmean(high_values) if high_values.size else np.nan)
            low_means.append(np.nanmean(low_values) if low_values.size else np.nan)
            high_stds.append(np.nanstd(high_values, ddof=1) if high_values.size > 1 else 0.0)
            low_stds.append(np.nanstd(low_values, ddof=1) if low_values.size > 1 else 0.0)
        axis.errorbar(
            x - 0.08,
            high_means,
            yerr=high_stds,
            marker="o",
            linestyle="none",
            capsize=4,
            label=str(run_frame.iloc[0]["group_high_display"]),
        )
        axis.errorbar(
            x + 0.08,
            low_means,
            yerr=low_stds,
            marker="s",
            linestyle="none",
            capsize=4,
            label=str(run_frame.iloc[0]["group_low_display"]),
        )
        axis.set_xticks(x)
        axis.set_xticklabels(plot_labels, rotation=25, ha="right")
        axis.set_ylim(-0.02, 1.02)
        axis.set_ylabel("Fraction")
        axis.set_title(title)
        axis.grid(True, axis="y", alpha=0.3)
        axis.legend()
    figure.suptitle("RQ2-label composition of selector-informed PRBCD")
    figure.tight_layout()
    figure.savefig(
        RUN_PLOTS_DIR / f"block_composition__{safe_config}.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()

    # --------------------------------------------------------
    # 6. Paired selector advantages for this RQ2 configuration
    # --------------------------------------------------------
    paired_mask = np.ones(len(rq3m_paired_wide_df), dtype=bool)
    for column, value in config_mapping.items():
        if pd.isna(value):
            paired_mask &= rq3m_paired_wide_df[column].isna().to_numpy()
        else:
            paired_mask &= (rq3m_paired_wide_df[column] == value).to_numpy()
    paired_frame = rq3m_paired_wide_df.loc[paired_mask].copy()

    paired_metrics = {
        "Selector fixed − high fixed": "selector_fixed_minus_high_fixed",
        "Selector fixed − low fixed": "selector_fixed_minus_low_fixed",
        "Selector fixed − random fixed": "selector_fixed_minus_random_fixed",
        "Selector refill − high refill": "selector_resampling_minus_high_resampling",
        "Selector refill − low refill": "selector_resampling_minus_low_resampling",
        "Selector refill − random refill": "selector_resampling_minus_random_resampling",
        "Selector refill − selector fixed": "selector_resampling_minus_selector_fixed",
    }
    values = [
        pd.to_numeric(paired_frame[column], errors="coerce").dropna().to_numpy(float)
        for column in paired_metrics.values()
    ]
    figure, axis = plt.subplots(figsize=(12, 6))
    axis.boxplot(values, showmeans=True, labels=list(paired_metrics.keys()), vert=False)
    axis.axvline(0, linestyle="--", linewidth=1)
    axis.set_xlabel(
        "Difference in final accuracy reduction\n"
        "(positive = selector condition is stronger)"
    )
    axis.set_title("Paired selector advantages")
    axis.grid(True, axis="x", alpha=0.3)
    figure.tight_layout()
    figure.savefig(
        RUN_PLOTS_DIR / f"paired_selector_advantages__{safe_config}.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()


# ============================================================
# Cross-configuration recovery summary
# ============================================================

recovery_columns = [
    "fixed_high_advantage_recovered",
    "resampling_high_advantage_recovered",
]
recovery_long = rq3m_paired_wide_df.melt(
    id_vars=RQ3M_PLOT_CONFIG_COLUMNS,
    value_vars=recovery_columns,
    var_name="recovery_type",
    value_name="advantage_recovered",
)
recovery_long.to_csv(
    RQ3M_OUT_DIR / "selector_advantage_recovery_long.csv",
    index=False,
)

print("All comparison tables and plots saved to:", RQ3M_OUT_DIR.resolve())


## 8. Model Guided PR-BCD

In [ ]:
# %% Cell 1: define the sweep
from copy import deepcopy
from itertools import product

# Values in these dictionaries are combined with a Cartesian product.
EXPERIMENT_SWEEP = {
    "attack": ["PRBCD"],
    "semi": [True],
    "epsilon": [0.05, 0.1, 0.20],
    "use_cert": ["accuracy_drop_selector_with_resampling", "none"],
    "seed": list(SEEDS),
}

ATTACK_SWEEP = {
    "block_size": [5000, 10000, 20000],
    "epochs": [150],
    "fine_tune_epochs": [100],
    "keep_heuristic": ["WeightOnly"],
    "do_synchronize": [True],
    "loss_type": ["tanhMargin"],
}

SELECTOR_SWEEP = {
    "accuracy_drop_selector_mode": ["one_sample"],
    "n_candidates_k_sample": [2_000],
    "n_candidates_one_sample": [5_000],
    "drop_mode": ["endpoint"],
    "acc_drop_threshold_k_samples": [1e-3],
    "loss_drop_threshold_k_samples": [1e-3],
    "k_samples_batch": [10],
    "training_data_node_cap": [15],
    "tau": [0.7],
    "score_batch_size": [1_000],
    "max_sampling_tries": [2_000_000],
    "exclude_tried": [True],
    "resampling_mode": ["topk"],  # "threshold", "topk"
}

# Passed to every selector run, but not expanded as sweep dimensions.
SELECTOR_FIXED = {
    "lp_hit_rate_detour": False,
    "lp_hit_rate_top_k": 200,
    "lp_hit_rate_out_dir": "extendedPlotting/lpEndpointHitRate",

    # Default for all selector-guided resampling configurations.
    # Overridden to 300 only for Citeseer with B=10,000.
    "top_k_per_batch": 100,
}

# Temporary filter:
# Do not include these LP/mining variants in sweep_configs at all.
EXCLUDED_SCORING_MODES = {
    "subset_accuracy_drop",
}

EXCLUDED_LABEL_MODES = {
    "subset_accuracy_drop",
}

# Keep Python model objects out of the CSV. §8 deliberately consumes the
# complete RQ3 subgraph selector sweep, while earlier cells keep using the
# primary-only `LP_MODEL_VARIANTS` compatibility registry.
if (
    "MODEL_GUIDED_PRBCD_LP_MODEL_VARIANTS" in globals()
    and MODEL_GUIDED_PRBCD_LP_MODEL_VARIANTS
):
    MODEL_GUIDED_LP_MODEL_VARIANTS = (
        MODEL_GUIDED_PRBCD_LP_MODEL_VARIANTS
    )
elif (
    "RQ3_SUBGRAPH_LP_MODEL_VARIANTS" in globals()
    and RQ3_SUBGRAPH_LP_MODEL_VARIANTS
):
    MODEL_GUIDED_LP_MODEL_VARIANTS = (
        RQ3_SUBGRAPH_LP_MODEL_VARIANTS
    )
elif "lp_model_variants" in globals() and lp_model_variants:
    MODEL_GUIDED_LP_MODEL_VARIANTS = lp_model_variants
else:
    MODEL_GUIDED_LP_MODEL_VARIANTS = {
        "default": {
            "model": lp_model,
            "seed": globals().get("SEED", 0),
            "candidate_config_id": "default",
            "candidate_set_size": globals().get("n_cands", ""),
            "training_candidate_size": globals().get("n_cands", ""),
            "subgraph_seed": "",
            "subgraph_fraction_requested": "",
            "subgraph_fraction_actual": "",
            "n_source_nodes": "",
            "n_target_nodes": "",
            "prbcd_candidate_fraction": globals().get(
                "PRBCD_CANDIDATE_FRACTION",
                "",
            ),
            "actual_prbcd_fraction": globals().get(
                "actual_prbcd_fraction",
                "",
            ),
            "scoring_mode": globals().get("SCORING_MODE", ""),
            "endpoint_mining_hop": globals().get(
                "ENDPOINT_MINING_HOP",
                0,
            ),
            "n_prbcd_requested": "",
            "n_prbcd_mined_unique": "",
            "n_prbcd_selected": "",
            "n_random_candidates": "",
            "n_positive_labels": "",
            "label_mode": globals().get("LABEL_MODE", ""),
            "extreme_fraction": globals().get(
                "EXTREME_FRACTION",
                "",
            ),
        }
    }

ADS_CERTS = {
    "accuracy_drop_selector",
    "accuracy_drop_selector_with_resampling",
    "direct_selector",
}

RESAMPLING_CERT = "accuracy_drop_selector_with_resampling"

ADS_ONLY_SELECTOR_KEYS = {
    "accuracy_drop_selector_mode",
    "n_candidates_k_sample",
    "n_candidates_one_sample",
    "drop_mode",
    "acc_drop_threshold_k_samples",
    "loss_drop_threshold_k_samples",
    "k_samples_batch",
    "training_data_node_cap",
    "resampling_mode",
    "top_k_per_batch",
}


def expand_grid(grid):
    """Yield one dictionary for every Cartesian-product combination."""
    keys = list(grid)

    for values in product(*(grid[key] for key in keys)):
        yield dict(zip(keys, values))


def _normalize_dataset_name(value):
    """Normalize common spellings such as cite-seer and cite_seer."""
    return (
        str(value)
        .strip()
        .lower()
        .replace("-", "")
        .replace("_", "")
        .replace(" ", "")
    )


def _split_lp_model_variant(lp_model_variant):
    if isinstance(lp_model_variant, dict) and "model" in lp_model_variant:
        lp_model_object = lp_model_variant["model"]
        lp_model_metadata = {
            key: value
            for key, value in lp_model_variant.items()
            if key != "model"
        }
    else:
        lp_model_object = lp_model_variant
        lp_model_metadata = {}

    return lp_model_object, lp_model_metadata


def _is_excluded_lp_variant(lp_model_metadata):
    scoring_mode = str(
        lp_model_metadata.get("scoring_mode", "")
    ).strip()

    label_mode = str(
        lp_model_metadata.get("label_mode", "")
    ).strip()

    return (
        scoring_mode in EXCLUDED_SCORING_MODES
        or label_mode in EXCLUDED_LABEL_MODES
    )


def build_sweep_configs():
    experiment_configs = list(expand_grid(EXPERIMENT_SWEEP))
    attack_configs = list(expand_grid(ATTACK_SWEEP))
    selector_configs = list(expand_grid(SELECTOR_SWEEP))

    dataset_key = _normalize_dataset_name(
        globals().get("DATASET", "")
    )

    configs = []
    skipped_configs = []

    for experiment_cfg in experiment_configs:
        use_cert = experiment_cfg["use_cert"]

        for attack_cfg in attack_configs:
            for selector_cfg_full in selector_configs:
                # resampling_mode only applies to the resampling attack.
                # Without this, "none" and other methods are duplicated once
                # for every configured resampling mode.
                if (
                    use_cert != RESAMPLING_CERT
                    and selector_cfg_full.get("resampling_mode")
                    != SELECTOR_SWEEP["resampling_mode"][0]
                ):
                    continue

                selector_cfg = deepcopy(selector_cfg_full)
                selector_cfg.update(deepcopy(SELECTOR_FIXED))

                # Use k=300 only for selector-guided Citeseer attacks
                # with block size B=10,000. All other selector-guided
                # resampling configurations retain the default k=100.
                if use_cert == RESAMPLING_CERT:
                    is_citeseer = dataset_key == "citeseer"
                    is_block_10000 = (
                        int(attack_cfg["block_size"]) == 10_000
                    )

                    selector_cfg["top_k_per_batch"] = (
                        300
                        if is_citeseer and is_block_10000
                        else 100
                    )

                # These parameters are relevant only for the selector-guided
                # resampling condition.
                if use_cert != RESAMPLING_CERT:
                    selector_cfg.pop(
                        "resampling_mode",
                        None,
                    )
                    selector_cfg.pop(
                        "top_k_per_batch",
                        None,
                    )

                # Do not pass ADS-only parameters to unrelated attack modes.
                if use_cert not in ADS_CERTS:
                    selector_cfg = {
                        key: value
                        for key, value in selector_cfg.items()
                        if key not in ADS_ONLY_SELECTOR_KEYS
                    }

                matching_variants = []

                for lp_model_label, lp_model_variant in (
                    MODEL_GUIDED_LP_MODEL_VARIANTS.items()
                ):
                    lp_model_object, lp_model_metadata = (
                        _split_lp_model_variant(lp_model_variant)
                    )

                    variant_seed = lp_model_metadata.get(
                        "seed",
                        experiment_cfg.get("seed"),
                    )

                    if int(variant_seed) != int(
                        experiment_cfg.get("seed")
                    ):
                        continue

                    matching_variants.append(
                        (
                            lp_model_label,
                            lp_model_object,
                            lp_model_metadata,
                        )
                    )

                if not matching_variants:
                    continue

                # Selector-enabled PRBCD expands over every trained subgraph
                # selector. Non-selector baselines are run only once per seed.
                if use_cert not in ADS_CERTS:
                    matching_variants = [matching_variants[0]]

                for (
                    lp_model_label,
                    lp_model_object,
                    lp_model_metadata,
                ) in matching_variants:
                    # --------------------------------------------------------
                    # Filter unwanted LP/mining variants at sweep-definition
                    # time. These configs are not added to sweep_configs.
                    # --------------------------------------------------------
                    if (
                        use_cert in ADS_CERTS
                        and _is_excluded_lp_variant(lp_model_metadata)
                    ):
                        skipped_configs.append(
                            {
                                "use_cert": use_cert,
                                "lp_model_label": lp_model_label,
                                "candidate_config_id": (
                                    lp_model_metadata.get(
                                        "candidate_config_id",
                                        "",
                                    )
                                ),
                                "scoring_mode": (
                                    lp_model_metadata.get(
                                        "scoring_mode",
                                        "",
                                    )
                                ),
                                "endpoint_mining_hop": (
                                    lp_model_metadata.get(
                                        "endpoint_mining_hop",
                                        0,
                                    )
                                ),
                                "label_mode": (
                                    lp_model_metadata.get(
                                        "label_mode",
                                        "",
                                    )
                                ),
                                "extreme_fraction": (
                                    lp_model_metadata.get(
                                        "extreme_fraction",
                                        "",
                                    )
                                ),
                                "candidate_set_size": (
                                    lp_model_metadata.get(
                                        "candidate_set_size",
                                        "",
                                    )
                                ),
                                "prbcd_candidate_fraction": (
                                    lp_model_metadata.get(
                                        "prbcd_candidate_fraction",
                                        "",
                                    )
                                ),
                            }
                        )
                        continue

                    log_label = lp_model_label
                    log_metadata = deepcopy(lp_model_metadata)

                    if use_cert not in ADS_CERTS:
                        log_label = "no_selector_baseline"
                        log_metadata = {
                            "seed": int(experiment_cfg["seed"]),
                            "candidate_config_id": "no_selector",
                            "candidate_set_size": "",
                            "training_candidate_size": "",
                            "subgraph_seed": "",
                            "subgraph_fraction_requested": "",
                            "subgraph_fraction_actual": "",
                            "n_source_nodes": "",
                            "n_target_nodes": "",
                            "selector_training_scope": (
                                "not_applicable"
                            ),
                            "prbcd_candidate_fraction": "",
                            "actual_prbcd_fraction": "",
                            "scoring_mode": "",
                            "endpoint_mining_hop": "",
                            "n_prbcd_requested": "",
                            "n_prbcd_mined_unique": "",
                            "n_prbcd_selected": "",
                            "n_random_candidates": "",
                            "n_positive_labels": "",
                            "label_mode": "",
                            "extreme_fraction": "",
                        }

                    configs.append(
                        {
                            "experiment": deepcopy(experiment_cfg),
                            "attack_params": {
                                **deepcopy(attack_cfg),
                                "lp_model": lp_model_object,
                            },
                            "selector_params": deepcopy(
                                selector_cfg
                            ),
                            "log_values": {
                                "lp_model_label": log_label,
                                **log_metadata,
                            },
                        }
                    )

    print(
        "Skipped configurations during sweep construction: "
        f"{len(skipped_configs)}"
    )

    if skipped_configs:
        print("Skipped variants:")

        for index, skipped in enumerate(
            skipped_configs,
            start=1,
        ):
            print(
                f"  {index:03d}: "
                f"cert={skipped['use_cert']}, "
                f"lp_model={skipped['lp_model_label']}, "
                f"candidate_config="
                f"{skipped.get('candidate_config_id', '-')}, "
                f"scoring_mode="
                f"{skipped.get('scoring_mode', '-')}, "
                f"h={skipped.get('endpoint_mining_hop', 0)}, "
                f"label_mode="
                f"{skipped.get('label_mode', '-')}, "
                f"extreme_fraction="
                f"{skipped.get('extreme_fraction', '-')}, "
                f"candidate_set_size="
                f"{skipped.get('candidate_set_size', '-')}, "
                f"prbcd_fraction="
                f"{skipped.get('prbcd_candidate_fraction', '-')}"
            )

    return configs


EXPERIMENT_COLUMNS = list(EXPERIMENT_SWEEP)
ATTACK_COLUMNS = list(ATTACK_SWEEP)

SELECTOR_COLUMNS = list(SELECTOR_SWEEP) + [
    key
    for key in SELECTOR_FIXED
    if key not in SELECTOR_SWEEP
]

LOG_COLUMNS = [
    "lp_model_label",
    "lp_model_group_label",
    "seed",
    "candidate_config_id",
    "candidate_set_size",
    "training_candidate_size",
    "subgraph_seed",
    "subgraph_fraction_requested",
    "subgraph_fraction_actual",
    "n_source_nodes",
    "n_target_nodes",
    "selector_training_scope",
    "prbcd_candidate_fraction",
    "actual_prbcd_fraction",
    "scoring_mode",
    "endpoint_mining_hop",
    "n_prbcd_requested",
    "n_prbcd_mined_unique",
    "n_prbcd_selected",
    "n_random_candidates",
    "n_positive_labels",
    "label_mode",
    "extreme_fraction",
]

ALL_CONFIG_COLUMNS = (
    [
        f"experiment__{key}"
        for key in EXPERIMENT_COLUMNS
    ]
    + [
        f"attack__{key}"
        for key in ATTACK_COLUMNS
    ]
    + [
        f"selector__{key}"
        for key in SELECTOR_COLUMNS
    ]
    + [
        f"log__{key}"
        for key in LOG_COLUMNS
    ]
)


def config_to_row(cfg):
    row = {}

    for key in EXPERIMENT_COLUMNS:
        row[f"experiment__{key}"] = (
            cfg["experiment"].get(key, "")
        )

    for key in ATTACK_COLUMNS:
        row[f"attack__{key}"] = (
            cfg["attack_params"].get(key, "")
        )

    for key in SELECTOR_COLUMNS:
        row[f"selector__{key}"] = (
            cfg["selector_params"].get(key, "")
        )

    for key in LOG_COLUMNS:
        row[f"log__{key}"] = (
            cfg["log_values"].get(key, "")
        )

    return row


sweep_configs = build_sweep_configs()
print(f"Number of configurations: {len(sweep_configs)}")

# Strictly verify the conditional top-k assignment before starting the runs.
dataset_key = _normalize_dataset_name(
    globals().get("DATASET", "")
)

for cfg in sweep_configs:
    use_cert = cfg["experiment"]["use_cert"]
    block_size = int(cfg["attack_params"]["block_size"])
    selector = cfg["selector_params"]

    if use_cert != RESAMPLING_CERT:
        assert "top_k_per_batch" not in selector
        continue

    expected_top_k = (
        300
        if dataset_key == "citeseer"
        and block_size == 10_000
        else 100
    )

    actual_top_k = int(
        selector["top_k_per_batch"]
    )

    assert actual_top_k == expected_top_k, (
        "Unexpected top_k_per_batch: "
        f"dataset={globals().get('DATASET', '')}, "
        f"block_size={block_size}, "
        f"expected={expected_top_k}, "
        f"actual={actual_top_k}"
    )

print("Conditional top-k configuration verified.")

for index, cfg in enumerate(
    sweep_configs,
    start=1,
):
    exp = cfg["experiment"]
    attack_params = cfg["attack_params"]
    selector = cfg["selector_params"]
    log_values = cfg["log_values"]

    print(
        f"{index:03d}: "
        f"attack={exp['attack']}, "
        f"cert={exp['use_cert']}, "
        f"epsilon={exp['epsilon']}, "
        f"seed={exp['seed']}, "
        f"block_size={attack_params['block_size']}, "
        f"candidate_config="
        f"{log_values.get('candidate_config_id', '-')}, "
        f"scoring_mode="
        f"{log_values.get('scoring_mode', '-')}, "
        f"label_mode="
        f"{log_values.get('label_mode', '-')}, "
        f"extreme_fraction="
        f"{log_values.get('extreme_fraction', '-')}, "
        f"candidate_set_size="
        f"{log_values.get('candidate_set_size', '-')}, "
        f"subgraph_fraction="
        f"{log_values.get('subgraph_fraction_requested', '-')}, "
        f"subgraph_seed="
        f"{log_values.get('subgraph_seed', '-')}, "
        f"prbcd_fraction="
        f"{log_values.get('prbcd_candidate_fraction', '-')}, "
        f"drop_mode={selector.get('drop_mode', '-')}, "
        f"tau={selector.get('tau', '-')}, "
        f"resampling_mode="
        f"{selector.get('resampling_mode', '-')}, "
        f"top_k_per_batch="
        f"{selector.get('top_k_per_batch', '-')}"
    )

In [ ]:
# ============================================================
# Verify which selector object is stored in every sweep config
# ============================================================

selector_configs_checked = 0

for config_index, cfg in enumerate(sweep_configs, start=1):
    experiment_seed = int(cfg["experiment"]["seed"])
    use_cert = str(cfg["experiment"]["use_cert"])

    if use_cert not in ADS_CERTS:
        continue

    selector_label = cfg["log_values"]["lp_model_label"]
    passed_model = cfg["attack_params"]["lp_model"]

    registry_variant = MODEL_GUIDED_LP_MODEL_VARIANTS[
        selector_label
    ]
    expected_model, metadata = _split_lp_model_variant(
        registry_variant
    )
    selector_seed = int(metadata["seed"])

    print(
        f"config={config_index:03d}",
        f"experiment_seed={experiment_seed}",
        f"selector_seed={selector_seed}",
        f"is_global_lp_model={passed_model is lp_model}",
        f"is_expected_registry_model={passed_model is expected_model}",
        f"label={selector_label}",
    )

    assert selector_seed == experiment_seed, (
        f"Seed mismatch in config {config_index}: "
        f"experiment={experiment_seed}, selector={selector_seed}"
    )

    assert passed_model is expected_model, (
        f"Wrong selector object in config {config_index}"
    )

    if experiment_seed != 0:
        assert passed_model is not lp_model, (
            f"Seed-{experiment_seed} config is using the global "
            "first-model alias!"
        )

    selector_configs_checked += 1

print(
    f"Successfully checked {selector_configs_checked} "
    "selector-enabled sweep configurations."
)

In [ ]:
from IPython.core.display_functions import clear_output
# %% Cell 2: run the sweep and append CSV logs
from DeleteCache import delete_pert_files_and_folders
from timeit import default_timer as timer
from datetime import datetime
import csv
import gc
import hashlib
import json
import os
import traceback

import numpy as np
import torch
from sparse_smoothing.utils import load_and_standardize

# -------------------------
# Behaviour
# -------------------------
STOP_ON_ERROR = False
SAVE_RAW_RESULTS = False

# Temporary skip settings
SKIP_LABEL_MODES = {
    "subset_accuracy_drop",
}

MODEL_STORAGE_TYPE = "demo_custom_split"
ARTIFACT_DIR = "cache"
PERT_ADJ_STORAGE_TYPE = "evasion_global_adj"
PERT_ATTR_STORAGE_TYPE = "evasion_global_attr"


def _folder_safe_value(value):
    """Convert a config value into a compact, filesystem-safe string."""
    if isinstance(value, torch.Tensor):
        value = value.detach().cpu().item() if value.numel() == 1 else value.detach().cpu().tolist()
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, (list, tuple)):
        return "-".join(_folder_safe_value(item) for item in value)
    if value is None:
        return "na"

    text = f"{value:.12g}" if isinstance(value, float) else str(value)
    return (
        text.strip()
        .replace(" ", "")
        .replace("/", "-")
        .replace("\\", "-")
        .replace(".", "p")
        .replace("+", "")
        .replace("-", "m")
    )


def _read_nested_config(cfg, candidate_paths):
    """Return the first configured value found among the candidate paths."""
    for section_name, key in candidate_paths:
        section = cfg.get(section_name, {})
        if key in section and section[key] is not None:
            return section[key]
    return None


def _collect_sweep_values(candidate_paths):
    """Collect unique values in sweep order for one signature field."""
    values = []
    seen = set()

    for cfg in sweep_configs:
        value = _read_nested_config(cfg, candidate_paths)
        safe_value = _folder_safe_value(value)
        if safe_value not in seen:
            seen.add(safe_value)
            values.append(safe_value)

    return "+".join(values) if values else "na"


# -------------------------
# Safe output directory
# -------------------------
FULL_FOLDER_SIGNATURE = "__".join([
    "block_size-" + _collect_sweep_values([
        ("attack_params", "block_size"),
    ]),
    "candN-" + _collect_sweep_values([
        ("log_values", "candidate_set_size"),
    ]),
    "subgraphFrac-" + _collect_sweep_values([
        ("log_values", "subgraph_fraction_requested"),
    ]),
    "subgraphSeed-" + _collect_sweep_values([
        ("log_values", "subgraph_seed"),
    ]),
    "prbcdFrac-" + _collect_sweep_values([
        ("log_values", "prbcd_candidate_fraction"),
    ]),
    "scoringMode-" + _collect_sweep_values([
        ("log_values", "scoring_mode"),
    ]),
    "h-" + _collect_sweep_values([
        ("log_values", "endpoint_mining_hop"),
    ]),
    "labelMode-" + _collect_sweep_values([
        ("log_values", "label_mode"),
    ]),
    "extremeFrac-" + _collect_sweep_values([
        ("log_values", "extreme_fraction"),
    ]),
    "tau-" + _collect_sweep_values([
        ("selector_params", "tau"),
        ("attack_params", "tau"),
        ("experiment", "tau"),
    ]),
    "resamplingMode-" + _collect_sweep_values([
        ("selector_params", "resampling_mode"),
        ("attack_params", "resampling_mode"),
    ]),
    "epsilon-" + _collect_sweep_values([
        ("experiment", "epsilon"),
        ("attack_params", "epsilon"),
        ("attack_params", "eps"),
    ]),
    "epochs-" + _collect_sweep_values([
        ("attack_params", "epochs"),
    ]),
])

SIGNATURE_HASH = hashlib.sha1(
    FULL_FOLDER_SIGNATURE.encode("utf-8")
).hexdigest()[:10]

FOLDER_SIGNATURE = f"sig-{SIGNATURE_HASH}"
RUN_GROUP_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

OUT_DIR = os.path.join(
    "extendedPlotting",
    "sweeps",
    f"global_prbcd_{DATASET}__{FOLDER_SIGNATURE}__{RUN_GROUP_ID}",
)

os.makedirs(OUT_DIR, exist_ok=True)

# Store the full long signature separately, so it is not lost.
with open(os.path.join(OUT_DIR, "sweep_signature.json"), "w", encoding="utf-8") as handle:
    json.dump(
        {
            "run_group_id": RUN_GROUP_ID,
            "dataset": DATASET,
            "signature_hash": SIGNATURE_HASH,
            "folder_signature": FOLDER_SIGNATURE,
            "full_folder_signature": FULL_FOLDER_SIGNATURE,
            "endpoint_mining_hops": _collect_sweep_values([("log_values", "endpoint_mining_hop")]),
            "subgraph_fractions": _collect_sweep_values([("log_values", "subgraph_fraction_requested")]),
            "subgraph_seeds": _collect_sweep_values([("log_values", "subgraph_seed")]),
            "training_candidate_sizes": _collect_sweep_values([("log_values", "training_candidate_size")]),
            "resampling_modes": _collect_sweep_values([
                ("selector_params", "resampling_mode"),
                ("attack_params", "resampling_mode"),
            ]),
            "skipped_label_modes": sorted(SKIP_LABEL_MODES),
        },
        handle,
        indent=2,
        ensure_ascii=False,
    )

SUMMARY_CSV = os.path.join(OUT_DIR, "summary.csv")
EPOCHS_CSV = os.path.join(OUT_DIR, "epochs.csv")
RAW_RESULT_DIR = os.path.join(OUT_DIR, "raw_results")
if SAVE_RAW_RESULTS:
    os.makedirs(RAW_RESULT_DIR, exist_ok=True)

SUMMARY_HEADER = [
    "run_group_id",
    "run_id",
    "config_index",
    "status",
    "dataset",
    "model_label",
    "model_storage_type",
    "use_cert_label",
    "accuracy",
    "runtime_seconds",
    "error_type",
    "error_message",
    "raw_result_path",
] + ALL_CONFIG_COLUMNS

EPOCHS_HEADER = [
    "run_group_id",
    "run_id",
    "config_index",
    "dataset",
    "model_label",
    "use_cert_label",
    "stat_idx",
    "epoch_idx",
    "phase",
    "accuracy",
    "loss",
    "nonzero_weights",
    "prob_mass_update",
    "prob_mass_projected",
] + ALL_CONFIG_COLUMNS


def to_python_scalar(value):
    if isinstance(value, torch.Tensor):
        if value.numel() == 1:
            return value.detach().cpu().item()
        return str(value.detach().cpu().tolist())
    if isinstance(value, np.generic):
        return value.item()
    if value is None:
        return ""
    if isinstance(value, (str, int, float, bool)):
        return value
    return str(value)


def append_dict_rows(path, fieldnames, rows):
    rows = list(rows)
    if not rows:
        return

    file_exists = os.path.isfile(path)
    if file_exists:
        with open(path, "r", newline="", encoding="utf-8") as handle:
            existing_header = next(csv.reader(handle), None)
        if existing_header != fieldnames:
            raise RuntimeError(
                f"CSV header mismatch in {path}.\n"
                f"Found:    {existing_header}\n"
                f"Expected: {fieldnames}"
            )

    with open(path, "a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="raise")
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)
        handle.flush()
        os.fsync(handle.fileno())


def make_use_cert_label(
        experiment_cfg,
        selector_params,
):
    use_cert = experiment_cfg["use_cert"]
    parts = [str(use_cert)]

    if use_cert in ADS_CERTS:
        for key, prefix in (
            ("accuracy_drop_selector_mode", "mode"),
            ("drop_mode", "drop"),
            ("training_data_node_cap", "cap"),
        ):
            value = selector_params.get(key)

            if value not in (None, ""):
                parts.append(
                    f"{prefix}-{value}"
                )

    if use_cert == "accuracy_drop_selector_with_resampling":
        resampling_mode = selector_params.get(
            "resampling_mode"
        )

        if resampling_mode not in (None, ""):
            parts.append(
                f"resample-{resampling_mode}"
            )

    return "__".join(parts)


def make_run_id(config_index, cfg):
    serializable_cfg = {
        key: to_python_scalar(value)
        for key, value in config_to_row(cfg).items()
    }
    digest = hashlib.sha1(
        json.dumps(serializable_cfg, sort_keys=True).encode("utf-8")
    ).hexdigest()[:10]
    return f"{RUN_GROUP_ID}_{config_index:04d}_{digest}"


def make_config_run_group_id(cfg):
    """
    Keep all rows from one candidate-size × PRBCD-fraction × mining-mode
    configuration together, while still storing them in one sweep folder.
    This prevents plotting cells from averaging different candidate configs.
    """
    candidate_label = (
        cfg.get("log_values", {}).get("candidate_config_id")
        or cfg.get("log_values", {}).get("lp_model_label")
    )

    scoring_mode = cfg.get("log_values", {}).get("scoring_mode")
    endpoint_mining_hop = cfg.get("log_values", {}).get("endpoint_mining_hop")
    label_mode = cfg.get("log_values", {}).get("label_mode")
    extreme_fraction = cfg.get("log_values", {}).get("extreme_fraction")

    if candidate_label:
        suffix_parts = [str(candidate_label)]
        if scoring_mode:
            suffix_parts.append(f"score-{scoring_mode}")
        if endpoint_mining_hop not in (None, ""):
            suffix_parts.append(f"h-{endpoint_mining_hop}")
        if label_mode:
            suffix_parts.append(f"label-{label_mode}")
        if extreme_fraction not in (None, ""):
            suffix_parts.append(f"extremeFrac-{extreme_fraction}")
        return RUN_GROUP_ID + "__" + "__".join(suffix_parts)

    return RUN_GROUP_ID


def as_list(value):
    if value is None:
        return []
    if isinstance(value, torch.Tensor):
        value = value.detach().cpu()
        return [value.item()] if value.ndim == 0 else value.reshape(-1).tolist()
    if isinstance(value, np.ndarray):
        return value.reshape(-1).tolist()
    if isinstance(value, (list, tuple)):
        return [to_python_scalar(item) for item in value]
    return [to_python_scalar(value)]


def is_metric_series(value):
    return isinstance(value, (list, tuple, np.ndarray, torch.Tensor))


def find_attack_statistics(obj, seen=None):
    """Find a nested PRBCD attack-statistics dictionary."""
    if seen is None:
        seen = set()

    obj_id = id(obj)
    if obj_id in seen:
        return None
    seen.add(obj_id)

    if isinstance(obj, dict):
        for key in ("attack_statistics", "attack_stats", "stats"):
            candidate = obj.get(key)
            if isinstance(candidate, dict) and (
                is_metric_series(candidate.get("accuracy"))
                or is_metric_series(candidate.get("loss"))
            ):
                return candidate

        if (
            is_metric_series(obj.get("accuracy"))
            and is_metric_series(obj.get("loss"))
        ):
            return obj

        for value in obj.values():
            found = find_attack_statistics(value, seen)
            if found is not None:
                return found

    elif isinstance(obj, (list, tuple)):
        for value in obj:
            found = find_attack_statistics(value, seen)
            if found is not None:
                return found

    return None


def extract_final_accuracy(result):
    if isinstance(result, dict):
        results = result.get("results")
        if isinstance(results, (list, tuple)) and results:
            first = results[0]
            if isinstance(first, dict) and "accuracy" in first:
                return to_python_scalar(first["accuracy"])

        accuracy = result.get("accuracy")
        if accuracy is not None and not is_metric_series(accuracy):
            return to_python_scalar(accuracy)

    raise KeyError("Could not find final accuracy in the experiment result.")


def build_epoch_rows(stats, cfg, run_id, config_index, use_cert_label):
    if not stats:
        return []

    metric_lists = {
        "accuracy": as_list(stats.get("accuracy")),
        "loss": as_list(stats.get("loss")),
        "nonzero_weights": as_list(stats.get("nonzero_weights")),
        "prob_mass_update": as_list(stats.get("probability_mass_update")),
        "prob_mass_projected": as_list(stats.get("probability_mass_projected")),
    }

    length = max((len(values) for values in metric_lists.values()), default=0)
    configured_epochs = int(cfg["attack_params"].get("epochs", 0))
    has_baseline = configured_epochs > 0 and length == configured_epochs + 1
    config_row = {
        key: to_python_scalar(value)
        for key, value in config_to_row(cfg).items()
    }

    rows = []
    for stat_idx in range(length):
        is_baseline = has_baseline and stat_idx == 0
        epoch_idx = -1 if is_baseline else stat_idx - 1 if has_baseline else stat_idx

        row = {
            "run_group_id": make_config_run_group_id(cfg),
            "run_id": run_id,
            "config_index": config_index,
            "dataset": DATASET,
            "model_label": MODEL_LABEL,
            "use_cert_label": use_cert_label,
            "stat_idx": stat_idx,
            "epoch_idx": epoch_idx,
            "phase": "baseline" if is_baseline else "attack",
        }

        for metric_name, values in metric_lists.items():
            row[metric_name] = (
                to_python_scalar(values[stat_idx])
                if stat_idx < len(values)
                else ""
            )

        row.update(config_row)
        rows.append(row)

    return rows


# -------------------------
# Load the exact graph type required by the current run call
# -------------------------
graph_sparse = load_and_standardize(os.path.join("data", f"{DATASET}.npz"))

assert hasattr(graph_sparse, "attr_matrix")
assert hasattr(graph_sparse, "adj_matrix")
assert hasattr(graph_sparse, "labels")

print(
    "graph_sparse:",
    f"nodes={graph_sparse.attr_matrix.shape[0]}",
    f"features={graph_sparse.attr_matrix.shape[1]}",
    f"directed_edges={graph_sparse.adj_matrix.nnz}",
)
print("Output dir:", OUT_DIR)
print("Signature hash:", SIGNATURE_HASH)
print("Skipped label modes:", sorted(SKIP_LABEL_MODES))
print("Summary CSV:", SUMMARY_CSV)
print("Epoch CSV:", EPOCHS_CSV)

# -------------------------
# Sweep
# -------------------------
results_global_prbcd_cert = []

for config_index, cfg in enumerate(sweep_configs, start=1):
    experiment_cfg = dict(cfg["experiment"])
    attack_params = dict(cfg["attack_params"])
    selector_params = dict(cfg["selector_params"])

    epsilon = experiment_cfg["epsilon"]
    semi = experiment_cfg["semi"]
    use_cert = experiment_cfg["use_cert"]
    seed = experiment_cfg["seed"]
    attack_name = experiment_cfg["attack"]

    use_cert_label = make_use_cert_label(experiment_cfg, selector_params)
    run_id = make_run_id(config_index, cfg)
    config_row = {
        key: to_python_scalar(value)
        for key, value in config_to_row(cfg).items()
    }

    # --------------------------------------------------------
    # Temporary skip:
    # Skip all configurations with label_mode="subset_accuracy_drop".
    # --------------------------------------------------------
    log_values = cfg.get("log_values", {})
    label_mode = str(log_values.get("label_mode", "")).strip()

    if label_mode in SKIP_LABEL_MODES:
        print("=" * 80)
        print(f"Skipping run {config_index}/{len(sweep_configs)}: {run_id}")
        print("Reason: label_mode is disabled for now.")
        print("label_mode:", label_mode)

        summary_row = {
            "run_group_id": make_config_run_group_id(cfg),
            "run_id": run_id,
            "config_index": config_index,
            "status": "skipped",
            "dataset": DATASET,
            "model_label": MODEL_LABEL,
            "model_storage_type": MODEL_STORAGE_TYPE,
            "use_cert_label": use_cert_label,
            "accuracy": "",
            "runtime_seconds": 0,
            "error_type": "",
            "error_message": f"Skipped because label_mode={label_mode} is disabled for now.",
            "raw_result_path": "",
            **config_row,
        }

        append_dict_rows(SUMMARY_CSV, SUMMARY_HEADER, [summary_row])

        results_global_prbcd_cert.append({
            "run_id": run_id,
            "use_cert_label": use_cert_label,
            "accuracy": None,
            "runtime_seconds": 0,
            "status": "skipped",
            "reason": f"label_mode={label_mode} disabled",
        })

        continue

    print("=" * 80)
    print(f"Run {config_index}/{len(sweep_configs)}: {run_id}")
    print("attack:", attack_name)
    print("use_cert:", use_cert)
    print("attack_params:", attack_params)
    print("selector_params:", selector_params)
    print(
        "resampling_mode:",
        selector_params.get(
            "resampling_mode",
            "not-applicable",
        ),
    )

    delete_pert_files_and_folders(
        cache_dir=ARTIFACT_DIR,
        pert_adj_storage_type=PERT_ADJ_STORAGE_TYPE,
        pert_attr_storage_type=PERT_ATTR_STORAGE_TYPE,
    )

    start = timer()
    result = None
    raw_result_path = ""

    try:
        result = experiment_global_attack_direct.run(
            graph=graph_sparse,
            data_dir="./data",
            dataset=DATASET,
            attack=attack_name,
            attack_params=attack_params,
            selector_params=selector_params,
            epsilons=[epsilon],
            binary_attr=False,
            make_undirected=True,
            seed=seed,
            artifact_dir=ARTIFACT_DIR,
            pert_adj_storage_type=PERT_ADJ_STORAGE_TYPE,
            pert_attr_storage_type=PERT_ATTR_STORAGE_TYPE,
            model_label=MODEL_LABEL,
            model_storage_type=MODEL_STORAGE_TYPE,
            device="cpu",
            data_device="cpu",
            debug_level="info",
            semi=semi,
            use_cert=use_cert,
        )

        elapsed = timer() - start
        final_accuracy = extract_final_accuracy(result)

        stats = find_attack_statistics(result)

        if stats is None:
            raise RuntimeError(
                "No PRBCD attack statistics found in the returned result."
            )
        
        # Make attack statistics consistently available at the top level.
        result["attack_statistics"] = stats

        epoch_rows = build_epoch_rows(
            stats=stats,
            cfg=cfg,
            run_id=run_id,
            config_index=config_index,
            use_cert_label=use_cert_label,
        )
        append_dict_rows(EPOCHS_CSV, EPOCHS_HEADER, epoch_rows)

        if not epoch_rows:
            print("[warn] No per-epoch attack statistics found in the returned result.")

        if SAVE_RAW_RESULTS:
            raw_result_path = os.path.join(RAW_RESULT_DIR, f"{run_id}.pt")
            torch.save(result, raw_result_path)

        summary_row = {
            "run_group_id": make_config_run_group_id(cfg),
            "run_id": run_id,
            "config_index": config_index,
            "status": "ok",
            "dataset": DATASET,
            "model_label": MODEL_LABEL,
            "model_storage_type": MODEL_STORAGE_TYPE,
            "use_cert_label": use_cert_label,
            "accuracy": final_accuracy,
            "runtime_seconds": elapsed,
            "error_type": "",
            "error_message": "",
            "raw_result_path": raw_result_path,
            **config_row,
        }
        append_dict_rows(SUMMARY_CSV, SUMMARY_HEADER, [summary_row])

        results_global_prbcd_cert.append({
            "run_group_id": make_config_run_group_id(cfg),
            "run_id": run_id,
            "use_cert_label": use_cert_label,
            "seed": seed,
            "block_size": attack_params.get("block_size"),
            "candidate_config_id": cfg.get(
                "log_values",
                {},
            ).get(
                "candidate_config_id",
                "",
            ),
            "scoring_mode": cfg.get(
                "log_values",
                {},
            ).get(
                "scoring_mode",
                "",
            ),
            "training_candidate_size": cfg.get(
                "log_values", {}
            ).get("training_candidate_size", ""),
            "subgraph_seed": cfg.get(
                "log_values", {}
            ).get("subgraph_seed", ""),
            "subgraph_fraction_requested": cfg.get(
                "log_values", {}
            ).get("subgraph_fraction_requested", ""),
            "subgraph_fraction_actual": cfg.get(
                "log_values", {}
            ).get("subgraph_fraction_actual", ""),
            "n_source_nodes": cfg.get(
                "log_values", {}
            ).get("n_source_nodes", ""),
            "n_target_nodes": cfg.get(
                "log_values", {}
            ).get("n_target_nodes", ""),
            "accuracy": final_accuracy,
            "runtime_seconds": elapsed,
            "status": "ok",
            "attack_statistics": result["attack_statistics"],
        })

        print(f"Final accuracy: {final_accuracy}")
        print(f"Runtime: {elapsed:.2f} seconds")

        clear_output(wait=True)

    except Exception as exc:
        elapsed = timer() - start
        error_message = "".join(
            traceback.format_exception_only(type(exc), exc)
        ).strip()

        summary_row = {
            "run_group_id": make_config_run_group_id(cfg),
            "run_id": run_id,
            "config_index": config_index,
            "status": "error",
            "dataset": DATASET,
            "model_label": MODEL_LABEL,
            "model_storage_type": MODEL_STORAGE_TYPE,
            "use_cert_label": use_cert_label,
            "accuracy": "",
            "runtime_seconds": elapsed,
            "error_type": type(exc).__name__,
            "error_message": error_message,
            "raw_result_path": "",
            **config_row,
        }
        append_dict_rows(SUMMARY_CSV, SUMMARY_HEADER, [summary_row])

        results_global_prbcd_cert.append({
            "run_id": run_id,
            "use_cert_label": use_cert_label,
            "accuracy": None,
            "runtime_seconds": elapsed,
            "status": "error",
            "error": error_message,
        })

        print(f"[error] {error_message}")
        traceback.print_exc()

        if STOP_ON_ERROR:
            raise

    finally:
        del result
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("=" * 80)
print("Sweep finished.")
print("Output dir:", OUT_DIR)
print("Summary CSV:", SUMMARY_CSV)
print("Epoch CSV:", EPOCHS_CSV)

# -------------------------
# Seed-averaged sweep outputs
# -------------------------
if os.path.exists(SUMMARY_CSV):
    sweep_summary_raw_df = pd.read_csv(SUMMARY_CSV)
    completed = sweep_summary_raw_df[sweep_summary_raw_df["status"] == "ok"].copy()
    seed_col = "experiment__seed"
    non_group = {"run_id", "config_index", "run_group_id", seed_col, "log__seed", "log__lp_model_label", "accuracy", "runtime_seconds", "status", "error_type", "error_message", "raw_result_path"}
    sweep_group_cols = [column for column in completed.columns if column not in non_group and not column.startswith("selector__lp_model")]
    if not completed.empty:
        completed["seed"] = pd.to_numeric(completed[seed_col], errors="coerce")
        completed["accuracy"] = pd.to_numeric(completed["accuracy"], errors="coerce")
        completed["runtime_seconds"] = pd.to_numeric(completed["runtime_seconds"], errors="coerce")
        sweep_summary_averaged_df = aggregate_over_seeds(completed, sweep_group_cols, ["accuracy", "runtime_seconds"])
        sweep_summary_averaged_df.to_csv(os.path.join(OUT_DIR, "summary_averaged_over_seeds.csv"), index=False)
        display(sweep_summary_averaged_df)

if os.path.exists(EPOCHS_CSV):
    sweep_epochs_raw_df = pd.read_csv(EPOCHS_CSV)
    if not sweep_epochs_raw_df.empty:
        sweep_epochs_raw_df["seed"] = pd.to_numeric(sweep_epochs_raw_df["experiment__seed"], errors="coerce")
        epoch_group_cols = [column for column in ["run_group_id", "use_cert_label", "epoch_idx", "phase", "attack__block_size", "selector__resampling_mode", "log__candidate_config_id", "log__scoring_mode", "log__endpoint_mining_hop", "log__label_mode", "log__extreme_fraction"] if column in sweep_epochs_raw_df.columns]
        sweep_epochs_averaged_df = aggregate_over_seeds(sweep_epochs_raw_df, epoch_group_cols, ["accuracy", "loss", "nonzero_weights", "prob_mass_update", "prob_mass_projected"])
        sweep_epochs_averaged_df.to_csv(os.path.join(OUT_DIR, "epochs_averaged_over_seeds.csv"), index=False)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


rows = []

for run in results_global_prbcd_cert:
    if run.get("status") != "ok":
        continue

    stats = run.get("attack_statistics", {})

    epochs = as_list(stats.get("epoch"))
    mean_scores = as_list(
        stats.get("selector_score_mean")
    )
    max_scores = as_list(
        stats.get("selector_score_max")
    )

    # Backward-compatible fallback:
    # calculate the epoch maximum from per-try maxima.
    if not max_scores:
        try_maxes_by_epoch = as_list(
            stats.get("selector_score_try_maxes")
        )

        max_scores = []

        for try_maxes in try_maxes_by_epoch: 
            try:
                finite_try_maxes = [
                    float(value)
                    for value in as_list(try_maxes)
                    if np.isfinite(float(value))
                ]
            except (TypeError, ValueError):
                finite_try_maxes = []

            max_scores.append(
                max(finite_try_maxes)
                if finite_try_maxes
                else float("nan")
            )

    # Fallback when no explicit epoch list was recorded.
    # Index 0 is treated as the pre-attack baseline.
    number_of_rows = max(
        len(mean_scores),
        len(max_scores),
    )

    if not epochs and number_of_rows:
        epochs = list(
            range(-1, number_of_rows - 1)
        )

    for index, epoch in enumerate(epochs):
        try:
            epoch = int(epoch)
        except (TypeError, ValueError):
            continue

        mean_score = (
            mean_scores[index]
            if index < len(mean_scores)
            else float("nan")
        )

        max_score = (
            max_scores[index]
            if index < len(max_scores)
            else float("nan")
        )

        try:
            mean_score = float(mean_score)
        except (TypeError, ValueError):
            mean_score = float("nan")

        try:
            max_score = float(max_score)
        except (TypeError, ValueError):
            max_score = float("nan")

        # Remove baseline and epochs without selector resampling.
        if epoch < 0:
            continue

        if (
            not np.isfinite(mean_score)
            and not np.isfinite(max_score)
        ):
            continue

        rows.append({
            "run_group_id": run.get(
                "run_group_id",
                "",
            ),
            "use_cert_label": run.get(
                "use_cert_label",
                "",
            ),
            "seed": run.get("seed"),
            "block_size": run.get(
                "block_size"
            ),
            "candidate_config_id": run.get(
                "candidate_config_id",
                "",
            ),
            "scoring_mode": run.get(
                "scoring_mode",
                "",
            ),
            "training_candidate_size": run.get(
                "training_candidate_size",
                "",
            ),
            "subgraph_seed": run.get(
                "subgraph_seed",
                "",
            ),
            "subgraph_fraction_requested": run.get(
                "subgraph_fraction_requested",
                "",
            ),
            "epoch": epoch,
            "selector_score_mean": mean_score,
            "selector_score_max": max_score,
        })


selector_score_df = pd.DataFrame(rows)

if selector_score_df.empty:
    print(
        "No selector score values found. "
        "Run the selector-resampling attack first."
    )

else:
    group_columns = [
        "run_group_id",
        "use_cert_label",
        "block_size",
        "candidate_config_id",
        "scoring_mode",
        "training_candidate_size",
        "subgraph_seed",
        "subgraph_fraction_requested",
        "epoch",
    ]

    selector_score_averaged_df = (
        selector_score_df
        .groupby(
            group_columns,
            dropna=False,
            as_index=False,
        )
        .agg(
            mean_selector_score=(
                "selector_score_mean",
                "mean",
            ),
            std_selector_score=(
                "selector_score_mean",
                "std",
            ),
            mean_max_selector_score=(
                "selector_score_max",
                "mean",
            ),
            std_max_selector_score=(
                "selector_score_max",
                "std",
            ),
            n_seeds=(
                "seed",
                "nunique",
            ),
        )
        .sort_values("epoch")
    )

    display(selector_score_averaged_df)

    fig, ax = plt.subplots(
        figsize=(11, 6)
    )

    curve_columns = [
        "run_group_id",
        "use_cert_label",
        "block_size",
        "candidate_config_id",
        "scoring_mode",
        "training_candidate_size",
        "subgraph_seed",
        "subgraph_fraction_requested",
    ]

    for curve_key, curve in (
        selector_score_averaged_df.groupby(
            curve_columns,
            dropna=False,
        )
    ):
        (
            run_group_id,
            use_cert_label,
            block_size,
            candidate_config_id,
            scoring_mode,
            training_candidate_size,
            subgraph_seed,
            subgraph_fraction_requested,
        ) = curve_key

        curve = curve.sort_values("epoch")

        base_label = (
            f"{use_cert_label} | "
            f"block={block_size} | "
            f"{candidate_config_id} | "
            f"{scoring_mode} | "
            f"frac={subgraph_fraction_requested} | "
            f"trainN={training_candidate_size} | "
            f"sgSeed={subgraph_seed}"
        )

        # Mean selector score
        mean_mask = np.isfinite(
            curve["mean_selector_score"]
        )

        mean_curve = curve.loc[mean_mask]

        if not mean_curve.empty:
            mean_line, = ax.plot(
                mean_curve["epoch"],
                mean_curve["mean_selector_score"],
                marker="o",
                linewidth=1.8,
                linestyle="-",
                label=f"{base_label} — mean",
            )

            mean_std = (
                mean_curve["std_selector_score"]
                .fillna(0.0)
            )

            ax.fill_between(
                mean_curve["epoch"],
                mean_curve["mean_selector_score"]
                - mean_std,
                mean_curve["mean_selector_score"]
                + mean_std,
                alpha=0.12,
                color=mean_line.get_color(),
            )

            curve_color = mean_line.get_color()

        else:
            curve_color = None

        # Maximum selector score
        max_mask = np.isfinite(
            curve["mean_max_selector_score"]
        )

        max_curve = curve.loc[max_mask]

        if not max_curve.empty:
            max_line, = ax.plot(
                max_curve["epoch"],
                max_curve[
                    "mean_max_selector_score"
                ],
                marker="s",
                linewidth=1.8,
                linestyle="--",
                label=f"{base_label} — max",
                color=curve_color,
            )

            max_std = (
                max_curve[
                    "std_max_selector_score"
                ]
                .fillna(0.0)
            )

            ax.fill_between(
                max_curve["epoch"],
                max_curve[
                    "mean_max_selector_score"
                ] - max_std,
                max_curve[
                    "mean_max_selector_score"
                ] + max_std,
                alpha=0.08,
                color=max_line.get_color(),
            )

    ax.set_xlabel("PRBCD epoch")
    ax.set_ylabel("Selector score")
    ax.set_title(
        "Mean and maximum selector scores "
        "during PRBCD resampling"
    )
    ax.grid(True, alpha=0.3)
    ax.legend(
        fontsize=8,
        loc="best",
    )

    fig.tight_layout()
    fig.savefig(
        RUN_PLOTS_DIR / "selector_scores_during_resampling.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
from pathlib import Path
import re
import json
import hashlib

import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection


# ============================================================
# Configuration
# ============================================================

SWEEP_ROOT = Path("extendedPlotting") / "sweeps"

# Plot only the sweep created by the current notebook run.
#
# In the sweep runner, OUT_DIR should point to the active sweep folder,
# for example:
#   extendedPlotting/sweeps/global_prbcd_...__20260721_153440
#
# Set this manually to a folder path only when the plotting cell is run
# independently of the sweep cell.
CURRENT_SWEEP_DIR = globals().get("OUT_DIR", None)

# RUN_GROUP_ID is used to exclude unrelated rows if an epochs.csv happens
# to contain data from more than one sweep.
CURRENT_SWEEP_ID = str(
    globals().get("RUN_GROUP_ID", "")
).strip()

# Epoch -1 represents the pre-attack baseline.
BASELINE_EPOCH = -1
INCLUDE_BASELINE = True

SAVE_PLOTS = True
SHOW_PLOTS = True

# Fixed accuracy range for epoch-accuracy plots.
ACCURACY_Y_MIN = 0.75
ACCURACY_Y_MAX = 0.90

# Show mean ± one standard deviation across seeds.
SHOW_STD_REGION = True
STD_REGION_ALPHA = 0.20

# Exclude selected methods from all plots.
EXCLUDED_METHODS = {
    "accuracy_drop_selector_k_hop",
}

SIGNATURE_JSON_NAME = "sweep_signature.json"


# ============================================================
# Possible column names
# ============================================================

EPOCH_CANDIDATES = [
    "epoch_idx",
    "epoch",
    "Epoch",
    "step",
]

ACCURACY_CANDIDATES = [
    "accuracy",
    "acc",
    "Accuracy",
]

METHOD_CANDIDATES = [
    "use_cert_label",
    "attack__use_cert",
    "use_cert",
]

BLOCK_SIZE_CANDIDATES = [
    "block_size",
    "attack__block_size",
    "attack_params__block_size",
    "attack_params.block_size",
]

RUN_GROUP_CANDIDATES = [
    "run_group_id",
    "sweep_id",
    "experiment_id",
]

DATASET_CANDIDATES = [
    "dataset",
]

SEED_CANDIDATES = [
    "seed",
    "experiment__seed",
    "experiment_params__seed",
    "experiment_params.seed",
    "log__seed",
    "log_values__seed",
    "log_values.seed",
]

CANDIDATE_SET_SIZE_CANDIDATES = [
    "candidate_set_size",
    "log_values__candidate_set_size",
    "log_values.candidate_set_size",
]

PRBCD_FRACTION_CANDIDATES = [
    "prbcd_candidate_fraction",
    "log_values__prbcd_candidate_fraction",
    "log_values.prbcd_candidate_fraction",
]

SCORING_MODE_CANDIDATES = [
    "scoring_mode",
    "log_values__scoring_mode",
    "log_values.scoring_mode",
]

ENDPOINT_MINING_HOP_CANDIDATES = [
    "endpoint_mining_hop",
    "log__endpoint_mining_hop",
    "log_values__endpoint_mining_hop",
    "log_values.endpoint_mining_hop",
    "h",
]

LABEL_MODE_CANDIDATES = [
    "label_mode",
    "log_values__label_mode",
    "log_values.label_mode",
]

EXTREME_FRACTION_CANDIDATES = [
    "extreme_fraction",
    "log_values__extreme_fraction",
    "log_values.extreme_fraction",
]

TAU_CANDIDATES = [
    "tau",
    "selector__tau",
    "selector_params__tau",
    "selector_params.tau",
    "attack__tau",
    "attack_params__tau",
    "attack_params.tau",
    "experiment__tau",
    "experiment.tau",
]

RESAMPLING_MODE_CANDIDATES = [
    "resampling_mode",
    "selector__resampling_mode",
    "selector_params__resampling_mode",
    "selector_params.resampling_mode",
    "attack__resampling_mode",
    "attack_params__resampling_mode",
    "attack_params.resampling_mode",
]

EPSILON_CANDIDATES = [
    "epsilon",
    "eps",
    "experiment__epsilon",
    "experiment.epsilon",
    "attack_params__epsilon",
    "attack_params.epsilon",
    "attack_params__eps",
    "attack_params.eps",
]

EPOCHS_TOTAL_CANDIDATES = [
    "epochs",
    "attack_params__epochs",
    "attack_params.epochs",
]

CANDIDATE_CONFIG_ID_CANDIDATES = [
    "candidate_config_id",
    "log__candidate_config_id",
    "log_values__candidate_config_id",
    "log_values.candidate_config_id",
]

LP_MODEL_LABEL_CANDIDATES = [
    "lp_model_label",
    "log__lp_model_label",
    "log_values__lp_model_label",
    "log_values.lp_model_label",
]

# Derived plotting columns. These deliberately remove only the victim/model
# seed token. They do NOT remove subgraphSeed, which is a real configuration
# variable and must remain fixed.
SEED_AGNOSTIC_RUN_GROUP_COLUMN = "_plot_run_group_seed_agnostic"
SEED_AGNOSTIC_CANDIDATE_CONFIG_COLUMN = (
    "_plot_candidate_config_seed_agnostic"
)


# ============================================================
# Helper functions
# ============================================================

def normalize_column_name(name):
    return (
        str(name)
        .lower()
        .replace(".", "_")
        .replace("-", "_")
        .replace(" ", "_")
    )


def strip_victim_seed_token(value):
    """
    Remove only seed tokens that identify the victim/model instance.

    Examples
    --------
    "...__seed-0"              -> "..."
    "...__seed-3__score-x"     -> "...__score-x"
    "...__victimSeed-2"        -> "..."

    ``subgraphSeed-20`` is intentionally preserved because it is part of the
    experimental configuration rather than the replicate identifier.
    """
    if pd.isna(value):
        return value

    result = str(value)

    result = re.sub(
        r"__(?:seed|victimSeed|experimentSeed)-[0-9]+(?=__|$)",
        "",
        result,
        flags=re.IGNORECASE,
    )

    # Defensive cleanup in case removal leaves repeated separators.
    result = re.sub(r"_{4,}", "__", result)
    return result.strip("_")


def find_candidate_configuration_column(dataframe):
    """
    Prefer a genuine candidate_config_id column.

    Only fall back to lp_model_label when no candidate_config_id exists.
    The fallback is safe because its seed suffix is removed before grouping.
    """
    column = find_flexible_column(
        dataframe,
        CANDIDATE_CONFIG_ID_CANDIDATES,
    )

    if column is not None:
        return column

    return find_flexible_column(
        dataframe,
        LP_MODEL_LABEL_CANDIDATES,
    )


def add_seed_agnostic_plot_identifiers(
        dataframe,
        run_group_col,
):
    """
    Add identifiers that are identical across victim seeds for the same
    experimental configuration.
    """
    result = dataframe.copy()

    if run_group_col is not None:
        result[SEED_AGNOSTIC_RUN_GROUP_COLUMN] = (
            result[run_group_col]
            .map(strip_victim_seed_token)
        )

    candidate_column = find_candidate_configuration_column(result)

    if candidate_column is not None:
        result[SEED_AGNOSTIC_CANDIDATE_CONFIG_COLUMN] = (
            result[candidate_column]
            .map(strip_victim_seed_token)
        )

    return result


def find_first_existing_column(dataframe, candidates):
    return next(
        (
            column
            for column in candidates
            if column in dataframe.columns
        ),
        None,
    )


def find_column_by_suffix(dataframe, suffixes):
    normalized_suffixes = [
        normalize_column_name(suffix)
        for suffix in suffixes
    ]

    for column in dataframe.columns:
        normalized_column = normalize_column_name(column)

        for suffix in normalized_suffixes:
            # One-character suffix matching is too permissive (for example,
            # candidate "h" would incorrectly match "epoch"). Exact
            # matches were already checked by find_first_existing_column().
            if len(suffix) < 2:
                continue

            if normalized_column.endswith(suffix):
                return column

    return None


def find_flexible_column(dataframe, candidates):
    column = find_first_existing_column(dataframe, candidates)

    if column is not None:
        return column

    return find_column_by_suffix(dataframe, candidates)


def find_block_size_column(dataframe):
    """
    Find the block-size column using known names or a normalized
    column name ending in 'block_size'.
    """
    column = find_first_existing_column(
        dataframe,
        BLOCK_SIZE_CANDIDATES,
    )

    if column is not None:
        return column

    return next(
        (
            column
            for column in dataframe.columns
            if normalize_column_name(column).endswith("block_size")
        ),
        None,
    )


def safe_filename(value):
    """
    Convert a run identifier into a filesystem-safe string.
    """
    value = str(value).strip()

    value = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        value,
    )

    return value.strip("._-") or "unknown_run"


def compact_safe_filename(value, max_length=90):
    """
    Make safe filenames short enough for operating systems.
    """
    safe = safe_filename(value)

    if len(safe) <= max_length:
        return safe

    digest = hashlib.sha1(
        safe.encode("utf-8")
    ).hexdigest()[:10]

    prefix_length = max(20, max_length - len(digest) - 2)

    return f"{safe[:prefix_length].rstrip('._-')}__{digest}"


def format_block_size(block_size):
    """
    Format block sizes without unnecessary decimal places.
    """
    value = float(block_size)

    if value.is_integer():
        return str(int(value))

    return str(value).replace(".", "p")


def normalize_method_name(value):
    return str(value).strip()


def is_excluded_method(method):
    return normalize_method_name(method) in EXCLUDED_METHODS


def attach_matching_none_baseline(
        all_df,
        run_df,
        method_col,
        block_size_col,
):
    """
    Add the matching ``use_cert=none`` rows to a method-specific run.

    ``run_group_id`` may identify individual configurations, including the
    method itself. Grouping by the full run id therefore separates the attack
    run from its ``none`` baseline. This helper restores the comparison by
    matching baseline rows on experiment properties that must be shared.
    """
    if method_col is None:
        return run_df.copy()

    run_method_labels = {
        normalize_method_name(value).lower()
        for value in run_df[method_col].dropna().unique()
    }

    # The baseline is already present, so do not add it twice.
    if "none" in run_method_labels:
        return run_df.copy()

    none_mask = (
        all_df[method_col]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("none")
    )

    none_df = all_df.loc[none_mask].copy()

    if none_df.empty:
        print("Warning: the current sweep contains no 'none' rows.")
        return run_df.copy()

    dataset_col = find_flexible_column(
        all_df,
        DATASET_CANDIDATES,
    )

    seed_col = find_flexible_column(
        all_df,
        SEED_CANDIDATES,
    )

    epsilon_col = find_flexible_column(
        all_df,
        EPSILON_CANDIDATES,
    )

    epochs_total_col = find_flexible_column(
        all_df,
        EPOCHS_TOTAL_CANDIDATES,
    )

    match_columns = []

    for column in [
        dataset_col,
        seed_col,
        block_size_col,
        epsilon_col,
        epochs_total_col,
    ]:
        if (
            column is not None
            and column in all_df.columns
            and column not in match_columns
        ):
            match_columns.append(column)

    # block_size_col is always available by this point, so this normally
    # performs a precise merge rather than attaching every baseline row.
    comparison_keys = (
        run_df[match_columns]
        .drop_duplicates()
    )

    matching_none_df = none_df.merge(
        comparison_keys,
        on=match_columns,
        how="inner",
    )

    if matching_none_df.empty:
        print(
            "Warning: no matching 'none' baseline found for run values "
            f"on columns {match_columns}."
        )
        return run_df.copy()

    print(
        f"Added {len(matching_none_df)} matching 'none' row(s) "
        f"using {match_columns}."
    )

    return pd.concat(
        [
            run_df.copy(),
            matching_none_df,
        ],
        ignore_index=True,
        sort=False,
    )


def display_encoded_value(value):
    """
    Convert folder-safe values such as 0p25 back to compact display values.
    Keeps categorical values unchanged.
    """
    if value is None:
        return ""

    value = str(value)

    if value in ("", "na"):
        return "na"

    parts = [
        part
        for part in value.split("+")
        if part != ""
    ]

    if not parts:
        return "na"

    decoded_parts = []

    for part in parts:
        decoded = part.replace("p", ".")

        decoded_parts.append(decoded)

    return ", ".join(decoded_parts)


def shorten_text(text, max_chars=115):
    text = str(text)

    if len(text) <= max_chars:
        return text

    return text[: max_chars - 3].rstrip() + "..."


def parse_folder_signature(signature):
    """
    Parse strings like:
        block_size-4000__candN-1000+2500__tau-0p8

    into:
        {"block_size": "4000", "candN": "1000+2500", "tau": "0p8"}
    """
    fields = {}

    if not signature:
        return fields

    for part in str(signature).split("__"):
        if "-" not in part:
            continue

        key, value = part.split("-", 1)
        fields[key] = value

    return fields


def load_sweep_metadata(sweep_dir):
    """
    Load sweep_signature.json from a sweep folder, if present.
    Works with both new hashed folders and older folders without JSON.
    """
    json_path = sweep_dir / SIGNATURE_JSON_NAME

    metadata = {
        "exists": False,
        "path": str(json_path),
        "raw": {},
        "fields": {},
        "error": "",
    }

    if not json_path.exists():
        return metadata

    try:
        with open(json_path, "r", encoding="utf-8") as handle:
            raw = json.load(handle)

        full_signature = raw.get("full_folder_signature", "")
        fields = parse_folder_signature(full_signature)

        metadata.update({
            "exists": True,
            "raw": raw,
            "fields": fields,
        })

    except Exception as exc:
        metadata["error"] = f"{type(exc).__name__}: {exc}"

    return metadata


def unique_values_from_column(dataframe, candidates, max_values=12):
    column = find_flexible_column(dataframe, candidates)

    if column is None:
        return None

    values = (
        dataframe[column]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    if not values:
        return None

    values = sorted(values)

    if len(values) > max_values:
        shown = values[:max_values]
        return ", ".join(shown) + f", ... (+{len(values) - max_values})"

    return ", ".join(values)


def get_context_value(run_df, candidates, sweep_fields, sweep_key):
    """
    Prefer run-specific values from epochs.csv.
    Fall back to sweep-wide values from sweep_signature.json.
    """
    run_value = unique_values_from_column(run_df, candidates)

    if run_value not in (None, ""):
        return run_value

    sweep_value = sweep_fields.get(sweep_key)

    if sweep_value not in (None, ""):
        return display_encoded_value(sweep_value)

    return None


def build_plot_context_text(
        csv_path,
        run_df,
        sweep_metadata,
):
    """
    Create a compact text box with experimental metadata.

    The seed information is derived from the rows actually used for
    the plot. When multiple seeds are present, the plot is explicitly
    marked as a seed-averaged plot.
    """
    raw = sweep_metadata.get(
        "raw",
        {},
    )

    fields = sweep_metadata.get(
        "fields",
        {},
    )

    dataset = (
        raw.get("dataset")
        or unique_values_from_column(
            run_df,
            DATASET_CANDIDATES,
        )
        or "unknown"
    )

    signature_hash = (
        raw.get("signature_hash")
        or raw.get("folder_signature")
        or "no-signature-json"
    )

    # ========================================================
    # Determine the actual seeds represented in this plot
    # ========================================================

    seed_candidates = [
        "seed",
        "experiment__seed",
        "experiment_params__seed",
        "experiment_params.seed",
        "log__seed",
        "log_values__seed",
        "log_values.seed",
    ]

    seed_column = find_flexible_column(
        run_df,
        seed_candidates,
    )

    seed_label = None

    if seed_column is not None:
        seed_values = (
            run_df[seed_column]
            .dropna()
            .unique()
            .tolist()
        )

        def seed_sort_key(value):
            try:
                return float(value)
            except (TypeError, ValueError):
                return str(value)

        seed_values = sorted(
            seed_values,
            key=seed_sort_key,
        )

        seed_count = len(
            seed_values
        )

        if seed_count > 1:
            seed_label = (
                f"mean over {seed_count} seeds"
            )

        elif seed_count == 1:
            seed_value = seed_values[0]

            try:
                numeric_seed = float(
                    seed_value
                )

                if numeric_seed.is_integer():
                    seed_value = int(
                        numeric_seed
                    )

            except (TypeError, ValueError):
                pass

            seed_label = (
                f"single seed: {seed_value}"
            )

    # ========================================================
    # Construct context entries
    # ========================================================

    items = [
        (
            "dataset",
            dataset,
        ),
        (
            "sig",
            signature_hash,
        ),
        (
            "seed aggregation",
            seed_label,
        ),
        (
            "candN",
            get_context_value(
                run_df,
                CANDIDATE_SET_SIZE_CANDIDATES,
                fields,
                "candN",
            ),
        ),
        (
            "prbcdFrac",
            get_context_value(
                run_df,
                PRBCD_FRACTION_CANDIDATES,
                fields,
                "prbcdFrac",
            ),
        ),
        (
            "scoring",
            get_context_value(
                run_df,
                SCORING_MODE_CANDIDATES,
                fields,
                "scoringMode",
            ),
        ),
        (
            "h",
            get_context_value(
                run_df,
                ENDPOINT_MINING_HOP_CANDIDATES,
                fields,
                "h",
            ),
        ),
        (
            "label",
            get_context_value(
                run_df,
                LABEL_MODE_CANDIDATES,
                fields,
                "labelMode",
            ),
        ),
        (
            "extremeFrac",
            get_context_value(
                run_df,
                EXTREME_FRACTION_CANDIDATES,
                fields,
                "extremeFrac",
            ),
        ),
        (
            "tau",
            get_context_value(
                run_df,
                TAU_CANDIDATES,
                fields,
                "tau",
            ),
        ),
        (
            "resampling",
            get_context_value(
                run_df,
                RESAMPLING_MODE_CANDIDATES,
                fields,
                "resamplingMode",
            ),
        ),
        (
            "eps",
            get_context_value(
                run_df,
                EPSILON_CANDIDATES,
                fields,
                "epsilon",
            ),
        ),
        (
            "epochs",
            get_context_value(
                run_df,
                EPOCHS_TOTAL_CANDIDATES,
                fields,
                "epochs",
            ),
        ),
        (
            "candidate_cfg",
            unique_values_from_column(
                run_df,
                [SEED_AGNOSTIC_CANDIDATE_CONFIG_COLUMN],
            ),
        ),
    ]

    rendered = []

    for key, value in items:
        if value is None:
            continue

        value_text = str(
            value
        ).strip()

        if value_text.lower() in {
            "",
            "nan",
            "none",
        }:
            continue

        rendered.append(
            f"{key}="
            f"{shorten_text(value_text, max_chars=70)}"
        )

    if not rendered:
        return ""

    # Split context across lines to prevent overly wide boxes.
    lines = []
    current_line = ""

    for item in rendered:
        candidate_line = (
            item
            if not current_line
            else current_line + " | " + item
        )

        if len(candidate_line) <= 115:
            current_line = candidate_line

        else:
            if current_line:
                lines.append(
                    current_line
                )

            current_line = item

    if current_line:
        lines.append(
            current_line
        )

    return "\n".join(
        lines
    )




# ============================================================
# Thesis-oriented plotting helpers
# ============================================================

ACCURACY_DROP_COLUMN = "accuracy_drop_pp"
BASELINE_ACCURACY_COLUMN = "seed_baseline_accuracy"

# These columns define experiment configurations. Only columns that actually
# vary within a run group are used to split plots. Method and seed are handled
# separately, and block size remains a plot dimension.
CONFIGURATION_COLUMN_GROUPS = [
    ("epsilon", EPSILON_CANDIDATES),
    ("candidate_set_size", CANDIDATE_SET_SIZE_CANDIDATES),
    ("prbcd_candidate_fraction", PRBCD_FRACTION_CANDIDATES),
    ("scoring_mode", SCORING_MODE_CANDIDATES),
    ("endpoint_mining_hop", ENDPOINT_MINING_HOP_CANDIDATES),
    ("label_mode", LABEL_MODE_CANDIDATES),
    ("extreme_fraction", EXTREME_FRACTION_CANDIDATES),
    ("tau", TAU_CANDIDATES),
    ("resampling_mode", RESAMPLING_MODE_CANDIDATES),
    ("epochs", EPOCHS_TOTAL_CANDIDATES),
    (
        "candidate_config_id",
        [SEED_AGNOSTIC_CANDIDATE_CONFIG_COLUMN],
    ),
]


def display_method_name(method):
    """Return short, thesis-friendly method names."""
    normalized = normalize_method_name(method)
    lower = normalized.lower()

    if "accuracy_drop_selector_with_resampling" in lower:
        return "Selector with Resampling"

    if lower == "none":
        return "Standard PR-BCD"

    # Keep unknown methods readable without exposing snake_case internals.
    return normalized.replace("_", " ")


def format_config_value(value):
    if pd.isna(value):
        return "na"

    if isinstance(value, float) and value.is_integer():
        return str(int(value))

    return str(value)


def get_varying_configuration_columns(
        dataframe,
        epoch_col,
        method_col,
        block_size_col,
):
    """
    Return ``[(logical_name, real_column), ...]`` for configuration columns
    that vary among the non-baseline attack rows in this run group.

    Epsilon is therefore never averaged together with another epsilon. The
    same rule also applies to any other recognized configuration field.
    """
    reference_df = dataframe[dataframe[epoch_col] != BASELINE_EPOCH].copy()

    if method_col is not None:
        non_standard_mask = ~(
            reference_df[method_col]
            .astype(str)
            .str.strip()
            .str.lower()
            .eq("none")
        )
        if non_standard_mask.any():
            reference_df = reference_df.loc[non_standard_mask].copy()

    varying = []
    seen_columns = set()

    for logical_name, candidates in CONFIGURATION_COLUMN_GROUPS:
        column = find_flexible_column(reference_df, candidates)

        if (
            column is None
            or column in seen_columns
            or column in {epoch_col, method_col, block_size_col}
        ):
            continue

        if reference_df[column].nunique(dropna=False) > 1:
            varying.append((logical_name, column))
            seen_columns.add(column)

    return varying


def configuration_label(config_values):
    if not config_values:
        return "default configuration"

    return ", ".join(
        f"{name}={format_config_value(value)}"
        for name, value in config_values.items()
    )


def configuration_filename(config_values):
    if not config_values:
        return "config-default"

    raw = "__".join(
        f"{name}-{format_config_value(value)}"
        for name, value in config_values.items()
    )
    return compact_safe_filename(raw, max_length=110)


def select_configuration_rows(dataframe, columns, values):
    mask = pd.Series(True, index=dataframe.index)

    for column, value in zip(columns, values):
        if pd.isna(value):
            mask &= dataframe[column].isna()
        else:
            mask &= dataframe[column].eq(value)

    return dataframe.loc[mask].copy()


def iter_configuration_dataframes(
        all_df,
        run_df,
        epoch_col,
        method_col,
        block_size_col,
):
    """
    Split a run into fixed configurations, then attach the matching standard
    PR-BCD rows. Seeds remain inside each configuration and are the only
    dimension averaged by the plotting summaries.
    """
    varying_pairs = get_varying_configuration_columns(
        dataframe=run_df,
        epoch_col=epoch_col,
        method_col=method_col,
        block_size_col=block_size_col,
    )
    varying_columns = [column for _, column in varying_pairs]

    if method_col is not None:
        attack_rows = run_df.loc[
            ~(
                run_df[method_col]
                .astype(str)
                .str.strip()
                .str.lower()
                .eq("none")
            )
        ].copy()
    else:
        attack_rows = run_df.copy()

    if attack_rows.empty:
        return

    if varying_columns:
        grouped = attack_rows.groupby(
            varying_columns,
            dropna=False,
            sort=True,
        )
    else:
        grouped = [((), attack_rows)]

    for group_key, attack_config_df in grouped:
        if varying_columns:
            if len(varying_columns) == 1:
                single_value = (
                    group_key[0]
                    if isinstance(group_key, tuple)
                    else group_key
                )
                values = (single_value,)
            else:
                values = tuple(group_key)

            # Re-select explicitly so pandas' treatment of NaN group keys does
            # not leak rows from another configuration.
            attack_config_df = select_configuration_rows(
                attack_rows,
                varying_columns,
                values,
            )
        else:
            values = ()

        config_values = {
            logical_name: value
            for (logical_name, _), value in zip(varying_pairs, values)
        }

        if method_col is not None:
            config_df = attach_matching_none_baseline(
                all_df=all_df,
                run_df=attack_config_df,
                method_col=method_col,
                block_size_col=block_size_col,
            )
        else:
            config_df = attack_config_df.copy()

        yield config_values, config_df


def baseline_identity_columns(
        dataframe,
        seed_col,
        block_size_col,
):
    """Columns that identify the clean baseline belonging to an observation."""
    columns = [seed_col]

    for candidates in [
        DATASET_CANDIDATES,
        [block_size_col],
        EPSILON_CANDIDATES,
        EPOCHS_TOTAL_CANDIDATES,
    ]:
        column = find_flexible_column(dataframe, candidates)

        if column is not None and column not in columns:
            columns.append(column)

    return columns


def add_seed_specific_accuracy_drop(
        dataframe,
        epoch_col,
        accuracy_col,
        method_col,
        block_size_col,
):
    """
    Add accuracy drop in percentage points relative to each seed's own
    pre-attack baseline.

    The exact method-specific baseline is preferred. If a method does not have
    its own epoch -1 row, the matching seed/configuration baseline from another
    method (normally Standard PR-BCD) is used as a fallback.
    """
    seed_col = find_flexible_column(dataframe, SEED_CANDIDATES)

    if seed_col is None:
        raise RuntimeError(
            "Cannot compute seed-specific accuracy drops because no seed "
            "column was found."
        )

    work_df = dataframe.copy()
    baseline_rows = work_df.loc[
        work_df[epoch_col] == BASELINE_EPOCH
    ].copy()

    if baseline_rows.empty:
        raise RuntimeError(
            f"No epoch {BASELINE_EPOCH} rows were found, so seed-specific "
            "baselines cannot be computed."
        )

    identity_cols = baseline_identity_columns(
        dataframe=work_df,
        seed_col=seed_col,
        block_size_col=block_size_col,
    )

    # Shared fallback baseline. Duplicate method baselines are collapsed only
    # after matching the seed and configuration identity.
    shared_stats = (
        baseline_rows
        .groupby(identity_cols, dropna=False)[accuracy_col]
        .agg(["mean", "min", "max", "count"])
        .reset_index()
        .rename(columns={
            "mean": "_baseline_shared",
            "min": "_baseline_min",
            "max": "_baseline_max",
            "count": "_baseline_count",
        })
    )

    inconsistent = shared_stats.loc[
        (shared_stats["_baseline_max"] - shared_stats["_baseline_min"]).abs()
        > 1e-10
    ]
    if not inconsistent.empty:
        print(
            "Warning: baseline values differ across duplicate method rows for "
            f"{len(inconsistent)} seed/configuration combination(s). The "
            "method-specific baseline is used where available."
        )

    work_df = work_df.merge(
        shared_stats[identity_cols + ["_baseline_shared"]],
        on=identity_cols,
        how="left",
        validate="many_to_one",
    )

    if method_col is not None:
        exact_cols = identity_cols + [method_col]
        exact_baseline = (
            baseline_rows
            .groupby(exact_cols, dropna=False)[accuracy_col]
            .mean()
            .reset_index(name="_baseline_exact")
        )
        work_df = work_df.merge(
            exact_baseline,
            on=exact_cols,
            how="left",
            validate="many_to_one",
        )
        work_df[BASELINE_ACCURACY_COLUMN] = (
            work_df["_baseline_exact"]
            .combine_first(work_df["_baseline_shared"])
        )
    else:
        work_df[BASELINE_ACCURACY_COLUMN] = work_df["_baseline_shared"]

    missing_baseline = work_df[BASELINE_ACCURACY_COLUMN].isna()
    if missing_baseline.any():
        print(
            f"Warning: dropping {int(missing_baseline.sum())} row(s) without "
            "a matching seed-specific baseline."
        )
        work_df = work_df.loc[~missing_baseline].copy()

    work_df[ACCURACY_DROP_COLUMN] = 100.0 * (
        work_df[BASELINE_ACCURACY_COLUMN] - work_df[accuracy_col]
    )

    removable = ["_baseline_shared", "_baseline_exact"]
    work_df = work_df.drop(
        columns=[column for column in removable if column in work_df.columns]
    )

    return work_df


def aggregate_metric_across_seeds(
        dataframe,
        epoch_col,
        metric_col,
):
    """
    Aggregate one seed-independent fixed configuration over seeds only.

    Duplicate measurements inside the same seed and epoch are averaged first,
    so every seed receives equal weight. The returned ``std`` is the sample
    standard deviation across seed-level values (pandas ddof=1).
    """
    seed_col = find_flexible_column(dataframe, SEED_CANDIDATES)

    if seed_col is None:
        raise RuntimeError(
            "Seed aggregation requested, but no seed column was found."
        )

    work_df = (
        dataframe[[seed_col, epoch_col, metric_col]]
        .dropna(subset=[seed_col, epoch_col, metric_col])
        .copy()
    )

    if work_df.empty:
        return pd.DataFrame(columns=["mean", "std", "seed_count"])

    per_seed = (
        work_df
        .groupby([seed_col, epoch_col], dropna=False)[metric_col]
        .mean()
        .reset_index()
    )

    summary = (
        per_seed
        .groupby(epoch_col, dropna=False)[metric_col]
        .agg(mean="mean", std="std", seed_count="count")
        .sort_index()
    )
    summary["std"] = summary["std"].fillna(0.0)

    return summary


def add_std_ribbon_3d(
        axis,
        epoch_values,
        mean_values,
        std_values,
        block_size,
        color,
):
    if (
        not SHOW_STD_REGION
        or len(epoch_values) < 2
        or not any(value > 0 for value in std_values)
    ):
        return

    lower_values = mean_values - std_values
    upper_values = mean_values + std_values

    vertices = [
        list(zip(
            epoch_values,
            lower_values,
            [float(block_size)] * len(epoch_values),
        ))
        + list(zip(
            epoch_values[::-1],
            upper_values[::-1],
            [float(block_size)] * len(epoch_values),
        ))
    ]

    axis.add_collection3d(
        Poly3DCollection(
            vertices,
            facecolor=color,
            edgecolor="none",
            alpha=STD_REGION_ALPHA,
        )
    )


def deduplicated_legend(axis, **kwargs):
    handles, labels = axis.get_legend_handles_labels()
    unique = {}

    for handle, label in zip(handles, labels):
        if label and label != "_nolegend_" and label not in unique:
            unique[label] = handle

    if unique:
        axis.legend(
            unique.values(),
            unique.keys(),
            frameon=False,
            **kwargs,
        )


def style_2d_drop_axis(axis):
    axis.axhline(0.0, color="0.35", linewidth=0.9, alpha=0.75)
    axis.grid(visible=True, axis="y", alpha=0.20)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)


def save_or_show_figure(figure, plot_path):
    figure.tight_layout()

    if SAVE_PLOTS:
        figure.savefig(
            plot_path,
            dpi=300,
            bbox_inches="tight",
        )
        print("Saved:", plot_path)

    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(figure)


def calculate_maximum_accuracy_drops(
        dataframe,
        epoch_col,
        method_col,
        block_size_col,
):
    """Maximum mean seed-normalized accuracy drop for every method/block size."""
    rows = []

    if method_col is None:
        method_groups = [("accuracy", dataframe)]
    else:
        method_groups = dataframe.groupby(method_col, dropna=False)

    for method, method_df in method_groups:
        raw_method = normalize_method_name(method)

        if is_excluded_method(raw_method):
            continue

        for block_size, block_df in method_df.groupby(
            block_size_col,
            dropna=False,
        ):
            if pd.isna(block_size):
                continue

            attack_df = block_df.loc[
                block_df[epoch_col] != BASELINE_EPOCH
            ].copy()

            summary = aggregate_metric_across_seeds(
                dataframe=attack_df,
                epoch_col=epoch_col,
                metric_col=ACCURACY_DROP_COLUMN,
            )

            if summary.empty:
                continue

            max_epoch = summary["mean"].idxmax()
            rows.append({
                "block_size": float(block_size),
                "raw_method": raw_method,
                "method": display_method_name(raw_method),
                "epoch": int(max_epoch),
                "mean_accuracy_drop_pp": float(summary.loc[max_epoch, "mean"]),
                "std_accuracy_drop_pp": float(summary.loc[max_epoch, "std"]),
                "seed_count": int(summary.loc[max_epoch, "seed_count"]),
            })

    return pd.DataFrame(rows)


# ============================================================
# Resolve the epochs.csv belonging to the current sweep only
# ============================================================

if CURRENT_SWEEP_DIR is None or str(CURRENT_SWEEP_DIR).strip() == "":
    raise RuntimeError(
        "OUT_DIR is not defined. Run the sweep cell first, or set "
        "CURRENT_SWEEP_DIR manually to the active sweep folder."
    )

current_sweep_dir = Path(CURRENT_SWEEP_DIR).expanduser()

if current_sweep_dir.name == "epochs.csv":
    current_epochs_csv = current_sweep_dir
    current_sweep_dir = current_sweep_dir.parent
else:
    current_epochs_csv = current_sweep_dir / "epochs.csv"

if not current_epochs_csv.exists():
    raise FileNotFoundError(
        "The current sweep has no epochs.csv:\n"
        f"  sweep folder: {current_sweep_dir.resolve()}\n"
        f"  expected CSV: {current_epochs_csv.resolve()}"
    )

# Preserve a notebook-provided output folder, but make the script executable
# independently as well.
RUN_PLOTS_DIR = Path(
    globals().get("RUN_PLOTS_DIR", current_sweep_dir / "plots")
)

csv_files = [current_epochs_csv]

print("Current sweep folder:", current_sweep_dir)
print("Current sweep id:", CURRENT_SWEEP_ID or "not available")
print("Processing only:", current_epochs_csv)
print(
    "Metric: accuracy drop in percentage points relative to each seed's "
    f"epoch {BASELINE_EPOCH} baseline."
)


# ============================================================
# Process the current epochs.csv
# ============================================================

for csv_path in csv_files:
    print()
    print("=" * 100)
    print("Processing:", csv_path)

    sweep_metadata = load_sweep_metadata(csv_path.parent)

    if sweep_metadata["exists"]:
        print("Sweep metadata:", sweep_metadata["path"])
        print("Signature hash:", sweep_metadata["raw"].get("signature_hash", ""))
        print("Parsed signature fields:", sweep_metadata["fields"])
    else:
        print("Sweep metadata: no sweep_signature.json found.")
        if sweep_metadata["error"]:
            print("Metadata error:", sweep_metadata["error"])

    df = pd.read_csv(csv_path)

    if df.empty:
        print("Skipping empty file.")
        continue

    epoch_col = find_flexible_column(df, EPOCH_CANDIDATES)
    accuracy_col = find_flexible_column(df, ACCURACY_CANDIDATES)
    method_col = find_flexible_column(df, METHOD_CANDIDATES)
    block_size_col = find_block_size_column(df)
    run_group_col = find_flexible_column(df, RUN_GROUP_CANDIDATES)

    if CURRENT_SWEEP_ID and run_group_col is not None:
        current_sweep_mask = (
            df[run_group_col]
            .astype(str)
            .str.startswith(CURRENT_SWEEP_ID)
        )

        if current_sweep_mask.any():
            skipped_rows = int((~current_sweep_mask).sum())
            df = df.loc[current_sweep_mask].copy()

            if skipped_rows:
                print(
                    f"Filtered out {skipped_rows} row(s) not belonging "
                    f"to current sweep {CURRENT_SWEEP_ID}."
                )
        else:
            raise RuntimeError(
                f"No rows in {csv_path} match current RUN_GROUP_ID "
                f"{CURRENT_SWEEP_ID!r}."
            )

    if epoch_col is None:
        print("Skipping: no compatible epoch column found.")
        continue
    if accuracy_col is None:
        print("Skipping: no compatible accuracy column found.")
        continue
    if block_size_col is None:
        print("Skipping: no compatible block-size column found.")
        continue

    df[epoch_col] = pd.to_numeric(df[epoch_col], errors="coerce")
    df[accuracy_col] = pd.to_numeric(df[accuracy_col], errors="coerce")
    df[block_size_col] = pd.to_numeric(df[block_size_col], errors="coerce")

    df = df.dropna(
        subset=[epoch_col, accuracy_col, block_size_col]
    ).copy()

    if df.empty:
        print("Skipping: no valid rows remained.")
        continue

    df[epoch_col] = df[epoch_col].astype(int)

    # --------------------------------------------------------
    # Collapse seed-specific selector labels into one configuration.
    #
    # Without this step, labels ending in ``__seed-0``, ``__seed-1``,
    # etc. are treated as different configurations before
    # aggregate_metric_across_seeds() is reached.
    # --------------------------------------------------------
    df = add_seed_agnostic_plot_identifiers(
        dataframe=df,
        run_group_col=run_group_col,
    )

    if run_group_col is not None:
        run_groups = df.groupby(
            SEED_AGNOSTIC_RUN_GROUP_COLUMN,
            dropna=False,
        )
    else:
        run_groups = [(csv_path.parent.name, df)]

    for run_group_value, run_df in run_groups:
        run_identifier = (
            csv_path.parent.name
            if pd.isna(run_group_value)
            else str(run_group_value)
        )

        if method_col is not None:
            original_method_labels = {
                normalize_method_name(value).lower()
                for value in run_df[method_col].dropna().unique()
            }

            if original_method_labels == {"none"}:
                print(
                    f"Skipping standalone none run {run_identifier}; "
                    "it is used as the Standard PR-BCD comparison."
                )
                continue

        safe_run_identifier = compact_safe_filename(run_identifier)

        for config_values, config_df in iter_configuration_dataframes(
            all_df=df,
            run_df=run_df,
            epoch_col=epoch_col,
            method_col=method_col,
            block_size_col=block_size_col,
        ):
            config_name = configuration_label(config_values)
            safe_config_name = configuration_filename(config_values)

            try:
                config_df = add_seed_specific_accuracy_drop(
                    dataframe=config_df,
                    epoch_col=epoch_col,
                    accuracy_col=accuracy_col,
                    method_col=method_col,
                    block_size_col=block_size_col,
                )
            except RuntimeError as exc:
                print(
                    f"Skipping run {run_identifier}, {config_name}: {exc}"
                )
                continue

            if not INCLUDE_BASELINE:
                config_df = config_df.loc[
                    config_df[epoch_col] != BASELINE_EPOCH
                ].copy()

            if config_df.empty:
                print(
                    f"Skipping run {run_identifier}, {config_name}: "
                    "no rows remained."
                )
                continue

            block_sizes = sorted(
                config_df[block_size_col]
                .dropna()
                .unique()
                .tolist()
            )

            plot_output_dir = RUN_PLOTS_DIR
            if SAVE_PLOTS:
                plot_output_dir.mkdir(parents=True, exist_ok=True)

            plot_context_text = build_plot_context_text(
                csv_path=csv_path,
                run_df=config_df,
                sweep_metadata=sweep_metadata,
            )

            seed_col = find_flexible_column(
                config_df,
                SEED_CANDIDATES,
            )
            seed_count_for_plot = (
                int(config_df[seed_col].nunique(dropna=True))
                if seed_col is not None
                else 0
            )

            if seed_count_for_plot < 2:
                print(
                    "Warning: this configuration contains only "
                    f"{seed_count_for_plot} seed(s); its SD ribbon will be zero."
                )
            else:
                print(
                    f"Aggregating mean ± 1 SD over "
                    f"{seed_count_for_plot} seeds."
                )

            print()
            print("-" * 100)
            print("Run:", run_identifier)
            print("Configuration:", config_name)
            print(
                "Block sizes:",
                [format_block_size(value) for value in block_sizes],
            )
            if method_col is not None:
                method_pairs = sorted({
                    (
                        normalize_method_name(value),
                        display_method_name(value),
                    )
                    for value in config_df[method_col].dropna().unique()
                })
                print("Method labels:")
                for raw_method, short_method in method_pairs:
                    print(f"  {raw_method} -> {short_method}")
            print("Plot context (printed, not embedded in figure):")
            print(plot_context_text or "na")
            print("Output folder:", plot_output_dir)

            if SAVE_PLOTS:
                plot_metadata_path = (
                    plot_output_dir
                    / (
                        "plot_metadata"
                        f"__run-{safe_run_identifier}"
                        f"__{safe_config_name}.json"
                    )
                )
                with open(plot_metadata_path, "w", encoding="utf-8") as handle:
                    json.dump(
                        {
                            "csv_path": str(csv_path),
                            "run_identifier": run_identifier,
                            "configuration": config_values,
                            "block_sizes": [
                                format_block_size(value)
                                for value in block_sizes
                            ],
                            "metric": ACCURACY_DROP_COLUMN,
                            "baseline_epoch": BASELINE_EPOCH,
                            "plot_context_text": plot_context_text,
                            "sweep_metadata": sweep_metadata,
                        },
                        handle,
                        indent=2,
                        ensure_ascii=False,
                        default=str,
                    )
                print("Saved:", plot_metadata_path)

            # ====================================================
            # 1. 3D plot across block sizes
            # ====================================================

            if len(block_sizes) > 1:
                figure = plt.figure(figsize=(11, 7))
                axis = figure.add_subplot(111, projection="3d")
                plotted_lines = 0

                if method_col is not None:
                    grouped_series = config_df.groupby(
                        [method_col, block_size_col],
                        dropna=False,
                    )
                else:
                    grouped_series = config_df.groupby(
                        block_size_col,
                        dropna=False,
                    )

                for group_key, group in grouped_series:
                    if method_col is not None:
                        method, block_size = group_key
                        raw_method = normalize_method_name(method)
                    else:
                        block_size = group_key
                        raw_method = "accuracy"

                    if is_excluded_method(raw_method) or pd.isna(block_size):
                        continue

                    summary = aggregate_metric_across_seeds(
                        dataframe=group,
                        epoch_col=epoch_col,
                        metric_col=ACCURACY_DROP_COLUMN,
                    )
                    if summary.empty:
                        continue

                    epoch_values = summary.index.to_numpy()
                    mean_values = summary["mean"].to_numpy()
                    std_values = summary["std"].to_numpy()
                    block_size_values = [float(block_size)] * len(summary)

                    line, = axis.plot(
                        epoch_values,
                        mean_values,
                        block_size_values,
                        label=display_method_name(raw_method),
                    )

                    add_std_ribbon_3d(
                        axis=axis,
                        epoch_values=epoch_values,
                        mean_values=mean_values,
                        std_values=std_values,
                        block_size=block_size,
                        color=line.get_color(),
                    )
                    plotted_lines += 1

                if plotted_lines:
                    axis.set_xlabel(
                        "Epoch (-1 = baseline)"
                        if INCLUDE_BASELINE
                        else "Epoch"
                    )
                    axis.set_ylabel("Accuracy drop (pp)")
                    axis.set_zlabel("Block size")
                    axis.set_title("Accuracy drop across block sizes")
                    axis.set_zticks(block_sizes)
                    axis.view_init(elev=25, azim=-60)
                    axis.grid(visible=True, alpha=0.20)
                    deduplicated_legend(
                        axis,
                        loc="upper left",
                        bbox_to_anchor=(1.02, 1.0),
                    )

                    plot_path = (
                        plot_output_dir
                        / (
                            "accuracy_drop_3d"
                            f"__run-{safe_run_identifier}"
                            f"__{safe_config_name}.png"
                        )
                    )
                    save_or_show_figure(figure, plot_path)
                else:
                    plt.close(figure)
                    print("No valid series remained for the 3D plot.")

            # ====================================================
            # 2. One accuracy-drop trajectory per block size
            # ====================================================

            for block_size in block_sizes:
                block_df = config_df.loc[
                    config_df[block_size_col] == block_size
                ].copy()
                if block_df.empty:
                    continue

                figure, axis = plt.subplots(figsize=(8.5, 5.2))
                plotted_lines = 0

                if method_col is not None:
                    method_groups = block_df.groupby(method_col, dropna=False)
                else:
                    method_groups = [("accuracy", block_df)]

                for method, method_df in method_groups:
                    raw_method = normalize_method_name(method)
                    if is_excluded_method(raw_method):
                        continue

                    summary = aggregate_metric_across_seeds(
                        dataframe=method_df,
                        epoch_col=epoch_col,
                        metric_col=ACCURACY_DROP_COLUMN,
                    )
                    if summary.empty:
                        continue

                    epoch_values = summary.index.to_numpy()
                    mean_values = summary["mean"].to_numpy()
                    std_values = summary["std"].to_numpy()

                    line, = axis.plot(
                        epoch_values,
                        mean_values,
                        linewidth=2.0,
                        label=display_method_name(raw_method),
                    )

                    if SHOW_STD_REGION:
                        axis.fill_between(
                            epoch_values,
                            mean_values - std_values,
                            mean_values + std_values,
                            color=line.get_color(),
                            alpha=STD_REGION_ALPHA,
                            linewidth=0,
                            label="_nolegend_",
                        )
                    plotted_lines += 1

                if not plotted_lines:
                    plt.close(figure)
                    print(
                        f"Skipping block_size={format_block_size(block_size)}: "
                        "no valid series."
                    )
                    continue

                axis.set_xlabel(
                    "Epoch (-1 = baseline)"
                    if INCLUDE_BASELINE
                    else "Epoch"
                )
                axis.set_ylabel("Accuracy drop (pp)")
                axis.set_title("Accuracy drop from baseline")
                style_2d_drop_axis(axis)
                deduplicated_legend(axis, loc="best")

                block_size_label = format_block_size(block_size)
                plot_path = (
                    plot_output_dir
                    / (
                        "accuracy_drop"
                        f"__run-{safe_run_identifier}"
                        f"__{safe_config_name}"
                        f"__block_size-{block_size_label}.png"
                    )
                )
                save_or_show_figure(figure, plot_path)

            # ====================================================
            # 3. Maximum mean accuracy drop, with details printed
            # ====================================================

            max_drop_df = calculate_maximum_accuracy_drops(
                dataframe=config_df,
                epoch_col=epoch_col,
                method_col=method_col,
                block_size_col=block_size_col,
            )

            if max_drop_df.empty:
                print("No maximum-drop summary could be calculated.")
                continue

            print("Maximum mean accuracy drop by method and block size:")
            print(
                max_drop_df[
                    [
                        "block_size",
                        "raw_method",
                        "method",
                        "epoch",
                        "mean_accuracy_drop_pp",
                        "std_accuracy_drop_pp",
                        "seed_count",
                    ]
                ].to_string(index=False)
            )

            if SAVE_PLOTS:
                max_drop_csv_path = (
                    plot_output_dir
                    / (
                        "maximum_accuracy_drop"
                        f"__run-{safe_run_identifier}"
                        f"__{safe_config_name}.csv"
                    )
                )
                max_drop_df.to_csv(max_drop_csv_path, index=False)
                print("Saved:", max_drop_csv_path)

            for block_size in sorted(max_drop_df["block_size"].unique()):
                block_result = (
                    max_drop_df.loc[
                        max_drop_df["block_size"] == block_size
                    ]
                    .sort_values("mean_accuracy_drop_pp", ascending=False)
                    .copy()
                )
                if block_result.empty:
                    continue

                figure, axis = plt.subplots(figsize=(7.5, 4.8))
                axis.bar(
                    block_result["method"],
                    block_result["mean_accuracy_drop_pp"],
                    yerr=block_result["std_accuracy_drop_pp"],
                    capsize=4,
                )
                axis.set_ylabel("Maximum mean accuracy drop (pp)")
                axis.set_title("Maximum accuracy drop")
                axis.grid(visible=True, axis="y", alpha=0.20)
                axis.spines["top"].set_visible(False)
                axis.spines["right"].set_visible(False)

                block_size_label = format_block_size(block_size)
                plot_path = (
                    plot_output_dir
                    / (
                        "maximum_accuracy_drop"
                        f"__run-{safe_run_identifier}"
                        f"__{safe_config_name}"
                        f"__block_size-{block_size_label}.png"
                    )
                )
                save_or_show_figure(figure, plot_path)

print()
print("=" * 100)
print("Finished processing the current sweep only.")


In [ ]:
from pathlib import Path
import re
import json
import hashlib

import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection


# ============================================================
# Configuration
# ============================================================

SWEEP_ROOT = Path("extendedPlotting") / "sweeps"

# Plot only the sweep created by the current notebook run.
#
# In the sweep runner, OUT_DIR should point to the active sweep folder,
# for example:
#   extendedPlotting/sweeps/global_prbcd_...__20260721_153440
#
# Set this manually to a folder path only when the plotting cell is run
# independently of the sweep cell.
CURRENT_SWEEP_DIR = globals().get("OUT_DIR", None)

# RUN_GROUP_ID is used to exclude unrelated rows if an epochs.csv happens
# to contain data from more than one sweep.
CURRENT_SWEEP_ID = str(
    globals().get("RUN_GROUP_ID", "")
).strip()

# Epoch -1 represents the pre-attack baseline.
INCLUDE_BASELINE = True

SAVE_PLOTS = True
SHOW_PLOTS = True

# Fixed accuracy range for epoch-accuracy plots.
ACCURACY_Y_MIN = 0.75
ACCURACY_Y_MAX = 0.90

# Show mean ± one standard deviation across seeds.
SHOW_STD_REGION = True
STD_REGION_ALPHA = 0.20

# Exclude selected methods from all plots.
EXCLUDED_METHODS = {
    "accuracy_drop_selector_k_hop",
}

SIGNATURE_JSON_NAME = "sweep_signature.json"


# ============================================================
# Possible column names
# ============================================================

EPOCH_CANDIDATES = [
    "epoch_idx",
    "epoch",
    "Epoch",
    "step",
]

ACCURACY_CANDIDATES = [
    "accuracy",
    "acc",
    "Accuracy",
]

METHOD_CANDIDATES = [
    "use_cert_label",
    "attack__use_cert",
    "use_cert",
]

BLOCK_SIZE_CANDIDATES = [
    "block_size",
    "attack__block_size",
    "attack_params__block_size",
    "attack_params.block_size",
]

RUN_GROUP_CANDIDATES = [
    "run_group_id",
    "sweep_id",
    "experiment_id",
]

DATASET_CANDIDATES = [
    "dataset",
]

SEED_CANDIDATES = [
    "seed",
    "experiment__seed",
    "experiment_params__seed",
    "experiment_params.seed",
    "log__seed",
    "log_values__seed",
    "log_values.seed",
]

CANDIDATE_SET_SIZE_CANDIDATES = [
    "candidate_set_size",
    "log_values__candidate_set_size",
    "log_values.candidate_set_size",
]

PRBCD_FRACTION_CANDIDATES = [
    "prbcd_candidate_fraction",
    "log_values__prbcd_candidate_fraction",
    "log_values.prbcd_candidate_fraction",
]

SCORING_MODE_CANDIDATES = [
    "scoring_mode",
    "log_values__scoring_mode",
    "log_values.scoring_mode",
]

ENDPOINT_MINING_HOP_CANDIDATES = [
    "endpoint_mining_hop",
    "log__endpoint_mining_hop",
    "log_values__endpoint_mining_hop",
    "log_values.endpoint_mining_hop",
    "h",
]

LABEL_MODE_CANDIDATES = [
    "label_mode",
    "log_values__label_mode",
    "log_values.label_mode",
]

EXTREME_FRACTION_CANDIDATES = [
    "extreme_fraction",
    "log_values__extreme_fraction",
    "log_values.extreme_fraction",
]

TAU_CANDIDATES = [
    "tau",
    "selector__tau",
    "selector_params__tau",
    "selector_params.tau",
    "attack__tau",
    "attack_params__tau",
    "attack_params.tau",
    "experiment__tau",
    "experiment.tau",
]

RESAMPLING_MODE_CANDIDATES = [
    "resampling_mode",
    "selector__resampling_mode",
    "selector_params__resampling_mode",
    "selector_params.resampling_mode",
    "attack__resampling_mode",
    "attack_params__resampling_mode",
    "attack_params.resampling_mode",
]

EPSILON_CANDIDATES = [
    "epsilon",
    "eps",
    "experiment__epsilon",
    "experiment.epsilon",
    "attack_params__epsilon",
    "attack_params.epsilon",
    "attack_params__eps",
    "attack_params.eps",
]

EPOCHS_TOTAL_CANDIDATES = [
    "epochs",
    "attack_params__epochs",
    "attack_params.epochs",
]

CANDIDATE_CONFIG_ID_CANDIDATES = [
    "candidate_config_id",
    "log_values__candidate_config_id",
    "log_values.candidate_config_id",
    "lp_model_label",
    "log_values__lp_model_label",
    "log_values.lp_model_label",
]


# ============================================================
# Helper functions
# ============================================================

def normalize_column_name(name):
    return (
        str(name)
        .lower()
        .replace(".", "_")
        .replace("-", "_")
        .replace(" ", "_")
    )


def find_first_existing_column(dataframe, candidates):
    return next(
        (
            column
            for column in candidates
            if column in dataframe.columns
        ),
        None,
    )


def find_column_by_suffix(dataframe, suffixes):
    normalized_suffixes = [
        normalize_column_name(suffix)
        for suffix in suffixes
    ]

    for column in dataframe.columns:
        normalized_column = normalize_column_name(column)

        for suffix in normalized_suffixes:
            if normalized_column.endswith(suffix):
                return column

    return None


def find_flexible_column(dataframe, candidates):
    column = find_first_existing_column(dataframe, candidates)

    if column is not None:
        return column

    return find_column_by_suffix(dataframe, candidates)


def find_block_size_column(dataframe):
    """
    Find the block-size column using known names or a normalized
    column name ending in 'block_size'.
    """
    column = find_first_existing_column(
        dataframe,
        BLOCK_SIZE_CANDIDATES,
    )

    if column is not None:
        return column

    return next(
        (
            column
            for column in dataframe.columns
            if normalize_column_name(column).endswith("block_size")
        ),
        None,
    )


def safe_filename(value):
    """
    Convert a run identifier into a filesystem-safe string.
    """
    value = str(value).strip()

    value = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        value,
    )

    return value.strip("._-") or "unknown_run"


def compact_safe_filename(value, max_length=90):
    """
    Make safe filenames short enough for operating systems.
    """
    safe = safe_filename(value)

    if len(safe) <= max_length:
        return safe

    digest = hashlib.sha1(
        safe.encode("utf-8")
    ).hexdigest()[:10]

    prefix_length = max(20, max_length - len(digest) - 2)

    return f"{safe[:prefix_length].rstrip('._-')}__{digest}"


def format_block_size(block_size):
    """
    Format block sizes without unnecessary decimal places.
    """
    value = float(block_size)

    if value.is_integer():
        return str(int(value))

    return str(value).replace(".", "p")


def normalize_method_name(value):
    return str(value).strip()


def is_excluded_method(method):
    return normalize_method_name(method) in EXCLUDED_METHODS


def attach_matching_none_baseline(
        all_df,
        run_df,
        method_col,
        block_size_col,
):
    """
    Add the matching ``use_cert=none`` rows to a method-specific run.

    ``run_group_id`` may identify individual configurations, including the
    method itself. Grouping by the full run id therefore separates the attack
    run from its ``none`` baseline. This helper restores the comparison by
    matching baseline rows on experiment properties that must be shared.
    """
    if method_col is None:
        return run_df.copy()

    run_method_labels = {
        normalize_method_name(value).lower()
        for value in run_df[method_col].dropna().unique()
    }

    # The baseline is already present, so do not add it twice.
    if "none" in run_method_labels:
        return run_df.copy()

    none_mask = (
        all_df[method_col]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("none")
    )

    none_df = all_df.loc[none_mask].copy()

    if none_df.empty:
        print("Warning: the current sweep contains no 'none' rows.")
        return run_df.copy()

    dataset_col = find_flexible_column(
        all_df,
        DATASET_CANDIDATES,
    )

    seed_col = find_flexible_column(
        all_df,
        SEED_CANDIDATES,
    )

    epsilon_col = find_flexible_column(
        all_df,
        EPSILON_CANDIDATES,
    )

    epochs_total_col = find_flexible_column(
        all_df,
        EPOCHS_TOTAL_CANDIDATES,
    )

    match_columns = []

    for column in [
        dataset_col,
        seed_col,
        block_size_col,
        epsilon_col,
        epochs_total_col,
    ]:
        if (
            column is not None
            and column in all_df.columns
            and column not in match_columns
        ):
            match_columns.append(column)

    # block_size_col is always available by this point, so this normally
    # performs a precise merge rather than attaching every baseline row.
    comparison_keys = (
        run_df[match_columns]
        .drop_duplicates()
    )

    matching_none_df = none_df.merge(
        comparison_keys,
        on=match_columns,
        how="inner",
    )

    if matching_none_df.empty:
        print(
            "Warning: no matching 'none' baseline found for run values "
            f"on columns {match_columns}."
        )
        return run_df.copy()

    print(
        f"Added {len(matching_none_df)} matching 'none' row(s) "
        f"using {match_columns}."
    )

    return pd.concat(
        [
            run_df.copy(),
            matching_none_df,
        ],
        ignore_index=True,
        sort=False,
    )


def display_encoded_value(value):
    """
    Convert folder-safe values such as 0p25 back to compact display values.
    Keeps categorical values unchanged.
    """
    if value is None:
        return ""

    value = str(value)

    if value in ("", "na"):
        return "na"

    parts = [
        part
        for part in value.split("+")
        if part != ""
    ]

    if not parts:
        return "na"

    decoded_parts = []

    for part in parts:
        decoded = part.replace("p", ".")

        decoded_parts.append(decoded)

    return ", ".join(decoded_parts)


def shorten_text(text, max_chars=115):
    text = str(text)

    if len(text) <= max_chars:
        return text

    return text[: max_chars - 3].rstrip() + "..."


def parse_folder_signature(signature):
    """
    Parse strings like:
        block_size-4000__candN-1000+2500__tau-0p8

    into:
        {"block_size": "4000", "candN": "1000+2500", "tau": "0p8"}
    """
    fields = {}

    if not signature:
        return fields

    for part in str(signature).split("__"):
        if "-" not in part:
            continue

        key, value = part.split("-", 1)
        fields[key] = value

    return fields


def load_sweep_metadata(sweep_dir):
    """
    Load sweep_signature.json from a sweep folder, if present.
    Works with both new hashed folders and older folders without JSON.
    """
    json_path = sweep_dir / SIGNATURE_JSON_NAME

    metadata = {
        "exists": False,
        "path": str(json_path),
        "raw": {},
        "fields": {},
        "error": "",
    }

    if not json_path.exists():
        return metadata

    try:
        with open(json_path, "r", encoding="utf-8") as handle:
            raw = json.load(handle)

        full_signature = raw.get("full_folder_signature", "")
        fields = parse_folder_signature(full_signature)

        metadata.update({
            "exists": True,
            "raw": raw,
            "fields": fields,
        })

    except Exception as exc:
        metadata["error"] = f"{type(exc).__name__}: {exc}"

    return metadata


def unique_values_from_column(dataframe, candidates, max_values=12):
    column = find_flexible_column(dataframe, candidates)

    if column is None:
        return None

    values = (
        dataframe[column]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    if not values:
        return None

    values = sorted(values)

    if len(values) > max_values:
        shown = values[:max_values]
        return ", ".join(shown) + f", ... (+{len(values) - max_values})"

    return ", ".join(values)


def get_context_value(run_df, candidates, sweep_fields, sweep_key):
    """
    Prefer run-specific values from epochs.csv.
    Fall back to sweep-wide values from sweep_signature.json.
    """
    run_value = unique_values_from_column(run_df, candidates)

    if run_value not in (None, ""):
        return run_value

    sweep_value = sweep_fields.get(sweep_key)

    if sweep_value not in (None, ""):
        return display_encoded_value(sweep_value)

    return None


def build_plot_context_text(
        csv_path,
        run_df,
        sweep_metadata,
):
    """
    Create a compact text box with experimental metadata.

    The seed information is derived from the rows actually used for
    the plot. When multiple seeds are present, the plot is explicitly
    marked as a seed-averaged plot.
    """
    raw = sweep_metadata.get(
        "raw",
        {},
    )

    fields = sweep_metadata.get(
        "fields",
        {},
    )

    dataset = (
        raw.get("dataset")
        or unique_values_from_column(
            run_df,
            DATASET_CANDIDATES,
        )
        or "unknown"
    )

    signature_hash = (
        raw.get("signature_hash")
        or raw.get("folder_signature")
        or "no-signature-json"
    )

    # ========================================================
    # Determine the actual seeds represented in this plot
    # ========================================================

    seed_candidates = [
        "seed",
        "experiment__seed",
        "experiment_params__seed",
        "experiment_params.seed",
        "log__seed",
        "log_values__seed",
        "log_values.seed",
    ]

    seed_column = find_flexible_column(
        run_df,
        seed_candidates,
    )

    seed_label = None

    if seed_column is not None:
        seed_values = (
            run_df[seed_column]
            .dropna()
            .unique()
            .tolist()
        )

        def seed_sort_key(value):
            try:
                return float(value)
            except (TypeError, ValueError):
                return str(value)

        seed_values = sorted(
            seed_values,
            key=seed_sort_key,
        )

        seed_count = len(
            seed_values
        )

        if seed_count > 1:
            seed_label = (
                f"mean over {seed_count} seeds"
            )

        elif seed_count == 1:
            seed_value = seed_values[0]

            try:
                numeric_seed = float(
                    seed_value
                )

                if numeric_seed.is_integer():
                    seed_value = int(
                        numeric_seed
                    )

            except (TypeError, ValueError):
                pass

            seed_label = (
                f"single seed: {seed_value}"
            )

    # ========================================================
    # Construct context entries
    # ========================================================

    items = [
        (
            "dataset",
            dataset,
        ),
        (
            "sig",
            signature_hash,
        ),
        (
            "seed aggregation",
            seed_label,
        ),
        (
            "candN",
            get_context_value(
                run_df,
                CANDIDATE_SET_SIZE_CANDIDATES,
                fields,
                "candN",
            ),
        ),
        (
            "prbcdFrac",
            get_context_value(
                run_df,
                PRBCD_FRACTION_CANDIDATES,
                fields,
                "prbcdFrac",
            ),
        ),
        (
            "scoring",
            get_context_value(
                run_df,
                SCORING_MODE_CANDIDATES,
                fields,
                "scoringMode",
            ),
        ),
        (
            "h",
            get_context_value(
                run_df,
                ENDPOINT_MINING_HOP_CANDIDATES,
                fields,
                "h",
            ),
        ),
        (
            "label",
            get_context_value(
                run_df,
                LABEL_MODE_CANDIDATES,
                fields,
                "labelMode",
            ),
        ),
        (
            "extremeFrac",
            get_context_value(
                run_df,
                EXTREME_FRACTION_CANDIDATES,
                fields,
                "extremeFrac",
            ),
        ),
        (
            "tau",
            get_context_value(
                run_df,
                TAU_CANDIDATES,
                fields,
                "tau",
            ),
        ),
        (
            "resampling",
            get_context_value(
                run_df,
                RESAMPLING_MODE_CANDIDATES,
                fields,
                "resamplingMode",
            ),
        ),
        (
            "eps",
            get_context_value(
                run_df,
                EPSILON_CANDIDATES,
                fields,
                "epsilon",
            ),
        ),
        (
            "epochs",
            get_context_value(
                run_df,
                EPOCHS_TOTAL_CANDIDATES,
                fields,
                "epochs",
            ),
        ),
        (
            "candidate_cfg",
            unique_values_from_column(
                run_df,
                CANDIDATE_CONFIG_ID_CANDIDATES,
            ),
        ),
    ]

    rendered = []

    for key, value in items:
        if value is None:
            continue

        value_text = str(
            value
        ).strip()

        if value_text.lower() in {
            "",
            "nan",
            "none",
        }:
            continue

        rendered.append(
            f"{key}="
            f"{shorten_text(value_text, max_chars=70)}"
        )

    if not rendered:
        return ""

    # Split context across lines to prevent overly wide boxes.
    lines = []
    current_line = ""

    for item in rendered:
        candidate_line = (
            item
            if not current_line
            else current_line + " | " + item
        )

        if len(candidate_line) <= 115:
            current_line = candidate_line

        else:
            if current_line:
                lines.append(
                    current_line
                )

            current_line = item

    if current_line:
        lines.append(
            current_line
        )

    return "\n".join(
        lines
    )


def add_run_label(axis, run_identifier):
    """
    Add a visible run identifier inside a 2D plot.
    """
    axis.text(
        0.01,
        0.01,
        f"Run: {run_identifier}",
        transform=axis.transAxes,
        fontsize=8,
        alpha=0.75,
        horizontalalignment="left",
        verticalalignment="bottom",
    )


def add_context_box(axis, context_text):
    """
    Add metadata from sweep_signature.json / epochs.csv into the plot.
    Works for both 2D and 3D axes.
    """
    if not context_text:
        return

    box_style = {
        "boxstyle": "round,pad=0.35",
        "facecolor": "white",
        "edgecolor": "none",
        "alpha": 0.70,
    }

    if hasattr(axis, "text2D"):
        axis.text2D(
            0.01,
            0.98,
            context_text,
            transform=axis.transAxes,
            fontsize=8,
            alpha=0.90,
            horizontalalignment="left",
            verticalalignment="top",
            bbox=box_style,
        )
    else:
        axis.text(
            0.01,
            0.98,
            context_text,
            transform=axis.transAxes,
            fontsize=8,
            alpha=0.90,
            horizontalalignment="left",
            verticalalignment="top",
            bbox=box_style,
        )


def save_or_show_figure(figure, plot_path):
    figure.tight_layout()

    if SAVE_PLOTS:
        figure.savefig(
            plot_path,
            dpi=200,
            bbox_inches="tight",
        )

        print("Saved:", plot_path)

    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(figure)



def aggregate_accuracy_across_seeds(
        dataframe,
        epoch_col,
        accuracy_col,
):
    """
    Return epoch-wise mean and standard deviation across seeds.

    Duplicate rows within the same seed and epoch are averaged first so that
    every seed has equal weight. If no seed column exists, the function falls
    back to aggregating all rows per epoch.

    The returned frame contains:
        mean, std, seed_count
    """
    seed_col = find_flexible_column(
        dataframe,
        SEED_CANDIDATES,
    )

    required_columns = [
        epoch_col,
        accuracy_col,
    ]

    if seed_col is not None:
        required_columns.append(seed_col)

    work_df = (
        dataframe[required_columns]
        .dropna(
            subset=[
                epoch_col,
                accuracy_col,
            ]
        )
        .copy()
    )

    if work_df.empty:
        return pd.DataFrame(
            columns=[
                "mean",
                "std",
                "seed_count",
            ]
        )

    if seed_col is not None:
        # First collapse possible duplicate measurements within each seed.
        per_seed = (
            work_df
            .groupby(
                [
                    seed_col,
                    epoch_col,
                ],
                dropna=False,
            )[accuracy_col]
            .mean()
            .reset_index()
        )

        summary = (
            per_seed
            .groupby(
                epoch_col,
                dropna=False,
            )[accuracy_col]
            .agg(
                mean="mean",
                std="std",
                seed_count="count",
            )
            .sort_index()
        )

    else:
        summary = (
            work_df
            .groupby(
                epoch_col,
                dropna=False,
            )[accuracy_col]
            .agg(
                mean="mean",
                std="std",
                seed_count="count",
            )
            .sort_index()
        )

    # pandas returns NaN for sample std when only one seed is available.
    summary["std"] = (
        summary["std"]
        .fillna(0.0)
    )

    return summary


def add_std_ribbon_3d(
        axis,
        epoch_values,
        mean_values,
        std_values,
        block_size,
        color,
):
    """
    Add a translucent mean ± std ribbon at a constant block-size plane.
    """
    if (
        not SHOW_STD_REGION
        or len(epoch_values) < 2
        or not any(value > 0 for value in std_values)
    ):
        return

    lower_values = mean_values - std_values
    upper_values = mean_values + std_values

    vertices = [
        list(
            zip(
                epoch_values,
                lower_values,
                [float(block_size)] * len(epoch_values),
            )
        )
        + list(
            zip(
                epoch_values[::-1],
                upper_values[::-1],
                [float(block_size)] * len(epoch_values),
            )
        )
    ]

    ribbon = Poly3DCollection(
        vertices,
        facecolor=color,
        edgecolor="none",
        alpha=STD_REGION_ALPHA,
    )

    axis.add_collection3d(
        ribbon
    )


def calculate_max_accuracy_differences(
    run_df,
    epoch_col,
    accuracy_col,
    method_col,
    block_size_col,
):
    """
    For every block size and every non-'none' method:

    1. Average duplicate rows for each epoch and method.
    2. Align the method with 'none' by epoch.
    3. Calculate:

           signed_difference = none_accuracy - method_accuracy

    4. Select the epoch with the largest absolute difference.

    Interpretation:
        positive -> use_cert achieved lower accuracy than none
        negative -> none achieved lower accuracy than use_cert
    """
    rows = []

    if method_col is None:
        return pd.DataFrame()

    for block_size, block_df in run_df.groupby(
        block_size_col,
        dropna=False,
    ):
        if pd.isna(block_size):
            continue

        averaged = (
            block_df
            .groupby(
                [epoch_col, method_col],
                dropna=False,
            )[accuracy_col]
            .mean()
            .reset_index()
        )

        pivot = averaged.pivot(
            index=epoch_col,
            columns=method_col,
            values=accuracy_col,
        )

        none_column = next(
            (
                column
                for column in pivot.columns
                if str(column).strip().lower() == "none"
            ),
            None,
        )

        if none_column is None:
            print(
                f"Skipping difference analysis for "
                f"block_size={format_block_size(block_size)}: "
                "no 'none' method found."
            )
            continue

        for method in pivot.columns:
            method_label = normalize_method_name(method)

            if method == none_column:
                continue

            if is_excluded_method(method_label):
                continue

            comparison = pivot[
                [none_column, method]
            ].dropna()

            if comparison.empty:
                continue

            signed_difference = (
                comparison[none_column]
                - comparison[method]
            )

            max_epoch = signed_difference.abs().idxmax()

            none_accuracy = float(
                comparison.loc[max_epoch, none_column]
            )

            method_accuracy = float(
                comparison.loc[max_epoch, method]
            )

            max_signed_difference = float(
                signed_difference.loc[max_epoch]
            )

            rows.append({
                "block_size": float(block_size),
                "use_cert": method_label,
                "epoch": int(max_epoch),
                "none_accuracy": none_accuracy,
                "use_cert_accuracy": method_accuracy,
                "signed_difference": max_signed_difference,
                "absolute_difference": abs(max_signed_difference),
                "better_method": (
                    method_label
                    if max_signed_difference > 0
                    else "none"
                    if max_signed_difference < 0
                    else "equal"
                ),
            })

    return pd.DataFrame(rows)


# ============================================================
# Resolve the epochs.csv belonging to the current sweep only
# ============================================================

if CURRENT_SWEEP_DIR is None or str(CURRENT_SWEEP_DIR).strip() == "":
    raise RuntimeError(
        "OUT_DIR is not defined. Run the sweep cell first, or set "
        "CURRENT_SWEEP_DIR manually to the active sweep folder."
    )

current_sweep_dir = Path(CURRENT_SWEEP_DIR).expanduser()

# Allow CURRENT_SWEEP_DIR to be either the folder or epochs.csv itself.
if current_sweep_dir.name == "epochs.csv":
    current_epochs_csv = current_sweep_dir
    current_sweep_dir = current_sweep_dir.parent
else:
    current_epochs_csv = current_sweep_dir / "epochs.csv"

if not current_epochs_csv.exists():
    raise FileNotFoundError(
        "The current sweep has no epochs.csv:\n"
        f"  sweep folder: {current_sweep_dir.resolve()}\n"
        f"  expected CSV: {current_epochs_csv.resolve()}"
    )

csv_files = [current_epochs_csv]

print("Current sweep folder:", current_sweep_dir)
print("Current sweep id:", CURRENT_SWEEP_ID or "not available")
print("Processing only:", current_epochs_csv)


# ============================================================
# Process every epochs.csv
# ============================================================

for csv_path in csv_files:
    print()
    print("=" * 100)
    print("Processing:", csv_path)

    sweep_metadata = load_sweep_metadata(csv_path.parent)

    if sweep_metadata["exists"]:
        print("Sweep metadata:", sweep_metadata["path"])
        print("Signature hash:", sweep_metadata["raw"].get("signature_hash", ""))
        print("Parsed signature fields:", sweep_metadata["fields"])
    else:
        print("Sweep metadata: no sweep_signature.json found.")
        if sweep_metadata["error"]:
            print("Metadata error:", sweep_metadata["error"])

    df = pd.read_csv(csv_path)

    if df.empty:
        print("Skipping empty file.")
        continue

    epoch_col = find_first_existing_column(
        df,
        EPOCH_CANDIDATES,
    )

    accuracy_col = find_first_existing_column(
        df,
        ACCURACY_CANDIDATES,
    )

    method_col = find_first_existing_column(
        df,
        METHOD_CANDIDATES,
    )

    block_size_col = find_block_size_column(df)

    run_group_col = find_first_existing_column(
        df,
        RUN_GROUP_CANDIDATES,
    )

    # Keep only rows belonging to the current sweep. Configuration-level
    # run_group_id values may extend RUN_GROUP_ID with "__...", so prefix
    # matching is intentional.
    if CURRENT_SWEEP_ID and run_group_col is not None:
        current_sweep_mask = (
            df[run_group_col]
            .astype(str)
            .str.startswith(CURRENT_SWEEP_ID)
        )

        if current_sweep_mask.any():
            skipped_rows = int((~current_sweep_mask).sum())
            df = df.loc[current_sweep_mask].copy()

            if skipped_rows:
                print(
                    f"Filtered out {skipped_rows} row(s) not belonging "
                    f"to current sweep {CURRENT_SWEEP_ID}."
                )
        else:
            raise RuntimeError(
                f"No rows in {csv_path} match current RUN_GROUP_ID "
                f"{CURRENT_SWEEP_ID!r}."
            )

    if epoch_col is None:
        print("Skipping: no compatible epoch column found.")
        continue

    if accuracy_col is None:
        print("Skipping: no compatible accuracy column found.")
        continue

    if block_size_col is None:
        print("Skipping: no compatible block-size column found.")
        continue

    # --------------------------------------------------------
    # Convert relevant columns to numeric
    # --------------------------------------------------------

    df[epoch_col] = pd.to_numeric(
        df[epoch_col],
        errors="coerce",
    )

    df[accuracy_col] = pd.to_numeric(
        df[accuracy_col],
        errors="coerce",
    )

    df[block_size_col] = pd.to_numeric(
        df[block_size_col],
        errors="coerce",
    )

    df = df.dropna(
        subset=[
            epoch_col,
            accuracy_col,
            block_size_col,
        ]
    ).copy()

    if not INCLUDE_BASELINE:
        df = df[df[epoch_col] >= 0]

    if df.empty:
        print("Skipping: no valid rows remained.")
        continue

    df[epoch_col] = df[epoch_col].astype(int)

    # --------------------------------------------------------
    # Split CSV into run groups
    # --------------------------------------------------------

    if run_group_col is not None:
        run_groups = df.groupby(
            run_group_col,
            dropna=False,
        )
    else:
        run_groups = [
            (
                csv_path.parent.name,
                df,
            )
        ]

    # ========================================================
    # Process every run group separately
    # ========================================================

    for run_group_value, run_df in run_groups:
        if pd.isna(run_group_value):
            run_identifier = csv_path.parent.name
        else:
            run_identifier = str(run_group_value)

        if method_col is not None:
            original_method_labels = {
                normalize_method_name(value).lower()
                for value in run_df[method_col].dropna().unique()
            }

            # Do not create a redundant plot containing only the baseline.
            # The same rows are attached to every non-none run below.
            if original_method_labels == {"none"}:
                print(
                    f"Skipping standalone none run {run_identifier}; "
                    "it is used as the comparison baseline."
                )
                continue

            run_df = attach_matching_none_baseline(
                all_df=df,
                run_df=run_df,
                method_col=method_col,
                block_size_col=block_size_col,
            )

        safe_run_identifier = compact_safe_filename(
            run_identifier
        )

        block_sizes = sorted(
            run_df[block_size_col]
            .dropna()
            .unique()
            .tolist()
        )

        plot_output_dir = RUN_PLOTS_DIR

        if SAVE_PLOTS:
            plot_output_dir.mkdir(
                parents=True,
                exist_ok=True,
            )

        plot_context_text = build_plot_context_text(
            csv_path=csv_path,
            run_df=run_df,
            sweep_metadata=sweep_metadata,
        )

        print()
        print("-" * 100)
        print("Run:", run_identifier)
        print(
            "Block sizes:",
            [
                format_block_size(value)
                for value in block_sizes
            ],
        )
        print("Plot context:")
        print(plot_context_text or "na")
        print("Output folder:", plot_output_dir)

        if SAVE_PLOTS:
            plot_metadata_path = plot_output_dir / "plot_metadata.json"

            with open(plot_metadata_path, "w", encoding="utf-8") as handle:
                json.dump(
                    {
                        "csv_path": str(csv_path),
                        "run_identifier": run_identifier,
                        "safe_run_identifier": safe_run_identifier,
                        "block_sizes": [
                            format_block_size(value)
                            for value in block_sizes
                        ],
                        "plot_context_text": plot_context_text,
                        "sweep_metadata": sweep_metadata,
                    },
                    handle,
                    indent=2,
                    ensure_ascii=False,
                )

            print("Saved:", plot_metadata_path)

        # ====================================================
        # 1. 3D plot when the run has multiple block sizes
        # ====================================================

        if len(block_sizes) > 1:
            figure = plt.figure(
                figsize=(14, 8)
            )

            axis = figure.add_subplot(
                111,
                projection="3d",
            )

            plotted_lines = 0

            if method_col is not None:
                grouped_series = run_df.groupby(
                    [
                        method_col,
                        block_size_col,
                    ],
                    dropna=False,
                )
            else:
                grouped_series = run_df.groupby(
                    block_size_col,
                    dropna=False,
                )

            for group_key, group in grouped_series:
                if method_col is not None:
                    method, block_size = group_key
                    method_label = normalize_method_name(method)
                else:
                    block_size = group_key
                    method_label = "accuracy"

                if is_excluded_method(method_label):
                    continue

                if pd.isna(block_size):
                    continue

                summary = aggregate_accuracy_across_seeds(
                    dataframe=group,
                    epoch_col=epoch_col,
                    accuracy_col=accuracy_col,
                )

                if summary.empty:
                    continue

                epoch_values = summary.index.to_numpy()
                accuracy_values = summary["mean"].to_numpy()
                std_values = summary["std"].to_numpy()

                block_size_values = [
                    float(block_size)
                ] * len(summary)

                line, = axis.plot(
                    epoch_values,
                    accuracy_values,
                    block_size_values,
                    label=(
                        f"{method_label} | "
                        f"block_size="
                        f"{format_block_size(block_size)}"
                    ),
                )

                add_std_ribbon_3d(
                    axis=axis,
                    epoch_values=epoch_values,
                    mean_values=accuracy_values,
                    std_values=std_values,
                    block_size=block_size,
                    color=line.get_color(),
                )

                plotted_lines += 1

            if plotted_lines > 0:
                axis.set_xlabel(
                    "Epoch (-1 = pre-attack baseline)"
                    if INCLUDE_BASELINE
                    else "Epoch"
                )

                axis.set_ylabel("Accuracy")
                axis.set_zlabel("Block size")

                axis.set_title(
                    "Accuracy trajectories across block sizes "
                    "(mean ± 1 SD across seeds)"
                    if SHOW_STD_REGION
                    else "Accuracy trajectories across block sizes"
                )

                axis.set_ylim(
                    ACCURACY_Y_MIN,
                    ACCURACY_Y_MAX,
                )

                axis.set_yticks([
                    0.75,
                    0.80,
                    0.85,
                    0.90,
                ])

                axis.set_zticks(block_sizes)

                axis.view_init(
                    elev=25,
                    azim=-60,
                )

                axis.grid(
                    visible=True,
                    alpha=0.25,
                )

                axis.legend(
                    title=method_col or "series",
                    loc="upper left",
                    bbox_to_anchor=(1.02, 1.0),
                )

                add_context_box(
                    axis,
                    plot_context_text,
                )

                plot_path = (
                    plot_output_dir
                    / (
                        "epoch_accuracy_3d"
                        f"__run-{safe_run_identifier}.png"
                    )
                )

                save_or_show_figure(
                    figure,
                    plot_path,
                )

            else:
                plt.close(figure)

                print(
                    "No valid series remained for the 3D plot."
                )

        # ====================================================
        # 2. Separate epoch-accuracy plot for each block size
        # ====================================================

        for block_size in block_sizes:
            block_df = run_df[
                run_df[block_size_col] == block_size
            ].copy()

            if block_df.empty:
                continue

            figure, axis = plt.subplots(
                figsize=(12, 7)
            )

            plotted_lines = 0

            if method_col is not None:
                method_groups = block_df.groupby(
                    method_col,
                    dropna=False,
                )

                for method, method_df in method_groups:
                    method_label = normalize_method_name(method)

                    if is_excluded_method(method_label):
                        continue

                    summary = aggregate_accuracy_across_seeds(
                        dataframe=method_df,
                        epoch_col=epoch_col,
                        accuracy_col=accuracy_col,
                    )

                    if summary.empty:
                        continue

                    epoch_values = summary.index.to_numpy()
                    mean_values = summary["mean"].to_numpy()
                    std_values = summary["std"].to_numpy()

                    line, = axis.plot(
                        epoch_values,
                        mean_values,
                        label=method_label,
                    )

                    if SHOW_STD_REGION:
                        axis.fill_between(
                            epoch_values,
                            mean_values - std_values,
                            mean_values + std_values,
                            color=line.get_color(),
                            alpha=STD_REGION_ALPHA,
                            linewidth=0,
                            label="_nolegend_",
                        )

                    plotted_lines += 1

                if plotted_lines > 0:
                    axis.legend(
                        title=method_col,
                        loc="best",
                    )

            else:
                summary = aggregate_accuracy_across_seeds(
                    dataframe=block_df,
                    epoch_col=epoch_col,
                    accuracy_col=accuracy_col,
                )

                if not summary.empty:
                    epoch_values = summary.index.to_numpy()
                    mean_values = summary["mean"].to_numpy()
                    std_values = summary["std"].to_numpy()

                    line, = axis.plot(
                        epoch_values,
                        mean_values,
                    )

                    if SHOW_STD_REGION:
                        axis.fill_between(
                            epoch_values,
                            mean_values - std_values,
                            mean_values + std_values,
                            color=line.get_color(),
                            alpha=STD_REGION_ALPHA,
                            linewidth=0,
                            label="_nolegend_",
                        )

                    plotted_lines += 1

            if plotted_lines == 0:
                plt.close(figure)

                print(
                    f"Skipping block_size="
                    f"{format_block_size(block_size)}: "
                    "no valid series."
                )
                continue

            block_size_label = format_block_size(
                block_size
            )

            axis.set_xlabel(
                "Epoch (-1 = pre-attack baseline)"
                if INCLUDE_BASELINE
                else "Epoch"
            )

            axis.set_ylabel("Accuracy")

            axis.set_title(
                "Accuracy trajectory "
                "(mean ± 1 SD across seeds)"
                if SHOW_STD_REGION
                else "Accuracy trajectory"
            )

            axis.set_ylim(
                ACCURACY_Y_MIN,
                ACCURACY_Y_MAX,
            )

            axis.set_yticks([
                0.60,
                0.65,
                0.70,
                0.75,
                0.80,
                0.85,
                0.90,
            ])

            axis.grid(
                visible=True,
                alpha=0.25,
            )

            add_run_label(
                axis,
                run_identifier,
            )

            add_context_box(
                axis,
                plot_context_text,
            )

            plot_path = (
                plot_output_dir
                / (
                    "epoch_accuracy"
                    f"__run-{safe_run_identifier}"
                    f"__block_size-{block_size_label}.png"
                )
            )

            save_or_show_figure(
                figure,
                plot_path,
            )

        # ====================================================
        # 3. Maximum difference against "none"
        #    Only for runs with multiple block sizes
        # ====================================================

        if len(block_sizes) > 1 and method_col is not None:
            max_difference_df = (
                calculate_max_accuracy_differences(
                    run_df=run_df,
                    epoch_col=epoch_col,
                    accuracy_col=accuracy_col,
                    method_col=method_col,
                    block_size_col=block_size_col,
                )
            )

            if max_difference_df.empty:
                print(
                    "No valid none-vs-use_cert comparisons "
                    f"found for run {run_identifier}."
                )

            else:
                # Save all numerical difference results.
                difference_csv_path = (
                    plot_output_dir
                    / (
                        "max_accuracy_differences"
                        f"__run-{safe_run_identifier}.csv"
                    )
                )

                if SAVE_PLOTS:
                    max_difference_df.to_csv(
                        difference_csv_path,
                        index=False,
                    )

                    print(
                        "Saved:",
                        difference_csv_path,
                    )

                # --------------------------------------------
                # One difference plot for each block size
                # --------------------------------------------

                for block_size in sorted(
                    max_difference_df[
                        "block_size"
                    ].unique()
                ):
                    block_result = (
                        max_difference_df[
                            max_difference_df["block_size"]
                            == block_size
                        ]
                        .sort_values(
                            "absolute_difference",
                            ascending=True,
                        )
                        .copy()
                    )

                    if block_result.empty:
                        continue

                    figure_height = max(
                        6,
                        0.65 * len(block_result) + 3,
                    )

                    figure, axis = plt.subplots(
                        figsize=(13, figure_height)
                    )

                    methods = (
                        block_result["use_cert"]
                        .astype(str)
                        .tolist()
                    )

                    signed_differences = (
                        block_result[
                            "signed_difference"
                        ]
                        .to_numpy()
                    )

                    bars = axis.barh(
                        methods,
                        signed_differences,
                    )

                    axis.axvline(
                        0,
                        linewidth=1,
                        alpha=0.75,
                    )

                    largest_absolute_difference = max(
                        float(
                            block_result[
                                "absolute_difference"
                            ].max()
                        ),
                        0.001,
                    )

                    annotation_offset = (
                        largest_absolute_difference * 0.035
                    )

                    for bar, row in zip(
                        bars,
                        block_result.itertuples(
                            index=False
                        ),
                    ):
                        width = float(bar.get_width())

                        if width >= 0:
                            annotation_x = (
                                width + annotation_offset
                            )
                            horizontal_alignment = "left"
                        else:
                            annotation_x = (
                                width - annotation_offset
                            )
                            horizontal_alignment = "right"

                        axis.text(
                            annotation_x,
                            (
                                bar.get_y()
                                + bar.get_height() / 2
                            ),
                            (
                                f"{width:+.4f} | "
                                f"epoch {row.epoch} | "
                                f"none={row.none_accuracy:.4f} | "
                                f"method={row.use_cert_accuracy:.4f}"
                            ),
                            horizontalalignment=(
                                horizontal_alignment
                            ),
                            verticalalignment="center",
                            fontsize=9,
                        )

                    block_size_label = format_block_size(
                        block_size
                    )

                    axis.set_xlabel(
                        "Largest signed accuracy difference "
                        "(none accuracy − use_cert accuracy)"
                    )

                    axis.set_ylabel("use_cert")

                    axis.set_title(
                        "Largest accuracy difference against the baseline"
                    )

                    axis.grid(
                        visible=True,
                        axis="x",
                        alpha=0.25,
                    )

                    axis.margins(
                        x=0.30,
                    )

                    axis.text(
                        0.01,
                        0.01,
                        (
                            "Positive: use_cert produced lower accuracy "
                            "than none\n"
                            "Negative: none produced lower accuracy "
                            "than use_cert"
                        ),
                        transform=axis.transAxes,
                        fontsize=8,
                        alpha=0.75,
                        horizontalalignment="left",
                        verticalalignment="bottom",
                    )

                    add_context_box(
                        axis,
                        plot_context_text,
                    )

                    plot_path = (
                        plot_output_dir
                        / (
                            "max_accuracy_difference_vs_none"
                            f"__run-{safe_run_identifier}"
                            f"__block_size-{block_size_label}.png"
                        )
                    )

                    save_or_show_figure(
                        figure,
                        plot_path,
                    )

print()
print("=" * 100)
print("Finished processing the current sweep only.")

---
## §9  RQ1 — When Does PR-BCD Miss Harmful Edges?

This experiment constructs an **empirical high-budget PR-BCD reference** and compares ordinary PR-BCD runs across candidate block sizes.

The cell temporarily instruments PR-BCD at runtime and records:

- the initial candidate block;
- candidate blocks at every epoch;
- blocks immediately before and after resampling;
- accumulated positive gradients;
- maximum relaxed perturbation weights;
- final discrete perturbations;
- victim accuracy and runtime.

The optimization itself is unchanged. Reference and evaluation runs use separate PR-BCD sampling seeds so that small evaluation blocks are not trivial prefixes of the high-budget reference block.


In [ ]:
# ============================================================
# CELL 1: DATA GENERATION
#
# MULTISEED BLOCK-SIZE × RESAMPLING-DURATION EXPERIMENT
#
# This cell:
#   1. runs all PRBCD conditions,
#   2. records run-level and epoch-level metrics,
#   3. aggregates repeats and victim seeds,
#   4. saves everything as CSV files,
#   5. does not create plots.
#
# Definition:
#   n_resampling_epochs = EPOCHS - fine_tune_epochs
# ============================================================

from datetime import datetime
from pathlib import Path
from timeit import default_timer as timer

import gc
import inspect
import json
import os

import numpy as np
import pandas as pd
import torch

from IPython.display import display
from experiments import experiment_global_attack_direct
from rgnn_at_scale.attacks.prbcd import PRBCD
from sparse_smoothing.utils import load_and_standardize


# ============================================================
# Configuration
# ============================================================

BLOCK_EXPERIMENT_EPSILON = 0.1

BLOCK_SIZES = [
    1_000,
    5_000,
    20_000,
    50_000,
]

# Total optimization budget.
EPOCHS = 300

# Exploration/resampling durations.
N_RESAMPLING_EPOCHS_VALUES = [
    25,
    50,
    100,
    150,
    200,
]

# Corresponding fine-tuning durations are:
#
#   275, 250, 200, 150, 100
#
# because:
#
#   fine_tune_epochs = EPOCHS - n_resampling_epochs

REPEATS_PER_SEED = 1

# Disable early stopping so every run receives its assigned
# number of exploration and fine-tuning epochs.
WITH_EARLY_STOPPING = False

ARTIFACT_DIR = "cache"
PERT_ADJ_STORAGE_TYPE = "evasion_global_adj"
PERT_ATTR_STORAGE_TYPE = "evasion_global_attr"

BASE_OUT_DIR = (
    Path("extendedPlotting")
    / "prbcd_blocksize_resampling_multiseed"
)


# ============================================================
# Validate configuration
# ============================================================

if not BLOCK_SIZES:
    raise ValueError("BLOCK_SIZES must not be empty.")

if not N_RESAMPLING_EPOCHS_VALUES:
    raise ValueError(
        "N_RESAMPLING_EPOCHS_VALUES must not be empty."
    )

invalid_resampling_values = [
    value
    for value in N_RESAMPLING_EPOCHS_VALUES
    if value < 0 or value > EPOCHS
]

if invalid_resampling_values:
    raise ValueError(
        "Every n_resampling_epochs value must be between "
        f"0 and EPOCHS={EPOCHS}. Invalid values: "
        f"{invalid_resampling_values}"
    )

if len(set(N_RESAMPLING_EPOCHS_VALUES)) != len(
    N_RESAMPLING_EPOCHS_VALUES
):
    raise ValueError(
        "N_RESAMPLING_EPOCHS_VALUES contains duplicates."
    )


# ============================================================
# Verify patched PRBCD sampling-seed support
#
# No full-space reference attack is performed.
# ============================================================

required_sampler_args = {
    "rq1_enabled",
    "rq1_is_reference",
    "rq1_sampling_seed",
}

available_prbcd_args = set(
    inspect.signature(PRBCD.__init__).parameters
)

missing_sampler_args = (
    required_sampler_args
    - available_prbcd_args
)

if missing_sampler_args:
    raise RuntimeError(
        "Loaded PRBCD does not expose the patched sampling-seed "
        f"arguments: {sorted(missing_sampler_args)}"
    )


# ============================================================
# Dataset-dependent quantities
# ============================================================

N_NODES = int(
    SEED_CONTEXTS[0]["n_nodes"]
)

N_UNDIRECTED_EDGES = int(
    SEED_CONTEXTS[0]["n_undirected"]
)

N_POSSIBLE_EDGES = (
    N_NODES * (N_NODES - 1) // 2
)

ATTACK_BUDGET = max(
    1,
    round(
        BLOCK_EXPERIMENT_EPSILON
        * N_UNDIRECTED_EDGES
    ),
)

if ATTACK_BUDGET >= N_POSSIBLE_EDGES:
    raise ValueError(
        "Attack budget must be smaller than the full "
        "possible-edge space."
    )

invalid_block_sizes = [
    block_size
    for block_size in BLOCK_SIZES
    if block_size <= ATTACK_BUDGET
]

if invalid_block_sizes:
    raise ValueError(
        "Every block size must exceed the attack budget. "
        f"Attack budget: {ATTACK_BUDGET}; "
        f"invalid block sizes: {invalid_block_sizes}"
    )


# ============================================================
# Load graph and create output directory
# ============================================================

graph_sparse = load_and_standardize(
    os.path.join(
        "data",
        f"{DATASET}.npz",
    )
)

RUN_ID = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

OUT_DIR = (
    BASE_OUT_DIR
    / f"{DATASET}__{RUN_ID}"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

BASE_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Save experiment configuration
# ============================================================

experiment_config = {
    "dataset": DATASET,
    "epsilon": BLOCK_EXPERIMENT_EPSILON,
    "block_sizes": BLOCK_SIZES,
    "epochs": EPOCHS,
    "n_resampling_epochs_values": (
        N_RESAMPLING_EPOCHS_VALUES
    ),
    "fine_tune_epochs_values": [
        EPOCHS - value
        for value in N_RESAMPLING_EPOCHS_VALUES
    ],
    "repeats_per_seed": REPEATS_PER_SEED,
    "with_early_stopping": WITH_EARLY_STOPPING,
    "victim_seeds": [
        int(seed)
        for seed in SEEDS
    ],
    "n_nodes": N_NODES,
    "n_undirected_edges": N_UNDIRECTED_EDGES,
    "n_possible_edges": N_POSSIBLE_EDGES,
    "attack_budget": ATTACK_BUDGET,
}

with open(
    OUT_DIR / "experiment_config.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        experiment_config,
        file,
        indent=2,
    )


# Store a pointer to the most recent result directory.
# The plotting cell can use this automatically.
(
    BASE_OUT_DIR
    / "latest_run.txt"
).write_text(
    str(OUT_DIR.resolve()),
    encoding="utf-8",
)


# ============================================================
# Print configuration
# ============================================================

print("Dataset:", DATASET)
print("Nodes:", N_NODES)
print("Undirected clean edges:", N_UNDIRECTED_EDGES)
print("Possible edge flips:", N_POSSIBLE_EDGES)
print("Attack budget:", ATTACK_BUDGET)
print("Block sizes:", BLOCK_SIZES)
print(
    "Resampling epoch values:",
    N_RESAMPLING_EPOCHS_VALUES,
)
print(
    "Fine-tuning epoch values:",
    [
        EPOCHS - value
        for value in N_RESAMPLING_EPOCHS_VALUES
    ],
)
print("Total epochs:", EPOCHS)
print("Victim seeds:", SEEDS)
print("Repeats per condition:", REPEATS_PER_SEED)
print("Early stopping:", WITH_EARLY_STOPPING)
print("Output directory:", OUT_DIR.resolve())


# ============================================================
# Helper functions
# ============================================================

def _to_scalar(
    value,
    default=np.nan,
):
    if value is None:
        return default

    try:
        return float(
            torch.as_tensor(value)
            .detach()
            .cpu()
            .item()
        )
    except Exception:
        return default


def _final_accuracy(result):
    """
    Extract the final attacked accuracy.
    """
    rows = result.get(
        "results",
        [],
    ) or []

    if not rows:
        return np.nan

    first_row = rows[0]

    if not isinstance(first_row, dict):
        return np.nan

    return _to_scalar(
        first_row.get("accuracy")
    )


def _clean_accuracy(result):
    """Extract the clean accuracy logged before PRBCD epoch 0."""
    statistics = (
        result.get(
            "attack_statistics",
            {},
        )
        or {}
    )
    values = statistics.get(
        "accuracy",
        [],
    ) or []

    if not values:
        return np.nan

    return _to_scalar(
        values[0]
    )


def _epoch_rows(
    result,
    victim_seed,
    block_size,
    repeat,
    sampling_seed,
    n_resampling_epochs,
    fine_tune_epochs,
):
    """
    Convert PRBCD attack statistics into one row per logged step.

    Logging convention:

        step 0 = clean baseline
        step 1 = PRBCD epoch 0
        step 2 = PRBCD epoch 1
        ...
    """
    statistics = (
        result.get(
            "attack_statistics",
            {},
        )
        or {}
    )

    metric_names = [
        "accuracy",
        "loss",
        "probability_mass_update",
        "probability_mass_projected",
        "nonzero_weights",
    ]

    metrics = {
        metric_name: (
            statistics.get(
                metric_name,
                [],
            )
            or []
        )
        for metric_name in metric_names
    }

    rows = []

    for step, accuracy in enumerate(
        metrics["accuracy"]
    ):
        prbcd_epoch = step - 1

        is_resampling_phase = (
            0
            <= prbcd_epoch
            < n_resampling_epochs
        )

        row = {
            "seed": int(victim_seed),
            "block_size": int(block_size),
            "repeat": int(repeat),
            "sampling_seed": int(
                sampling_seed
            ),
            "epochs": int(EPOCHS),
            "n_resampling_epochs": int(
                n_resampling_epochs
            ),
            "fine_tune_epochs": int(
                fine_tune_epochs
            ),
            "step": int(step),
            "prbcd_epoch": int(
                prbcd_epoch
            ),
            "is_clean_baseline": bool(
                step == 0
            ),
            "is_resampling_phase": bool(
                is_resampling_phase
            ),
            "accuracy": _to_scalar(
                accuracy
            ),
        }

        for metric_name in [
            "loss",
            "probability_mass_update",
            "probability_mass_projected",
            "nonzero_weights",
        ]:
            values = metrics[
                metric_name
            ]

            row[metric_name] = (
                _to_scalar(values[step])
                if step < len(values)
                else np.nan
            )

        rows.append(row)

    return rows


def _run_prbcd(
    victim_seed,
    block_size,
    sampling_seed,
    n_resampling_epochs,
):
    """
    Run one PRBCD condition.
    """
    fine_tune_epochs = (
        EPOCHS
        - n_resampling_epochs
    )

    attack_params = {
        "block_size": int(
            block_size
        ),
        "epochs": int(
            EPOCHS
        ),
        "fine_tune_epochs": int(
            fine_tune_epochs
        ),
        "with_early_stopping": bool(
            WITH_EARLY_STOPPING
        ),
        "keep_heuristic": "WeightOnly",
        "do_synchronize": True,
        "loss_type": "tanhMargin",

        # Patched candidate-sampling controls.
        "rq1_enabled": True,
        "rq1_is_reference": False,
        "rq1_sampling_seed": int(
            sampling_seed
        ),
    }

    started = timer()

    result = experiment_global_attack_direct.run(
        graph=graph_sparse,
        data_dir="./data",
        dataset=DATASET,
        attack="PRBCD",
        attack_params=attack_params,
        selector_params={},
        epsilons=[
            BLOCK_EXPERIMENT_EPSILON
        ],
        binary_attr=False,
        make_undirected=True,
        seed=int(victim_seed),
        artifact_dir=ARTIFACT_DIR,
        pert_adj_storage_type=(
            PERT_ADJ_STORAGE_TYPE
        ),
        pert_attr_storage_type=(
            PERT_ATTR_STORAGE_TYPE
        ),
        model_label=MODEL_LABEL,
        model_storage_type=(
            "demo_custom_split"
        ),
        device="cpu",
        data_device="cpu",
        debug_level="info",
        semi=True,
        use_cert="none",
    )

    runtime_seconds = (
        timer() - started
    )

    return (
        result,
        runtime_seconds,
        fine_tune_epochs,
    )


# ============================================================
# Run experiment
# ============================================================

run_rows = []
epoch_rows = []

total_runs = (
    len(SEEDS)
    * len(BLOCK_SIZES)
    * len(N_RESAMPLING_EPOCHS_VALUES)
    * REPEATS_PER_SEED
)

completed_runs = 0

for victim_seed in SEEDS:

    for block_index, block_size in enumerate(
        BLOCK_SIZES
    ):

        for repeat in range(
            REPEATS_PER_SEED
        ):

            # Held constant across fine-tuning conditions for
            # the same victim seed, block size, and repeat.
            sampling_seed = (
                200_000
                + int(victim_seed) * 10_000
                + block_index * 1_000
                + repeat
            )

            for n_resampling_epochs in (
                N_RESAMPLING_EPOCHS_VALUES
            ):

                fine_tune_epochs = (
                    EPOCHS
                    - n_resampling_epochs
                )

                print(
                    "\n"
                    f"Run {completed_runs + 1}/"
                    f"{total_runs} | "
                    f"victim seed={victim_seed} | "
                    f"block size={block_size:,} | "
                    f"resampling epochs="
                    f"{n_resampling_epochs} | "
                    f"fine-tune epochs="
                    f"{fine_tune_epochs} | "
                    f"repeat={repeat + 1}/"
                    f"{REPEATS_PER_SEED} | "
                    f"sampling seed={sampling_seed}"
                )

                set_global_seed(
                    sampling_seed
                )

                (
                    result,
                    runtime_seconds,
                    fine_tune_epochs,
                ) = _run_prbcd(
                    victim_seed=(
                        victim_seed
                    ),
                    block_size=(
                        block_size
                    ),
                    sampling_seed=(
                        sampling_seed
                    ),
                    n_resampling_epochs=(
                        n_resampling_epochs
                    ),
                )

                final_accuracy = (
                    _final_accuracy(result)
                )
                clean_accuracy = (
                    _clean_accuracy(result)
                )

                if not np.isfinite(clean_accuracy):
                    clean_accuracy = float(
                        SEED_CONTEXT_BY_SEED[
                            int(victim_seed)
                        ]["clean_acc"]
                    )

                final_accuracy_drop = (
                    clean_accuracy
                    - final_accuracy
                )

                run_rows.append({
                    "seed": int(
                        victim_seed
                    ),
                    "block_size": int(
                        block_size
                    ),
                    "repeat": int(
                        repeat
                    ),
                    "sampling_seed": int(
                        sampling_seed
                    ),
                    "epsilon": float(
                        BLOCK_EXPERIMENT_EPSILON
                    ),
                    "attack_budget": int(
                        ATTACK_BUDGET
                    ),
                    "epochs": int(
                        EPOCHS
                    ),
                    "n_resampling_epochs": int(
                        n_resampling_epochs
                    ),
                    "fine_tune_epochs": int(
                        fine_tune_epochs
                    ),
                    "with_early_stopping": bool(
                        WITH_EARLY_STOPPING
                    ),
                    "clean_accuracy": (
                        clean_accuracy
                    ),
                    "final_accuracy": (
                        final_accuracy
                    ),
                    "final_accuracy_drop": (
                        final_accuracy_drop
                    ),
                    "runtime_seconds": float(
                        runtime_seconds
                    ),
                })

                epoch_rows.extend(
                    _epoch_rows(
                        result=result,
                        victim_seed=(
                            victim_seed
                        ),
                        block_size=(
                            block_size
                        ),
                        repeat=repeat,
                        sampling_seed=(
                            sampling_seed
                        ),
                        n_resampling_epochs=(
                            n_resampling_epochs
                        ),
                        fine_tune_epochs=(
                            fine_tune_epochs
                        ),
                    )
                )

                print(
                    f"Final accuracy: "
                    f"{final_accuracy:.6f} | "
                    f"runtime: "
                    f"{runtime_seconds:.1f} seconds"
                )

                completed_runs += 1

                del result
                gc.collect()

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()


# ============================================================
# Create raw DataFrames
# ============================================================

runs_df = pd.DataFrame(
    run_rows
)

epoch_metrics_df = pd.DataFrame(
    epoch_rows
)

if runs_df.empty:
    raise RuntimeError(
        "No PRBCD runs were completed."
    )

if epoch_metrics_df.empty:
    raise RuntimeError(
        "No epoch-level PRBCD statistics were recorded."
    )


# ============================================================
# Save raw data
# ============================================================

runs_df.to_csv(
    OUT_DIR
    / "prbcd_blocksize_resampling_runs.csv",
    index=False,
)

epoch_metrics_df.to_csv(
    OUT_DIR
    / "prbcd_blocksize_resampling_epoch_metrics.csv",
    index=False,
)


# ============================================================
# Aggregate repeats within victim seeds
# ============================================================

seed_level_df = (
    runs_df
    .groupby(
        [
            "seed",
            "block_size",
            "n_resampling_epochs",
            "fine_tune_epochs",
        ],
        as_index=False,
    )
    .agg(
        clean_accuracy=(
            "clean_accuracy",
            "mean",
        ),
        final_accuracy=(
            "final_accuracy",
            "mean",
        ),
        final_accuracy_repeat_std=(
            "final_accuracy",
            "std",
        ),
        final_accuracy_drop=(
            "final_accuracy_drop",
            "mean",
        ),
        final_accuracy_drop_repeat_std=(
            "final_accuracy_drop",
            "std",
        ),
        runtime_seconds=(
            "runtime_seconds",
            "mean",
        ),
        runtime_seconds_repeat_std=(
            "runtime_seconds",
            "std",
        ),
        n_repeats=(
            "repeat",
            "count",
        ),
    )
)

seed_level_df.to_csv(
    OUT_DIR
    / "prbcd_blocksize_resampling_seed_level.csv",
    index=False,
)


# ============================================================
# Aggregate final results across victim seeds
# ============================================================

blocksize_resampling_summary_df = (
    seed_level_df
    .groupby(
        [
            "block_size",
            "n_resampling_epochs",
            "fine_tune_epochs",
        ],
        as_index=False,
    )
    .agg(
        final_accuracy_mean=(
            "final_accuracy",
            "mean",
        ),
        final_accuracy_std=(
            "final_accuracy",
            "std",
        ),
        final_accuracy_drop_mean=(
            "final_accuracy_drop",
            "mean",
        ),
        final_accuracy_drop_std=(
            "final_accuracy_drop",
            "std",
        ),
        runtime_seconds_mean=(
            "runtime_seconds",
            "mean",
        ),
        runtime_seconds_std=(
            "runtime_seconds",
            "std",
        ),
        n_seeds=(
            "seed",
            "nunique",
        ),
    )
)

blocksize_resampling_summary_df[
    "final_accuracy_sem"
] = (
    blocksize_resampling_summary_df[
        "final_accuracy_std"
    ]
    /
    np.sqrt(
        blocksize_resampling_summary_df[
            "n_seeds"
        ]
    )
)

blocksize_resampling_summary_df[
    "final_accuracy_ci95"
] = (
    1.96
    * blocksize_resampling_summary_df[
        "final_accuracy_sem"
    ]
)

blocksize_resampling_summary_df[
    "final_accuracy_drop_sem"
] = (
    blocksize_resampling_summary_df[
        "final_accuracy_drop_std"
    ]
    /
    np.sqrt(
        blocksize_resampling_summary_df[
            "n_seeds"
        ]
    )
)

blocksize_resampling_summary_df[
    "final_accuracy_drop_ci95"
] = (
    1.96
    * blocksize_resampling_summary_df[
        "final_accuracy_drop_sem"
    ]
)

blocksize_resampling_summary_df[
    "runtime_seconds_sem"
] = (
    blocksize_resampling_summary_df[
        "runtime_seconds_std"
    ]
    /
    np.sqrt(
        blocksize_resampling_summary_df[
            "n_seeds"
        ]
    )
)

blocksize_resampling_summary_df[
    "runtime_seconds_ci95"
] = (
    1.96
    * blocksize_resampling_summary_df[
        "runtime_seconds_sem"
    ]
)

blocksize_resampling_summary_df.to_csv(
    OUT_DIR
    / "prbcd_blocksize_resampling_summary.csv",
    index=False,
)


# ============================================================
# Aggregate epoch trajectories:
# first within each victim seed
# ============================================================

seed_epoch_df = (
    epoch_metrics_df
    .groupby(
        [
            "seed",
            "block_size",
            "n_resampling_epochs",
            "fine_tune_epochs",
            "step",
            "prbcd_epoch",
            "is_clean_baseline",
            "is_resampling_phase",
        ],
        as_index=False,
    )
    .agg(
        accuracy=(
            "accuracy",
            "mean",
        ),
        loss=(
            "loss",
            "mean",
        ),
        probability_mass_update=(
            "probability_mass_update",
            "mean",
        ),
        probability_mass_projected=(
            "probability_mass_projected",
            "mean",
        ),
        nonzero_weights=(
            "nonzero_weights",
            "mean",
        ),
    )
)


# ============================================================
# Aggregate epoch trajectories across victim seeds
# ============================================================

epoch_summary_df = (
    seed_epoch_df
    .groupby(
        [
            "block_size",
            "n_resampling_epochs",
            "fine_tune_epochs",
            "step",
            "prbcd_epoch",
            "is_clean_baseline",
            "is_resampling_phase",
        ],
        as_index=False,
    )
    .agg(
        accuracy_mean=(
            "accuracy",
            "mean",
        ),
        accuracy_std=(
            "accuracy",
            "std",
        ),
        loss_mean=(
            "loss",
            "mean",
        ),
        loss_std=(
            "loss",
            "std",
        ),
        probability_mass_update_mean=(
            "probability_mass_update",
            "mean",
        ),
        probability_mass_projected_mean=(
            "probability_mass_projected",
            "mean",
        ),
        nonzero_weights_mean=(
            "nonzero_weights",
            "mean",
        ),
        n_seeds=(
            "seed",
            "nunique",
        ),
    )
)

epoch_summary_df[
    "accuracy_sem"
] = (
    epoch_summary_df[
        "accuracy_std"
    ]
    /
    np.sqrt(
        epoch_summary_df[
            "n_seeds"
        ]
    )
)

epoch_summary_df[
    "accuracy_ci95"
] = (
    1.96
    * epoch_summary_df[
        "accuracy_sem"
    ]
)

epoch_summary_df[
    "loss_sem"
] = (
    epoch_summary_df[
        "loss_std"
    ]
    /
    np.sqrt(
        epoch_summary_df[
            "n_seeds"
        ]
    )
)

epoch_summary_df[
    "loss_ci95"
] = (
    1.96
    * epoch_summary_df[
        "loss_sem"
    ]
)


# ============================================================
# Save aggregated epoch data
# ============================================================

seed_epoch_df.to_csv(
    OUT_DIR
    / "prbcd_blocksize_resampling_epoch_seed_level.csv",
    index=False,
)

epoch_summary_df.to_csv(
    OUT_DIR
    / "prbcd_blocksize_resampling_epoch_summary.csv",
    index=False,
)


# ============================================================
# Finish
# ============================================================

print("\nData generation finished.")
print("Saved to:", OUT_DIR.resolve())

display(
    blocksize_resampling_summary_df
    .sort_values(
        [
            "block_size",
            "fine_tune_epochs",
        ]
    )
)

In [ ]:
# ============================================================
# CELL 2: PLOTTING ONLY
#
# This cell does not run PRBCD.
# It reloads previously saved CSV files.
# ============================================================

from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401


# ============================================================
# Select result directory
# ============================================================

BASE_OUT_DIR = (
    Path("extendedPlotting")
    / "prbcd_blocksize_resampling_multiseed"
)

# Leave as None to use the most recent saved run.
#
# To use a specific run:
#
# RESULT_DIR = Path(
#     "extendedPlotting/"
#     "prbcd_blocksize_resampling_multiseed/"
#     "DATASET__YYYYMMDD_HHMMSS"
# )
RESULT_DIR = None


def _find_latest_result_directory(
    base_directory,
):
    pointer_file = (
        base_directory
        / "latest_run.txt"
    )

    if pointer_file.exists():
        candidate = Path(
            pointer_file.read_text(
                encoding="utf-8"
            ).strip()
        )

        if candidate.exists():
            return candidate

    candidates = [
        path
        for path in base_directory.iterdir()
        if path.is_dir()
    ]

    if not candidates:
        raise FileNotFoundError(
            "No saved experiment directories were found in "
            f"{base_directory.resolve()}."
        )

    return max(
        candidates,
        key=lambda path: path.stat().st_mtime,
    )


if RESULT_DIR is None:
    RESULT_DIR = _find_latest_result_directory(
        BASE_OUT_DIR
    )

RESULT_DIR = Path(
    RESULT_DIR
)

if not RESULT_DIR.exists():
    raise FileNotFoundError(
        f"Result directory does not exist: {RESULT_DIR}"
    )

print(
    "Loading results from:",
    RESULT_DIR.resolve(),
)


# ============================================================
# Load configuration and CSV files
# ============================================================

config_path = (
    RESULT_DIR
    / "experiment_config.json"
)

if config_path.exists():
    with open(
        config_path,
        "r",
        encoding="utf-8",
    ) as file:
        experiment_config = json.load(file)
else:
    experiment_config = {}

runs_df = pd.read_csv(
    RESULT_DIR
    / "prbcd_blocksize_resampling_runs.csv"
)

epoch_metrics_df = pd.read_csv(
    RESULT_DIR
    / "prbcd_blocksize_resampling_epoch_metrics.csv"
)

seed_level_df = pd.read_csv(
    RESULT_DIR
    / "prbcd_blocksize_resampling_seed_level.csv"
)

blocksize_resampling_summary_df = pd.read_csv(
    RESULT_DIR
    / "prbcd_blocksize_resampling_summary.csv"
)

seed_epoch_df = pd.read_csv(
    RESULT_DIR
    / "prbcd_blocksize_resampling_epoch_seed_level.csv"
)

epoch_summary_df = pd.read_csv(
    RESULT_DIR
    / "prbcd_blocksize_resampling_epoch_summary.csv"
)


# ============================================================
# Restore Boolean columns safely
# ============================================================

def _to_boolean_series(series):
    if series.dtype == bool:
        return series

    return (
        series
        .astype(str)
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
        .fillna(False)
        .astype(bool)
    )


for dataframe in [
    epoch_metrics_df,
    seed_epoch_df,
    epoch_summary_df,
]:
    if "is_clean_baseline" in dataframe.columns:
        dataframe[
            "is_clean_baseline"
        ] = _to_boolean_series(
            dataframe[
                "is_clean_baseline"
            ]
        )

    if "is_resampling_phase" in dataframe.columns:
        dataframe[
            "is_resampling_phase"
        ] = _to_boolean_series(
            dataframe[
                "is_resampling_phase"
            ]
        )


# ============================================================
# Derive final accuracy drop from each seed-specific clean baseline
# ============================================================

FINAL_RESULT_GROUP_COLUMNS = [
    "seed",
    "block_size",
    "n_resampling_epochs",
    "fine_tune_epochs",
]

if "final_accuracy_drop" not in seed_level_df.columns:
    baseline_by_seed = (
        seed_epoch_df[
            seed_epoch_df[
                "is_clean_baseline"
            ]
        ]
        .groupby(
            FINAL_RESULT_GROUP_COLUMNS,
            as_index=False,
        )["accuracy"]
        .mean()
        .rename(
            columns={
                "accuracy": "clean_accuracy"
            }
        )
    )

    seed_level_df = seed_level_df.merge(
        baseline_by_seed,
        on=FINAL_RESULT_GROUP_COLUMNS,
        how="left",
    )

    seed_level_df[
        "final_accuracy_drop"
    ] = (
        seed_level_df[
            "clean_accuracy"
        ]
        - seed_level_df[
            "final_accuracy"
        ]
    )

final_drop_summary_df = (
    seed_level_df
    .groupby(
        [
            "block_size",
            "n_resampling_epochs",
            "fine_tune_epochs",
        ],
        as_index=False,
    )
    .agg(
        final_accuracy_drop_mean=(
            "final_accuracy_drop",
            "mean",
        ),
        final_accuracy_drop_std=(
            "final_accuracy_drop",
            "std",
        ),
        n_seeds=(
            "seed",
            "nunique",
        ),
    )
)

final_drop_summary_df[
    "final_accuracy_drop_std"
] = (
    final_drop_summary_df[
        "final_accuracy_drop_std"
    ]
    .fillna(0.0)
)

final_drop_summary_df[
    "final_accuracy_drop_sem"
] = (
    final_drop_summary_df[
        "final_accuracy_drop_std"
    ]
    /
    np.sqrt(
        final_drop_summary_df[
            "n_seeds"
        ].clip(lower=1)
    )
)

final_drop_summary_df[
    "final_accuracy_drop_ci95"
] = (
    1.96
    * final_drop_summary_df[
        "final_accuracy_drop_sem"
    ]
)


# ============================================================
# Resolve experiment values from loaded data
# ============================================================

BLOCK_SIZES = sorted(
    epoch_summary_df[
        "block_size"
    ].dropna().astype(int).unique()
)

FINE_TUNE_EPOCH_VALUES = sorted(
    epoch_summary_df[
        "fine_tune_epochs"
    ].dropna().astype(int).unique()
)

N_RESAMPLING_EPOCH_VALUES = sorted(
    epoch_summary_df[
        "n_resampling_epochs"
    ].dropna().astype(int).unique()
)

if "epochs" in epoch_metrics_df.columns:
    EPOCHS = int(
        epoch_metrics_df[
            "epochs"
        ].dropna().iloc[0]
    )
else:
    EPOCHS = int(
        max(
            FINE_TUNE_EPOCH_VALUES
        )
        + min(
            N_RESAMPLING_EPOCH_VALUES
        )
    )

print("Block sizes:", BLOCK_SIZES)
print(
    "Fine-tuning epochs:",
    FINE_TUNE_EPOCH_VALUES,
)
print(
    "Resampling epochs:",
    N_RESAMPLING_EPOCH_VALUES,
)
print("Total epochs:", EPOCHS)

display(
    blocksize_resampling_summary_df
    .sort_values(
        [
            "block_size",
            "fine_tune_epochs",
        ]
    )
)


# ============================================================
# Plot settings
# ============================================================

SAVE_PLOTS = True

# Plot every nth epoch to make the 3D figure lighter.
# Set to 1 to plot every epoch.
EPOCH_STRIDE = 2

# Camera angle.
ELEVATION = 25
AZIMUTH = -125


# ============================================================
# Prepare epoch trajectories
# ============================================================

attack_epoch_summary = (
    epoch_summary_df[
        ~epoch_summary_df[
            "is_clean_baseline"
        ]
    ]
    .copy()
)

attack_epoch_summary = (
    attack_epoch_summary[
        attack_epoch_summary[
            "prbcd_epoch"
        ] >= 0
    ]
)

attack_epoch_summary = (
    attack_epoch_summary[
        attack_epoch_summary[
            "prbcd_epoch"
        ] % EPOCH_STRIDE
        == 0
    ]
)


# ============================================================
# Combined 3D plot
#
# x-axis: PRBCD epoch
# y-axis: fine-tuning epochs
# z-axis: accuracy
#
# Block size is represented by color.
# ============================================================

fig = plt.figure(
    figsize=(15, 10)
)

ax = fig.add_subplot(
    111,
    projection="3d",
)

default_colors = plt.rcParams[
    "axes.prop_cycle"
].by_key()["color"]

block_color_map = {
    block_size: default_colors[
        index % len(default_colors)
    ]
    for index, block_size in enumerate(
        BLOCK_SIZES
    )
}

for block_size in BLOCK_SIZES:

    block_data = (
        attack_epoch_summary[
            attack_epoch_summary[
                "block_size"
            ] == block_size
        ]
    )

    for fine_tune_epochs in (
        FINE_TUNE_EPOCH_VALUES
    ):

        part = (
            block_data[
                block_data[
                    "fine_tune_epochs"
                ] == fine_tune_epochs
            ]
            .sort_values(
                "prbcd_epoch"
            )
        )

        if part.empty:
            continue

        x = part[
            "prbcd_epoch"
        ].to_numpy()

        y = np.full(
            shape=len(part),
            fill_value=fine_tune_epochs,
            dtype=float,
        )

        z = part[
            "accuracy_mean"
        ].to_numpy()

        ax.plot(
            x,
            y,
            z,
            linewidth=1.8,
            alpha=0.85,
            color=block_color_map[
                block_size
            ],
        )

        # Mark the end of the resampling phase.
        n_resampling_epochs = int(
            part[
                "n_resampling_epochs"
            ].iloc[0]
        )

        transition_epoch = (
            n_resampling_epochs - 1
        )

        full_part = (
            epoch_summary_df[
                (
                    epoch_summary_df[
                        "block_size"
                    ] == block_size
                )
                &
                (
                    epoch_summary_df[
                        "fine_tune_epochs"
                    ] == fine_tune_epochs
                )
                &
                (
                    epoch_summary_df[
                        "prbcd_epoch"
                    ] == transition_epoch
                )
            ]
        )

        if not full_part.empty:
            ax.scatter(
                [transition_epoch],
                [fine_tune_epochs],
                [
                    full_part[
                        "accuracy_mean"
                    ].iloc[0]
                ],
                s=28,
                color=block_color_map[
                    block_size
                ],
                depthshade=True,
            )


# ============================================================
# 3D plot formatting
# ============================================================

ax.set_xlabel(
    "PRBCD epoch",
    labelpad=12,
)

ax.set_ylabel(
    "Number of fine-tuning epochs",
    labelpad=14,
)

ax.set_zlabel(
    "Accuracy",
    labelpad=12,
)

ax.set_title(
    "PRBCD accuracy trajectories across exploration schedules\n"
    "Lower fine-tuning duration means more resampling epochs",
    pad=22,
)

ax.set_yticks(
    FINE_TUNE_EPOCH_VALUES
)

ax.view_init(
    elev=ELEVATION,
    azim=AZIMUTH,
)

legend_handles = [
    Line2D(
        [0],
        [0],
        linewidth=2.5,
        color=block_color_map[
            block_size
        ],
        label=f"B = {block_size:,}",
    )
    for block_size in BLOCK_SIZES
]

ax.legend(
    handles=legend_handles,
    title="Block size",
    loc="upper left",
    bbox_to_anchor=(0.02, 0.98),
)

plt.tight_layout()

if SAVE_PLOTS:
    plt.savefig(
        RUN_PLOTS_DIR
        / "accuracy_epoch_finetune_3d.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.savefig(
        RUN_PLOTS_DIR
        / "accuracy_epoch_finetune_3d.pdf",
        bbox_inches="tight",
    )

plt.show()


# ============================================================
# Supporting plot:
# seed-specific final accuracy drop versus fine-tuning duration
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 6)
)

for block_size in BLOCK_SIZES:

    part = (
        final_drop_summary_df[
            final_drop_summary_df[
                "block_size"
            ] == block_size
        ]
        .sort_values(
            "fine_tune_epochs"
        )
    )

    ax.errorbar(
        part[
            "fine_tune_epochs"
        ],
        part[
            "final_accuracy_drop_mean"
        ],
        yerr=part[
            "final_accuracy_drop_ci95"
        ].fillna(0.0),
        marker="o",
        capsize=4,
        label=f"B = {block_size:,}",
    )

ax.axhline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)

ax.set_xlabel(
    "Number of fine-tuning epochs"
)

ax.set_ylabel(
    "Clean accuracy − final attacked accuracy"
)

ax.set_title(
    "Final PRBCD accuracy reduction by fine-tuning duration"
)

ax.legend(
    title="Block size"
)

ax.grid(
    alpha=0.3
)

plt.tight_layout()

if SAVE_PLOTS:
    plt.savefig(
        RUN_PLOTS_DIR
        / "final_accuracy_drop_vs_finetune_epochs.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.savefig(
        RUN_PLOTS_DIR
        / "final_accuracy_drop_vs_finetune_epochs.pdf",
        bbox_inches="tight",
    )

plt.show()

In [ ]:
### §9.1  RQ1 — Verifying the Miss Probability

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# Configuration
# ============================================================

# Number of possible undirected edge flips:
# M = n * (n - 1) // 2
N_NODES = adj_matrix.sizes()[0]
M = N_NODES * (N_NODES - 1) // 2

BLOCK_SIZES = [1_000, 5_000, 20_000, 50_000, 100_000]
N_RESAMPLING_STEPS = 50
N_RUNS = 200

# Fraction of the current block whose weights are <= eps.
#
# PRBCD discards:
# max(number of eps-weight edges, floor(current block size / 2))
EPS_FRACTION_SCENARIOS = [0.50, 0.75, 1.00]

BASE_SEED = 42


# ============================================================
# One sampler-only PRBCD coverage run
# ============================================================

def simulate_coverage_run(
    n_possible_edges: int,
    block_size: int,
    n_resampling_steps: int,
    eps_fraction: float,
    seed: int,
):
    """
    Simulates PRBCD candidate coverage without gradients or victim-model calls.

    The resampling rule follows the WeightOnly implementation:

        n_eps = number of weights <= eps
        n_discard = max(n_eps, floor(actual_block_size / 2))
        n_keep = actual_block_size - n_discard
        n_refill = nominal_block_size - n_keep

    Newly sampled IDs are merged with retained IDs using np.unique, matching
    the deduplication behavior of torch.unique.

    An edge is considered covered once it has appeared in a search block.

    Returns
    -------
    trace:
        One row for the initial block and every resampling step.

    summary:
        Final coverage statistics for the run.
    """
    if n_possible_edges <= 0:
        raise ValueError("n_possible_edges must be positive.")

    if block_size <= 0:
        raise ValueError("block_size must be positive.")

    if not 0.0 <= eps_fraction <= 1.0:
        raise ValueError("eps_fraction must be between 0 and 1.")

    rng = np.random.default_rng(seed)

    seen = set()

    # --------------------------------------------------------
    # Initial block
    # --------------------------------------------------------

    raw_initial = rng.integers(
        low=0,
        high=n_possible_edges,
        size=block_size,
        endpoint=False,
        dtype=np.int64,
    )

    current_block = np.unique(raw_initial)

    # Every unique initial draw enters the initial search space.
    seen.update(current_block.tolist())

    raw_draws_total = raw_initial.size

    p_never_seen = np.exp(
        raw_draws_total
        * np.log1p(-1.0 / n_possible_edges)
    )

    records = [{
        "stage": 0,
        "eps_fraction": eps_fraction,
        "block_size_nominal": block_size,
        "block_size_before": 0,
        "n_eps": np.nan,
        "n_discard": np.nan,
        "discard_fraction": np.nan,
        "n_keep": np.nan,
        "raw_draws_stage": raw_initial.size,
        "raw_draws_total": raw_draws_total,
        "actual_block_size": current_block.size,
        "new_unique_stage": current_block.size,
        "unique_seen": len(seen),
        "coverage_fraction": len(seen) / n_possible_edges,
        "repeat_fraction": 1.0 - len(seen) / raw_draws_total,
        "p_never_seen_theory": p_never_seen,
        "p_seen_theory": 1.0 - p_never_seen,
    }]

    # --------------------------------------------------------
    # Resampling steps
    # --------------------------------------------------------

    for stage in range(1, n_resampling_steps + 1):

        block_size_before = current_block.size

        if block_size_before == 0:
            n_eps = 0
            n_discard = 0
            n_keep = 0
            kept = np.empty(0, dtype=np.int64)

        else:
            # Simulated count of edges whose weight is <= eps.
            n_eps = int(
                np.floor(eps_fraction * block_size_before)
            )

            # WeightOnly rule from PRBCD:
            # discard at least half, potentially all eps edges.
            n_discard = max(
                n_eps,
                block_size_before // 2,
            )

            n_discard = min(
                n_discard,
                block_size_before,
            )

            n_keep = block_size_before - n_discard

            # Coverage is independent of which particular edges survive.
            # Random retention avoids adding a ranking assumption.
            if n_keep > 0:
                kept = rng.choice(
                    current_block,
                    size=n_keep,
                    replace=False,
                )
            else:
                kept = np.empty(0, dtype=np.int64)

        # Match PRBCD:
        # request enough raw candidates to refill the nominal block.
        n_refill = block_size - n_keep

        raw_new = rng.integers(
            low=0,
            high=n_possible_edges,
            size=n_refill,
            endpoint=False,
            dtype=np.int64,
        )

        raw_draws_total += raw_new.size

        unique_before = len(seen)

        # A newly drawn edge counts as covered if it was not seen before.
        seen.update(raw_new.tolist())

        new_unique = len(seen) - unique_before

        # Retained edges and new draws are deduplicated.
        # The resulting block may therefore be slightly smaller than the
        # nominal block size, matching the behavior of torch.unique.
        current_block = np.unique(
            np.concatenate([kept, raw_new])
        )

        p_never_seen = np.exp(
            raw_draws_total
            * np.log1p(-1.0 / n_possible_edges)
        )

        discard_fraction = (
            n_discard / block_size_before
            if block_size_before > 0
            else np.nan
        )

        records.append({
            "stage": stage,
            "eps_fraction": eps_fraction,
            "block_size_nominal": block_size,
            "block_size_before": block_size_before,
            "n_eps": n_eps,
            "n_discard": n_discard,
            "discard_fraction": discard_fraction,
            "n_keep": n_keep,
            "raw_draws_stage": raw_new.size,
            "raw_draws_total": raw_draws_total,
            "actual_block_size": current_block.size,
            "new_unique_stage": new_unique,
            "unique_seen": len(seen),
            "coverage_fraction": len(seen) / n_possible_edges,
            "repeat_fraction": 1.0 - len(seen) / raw_draws_total,
            "p_never_seen_theory": p_never_seen,
            "p_seen_theory": 1.0 - p_never_seen,
        })

    trace = pd.DataFrame(records)

    final_p_never_seen = np.exp(
        raw_draws_total
        * np.log1p(-1.0 / n_possible_edges)
    )

    expected_unique_seen = (
        n_possible_edges
        * (1.0 - final_p_never_seen)
    )

    resampling_rows = trace["stage"] > 0

    summary = {
        "eps_fraction": eps_fraction,
        "block_size": block_size,
        "seed": seed,
        "raw_draws_total": raw_draws_total,
        "unique_seen": len(seen),
        "coverage_fraction": len(seen) / n_possible_edges,
        "empirical_repeat_fraction": (
            1.0 - len(seen) / raw_draws_total
        ),
        "p_edge_never_seen_theory": final_p_never_seen,
        "p_edge_seen_theory": 1.0 - final_p_never_seen,
        "expected_unique_seen_theory": expected_unique_seen,
        "mean_discard_fraction": trace.loc[
            resampling_rows,
            "discard_fraction",
        ].mean(),
        "mean_raw_draws_per_resampling": trace.loc[
            resampling_rows,
            "raw_draws_stage",
        ].mean(),
        "mean_actual_block_size": trace.loc[
            resampling_rows,
            "actual_block_size",
        ].mean(),
        "actual_final_block_size": current_block.size,
    }

    return trace, summary


# ============================================================
# Run all scenarios, block sizes, and seeds
# ============================================================

all_traces = []
all_summaries = []

for scenario_index, eps_fraction in enumerate(
    EPS_FRACTION_SCENARIOS
):
    for block_size_index, block_size in enumerate(BLOCK_SIZES):
        for run in range(N_RUNS):

            seed = (
                BASE_SEED
                + scenario_index * 1_000_000
                + block_size_index * 10_000
                + run
            )

            trace, summary = simulate_coverage_run(
                n_possible_edges=M,
                block_size=block_size,
                n_resampling_steps=N_RESAMPLING_STEPS,
                eps_fraction=eps_fraction,
                seed=seed,
            )

            trace["run"] = run
            trace["block_size"] = block_size

            all_traces.append(trace)
            all_summaries.append(summary)

traces = pd.concat(
    all_traces,
    ignore_index=True,
)

summaries = pd.DataFrame(all_summaries)


# ============================================================
# Aggregated results
# ============================================================

coverage_summary = (
    summaries
    .groupby(["eps_fraction", "block_size"])
    .agg(
        raw_draws_mean=(
            "raw_draws_total",
            "mean",
        ),
        raw_draws_std=(
            "raw_draws_total",
            "std",
        ),
        unique_seen_mean=(
            "unique_seen",
            "mean",
        ),
        unique_seen_std=(
            "unique_seen",
            "std",
        ),
        coverage_mean=(
            "coverage_fraction",
            "mean",
        ),
        coverage_std=(
            "coverage_fraction",
            "std",
        ),
        empirical_repeat_mean=(
            "empirical_repeat_fraction",
            "mean",
        ),
        theoretical_p_never_seen=(
            "p_edge_never_seen_theory",
            "mean",
        ),
        theoretical_p_seen=(
            "p_edge_seen_theory",
            "mean",
        ),
        expected_unique_theory=(
            "expected_unique_seen_theory",
            "mean",
        ),
        mean_discard_fraction=(
            "mean_discard_fraction",
            "mean",
        ),
        mean_raw_draws_per_resampling=(
            "mean_raw_draws_per_resampling",
            "mean",
        ),
        mean_actual_block_size=(
            "mean_actual_block_size",
            "mean",
        ),
        final_block_size_mean=(
            "actual_final_block_size",
            "mean",
        ),
    )
    .reset_index()
)

display(coverage_summary)


# ============================================================
# Plot 1:
# Coverage over resampling steps, separately for each scenario
# ============================================================

coverage_by_stage = (
    traces
    .groupby(
        ["eps_fraction", "block_size", "stage"]
    )["coverage_fraction"]
    .agg(["mean", "std"])
    .reset_index()
)

for eps_fraction in EPS_FRACTION_SCENARIOS:

    plt.figure(figsize=(9, 6))

    scenario_data = coverage_by_stage[
        coverage_by_stage["eps_fraction"] == eps_fraction
    ]

    for block_size in BLOCK_SIZES:

        part = scenario_data[
            scenario_data["block_size"] == block_size
        ]

        plt.plot(
            part["stage"],
            part["mean"],
            label=f"B = {block_size:,}",
        )

        plt.fill_between(
            part["stage"],
            part["mean"] - part["std"],
            part["mean"] + part["std"],
            alpha=0.15,
        )

    plt.xlabel("Initial block / resampling step")
    plt.ylabel("Fraction of possible edges ever seen")
    plt.title(
        "PRBCD candidate coverage\n"
        f"{eps_fraction:.0%} of block weights <= eps"
    )
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        RUN_PLOTS_DIR
        / f"coverage_by_resampling_step__eps-{eps_fraction:.2f}.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()


# ============================================================
# Plot 2:
# Empirical versus theoretical unique coverage
# ============================================================

for eps_fraction in EPS_FRACTION_SCENARIOS:

    part = coverage_summary[
        coverage_summary["eps_fraction"] == eps_fraction
    ]

    plt.figure(figsize=(8, 6))

    plt.plot(
        part["block_size"],
        part["unique_seen_mean"],
        marker="o",
        label="Empirical unique edges seen",
    )

    plt.plot(
        part["block_size"],
        part["expected_unique_theory"],
        marker="x",
        linestyle="--",
        label="Theoretical expectation",
    )

    plt.xscale("log")
    plt.xlabel("Block size")
    plt.ylabel("Unique edges ever seen")
    plt.title(
        "Empirical and theoretical coverage\n"
        f"{eps_fraction:.0%} of block weights <= eps"
    )
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        RUN_PLOTS_DIR
        / f"empirical_theoretical_coverage__eps-{eps_fraction:.2f}.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()


# ============================================================
# Plot 3:
# Probability that one fixed edge is never seen
# ============================================================

plt.figure(figsize=(9, 6))

for eps_fraction in EPS_FRACTION_SCENARIOS:

    part = coverage_summary[
        coverage_summary["eps_fraction"] == eps_fraction
    ]

    plt.plot(
        part["block_size"],
        part["theoretical_p_never_seen"],
        marker="o",
        label=f"{eps_fraction:.0%} at eps",
    )

plt.xscale("log")
plt.xlabel("Block size")
plt.ylabel("P(fixed edge is never seen)")
plt.title(
    "Coverage-miss probability under variable "
    "PRBCD resampling"
)
plt.legend(title="Resampling scenario")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    RUN_PLOTS_DIR / "coverage_miss_probability.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


# ============================================================
# Plot 4:
# Total raw candidate draws
# ============================================================

plt.figure(figsize=(9, 6))

for eps_fraction in EPS_FRACTION_SCENARIOS:

    part = coverage_summary[
        coverage_summary["eps_fraction"] == eps_fraction
    ]

    plt.plot(
        part["block_size"],
        part["raw_draws_mean"],
        marker="o",
        label=f"{eps_fraction:.0%} at eps",
    )

plt.xscale("log")
plt.xlabel("Block size")
plt.ylabel("Total raw candidate draws")
plt.title(
    "Realized search effort under different "
    "resampling scenarios"
)
plt.legend(title="Resampling scenario")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    RUN_PLOTS_DIR / "realized_search_effort.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


# ============================================================
# Compact interpretation table
# ============================================================

result_table = coverage_summary[[
    "eps_fraction",
    "block_size",
    "mean_discard_fraction",
    "raw_draws_mean",
    "unique_seen_mean",
    "coverage_mean",
    "theoretical_p_never_seen",
    "final_block_size_mean",
]].copy()

result_table["eps_fraction"] = (
    100 * result_table["eps_fraction"]
).round(0).astype(int).astype(str) + "%"

display(result_table)

In [ ]:
# ============================================================
# RQ3 CSV run archive helper
#
# Usage:
#   RQ3_CSV_RUN_CAPTURE = start_rq3_csv_run(...)
#   ... run the RQ3 cells ...
#   RQ3_CSV_ARCHIVE_DIR = finish_rq3_csv_run(RQ3_CSV_RUN_CAPTURE)
#
# The existing RQ3 cells may keep writing to their fixed output folders.
# This helper creates a unique archive for every run and preserves the
# previous flat outputs before a new run can overwrite them.
# ============================================================

from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Optional
import csv
import hashlib
import json
import os
import re
import shutil
import uuid


RQ3_DEFAULT_CSV_PATTERNS = (
    "rq3_subgraph_candidate_mining/**/*.csv",
    "lp_training_roc_data/**/*.csv",
    "lp_training_roc_data_multiseed/**/*.csv",
    "lp_training_roc_curves_averaged/**/*.csv",
    "lp_training_dynamics_multiseed/**/*.csv",
    "lp_training_heldout_multiseed/**/*.csv",
    "selector_evaluation_multiseed/**/*.csv",
    "whole_graph_transfer_multiseed/**/*.csv",
    "score_diagnostics_multiseed/**/*.csv",
    "rq3_matched_selector_vs_rq2/**/*.csv",
    "sweeps/global_prbcd_*/**/*.csv",
    "rq3*/**/*.csv",
)


def _safe_token(value, fallback):
    text = str(value).strip() if value is not None else ""
    text = text or fallback
    text = re.sub(r"[^A-Za-z0-9._-]+", "-", text)
    return text.strip("._-") or fallback


def _relative_to_project(path, project_dir):
    path = Path(path).resolve()
    project_dir = Path(project_dir).resolve()
    try:
        return path.relative_to(project_dir).as_posix()
    except ValueError:
        drive, tail = os.path.splitdrive(str(path))
        return (
            "_external/"
            + _safe_token(drive, "external")
            + "/"
            + tail.lstrip("/\\").replace("\\", "/")
        )


def _fingerprint(path):
    stat = Path(path).stat()
    return {
        "size_bytes": int(stat.st_size),
        "mtime_ns": int(stat.st_mtime_ns),
    }


def _sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def _modified_utc(path):
    return datetime.fromtimestamp(
        Path(path).stat().st_mtime,
        tz=timezone.utc,
    ).isoformat()


def _collect_csv_files(
    extended_plotting_dir,
    archive_base_dir,
    patterns,
):
    extended_plotting_dir = Path(extended_plotting_dir)
    archive_base_dir = Path(archive_base_dir).resolve()

    if not extended_plotting_dir.exists():
        return []

    found = {}

    for pattern in patterns:
        for path in extended_plotting_dir.glob(str(pattern)):
            if not path.is_file() or path.suffix.lower() != ".csv":
                continue

            resolved = path.resolve()

            try:
                resolved.relative_to(archive_base_dir)
                continue
            except ValueError:
                pass

            found[str(resolved)] = resolved

    return sorted(
        found.values(),
        key=lambda path: path.as_posix(),
    )


def _copy_preserving_tree(
    source,
    destination_root,
    project_dir,
):
    source = Path(source)
    relative_path = _relative_to_project(
        source,
        project_dir,
    )
    destination = Path(destination_root) / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    return destination, relative_path


def _write_csv_atomic(path, rows):
    columns = [
        "status",
        "changed_during_run",
        "source_path",
        "relative_path",
        "archived_path",
        "size_bytes",
        "modified_utc",
        "sha256",
    ]

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        for row in rows:
            writer.writerow({
                column: row.get(column, "")
                for column in columns
            })

    temporary_path.replace(path)


def _write_json_atomic(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")

    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

    temporary_path.replace(path)


def start_rq3_csv_run(
    *,
    run_label="rq3",
    dataset=None,
    project_dir=None,
    archive_relative_dir=(
        "extendedPlotting/rq3_csv_run_archives"
    ),
    patterns=None,
    additional_patterns=None,
    backup_existing=True,
):
    """
    Start an RQ3 CSV archive.

    Run this before the RQ3 cells. Existing matching CSVs are copied into
    pre_run_snapshot/ by default, protecting the previous run before the fixed
    output locations are overwritten.
    """
    project_dir = Path(
        project_dir if project_dir is not None else Path.cwd()
    ).expanduser().resolve()

    extended_plotting_dir = project_dir / "extendedPlotting"
    archive_base_dir = project_dir / archive_relative_dir
    archive_base_dir.mkdir(parents=True, exist_ok=True)

    selected_patterns = list(
        patterns
        if patterns is not None
        else RQ3_DEFAULT_CSV_PATTERNS
    )
    if additional_patterns:
        selected_patterns.extend(
            str(pattern)
            for pattern in additional_patterns
        )
    selected_patterns = list(dict.fromkeys(selected_patterns))

    started = datetime.now(timezone.utc)
    dataset_token = _safe_token(dataset, "dataset-unknown")
    label_token = _safe_token(run_label, "rq3")
    timestamp = started.strftime("%Y%m%d_%H%M%S_%fZ")
    run_id = (
        f"{dataset_token}__{timestamp}"
        f"__{label_token}__{uuid.uuid4().hex[:8]}"
    )

    archive_dir = archive_base_dir / run_id
    archive_dir.mkdir(parents=False, exist_ok=False)

    existing_files = _collect_csv_files(
        extended_plotting_dir,
        archive_base_dir,
        selected_patterns,
    )

    baseline = {
        _relative_to_project(path, project_dir): _fingerprint(path)
        for path in existing_files
    }

    pre_run_rows = []

    if backup_existing:
        pre_run_root = archive_dir / "pre_run_snapshot"

        for source in existing_files:
            archived, relative = _copy_preserving_tree(
                source,
                pre_run_root,
                project_dir,
            )
            pre_run_rows.append({
                "status": "pre_run_snapshot",
                "changed_during_run": False,
                "source_path": str(source),
                "relative_path": relative,
                "archived_path": str(archived),
                "size_bytes": archived.stat().st_size,
                "modified_utc": _modified_utc(source),
                "sha256": _sha256(archived),
            })

        _write_csv_atomic(
            archive_dir / "pre_run_manifest.csv",
            pre_run_rows,
        )

    state = {
        "run_id": run_id,
        "run_label": label_token,
        "dataset": dataset_token,
        "project_dir": project_dir,
        "extended_plotting_dir": extended_plotting_dir,
        "archive_base_dir": archive_base_dir,
        "archive_dir": archive_dir,
        "patterns": tuple(selected_patterns),
        "started_at_utc": started.isoformat(),
        "baseline": baseline,
        "pre_run_file_count": len(existing_files),
        "backup_existing": bool(backup_existing),
        "finalized": False,
    }

    _write_json_atomic(
        archive_dir / "run_metadata.json",
        {
            "run_id": run_id,
            "run_label": label_token,
            "dataset": dataset_token,
            "project_dir": str(project_dir),
            "archive_dir": str(archive_dir),
            "started_at_utc": state["started_at_utc"],
            "finished_at_utc": None,
            "finalized": False,
            "backup_existing": bool(backup_existing),
            "pre_run_file_count": len(existing_files),
            "patterns": selected_patterns,
        },
    )

    (archive_base_dir / "latest_started_run.txt").write_text(
        str(archive_dir.resolve()),
        encoding="utf-8",
    )

    print("[RQ3 CSV archive] started:", run_id)
    print("[RQ3 CSV archive] directory:", archive_dir)
    print("[RQ3 CSV archive] existing matching CSVs:", len(existing_files))

    if backup_existing:
        print(
            "[RQ3 CSV archive] previous state protected in:",
            archive_dir / "pre_run_snapshot",
        )

    return state


def finish_rq3_csv_run(
    state,
    *,
    copy_all_current=True,
):
    """
    Finish the RQ3 CSV archive.

    copy_all_current=True produces a complete snapshot. run_manifest.csv marks
    each copied file as new, modified, or unchanged relative to the beginning
    of the run.
    """
    required_keys = {
        "run_id",
        "project_dir",
        "extended_plotting_dir",
        "archive_base_dir",
        "archive_dir",
        "patterns",
        "baseline",
        "started_at_utc",
    }

    if not isinstance(state, dict):
        raise TypeError(
            "state must be returned by start_rq3_csv_run()."
        )

    missing = required_keys.difference(state)
    if missing:
        raise KeyError(
            "Invalid RQ3 archive state. Missing: "
            + ", ".join(sorted(missing))
        )

    if state.get("finalized"):
        raise RuntimeError(
            f"Run {state['run_id']!r} is already finalized."
        )

    archive_dir = Path(state["archive_dir"])
    finished_marker = archive_dir / "FINISHED"

    if finished_marker.exists():
        raise RuntimeError(
            "This archive already contains a FINISHED marker."
        )

    current_files = _collect_csv_files(
        state["extended_plotting_dir"],
        state["archive_base_dir"],
        state["patterns"],
    )

    results_root = archive_dir / "results"
    result_rows = []
    status_counts = {
        "new": 0,
        "modified": 0,
        "unchanged": 0,
    }

    for source in current_files:
        relative = _relative_to_project(
            source,
            state["project_dir"],
        )
        baseline_fingerprint = state["baseline"].get(relative)
        current_fingerprint = _fingerprint(source)

        if baseline_fingerprint is None:
            status = "new"
        elif current_fingerprint != baseline_fingerprint:
            status = "modified"
        else:
            status = "unchanged"

        status_counts[status] += 1

        if not copy_all_current and status == "unchanged":
            continue

        archived, relative = _copy_preserving_tree(
            source,
            results_root,
            state["project_dir"],
        )

        result_rows.append({
            "status": status,
            "changed_during_run": status in {"new", "modified"},
            "source_path": str(source),
            "relative_path": relative,
            "archived_path": str(archived),
            "size_bytes": archived.stat().st_size,
            "modified_utc": _modified_utc(source),
            "sha256": _sha256(archived),
        })

    _write_csv_atomic(
        archive_dir / "run_manifest.csv",
        result_rows,
    )

    finished = datetime.now(timezone.utc)

    _write_json_atomic(
        archive_dir / "run_metadata.json",
        {
            "run_id": state["run_id"],
            "run_label": state.get("run_label"),
            "dataset": state.get("dataset"),
            "project_dir": str(state["project_dir"]),
            "archive_dir": str(archive_dir),
            "started_at_utc": state["started_at_utc"],
            "finished_at_utc": finished.isoformat(),
            "finalized": True,
            "backup_existing": state.get("backup_existing", False),
            "copy_all_current": bool(copy_all_current),
            "pre_run_file_count": state.get("pre_run_file_count", 0),
            "current_matching_file_count": len(current_files),
            "result_file_count": len(result_rows),
            "new_file_count": status_counts["new"],
            "modified_file_count": status_counts["modified"],
            "unchanged_file_count": status_counts["unchanged"],
            "patterns": list(state["patterns"]),
        },
    )

    finished_marker.write_text(
        finished.isoformat(),
        encoding="utf-8",
    )

    archive_base_dir = Path(state["archive_base_dir"])
    (archive_base_dir / "latest_run.txt").write_text(
        str(archive_dir.resolve()),
        encoding="utf-8",
    )

    state["finalized"] = True

    print("[RQ3 CSV archive] finalized:", state["run_id"])
    print("[RQ3 CSV archive] copied CSV files:", len(result_rows))
    print(
        "[RQ3 CSV archive] new / modified / unchanged:",
        status_counts["new"],
        "/",
        status_counts["modified"],
        "/",
        status_counts["unchanged"],
    )
    print("[RQ3 CSV archive] results:", results_root)
    print(
        "[RQ3 CSV archive] manifest:",
        archive_dir / "run_manifest.csv",
    )

    return archive_dir


def archive_current_rq3_csvs(
    *,
    run_label="manual-snapshot",
    dataset=None,
    project_dir=None,
    patterns=None,
    additional_patterns=None,
):
    """
    One-shot snapshot for a run that has already completed.
    """
    state = start_rq3_csv_run(
        run_label=run_label,
        dataset=dataset,
        project_dir=project_dir,
        patterns=patterns,
        additional_patterns=additional_patterns,
        backup_existing=False,
    )

    # Empty baseline means every current file is recorded as new.
    state["baseline"] = {}

    return finish_rq3_csv_run(
        state,
        copy_all_current=True,
    )
